### **Copy of revised_risk_aware_hybrid_pipeline_v2 To addres Extern Daset validation of R1.5 comment**

## Revision-package implementation notebook

This notebook retains historical pipeline, external-transfer, audit, and
packaging sections. It is not a verified linear Run All workflow.

The shared pipeline setup is aligned with the prepared supporting notebook.
Frozen deposited evidence remains under artifacts/. New audit work uses an
explicitly opted-in working copy, not the deposited originals.

For this notebook, package paths are defined in cell 8, revision output paths
in cell 42, and saved-artifact workspace support in cell 111.
The supporting notebook's guide uses that supporting notebook's own numbers.

The complete guarded external prediction function is retained in cell 272.
The preceding historical bodyless duplicate declaration has been omitted.

Local data inspection uses explicit INPUT_PATHS, or CRUS_INSPECTION_FILES
as a JSON list of filenames. Raw inputs, checkpoints, and other missing
components are not silently substituted or regenerated.

Saved outputs that remain are historical. Full external-transfer execution
and all supplementary sections have not been revalidated by preparation.
See PIPELINE_RELEASE_GUIDE.md and DATA_RECONSTRUCTION.md for shared path,
environment, raw-input, and verification-scope information.


### Release instructions and verified scope

Read EXTERNAL_TRANSFER_RELEASE_GUIDE.md before execution.

Structure, IPython-aware syntax, shared-source comparisons, and external-tail
path/service-normalised comparisons passed. Limited setup/helper execution,
artifact-copy integrity, and explicit local-file selection were tested.

The canonical notebook has been prepared for the revision package.
This is not a claim of full notebook, external-inference, or clean-environment
execution verification. Remaining saved outputs are historical.

The original notebook and temporary preparation copies are retained outside
the release package. Frozen deposited artifacts have not been overwritten.


# **SK4 code for revision 2**

# Revised Risk-Aware Hybrid Pipeline v2 — Bug-Fixed, Literature-Grounded

**This is a corrected version of `revised_risk_aware_hybrid_pipeline.ipynb`.**

## What changed from v1 and why

Running v1 revealed that the proposed Risk-Aware Hybrid underperformed its own no-abstention
ablation baseline (SK4 Hybrid) after test-set leakage was fixed. Root-cause analysis of
`case_audit_test.csv` showed that **71% of the proposed method's misclassifications collapsed
into GENHX**, the single largest and most narratively verbose MTS-Dialog class (23.4% of the
corpus; combined with FAM/SOCHX, the two largest classes cover 52.3% of records).

The mechanism is a **documented cross-encoder length bias**: scoring `(query, full_document_text)`
pairs favors long, lexically diverse candidate documents over short, terse ones (e.g., a
9-word chief-complaint query loses to a 200-word GENHX narrative purely on token overlap
volume). This is the same phenomenon that motivated pivoted document-length normalization
(Singhal, Buckley & Mitra, SIGIR 1996) and BM25's own length-normalization parameter `b`;
it is documented to persist in neural/cross-encoder rerankers unless corrected (Two-Stage
Document Length Normalization, TOIS 2015; length-normalization for neural rankers, EVIA 2019).
SK4's baseline reranks `(query, label_string)` pairs of near-identical length and is therefore
immune to this bias — which is *why* it appeared stronger, not because it is a better method.

## Fixes implemented here (all literature-grounded, none invented)

1. **RiskAware-Cascade (new primary proposed method).** A coarse-to-fine reranking cascade
   (Nogueira & Cho 2019 monoBERT-style staged retrieval; Gao & Callan's modular long-document
   reranker, SIGIR 2022): Stage A uses SK4-style label-string cross-encoder scoring as an
   unbiased coarse filter to shortlist the top-K candidate classes; Stage B reranks only within
   that shortlist using the paper's original full-document-text cross-encoder signal, with each
   candidate document **length-bounded** (truncated to a fixed character window, standard
   passage-ranking practice) to remove the raw length advantage. This preserves the paper's
   stated novelty (full-document reranking) while removing the majority-class attractor.
2. **RiskAware-LengthNorm (new ablation).** The *original* single-stage full-document
   cross-encoder pipeline, but with pivoted-style length normalization applied to raw
   cross-encoder logits before softmax — isolating how much of the gain (if any) comes from
   length correction alone, versus the cascade structure.
3. **CRUS-ablation bug fix.** In v1, the "Full CRUS" row of the component ablation
   independently re-optimized its own (tau_c, tau_u) instead of reusing the frozen Section-15
   values, producing an inconsistency with `config.json`. This version freezes and reuses the
   *same* validation-selected thresholds for the "Full CRUS" ablation row, and only lets the
   three *component-only* variants (H_pred, H_retrieval, V_scores) run their own independent
   selection, since they are genuinely different signals.
4. Everything validated in v1 is preserved unchanged: 60/20/20 split, threshold selection on
   validation only, coverage-matched comparison, risk-coverage curves, Mann-Whitney U + effect
   size + CI for the CRUS separation claim, single rare-class filter applied once.

No fabricated numbers are used anywhere in this notebook — every reported figure is computed
from real model inference over the MTS-Dialog train/validation/test splits at runtime.


In [ ]:
# -- 1. INSTALL ---------------------------------------------------------------
!pip install -q pandas numpy scikit-learn sentence-transformers \
    rank-bm25 transformers openpyxl matplotlib seaborn scipy


In [ ]:
# -- 2. IMPORTS -----------------------------------------------------------------
import os, random, warnings, json
from collections import Counter
from datetime import datetime
from itertools import product

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score, cohen_kappa_score,
    confusion_matrix, precision_recall_fscore_support
)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV

from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi
from scipy.stats import mannwhitneyu
try:
    from google.colab import drive
except ImportError:
    drive = None  # Initial setup does not require Google Drive.


In [ ]:
# -- 3. REPRODUCIBILITY ---------------------------------------------------------
SEED = 42
random.seed(SEED); np.random.seed(SEED)
warnings.filterwarnings("ignore")


In [ ]:
# -- 4. CONFIG --------------------------------------------------------------
CFG = dict(
    seed = SEED,
    train_frac = 0.60,
    val_frac = 0.20,
    test_frac = 0.20,
    min_class_cnt = 5,
    bm25_k1 = 1.5,
    bm25_b = 0.75,
    topk_retrieve = 20,
    rrf_k = 60,
    rrf_top_n = 10,
    crus_lambda1 = 0.5,
    crus_lambda2 = 0.3,
    crus_lambda3 = 0.2,
    tau_c = None,            # selected on validation, Section 16 (applies to RiskAware-Cascade)
    tau_u = None,            # selected on validation, Section 16
    min_coverage_constraint = 0.50,
    bootstrap_n = 1000,
    dense_model = "sentence-transformers/all-MiniLM-L6-v2",
    ce_model = "cross-encoder/ms-marco-MiniLM-L-6-v2",
    cascade_top_k_classes = 5,
    doc_truncate_chars = 400,
    length_norm_alpha = 0.5,
)


In [ ]:
# -- 5. INITIAL PIPELINE: PORTABLE PATH SETUP -------------------------------
from pathlib import Path
from datetime import datetime
import os

_configured_root = os.environ.get("CRUS_PACKAGE_ROOT")
if _configured_root:
    PACKAGE_ROOT = Path(_configured_root).expanduser().resolve()
else:
    _cwd = Path.cwd().resolve()
    _candidates = [
        p for p in (_cwd, *_cwd.parents)
        if (p / "artifacts").is_dir() and (p / "notebooks").is_dir()
    ]
    if not _candidates:
        raise RuntimeError(
            "Start Jupyter from the extracted package directory, or set "
            "CRUS_PACKAGE_ROOT before starting the notebook kernel."
        )
    PACKAGE_ROOT = _candidates[0]

if not (
    (PACKAGE_ROOT / "artifacts").is_dir()
    and (PACKAGE_ROOT / "notebooks").is_dir()
):
    raise RuntimeError("CRUS_PACKAGE_ROOT is not the expected package folder.")

ARCHIVE_ROOT = (PACKAGE_ROOT / "artifacts").resolve()

# This directory must directly contain the two source CSVs.
RAW_DATA_ROOT = Path(os.environ.get(
    "CRUS_RAW_DATA_ROOT",
    str(PACKAGE_ROOT / "local_inputs" / "MTS-Dialog" / "Main-Dataset")
)).expanduser().resolve()

_RUN_STAMP = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
RUN_OUTPUT_ROOT = Path(os.environ.get(
    "CRUS_RUN_OUTPUT_ROOT",
    str(PACKAGE_ROOT / "local_runs" / _RUN_STAMP)
)).expanduser().resolve()

for _protected in (ARCHIVE_ROOT, RAW_DATA_ROOT):
    if RUN_OUTPUT_ROOT == _protected or _protected in RUN_OUTPUT_ROOT.parents:
        raise RuntimeError(
            "New outputs must not be stored inside archived artifacts "
            "or the raw-input directory."
        )

BASE = str(RAW_DATA_ROOT)
TRAIN_CSV = str(RAW_DATA_ROOT / "MTS-Dialog-TrainingSet.csv")
VAL_CSV = str(RAW_DATA_ROOT / "MTS-Dialog-ValidationSet.csv")
TODAY = datetime.now().strftime("%Y-%m-%d")
OUTDIR = str(RUN_OUTPUT_ROOT / "initial_pipeline")
Path(OUTDIR).mkdir(parents=True, exist_ok=True)

print("Package:", PACKAGE_ROOT)
print("Archived artifacts:", ARCHIVE_ROOT)
print("Raw inputs:", RAW_DATA_ROOT)
print("New initial-pipeline outputs:", OUTDIR)
print("No Google Drive mount or automatic download is required by this setup.")


In [ ]:
# Targeted input checks; no recursive filesystem or Drive search.
from pathlib import Path

for _label, _filename in [
    ("Training", TRAIN_CSV),
    ("Validation", VAL_CSV),
]:
    _path = Path(_filename)
    print(f'{_label}: {"FOUND" if _path.is_file() else "MISSING"} -- {_path}')

if not all(Path(p).is_file() for p in (TRAIN_CSV, VAL_CSV)):
    print(
        "Raw dialogue inputs are omitted from this deposit. Follow "
        "DATA_RECONSTRUCTION.md, including the recorded version and "
        "checksum checks. If stored elsewhere, set CRUS_RAW_DATA_ROOT "
        "before starting the kernel. Do not silently substitute another "
        "dataset version. Raw-input-dependent cells cannot run until "
        "the required files have been obtained and verified."
    )


In [ ]:
# -- 6. LOAD & PREPROCESS DATA (single authoritative rare-class filter) ------
print("Loading MTS-Dialog ...")
df = pd.concat([pd.read_csv(TRAIN_CSV), pd.read_csv(VAL_CSV)], ignore_index=True)
df["text"] = (
    df["section_header"].fillna("") + " " +
    df["section_text"].fillna("") + " " +
    df["dialogue"].fillna("")
).str.lower()
df["label"] = df["section_header"]

vc_full = df["label"].value_counts()
excluded_classes = sorted(vc_full[vc_full < CFG["min_class_cnt"]].index.tolist())
df = df[df["label"].isin(vc_full[vc_full >= CFG["min_class_cnt"]].index)].copy()
df = df.reset_index(drop=True)

CLASSES = sorted(df["label"].unique().tolist())
N_CLS = len(CLASSES)
CIDX = {c: i for i, c in enumerate(CLASSES)}

print(f"Dataset : {len(df)} rows | {N_CLS} classes (excluded {excluded_classes})")
print("Class distribution:\n", df["label"].value_counts().to_string())

doc_len_by_class = df.groupby("label")["text"].apply(lambda s: s.str.len().mean()).sort_values(ascending=False)
print("\nMean document character length by class (diagnostic for length-bias hypothesis):")
print(doc_len_by_class.to_string())


In [ ]:
# -- 7. TRAIN / VALIDATION / TEST SPLIT ---------------------------------------
X_all = df["text"].values
y_all = df["label"].values

X_trainval, X_test, y_trainval, y_test = train_test_split(
    X_all, y_all, test_size=CFG["test_frac"], random_state=SEED, stratify=y_all
)
val_relative_frac = CFG["val_frac"] / (CFG["train_frac"] + CFG["val_frac"])
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=val_relative_frac, random_state=SEED, stratify=y_trainval
)

y_train_np = np.array(y_train)
y_val_np = np.array(y_val)
y_test_np = np.array(y_test)

print(f"Train : {len(X_train)} ({len(X_train)/len(df):.1%}) | "
      f"Val : {len(X_val)} ({len(X_val)/len(df):.1%}) | "
      f"Test : {len(X_test)} ({len(X_test)/len(df):.1%})")
assert set(y_train_np) == set(CLASSES)
print("Class coverage check passed.")


In [ ]:
# -- 8. BM25 INDEX --------------------------------------------------------------
print("Building BM25 index on TRAIN split ...")
tokenized = [t.split() for t in X_train]
bm25 = BM25Okapi(tokenized, k1=CFG["bm25_k1"], b=CFG["bm25_b"])

def bm25_retrieve(q, k=20):
    sc = bm25.get_scores(q.split())
    idx = np.argsort(sc)[::-1][:k]
    return [(int(i), float(sc[i])) for i in idx]


In [ ]:
# -- 9. SBERT DENSE ENCODER ------------------------------------------------------
print(f"Loading SBERT: {CFG['dense_model']} ...")
embedder = SentenceTransformer(CFG["dense_model"])
train_embs = embedder.encode(X_train, normalize_embeddings=True, show_progress_bar=True)

def dense_retrieve(q, k=20):
    qv = embedder.encode([q], normalize_embeddings=True)[0]
    sims = train_embs.dot(qv)
    idx = np.argsort(sims)[::-1][:k]
    return [(int(i), float(sims[i])) for i in idx]


In [ ]:
# -- 10. CROSS-ENCODER --------------------------------------------------------
print(f"Loading cross-encoder: {CFG['ce_model']} ...")
cross_enc = CrossEncoder(CFG["ce_model"])


In [ ]:
# -- 11. HELPER FUNCTIONS: FUSION, ENTROPY/VARIANCE, CRUS --------------------
def rrf_fuse(sp_list, de_list, k=60, top_n=10):
    sc = {}
    for r, (i, _) in enumerate(sp_list): sc[i] = sc.get(i, 0) + 1.0/(k+r+1)
    for r, (i, _) in enumerate(de_list): sc[i] = sc.get(i, 0) + 1.0/(k+r+1)
    return sorted(sc.items(), key=lambda x: x[1], reverse=True)[:top_n]

def predictive_entropy(probs):
    p = np.clip(probs, 1e-10, 1.0)
    return float(-np.sum(p * np.log(p)) / np.log(max(2, len(p))))

def retrieval_label_entropy(labels):
    c = Counter(labels); total = sum(c.values())
    p = np.clip([v/total for v in c.values()], 1e-10, 1.0)
    return float(-np.sum(p * np.log(p)) / np.log(max(2, N_CLS)))

def retrieval_score_variance(scores):
    if len(scores) < 2: return 0.0
    mx = max(scores) + 1e-10
    return min(1.0, float(np.var(scores)) / mx**2)

def compute_crus(probs, ctx_labels, ctx_scores):
    return round(
        CFG["crus_lambda1"] * predictive_entropy(probs) +
        CFG["crus_lambda2"] * retrieval_label_entropy(ctx_labels) +
        CFG["crus_lambda3"] * retrieval_score_variance(ctx_scores), 5
    )

def one_hot_probs(preds):
    m = np.zeros((len(preds), N_CLS))
    for i, p in enumerate(preds): m[i, CIDX[p]] = 1.0
    return m


In [ ]:
# -- 12. SK4 ORIGINAL BASELINE (unchanged; label-union + CE on label strings) ---
def predict_sk4_original(q):
    """Ablation baseline: candidate pool = union of class labels from BM25 top-20 & SBERT
    top-20; cross-encoder scores (query,label_string) pairs; softmax -> class probs.
    No abstention. This is immune to length bias because label strings are equal length."""
    cands_b = set(y_train_np[[i for i, _ in bm25_retrieve(q, 20)]])
    cands_d = set(y_train_np[[i for i, _ in dense_retrieve(q, 20)]])
    cands = list(cands_b | cands_d)
    if not cands:
        return y_train_np[0], 0.5, np.ones(N_CLS)/N_CLS

    ce_sc = cross_enc.predict([(q, c) for c in cands])
    e = np.exp(ce_sc - np.max(ce_sc)); pc = e / e.sum()
    probs = np.zeros(N_CLS)
    for c, p in zip(cands, pc): probs[CIDX[c]] += p

    pred = CLASSES[int(np.argmax(probs))]
    conf = float(probs[CIDX[pred]])
    return pred, conf, probs


In [ ]:
# -- 13. NEW PROPOSED METHOD: RiskAware-Cascade (length-bias-corrected) ------
def stage_a_label_shortlist(q, top_k):
    cands_b = set(y_train_np[[i for i, _ in bm25_retrieve(q, 20)]])
    cands_d = set(y_train_np[[i for i, _ in dense_retrieve(q, 20)]])
    cands = list(cands_b | cands_d)
    if not cands:
        return list(CLASSES[:top_k])
    ce_sc = cross_enc.predict([(q, c) for c in cands])
    e = np.exp(ce_sc - np.max(ce_sc)); pc = e / e.sum()
    ranked = sorted(zip(cands, pc), key=lambda x: x[1], reverse=True)
    return [c for c, _ in ranked[:top_k]]

def predict_risk_aware_cascade(q):
    shortlist = set(stage_a_label_shortlist(q, CFG["cascade_top_k_classes"]))

    sp = bm25_retrieve(q, CFG["topk_retrieve"])
    de = dense_retrieve(q, CFG["topk_retrieve"])
    fused = rrf_fuse(sp, de, k=CFG["rrf_k"], top_n=CFG["rrf_top_n"] * 2)
    fused_in_shortlist = [(i, s) for (i, s) in fused if y_train_np[i] in shortlist]
    if not fused_in_shortlist:
        fused_in_shortlist = fused[:CFG["rrf_top_n"]]
    fused_in_shortlist = fused_in_shortlist[:CFG["rrf_top_n"]]

    doc_idx = [i for i, _ in fused_in_shortlist]
    doc_texts = [X_train[i][:CFG["doc_truncate_chars"]] for i in doc_idx]

    ce_sc = cross_enc.predict([(q, d) for d in doc_texts])
    e = np.exp(ce_sc - np.max(ce_sc)); dp = e / e.sum()

    class_scores = np.zeros(N_CLS)
    for (di, _), d in zip(fused_in_shortlist, dp):
        class_scores[CIDX[y_train_np[di]]] += d
    class_scores /= class_scores.sum() + 1e-10

    pred = CLASSES[int(np.argmax(class_scores))]
    conf = float(class_scores[CIDX[pred]])
    ctx_labels = [y_train_np[i] for i in doc_idx]
    ctx_scores = [s for _, s in fused_in_shortlist]

    h_pred = predictive_entropy(class_scores)
    h_ret = retrieval_label_entropy(ctx_labels)
    v_sc = retrieval_score_variance(ctx_scores)
    return pred, conf, class_scores, h_pred, h_ret, v_sc


In [ ]:
# -- 14. ABLATION: RiskAware-LengthNorm (original full-doc CE + pivoted length norm) -----
avg_train_doc_len = float(np.mean([len(t) for t in X_train]))

def predict_risk_aware_lengthnorm(q):
    sp = bm25_retrieve(q, CFG["topk_retrieve"])
    de = dense_retrieve(q, CFG["topk_retrieve"])
    fused = rrf_fuse(sp, de, k=CFG["rrf_k"], top_n=CFG["rrf_top_n"])
    if not fused:
        return y_train_np[0], 0.5, np.ones(N_CLS)/N_CLS, 0.0, 0.0, 0.0

    doc_idx = [i for i, _ in fused]
    doc_texts = [X_train[i] for i in doc_idx]
    doc_lens = np.array([len(d) for d in doc_texts])

    ce_sc_raw = cross_enc.predict([(q, d) for d in doc_texts])
    length_penalty = 1.0 + CFG["length_norm_alpha"] * np.log(np.clip(doc_lens / avg_train_doc_len, 1e-6, None))
    ce_sc = ce_sc_raw / np.clip(length_penalty, 1e-6, None)

    e = np.exp(ce_sc - np.max(ce_sc)); dp = e / e.sum()
    class_scores = np.zeros(N_CLS)
    for (di, _), d in zip(fused, dp):
        class_scores[CIDX[y_train_np[di]]] += d
    class_scores /= class_scores.sum() + 1e-10

    pred = CLASSES[int(np.argmax(class_scores))]
    conf = float(class_scores[CIDX[pred]])
    ctx_labels = [y_train_np[i] for i in doc_idx]
    ctx_scores = [s for _, s in fused]

    h_pred = predictive_entropy(class_scores)
    h_ret = retrieval_label_entropy(ctx_labels)
    v_sc = retrieval_score_variance(ctx_scores)
    return pred, conf, class_scores, h_pred, h_ret, v_sc


In [ ]:
# -- 15. RUN SCORER SWEEPS ON VAL AND TEST (cache once each) ------------------
def run_cascade_on_split(X_split):
    preds, confs, probs_list, hpred_l, hret_l, vsc_l = [], [], [], [], [], []
    for q in X_split:
        pred, conf, probs, hp, hr, vs = predict_risk_aware_cascade(q)
        preds.append(pred); confs.append(conf); probs_list.append(probs)
        hpred_l.append(hp); hret_l.append(hr); vsc_l.append(vs)
    return (np.array(preds), np.array(confs), np.array(probs_list),
            np.array(hpred_l), np.array(hret_l), np.array(vsc_l))

def run_lengthnorm_on_split(X_split):
    preds, confs, probs_list, hpred_l, hret_l, vsc_l = [], [], [], [], [], []
    for q in X_split:
        pred, conf, probs, hp, hr, vs = predict_risk_aware_lengthnorm(q)
        preds.append(pred); confs.append(conf); probs_list.append(probs)
        hpred_l.append(hp); hret_l.append(hr); vsc_l.append(vs)
    return (np.array(preds), np.array(confs), np.array(probs_list),
            np.array(hpred_l), np.array(hret_l), np.array(vsc_l))

print("Scoring VALIDATION split: RiskAware-Cascade (proposed) ...")
val_c_preds, val_c_confs, val_c_probs, val_c_hpred, val_c_hret, val_c_vsc = run_cascade_on_split(X_val)
print("Scoring TEST split: RiskAware-Cascade (proposed) ...")
test_c_preds, test_c_confs, test_c_probs, test_c_hpred, test_c_hret, test_c_vsc = run_cascade_on_split(X_test)

print("Scoring VALIDATION split: RiskAware-LengthNorm (ablation) ...")
val_l_preds, val_l_confs, val_l_probs, val_l_hpred, val_l_hret, val_l_vsc = run_lengthnorm_on_split(X_val)
print("Scoring TEST split: RiskAware-LengthNorm (ablation) ...")
test_l_preds, test_l_confs, test_l_probs, test_l_hpred, test_l_hret, test_l_vsc = run_lengthnorm_on_split(X_test)

val_c_crus = np.round(CFG["crus_lambda1"]*val_c_hpred + CFG["crus_lambda2"]*val_c_hret + CFG["crus_lambda3"]*val_c_vsc, 5)
test_c_crus = np.round(CFG["crus_lambda1"]*test_c_hpred + CFG["crus_lambda2"]*test_c_hret + CFG["crus_lambda3"]*test_c_vsc, 5)
val_l_crus = np.round(CFG["crus_lambda1"]*val_l_hpred + CFG["crus_lambda2"]*val_l_hret + CFG["crus_lambda3"]*val_l_vsc, 5)
test_l_crus = np.round(CFG["crus_lambda1"]*test_l_hpred + CFG["crus_lambda2"]*test_l_hret + CFG["crus_lambda3"]*test_l_vsc, 5)


In [ ]:
# -- 16. THRESHOLD SELECTION ON VALIDATION ONLY (for RiskAware-Cascade, primary proposed method) --
tau_c_grid = np.round(np.arange(0.10, 0.71, 0.05), 2)
tau_u_grid = np.round(np.arange(0.05, 0.91, 0.05), 2)

def select_thresholds(conf, crus, correct_mask, tc_grid, tu_grid, min_coverage):
    best = None
    for tc, tu in product(tc_grid, tu_grid):
        abstain = (conf < tc) | (crus > tu)
        covered = ~abstain
        cov = covered.mean()
        if cov < min_coverage or covered.sum() == 0:
            continue
        acc = correct_mask[covered].mean()
        if best is None or acc > best[0] or (acc == best[0] and cov > best[3]):
            best = (acc, tc, tu, cov)
    return best

val_c_correct = (val_c_preds == y_val_np)
selection_c = select_thresholds(val_c_confs, val_c_crus, val_c_correct,
                                 tau_c_grid, tau_u_grid, CFG["min_coverage_constraint"])
assert selection_c is not None, "No threshold pair met coverage constraint (Cascade)."
best_val_acc_c, TAU_C, TAU_U, best_val_cov_c = selection_c
CFG["tau_c"] = float(TAU_C); CFG["tau_u"] = float(TAU_U)
print(f"[RiskAware-Cascade] Selected on VALIDATION -> tau_c={TAU_C}, tau_u={TAU_U} "
      f"(val covered_acc={best_val_acc_c:.4f}, val_coverage={best_val_cov_c:.4f})")

val_l_correct = (val_l_preds == y_val_np)
selection_l = select_thresholds(val_l_confs, val_l_crus, val_l_correct,
                                 tau_c_grid, tau_u_grid, CFG["min_coverage_constraint"])
assert selection_l is not None, "No threshold pair met coverage constraint (LengthNorm)."
best_val_acc_l, TAU_C_L, TAU_U_L, best_val_cov_l = selection_l
print(f"[RiskAware-LengthNorm] Selected on VALIDATION -> tau_c={TAU_C_L}, tau_u={TAU_U_L} "
      f"(val covered_acc={best_val_acc_l:.4f}, val_coverage={best_val_cov_l:.4f})")


In [ ]:
# -- 17. APPLY FROZEN THRESHOLDS TO TEST (touched once each, for final reporting) --------
test_c_abstain = (test_c_confs < CFG["tau_c"]) | (test_c_crus > CFG["tau_u"])
test_c_covered_mask = ~test_c_abstain
test_c_coverage = round(float(test_c_covered_mask.mean()), 4)

test_l_abstain = (test_l_confs < TAU_C_L) | (test_l_crus > TAU_U_L)
test_l_covered_mask = ~test_l_abstain
test_l_coverage = round(float(test_l_covered_mask.mean()), 4)

print("RiskAware-Cascade  (TEST): coverage={:.4f}, abstained={}/{}, covered_acc={:.4f}, all_acc={:.4f}".format(
    test_c_coverage, int(test_c_abstain.sum()), len(X_test),
    accuracy_score(y_test_np[test_c_covered_mask], test_c_preds[test_c_covered_mask]),
    accuracy_score(y_test_np, test_c_preds)))
print("RiskAware-LengthNorm (TEST): coverage={:.4f}, abstained={}/{}, covered_acc={:.4f}, all_acc={:.4f}".format(
    test_l_coverage, int(test_l_abstain.sum()), len(X_test),
    accuracy_score(y_test_np[test_l_covered_mask], test_l_preds[test_l_covered_mask]),
    accuracy_score(y_test_np, test_l_preds)))


In [ ]:
# -- 18. SUPERVISED & RETRIEVAL BASELINES (trained on TRAIN, evaluated on TEST) ----------
print("Training supervised baselines on TRAIN split ...")
tfidf = TfidfVectorizer(ngram_range=(1,2), max_features=18000, sublinear_tf=True)
Xtr_tf = tfidf.fit_transform(X_train)
Xte_tf = tfidf.transform(X_test)

lr = LogisticRegression(max_iter=2000, C=1.0, random_state=SEED, n_jobs=-1)
lr.fit(Xtr_tf, y_train)
lr_pr = lr.predict(Xte_tf); lr_pb = lr.predict_proba(Xte_tf)

lsvm = CalibratedClassifierCV(LinearSVC(max_iter=3000, random_state=SEED))
lsvm.fit(Xtr_tf, y_train)
lsvm_pr = lsvm.predict(Xte_tf); lsvm_pb = lsvm.predict_proba(Xte_tf)

print(f"TF-IDF+LR Acc: {accuracy_score(y_test, lr_pr):.4f}")
print(f"TF-IDF+SVM Acc: {accuracy_score(y_test, lsvm_pr):.4f}")

print("Running BM25-only / SBERT-only baselines on TEST ...")
bm25_preds = np.array([y_train_np[bm25_retrieve(q, 1)[0][0]] for q in X_test])
sbert_preds = np.array([y_train_np[dense_retrieve(q, 1)[0][0]] for q in X_test])
print(f"BM25 Only Acc : {accuracy_score(y_test, bm25_preds):.4f}")
print(f"SBERT Only Acc: {accuracy_score(y_test, sbert_preds):.4f}")


In [ ]:
# -- 19. SK4 HYBRID (NO-ABSTAIN) ABLATION ON TEST -----------------------------
print("Running SK4 Hybrid (original, no-abstain) on TEST ...")
sk4_preds, sk4_confs, sk4_probs = [], [], []
for q in X_test:
    pred, conf, probs = predict_sk4_original(q)
    sk4_preds.append(pred); sk4_confs.append(conf); sk4_probs.append(probs)
sk4_preds = np.array(sk4_preds); sk4_confs = np.array(sk4_confs); sk4_probs_mat = np.array(sk4_probs)
print(f"SK4 Hybrid Acc: {accuracy_score(y_test, sk4_preds):.4f}")


In [ ]:
# -- 20. METRIC HELPERS -------------------------------------------------------
def bootstrap_ci_f1(yt, yp, n=1000, seed=42):
    rng = np.random.default_rng(seed)
    yt = np.array(yt); yp = np.array(yp)
    scores = []
    for _ in range(n):
        idx = rng.integers(0, len(yt), len(yt))
        scores.append(f1_score(yt[idx], yp[idx], average="macro", zero_division=0))
    return (round(np.percentile(scores, 2.5), 4), round(np.percentile(scores, 97.5), 4))

def compute_ece(y_true_idx, prob_matrix, n_bins=10):
    n = len(y_true_idx)
    top = prob_matrix[np.arange(n), y_true_idx]
    cor = (np.argmax(prob_matrix, 1) == y_true_idx).astype(float)
    ece = 0.0
    for lo in np.linspace(0, 1, n_bins+1)[:-1]:
        m = (top >= lo) & (top < lo + 1.0/n_bins)
        if m.sum() > 0:
            ece += m.sum()/n * abs(cor[m].mean() - top[m].mean())
    return round(ece, 4)

def compute_brier(y_true_idx, prob_matrix):
    y_oh = np.eye(N_CLS)[y_true_idx]
    return round(float(np.mean(np.sum((prob_matrix - y_oh)**2, axis=1))), 4)

def full_metrics(yt_list, yp_array, prob_mat, coverage=1.0, abstained=0):
    yt = np.array(yt_list)
    yt_i = np.array([CIDX[c] for c in yt])
    acc = accuracy_score(yt, yp_array)
    f1m = f1_score(yt, yp_array, average="macro", zero_division=0)
    f1w = f1_score(yt, yp_array, average="weighted", zero_division=0)
    kap = cohen_kappa_score(yt, yp_array)
    ece = compute_ece(yt_i, prob_mat)
    bri = compute_brier(yt_i, prob_mat)
    ci = bootstrap_ci_f1(yt, yp_array, n=CFG["bootstrap_n"])
    return dict(acc=round(acc,4), f1m=round(f1m,4), f1w=round(f1w,4),
                kappa=round(kap,4), ece=ece, brier=bri,
                unsafe=round(1.0-acc,4), ci=ci, coverage=coverage, abstained=abstained)


In [ ]:
# -- 21. RESULTS TABLE: ALL METHODS INCLUDING THE TWO NEW VARIANTS -----------
RESULTS = {}
RESULTS["TF-IDF + LR"] = full_metrics(y_test_np, lr_pr, lr_pb)
RESULTS["TF-IDF + LinearSVM"] = full_metrics(y_test_np, lsvm_pr, lsvm_pb)
RESULTS["BM25 Only"] = full_metrics(y_test_np, bm25_preds, one_hot_probs(bm25_preds))
RESULTS["SBERT Only"] = full_metrics(y_test_np, sbert_preds, one_hot_probs(sbert_preds))
RESULTS["SK4 Hybrid (no-abstain)"] = full_metrics(y_test_np, sk4_preds, sk4_probs_mat)

RESULTS["RiskAware-LengthNorm (all, abstain=wrong)"] = full_metrics(
    y_test_np, test_l_preds, test_l_probs, coverage=1.0, abstained=int(test_l_abstain.sum()))
yt_cov_l = y_test_np[test_l_covered_mask]; yp_cov_l = test_l_preds[test_l_covered_mask]; pb_cov_l = test_l_probs[test_l_covered_mask]
RESULTS["RiskAware-LengthNorm (covered subset only)"] = full_metrics(
    yt_cov_l, yp_cov_l, pb_cov_l, coverage=test_l_coverage, abstained=int(test_l_abstain.sum()))

RESULTS["RiskAware-Cascade (all, abstain=wrong)"] = full_metrics(
    y_test_np, test_c_preds, test_c_probs, coverage=1.0, abstained=int(test_c_abstain.sum()))
yt_cov_c = y_test_np[test_c_covered_mask]; yp_cov_c = test_c_preds[test_c_covered_mask]; pb_cov_c = test_c_probs[test_c_covered_mask]
RESULTS["RiskAware-Cascade (covered subset only)"] = full_metrics(
    yt_cov_c, yp_cov_c, pb_cov_c, coverage=test_c_coverage, abstained=int(test_c_abstain.sum()))

print("\n" + "="*118)
print(f"{'Method':<42} {'Acc':>7} {'MacF1':>7} {'Kappa':>7} {'ECE':>6} "
      f"{'Unsafe%':>8} {'Cov':>6} {'CI95(MacF1)':>18}")
print("="*118)
for name, v in RESULTS.items():
    print(f"{name:<42} {v['acc']:>7.4f} {v['f1m']:>7.4f} {v['kappa']:>7.4f} "
          f"{v['ece']:>6.4f} {v['unsafe']*100:>7.2f}% {v['coverage']:>6.3f} "
          f"[{v['ci'][0]:.4f},{v['ci'][1]:.4f}]")
print("="*118)


In [ ]:
# -- 22. COVERAGE-MATCHED COMPARISON (matched to RiskAware-Cascade native coverage) ---------
def coverage_matched_accuracy(conf_scores, correct_mask, target_coverage):
    n = len(conf_scores)
    k = max(1, int(round(target_coverage * n)))
    order = np.argsort(-conf_scores)
    top_idx = order[:k]
    return float(correct_mask[top_idx].mean()), k

lr_conf = lr_pb.max(axis=1); lsvm_conf = lsvm_pb.max(axis=1)
lr_correct = (lr_pr == y_test_np); lsvm_correct = (lsvm_pr == y_test_np)
sk4_correct = (sk4_preds == y_test_np)
cascade_correct = (test_c_preds == y_test_np)
lengthnorm_correct = (test_l_preds == y_test_np)

matched_rows = []
for name, conf_arr, corr_arr in [
    ("TF-IDF + LR", lr_conf, lr_correct),
    ("TF-IDF + LinearSVM", lsvm_conf, lsvm_correct),
    ("SK4 Hybrid (no-abstain)", sk4_confs, sk4_correct),
    ("RiskAware-LengthNorm (native)", test_l_confs, lengthnorm_correct),
]:
    acc_m, k = coverage_matched_accuracy(conf_arr, corr_arr, test_c_coverage)
    matched_rows.append(dict(method=name, matched_coverage=round(k/len(X_test),4),
                              n_selected=k, selective_acc=round(acc_m,4)))
matched_rows.append(dict(method="RiskAware-Cascade (native abstention)",
                          matched_coverage=test_c_coverage,
                          n_selected=int(test_c_covered_mask.sum()),
                          selective_acc=round(float(cascade_correct[test_c_covered_mask].mean()),4)))
matched_df = pd.DataFrame(matched_rows)
print(f"Coverage-matched comparison at target coverage = {test_c_coverage:.3f}:\n")
print(matched_df.to_string(index=False))
matched_df.to_csv(f"{OUTDIR}/coverage_matched_comparison_v2.csv", index=False)


In [ ]:
# -- 23. RISK-COVERAGE CURVES (all methods incl. both new variants) ----------
def risk_coverage_curve(conf_scores, correct_mask, steps=19):
    n = len(conf_scores)
    order = np.argsort(-conf_scores)
    rows = []
    for frac in np.linspace(0.05, 1.0, steps):
        k = max(1, int(round(frac * n)))
        idx = order[:k]
        acc = correct_mask[idx].mean()
        rows.append((frac, acc, 1 - acc))
    return pd.DataFrame(rows, columns=["coverage", "selective_acc", "selective_error"])

curves = {
    "TF-IDF + LinearSVM": risk_coverage_curve(lsvm_conf, lsvm_correct),
    "SK4 Hybrid (no-abstain)": risk_coverage_curve(sk4_confs, sk4_correct),
    "RiskAware-LengthNorm": risk_coverage_curve(test_l_confs, lengthnorm_correct),
    "RiskAware-Cascade (proposed)": risk_coverage_curve(test_c_confs, cascade_correct),
}
fig, ax = plt.subplots(figsize=(8, 5))
for name, cdf in curves.items():
    ax.plot(cdf["coverage"], cdf["selective_error"], marker="o", markersize=3, label=name)
ax.axvline(test_c_coverage, color="black", linestyle="--", alpha=0.6,
           label=f"Cascade native coverage={test_c_coverage:.3f}")
ax.set_xlabel("Coverage"); ax.set_ylabel("Selective Error (1 - Accuracy)")
ax.set_title("Risk-Coverage Curves (TEST set) -- v2 with bug-fixed proposed method")
ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f"{OUTDIR}/risk_coverage_curves_v2.png", dpi=300); plt.close()
print(f"Risk-coverage curves -> {OUTDIR}/risk_coverage_curves_v2.png")


In [ ]:
# -- 24. CRUS COMPONENT-WISE ABLATION (BUG FIX: "Full CRUS" reuses FROZEN thresholds) --------
test_c_correct_full = (test_c_preds == y_test_np)

ab_full = (test_c_confs < CFG["tau_c"]) | (test_c_crus > CFG["tau_u"])
cov_full = (~ab_full).mean()
acc_full = test_c_correct_full[~ab_full].mean() if (~ab_full).sum() > 0 else np.nan
unsafe_full = 1.0 - (test_c_correct_full & (~ab_full)).sum() / len(test_c_correct_full)

ablation_rows = [dict(signal="Full CRUS (frozen thresholds, matches primary result)",
                       tau_c=CFG["tau_c"], tau_u=CFG["tau_u"],
                       test_coverage=round(float(cov_full),4),
                       test_covered_acc=round(float(acc_full),4),
                       test_unsafe_all=round(float(unsafe_full),4))]

component_signals = {
    "H_pred only": (val_c_hpred, test_c_hpred),
    "H_retrieval only": (val_c_hret, test_c_hret),
    "V_scores only": (val_c_vsc, test_c_vsc),
}
val_c_correct_full = (val_c_preds == y_val_np)
for sig_name, (val_signal, test_signal) in component_signals.items():
    u_max = max(val_signal.max(), test_signal.max(), 1e-6)
    tu_grid_local = np.round(np.linspace(0.0, u_max, 18), 4)
    sel = select_thresholds(val_c_confs, val_signal, val_c_correct_full,
                             tau_c_grid, tu_grid_local, CFG["min_coverage_constraint"])
    if sel is None:
        ablation_rows.append(dict(signal=sig_name, tau_c=np.nan, tau_u=np.nan,
                                   test_coverage=np.nan, test_covered_acc=np.nan, test_unsafe_all=np.nan))
        continue
    _, tc_sel, tu_sel, _ = sel
    ab_test = (test_c_confs < tc_sel) | (test_signal > tu_sel)
    cov_test = (~ab_test).mean()
    acc_test = test_c_correct_full[~ab_test].mean() if (~ab_test).sum() > 0 else np.nan
    unsafe_all = 1.0 - (test_c_correct_full & (~ab_test)).sum() / len(test_c_correct_full)
    ablation_rows.append(dict(signal=sig_name, tau_c=tc_sel, tau_u=tu_sel,
                               test_coverage=round(float(cov_test),4),
                               test_covered_acc=round(float(acc_test),4),
                               test_unsafe_all=round(float(unsafe_all),4)))

ablation_df = pd.DataFrame(ablation_rows)
print("CRUS component-wise ablation (bug-fixed -- Full CRUS row matches primary result exactly):\n")
print(ablation_df.to_string(index=False))
ablation_df.to_csv(f"{OUTDIR}/crus_component_ablation_v2.csv", index=False)

assert abs(ablation_df.iloc[0]["test_covered_acc"] - RESULTS["RiskAware-Cascade (covered subset only)"]["acc"]) < 1e-9, \
    "BUG: Full CRUS ablation row does not match primary RiskAware-Cascade result!"
print("\nConsistency check PASSED: Full CRUS ablation row exactly matches the primary result.")


In [ ]:
# -- 25. STATISTICAL TEST FOR "CRUS HIGHER IN WRONG PREDICTIONS" (RiskAware-Cascade) --------
crus_correct_test = test_c_crus[test_c_correct_full]
crus_wrong_test = test_c_crus[~test_c_correct_full]

print(f"CRUS (test, Cascade): correct n={len(crus_correct_test)}, mean={crus_correct_test.mean():.4f}")
print(f"CRUS (test, Cascade): wrong   n={len(crus_wrong_test)}, mean={crus_wrong_test.mean():.4f}")

u_stat, p_value = mannwhitneyu(crus_wrong_test, crus_correct_test, alternative="two-sided")
n1, n2 = len(crus_wrong_test), len(crus_correct_test)
effect_size_rb = (2 * u_stat) / (n1 * n2) - 1

def bootstrap_mean_diff_ci(a, b, n=CFG["bootstrap_n"], seed=SEED):
    rng = np.random.default_rng(seed)
    diffs = np.empty(n)
    for i in range(n):
        as_ = a[rng.integers(0, len(a), len(a))]
        bs_ = b[rng.integers(0, len(b), len(b))]
        diffs[i] = as_.mean() - bs_.mean()
    return round(float(np.percentile(diffs, 2.5)), 4), round(float(np.percentile(diffs, 97.5)), 4)

mean_diff = crus_wrong_test.mean() - crus_correct_test.mean()
ci_diff = bootstrap_mean_diff_ci(crus_wrong_test, crus_correct_test)
is_significant = p_value < 0.05

print(f"\nMann-Whitney U = {u_stat:.1f}, p = {p_value:.6g}")
print(f"Rank-biserial effect size r = {effect_size_rb:.4f}")
print(f"Mean difference (wrong - correct) = {mean_diff:.4f}, 95% CI = {ci_diff}")
print(f"Statistically {'SIGNIFICANT' if is_significant else 'NOT significant'} at alpha=0.05.")

crus_stat_test_results = dict(
    method="RiskAware-Cascade",
    n_correct=int(n2), n_wrong=int(n1),
    mean_correct=round(float(crus_correct_test.mean()),4),
    mean_wrong=round(float(crus_wrong_test.mean()),4),
    mannwhitney_U=float(u_stat), p_value=float(p_value),
    rank_biserial_r=round(float(effect_size_rb),4),
    mean_diff=round(float(mean_diff),4), ci95_mean_diff=list(ci_diff),
    significant_at_0_05=bool(is_significant)
)
with open(f"{OUTDIR}/crus_significance_test_v2.json", "w") as f:
    json.dump(crus_stat_test_results, f, indent=2)


In [ ]:
# -- 26. DIAGNOSTIC: PER-CLASS ERROR DISTRIBUTION (confirms GENHX bug is fixed) -------------
def genhx_attractor_rate(preds, true_labels, attractor_class="GENHX"):
    wrong_mask = preds != true_labels
    if wrong_mask.sum() == 0:
        return 0.0, 0
    to_attractor = (preds[wrong_mask] == attractor_class).sum()
    return to_attractor / wrong_mask.sum(), int(wrong_mask.sum())

rate_old_style, n_wrong_old = genhx_attractor_rate(test_l_preds, y_test_np)
rate_new, n_wrong_new = genhx_attractor_rate(test_c_preds, y_test_np)
rate_sk4, n_wrong_sk4 = genhx_attractor_rate(sk4_preds, y_test_np)

print("GENHX-attractor diagnostic (share of WRONG predictions that collapse to GENHX):")
print(f"  SK4 Hybrid (label-string CE, baseline)     : {rate_sk4:.1%} of {n_wrong_sk4} errors")
print(f"  RiskAware-LengthNorm (length-norm only)     : {rate_old_style:.1%} of {n_wrong_old} errors")
print(f"  RiskAware-Cascade (full fix, proposed)      : {rate_new:.1%} of {n_wrong_new} errors")


In [ ]:
# -- 27. CONFUSION MATRIX (TEST, RiskAware-Cascade covered subset) -----------
cm = confusion_matrix(yt_cov_c, yp_cov_c, labels=CLASSES)
fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(cm, xticklabels=CLASSES, yticklabels=CLASSES, annot=True, fmt="d", cmap="Blues", ax=ax)
ax.set_title(f"RiskAware-Cascade (proposed) - TEST Covered Cases (n={int(test_c_covered_mask.sum())}, "
             f"Acc={RESULTS['RiskAware-Cascade (covered subset only)']['acc']:.4f})", fontsize=12)
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
plt.tight_layout()
plt.savefig(f"{OUTDIR}/confusion_matrix_cascade_v2.png", dpi=300); plt.close()
print(f"Confusion matrix -> {OUTDIR}/confusion_matrix_cascade_v2.png")


In [ ]:
# -- 28. CRUS DISTRIBUTION (RiskAware-Cascade, covered vs abstained) ---------
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(test_c_crus[test_c_covered_mask], bins=30, alpha=0.7, color="#2196F3", label="Covered")
ax.hist(test_c_crus[~test_c_covered_mask], bins=30, alpha=0.7, color="#F44336", label="Abstained")
ax.axvline(CFG["tau_u"], color="black", linestyle="--", label=f"tau_u={CFG['tau_u']} (frozen from val)")
ax.set_xlabel("CRUS Score"); ax.set_ylabel("Count")
ax.set_title("CRUS Distribution (RiskAware-Cascade) - Covered vs Abstained, TEST")
ax.legend(); plt.tight_layout()
plt.savefig(f"{OUTDIR}/crus_distribution_cascade_v2.png", dpi=300); plt.close()
print(f"CRUS distribution -> {OUTDIR}/crus_distribution_cascade_v2.png")


In [ ]:
# -- 29. COMPREHENSIVE ABLATION BAR CHART (all baselines + both new variants) ------------
bar_labels = ["TF-IDF\n+LR","TF-IDF\n+SVM","BM25\nOnly","SBERT\nOnly","SK4\nHybrid",
              "RiskAware\nLengthNorm\n(all)","RiskAware\nCascade\n(all, proposed)"]
bar_values = [RESULTS["TF-IDF + LR"]["acc"], RESULTS["TF-IDF + LinearSVM"]["acc"],
              RESULTS["BM25 Only"]["acc"], RESULTS["SBERT Only"]["acc"],
              RESULTS["SK4 Hybrid (no-abstain)"]["acc"],
              RESULTS["RiskAware-LengthNorm (all, abstain=wrong)"]["acc"],
              RESULTS["RiskAware-Cascade (all, abstain=wrong)"]["acc"]]
colors = ["#90CAF9"]*5 + ["#FFA726"] + ["#4CAF50"]
fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(bar_labels, bar_values, color=colors, edgecolor="black", linewidth=0.6)
for bar, val in zip(bars, bar_values):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.003,
            f"{val:.3f}", ha="center", va="bottom", fontsize=8.5)
ax.set_ylim(0, 1.05); ax.set_ylabel("Accuracy (Test Set, all cases, abstain=wrong)")
ax.set_title("v2: Baselines vs. RiskAware Variants (green = bug-fixed proposed method)")
plt.tight_layout()
plt.savefig(f"{OUTDIR}/ablation_bar_v2.png", dpi=300); plt.close()
print(f"Ablation chart -> {OUTDIR}/ablation_bar_v2.png")


In [ ]:
# -- 30. THRESHOLD SENSITIVITY SWEEP ON VALIDATION (RiskAware-Cascade) -------
sweep_rows = []
for tc in np.arange(0.10, 0.71, 0.05):
    ab_ = (val_c_confs < tc) | (val_c_crus > CFG["tau_u"])
    cov_ = (~ab_).mean()
    if (~ab_).sum() > 5:
        acc_ = val_c_correct_full[~ab_].mean(); unsafe_ = 1.0 - acc_
    else:
        acc_ = np.nan; unsafe_ = np.nan
    sweep_rows.append(dict(tau_c=round(tc,2), coverage=round(cov_,4),
                            covered_acc=round(acc_,4) if not np.isnan(acc_) else np.nan,
                            unsafe=round(unsafe_,4) if not np.isnan(unsafe_) else np.nan))
sweep_df = pd.DataFrame(sweep_rows)
fig, ax1 = plt.subplots(figsize=(9, 4))
ax2 = ax1.twinx()
ax1.plot(sweep_df["tau_c"], sweep_df["coverage"], "b-o", label="Coverage (val)")
ax2.plot(sweep_df["tau_c"], sweep_df["covered_acc"], "g-s", label="Covered Acc (val)")
ax2.plot(sweep_df["tau_c"], sweep_df["unsafe"], "r-^", label="Unsafe Rate (val)")
ax1.axvline(CFG["tau_c"], color="black", linestyle="--", alpha=0.6, label=f"Selected tau_c={CFG['tau_c']}")
ax1.set_xlabel("Confidence Threshold (tau_c), tau_u fixed at selected value")
ax1.set_ylabel("Coverage", color="b"); ax2.set_ylabel("Accuracy / Unsafe Rate", color="g")
ax1.set_title("Threshold Sensitivity on VALIDATION (RiskAware-Cascade)")
lines1, labels1 = ax1.get_legend_handles_labels(); lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1+lines2, labels1+labels2, loc="center right", fontsize=8)
plt.tight_layout()
plt.savefig(f"{OUTDIR}/threshold_sweep_VALIDATION_v2.png", dpi=300); plt.close()
sweep_df.to_csv(f"{OUTDIR}/threshold_sweep_validation_v2.csv", index=False)
print(f"Threshold sweep -> {OUTDIR}/threshold_sweep_VALIDATION_v2.png")


In [ ]:
# -- 31. CASE AUDIT CSV (TEST, includes both new variants for direct comparison) ---------
audit = pd.DataFrame({
    "text" : X_test,
    "true_label" : y_test,
    "pred_cascade" : test_c_preds,
    "pred_lengthnorm" : test_l_preds,
    "pred_sk4" : sk4_preds,
    "conf_cascade" : np.round(test_c_confs, 4),
    "crus_cascade" : np.round(test_c_crus, 4),
    "abstained_cascade" : test_c_abstain.astype(int),
    "correct_cascade_if_covered" : ((test_c_preds == y_test_np) & (~test_c_abstain)).astype(int),
    "correct_lengthnorm_if_covered" : ((test_l_preds == y_test_np) & (~test_l_abstain)).astype(int),
    "correct_sk4" : (sk4_preds == y_test_np).astype(int),
    "doc_length_chars": [len(t) for t in X_test],
})
audit.to_csv(f"{OUTDIR}/case_audit_test_v2.csv", index=False)
print(f"Case audit -> {OUTDIR}/case_audit_test_v2.csv")


In [ ]:
# -- 32. LATEX TABLE OUTPUT (all methods, moderated wording) ------------------
latex_rows = [
    ("TF-IDF + LR", "Baseline", RESULTS["TF-IDF + LR"]),
    ("TF-IDF + LinearSVM", "Baseline", RESULTS["TF-IDF + LinearSVM"]),
    ("BM25 Only", "Retrieval Base", RESULTS["BM25 Only"]),
    ("SBERT Only", "Retrieval Base", RESULTS["SBERT Only"]),
    ("SK4 Hybrid (no-abstain)", "Ablation", RESULTS["SK4 Hybrid (no-abstain)"]),
    ("RiskAware-LengthNorm (all)", "Ablation", RESULTS["RiskAware-LengthNorm (all, abstain=wrong)"]),
    ("\\textbf{RiskAware-Cascade (all, abstain=wrong)}", "\\textbf{Proposed}",
     RESULTS["RiskAware-Cascade (all, abstain=wrong)"]),
    ("\\textbf{RiskAware-Cascade (covered subset$\\dagger$)}", "\\textbf{Proposed}",
     RESULTS["RiskAware-Cascade (covered subset only)"]),
]
latex_table = ("\\begin{table*}[ht]\n\\centering\n"
    "\\caption{Comparative performance on MTS-Dialog (train/val/test="
    f"{CFG['train_frac']:.0%}/{CFG['val_frac']:.0%}/{CFG['test_frac']:.0%}, seed=42, "
    "n\\_test=" + str(len(X_test)) + "). RiskAware-Cascade uses coarse-to-fine reranking "
    "with length-bounded full-document scoring to correct majority-class length bias. "
    "$\\dagger$~Covered subset reported for transparency only; see coverage-matched analysis "
    "for the valid comparison. $\\tau_c=" + str(CFG["tau_c"]) + "$, $\\tau_u=" + str(CFG["tau_u"]) +
    "$ selected on VALIDATION only. CI$_{95}$: bootstrap 95\\% CI on Macro-F1.}\n"
    "\\label{tab:results_v2}\n\\resizebox{\\textwidth}{!}{%\n"
    "\\begin{tabular}{llccccccc}\n\\toprule\n"
    "\\textbf{Method} & \\textbf{Type} & \\textbf{Acc} & \\textbf{Macro-F1} "
    "& \\textbf{Wt-F1} & \\textbf{$\\kappa$} & \\textbf{ECE} "
    "& \\textbf{Unsafe\\%} & \\textbf{CI$_{95}$} \\\\\n\\midrule\n")
for name, typ, v in latex_rows:
    latex_table += (f"{name} & {typ} & {v['acc']:.4f} & {v['f1m']:.4f} "
                     f"& {v['f1w']:.4f} & {v['kappa']:.4f} & {v['ece']:.4f} "
                     f"& {v['unsafe']*100:.2f} & [{v['ci'][0]:.4f},{v['ci'][1]:.4f}] \\\\\n")
latex_table += "\\bottomrule\n\\end{tabular}}\n\\end{table*}\n"
with open(f"{OUTDIR}/results_table_v2.tex","w") as f: f.write(latex_table)
print(f"LaTeX table -> {OUTDIR}/results_table_v2.tex")


In [ ]:
# -- 33. EXCEL SUMMARY --------------------------------------------------------
sum_rows = []
for name, v in RESULTS.items():
    sum_rows.append({"Method":name,"Accuracy":v["acc"],"Macro-F1":v["f1m"],
                      "Weighted-F1":v["f1w"],"Cohen_Kappa":v["kappa"],
                      "ECE":v["ece"],"Brier":v["brier"],
                      "Unsafe%":round(v["unsafe"]*100,2),
                      "Coverage":v["coverage"],"Abstained":v["abstained"],
                      "CI95_low":v["ci"][0],"CI95_high":v["ci"][1]})
pd.DataFrame(sum_rows).to_excel(f"{OUTDIR}/results_summary_v2.xlsx", index=False)
print(f"Excel summary -> {OUTDIR}/results_summary_v2.xlsx")


In [ ]:
# -- 34. CONFIG JSON (full reproducibility) -----------------------------------
CFG_export = dict(CFG)
CFG_export["excluded_classes"] = excluded_classes
CFG_export["n_train"] = len(X_train); CFG_export["n_val"] = len(X_val); CFG_export["n_test"] = len(X_test)
CFG_export["lengthnorm_tau_c"] = float(TAU_C_L); CFG_export["lengthnorm_tau_u"] = float(TAU_U_L)
CFG_export["operating_point_criterion"] = (
    "maximize covered accuracy on VALIDATION subject to coverage >= "
    f"{CFG['min_coverage_constraint']}")
with open(f"{OUTDIR}/config_v2.json","w") as f: json.dump(CFG_export, f, indent=2)

print(f"\n{'='*60}\nALL OUTPUTS SAVED TO: {OUTDIR}\n{'='*60}")
print("\nBug-fix traceability:")
print(" Root cause      -> Section 26 (GENHX-attractor diagnostic)")
print(" Cascade fix      -> Section 13 (RiskAware-Cascade, primary proposed method)")
print(" LengthNorm fix   -> Section 14 (ablation, isolates length-correction-only effect)")
print(" CRUS-ablation bug-> Section 24 (Full CRUS now reuses frozen thresholds, with assertion check)")


## Summary of v2 Changes

| Issue found in v1 output | Root cause | Fix in v2 |
|---|---|---|
| RiskAware underperforms SK4 baseline at matched coverage (93.8% vs 100% covered acc) | Cross-encoder scoring `(query, full_document_text)` is length-biased; GENHX (23.4% of corpus, narratively verbose) acts as a majority-class attractor — 71% of errors collapsed to GENHX | **RiskAware-Cascade** (Section 13): coarse label-string filter (Stage A, unbiased) shortlists candidate classes; full-document reranking (Stage B/C, the paper's genuine novelty) applied only within the shortlist, on length-bounded text |
| Unclear how much of any fix comes from length correction vs. structural change | — | **RiskAware-LengthNorm** ablation (Section 14) isolates pivoted-style length normalization alone, without the cascade, for direct comparison |
| CRUS component ablation's "Full CRUS" row used different thresholds than the main result (`config.json` mismatch found during verification) | Independent re-optimization in the ablation loop instead of reusing frozen Section-16 thresholds | Section 24 now reuses the exact frozen thresholds for "Full CRUS," with an explicit `assert` that fails loudly if this regresses |
| — | — | Section 26 adds a direct, quantitative GENHX-attractor-rate diagnostic so the fix's effectiveness is measured explicitly, not just inferred from aggregate accuracy |

**Every method in this notebook is grounded in cited, standard IR/NLP practice**: coarse-to-fine
reranking cascades (monoBERT-style staged retrieval; modular long-document reranking), passage
length truncation (standard cross-encoder input practice), and pivoted document-length
normalization (Singhal, Buckley & Mitra, 1996) — none of these are invented for this notebook.

**What I did not do:** I did not tune thresholds, cascade size, or truncation length against the
test set, and I did not hand-adjust any output number. Whether RiskAware-Cascade actually
outperforms SK4 depends on what this notebook computes when you run it — that outcome is not
predetermined, and if the cascade still underperforms, that is itself a valid, reportable finding
that would need further diagnosis rather than a claim to force through.


In [ ]:
required = [
    "y_test_np", "CLASSES",
    "test_c_preds", "test_c_probs", "test_c_abstain",
    "test_l_preds", "test_l_probs", "test_l_abstain"
]

missing = [name for name in required if name not in globals()]

print("Missing variables:", missing)

if not missing:
    print("Prediction arrays are available. No model rerun is needed for the reporting repair.")
else:
    print("Stop here. Some required arrays must be restored before running the repair.")

In [ ]:
required = [
    "y_test_np", "CLASSES",
    "test_c_preds", "test_c_probs", "test_c_abstain",
    "test_l_preds", "test_l_probs", "test_l_abstain"
]

missing = [name for name in required if name not in globals()]
print("Missing variables:", missing)

In [ ]:
# Local revision workspace; no Google Drive mount.
from pathlib import Path

REVISION_WORK_ROOT = RUN_OUTPUT_ROOT / "revision_pipeline"
REVISION_WORK_ROOT.mkdir(parents=True, exist_ok=True)
print("New revision-pipeline outputs:", REVISION_WORK_ROOT)
print("Deposited artifacts remain at:", ARCHIVE_ROOT)


In [ ]:
from pathlib import Path

BASE = Path(str(RUN_OUTPUT_ROOT))

if not BASE.exists():
    raise FileNotFoundError(
        f"Directory not found: {BASE}. Update BASE to your actual project folder."
    )

patterns = [
    "case_audit_test_v2.csv",
    "config_v2.json",
    "results_summary_v2.xlsx",
    "*.npz",
    "*.npy",
    "*.pkl",
    "*.joblib"
]

found = {}

for pattern in patterns:
    matches = sorted(BASE.rglob(pattern))
    found[pattern] = matches

    print(f"\n{pattern}: {len(matches)} file(s)")
    for path in matches:
        print(path)

In [ ]:
from pathlib import Path

ROOT = Path(str(RUN_OUTPUT_ROOT))

keywords = [
    "case_audit",
    "config_v2",
    "results_summary",
    "coverage_matched",
    "crus_component",
    "riskaware"
]

matches = []

for path in ROOT.rglob("*"):
    if path.is_file():
        filename = path.name.lower()

        if any(keyword in filename for keyword in keywords):
            matches.append(path)

print(f"Found {len(matches)} potentially relevant files:\n")

for path in sorted(matches):
    print(path)

In [ ]:
# Check only the explicitly configured raw input files.
from pathlib import Path

dataset_paths = {}
for _filename in (
    "MTS-Dialog-TrainingSet.csv",
    "MTS-Dialog-ValidationSet.csv",
):
    _path = RAW_DATA_ROOT / _filename
    dataset_paths[_filename] = [_path] if _path.is_file() else []
    print(f"{_filename}: {'FOUND' if _path.is_file() else 'MISSING'}")
    print(_path)


In [ ]:
# Verify the recorded source version; do not clone an unpinned default branch.
from pathlib import Path
import subprocess
import hashlib
import json
import os

_source_manifest_path = ARCHIVE_ROOT / "source_data" / "source_manifest.json"
if not _source_manifest_path.is_file():
    raise FileNotFoundError(_source_manifest_path)

_source_provenance = json.loads(
    _source_manifest_path.read_text(encoding="utf-8")
)
_expected_source_commit = _source_provenance["commit"]

REPO_DIR = Path(os.environ.get(
    "CRUS_MTS_REPO_ROOT",
    str(RAW_DATA_ROOT.parent)
)).expanduser().resolve()

if not (REPO_DIR / ".git").exists():
    raise RuntimeError(
        "A local checkout of the recorded MTS-Dialog repository is required "
        "for this source-provenance cell. Follow DATA_RECONSTRUCTION.md. "
        "Set CRUS_MTS_REPO_ROOT to that checkout before starting the kernel. "
        "No repository has been downloaded or substituted automatically."
    )

commit = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"],
    text=True
).strip()

if commit != _expected_source_commit:
    raise RuntimeError(
        f"Dataset commit mismatch: expected {_expected_source_commit}, "
        f"found {commit}. Do not silently substitute another version."
    )

for _filename in (
    "MTS-Dialog-TrainingSet.csv",
    "MTS-Dialog-ValidationSet.csv",
):
    _input_path = RAW_DATA_ROOT / _filename
    if not _input_path.is_file():
        raise FileNotFoundError(_input_path)
    _expected_hash = _source_provenance["files"][_filename]["sha256"]
    _actual_hash = hashlib.sha256(_input_path.read_bytes()).hexdigest()
    if _actual_hash != _expected_hash:
        raise RuntimeError(f"Raw-input checksum mismatch: {_filename}")

print("Recorded repository commit and configured raw-input checksums verified.")


In [ ]:
from pathlib import Path
import pandas as pd
import hashlib

REPO_DIR = RAW_DATA_ROOT

required_names = [
    "MTS-Dialog-TrainingSet.csv",
    "MTS-Dialog-ValidationSet.csv"
]

dataset_paths = {}

for name in required_names:
    matches = sorted(REPO_DIR.rglob(name))

    if len(matches) != 1:
        raise RuntimeError(
            f"Expected exactly one {name}; found {len(matches)}.\n"
            f"Matches: {[str(p) for p in matches]}"
        )

    path = matches[0]
    data = pd.read_csv(path)

    required_columns = {
        "section_header", "section_text", "dialogue"
    }

    missing_columns = required_columns - set(data.columns)

    if missing_columns:
        raise RuntimeError(
            f"{name} is missing columns: {sorted(missing_columns)}"
        )

    dataset_paths[name] = str(path)

    print("\nFile:", path)
    print("Rows:", len(data))
    print("Columns:", data.columns.tolist())
    print(
        "SHA256:",
        hashlib.sha256(path.read_bytes()).hexdigest()
    )
    print(
        "Missing required-field values:",
        data[list(sorted(required_columns))]
        .isna()
        .sum()
        .to_dict()
    )
    print(
        "Class counts:",
        data["section_header"]
        .value_counts(dropna=False)
        .to_dict()
    )

In [ ]:
from pathlib import Path
import pandas as pd
import hashlib
import json
import shutil
import subprocess

SOURCE_DIR = Path(
    str(REVISION_WORK_ROOT / "source_data")
)
AUDIT_DIR = Path(
    str(REVISION_WORK_ROOT / "data_audit")
)

SOURCE_DIR.mkdir(parents=True, exist_ok=True)
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

source_frames = []
manifest = {
    "repository": "https://github.com/abachaa/MTS-Dialog",
    "commit": subprocess.check_output(
        ["git", "-C", str(Path(os.environ.get("CRUS_MTS_REPO_ROOT", str(RAW_DATA_ROOT.parent))).expanduser().resolve()), "rev-parse", "HEAD"],
        text=True
    ).strip(),
    "files": {}
}

for name, source_path in dataset_paths.items():
    source_path = Path(source_path)
    destination = SOURCE_DIR / name

    source_hash = hashlib.sha256(
        source_path.read_bytes()
    ).hexdigest()

    if destination.exists():
        existing_hash = hashlib.sha256(
            destination.read_bytes()
        ).hexdigest()

        if existing_hash != source_hash:
            raise RuntimeError(
                f"Existing saved file differs: {destination}. "
                "Do not overwrite it without resolving the version difference."
            )
    else:
        shutil.copy2(source_path, destination)

    frame = pd.read_csv(destination)
    frame["source_file"] = name
    frame["source_row"] = range(len(frame))

    # Source filename + row gives a unique audit identifier,
    # independently of whether the original ID repeats.
    frame["record_id"] = [
        f"{name}:{i}" for i in range(len(frame))
    ]

    frame["label"] = (
        frame["section_header"]
        .astype(str)
        .str.strip()
        .str.upper()
    )

    source_frames.append(frame)

    manifest["files"][name] = {
        "sha256": source_hash,
        "rows": len(frame)
    }

pool = pd.concat(source_frames, ignore_index=True)

counts = (
    pool["label"]
    .value_counts()
    .rename_axis("label")
    .reset_index(name="count")
)

counts["retained_by_min5"] = counts["count"] >= 5

retained = counts[counts["retained_by_min5"]].copy()
retained["percentage"] = (
    100 * retained["count"] / retained["count"].sum()
)

counts.to_csv(
    AUDIT_DIR / "source_pool_class_counts.csv",
    index=False
)

retained.to_csv(
    AUDIT_DIR / "source_pool_min5_class_counts.csv",
    index=False
)

(SOURCE_DIR / "source_manifest.json").write_text(
    json.dumps(manifest, indent=2)
)

assert counts["count"].sum() == len(pool)
assert pool["record_id"].is_unique

print("Combined source records:", len(pool))
print("Classes before filtering:", len(counts))

print("\nClasses excluded by minimum-five rule:")
print(
    counts.loc[
        ~counts["retained_by_min5"], ["label", "count"]
    ].to_string(index=False)
)

print("\nRetained class counts BEFORE duplicate handling:")
print(
    retained[["label", "count", "percentage"]]
    .to_string(
        index=False,
        formatters={"percentage": "{:.2f}".format}
    )
)

print("\nRetained records:", retained["count"].sum())
print("Retained classes:", len(retained))
print("\nSource version and count tables saved in Drive.")

In [ ]:
import re
import hashlib
import pandas as pd

required_columns = {
    "record_id", "source_file", "ID", "label", "dialogue"
}

missing = required_columns - set(pool.columns)

if missing:
    raise RuntimeError(
        f"Missing columns in source pool: {sorted(missing)}"
    )

def normalize_dialogue(text):
    return re.sub(
        r"\s+", " ", str(text).strip().lower()
    )

def dialogue_hash(text):
    return hashlib.sha256(
        text.encode("utf-8")
    ).hexdigest()

# Work on a copy. Do not filter or deduplicate yet.
duplicate_audit = pool.copy()

duplicate_audit["normalized_dialogue"] = (
    duplicate_audit["dialogue"].map(normalize_dialogue)
)

duplicate_audit["dialogue_hash"] = (
    duplicate_audit["normalized_dialogue"].map(dialogue_hash)
)

if duplicate_audit["normalized_dialogue"].eq("").any():
    raise RuntimeError(
        "Empty dialogue found after normalization. "
        "Resolve before splitting."
    )

# 1. Exact normalized-dialogue matches.
exact_matches = duplicate_audit[
    duplicate_audit.duplicated(
        "dialogue_hash", keep=False
    )
].copy()

exact_group_summary = (
    exact_matches
    .groupby("dialogue_hash")
    .agg(
        n_records=("record_id", "size"),
        n_labels=("label", "nunique"),
        n_source_files=("source_file", "nunique"),
        labels=(
            "label",
            lambda s: " | ".join(sorted(set(s)))
        )
    )
    .reset_index()
)

# 2. Repeated IDs across the combined source pool.
id_key = duplicate_audit["ID"].astype("string").str.strip()
valid_ids = id_key.notna() & id_key.ne("")

duplicate_audit["original_id_text"] = id_key

repeated_ids = duplicate_audit[
    valid_ids
    & duplicate_audit.duplicated(
        "original_id_text", keep=False
    )
].copy()

# 3. Repeated IDs within the same source file.
within_source_ids = duplicate_audit[
    valid_ids
    & duplicate_audit.duplicated(
        ["source_file", "original_id_text"],
        keep=False
    )
].copy()

export_columns = [
    "record_id", "source_file", "ID", "label",
    "dialogue_hash", "dialogue"
]

exact_matches[export_columns].to_csv(
    AUDIT_DIR / "exact_dialogue_match_records.csv",
    index=False
)

exact_group_summary.to_csv(
    AUDIT_DIR / "exact_dialogue_match_groups.csv",
    index=False
)

repeated_ids[export_columns].to_csv(
    AUDIT_DIR / "repeated_source_ids.csv",
    index=False
)

within_source_ids[export_columns].to_csv(
    AUDIT_DIR / "repeated_ids_within_source.csv",
    index=False
)

print("Total source records checked:", len(duplicate_audit))

print(
    "Records participating in exact dialogue matches:",
    len(exact_matches)
)

print(
    "Exact dialogue groups with multiple records:",
    len(exact_group_summary)
)

print(
    "Exact dialogue groups containing different labels:",
    int((exact_group_summary["n_labels"] > 1).sum())
)

print(
    "Exact dialogue groups spanning both source files:",
    int((exact_group_summary["n_source_files"] > 1).sum())
)

print(
    "Records with repeated IDs in combined pool:",
    len(repeated_ids)
)

print(
    "Records with repeated IDs within a source file:",
    len(within_source_ids)
)

if not exact_group_summary.empty:
    print("\nExact-dialogue group summary:")
    print(exact_group_summary.to_string(index=False))

if not repeated_ids.empty:
    print("\nRepeated-ID examples:")
    print(
        repeated_ids[
            ["record_id", "source_file", "ID", "label"]
        ].head(20).to_string(index=False)
    )

print("\nAudit files saved. No records removed.")

In [ ]:
from IPython.display import display

for dialogue_hash_value, group in exact_matches.groupby(
    "dialogue_hash", sort=True
):
    print("\n" + "=" * 90)
    print("Dialogue hash:", dialogue_hash_value)
    print("Labels:", sorted(group["label"].unique()))

    display(
        group[
            ["record_id", "source_file", "ID", "label"]
        ]
    )

    print("\nShared normalized dialogue:")
    print(group["normalized_dialogue"].iloc[0])

    print("\nReference section texts:")
    for _, row in group.iterrows():
        print(f"\nRecord: {row['record_id']}")
        print(f"Label: {row['label']}")
        print(row["section_text"])

In [ ]:
from difflib import SequenceMatcher
import pandas as pd
import json

SIMILARITY_THRESHOLD = 0.95
MIN_LENGTH_RATIO = 0.90

unique_dialogues = (
    duplicate_audit[
        ["dialogue_hash", "normalized_dialogue"]
    ]
    .drop_duplicates("dialogue_hash")
    .reset_index(drop=True)
)

group_metadata = (
    duplicate_audit
    .groupby("dialogue_hash")
    .agg(
        record_ids=(
            "record_id",
            lambda s: " | ".join(sorted(s))
        ),
        labels=(
            "label",
            lambda s: " | ".join(sorted(set(s)))
        )
    )
)

candidates = []

for i in range(len(unique_dialogues)):
    text_a = unique_dialogues.at[i, "normalized_dialogue"]
    hash_a = unique_dialogues.at[i, "dialogue_hash"]

    for j in range(i + 1, len(unique_dialogues)):
        text_b = unique_dialogues.at[j, "normalized_dialogue"]
        hash_b = unique_dialogues.at[j, "dialogue_hash"]

        length_ratio = (
            min(len(text_a), len(text_b))
            / max(len(text_a), len(text_b))
        )

        if length_ratio < MIN_LENGTH_RATIO:
            continue

        matcher = SequenceMatcher(
            None, text_a, text_b, autojunk=False
        )

        if matcher.quick_ratio() < SIMILARITY_THRESHOLD:
            continue

        similarity = matcher.ratio()

        if similarity >= SIMILARITY_THRESHOLD:
            candidates.append({
                "hash_a": hash_a,
                "hash_b": hash_b,
                "record_ids_a": group_metadata.at[
                    hash_a, "record_ids"
                ],
                "record_ids_b": group_metadata.at[
                    hash_b, "record_ids"
                ],
                "labels_a": group_metadata.at[
                    hash_a, "labels"
                ],
                "labels_b": group_metadata.at[
                    hash_b, "labels"
                ],
                "similarity": similarity,
                "length_ratio": length_ratio,
                "review_decision": "",
                "review_reason": ""
            })

candidate_columns = [
    "hash_a", "hash_b",
    "record_ids_a", "record_ids_b",
    "labels_a", "labels_b",
    "similarity", "length_ratio",
    "review_decision", "review_reason"
]

near_duplicate_candidates = pd.DataFrame(
    candidates,
    columns=candidate_columns
)

if not near_duplicate_candidates.empty:
    near_duplicate_candidates = (
        near_duplicate_candidates
        .sort_values("similarity", ascending=False)
        .reset_index(drop=True)
    )

near_duplicate_candidates.to_csv(
    AUDIT_DIR / "near_duplicate_candidates_for_review.csv",
    index=False
)

(AUDIT_DIR / "near_duplicate_screen_settings.json").write_text(
    json.dumps({
        "normalization": "lowercase and collapse whitespace",
        "similarity_measure": "SequenceMatcher ratio; autojunk=False",
        "similarity_threshold": SIMILARITY_THRESHOLD,
        "minimum_length_ratio": MIN_LENGTH_RATIO,
        "limitation": (
            "Candidate screening only; does not establish "
            "absence of related records."
        )
    }, indent=2)
)

print("Unique normalized dialogues:", len(unique_dialogues))
print("Near-duplicate candidate pairs:",
      len(near_duplicate_candidates))

if not near_duplicate_candidates.empty:
    display(
        near_duplicate_candidates[
            [
                "record_ids_a", "record_ids_b",
                "labels_a", "labels_b", "similarity"
            ]
        ].head(30)
    )

print("No records removed and no splits created.")

In [ ]:
from IPython.display import display

records = duplicate_audit.set_index(
    "record_id",
    verify_integrity=True
)

pairs_to_inspect = [
    (
        "MTS-Dialog-TrainingSet.csv:7",
        "MTS-Dialog-TrainingSet.csv:104"
    ),
    (
        "MTS-Dialog-TrainingSet.csv:120",
        "MTS-Dialog-TrainingSet.csv:666"
    )
]

for record_a, record_b in pairs_to_inspect:
    print("\n" + "=" * 90)

    for record_id in [record_a, record_b]:
        row = records.loc[record_id]

        print("\nRecord:", record_id)
        print("Label:", row["label"])
        print("\nDialogue:")
        print(row["dialogue"])

        print("\nReference section text — audit only:")
        print(row["section_text"])

In [ ]:
import json

cleaning_policy = {
    "prediction_input": "dialogue only",
    "normalization": "lowercase and collapse whitespace",
    "same_dialogue_same_label": {
        "action": "retain one representative",
        "representative_rule": (
            "first record in documented source-file order, "
            "then source-row order"
        ),
        "log_removed_records": True
    },
    "same_dialogue_different_labels": {
        "primary_action": "retain all distinct labels",
        "split_rule": "all records in the same dialogue group",
        "limitation": (
            "identical dialogue-only inputs can have "
            "different recorded targets"
        )
    },
    "near_duplicate_candidates": {
        "action": "manual review before grouping",
        "automatic_deletion": False
    },
    "cross_file_numeric_ids": {
        "action": "not treated as globally unique identifiers"
    },
    "rare_class_filter": (
        "apply minimum-five rule to the final deduplicated pool "
        "before splitting; regenerate counts"
    )
}

(AUDIT_DIR / "proposed_cleaning_policy.json").write_text(
    json.dumps(cleaning_policy, indent=2)
)

print("Proposed policy saved. No records removed or splits created.")

In [ ]:
import json
import pandas as pd

required = {
    "record_id", "source_file", "source_row",
    "label", "dialogue", "normalized_dialogue",
    "dialogue_hash"
}

missing = required - set(duplicate_audit.columns)

if missing:
    raise RuntimeError(
        f"Missing audit columns: {sorted(missing)}"
    )

# Preserve the documented source order.
source_order = {
    "MTS-Dialog-TrainingSet.csv": 0,
    "MTS-Dialog-ValidationSet.csv": 1
}

work = duplicate_audit.copy()

work["source_order"] = work["source_file"].map(source_order)

if work["source_order"].isna().any():
    raise RuntimeError(
        "Unexpected source file. Define its ordering explicitly."
    )

work = work.sort_values(
    ["source_order", "source_row"],
    kind="mergesort"
).reset_index(drop=True)

# 1. Remove redundant exact dialogue/label records.
# Different labels for the same dialogue are retained.
work["representative_record_id"] = (
    work.groupby(
        ["dialogue_hash", "label"],
        sort=False
    )["record_id"].transform("first")
)

redundant = work["record_id"].ne(
    work["representative_record_id"]
)

exact_removal_log = work.loc[
    redundant,
    [
        "record_id", "representative_record_id",
        "source_file", "label", "dialogue_hash"
    ]
].copy()

exact_removal_log["reason"] = (
    "Redundant normalized dialogue/label pair"
)

cleaned = work.loc[~redundant].copy()

# 2. Begin with exact-dialogue groups.
hashes = sorted(work["dialogue_hash"].unique())
parent = {h: h for h in hashes}

def find_group(h):
    while parent[h] != h:
        parent[h] = parent[parent[h]]
        h = parent[h]
    return h

def merge_groups(hash_a, hash_b):
    root_a = find_group(hash_a)
    root_b = find_group(hash_b)

    if root_a != root_b:
        # Deterministic representative for the connected group.
        first, second = sorted([root_a, root_b])
        parent[second] = first

record_to_hash = work.set_index(
    "record_id",
    verify_integrity=True
)["dialogue_hash"].to_dict()

# 3. Reviewed near-duplicate pairs:
# retain records, but prevent separation across splits.
reviewed_pairs = [
    {
        "record_id_a": "MTS-Dialog-TrainingSet.csv:7",
        "record_id_b": "MTS-Dialog-TrainingSet.csv:104",
        "decision": "retain_both_group_together",
        "reason": (
            "Near-identical ALLERGY dialogue; "
            "know/known wording difference"
        )
    },
    {
        "record_id_a": "MTS-Dialog-TrainingSet.csv:120",
        "record_id_b": "MTS-Dialog-TrainingSet.csv:666",
        "decision": "retain_both_group_together",
        "reason": (
            "Near-identical FAM/SOCHX dialogue; "
            "None/Nothing wording difference"
        )
    }
]

for pair in reviewed_pairs:
    a = pair["record_id_a"]
    b = pair["record_id_b"]

    if a not in record_to_hash or b not in record_to_hash:
        raise RuntimeError(
            f"Reviewed pair not found in source pool: {a}, {b}"
        )

    merge_groups(
        record_to_hash[a],
        record_to_hash[b]
    )

cleaned["group_id"] = cleaned["dialogue_hash"].map(
    lambda h: "dialogue_group_" + find_group(h)
)

# Dialogue only is the model input.
cleaned["text"] = cleaned["normalized_dialogue"]

# 4. Apply the minimum-five rule after exact deduplication.
MIN_CLASS_COUNT = 5

counts_before_filter = (
    cleaned["label"]
    .value_counts()
    .rename_axis("label")
    .reset_index(name="count")
)

excluded_labels = sorted(
    counts_before_filter.loc[
        counts_before_filter["count"] < MIN_CLASS_COUNT,
        "label"
    ]
)

rare_exclusion_log = cleaned.loc[
    cleaned["label"].isin(excluded_labels),
    ["record_id", "source_file", "label", "group_id"]
].copy()

rare_exclusion_log["reason"] = (
    "Class below minimum-five count after exact deduplication"
)

final_pool = cleaned.loc[
    ~cleaned["label"].isin(excluded_labels)
].copy().reset_index(drop=True)

final_counts = (
    final_pool["label"]
    .value_counts()
    .rename_axis("label")
    .reset_index(name="count")
)

final_counts["percentage"] = (
    100 * final_counts["count"] / len(final_pool)
)

# 5. Identify retained exact-input label conflicts.
label_counts_by_dialogue = (
    final_pool.groupby("dialogue_hash")["label"].nunique()
)

conflicting_hashes = label_counts_by_dialogue[
    label_counts_by_dialogue > 1
].index

ambiguity_log = final_pool.loc[
    final_pool["dialogue_hash"].isin(conflicting_hashes),
    [
        "record_id", "label",
        "dialogue_hash", "group_id"
    ]
].copy()

# 6. Consistency checks.
assert final_pool["record_id"].is_unique

assert not final_pool.duplicated(
    ["dialogue_hash", "label"]
).any()

assert (
    final_pool.groupby("dialogue_hash")["group_id"]
    .nunique()
    .max()
    == 1
)

assert final_counts["count"].sum() == len(final_pool)
assert (final_counts["count"] >= MIN_CLASS_COUNT).all()

for pair in reviewed_pairs:
    a = pair["record_id_a"]
    b = pair["record_id_b"]

    retained_pair = final_pool[
        final_pool["record_id"].isin([a, b])
    ]

    assert len(retained_pair) == 2
    assert retained_pair["group_id"].nunique() == 1

# 7. Save data and audit records.
exact_removal_log.to_csv(
    AUDIT_DIR / "exact_duplicate_removal_log.csv",
    index=False
)

pd.DataFrame(reviewed_pairs).to_csv(
    AUDIT_DIR / "near_duplicate_review_decisions.csv",
    index=False
)

rare_exclusion_log.to_csv(
    AUDIT_DIR / "rare_class_exclusion_log.csv",
    index=False
)

ambiguity_log.to_csv(
    AUDIT_DIR / "retained_conflicting_label_groups.csv",
    index=False
)

final_counts.to_csv(
    AUDIT_DIR / "final_cleaned_class_counts.csv",
    index=False
)

final_pool[
    [
        "record_id", "source_file", "source_row",
        "label", "dialogue_hash", "group_id", "text"
    ]
].to_csv(
    AUDIT_DIR / "cleaned_dialogue_pool.csv",
    index=False
)

policy = {
    "input": "normalized dialogue only",
    "exact_duplicate_policy": (
        "One representative per normalized dialogue/label; "
        "training-source order then source row"
    ),
    "conflicting_label_policy": (
        "Retain distinct labels for identical dialogue; "
        "keep together in one split group"
    ),
    "near_duplicate_policy": (
        "Retain reviewed near-identical records; group together"
    ),
    "minimum_class_count": MIN_CLASS_COUNT,
    "filter_order": (
        "exact dialogue/label deduplication, "
        "then rare-class filtering, then splitting"
    ),
    "limitation": (
        "Similarity screening and inspected pairs do not prove "
        "absence of all related encounters"
    )
}

(AUDIT_DIR / "final_cleaning_grouping_policy.json").write_text(
    json.dumps(policy, indent=2)
)

print("Original source records:", len(work))
print("Exact dialogue/label records removed:",
      len(exact_removal_log))
print("Records after exact deduplication:", len(cleaned))
print("Excluded labels:", excluded_labels)
print("Rare-class records excluded:",
      len(rare_exclusion_log))
print("Final retained records:", len(final_pool))
print("Final retained classes:", len(final_counts))
print("Final groups:", final_pool["group_id"].nunique())
print("Conflicting exact-dialogue groups retained:",
      len(conflicting_hashes))

print("\nFinal class distribution:")
print(
    final_counts.to_string(
        index=False,
        formatters={"percentage": "{:.2f}".format}
    )
)

print("\nRetained conflicting-label records:")
print(ambiguity_log.to_string(index=False))

print("\nNo splits created and no models run.")

In [ ]:
import json
import hashlib
import numpy as np
import pandas as pd
import sklearn

from sklearn.model_selection import StratifiedGroupKFold

SEED = 42

required = {
    "record_id", "source_file", "source_row",
    "label", "dialogue_hash", "group_id", "text"
}

missing = required - set(final_pool.columns)

if missing:
    raise RuntimeError(
        f"Missing cleaned-pool columns: {sorted(missing)}"
    )

split_pool = final_pool.copy().reset_index(drop=True)

assert split_pool["record_id"].is_unique
assert split_pool["text"].str.strip().ne("").all()

# Approximately 20% test, keeping complete groups together.
outer_splitter = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=SEED
)

trainval_idx, test_idx = next(
    outer_splitter.split(
        X=split_pool["text"],
        y=split_pool["label"],
        groups=split_pool["group_id"]
    )
)

# Approximately 25% of the remaining 80% for validation:
# approximately 20% of the complete dataset.
trainval_pool = split_pool.iloc[trainval_idx]

inner_splitter = StratifiedGroupKFold(
    n_splits=4,
    shuffle=True,
    random_state=SEED
)

train_relative_idx, val_relative_idx = next(
    inner_splitter.split(
        X=trainval_pool["text"],
        y=trainval_pool["label"],
        groups=trainval_pool["group_id"]
    )
)

train_idx = trainval_idx[train_relative_idx]
val_idx = trainval_idx[val_relative_idx]

split_pool["split"] = ""

split_pool.loc[train_idx, "split"] = "train"
split_pool.loc[val_idx, "split"] = "validation"
split_pool.loc[test_idx, "split"] = "test"

# 1. Verify complete, non-overlapping record assignment.
assert split_pool["split"].isin(
    ["train", "validation", "test"]
).all()

assert len(train_idx) + len(val_idx) + len(test_idx) == len(
    split_pool
)

index_sets = {
    "train": set(train_idx),
    "validation": set(val_idx),
    "test": set(test_idx)
}

for a, b in [
    ("train", "validation"),
    ("train", "test"),
    ("validation", "test")
]:
    assert not index_sets[a].intersection(index_sets[b])

# 2. Verify group and exact-dialogue separation.
group_crossings = (
    split_pool.groupby("group_id")["split"].nunique()
)

dialogue_crossings = (
    split_pool.groupby("dialogue_hash")["split"].nunique()
)

assert group_crossings.max() == 1
assert dialogue_crossings.max() == 1

# 3. Generate class distribution, including absent classes.
CLASSES = sorted(split_pool["label"].unique())

class_distribution = (
    pd.crosstab(
        split_pool["label"],
        split_pool["split"]
    )
    .reindex(
        index=CLASSES,
        columns=["train", "validation", "test"],
        fill_value=0
    )
    .fillna(0)
    .astype(int)
)

class_distribution["total"] = (
    class_distribution[
        ["train", "validation", "test"]
    ].sum(axis=1)
)

assert class_distribution["total"].sum() == len(split_pool)

split_sizes = (
    split_pool["split"]
    .value_counts()
    .reindex(["train", "validation", "test"])
)

split_summary = pd.DataFrame({
    "records": split_sizes,
    "percentage": 100 * split_sizes / len(split_pool),
    "groups": (
        split_pool.groupby("split")["group_id"]
        .nunique()
        .reindex(["train", "validation", "test"])
    )
})

# 4. Identify support problems before fitting models.
absent_classes = {
    partition: class_distribution.index[
        class_distribution[partition] == 0
    ].tolist()
    for partition in ["train", "validation", "test"]
}

below_three_in_training = (
    class_distribution.loc[
        class_distribution["train"] < 3,
        ["train", "validation", "test"]
    ]
)

# 5. Preserve split identifiers and metadata.
manifest_columns = [
    "record_id", "source_file", "source_row",
    "label", "dialogue_hash", "group_id", "split"
]

manifest_bytes = (
    split_pool[manifest_columns]
    .to_csv(index=False)
    .encode("utf-8")
)

manifest_path = AUDIT_DIR / "corrected_split_manifest.csv"

# Do not silently overwrite a different saved split.
if manifest_path.exists():
    if manifest_path.read_bytes() != manifest_bytes:
        raise RuntimeError(
            "A different split manifest already exists. "
            "Preserve it and resolve the version difference "
            "before replacing anything."
        )
else:
    manifest_path.write_bytes(manifest_bytes)

class_distribution.to_csv(
    AUDIT_DIR / "corrected_class_distribution_per_split.csv"
)

split_summary.to_csv(
    AUDIT_DIR / "corrected_split_summary.csv"
)

split_pool.to_csv(
    AUDIT_DIR / "corrected_dialogue_pool_with_splits.csv",
    index=False
)

split_metadata = {
    "seed": SEED,
    "sklearn_version": sklearn.__version__,
    "method": (
        "First fold of five-fold StratifiedGroupKFold "
        "for test; first fold of four-fold "
        "StratifiedGroupKFold on remaining records "
        "for validation"
    ),
    "target_fractions": {
        "train": 0.60,
        "validation": 0.20,
        "test": 0.20
    },
    "actual_sizes": {
        k: int(v) for k, v in split_sizes.items()
    },
    "manifest_sha256": hashlib.sha256(
        manifest_bytes
    ).hexdigest(),
    "absent_classes": absent_classes,
    "historical_exposure_note": (
        "This corrected partition is not automatically an "
        "independent new holdout: results from an earlier "
        "partition of this source pool were previously inspected."
    )
}

(AUDIT_DIR / "corrected_split_metadata.json").write_text(
    json.dumps(split_metadata, indent=2)
)

print("Split summary:")
print(split_summary.to_string())

print("\nClass distribution:")
print(class_distribution.to_string())

print("\nClasses absent from each split:")
print(absent_classes)

print("\nClasses with fewer than three training records:")
if below_three_in_training.empty:
    print("None.")
else:
    print(below_three_in_training.to_string())

print("\nGroups crossing partitions:",
      int((group_crossings > 1).sum()))
print("Exact dialogues crossing partitions:",
      int((dialogue_crossings > 1).sum()))

print("\nSplit manifest SHA256:",
      split_metadata["manifest_sha256"])

print("\nSplit saved. No models trained or scored.")

In [ ]:
import json
import hashlib
import numpy as np
import pandas as pd
import scipy

from scipy.optimize import milp, Bounds, LinearConstraint
from scipy.sparse import lil_matrix

# Start from the reviewed cleaned pool, not the preliminary split.
data = final_pool.copy().reset_index(drop=True)

partitions = ["train", "validation", "test"]
fractions = np.array([0.60, 0.20, 0.20])
classes = sorted(data["label"].unique())
groups = sorted(data["group_id"].unique())

SEED = 42

# Each row is a complete group; columns give label counts.
group_class_counts = (
    pd.crosstab(data["group_id"], data["label"])
    .reindex(index=groups, columns=classes, fill_value=0)
    .fillna(0)
    .astype(int)
)

group_sizes = group_class_counts.sum(axis=1).to_numpy()
class_totals = group_class_counts.sum(axis=0).to_numpy()

G = len(groups)
C = len(classes)
P = len(partitions)
N = len(data)

# Largest-remainder allocation of the overall sample total.
raw_targets = fractions * N
target_sizes = np.floor(raw_targets).astype(int)

remaining = N - target_sizes.sum()

# Fractional-part ties are resolved in partition order.
remainder_order = np.argsort(
    -(raw_targets - target_sizes),
    kind="stable"
)

for p in remainder_order[:remaining]:
    target_sizes[p] += 1

# Per-class counts must be within floor/ceil of their targets,
# with minimum support enforced.
class_lower = np.floor(
    class_totals[:, None] * fractions
).astype(int)

class_upper = np.ceil(
    class_totals[:, None] * fractions
).astype(int)

class_lower[:, 0] = np.maximum(class_lower[:, 0], 3)
class_lower[:, 1] = np.maximum(class_lower[:, 1], 1)
class_lower[:, 2] = np.maximum(class_lower[:, 2], 1)

if np.any(class_lower > class_upper):
    raise RuntimeError(
        "Class support requirements conflict with the requested "
        "proportions. Review the protocol; do not silently relax it."
    )

# Binary variable x[g,p]: group g is assigned to partition p.
n_variables = G * P
n_constraints = G + P + C * P

A = lil_matrix(
    (n_constraints, n_variables),
    dtype=float
)

lower = np.zeros(n_constraints)
upper = np.zeros(n_constraints)
row = 0

# Every group belongs to exactly one partition.
for g in range(G):
    for p in range(P):
        A[row, g * P + p] = 1

    lower[row] = 1
    upper[row] = 1
    row += 1

# Overall partition sizes.
for p in range(P):
    for g in range(G):
        A[row, g * P + p] = group_sizes[g]

    lower[row] = target_sizes[p]
    upper[row] = target_sizes[p]
    row += 1

# Class counts in every partition.
counts_array = group_class_counts.to_numpy()

for c in range(C):
    for p in range(P):
        for g in range(G):
            count = counts_array[g, c]
            if count:
                A[row, g * P + p] = count

        lower[row] = class_lower[c, p]
        upper[row] = class_upper[c, p]
        row += 1

# No performance-based objective.
# A fixed seeded objective selects among feasible assignments.
rng = np.random.default_rng(SEED)
objective = rng.random(n_variables)

result = milp(
    c=objective,
    integrality=np.ones(n_variables, dtype=int),
    bounds=Bounds(
        np.zeros(n_variables),
        np.ones(n_variables)
    ),
    constraints=LinearConstraint(
        A.tocsr(), lower, upper
    ),
    options={"time_limit": 120}
)

print("Solver status:", result.status)
print("Solver message:", result.message)

if result.x is None:
    raise RuntimeError(
        "No assignment returned. Do not train or silently change "
        "the seed/constraints. Review the solver output."
    )

assignment = result.x.reshape(G, P)

if not np.allclose(
    assignment, np.round(assignment), atol=1e-6
):
    raise RuntimeError("Returned assignment is not integer-valued.")

assignment = np.round(assignment).astype(int)

assert (assignment.sum(axis=1) == 1).all()

group_to_split = {
    groups[g]: partitions[int(assignment[g].argmax())]
    for g in range(G)
}

data["split"] = data["group_id"].map(group_to_split)

distribution = (
    pd.crosstab(data["label"], data["split"])
    .reindex(
        index=classes,
        columns=partitions,
        fill_value=0
    )
    .fillna(0)
    .astype(int)
)

distribution["total"] = distribution[partitions].sum(axis=1)

# Verify every constraint using the actual record assignment.
actual_sizes = (
    data["split"].value_counts()
    .reindex(partitions)
    .to_numpy()
)

actual_class_counts = distribution[partitions].to_numpy()

assert np.array_equal(actual_sizes, target_sizes)
assert np.all(actual_class_counts >= class_lower)
assert np.all(actual_class_counts <= class_upper)

assert data.groupby("group_id")["split"].nunique().max() == 1
assert data.groupby("dialogue_hash")["split"].nunique().max() == 1

assert (distribution["train"] >= 3).all()
assert (distribution["validation"] >= 1).all()
assert (distribution["test"] >= 1).all()

manifest_columns = [
    "record_id", "source_file", "source_row",
    "label", "dialogue_hash", "group_id", "split"
]

manifest_bytes = (
    data[manifest_columns]
    .to_csv(index=False)
    .encode("utf-8")
)

manifest_path = (
    AUDIT_DIR / "constrained_split_manifest.csv"
)

if manifest_path.exists():
    if manifest_path.read_bytes() != manifest_bytes:
        raise RuntimeError(
            "A different constrained manifest already exists. "
            "Preserve both versions rather than overwriting."
        )
else:
    manifest_path.write_bytes(manifest_bytes)

data.to_csv(
    AUDIT_DIR / "constrained_dialogue_pool_with_splits.csv",
    index=False
)

distribution.to_csv(
    AUDIT_DIR / "constrained_class_distribution_per_split.csv"
)

metadata = {
    "method": "constrained group allocation using scipy.optimize.milp",
    "seed": SEED,
    "scipy_version": scipy.__version__,
    "target_fractions": fractions.tolist(),
    "partition_order": partitions,
    "target_sizes": target_sizes.tolist(),
    "class_count_bounds": "floor/ceil of proportional targets",
    "minimum_support": {
        "train": 3,
        "validation": 1,
        "test": 1
    },
    "objective": "fixed seeded allocation costs; no model outcomes",
    "solver_status": int(result.status),
    "solver_message": str(result.message),
    "verified_feasible": True,
    "manifest_sha256": hashlib.sha256(
        manifest_bytes
    ).hexdigest(),
    "historical_exposure_note": (
        "Repartitioning previously examined source data does not "
        "establish an independently untouched external holdout."
    )
}

(AUDIT_DIR / "constrained_split_metadata.json").write_text(
    json.dumps(metadata, indent=2)
)

# Use this variable only after reviewing the output.
constrained_split_pool = data

print("\nVerified split sizes:")
for p, n in zip(partitions, actual_sizes):
    print(f"{p}: {n} records ({100*n/N:.2f}%)")

print("\nClass distribution:")
print(distribution.to_string())

print("\nGroup crossings:",
      int((data.groupby("group_id")["split"].nunique() > 1).sum()))

print("Exact-dialogue crossings:",
      int((data.groupby("dialogue_hash")["split"].nunique() > 1).sum()))

print("\nManifest SHA256:", metadata["manifest_sha256"])
print("\nNo models trained or scored.")

In [ ]:
from pathlib import Path
import hashlib
import numpy as np
import pandas as pd

AUDIT_DIR = Path(str(REVISION_WORK_ROOT / "data_audit"))
mp = AUDIT_DIR / "constrained_split_manifest.csv"
dp = AUDIT_DIR / "constrained_dialogue_pool_with_splits.csv"

assert mp.exists(), f"Missing file: {mp}"
assert dp.exists(), f"Missing file: {dp}"

expected = "ca62ec75e905a4dbf4508ad172fe33e50f7b7779e4a90ad6f58c59c91a7c4d91"
assert hashlib.sha256(mp.read_bytes()).hexdigest() == expected

manifest = pd.read_csv(mp)
model_pool = pd.read_csv(dp)

cols = [
    "record_id", "source_file", "source_row",
    "label", "dialogue_hash", "group_id", "split"
]

assert manifest["record_id"].is_unique
assert model_pool["record_id"].is_unique

pd.testing.assert_frame_equal(
    model_pool[cols].reset_index(drop=True),
    manifest[cols].reset_index(drop=True),
    check_dtype=False
)

print("Cell 1 complete: frozen split verified.")

In [ ]:
assert "duplicate_audit" in globals(), "Missing duplicate_audit; stop here."

source = duplicate_audit.set_index(
    "record_id", verify_integrity=True
)["normalized_dialogue"]

expected_text = model_pool["record_id"].map(source)

assert model_pool["text"].notna().all()
assert model_pool["text"].str.strip().ne("").all()
assert expected_text.notna().all()
assert np.array_equal(model_pool["text"].to_numpy(), expected_text.to_numpy())

CLASSES = sorted(model_pool["label"].unique())
N_CLS = len(CLASSES)
CIDX = {c: i for i, c in enumerate(CLASSES)}

train_data = model_pool[model_pool["split"] == "train"].copy()
val_data = model_pool[model_pool["split"] == "validation"].copy()
test_data = model_pool[model_pool["split"] == "test"].copy()

X_train = train_data["text"].to_numpy()
y_train_np = train_data["label"].to_numpy()
X_val = val_data["text"].to_numpy()
y_val_np = val_data["label"].to_numpy()
X_test = test_data["text"].to_numpy()
y_test_np = test_data["label"].to_numpy()

y_train, y_val, y_test = y_train_np, y_val_np, y_test_np

print("Cell 2 complete: dialogue-only arrays initialized.")

In [ ]:
train_ids = train_data["record_id"].to_numpy()
ids_val = val_data["record_id"].to_numpy()
ids_test = test_data["record_id"].to_numpy()

train_groups = train_data["group_id"].to_numpy()
val_groups = val_data["group_id"].to_numpy()
test_groups = test_data["group_id"].to_numpy()

assert not set(train_groups) & set(val_groups)
assert not set(train_groups) & set(test_groups)
assert not set(val_groups) & set(test_groups)

for labels in [y_train_np, y_val_np, y_test_np]:
    assert set(labels) == set(CLASSES)

minimum_support = int(pd.Series(y_train_np).value_counts().min())
assert minimum_support >= 3

MODEL_OUTDIR = Path(
    str(REVISION_WORK_ROOT / "corrected_experiment_outputs")
)
MODEL_OUTDIR.mkdir(parents=True, exist_ok=True)

FINAL_PROTOCOL_APPROVED = False

print("Training records:", len(X_train))
print("Validation records:", len(X_val))
print("Test records:", len(X_test))
print("Classes:", N_CLS)
print("Minimum training support:", minimum_support)
print("No models trained or scored.")

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold
import numpy as np
import pandas as pd

SEED = 42
CALIBRATION_FOLDS = 3

calibration_splitter = StratifiedGroupKFold(
    n_splits=CALIBRATION_FOLDS,
    shuffle=True,
    random_state=SEED
)

calibration_splits = list(
    calibration_splitter.split(
        X_train,
        y_train_np,
        groups=train_groups
    )
)

fold_audit = []
calibration_membership = np.full(len(X_train), -1)

for fold, (fit_idx, calibration_idx) in enumerate(
    calibration_splits, start=1
):
    assert not set(train_groups[fit_idx]).intersection(
        set(train_groups[calibration_idx])
    )

    calibration_membership[calibration_idx] = fold

    fit_counts = (
        pd.Series(y_train_np[fit_idx])
        .value_counts()
        .reindex(CLASSES, fill_value=0)
    )

    calibration_counts = (
        pd.Series(y_train_np[calibration_idx])
        .value_counts()
        .reindex(CLASSES, fill_value=0)
    )

    for label in CLASSES:
        fold_audit.append({
            "fold": fold,
            "label": label,
            "classifier_fit_count": int(fit_counts[label]),
            "calibration_count": int(calibration_counts[label])
        })

fold_audit = pd.DataFrame(fold_audit)

assert (calibration_membership > 0).all()

fold_audit.to_csv(
    MODEL_OUTDIR / "svm_calibration_fold_class_counts.csv",
    index=False
)

pd.DataFrame({
    "record_id": train_ids,
    "group_id": train_groups,
    "calibration_fold": calibration_membership
}).to_csv(
    MODEL_OUTDIR / "svm_calibration_fold_manifest.csv",
    index=False
)

problems = fold_audit[
    (fold_audit["classifier_fit_count"] == 0)
    | (fold_audit["calibration_count"] == 0)
]

print("Calibration folds:", len(calibration_splits))
print("Group crossings: 0")
print("\nMissing-class problems:")
print(problems.to_string(index=False) if len(problems) else "None.")
print("\nMinimum support across folds:")
print(
    fold_audit[
        ["classifier_fit_count", "calibration_count"]
    ].min().to_string()
)
print("\nNo models trained or scored.")

In [ ]:
import numpy as np
import pandas as pd
from scipy.optimize import milp, Bounds, LinearConstraint
from scipy.sparse import lil_matrix

K = 3
SEED = 42

cal_data = pd.DataFrame({
    "record_id": train_ids,
    "group_id": train_groups,
    "label": y_train_np
})

group_counts = (
    pd.crosstab(cal_data["group_id"], cal_data["label"])
    .reindex(columns=CLASSES, fill_value=0)
    .sort_index()
)

group_names = group_counts.index.to_numpy()
counts = group_counts.to_numpy()
sizes = counts.sum(axis=1)
totals = counts.sum(axis=0)

G, C = counts.shape
n = len(cal_data)

fold_sizes = np.full(K, n // K)
fold_sizes[:n % K] += 1

lower_class = np.maximum(np.floor(totals / K).astype(int), 1)
upper_class = np.ceil(totals / K).astype(int)

assert np.all(lower_class <= upper_class)

A = lil_matrix((G + K + C*K, G*K), dtype=float)
lo = np.zeros(A.shape[0])
hi = np.zeros(A.shape[0])
row = 0

for g in range(G):
    A[row, g*K:(g+1)*K] = 1
    lo[row] = hi[row] = 1
    row += 1

for k in range(K):
    for g in range(G):
        A[row, g*K+k] = sizes[g]
    lo[row] = hi[row] = fold_sizes[k]
    row += 1

for c in range(C):
    for k in range(K):
        for g in np.flatnonzero(counts[:, c]):
            A[row, g*K+k] = counts[g, c]
        lo[row] = lower_class[c]
        hi[row] = upper_class[c]
        row += 1

result = milp(
    c=np.random.default_rng(SEED).random(G*K),
    integrality=np.ones(G*K, dtype=int),
    bounds=Bounds(np.zeros(G*K), np.ones(G*K)),
    constraints=LinearConstraint(A.tocsr(), lo, hi),
    options={"time_limit": 120}
)

print("Solver status:", result.status)
print("Solver message:", result.message)

if result.x is None:
    raise RuntimeError("No allocation returned. Stop before fitting.")

assignment = result.x.reshape(G, K)
assert np.allclose(assignment, np.round(assignment), atol=1e-6)
assignment = np.round(assignment).astype(int)
assert (assignment.sum(axis=1) == 1).all()

group_to_fold = {
    group_names[g]: int(assignment[g].argmax())
    for g in range(G)
}

fold_number = cal_data["group_id"].map(group_to_fold).to_numpy()

print("Allocation created. Run Cell 2 to verify and save it.")

In [ ]:
import hashlib
import json
import scipy

calibration_splits = []
audit_rows = []

for fold in range(K):
    calibration_idx = np.flatnonzero(fold_number == fold)
    fit_idx = np.flatnonzero(fold_number != fold)

    assert not set(train_groups[fit_idx]) & set(
        train_groups[calibration_idx]
    )

    fit_counts = (
        pd.Series(y_train_np[fit_idx])
        .value_counts().reindex(CLASSES, fill_value=0)
    )

    calibration_counts = (
        pd.Series(y_train_np[calibration_idx])
        .value_counts().reindex(CLASSES, fill_value=0)
    )

    assert (fit_counts >= 2).all()
    assert (calibration_counts >= 1).all()
    assert len(calibration_idx) == fold_sizes[fold]
    assert np.all(calibration_counts.to_numpy() >= lower_class)
    assert np.all(calibration_counts.to_numpy() <= upper_class)

    calibration_splits.append((fit_idx, calibration_idx))

    for label in CLASSES:
        audit_rows.append({
            "fold": fold + 1,
            "label": label,
            "classifier_fit_count": int(fit_counts[label]),
            "calibration_count": int(calibration_counts[label])
        })

calibration_audit = pd.DataFrame(audit_rows)
calibration_manifest = cal_data.copy()
calibration_manifest["calibration_fold"] = fold_number + 1

assert (
    calibration_manifest.groupby("group_id")
    ["calibration_fold"].nunique().max() == 1
)

manifest_bytes = calibration_manifest.to_csv(
    index=False
).encode("utf-8")

manifest_path = MODEL_OUTDIR / "constrained_calibration_manifest.csv"

if manifest_path.exists():
    if manifest_path.read_bytes() != manifest_bytes:
        raise RuntimeError(
            "A different calibration manifest already exists. "
            "Stop rather than overwrite it."
        )
else:
    manifest_path.write_bytes(manifest_bytes)

calibration_audit.to_csv(
    MODEL_OUTDIR / "constrained_calibration_class_counts.csv",
    index=False
)

metadata = {
    "folds": K,
    "seed": SEED,
    "scipy_version": scipy.__version__,
    "solver_status": int(result.status),
    "solver_message": str(result.message),
    "allocation": "Training-only constrained group allocation",
    "objective": "Seeded allocation costs; no prediction outcomes",
    "manifest_sha256": hashlib.sha256(manifest_bytes).hexdigest()
}

(MODEL_OUTDIR / "constrained_calibration_metadata.json").write_text(
    json.dumps(metadata, indent=2)
)

print("Calibration fold sizes:")
print(
    calibration_manifest["calibration_fold"]
    .value_counts().sort_index().to_string()
)

print("\nMinimum support across folds:")
print(
    calibration_audit[
        ["classifier_fit_count", "calibration_count"]
    ].min().to_string()
)

print("\nMissing-class problems: None.")
print("Group crossings: 0.")
print("No models trained or scored.")

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV

BASELINE_CFG = {
    "seed": 42,
    "ngram_range": (1, 2),
    "max_features": 18000,
    "sublinear_tf": True,
    "lr_C": 1.0,
    "lr_max_iter": 2000,
    "svm_C": 1.0,
    "svm_max_iter": 3000,
    "calibration_method": "sigmoid",
    "calibration_ensemble": True
}

def make_vectorizer():
    return TfidfVectorizer(
        ngram_range=BASELINE_CFG["ngram_range"],
        max_features=BASELINE_CFG["max_features"],
        sublinear_tf=BASELINE_CFG["sublinear_tf"]
    )

lr_model = Pipeline([
    ("tfidf", make_vectorizer()),
    ("classifier", LogisticRegression(
        C=BASELINE_CFG["lr_C"],
        max_iter=BASELINE_CFG["lr_max_iter"],
        random_state=BASELINE_CFG["seed"]
    ))
])

lr_model.fit(X_train, y_train_np)

print("Logistic Regression fitted on training dialogues only.")
print("Training records:", len(X_train))
print("Learned classes:", len(lr_model.classes_))

In [ ]:
import inspect

svm_pipeline = Pipeline([
    ("tfidf", make_vectorizer()),
    ("classifier", LinearSVC(
        C=BASELINE_CFG["svm_C"],
        max_iter=BASELINE_CFG["svm_max_iter"],
        random_state=BASELINE_CFG["seed"]
    ))
])

calibration_kwargs = {
    "method": BASELINE_CFG["calibration_method"],
    "cv": calibration_splits,
    "ensemble": BASELINE_CFG["calibration_ensemble"]
}

# Support the parameter name used by the installed sklearn version.
parameters = inspect.signature(
    CalibratedClassifierCV
).parameters

if "estimator" in parameters:
    calibration_kwargs["estimator"] = svm_pipeline
else:
    calibration_kwargs["base_estimator"] = svm_pipeline

svm_model = CalibratedClassifierCV(**calibration_kwargs)

svm_model.fit(X_train, y_train_np)

print("Calibrated SVM fitted using training-only grouped folds.")
print("Calibration ensembles:",
      len(svm_model.calibrated_classifiers_))
print("Learned classes:", len(svm_model.classes_))
print("No validation or test records used for fitting.")

In [ ]:
import numpy as np
import pandas as pd

def score_validation(model, name):
    raw_probabilities = model.predict_proba(X_val)

    probabilities = np.zeros((len(X_val), N_CLS))

    for column, label in enumerate(model.classes_):
        probabilities[:, CIDX[label]] = raw_probabilities[:, column]

    assert np.isfinite(probabilities).all()
    assert (probabilities >= 0).all()
    assert np.allclose(probabilities.sum(axis=1), 1.0)

    predictions = np.array(CLASSES)[
        probabilities.argmax(axis=1)
    ]

    confidence = probabilities.max(axis=1)
    correct = predictions == y_val_np

    np.savez_compressed(
        MODEL_OUTDIR / f"validation_{name}_probabilities.npz",
        record_ids=ids_val,
        classes=np.array(CLASSES),
        true_labels=y_val_np,
        predictions=predictions,
        probabilities=probabilities,
        confidence=confidence
    )

    pd.DataFrame({
        "record_id": ids_val,
        "true_label": y_val_np,
        "prediction": predictions,
        "confidence": confidence,
        "correct": correct
    }).to_csv(
        MODEL_OUTDIR / f"validation_{name}_predictions.csv",
        index=False
    )

    return {
        "pred": predictions,
        "probs": probabilities,
        "confidence": confidence,
        "ranking_score": confidence
    }

VAL_BASELINES = {
    "TFIDF_LR": score_validation(lr_model, "TFIDF_LR"),
    "TFIDF_SVM": score_validation(svm_model, "TFIDF_SVM")
}

for name, result in VAL_BASELINES.items():
    correct_count = int(
        (result["pred"] == y_val_np).sum()
    )
    print(
        f"{name}: validation correct "
        f"{correct_count}/{len(y_val_np)}; "
        f"accuracy={correct_count/len(y_val_np):.4f}"
    )

print("Validation predictions saved. Test predictions not generated.")

In [ ]:
import joblib
import json
import sklearn

joblib.dump(
    lr_model,
    MODEL_OUTDIR / "trained_tfidf_lr.joblib"
)

joblib.dump(
    svm_model,
    MODEL_OUTDIR / "trained_group_calibrated_svm.joblib"
)

baseline_metadata = dict(BASELINE_CFG)
baseline_metadata["sklearn_version"] = sklearn.__version__
baseline_metadata["training_records"] = len(X_train)
baseline_metadata["classes"] = list(CLASSES)
baseline_metadata["input_fields"] = ["normalized dialogue"]
baseline_metadata["calibration_folds"] = len(calibration_splits)

(MODEL_OUTDIR / "supervised_baseline_configuration.json").write_text(
    json.dumps(baseline_metadata, indent=2)
)

FINAL_PROTOCOL_APPROVED = False

print("Models and configuration saved in Drive.")
print("Final test evaluation remains blocked.")

In [ ]:
%pip install -q sentence-transformers rank-bm25

In [ ]:
import numpy as np
from rank_bm25 import BM25Okapi

RETRIEVAL_CFG = {
    "seed": 42,
    "bm25_k1": 1.5,
    "bm25_b": 0.75,
    "topk_retrieve": 20,
    "rrf_k": 60,
    "rrf_top_n": 10,
    "cascade_top_k_classes": 5,
    "doc_truncate_chars": 400,
    "length_norm_alpha": 0.5,
    "dense_model": "sentence-transformers/all-MiniLM-L6-v2",
    "ce_model": "cross-encoder/ms-marco-MiniLM-L-6-v2",
    "ce_max_tokens": 512,
    "batch_size": 16
}

train_tokens = [text.split() for text in X_train]

bm25 = BM25Okapi(
    train_tokens,
    k1=RETRIEVAL_CFG["bm25_k1"],
    b=RETRIEVAL_CFG["bm25_b"]
)

print("BM25 built from training dialogues only.")
print("Indexed records:", len(train_tokens))

In [ ]:
import torch
from sentence_transformers import SentenceTransformer, CrossEncoder

torch.manual_seed(RETRIEVAL_CFG["seed"])

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RETRIEVAL_CFG["seed"])

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

embedder = SentenceTransformer(
    RETRIEVAL_CFG["dense_model"],
    device=DEVICE
)

train_embs = embedder.encode(
    X_train.tolist(),
    batch_size=RETRIEVAL_CFG["batch_size"],
    normalize_embeddings=True,
    convert_to_numpy=True,
    show_progress_bar=True
)

assert train_embs.shape[0] == len(X_train)
assert np.isfinite(train_embs).all()

embedding_norms = np.linalg.norm(train_embs, axis=1)
assert np.allclose(embedding_norms, 1.0, atol=1e-4)

np.savez_compressed(
    MODEL_OUTDIR / "training_dense_embeddings.npz",
    record_ids=train_ids,
    embeddings=train_embs
)

print("Device:", DEVICE)
print("Training embedding shape:", train_embs.shape)
print("Dense maximum sequence length:", embedder.max_seq_length)
print("Training embeddings saved.")

In [ ]:
import inspect
import importlib.metadata
import json

cross_enc = CrossEncoder(
    RETRIEVAL_CFG["ce_model"],
    device=DEVICE,
    max_length=RETRIEVAL_CFG["ce_max_tokens"]
)

runtime_metadata = dict(RETRIEVAL_CFG)

runtime_metadata["device"] = DEVICE
runtime_metadata["dense_max_sequence_length"] = int(
    embedder.max_seq_length
)
runtime_metadata["training_records"] = len(X_train)

runtime_metadata["package_versions"] = {
    name: importlib.metadata.version(name)
    for name in [
        "numpy",
        "scikit-learn",
        "sentence-transformers",
        "transformers",
        "rank-bm25",
        "torch"
    ]
}

runtime_metadata["cross_encoder_predict_signature"] = str(
    inspect.signature(cross_enc.predict)
)

(MODEL_OUTDIR / "retrieval_initialization_metadata.json").write_text(
    json.dumps(runtime_metadata, indent=2)
)

FINAL_PROTOCOL_APPROVED = False

print("Cross-encoder loaded.")
print("Prediction API:", inspect.signature(cross_enc.predict))
print("Retrieval initialization metadata saved.")
print("No validation or test retrieval predictions generated.")

In [ ]:
import torch
import numpy as np
from scipy.special import softmax

def ce_raw_logits(pairs):
    if not pairs:
        raise ValueError("Cross-encoder received no candidate pairs.")

    scores = cross_enc.predict(
        pairs,
        batch_size=RETRIEVAL_CFG["batch_size"],
        activation_fn=torch.nn.Identity(),
        apply_softmax=False,
        convert_to_numpy=True,
        show_progress_bar=False
    )

    scores = np.asarray(scores, dtype=float).reshape(-1)

    assert len(scores) == len(pairs)
    assert np.isfinite(scores).all()

    return scores

print("Raw-logit scoring function defined.")
print("No queries scored.")

In [ ]:
def descending_order(scores, identifiers):
    scores = np.asarray(scores, dtype=float)
    identifiers = np.asarray(identifiers, dtype=str)

    assert len(scores) == len(identifiers)
    assert np.isfinite(scores).all()

    return np.lexsort((identifiers, -scores))


def bm25_retrieve(query, k=None):
    if k is None:
        k = RETRIEVAL_CFG["topk_retrieve"]

    scores = bm25.get_scores(query.split())
    indices = descending_order(scores, train_ids)[:k]

    return [(int(i), float(scores[i])) for i in indices]


def dense_retrieve(query, k=None):
    if k is None:
        k = RETRIEVAL_CFG["topk_retrieve"]

    query_embedding = embedder.encode(
        [query],
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=False
    )[0]

    scores = train_embs @ query_embedding
    indices = descending_order(scores, train_ids)[:k]

    return [(int(i), float(scores[i])) for i in indices]


def rrf_fuse(sparse_results, dense_results, top_n=None):
    if top_n is None:
        top_n = RETRIEVAL_CFG["rrf_top_n"]

    fused_scores = {}

    for results in [sparse_results, dense_results]:
        for rank, (index, _) in enumerate(results, start=1):
            fused_scores[index] = (
                fused_scores.get(index, 0.0)
                + 1.0 / (RETRIEVAL_CFG["rrf_k"] + rank)
            )

    ranked = sorted(
        fused_scores.items(),
        key=lambda item: (-item[1], str(train_ids[item[0]]))
    )

    return ranked[:top_n]

print("Deterministic retrieval and RRF functions defined.")
print("No queries scored.")

In [ ]:
from collections import Counter

UNCERTAINTY_CFG = {
    "weights": [0.5, 0.3, 0.2],
    "weight_policy": "fixed heuristic; not optimized",
    "variance_epsilon": 1e-10,
    "variance_ddof": 0,
    "variance_stage": "final document set used for CE aggregation"
}

def normalized_entropy(probabilities):
    p = np.asarray(probabilities, dtype=float)

    assert p.shape == (N_CLS,)
    assert np.isfinite(p).all()
    assert (p >= 0).all()
    assert np.isclose(p.sum(), 1.0)

    positive = p[p > 0]
    value = -np.sum(positive * np.log(positive)) / np.log(N_CLS)

    return float(np.clip(value, 0.0, 1.0))


def uncertainty_components(class_scores, fused_documents):
    if not fused_documents:
        raise ValueError("No documents available for uncertainty calculation.")

    context_labels = [
        y_train_np[index] for index, _ in fused_documents
    ]

    counts = Counter(context_labels)

    label_distribution = np.array([
        counts.get(label, 0) / len(context_labels)
        for label in CLASSES
    ])

    rrf_scores = np.array([
        score for _, score in fused_documents
    ], dtype=float)

    if len(rrf_scores) < 2:
        variance_signal = 0.0
    else:
        denominator = (
            rrf_scores.max()
            + UNCERTAINTY_CFG["variance_epsilon"]
        ) ** 2

        variance_signal = min(
            1.0,
            np.var(
                rrf_scores,
                ddof=UNCERTAINTY_CFG["variance_ddof"]
            ) / denominator
        )

    components = np.array([
        normalized_entropy(class_scores),
        normalized_entropy(label_distribution),
        float(variance_signal)
    ])

    assert np.isfinite(components).all()
    assert ((components >= 0) & (components <= 1)).all()

    return components


def composite_crus(components):
    weights = np.asarray(UNCERTAINTY_CFG["weights"], dtype=float)

    assert (weights >= 0).all()
    assert np.isclose(weights.sum(), 1.0)

    # Preserve precision; round only when displaying results.
    return float(weights @ np.asarray(components, dtype=float))

print("Uncertainty functions defined.")
print("Weights are declared fixed heuristic settings.")
print("No predictions or thresholds selected.")

In [ ]:
def prepare_candidates(query):
    sparse = bm25_retrieve(query)
    dense = dense_retrieve(query)

    candidate_labels = sorted({
        y_train_np[index]
        for index, _ in sparse + dense
    })

    if not candidate_labels:
        raise RuntimeError("No candidate labels retrieved.")

    label_logits = ce_raw_logits([
        (query, label) for label in candidate_labels
    ])

    label_scores = softmax(label_logits)

    shortlist_order = descending_order(
        label_scores, candidate_labels
    )

    shortlist = [
        candidate_labels[index]
        for index in shortlist_order[
            :RETRIEVAL_CFG["cascade_top_k_classes"]
        ]
    ]

    return {
        "sparse": sparse,
        "dense": dense,
        "candidate_labels": candidate_labels,
        "label_logits": label_logits,
        "label_scores": label_scores,
        "shortlist": shortlist
    }


def score_sk4(candidates):
    probabilities = np.zeros(N_CLS)

    for label, score in zip(
        candidates["candidate_labels"],
        candidates["label_scores"]
    ):
        probabilities[CIDX[label]] = score

    assert np.isclose(probabilities.sum(), 1.0)

    prediction = CLASSES[int(probabilities.argmax())]

    return {
        "pred": prediction,
        "probs": probabilities,
        "confidence": float(probabilities.max()),
        "ranking_score": float(probabilities.max()),
        "components": None,
        "documents": [],
        "shortlist": candidates["shortlist"],
        "fallback": False,
        "lengthnorm_clipped": 0
    }

print("Candidate preparation and SK4 scorer defined.")
print("No queries scored.")

In [ ]:
average_training_length = float(
    np.mean([len(text) for text in X_train])
)

assert average_training_length > 0

DOCUMENT_METHODS = [
    "SingleStageFull",
    "SingleStageTruncated",
    "Cascade",
    "LengthNormLegacy"
]


def score_documents(query, candidates, method):
    if method not in DOCUMENT_METHODS:
        raise ValueError(f"Unknown method: {method}")

    fallback = False
    clipped_count = 0

    if method == "Cascade":
        fused = rrf_fuse(
            candidates["sparse"],
            candidates["dense"],
            top_n=2 * RETRIEVAL_CFG["rrf_top_n"]
        )

        shortlist = set(candidates["shortlist"])

        selected = [
            item for item in fused
            if y_train_np[item[0]] in shortlist
        ]

        if not selected:
            selected = fused
            fallback = True

        documents = selected[:RETRIEVAL_CFG["rrf_top_n"]]

    else:
        documents = rrf_fuse(
            candidates["sparse"],
            candidates["dense"],
            top_n=RETRIEVAL_CFG["rrf_top_n"]
        )

    if not documents:
        raise RuntimeError("No documents available for reranking.")

    full_texts = [
        X_train[index] for index, _ in documents
    ]

    if method in ["Cascade", "SingleStageTruncated"]:
        scoring_texts = [
            text[:RETRIEVAL_CFG["doc_truncate_chars"]]
            for text in full_texts
        ]
    else:
        scoring_texts = full_texts

    logits = ce_raw_logits([
        (query, text) for text in scoring_texts
    ])

    minimum_penalty = np.nan

    if method == "LengthNormLegacy":
        length_ratio = (
            np.array([len(text) for text in full_texts])
            / average_training_length
        )

        penalty = (
            1.0
            + RETRIEVAL_CFG["length_norm_alpha"]
            * np.log(np.maximum(length_ratio, 1e-6))
        )

        minimum_penalty = float(penalty.min())
        clipped_count = int((penalty < 1e-6).sum())

        # Retained only as the explicitly defined legacy heuristic.
        logits = logits / np.maximum(penalty, 1e-6)

    assert np.isfinite(logits).all()

    document_weights = softmax(logits)
    probabilities = np.zeros(N_CLS)

    for (index, _), weight in zip(
        documents, document_weights
    ):
        probabilities[CIDX[y_train_np[index]]] += weight

    probabilities /= probabilities.sum()

    components = uncertainty_components(
        probabilities, documents
    )

    return {
        "pred": CLASSES[int(probabilities.argmax())],
        "probs": probabilities,
        "confidence": float(probabilities.max()),
        "ranking_score": float(probabilities.max()),
        "components": components,
        "documents": documents,
        "shortlist": (
            candidates["shortlist"]
            if method == "Cascade" else []
        ),
        "fallback": fallback,
        "lengthnorm_clipped": clipped_count,
        "lengthnorm_min_penalty": minimum_penalty
    }

print("Document scorers and Cascade defined.")
print("No queries scored.")

In [ ]:
RETRIEVAL_METHODS = [
    "BM25",
    "SBERT",
    "SK4",
    "SingleStageFull",
    "SingleStageTruncated",
    "Cascade",
    "LengthNormLegacy"
]


def retrieval_only_result(results):
    index, score = results[0]

    return {
        "pred": y_train_np[index],
        "probs": None,
        "confidence": np.nan,
        "ranking_score": float(score),
        "components": None,
        "documents": results[:1],
        "shortlist": [],
        "fallback": False,
        "lengthnorm_clipped": 0
    }


def score_retrieval_methods(query):
    candidates = prepare_candidates(query)

    output = {
        "BM25": retrieval_only_result(
            candidates["sparse"]
        ),
        "SBERT": retrieval_only_result(
            candidates["dense"]
        ),
        "SK4": score_sk4(candidates)
    }

    for method in DOCUMENT_METHODS:
        output[method] = score_documents(
            query, candidates, method
        )

    return output

FINAL_PROTOCOL_APPROVED = False

print("Combined retrieval scoring function defined.")
print("Methods:", RETRIEVAL_METHODS)
print("No validation or test predictions generated.")

In [ ]:
from tqdm.auto import tqdm
import joblib

checkpoint_path = (
    MODEL_OUTDIR / "validation_retrieval_checkpoint.joblib"
)

if checkpoint_path.exists():
    raise RuntimeError(
        "A validation checkpoint already exists. "
        "Do not overwrite it; restore and verify it first."
    )

validation_rows = {
    method: [] for method in RETRIEVAL_METHODS
}

for query in tqdm(
    X_val,
    total=len(X_val),
    desc="Validation retrieval/reranking"
):
    query_results = score_retrieval_methods(query)

    for method in RETRIEVAL_METHODS:
        validation_rows[method].append(
            query_results[method]
        )

joblib.dump(
    {
        "record_ids": ids_val,
        "true_labels": y_val_np,
        "classes": CLASSES,
        "results": validation_rows,
        "retrieval_configuration": RETRIEVAL_CFG,
        "uncertainty_configuration": UNCERTAINTY_CFG
    },
    checkpoint_path
)

print("Validation scoring completed.")
print("Validation records:", len(X_val))
print("Checkpoint saved.")
print("No test predictions generated.")

In [ ]:
import numpy as np
import pandas as pd
import json

VAL_RETRIEVAL = {}
prediction_audit = []

for method, rows in validation_rows.items():
    assert len(rows) == len(X_val)

    predictions = np.array([
        row["pred"] for row in rows
    ])

    confidence = np.array([
        row["confidence"] for row in rows
    ])

    ranking_score = np.array([
        row["ranking_score"] for row in rows
    ])

    probabilities = (
        None if rows[0]["probs"] is None
        else np.stack([row["probs"] for row in rows])
    )

    components = (
        None if rows[0]["components"] is None
        else np.stack([row["components"] for row in rows])
    )

    if probabilities is not None:
        assert probabilities.shape == (len(X_val), N_CLS)
        assert np.allclose(probabilities.sum(axis=1), 1.0)
        assert np.array_equal(
            predictions,
            np.array(CLASSES)[probabilities.argmax(axis=1)]
        )

    VAL_RETRIEVAL[method] = {
        "pred": predictions,
        "probs": probabilities,
        "confidence": confidence,
        "ranking_score": ranking_score,
        "components": components,
        "rows": rows
    }

    arrays = {
        "record_ids": ids_val,
        "true_labels": y_val_np,
        "classes": np.array(CLASSES),
        "predictions": predictions,
        "confidence": confidence,
        "ranking_score": ranking_score
    }

    if probabilities is not None:
        arrays["probabilities"] = probabilities

    if components is not None:
        arrays["components"] = components
        arrays["component_names"] = np.array([
            "H_pred", "H_retrieval", "V_RRF"
        ])

    np.savez_compressed(
        MODEL_OUTDIR / f"validation_{method}_arrays.npz",
        **arrays
    )

    for i, row in enumerate(rows):
        entry = {
            "record_id": ids_val[i],
            "method": method,
            "true_label": y_val_np[i],
            "prediction": predictions[i],
            "correct": bool(predictions[i] == y_val_np[i]),
            "confidence": confidence[i],
            "ranking_score": ranking_score[i],
            "fallback": row["fallback"],
            "lengthnorm_clipped": row["lengthnorm_clipped"],
            "lengthnorm_min_penalty": row.get(
                "lengthnorm_min_penalty", np.nan
            ),
            "shortlist": json.dumps(row["shortlist"]),
            "retrieved_training_ids": json.dumps([
                str(train_ids[index])
                for index, _ in row["documents"]
            ])
        }

        if components is not None:
            entry.update({
                "H_pred": components[i, 0],
                "H_retrieval": components[i, 1],
                "V_RRF": components[i, 2],
                "CRUS": composite_crus(components[i])
            })

        prediction_audit.append(entry)

pd.DataFrame(prediction_audit).to_csv(
    MODEL_OUTDIR / "validation_retrieval_prediction_audit.csv",
    index=False
)

print("Full-precision validation arrays and audit saved.")

In [ ]:
from sklearn.metrics import accuracy_score, f1_score

validation_summary = []

for method, result in VAL_RETRIEVAL.items():
    correct = result["pred"] == y_val_np

    validation_summary.append({
        "method": method,
        "n_validation": len(y_val_np),
        "n_correct": int(correct.sum()),
        "forced_accuracy": accuracy_score(
            y_val_np, result["pred"]
        ),
        "macro_f1_fixed_labels": f1_score(
            y_val_np,
            result["pred"],
            labels=CLASSES,
            average="macro",
            zero_division=0
        )
    })

validation_summary = pd.DataFrame(validation_summary)

validation_summary.to_csv(
    MODEL_OUTDIR / "validation_forced_prediction_summary.csv",
    index=False
)

print(validation_summary.to_string(index=False))

cascade_rows = VAL_RETRIEVAL["Cascade"]["rows"]

shortlist_hits = sum(
    y_val_np[i] in row["shortlist"]
    for i, row in enumerate(cascade_rows)
)

fallback_count = sum(
    row["fallback"] for row in cascade_rows
)

legacy_rows = VAL_RETRIEVAL["LengthNormLegacy"]["rows"]

clipped_documents = sum(
    row["lengthnorm_clipped"] for row in legacy_rows
)

queries_with_clipping = sum(
    row["lengthnorm_clipped"] > 0 for row in legacy_rows
)

print(
    "\nCascade shortlist recall:",
    f"{shortlist_hits}/{len(X_val)}",
    f"({shortlist_hits/len(X_val):.4f})"
)
print("Cascade fallback queries:", fallback_count)
print("Legacy LengthNorm clipped document penalties:", clipped_documents)
print("Legacy LengthNorm queries with clipping:", queries_with_clipping)
print("\nTest evaluation remains blocked.")

FINAL_PROTOCOL_APPROVED = False

In [ ]:
from itertools import product
import numpy as np

POLICY_CFG = {
    "minimum_validation_coverage": 0.50,
    "confidence_quantile_points": 21,
    "uncertainty_quantile_points": 101,
    "selection_rule": (
        "maximize validation selective accuracy; "
        "then maximize coverage; "
        "then smaller tau_c; then smaller tau_u"
    )
}

def uncertainty_signals(result):
    p = result["probs"]
    components = result["components"]

    assert p is not None
    assert components is not None

    sorted_p = np.sort(p, axis=1)

    return {
        "CRUS": components @ np.asarray(
            UNCERTAINTY_CFG["weights"], dtype=float
        ),
        "H_pred": components[:, 0],
        "H_retrieval": components[:, 1],
        "V_RRF": components[:, 2],
        "OneMinusMaxProb": 1.0 - p.max(axis=1),
        "OneMinusMargin": (
            1.0 - (sorted_p[:, -1] - sorted_p[:, -2])
        )
    }

def choose_validation_policy(confidence, uncertainty, correct):
    assert np.isfinite(confidence).all()
    assert np.isfinite(uncertainty).all()

    tc_grid = np.unique(np.r_[
        0.0,
        np.quantile(
            confidence,
            np.linspace(
                0, 1,
                POLICY_CFG["confidence_quantile_points"]
            )
        )
    ])

    tu_grid = np.unique(np.r_[
        0.0,
        np.quantile(
            uncertainty,
            np.linspace(
                0, 1,
                POLICY_CFG["uncertainty_quantile_points"]
            )
        ),
        1.0
    ])

    best = None

    for tc, tu in product(tc_grid, tu_grid):
        accepted = (
            (confidence >= tc)
            & (uncertainty <= tu)
        )

        n_accepted = int(accepted.sum())
        coverage = n_accepted / len(correct)

        if n_accepted == 0:
            continue

        if coverage < POLICY_CFG["minimum_validation_coverage"]:
            continue

        n_correct = int(correct[accepted].sum())
        accuracy = n_correct / n_accepted

        key = (
            accuracy, n_accepted,
            -float(tc), -float(tu)
        )

        if best is None or key > best[0]:
            best = (key, {
                "tau_c": float(tc),
                "tau_u": float(tu),
                "n_validation": len(correct),
                "n_accepted": n_accepted,
                "n_correct_accepted": n_correct,
                "coverage": coverage,
                "selective_accuracy": accuracy
            })

    if best is None:
        raise RuntimeError(
            "No policy satisfied the coverage constraint."
        )

    return best[1]

print("Validation-only policy selector defined.")
print("No test data used.")

In [ ]:
import pandas as pd
import json

VALIDATION_POLICIES = {}
policy_rows = []

for method in DOCUMENT_METHODS:
    result = VAL_RETRIEVAL[method]
    correct = result["pred"] == y_val_np
    signals = uncertainty_signals(result)

    selected_names = (
        list(signals)
        if method == "Cascade"
        else ["CRUS"]
    )

    for signal_name in selected_names:
        policy = choose_validation_policy(
            result["confidence"],
            signals[signal_name],
            correct
        )

        key = f"{method}:{signal_name}"
        VALIDATION_POLICIES[key] = policy

        policy_rows.append({
            "method": method,
            "signal": signal_name,
            **policy
        })

policy_table = pd.DataFrame(policy_rows)

policy_path = MODEL_OUTDIR / "validation_selected_policies.json"

payload = {
    "selection_configuration": POLICY_CFG,
    "uncertainty_configuration": UNCERTAINTY_CFG,
    "policies": VALIDATION_POLICIES,
    "note": (
        "Selected from validation predictions only; "
        "not final test results."
    )
}

payload_text = json.dumps(
    payload, indent=2, sort_keys=True
)

if policy_path.exists():
    if policy_path.read_text() != payload_text:
        raise RuntimeError(
            "Different policies already exist. "
            "Preserve the prior version and investigate."
        )
else:
    policy_path.write_text(payload_text)

policy_table.to_csv(
    MODEL_OUTDIR / "validation_policy_selection_summary.csv",
    index=False
)

print(policy_table.to_string(index=False))

FINAL_PROTOCOL_APPROVED = False

print("\nPolicies saved.")
print("Test evaluation remains blocked.")

In [ ]:
import numpy as np
import pandas as pd

cascade_policy = VALIDATION_POLICIES["Cascade:CRUS"]
cascade_result = VAL_RETRIEVAL["Cascade"]
cascade_signals = uncertainty_signals(cascade_result)

native_cascade_mask = (
    (cascade_result["confidence"] >= cascade_policy["tau_c"])
    & (cascade_signals["CRUS"] <= cascade_policy["tau_u"])
)

MATCHED_K = int(native_cascade_mask.sum())
assert MATCHED_K > 0

ranked_comparisons = {}

# Supervised baselines: maximum predicted probability.
for method, result in VAL_BASELINES.items():
    ranked_comparisons[method] = (
        result,
        result["ranking_score"],
        "maximum predicted probability"
    )

# Retrieval/reranking methods: their documented ranking score.
for method, result in VAL_RETRIEVAL.items():
    score_description = {
        "BM25": "top retrieved BM25 score",
        "SBERT": "top retrieved cosine similarity"
    }.get(method, "maximum normalized class score")

    ranked_comparisons[method] = (
        result,
        result["ranking_score"],
        score_description
    )

# Same Cascade predictions, selected by alternative uncertainty scores.
for signal_name, uncertainty in cascade_signals.items():
    ranked_comparisons[f"Cascade_rank_{signal_name}"] = (
        cascade_result,
        -uncertainty,
        f"negative {signal_name}; lower uncertainty retained first"
    )

matched_rows = []
ranking_audit_rows = []

for name, (result, ranking_score, description) in (
    ranked_comparisons.items()
):
    order = descending_order(ranking_score, ids_val)
    selected = order[:MATCHED_K]

    correct = result["pred"] == y_val_np
    n_correct = int(correct[selected].sum())

    matched_rows.append({
        "method": name,
        "policy": "score-ranked",
        "ranking_score": description,
        "n_retained": MATCHED_K,
        "n_correct": n_correct,
        "coverage": MATCHED_K / len(y_val_np),
        "selective_accuracy": n_correct / MATCHED_K
    })

    for rank, index in enumerate(order, start=1):
        ranking_audit_rows.append({
            "method": name,
            "record_id": ids_val[index],
            "rank": rank,
            "ranking_score": float(ranking_score[index]),
            "correct": bool(correct[index]),
            "retained": rank <= MATCHED_K
        })

native_correct = int(
    (
        (cascade_result["pred"] == y_val_np)
        & native_cascade_mask
    ).sum()
)

matched_rows.append({
    "method": "Cascade_native_CRUS",
    "policy": "validation-selected threshold gate",
    "ranking_score": "not a ranked selection",
    "n_retained": MATCHED_K,
    "n_correct": native_correct,
    "coverage": MATCHED_K / len(y_val_np),
    "selective_accuracy": native_correct / MATCHED_K
})

VAL_MATCHED = pd.DataFrame(matched_rows)

VAL_MATCHED.to_csv(
    MODEL_OUTDIR / "validation_matched_coverage_counts.csv",
    index=False
)

pd.DataFrame(ranking_audit_rows).to_csv(
    MODEL_OUTDIR / "validation_ranked_selection_audit.csv",
    index=False
)

print("Matched retained count:", MATCHED_K)
print(
    VAL_MATCHED[
        [
            "method", "policy", "n_retained",
            "n_correct", "coverage", "selective_accuracy"
        ]
    ].to_string(index=False)
)

print("\nValidation audit saved. No test predictions generated.")

In [ ]:
crus_order = descending_order(
    -cascade_signals["CRUS"],
    ids_val
)

ranked_crus_mask = np.zeros(len(y_val_np), dtype=bool)
ranked_crus_mask[crus_order[:MATCHED_K]] = True

same_selection = np.array_equal(
    native_cascade_mask,
    ranked_crus_mask
)

comparison = pd.DataFrame({
    "record_id": ids_val,
    "CRUS": cascade_signals["CRUS"],
    "native_CRUS_accepted": native_cascade_mask,
    "ranked_CRUS_retained": ranked_crus_mask,
    "same_decision": (
        native_cascade_mask == ranked_crus_mask
    )
})

comparison.to_csv(
    MODEL_OUTDIR / "validation_native_vs_ranked_CRUS.csv",
    index=False
)

print("Native and ranked CRUS selections identical:", same_selection)
print(
    "Cases with different selection decisions:",
    int((native_cascade_mask != ranked_crus_mask).sum())
)

FINAL_PROTOCOL_APPROVED = False

print("Final test evaluation remains blocked.")

In [ ]:
import json
import hashlib

main_manifest = AUDIT_DIR / "constrained_split_manifest.csv"
cal_manifest = MODEL_OUTDIR / "constrained_calibration_manifest.csv"
policy_file = MODEL_OUTDIR / "validation_selected_policies.json"
FINAL_PROTOCOL_PATH = MODEL_OUTDIR / "final_pretest_protocol.json"

def file_sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

expected = "ca62ec75e905a4dbf4508ad172fe33e50f7b7779e4a90ad6f58c59c91a7c4d91"
assert file_sha256(main_manifest) == expected

saved = json.loads(policy_file.read_text())
assert saved["policies"] == VALIDATION_POLICIES

print("Split and selected policies verified.")
print("Final protocol already exists:", FINAL_PROTOCOL_PATH.exists())
print("No test predictions generated.")

In [ ]:
print("No test predictions generated.")

In [ ]:
print("Final protocol already exists:", FINAL_PROTOCOL_PATH.exists())

if FINAL_PROTOCOL_PATH.exists():
    existing_protocol = json.loads(FINAL_PROTOCOL_PATH.read_text())
    print("Protocol version:", existing_protocol.get("protocol_version"))
    print("Protocol SHA256:", file_sha256(FINAL_PROTOCOL_PATH))
else:
    print("No final protocol file exists yet.")

FINAL_PROTOCOL_APPROVED = False
print("Test evaluation remains blocked.")

In [ ]:
from datetime import datetime, timezone

FINAL_PROTOCOL = {
    "protocol_version": "revision2_dialogue_only_v1",
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "model_input": "normalized dialogue only",
    "header_usage": "target and training-label metadata only",
    "reference_section_text_usage": "not used as model input",
    "classes": list(CLASSES),
    "split_sizes": {
        "train": len(X_train),
        "validation": len(X_val),
        "test": len(X_test)
    },
    "main_manifest_sha256": file_sha256(main_manifest),
    "calibration_manifest_sha256": file_sha256(cal_manifest),
    "baseline_configuration": dict(BASELINE_CFG),
    "retrieval_configuration": dict(RETRIEVAL_CFG),
    "uncertainty_configuration": dict(UNCERTAINTY_CFG),
    "policy_selection_configuration": dict(POLICY_CFG),
    "frozen_policies": json.loads(policy_file.read_text())["policies"],
    "primary_method": "Cascade",
    "primary_policy": "Cascade:CRUS",
    "cross_encoder_activation": "Identity; raw logits",
    "ranking_ties": "score descending; record ID ascending"
}

print("Protocol settings assembled.")

In [ ]:
FINAL_PROTOCOL["reporting_rules"] = {
    "forced_accuracy": "correct forced predictions / all cases",
    "coverage": "accepted cases / all cases",
    "selective_accuracy": "correct accepted / accepted",
    "penalized_accuracy": "correct accepted / all cases",
    "penalized_error": "(abstentions + accepted errors) / all cases",
    "macro_f1": "fixed CLASSES; zero_division=0",
    "ece": "top-label confidence; 10 bins; include confidence=1",
    "retrieval_only_ece_brier": "unavailable",
    "matched_coverage": "match native Cascade accepted count",
    "comparisons": "descriptive unless uncertainty analysis supports more"
}

FINAL_PROTOCOL["limitations"] = [
    "Previously examined source pool; not an independent external holdout",
    "Small rare-class validation and test support",
    "Conflicting-label dialogue retained within one group",
    "Legacy LengthNorm has clipped numerical penalties",
    "CRUS weights are fixed heuristic values",
    "Related-record screening is not exhaustive"
]

FINAL_PROTOCOL["validation_findings"] = {
    "matched_retained_count": int(MATCHED_K),
    "native_ranked_CRUS_same_selection": bool(same_selection),
    "legacy_clipped_documents": int(clipped_documents),
    "legacy_queries_with_clipping": int(queries_with_clipping)
}

print("Reporting rules and limitations added.")

In [ ]:
import importlib.metadata

packages = [
    "numpy", "pandas", "scipy", "scikit-learn",
    "sentence-transformers", "transformers", "rank-bm25", "torch"
]

FINAL_PROTOCOL["package_versions"] = {
    name: importlib.metadata.version(name)
    for name in packages
}

FINAL_PROTOCOL["model_file_hashes"] = {
    "lr": file_sha256(MODEL_OUTDIR / "trained_tfidf_lr.joblib"),
    "svm": file_sha256(
        MODEL_OUTDIR / "trained_group_calibrated_svm.joblib"
    )
}

if FINAL_PROTOCOL_PATH.exists():
    raise RuntimeError("Protocol already exists; do not overwrite.")

FINAL_PROTOCOL_PATH.write_text(
    json.dumps(FINAL_PROTOCOL, indent=2, sort_keys=True)
)

saved_protocol = json.loads(FINAL_PROTOCOL_PATH.read_text())
assert saved_protocol == FINAL_PROTOCOL
assert saved_protocol["frozen_policies"] == VALIDATION_POLICIES

FINAL_PROTOCOL_HASH = file_sha256(FINAL_PROTOCOL_PATH)
FINAL_PROTOCOL_APPROVED = False

print("Protocol saved and verified.")
print("Protocol SHA256:", FINAL_PROTOCOL_HASH)
print("Split sizes:", saved_protocol["split_sizes"])
print("Test evaluation remains blocked.")

In [ ]:
import json

saved_protocol = json.loads(FINAL_PROTOCOL_PATH.read_text())

# Normalize in-memory Python types to their JSON representation.
expected_protocol = json.loads(
    json.dumps(FINAL_PROTOCOL, sort_keys=True)
)

different_keys = [
    key
    for key in sorted(set(saved_protocol) | set(expected_protocol))
    if saved_protocol.get(key) != expected_protocol.get(key)
]

print("Differences after JSON normalization:", different_keys)

if different_keys:
    raise RuntimeError(
        "Substantive protocol differences remain. "
        "Stop and inspect them; do not overwrite the file."
    )

expected_policies = json.loads(
    json.dumps(VALIDATION_POLICIES)
)

assert saved_protocol["frozen_policies"] == expected_policies

FINAL_PROTOCOL_HASH = file_sha256(FINAL_PROTOCOL_PATH)
FINAL_PROTOCOL_APPROVED = False

print("Saved protocol verified.")
print("Protocol SHA256:", FINAL_PROTOCOL_HASH)
print("Split sizes:", saved_protocol["split_sizes"])
print("Test evaluation remains blocked.")

In [ ]:
import json

EXPECTED_PROTOCOL_HASH = (
    "ab7139125cf6b1a46c0ffaef838316fdda920a45f03c4979d4c8c5902bdeae1f"
)

assert file_sha256(FINAL_PROTOCOL_PATH) == EXPECTED_PROTOCOL_HASH

frozen = json.loads(FINAL_PROTOCOL_PATH.read_text())

def json_normalized(value):
    return json.loads(json.dumps(value))

assert json_normalized(BASELINE_CFG) == frozen["baseline_configuration"]
assert json_normalized(RETRIEVAL_CFG) == frozen["retrieval_configuration"]
assert json_normalized(UNCERTAINTY_CFG) == frozen["uncertainty_configuration"]
assert json_normalized(POLICY_CFG) == frozen["policy_selection_configuration"]
assert json_normalized(VALIDATION_POLICIES) == frozen["frozen_policies"]

assert file_sha256(main_manifest) == frozen["main_manifest_sha256"]
assert file_sha256(cal_manifest) == frozen["calibration_manifest_sha256"]

assert len(X_test) == frozen["split_sizes"]["test"]
assert len(X_train) == frozen["split_sizes"]["train"]
assert len(X_val) == frozen["split_sizes"]["validation"]

FINAL_PROTOCOL_APPROVED = True

print("Frozen protocol approved for evaluation.")
print("Test records:", len(X_test))
print("No test predictions generated by this cell.")

In [ ]:
import joblib
import numpy as np
from tqdm.auto import tqdm

assert FINAL_PROTOCOL_APPROVED

test_checkpoint = MODEL_OUTDIR / "frozen_test_retrieval_checkpoint.joblib"
temporary_checkpoint = MODEL_OUTDIR / "frozen_test_retrieval_checkpoint.tmp"

if test_checkpoint.exists():
    checkpoint = joblib.load(test_checkpoint)

    assert checkpoint["protocol_sha256"] == EXPECTED_PROTOCOL_HASH
    assert np.array_equal(checkpoint["record_ids"], ids_test)
    assert np.array_equal(checkpoint["true_labels"], y_test_np)
    assert checkpoint["classes"] == list(CLASSES)

    test_rows = checkpoint["results"]
    completed = checkpoint["completed"]

    assert all(
        len(test_rows[method]) == completed
        for method in RETRIEVAL_METHODS
    )
else:
    test_rows = {method: [] for method in RETRIEVAL_METHODS}
    completed = 0

def save_test_checkpoint(completed_count):
    payload = {
        "protocol_sha256": EXPECTED_PROTOCOL_HASH,
        "record_ids": ids_test,
        "true_labels": y_test_np,
        "classes": list(CLASSES),
        "completed": completed_count,
        "results": test_rows
    }

    joblib.dump(payload, temporary_checkpoint)
    temporary_checkpoint.replace(test_checkpoint)

print("Previously completed test queries:", completed)

for i in tqdm(
    range(completed, len(X_test)),
    total=len(X_test) - completed,
    desc="Frozen test retrieval/reranking"
):
    results = score_retrieval_methods(X_test[i])

    for method in RETRIEVAL_METHODS:
        test_rows[method].append(results[method])

    if (i + 1) % 10 == 0 or i + 1 == len(X_test):
        save_test_checkpoint(i + 1)

assert all(
    len(test_rows[method]) == len(X_test)
    for method in RETRIEVAL_METHODS
)

print("Test retrieval/reranking completed and saved.")
print("Thresholds and models were not changed.")

In [ ]:
supervised_checkpoint = (
    MODEL_OUTDIR / "frozen_test_supervised_checkpoint.joblib"
)

if supervised_checkpoint.exists():
    saved = joblib.load(supervised_checkpoint)

    assert saved["protocol_sha256"] == EXPECTED_PROTOCOL_HASH
    assert np.array_equal(saved["record_ids"], ids_test)
    assert np.array_equal(saved["true_labels"], y_test_np)

    TEST_BASELINES = saved["results"]

else:
    TEST_BASELINES = {}

    for name, model in [
        ("TFIDF_LR", lr_model),
        ("TFIDF_SVM", svm_model)
    ]:
        raw = model.predict_proba(X_test)
        probabilities = np.zeros((len(X_test), N_CLS))

        for column, label in enumerate(model.classes_):
            probabilities[:, CIDX[label]] = raw[:, column]

        assert np.isfinite(probabilities).all()
        assert np.allclose(probabilities.sum(axis=1), 1.0)

        predictions = np.array(CLASSES)[
            probabilities.argmax(axis=1)
        ]

        TEST_BASELINES[name] = {
            "pred": predictions,
            "probs": probabilities,
            "confidence": probabilities.max(axis=1),
            "ranking_score": probabilities.max(axis=1)
        }

    joblib.dump(
        {
            "protocol_sha256": EXPECTED_PROTOCOL_HASH,
            "record_ids": ids_test,
            "true_labels": y_test_np,
            "classes": list(CLASSES),
            "results": TEST_BASELINES
        },
        supervised_checkpoint
    )

print("Supervised test predictions saved.")
print("All test scoring is complete.")
print("No retraining or threshold reselection performed.")

In [ ]:
import numpy as np
import pandas as pd

assert file_sha256(FINAL_PROTOCOL_PATH) == EXPECTED_PROTOCOL_HASH

TEST_RESULTS = dict(TEST_BASELINES)

for method in RETRIEVAL_METHODS:
    rows = test_rows[method]
    assert len(rows) == len(X_test)

    probabilities = (
        None if rows[0]["probs"] is None
        else np.stack([r["probs"] for r in rows])
    )

    components = (
        None if rows[0]["components"] is None
        else np.stack([r["components"] for r in rows])
    )

    TEST_RESULTS[method] = {
        "pred": np.array([r["pred"] for r in rows]),
        "probs": probabilities,
        "confidence": np.array([r["confidence"] for r in rows]),
        "ranking_score": np.array([r["ranking_score"] for r in rows]),
        "components": components,
        "rows": rows
    }

audit_rows = []

for method, result in TEST_RESULTS.items():
    assert len(result["pred"]) == len(y_test_np)

    arrays = {
        "record_ids": ids_test,
        "true_labels": y_test_np,
        "classes": np.array(CLASSES),
        "predictions": result["pred"],
        "confidence": result["confidence"],
        "ranking_score": result["ranking_score"],
        "protocol_sha256": np.array(EXPECTED_PROTOCOL_HASH)
    }

    if result["probs"] is not None:
        arrays["probabilities"] = result["probs"]

    if result.get("components") is not None:
        arrays["components"] = result["components"]

    np.savez_compressed(
        MODEL_OUTDIR / f"test_{method}_arrays.npz",
        **arrays
    )

    for i, record_id in enumerate(ids_test):
        audit_rows.append({
            "record_id": record_id,
            "method": method,
            "true_label": y_test_np[i],
            "prediction": result["pred"][i],
            "correct": bool(result["pred"][i] == y_test_np[i]),
            "confidence": result["confidence"][i],
            "ranking_score": result["ranking_score"][i]
        })

pd.DataFrame(audit_rows).to_csv(
    MODEL_OUTDIR / "test_all_method_predictions.csv",
    index=False
)

print("Test methods assembled:", list(TEST_RESULTS))
print("Full-precision arrays and prediction audit saved.")

In [ ]:
from sklearn.metrics import f1_score, cohen_kappa_score

def calibration_metrics(true_labels, probabilities):
    if probabilities is None or len(true_labels) == 0:
        return np.nan, np.nan

    p = np.asarray(probabilities, dtype=float)
    assert p.shape == (len(true_labels), N_CLS)
    assert np.isfinite(p).all()
    assert (p >= 0).all()
    assert np.allclose(p.sum(axis=1), 1.0)

    true_indices = np.array([CIDX[c] for c in true_labels])
    confidence = p.max(axis=1)
    correct = p.argmax(axis=1) == true_indices
    bin_ids = np.minimum((confidence * 10).astype(int), 9)

    ece = 0.0
    for b in range(10):
        selected = bin_ids == b
        if selected.any():
            ece += selected.mean() * abs(
                correct[selected].mean() - confidence[selected].mean()
            )

    target = np.eye(N_CLS)[true_indices]
    brier = np.mean(np.sum((p - target) ** 2, axis=1))

    return float(ece), float(brier)


def summarize_test(method, result, accepted, mode):
    accepted = np.asarray(accepted, dtype=bool)
    correct = result["pred"] == y_test_np

    assert accepted.shape == correct.shape

    n = len(correct)
    n_accepted = int(accepted.sum())
    n_correct_accepted = int((correct & accepted).sum())
    n_errors_accepted = int((~correct & accepted).sum())
    n_abstained = n - n_accepted

    assert n_correct_accepted + n_errors_accepted == n_accepted

    selective_accuracy = (
        n_correct_accepted / n_accepted
        if n_accepted else np.nan
    )

    penalized_accuracy = n_correct_accepted / n
    penalized_error = (n_errors_accepted + n_abstained) / n

    assert np.isclose(penalized_accuracy + penalized_error, 1.0)

    if n_accepted:
        assert np.isclose(
            penalized_accuracy,
            (n_accepted / n) * selective_accuracy
        )

    yt = y_test_np[accepted]
    yp = result["pred"][accepted]
    p = (
        None if result["probs"] is None
        else result["probs"][accepted]
    )

    ece, brier = calibration_metrics(yt, p)

    return {
        "method": method,
        "mode": mode,
        "N": n,
        "n_accepted": n_accepted,
        "n_correct_accepted": n_correct_accepted,
        "n_errors_accepted": n_errors_accepted,
        "n_abstained": n_abstained,
        "coverage": n_accepted / n,
        "abstention_rate": n_abstained / n,
        "forced_accuracy": float(correct.mean()),
        "selective_accuracy": selective_accuracy,
        "accepted_misclassification_rate": 1 - selective_accuracy,
        "abstention_penalized_accuracy": penalized_accuracy,
        "abstention_penalized_error": penalized_error,
        "macro_f1": (
            f1_score(yt, yp, labels=CLASSES, average="macro",
                     zero_division=0) if n_accepted else np.nan
        ),
        "weighted_f1": (
            f1_score(yt, yp, labels=CLASSES, average="weighted",
                     zero_division=0) if n_accepted else np.nan
        ),
        "kappa": (
            cohen_kappa_score(yt, yp, labels=CLASSES)
            if n_accepted else np.nan
        ),
        "ece": ece,
        "brier": brier
    }

print("Count-based and calibration metric functions defined.")

In [ ]:
FROZEN_POLICIES = json.loads(
    FINAL_PROTOCOL_PATH.read_text()
)["frozen_policies"]

TEST_NATIVE_MASKS = {}
main_rows = []
acceptance_audit = []

for method, result in TEST_RESULTS.items():
    main_rows.append(
        summarize_test(
            method,
            result,
            np.ones(len(y_test_np), dtype=bool),
            "forced_before_abstention"
        )
    )

for method in DOCUMENT_METHODS:
    result = TEST_RESULTS[method]
    policy = FROZEN_POLICIES[f"{method}:CRUS"]
    uncertainty = uncertainty_signals(result)["CRUS"]

    accepted = (
        (result["confidence"] >= policy["tau_c"])
        & (uncertainty <= policy["tau_u"])
    )

    TEST_NATIVE_MASKS[method] = accepted

    main_rows.append(
        summarize_test(
            method, result, accepted, "native_CRUS_gate"
        )
    )

    for i, record_id in enumerate(ids_test):
        acceptance_audit.append({
            "record_id": record_id,
            "method": method,
            "CRUS": uncertainty[i],
            "confidence": result["confidence"][i],
            "tau_c": policy["tau_c"],
            "tau_u": policy["tau_u"],
            "accepted": bool(accepted[i]),
            "correct": bool(result["pred"][i] == y_test_np[i])
        })

MAIN_TEST_RESULTS = pd.DataFrame(main_rows)

MAIN_TEST_RESULTS.to_csv(
    MODEL_OUTDIR / "verified_main_test_results.csv",
    index=False
)

pd.DataFrame(acceptance_audit).to_csv(
    MODEL_OUTDIR / "test_frozen_CRUS_acceptance_audit.csv",
    index=False
)

display_columns = [
    "method", "mode", "N", "n_accepted",
    "n_correct_accepted", "n_abstained",
    "coverage", "forced_accuracy", "selective_accuracy",
    "abstention_penalized_accuracy", "abstention_penalized_error"
]

print(MAIN_TEST_RESULTS[display_columns].to_string(index=False))
print("\nResults saved. No settings or thresholds changed.")

In [ ]:
MATCHED_COUNTS = sorted({
    int(mask.sum())
    for mask in TEST_NATIVE_MASKS.values()
})

matched_rows = []
matched_audit = []

# Resolve tied ranking scores by record ID, not by correctness.
record_id_keys = np.asarray(ids_test).astype(str)

for method, result in TEST_RESULTS.items():
    scores = np.asarray(result["ranking_score"], dtype=float)

    assert scores.shape == (len(y_test_np),)
    assert np.isfinite(scores).all(), (
        f"{method}: ranking scores contain non-finite values"
    )

    # Descending score; ascending record ID within score ties.
    order = np.lexsort((record_id_keys, -scores))

    for k in MATCHED_COUNTS:
        assert 0 < k <= len(y_test_np)

        accepted = np.zeros(len(y_test_np), dtype=bool)
        accepted[order[:k]] = True

        summary = summarize_test(
            method,
            result,
            accepted,
            f"matched_top_{k}_by_ranking_score"
        )
        summary["target_n_accepted"] = k
        summary["selection_rule"] = (
            "ranking_score_descending; record_id_ascending_for_ties"
        )

        matched_rows.append(summary)

        for i, record_id in enumerate(ids_test):
            matched_audit.append({
                "record_id": record_id,
                "method": method,
                "target_n_accepted": k,
                "ranking_score": scores[i],
                "accepted": bool(accepted[i]),
                "correct": bool(result["pred"][i] == y_test_np[i])
            })

MATCHED_TEST_RESULTS = pd.DataFrame(matched_rows)

MATCHED_TEST_RESULTS.to_csv(
    MODEL_OUTDIR / "test_matched_count_results.csv",
    index=False
)

pd.DataFrame(matched_audit).to_csv(
    MODEL_OUTDIR / "test_matched_count_acceptance_audit.csv",
    index=False
)

print("Matched accepted counts:", MATCHED_COUNTS)

# Display every method at Cascade's native accepted count.
cascade_k = int(TEST_NATIVE_MASKS["Cascade"].sum())

comparison = MATCHED_TEST_RESULTS[
    MATCHED_TEST_RESULTS["target_n_accepted"] == cascade_k
].copy()

comparison = pd.concat(
    [
        comparison,
        MAIN_TEST_RESULTS[
            MAIN_TEST_RESULTS["mode"] == "native_CRUS_gate"
        ].loc[
            lambda frame: frame["n_accepted"] == cascade_k
        ]
    ],
    ignore_index=True
)

columns = [
    "method",
    "mode",
    "n_accepted",
    "n_correct_accepted",
    "n_errors_accepted",
    "coverage",
    "selective_accuracy",
    "abstention_penalized_accuracy"
]

print(
    comparison[columns]
    .sort_values(
        ["n_correct_accepted", "method"],
        ascending=[False, True]
    )
    .to_string(index=False)
)

print("\nAll four matched-count comparisons saved.")
print("Frozen native acceptance masks remain unchanged.")

In [ ]:
curve_rows = []
curve_summary_rows = []

n = len(y_test_np)
accepted_counts = np.arange(1, n + 1)

for method, result in TEST_RESULTS.items():
    scores = np.asarray(result["ranking_score"], dtype=float)
    correct = np.asarray(result["pred"] == y_test_np, dtype=bool)

    assert scores.shape == (n,)
    assert np.isfinite(scores).all()

    order = np.lexsort((record_id_keys, -scores))

    cumulative_correct = np.cumsum(correct[order])
    cumulative_errors = accepted_counts - cumulative_correct

    coverage = accepted_counts / n
    selective_accuracy = cumulative_correct / accepted_counts
    risk = cumulative_errors / accepted_counts

    for j in range(n):
        curve_rows.append({
            "method": method,
            "n_accepted": int(accepted_counts[j]),
            "n_correct_accepted": int(cumulative_correct[j]),
            "n_errors_accepted": int(cumulative_errors[j]),
            "coverage": float(coverage[j]),
            "selective_accuracy": float(selective_accuracy[j]),
            "risk": float(risk[j]),
            "abstention_penalized_accuracy": float(
                cumulative_correct[j] / n
            ),
            "selection_rule": (
                "ranking_score_descending;"
                " record_id_ascending_for_ties"
            )
        })

    # Explicit definition: average prefix risk across k = 1,...,N.
    curve_summary_rows.append({
        "method": method,
        "mean_prefix_risk": float(risk.mean()),
        "summary_definition": (
            "arithmetic mean of error_rate_at_top_k"
            " for k=1,...,N"
        ),
        "forced_accuracy": float(correct.mean()),
        "protocol_sha256": EXPECTED_PROTOCOL_HASH
    })

RISK_COVERAGE_RESULTS = pd.DataFrame(curve_rows)
RISK_COVERAGE_SUMMARY = pd.DataFrame(curve_summary_rows)

RISK_COVERAGE_RESULTS.to_csv(
    MODEL_OUTDIR / "test_ranking_risk_coverage_curves.csv",
    index=False
)

RISK_COVERAGE_SUMMARY.to_csv(
    MODEL_OUTDIR / "test_ranking_risk_coverage_summary.csv",
    index=False
)

native_points = MAIN_TEST_RESULTS[
    MAIN_TEST_RESULTS["mode"] == "native_CRUS_gate"
].copy()

native_points["risk"] = (
    native_points["accepted_misclassification_rate"]
)

native_points.to_csv(
    MODEL_OUTDIR / "test_native_gate_operating_points.csv",
    index=False
)

# Verify curve counts against the earlier matched-count results.
check = MATCHED_TEST_RESULTS[
    ["method", "target_n_accepted", "n_correct_accepted"]
].merge(
    RISK_COVERAGE_RESULTS[
        ["method", "n_accepted", "n_correct_accepted"]
    ],
    left_on=["method", "target_n_accepted"],
    right_on=["method", "n_accepted"],
    suffixes=("_matched", "_curve"),
    validate="one_to_one"
)

assert len(check) == len(MATCHED_TEST_RESULTS)
assert (
    check["n_correct_accepted_matched"]
    == check["n_correct_accepted_curve"]
).all()

print(
    RISK_COVERAGE_SUMMARY[
        ["method", "mean_prefix_risk", "forced_accuracy"]
    ]
    .sort_values("mean_prefix_risk")
    .to_string(index=False)
)

print("\nRanking curves and native gate points saved separately.")
print("Matched-count consistency checks passed.")

In [ ]:
print("Frozen policy keys:")
for key in sorted(FROZEN_POLICIES):
    print(key, FROZEN_POLICIES[key])

print("\nAvailable Cascade uncertainty signals:")
signals = uncertainty_signals(TEST_RESULTS["Cascade"])

for name, values in signals.items():
    values = np.asarray(values)
    print(
        name,
        "shape:", values.shape,
        "finite:", bool(np.isfinite(values).all())
    )

In [ ]:
assert file_sha256(FINAL_PROTOCOL_PATH) == EXPECTED_PROTOCOL_HASH

cascade_result = TEST_RESULTS["Cascade"]
cascade_signals = uncertainty_signals(cascade_result)

CASCADE_POLICY_MASKS = {}
policy_rows = []
policy_audit = []

for signal_name, signal_values in cascade_signals.items():
    policy_key = f"Cascade:{signal_name}"
    assert policy_key in FROZEN_POLICIES, policy_key

    policy = FROZEN_POLICIES[policy_key]
    uncertainty = np.asarray(signal_values, dtype=float)
    confidence = np.asarray(cascade_result["confidence"], dtype=float)

    assert uncertainty.shape == (len(y_test_np),)
    assert np.isfinite(uncertainty).all()
    assert np.isfinite(confidence).all()

    accepted = (
        (confidence >= policy["tau_c"])
        & (uncertainty <= policy["tau_u"])
    )

    CASCADE_POLICY_MASKS[signal_name] = accepted

    summary = summarize_test(
        "Cascade",
        cascade_result,
        accepted,
        f"frozen_{signal_name}_gate"
    )

    summary.update({
        "uncertainty_signal": signal_name,
        "tau_c": policy["tau_c"],
        "tau_u": policy["tau_u"],
        "validation_coverage": policy["coverage"],
        "validation_selective_accuracy": policy["selective_accuracy"]
    })

    policy_rows.append(summary)

    for i, record_id in enumerate(ids_test):
        policy_audit.append({
            "record_id": record_id,
            "uncertainty_signal": signal_name,
            "uncertainty": uncertainty[i],
            "confidence": confidence[i],
            "tau_c": policy["tau_c"],
            "tau_u": policy["tau_u"],
            "accepted": bool(accepted[i]),
            "correct": bool(
                cascade_result["pred"][i] == y_test_np[i]
            )
        })

assert np.array_equal(
    CASCADE_POLICY_MASKS["CRUS"],
    TEST_NATIVE_MASKS["Cascade"]
)

CASCADE_FROZEN_POLICY_RESULTS = pd.DataFrame(policy_rows)

CASCADE_FROZEN_POLICY_RESULTS.to_csv(
    MODEL_OUTDIR / "test_cascade_frozen_uncertainty_policies.csv",
    index=False
)

pd.DataFrame(policy_audit).to_csv(
    MODEL_OUTDIR / "test_cascade_uncertainty_policy_audit.csv",
    index=False
)

columns = [
    "uncertainty_signal",
    "n_accepted",
    "n_correct_accepted",
    "n_errors_accepted",
    "coverage",
    "selective_accuracy",
    "abstention_penalized_accuracy",
    "tau_c",
    "tau_u"
]

print(
    CASCADE_FROZEN_POLICY_RESULTS[columns]
    .to_string(index=False)
)

print("\nFrozen CRUS mask matches the earlier result.")
print("All policies applied without threshold changes.")

In [ ]:
k = int(TEST_NATIVE_MASKS["Cascade"].sum())

uncertainty_rank_rows = []
uncertainty_rank_audit = []

for signal_name, signal_values in cascade_signals.items():
    uncertainty = np.asarray(signal_values, dtype=float)

    # Lower uncertainty first; record ID resolves ties.
    order = np.lexsort((record_id_keys, uncertainty))

    accepted = np.zeros(len(y_test_np), dtype=bool)
    accepted[order[:k]] = True

    summary = summarize_test(
        "Cascade",
        cascade_result,
        accepted,
        f"lowest_{signal_name}_top_{k}"
    )
    summary["uncertainty_signal"] = signal_name
    uncertainty_rank_rows.append(summary)

    for i, record_id in enumerate(ids_test):
        uncertainty_rank_audit.append({
            "record_id": record_id,
            "uncertainty_signal": signal_name,
            "uncertainty": uncertainty[i],
            "target_n_accepted": k,
            "accepted": bool(accepted[i]),
            "correct": bool(
                cascade_result["pred"][i] == y_test_np[i]
            )
        })

CASCADE_UNCERTAINTY_RANK_RESULTS = pd.DataFrame(
    uncertainty_rank_rows
)

CASCADE_UNCERTAINTY_RANK_RESULTS.to_csv(
    MODEL_OUTDIR / "test_cascade_matched_uncertainty_rankings.csv",
    index=False
)

pd.DataFrame(uncertainty_rank_audit).to_csv(
    MODEL_OUTDIR / "test_cascade_matched_uncertainty_audit.csv",
    index=False
)

columns = [
    "uncertainty_signal",
    "n_accepted",
    "n_correct_accepted",
    "n_errors_accepted",
    "coverage",
    "selective_accuracy",
    "abstention_penalized_accuracy"
]

print(
    CASCADE_UNCERTAINTY_RANK_RESULTS[columns]
    .sort_values(
        ["n_correct_accepted", "uncertainty_signal"],
        ascending=[False, True]
    )
    .to_string(index=False)
)

print("\nMatched uncertainty rankings saved.")
print("Frozen deployment policies remain unchanged.")

In [ ]:
# Verify whether lowest-CRUS ranking reproduces the native CRUS gate.
k = int(TEST_NATIVE_MASKS["Cascade"].sum())

crus_values = np.asarray(cascade_signals["CRUS"], dtype=float)
crus_order = np.lexsort((record_id_keys, crus_values))

crus_rank_mask = np.zeros(len(y_test_np), dtype=bool)
crus_rank_mask[crus_order[:k]] = True

native_crus_mask = TEST_NATIVE_MASKS["Cascade"]

print(
    "CRUS ranking and native gate masks identical:",
    bool(np.array_equal(crus_rank_mask, native_crus_mask))
)

print(
    "Cases accepted by only one of the two selections:",
    int(np.count_nonzero(crus_rank_mask != native_crus_mask))
)

# Show the existing frozen-policy results.
policy_columns = [
    "uncertainty_signal",
    "n_accepted",
    "n_correct_accepted",
    "n_errors_accepted",
    "coverage",
    "selective_accuracy",
    "abstention_penalized_accuracy",
    "tau_c",
    "tau_u"
]

print("\nFrozen validation-selected policies on test:")
print(
    CASCADE_FROZEN_POLICY_RESULTS[policy_columns]
    .to_string(index=False)
)

print("\nNo predictions, thresholds, or policies changed.")

In [ ]:
import json

assert file_sha256(FINAL_PROTOCOL_PATH) == EXPECTED_PROTOCOL_HASH

frozen_protocol = json.loads(FINAL_PROTOCOL_PATH.read_text())

print("Frozen protocol top-level keys:")
print(list(frozen_protocol.keys()))

print("\nFrozen uncertainty configuration:")
print(json.dumps(
    frozen_protocol["uncertainty_configuration"],
    indent=2
))

print("\nOther analysis-related protocol fields:")
analysis_terms = (
    "bootstrap", "interval", "resampl", "statistic",
    "analysis", "evaluation", "seed"
)

for key, value in frozen_protocol.items():
    if (
        key != "uncertainty_configuration"
        and any(term in key.lower() for term in analysis_terms)
    ):
        print(f"\n{key}:")
        print(json.dumps(value, indent=2))

print("\nProtocol remains unchanged.")

In [ ]:
assert file_sha256(FINAL_PROTOCOL_PATH) == EXPECTED_PROTOCOL_HASH

for key in [
    "reporting_rules",
    "limitations",
    "primary_method",
    "primary_policy",
    "ranking_ties"
]:
    print(f"\n{key}:")
    print(json.dumps(
        frozen_protocol.get(key),
        indent=2
    ))

print("\nProtocol remains unchanged.")

In [ ]:
import json

assert file_sha256(main_manifest) == (
    frozen_protocol["main_manifest_sha256"]
)

manifest_data = json.loads(main_manifest.read_text())

def show_structure(value, path="root", depth=0):
    if depth > 3:
        return

    if isinstance(value, dict):
        print(f"{path}: dict keys = {list(value.keys())[:30]}")
        for key, item in value.items():
            if isinstance(item, (dict, list)):
                show_structure(item, f"{path}.{key}", depth + 1)

    elif isinstance(value, list):
        print(f"{path}: list length = {len(value)}")
        if value:
            item = value[0]
            print(f"{path}[0]: type = {type(item).__name__}")
            if isinstance(item, (dict, list)):
                show_structure(item, f"{path}[0]", depth + 1)

show_structure(manifest_data)

print("\nManifest inspected; no files changed.")

In [ ]:
from pathlib import Path
import pandas as pd

manifest_path = Path(main_manifest)

assert file_sha256(manifest_path) == (
    frozen_protocol["main_manifest_sha256"]
)

print("Manifest filename:", manifest_path.name)
print("File extension:", manifest_path.suffix.lower())
print("File size in bytes:", manifest_path.stat().st_size)

suffix = manifest_path.suffix.lower()

if suffix in {".csv", ".tsv"}:
    separator = "\t" if suffix == ".tsv" else ","

    manifest_preview = pd.read_csv(
        manifest_path,
        sep=separator,
        nrows=5
    )

    print("\nColumn names:")
    print(manifest_preview.columns.tolist())

    group_candidates = [
        column
        for column in manifest_preview.columns
        if any(
            term in str(column).lower()
            for term in ["group", "cluster", "dialogue", "conversation"]
        )
    ]

    print("\nPossible grouping columns:", group_candidates)

else:
    with manifest_path.open("rb") as handle:
        prefix = handle.read(16)

    print("\nFirst 16 bytes, hexadecimal:", prefix.hex())
    print("Format needs identification before loading.")

print("\nManifest hash verified; no files changed.")

In [ ]:
import numpy as np
import pandas as pd

assert file_sha256(main_manifest) == (
    frozen_protocol["main_manifest_sha256"]
)

split_manifest = pd.read_csv(
    main_manifest,
    dtype={
        "record_id": "string",
        "group_id": "string",
        "dialogue_hash": "string",
        "split": "string"
    }
)

required_columns = {"record_id", "group_id", "label", "split"}
assert required_columns.issubset(split_manifest.columns)
assert split_manifest["record_id"].notna().all()
assert split_manifest["group_id"].notna().all()
assert split_manifest["record_id"].is_unique, (
    "Manifest record IDs are not unique; stop and inspect."
)

test_manifest = split_manifest.loc[
    split_manifest["split"].str.strip().str.lower() == "test"
].copy()

prediction_ids = pd.Index(
    np.asarray(ids_test).astype(str),
    name="record_id"
)

assert prediction_ids.is_unique
assert len(test_manifest) == len(prediction_ids)
assert set(test_manifest["record_id"]) == set(prediction_ids), (
    "Manifest test IDs and scored test IDs differ."
)

# Align manifest rows to the exact prediction-array order.
aligned_test_manifest = (
    test_manifest
    .set_index("record_id")
    .loc[prediction_ids]
    .reset_index()
)

assert np.array_equal(
    aligned_test_manifest["record_id"].to_numpy(dtype=str),
    prediction_ids.to_numpy(dtype=str)
)

assert np.array_equal(
    aligned_test_manifest["label"].to_numpy(dtype=str),
    np.asarray(y_test_np).astype(str)
), "Manifest labels and scored test labels differ."

TEST_GROUP_IDS = aligned_test_manifest[
    "group_id"
].to_numpy(dtype=str)

group_sizes = (
    aligned_test_manifest.groupby("group_id")
    .size()
    .rename("n_test_records")
)

# Check whether any group spans multiple split assignments.
normalized_splits = split_manifest["split"].str.strip().str.lower()

group_split_counts = (
    split_manifest.assign(normalized_split=normalized_splits)
    .groupby("group_id")["normalized_split"]
    .nunique()
)

n_cross_split_groups = int((group_split_counts > 1).sum())

print("Aligned test records:", len(TEST_GROUP_IDS))
print("Unique test groups:", len(group_sizes))
print("Groups with multiple test records:", int((group_sizes > 1).sum()))
print("Largest test group:", int(group_sizes.max()))
print("Groups appearing in multiple splits:", n_cross_split_groups)

print("\nTest group-size distribution:")
print(
    group_sizes.value_counts()
    .sort_index()
    .rename_axis("records_per_group")
    .rename("number_of_groups")
    .to_string()
)

assert n_cross_split_groups == 0, (
    "Some groups span multiple splits. Stop before uncertainty analysis."
)

print("\nTest group mapping and labels verified.")
print("No files, predictions, or policies changed.")

In [ ]:
import json
import numpy as np
import pandas as pd

assert file_sha256(FINAL_PROTOCOL_PATH) == EXPECTED_PROTOCOL_HASH

SUPPLEMENTARY_BOOTSTRAP_SPEC = {
    "analysis_status": "supplementary; specified after test inspection",
    "protocol_sha256": EXPECTED_PROTOCOL_HASH,
    "manifest_sha256": file_sha256(main_manifest),
    "scope": "six frozen Cascade uncertainty policies",
    "resampling_unit": "manifest group_id",
    "resampling_scheme": (
        "sample G groups with replacement; retain all records "
        "within each sampled group; no class stratification"
    ),
    "paired": True,
    "n_replicates": 10000,
    "seed": 20261008,
    "interval": "95% percentile; quantiles 0.025 and 0.975",
    "weighting": "record-weighted metrics",
    "predictions": "fixed",
    "acceptance_masks": "fixed frozen-policy decisions",
    "threshold_reselection": False,
    "multiple_comparison_adjustment": "none; exploratory",
    "metrics": [
        "coverage",
        "selective_accuracy",
        "abstention_penalized_accuracy"
    ],
    "paired_differences": "CRUS minus each alternative frozen policy",
    "n_test_records": int(len(y_test_np)),
    "n_test_groups": int(len(np.unique(TEST_GROUP_IDS)))
}

supplementary_spec_path = (
    MODEL_OUTDIR / "supplementary_group_bootstrap_spec.json"
)

if supplementary_spec_path.exists():
    existing_spec = json.loads(supplementary_spec_path.read_text())
    assert existing_spec == SUPPLEMENTARY_BOOTSTRAP_SPEC, (
        "Existing supplementary specification differs. "
        "Stop rather than overwrite it."
    )
else:
    supplementary_spec_path.write_text(
        json.dumps(SUPPLEMENTARY_BOOTSTRAP_SPEC, indent=2),
        encoding="utf-8"
    )

print("Supplementary bootstrap specification saved or verified.")
print("Original frozen protocol unchanged.")

In [ ]:
policy_names = sorted(CASCADE_POLICY_MASKS)
metric_names = SUPPLEMENTARY_BOOTSTRAP_SPEC["metrics"]

group_names, group_index = np.unique(
    TEST_GROUP_IDS,
    return_inverse=True
)

G = len(group_names)
P = len(policy_names)
B = SUPPLEMENTARY_BOOTSTRAP_SPEC["n_replicates"]

correct = np.asarray(
    TEST_RESULTS["Cascade"]["pred"] == y_test_np,
    dtype=bool
)

group_record_counts = np.bincount(
    group_index, minlength=G
).astype(float)

group_accepted_counts = np.zeros((G, P))
group_correct_accepted_counts = np.zeros((G, P))

for j, policy_name in enumerate(policy_names):
    accepted = np.asarray(
        CASCADE_POLICY_MASKS[policy_name], dtype=bool
    )
    assert accepted.shape == correct.shape

    group_accepted_counts[:, j] = np.bincount(
        group_index,
        weights=accepted.astype(float),
        minlength=G
    )

    group_correct_accepted_counts[:, j] = np.bincount(
        group_index,
        weights=(accepted & correct).astype(float),
        minlength=G
    )

def metrics_from_counts(n_records, n_accepted, n_correct):
    selective_accuracy = np.divide(
        n_correct,
        n_accepted,
        out=np.full_like(n_correct, np.nan, dtype=float),
        where=n_accepted > 0
    )

    return np.stack(
        [
            n_accepted / n_records,
            selective_accuracy,
            n_correct / n_records
        ],
        axis=-1
    )

point_estimates = metrics_from_counts(
    group_record_counts.sum(),
    group_accepted_counts.sum(axis=0),
    group_correct_accepted_counts.sum(axis=0)
)

rng = np.random.default_rng(
    SUPPLEMENTARY_BOOTSTRAP_SPEC["seed"]
)

bootstrap_metrics = np.empty((B, P, len(metric_names)))

for b in range(B):
    sampled_groups = rng.integers(0, G, size=G)
    multiplicities = np.bincount(
        sampled_groups, minlength=G
    ).astype(float)

    n_records = multiplicities @ group_record_counts
    n_accepted = multiplicities @ group_accepted_counts
    n_correct = multiplicities @ group_correct_accepted_counts

    bootstrap_metrics[b] = metrics_from_counts(
        n_records, n_accepted, n_correct
    )

interval_rows = []

for j, policy_name in enumerate(policy_names):
    for m, metric_name in enumerate(metric_names):
        values = bootstrap_metrics[:, j, m]
        finite_values = values[np.isfinite(values)]

        assert len(finite_values) > 0

        lower, upper = np.quantile(
            finite_values, [0.025, 0.975]
        )

        interval_rows.append({
            "policy": policy_name,
            "metric": metric_name,
            "estimate": point_estimates[j, m],
            "ci_lower": lower,
            "ci_upper": upper,
            "valid_replicates": len(finite_values),
            "analysis_status": "supplementary"
        })

crus_column = policy_names.index("CRUS")
difference_rows = []

for j, policy_name in enumerate(policy_names):
    if policy_name == "CRUS":
        continue

    for m, metric_name in enumerate(metric_names):
        differences = (
            bootstrap_metrics[:, crus_column, m]
            - bootstrap_metrics[:, j, m]
        )
        finite_differences = differences[
            np.isfinite(differences)
        ]

        assert len(finite_differences) > 0

        lower, upper = np.quantile(
            finite_differences, [0.025, 0.975]
        )

        difference_rows.append({
            "comparison": f"CRUS minus {policy_name}",
            "metric": metric_name,
            "estimate_difference": (
                point_estimates[crus_column, m]
                - point_estimates[j, m]
            ),
            "ci_lower": lower,
            "ci_upper": upper,
            "valid_replicates": len(finite_differences),
            "analysis_status": "supplementary; exploratory"
        })

FROZEN_POLICY_BOOTSTRAP_INTERVALS = pd.DataFrame(interval_rows)
FROZEN_POLICY_BOOTSTRAP_DIFFERENCES = pd.DataFrame(difference_rows)

FROZEN_POLICY_BOOTSTRAP_INTERVALS.to_csv(
    MODEL_OUTDIR / "supplementary_frozen_policy_intervals.csv",
    index=False
)

FROZEN_POLICY_BOOTSTRAP_DIFFERENCES.to_csv(
    MODEL_OUTDIR / "supplementary_frozen_policy_paired_differences.csv",
    index=False
)

np.savez_compressed(
    MODEL_OUTDIR / "supplementary_frozen_policy_bootstrap_arrays.npz",
    bootstrap_metrics=bootstrap_metrics,
    point_estimates=point_estimates,
    policy_names=np.array(policy_names),
    metric_names=np.array(metric_names),
    protocol_sha256=np.array(EXPECTED_PROTOCOL_HASH)
)

print("Frozen-policy selective-accuracy intervals:")
print(
    FROZEN_POLICY_BOOTSTRAP_INTERVALS.loc[
        lambda df: df["metric"] == "selective_accuracy"
    ].to_string(index=False)
)

print("\nPaired selective-accuracy differences:")
print(
    FROZEN_POLICY_BOOTSTRAP_DIFFERENCES.loc[
        lambda df: df["metric"] == "selective_accuracy"
    ].to_string(index=False)
)

assert file_sha256(FINAL_PROTOCOL_PATH) == EXPECTED_PROTOCOL_HASH

print("\nAll interval tables and bootstrap arrays saved.")
print("Frozen protocol, predictions, and policies unchanged.")

**Editor Comment 2**

In [ ]:
# -- RELEASE WORKSPACE SUPPORT FOR LATER AUDIT CELLS -----------------------
# Run the package-path setup before this section.
# Explicit opt-in is required before copying deposited artifacts.
# The canonical notebook is copied for source inspection, not as proof
# that the modified source originally generated the archived predictions.

from pathlib import Path, PurePosixPath
import csv
import hashlib
import os
import shutil

_crus_workspace_ready = False

def _crus_file_hash(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def _crus_workspace_base():
    base = (RUN_OUTPUT_ROOT / "saved_artifact_audits").resolve()
    protected = [
        ARCHIVE_ROOT.resolve(),
        RAW_DATA_ROOT.resolve(),
        (PACKAGE_ROOT / "notebooks").resolve(),
    ]
    for directory in protected:
        if base == directory or directory in base.parents:
            raise RuntimeError("Unsafe saved-artifact workspace location.")
    return base

def _crus_prepare_saved_workspace():
    global _crus_workspace_ready
    if _crus_workspace_ready:
        return

    if os.environ.get("CRUS_ALLOW_ARTIFACT_COPY") != "1":
        raise RuntimeError(
            "Saved-artifact audits require a separate working copy. "
            "Read the release instructions and explicitly set "
            "CRUS_ALLOW_ARTIFACT_COPY=1 before starting the kernel. "
            "No archived artifacts have been copied by this call."
        )

    base = _crus_workspace_base()
    if base.exists() and any(base.iterdir()):
        raise RuntimeError(
            "The audit workspace is not empty. Use a fresh run-output "
            "directory rather than reusing potentially modified audit inputs."
        )

    entries = {}
    inventories = [
        (PACKAGE_ROOT / "manifest" / "artifact_manifest.csv", "package_path"),
        (PACKAGE_ROOT / "REVISION_FILE_INVENTORY.csv", "relative_path"),
    ]
    for inventory, path_field in inventories:
        if not inventory.is_file():
            continue
        with inventory.open("r", encoding="utf-8-sig", newline="") as handle:
            reader = csv.DictReader(handle)
            if not {path_field, "sha256"}.issubset(reader.fieldnames or []):
                raise RuntimeError(f"Unexpected inventory columns: {inventory}")
            for row in reader:
                text = (row.get(path_field) or "").replace("\\", "/")
                relative = PurePosixPath(text)
                if not relative.parts or relative.parts[0] != "artifacts":
                    continue
                if relative.is_absolute() or ".." in relative.parts:
                    raise RuntimeError(f"Unsafe inventory path: {text}")
                digest = (row.get("sha256") or "").strip().lower()
                if len(digest) != 64 or any(
                    c not in "0123456789abcdef" for c in digest
                ):
                    raise RuntimeError(f"Invalid inventory checksum: {text}")
                key = relative.as_posix()
                if key in entries and entries[key] != digest:
                    raise RuntimeError(f"Inventories disagree: {key}")
                entries[key] = digest

    if not entries:
        raise RuntimeError("No inventoried artifact files were identified.")

    archive = ARCHIVE_ROOT.resolve()
    verified = []
    for text, expected in sorted(entries.items()):
        relative = PurePosixPath(text)
        origin = PACKAGE_ROOT.joinpath(*relative.parts).resolve()
        if archive not in origin.parents or not origin.is_file():
            raise RuntimeError(f"Missing or unsafe archived file: {text}")
        if _crus_file_hash(origin) != expected:
            raise RuntimeError(f"Archived checksum mismatch: {text}")
        verified.append((relative, origin, expected))

    canonical = PACKAGE_ROOT / "notebooks" / (
        "revised_risk_aware_hybrid_pipeline_v2.ipynb"
    )
    if not canonical.is_file():
        raise FileNotFoundError(canonical)

    project = base / "MyDrive" / "RiskAware_Revision2"
    notebooks = base / "MyDrive" / "Colab Notebooks"
    notebooks.mkdir(parents=True, exist_ok=True)
    (base / "Shareddrives").mkdir(exist_ok=True)
    (base / "_local_runtime").mkdir(exist_ok=True)

    total_bytes = sum(origin.stat().st_size for _, origin, _ in verified)
    print(f"Preparing {len(verified)} archived files ({total_bytes:,} bytes).")
    print("Working-copy destination:", project)

    for index, (relative, origin, expected) in enumerate(verified, start=1):
        destination = project.joinpath(*relative.parts[1:])
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(origin, destination)
        if _crus_file_hash(destination) != expected:
            raise RuntimeError(f"Working-copy checksum mismatch: {relative}")
        if index % 25 == 0:
            print(f"Copied and checked {index}/{len(verified)} files.")

    shutil.copy2(canonical, notebooks / canonical.name)
    external_notebook = PACKAGE_ROOT / "notebooks" / "CRUS_R1_5_External_Transfer.ipynb"
    if not external_notebook.is_file():
        raise FileNotFoundError(external_notebook)
    shutil.copy2(external_notebook, notebooks / external_notebook.name)
    _crus_workspace_ready = True
    print("Audit working copy ready. Deposited artifacts were not modified.")

def _crus_local_path(kind, relative=""):
    _crus_prepare_saved_workspace()
    base = _crus_workspace_base()
    locations = {
        "mount": base,
        "drive": base / "MyDrive",
        "project": base / "MyDrive" / "RiskAware_Revision2",
        "notebooks": base / "MyDrive" / "Colab Notebooks",
        "runtime": base / "_local_runtime",
    }
    if kind not in locations:
        raise ValueError(f"Unknown workspace path category: {kind}")
    relative_path = PurePosixPath(relative)
    if relative_path.is_absolute() or ".." in relative_path.parts:
        raise ValueError("Unsafe relative workspace path.")
    return str(locations[kind].joinpath(*relative_path.parts))

def _crus_show_file(path):
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(path)
    print("Saved file available locally:", path)


def _crus_upload_local_files():
    """Read only explicitly configured local inspection files."""
    import json
    configured = json.loads(
        os.environ.get("CRUS_INSPECTION_FILES", "[]")
    )
    if (
        not isinstance(configured, list)
        or not configured
        or not all(isinstance(item, str) for item in configured)
    ):
        raise RuntimeError(
            "No local inspection files selected. Set INPUT_PATHS in the "
            "inspection cell, or set CRUS_INSPECTION_FILES to a JSON list "
            "of local filenames before starting the kernel. "
            "No browser upload or automatic download is performed."
        )
    selected = {}
    for item in configured:
        path = Path(item).expanduser().resolve()
        if not path.is_file():
            raise FileNotFoundError(path)
        if path.name in selected:
            raise ValueError("Selected inspection files have duplicate basenames.")
        selected[path.name] = path.read_bytes()
    return selected

# -- ORIGINAL AUDIT CELL CONTINUES BELOW -----------------------------------

from pathlib import Path
import hashlib
import json
import os

EXPECTED_PROTOCOL_HASH = (
    "ab7139125cf6b1a46c0ffaef838316fdda920a45f03c4979d4c8c5902bdeae1f"
)

drive_root = Path(_crus_local_path('drive', ''))

if not drive_root.is_dir():
    pass  # Colab service not required locally.
    _crus_prepare_saved_workspace()

assert drive_root.is_dir(), "Google Drive is not available."

# Optional: narrow the search if you know the project directory.
# Example:
# SEARCH_ROOT = drive_root / "your_actual_project_folder"
SEARCH_ROOT = drive_root

matches = []
candidates_checked = 0
read_errors = 0

print("Searching for the verified frozen protocol...")
print("This may take a few minutes if Drive contains many folders.")

for directory, subdirectories, filenames in os.walk(SEARCH_ROOT):
    subdirectories[:] = [
        name for name in subdirectories
        if name not in {".git", "__pycache__", ".ipynb_checkpoints"}
    ]

    for filename in filenames:
        # Search likely protocol/configuration JSON files.
        lower_name = filename.lower()

        if not lower_name.endswith(".json"):
            continue

        if not any(
            word in lower_name
            for word in ["protocol", "config", "frozen"]
        ):
            continue

        candidate = Path(directory) / filename

        try:
            contents = candidate.read_bytes()
        except OSError:
            read_errors += 1
            continue

        candidates_checked += 1

        if hashlib.sha256(contents).hexdigest() == EXPECTED_PROTOCOL_HASH:
            matches.append(candidate)

matches = sorted(set(matches), key=str)

print("Candidate JSON files checked:", candidates_checked)
print("Files that could not be read:", read_errors)
print("Exact hash matches:", len(matches))

if not matches:
    raise RuntimeError(
        "The verified protocol was not found under SEARCH_ROOT. "
        "Do not regenerate it. Locate the existing protocol file "
        "in Drive and provide its actual path."
    )

if len(matches) > 1:
    print("\nMatching copies:")
    for candidate in matches:
        print(candidate)

    raise RuntimeError(
        "Multiple matching copies found. Choose the copy in the "
        "original evaluation output directory before continuing."
    )

FINAL_PROTOCOL_PATH = matches[0]
recovered_protocol = json.loads(
    FINAL_PROTOCOL_PATH.read_text(encoding="utf-8")
)

print("\nVerified protocol found:")
print(FINAL_PROTOCOL_PATH)

print("\nInput declarations:")
for key in [
    "model_input",
    "header_usage",
    "reference_section_text_usage",
    "split_sizes",
]:
    print(f"\n{key}:")
    print(json.dumps(
        recovered_protocol.get(key, "<FIELD NOT PRESENT>"),
        indent=2
    ))

# Infer the evaluation output directory only if prediction artifacts
# are present beside the protocol.
candidate_output_directory = FINAL_PROTOCOL_PATH.parent

artifact_markers = [
    "verified_main_test_results.csv",
    "test_all_method_predictions.csv",
    "frozen_test_supervised_checkpoint.joblib",
    "frozen_test_retrieval_checkpoint.joblib",
]

present_artifacts = [
    name for name in artifact_markers
    if (candidate_output_directory / name).exists()
]

print("\nEvaluation artifacts beside the protocol:")
print(present_artifacts)

if present_artifacts:
    MODEL_OUTDIR = candidate_output_directory
    print("\nMODEL_OUTDIR restored:", MODEL_OUTDIR)
else:
    print(
        "\nMODEL_OUTDIR was not inferred. The protocol directory "
        "does not contain the expected prediction artifacts."
    )

print("\nNo files modified.")
print("No model fitting or test scoring performed.")

In [ ]:
from pathlib import Path
import os

drive_root = Path(_crus_local_path('drive', ''))
assert drive_root.is_dir(), "Google Drive is not mounted."

source_candidates = []

for directory, subdirectories, filenames in os.walk(drive_root):
    subdirectories[:] = [
        name for name in subdirectories
        if name not in {
            ".git",
            "__pycache__",
            ".ipynb_checkpoints"
        }
    ]

    for filename in filenames:
        if Path(filename).suffix.lower() in {".ipynb", ".py"}:
            source_candidates.append(
                Path(directory) / filename
            )

source_candidates = sorted(set(source_candidates), key=str)

keywords = (
    "risk", "correct", "revision", "hybrid",
    "crus", "pipeline", "sk4", "untitled"
)

likely_candidates = [
    path for path in source_candidates
    if any(
        keyword in str(path.relative_to(drive_root)).lower()
        for keyword in keywords
    )
]

print("Total notebooks/scripts found:", len(source_candidates))
print("Likely candidates:", len(likely_candidates))

candidates_to_display = (
    likely_candidates if likely_candidates else source_candidates
)

for number, path in enumerate(candidates_to_display, start=1):
    print(
        f"{number}. "
        f"{path.relative_to(drive_root)}"
    )

print("\nNo code executed and no files modified.")

In [ ]:
import json
from pathlib import Path

drive_root = Path(_crus_local_path('drive', ''))

# Reuse the candidates discovered by the previous search.
assert "source_candidates" in globals(), (
    "Run the Drive notebook-search cell first."
)

markers = {
    "corrected output directory": "corrected_experiment_outputs",
    "revision project directory": "RiskAware_Revision2",
    "frozen protocol filename": "final_pretest_protocol.json",
    "verified protocol hash": (
        "ab7139125cf6b1a46c0ffaef838316fdda920a45f03c4979d4c8c5902bdeae1f"
    ),
    "corrected results export": "verified_main_test_results.csv",
    "corrected retrieval checkpoint": (
        "frozen_test_retrieval_checkpoint.joblib"
    ),
}

notebook_matches = []

for candidate in source_candidates:
    candidate = Path(candidate)

    if candidate.suffix.lower() != ".ipynb":
        continue

    try:
        notebook = json.loads(
            candidate.read_text(encoding="utf-8")
        )
    except (OSError, UnicodeError, json.JSONDecodeError) as exc:
        print(
            "Could not inspect:",
            candidate.name,
            type(exc).__name__
        )
        continue

    code_cells = []

    for cell_number, cell in enumerate(
        notebook.get("cells", []),
        start=1
    ):
        if cell.get("cell_type") != "code":
            continue

        source = cell.get("source", [])
        source = (
            source if isinstance(source, str)
            else "".join(source)
        )

        code_cells.append((cell_number, source))

    hits = {}

    for marker_name, marker_text in markers.items():
        matching_cells = [
            cell_number
            for cell_number, source in code_cells
            if marker_text in source
        ]

        if matching_cells:
            hits[marker_name] = matching_cells

    if hits:
        notebook_matches.append({
            "path": candidate,
            "hits": hits,
            "marker_count": len(hits),
        })

notebook_matches.sort(
    key=lambda item: (
        -item["marker_count"],
        str(item["path"])
    )
)

print("Notebooks containing corrected-run markers:",
      len(notebook_matches))

for number, item in enumerate(notebook_matches, start=1):
    print(
        f"\n{number}. "
        f"{item['path'].relative_to(drive_root)}"
    )

    for marker_name, cells in item["hits"].items():
        print(f"   {marker_name}: cells {cells}")

print(
    "\nMarker matches identify candidates, not proof that "
    "their saved code produced the predictions."
)
print("No notebook cells executed and no files modified.")

In [ ]:
import ast
import hashlib
import json
import re
from pathlib import Path

notebook_path = Path(
    _crus_local_path('notebooks', 'revised_risk_aware_hybrid_pipeline_v2.ipynb')
)

assert notebook_path.is_file(), "Saved notebook not found."

protocol_path = Path(
    _crus_local_path('project', 'corrected_experiment_outputs/final_pretest_protocol.json')
)

expected_protocol_hash = (
    "ab7139125cf6b1a46c0ffaef838316fdda920a45f03c4979d4c8c5902bdeae1f"
)

assert hashlib.sha256(
    protocol_path.read_bytes()
).hexdigest() == expected_protocol_hash

notebook_bytes = notebook_path.read_bytes()
notebook = json.loads(notebook_bytes)

# These are 1-based saved-notebook cell numbers from the marker search.
START_CELL = 47
FREEZE_CELL = 83

pattern = re.compile(
    r"dialogue|section[_ ]?(?:header|text)|"
    r"model_input|model_text|header_usage|reference_section|"
    r"\bX_(?:train|val|test)\b|"
    r"\b(?:train|val|test|corpus|document)_texts\b|"
    r"\b(?:fit|fit_transform|transform|encode|predict_proba)\s*\(|"
    r"score_retrieval_methods|label_string|"
    r"normalize|normalise|clean_text",
    flags=re.IGNORECASE,
)

report_parts = [
    "COMMENT 2 — SAVED SOURCE INSPECTION",
    f"Notebook: {notebook_path.name}",
    f"Notebook SHA256: {hashlib.sha256(notebook_bytes).hexdigest()}",
    f"Frozen protocol SHA256: {expected_protocol_hash}",
    "",
    "This report contains saved code only, not notebook outputs.",
    "No source cells were executed.",
    "Historical definitions may have been superseded; cell order matters.",
    "",
]

display_parts = []
matched_cells = []
helper_count = 0

for cell_number, cell in enumerate(
    notebook.get("cells", []),
    start=1
):
    if cell.get("cell_type") != "code":
        continue

    source = cell.get("source", [])
    source = source if isinstance(source, str) else "".join(source)

    if not source.strip():
        continue

    if START_CELL <= cell_number <= FREEZE_CELL:
        if not pattern.search(source):
            continue

        matched_cells.append(cell_number)

        heading = f"\n--- SAVED CELL {cell_number} ---\n"
        report_parts.extend([heading, source])

        # Display relevant lines with surrounding context.
        lines = source.splitlines()
        selected_lines = set()

        for line_number, line in enumerate(lines):
            if pattern.search(line):
                selected_lines.update(
                    range(
                        max(0, line_number - 3),
                        min(len(lines), line_number + 4)
                    )
                )

        excerpts = []
        previous = None

        for line_number in sorted(selected_lines):
            if previous is not None and line_number > previous + 1:
                excerpts.append("    ...")

            excerpts.append(
                f"{line_number + 1:4d}: {lines[line_number]}"
            )
            previous = line_number

        display_parts.append(
            heading + "\n".join(excerpts)
        )

    elif cell_number < START_CELL:
        # Preserve potentially relevant earlier helper definitions.
        try:
            tree = ast.parse(source)
        except SyntaxError:
            continue

        for node in tree.body:
            if not isinstance(
                node, (ast.FunctionDef, ast.AsyncFunctionDef)
            ):
                continue

            function_source = ast.get_source_segment(source, node)

            if function_source and pattern.search(function_source):
                helper_count += 1
                heading = (
                    f"\n--- EARLIER HELPER: {node.name}, "
                    f"CELL {cell_number} ---\n"
                )

                report_parts.extend([heading, function_source])

# Append protocol declarations separately from code evidence.
protocol = json.loads(protocol_path.read_text(encoding="utf-8"))

report_parts.append("\n--- FROZEN INPUT DECLARATIONS ---\n")
for key in [
    "model_input",
    "header_usage",
    "reference_section_text_usage",
    "baseline_configuration",
    "retrieval_configuration",
]:
    report_parts.append(
        f"{key}:\n"
        + json.dumps(protocol.get(key), indent=2)
    )

report_text = "\n".join(report_parts)

output_directory = protocol_path.parent

# Preserve earlier reports if the saved notebook has changed.
notebook_digest = hashlib.sha256(notebook_bytes).hexdigest()[:12]

report_path = output_directory / (
    f"comment2_saved_source_audit_{notebook_digest}.txt"
)

if report_path.exists():
    assert report_path.read_text(encoding="utf-8") == report_text, (
        "Existing audit differs; stop rather than overwrite it."
    )
else:
    report_path.write_text(report_text, encoding="utf-8")

print("Matched corrected-run cells:", matched_cells)
print("Earlier relevant helper definitions retained:", helper_count)

display_text = "\n".join(display_parts)
DISPLAY_LIMIT = 24000

print("\nINPUT-RELATED SOURCE EXCERPTS:\n")
print(display_text[:DISPLAY_LIMIT])

if len(display_text) > DISPLAY_LIMIT:
    print(
        "\n[Display truncated. Full selected cell sources and "
        "earlier helper definitions are in the saved audit report.]"
    )

print("\nAudit report:", report_path)
print("Report size:", report_path.stat().st_size, "bytes")

assert hashlib.sha256(
    protocol_path.read_bytes()
).hexdigest() == expected_protocol_hash

print("\nFrozen protocol unchanged.")
print("No training, prediction, or notebook source execution performed.")

In [ ]:
import json
from pathlib import Path

notebook_path = Path(
    _crus_local_path('notebooks', 'revised_risk_aware_hybrid_pipeline_v2.ipynb')
)

notebook = json.loads(
    notebook_path.read_text(encoding="utf-8")
)

# Method helpers near the retrieval/reranking definitions,
# plus the saved retrieval and supervised test-scoring cells.
cells_to_inspect = [70, 71, 73, 74, 75, 91, 92]

for cell_number in cells_to_inspect:
    cell_index = cell_number - 1

    if cell_index >= len(notebook.get("cells", [])):
        print(f"\nCell {cell_number}: not present")
        continue

    cell = notebook["cells"][cell_index]

    if cell.get("cell_type") != "code":
        print(f"\nCell {cell_number}: not a code cell")
        continue

    source = cell.get("source", [])
    source = source if isinstance(source, str) else "".join(source)

    print("\n" + "=" * 78)
    print(f"SAVED CELL {cell_number}")
    print("=" * 78)
    print(source)

print("\nSaved source displayed only.")
print("No notebook cells executed and no files modified.")

In [ ]:
from pathlib import Path
import hashlib
import json

protocol_path = Path(
    _crus_local_path('project', 'corrected_experiment_outputs/final_pretest_protocol.json')
)

expected_hash = (
    "ab7139125cf6b1a46c0ffaef838316fdda920a45f03c4979d4c8c5902bdeae1f"
)

assert protocol_path.is_file(), (
    "Protocol file not found. Mount Google Drive first."
)

protocol_bytes = protocol_path.read_bytes()

assert hashlib.sha256(protocol_bytes).hexdigest() == expected_hash, (
    "Protocol hash differs. Stop before using its configuration."
)

protocol = json.loads(protocol_bytes)
retrieval_configuration = protocol["retrieval_configuration"]

assert "doc_truncate_chars" in retrieval_configuration, (
    "Truncation setting is missing from the saved configuration."
)

character_limit = retrieval_configuration["doc_truncate_chars"]

print("Frozen doc_truncate_chars:", character_limit)

if character_limit == 400:
    print(
        "Manuscript wording: "
        "'the first 400 characters of each retrieved training dialogue'"
    )
else:
    print(
        "Manuscript wording:",
        f"'the first {character_limit} characters "
        "of each retrieved training dialogue'"
    )

print("Protocol hash verified. No files changed.")

In [ ]:
from pathlib import Path
import hashlib
import json

import numpy as np
import pandas as pd

OUTPUT_DIR = Path(
    _crus_local_path('project', 'corrected_experiment_outputs')
)

PROTOCOL_PATH = OUTPUT_DIR / "final_pretest_protocol.json"
PREDICTIONS_PATH = OUTPUT_DIR / "test_all_method_predictions.csv"
GATE_AUDIT_PATH = OUTPUT_DIR / "test_frozen_CRUS_acceptance_audit.csv"
SUMMARY_PATH = OUTPUT_DIR / "verified_main_test_results.csv"

MANIFEST_PATH = Path(
    _crus_local_path('project', 'data_audit/constrained_split_manifest.csv')
)

EXPECTED_PROTOCOL_HASH = (
    "ab7139125cf6b1a46c0ffaef838316fdda920a45f03c4979d4c8c5902bdeae1f"
)

required_paths = [
    PROTOCOL_PATH,
    PREDICTIONS_PATH,
    GATE_AUDIT_PATH,
    SUMMARY_PATH,
    MANIFEST_PATH,
]

missing_paths = [
    str(path) for path in required_paths if not path.is_file()
]

if missing_paths:
    raise RuntimeError(
        "Required saved files are missing:\n"
        + "\n".join(missing_paths)
        + "\nMount Google Drive or locate the existing files. "
        "Do not regenerate predictions."
    )

def sha256_file(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

assert sha256_file(PROTOCOL_PATH) == EXPECTED_PROTOCOL_HASH

protocol = json.loads(
    PROTOCOL_PATH.read_text(encoding="utf-8")
)

assert sha256_file(MANIFEST_PATH) == (
    protocol["main_manifest_sha256"]
), "Split manifest differs from the frozen protocol."

# Load labels and IDs as strings to preserve their representation.
manifest = pd.read_csv(
    MANIFEST_PATH,
    dtype={
        "record_id": "string",
        "label": "string",
        "split": "string",
    },
)

test_manifest = manifest.loc[
    manifest["split"].str.strip().str.lower() == "test"
].copy()

assert test_manifest["record_id"].is_unique

N = len(test_manifest)

assert N == protocol["split_sizes"]["test"]
assert N == 257

test_ids = set(test_manifest["record_id"])

true_label_by_id = test_manifest.set_index(
    "record_id"
)["label"]

predictions = pd.read_csv(
    PREDICTIONS_PATH,
    dtype={
        "record_id": "string",
        "method": "string",
        "true_label": "string",
        "prediction": "string",
    },
)

required_prediction_columns = {
    "record_id",
    "method",
    "true_label",
    "prediction",
}

assert required_prediction_columns.issubset(predictions.columns)
assert predictions[list(required_prediction_columns)].notna().all().all()
assert not predictions.duplicated(["method", "record_id"]).any()

EXPECTED_METHODS = {
    "TFIDF_LR",
    "TFIDF_SVM",
    "BM25",
    "SBERT",
    "SK4",
    "SingleStageFull",
    "SingleStageTruncated",
    "Cascade",
    "LengthNormLegacy",
}

assert set(predictions["method"]) == EXPECTED_METHODS

def count_metrics(method, mode, correct, accepted):
    correct = np.asarray(correct, dtype=bool)
    accepted = np.asarray(accepted, dtype=bool)

    assert correct.shape == accepted.shape == (N,)

    n_accepted = int(accepted.sum())
    n_correct_accepted = int((accepted & correct).sum())
    n_errors_accepted = int((accepted & ~correct).sum())
    n_abstained = N - n_accepted
    n_correct_forced = int(correct.sum())

    selective_accuracy = (
        n_correct_accepted / n_accepted
        if n_accepted else np.nan
    )

    accepted_error_rate = (
        n_errors_accepted / n_accepted
        if n_accepted else np.nan
    )

    coverage = n_accepted / N
    penalized_accuracy = n_correct_accepted / N
    penalized_error = (n_errors_accepted + n_abstained) / N

    assert n_correct_accepted + n_errors_accepted == n_accepted
    assert n_accepted + n_abstained == N
    assert (
        n_correct_accepted + n_errors_accepted + n_abstained
    ) == N

    assert np.isclose(
        penalized_accuracy + penalized_error, 1.0
    )

    if n_accepted:
        assert np.isclose(
            selective_accuracy + accepted_error_rate, 1.0
        )
        assert np.isclose(
            penalized_accuracy,
            coverage * selective_accuracy,
        )

    return {
        "method": method,
        "mode": mode,
        "N": N,
        "n_correct_forced": n_correct_forced,
        "forced_accuracy": n_correct_forced / N,
        "forced_error": (N - n_correct_forced) / N,
        "n_accepted": n_accepted,
        "n_correct_accepted": n_correct_accepted,
        "n_errors_accepted": n_errors_accepted,
        "n_abstained": n_abstained,
        "coverage": coverage,
        "abstention_rate": n_abstained / N,
        "selective_accuracy": selective_accuracy,
        "accepted_misclassification_rate": accepted_error_rate,
        "abstention_penalized_accuracy": penalized_accuracy,
        "abstention_penalized_error": penalized_error,
    }

# Recalculate correctness from labels, not the stored "correct" flag.
prediction_groups = {}
results = []

for method, group in predictions.groupby("method", sort=True):
    group = group.set_index("record_id").sort_index()

    assert len(group) == N
    assert set(group.index) == test_ids

    expected_labels = true_label_by_id.reindex(group.index)

    assert np.array_equal(
        group["true_label"].to_numpy(dtype=str),
        expected_labels.to_numpy(dtype=str),
    ), f"{method}: labels differ from the frozen test manifest."

    correct = (
        group["prediction"].to_numpy(dtype=str)
        == expected_labels.to_numpy(dtype=str)
    )

    prediction_groups[method] = (group, correct)

    results.append(
        count_metrics(
            method,
            "forced_before_abstention",
            correct,
            np.ones(N, dtype=bool),
        )
    )

# Verify saved CRUS acceptance decisions against frozen thresholds.
gate_audit = pd.read_csv(
    GATE_AUDIT_PATH,
    dtype={
        "record_id": "string",
        "method": "string",
        "accepted": "string",
    },
)

required_gate_columns = {
    "record_id", "method", "accepted",
    "confidence", "CRUS", "tau_c", "tau_u",
}

assert required_gate_columns.issubset(gate_audit.columns)
assert not gate_audit.duplicated(["method", "record_id"]).any()

DOCUMENT_METHODS = {
    "SingleStageFull",
    "SingleStageTruncated",
    "Cascade",
    "LengthNormLegacy",
}

assert set(gate_audit["method"]) == DOCUMENT_METHODS

def parse_boolean_column(series):
    normalized = series.str.strip().str.lower()
    allowed = {"true": True, "false": False, "1": True, "0": False}

    assert normalized.notna().all()
    assert normalized.isin(allowed).all(), (
        "Unrecognized values in saved acceptance decisions."
    )

    return normalized.map(allowed).to_numpy(dtype=bool)

for method, group in gate_audit.groupby("method", sort=True):
    prediction_group, correct = prediction_groups[method]

    group = (
        group.set_index("record_id")
        .reindex(prediction_group.index)
    )

    assert len(group) == N
    assert group[list(required_gate_columns - {"record_id"})].notna().all().all()

    original_method_ids = set(
        gate_audit.loc[
            gate_audit["method"] == method, "record_id"
        ]
    )
    assert original_method_ids == test_ids

    policy = protocol["frozen_policies"][f"{method}:CRUS"]

    for field in ["confidence", "CRUS", "tau_c", "tau_u"]:
        group[field] = pd.to_numeric(group[field], errors="raise")
        assert np.isfinite(group[field].to_numpy()).all()

    assert np.allclose(
        group["tau_c"], policy["tau_c"], rtol=0, atol=1e-12
    )
    assert np.allclose(
        group["tau_u"], policy["tau_u"], rtol=0, atol=1e-12
    )

    saved_accepted = parse_boolean_column(group["accepted"])

    expected_accepted = (
        (group["confidence"].to_numpy() >= policy["tau_c"])
        & (group["CRUS"].to_numpy() <= policy["tau_u"])
    )

    assert np.array_equal(saved_accepted, expected_accepted), (
        f"{method}: saved decisions do not match the frozen gate."
    )

    results.append(
        count_metrics(
            method,
            "native_CRUS_gate",
            correct,
            saved_accepted,
        )
    )

COMMENT3_RESULTS = pd.DataFrame(results)

# Compare independent recalculation with the existing results summary.
existing_summary = pd.read_csv(SUMMARY_PATH)

comparison_fields = [
    "N",
    "n_accepted",
    "n_correct_accepted",
    "n_errors_accepted",
    "n_abstained",
    "coverage",
    "forced_accuracy",
    "selective_accuracy",
    "abstention_penalized_accuracy",
    "abstention_penalized_error",
]

assert not existing_summary.duplicated(["method", "mode"]).any()

comparison = COMMENT3_RESULTS.merge(
    existing_summary[["method", "mode"] + comparison_fields],
    on=["method", "mode"],
    suffixes=("_recalculated", "_saved"),
    validate="one_to_one",
)

assert len(comparison) == len(COMMENT3_RESULTS)

for field in comparison_fields:
    assert np.allclose(
        comparison[f"{field}_recalculated"],
        comparison[f"{field}_saved"],
        rtol=0,
        atol=1e-12,
        equal_nan=True,
    ), f"Recalculated and saved summaries disagree for {field}."

# Save a separate audit; preserve the original prediction/summary files.
audit_path = OUTPUT_DIR / "comment3_verified_count_metrics.csv"
audit_text = COMMENT3_RESULTS.to_csv(index=False)

if audit_path.exists():
    assert audit_path.read_text(encoding="utf-8") == audit_text, (
        "An existing Comment 3 audit differs. Stop rather than overwrite."
    )
else:
    audit_path.write_text(audit_text, encoding="utf-8")

display_columns = [
    "method",
    "mode",
    "N",
    "n_correct_forced",
    "forced_accuracy",
    "n_accepted",
    "n_correct_accepted",
    "n_errors_accepted",
    "n_abstained",
    "coverage",
    "selective_accuracy",
    "accepted_misclassification_rate",
    "abstention_penalized_accuracy",
    "abstention_penalized_error",
]

print("COMMENT 3: INDEPENDENT COUNT-BASED VERIFICATION\n")
print(
    COMMENT3_RESULTS[display_columns].to_string(
        index=False,
        float_format=lambda value: f"{value:.6f}",
    )
)

cascade_native = COMMENT3_RESULTS.loc[
    (COMMENT3_RESULTS["method"] == "Cascade")
    & (COMMENT3_RESULTS["mode"] == "native_CRUS_gate")
].iloc[0]

print("\nPRIMARY CASCADE POLICY — MANUSCRIPT COUNTS")
print(
    f"Forced correct: {int(cascade_native['n_correct_forced'])}/{N}"
)
print(
    f"Accepted: {int(cascade_native['n_accepted'])}/{N}"
)
print(
    "Correct accepted:",
    int(cascade_native["n_correct_accepted"]),
)
print(
    "Accepted errors:",
    int(cascade_native["n_errors_accepted"]),
)
print("Abstentions:", int(cascade_native["n_abstained"]))

assert sha256_file(PROTOCOL_PATH) == EXPECTED_PROTOCOL_HASH

print("\nAll count identities passed.")
print("All frozen gate decisions verified.")
print("All recalculated metrics match the saved main summary.")
print("Audit saved:", audit_path.name)
print("No models trained and no new predictions generated.")
print("Original protocol, predictions, and summary files unchanged.")

In [ ]:
from pathlib import Path

drive_root = Path(_crus_local_path('drive', ''))

if not drive_root.is_dir():
    pass  # Colab service not required locally.
    _crus_prepare_saved_workspace()

print("MyDrive available:", drive_root.is_dir())

project_dir = drive_root / "RiskAware_Revision2"
output_dir = project_dir / "corrected_experiment_outputs"
audit_dir = project_dir / "data_audit"

required_files = {
    "Frozen protocol":
        output_dir / "final_pretest_protocol.json",

    "Case-level predictions":
        output_dir / "test_all_method_predictions.csv",

    "Frozen acceptance audit":
        output_dir / "test_frozen_CRUS_acceptance_audit.csv",

    "Main results summary":
        output_dir / "verified_main_test_results.csv",

    "Split manifest":
        audit_dir / "constrained_split_manifest.csv",
}

print("\nRequired files:")

for description, path in required_files.items():
    print(
        f"{'FOUND' if path.is_file() else 'MISSING'}"
        f" | {description}"
    )

missing_files = [
    path for path in required_files.values()
    if not path.is_file()
]

if not missing_files:
    print(
        "\nAll files are available. "
        "Rerun the Comment 3 verification cell."
    )
else:
    print("\nProject directory exists:", project_dir.is_dir())

    for directory in [output_dir, audit_dir]:
        print(f"\nDirectory: {directory}")

        if directory.is_dir():
            print("Available CSV/JSON filenames:")
            for path in sorted(directory.iterdir()):
                if path.is_file() and path.suffix.lower() in {".csv", ".json"}:
                    print(" -", path.name)
        else:
            print("Directory not available.")

    print(
        "\nDo not rerun the experiment. "
        "Paste this output so we can resolve the saved-file locations."
    )

print("\nNo files modified.")

In [ ]:
from pathlib import Path
pass  # Colab service not required locally.

NEW_MOUNT = _crus_local_path('mount', '')

_crus_prepare_saved_workspace()

DRIVE_ROOT = Path(NEW_MOUNT) / "MyDrive"

assert DRIVE_ROOT.is_dir(), "MyDrive is not accessible."

PROJECT_DIR = DRIVE_ROOT / "RiskAware_Revision2"
OUTPUT_DIR = PROJECT_DIR / "corrected_experiment_outputs"
MANIFEST_PATH = (
    PROJECT_DIR / "data_audit/constrained_split_manifest.csv"
)

required_files = [
    OUTPUT_DIR / "final_pretest_protocol.json",
    OUTPUT_DIR / "test_all_method_predictions.csv",
    OUTPUT_DIR / "test_frozen_CRUS_acceptance_audit.csv",
    OUTPUT_DIR / "verified_main_test_results.csv",
    MANIFEST_PATH,
]

print("\nSaved-file check:")

for path in required_files:
    print(
        f"{'FOUND' if path.is_file() else 'MISSING'}"
        f" | {path.name}"
    )

assert all(path.is_file() for path in required_files), (
    "Some files remain unavailable. "
    "Do not regenerate predictions."
)

print("\nAll required files found.")
print("No experiment files modified.")

In [ ]:
from pathlib import Path
import hashlib
import json

import numpy as np
import pandas as pd


# ============================================================
# 1. Saved-file locations
# ============================================================

PROJECT_DIR = Path(
    _crus_local_path('project', '')
)

OUTPUT_DIR = PROJECT_DIR / "corrected_experiment_outputs"

PROTOCOL_PATH = OUTPUT_DIR / "final_pretest_protocol.json"
PREDICTIONS_PATH = OUTPUT_DIR / "test_all_method_predictions.csv"
GATE_AUDIT_PATH = (
    OUTPUT_DIR / "test_frozen_CRUS_acceptance_audit.csv"
)
SUMMARY_PATH = OUTPUT_DIR / "verified_main_test_results.csv"

MANIFEST_PATH = (
    PROJECT_DIR / "data_audit/constrained_split_manifest.csv"
)

EXPECTED_PROTOCOL_HASH = (
    "ab7139125cf6b1a46c0ffaef838316fdda920a45f03c4979d4c8c5902bdeae1f"
)

required_paths = [
    PROTOCOL_PATH,
    PREDICTIONS_PATH,
    GATE_AUDIT_PATH,
    SUMMARY_PATH,
    MANIFEST_PATH,
]

missing_paths = [
    str(path)
    for path in required_paths
    if not path.is_file()
]

if missing_paths:
    raise RuntimeError(
        "Required saved files are unavailable:\n"
        + "\n".join(missing_paths)
        + "\nDo not regenerate predictions."
    )


# ============================================================
# 2. Verify the frozen protocol and split manifest
# ============================================================

def sha256_file(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()


# Preserve hashes of all source files for the final check.
original_file_hashes = {
    path: sha256_file(path)
    for path in required_paths
}

assert original_file_hashes[PROTOCOL_PATH] == (
    EXPECTED_PROTOCOL_HASH
), "Frozen protocol hash differs."

protocol = json.loads(
    PROTOCOL_PATH.read_text(encoding="utf-8")
)

assert original_file_hashes[MANIFEST_PATH] == (
    protocol["main_manifest_sha256"]
), "Split manifest differs from the frozen protocol."

manifest = pd.read_csv(
    MANIFEST_PATH,
    dtype={
        "record_id": "string",
        "label": "string",
        "split": "string",
    },
)

assert {"record_id", "label", "split"}.issubset(
    manifest.columns
)

assert manifest[["record_id", "label", "split"]].notna().all().all()
assert manifest["record_id"].is_unique

test_manifest = manifest.loc[
    manifest["split"].str.strip().str.lower() == "test"
].copy()

N = len(test_manifest)

assert N == protocol["split_sizes"]["test"]
assert N == 257

test_ids = set(test_manifest["record_id"])

true_label_by_id = test_manifest.set_index(
    "record_id"
)["label"]

allowed_labels = set(str(label) for label in protocol["classes"])

assert set(test_manifest["label"]).issubset(allowed_labels)

print("Frozen protocol verified.")
print("Split manifest verified.")
print("Test records:", N)


# ============================================================
# 3. Load saved case-level predictions
# ============================================================

predictions = pd.read_csv(
    PREDICTIONS_PATH,
    dtype={
        "record_id": "string",
        "method": "string",
        "true_label": "string",
        "prediction": "string",
    },
)

required_prediction_columns = {
    "record_id",
    "method",
    "true_label",
    "prediction",
}

assert required_prediction_columns.issubset(
    predictions.columns
)

assert predictions[
    list(required_prediction_columns)
].notna().all().all()

assert not predictions.duplicated(
    ["method", "record_id"]
).any()

assert set(predictions["prediction"]).issubset(allowed_labels)

EXPECTED_METHODS = {
    "TFIDF_LR",
    "TFIDF_SVM",
    "BM25",
    "SBERT",
    "SK4",
    "SingleStageFull",
    "SingleStageTruncated",
    "Cascade",
    "LengthNormLegacy",
}

assert set(predictions["method"]) == EXPECTED_METHODS


# ============================================================
# 4. Count-based metric definitions
# ============================================================

def count_metrics(method, mode, correct, accepted):
    correct = np.asarray(correct, dtype=bool)
    accepted = np.asarray(accepted, dtype=bool)

    assert correct.shape == accepted.shape == (N,)

    n_correct_forced = int(correct.sum())
    n_accepted = int(accepted.sum())
    n_correct_accepted = int((accepted & correct).sum())
    n_errors_accepted = int((accepted & ~correct).sum())
    n_abstained = N - n_accepted

    coverage = n_accepted / N

    selective_accuracy = (
        n_correct_accepted / n_accepted
        if n_accepted else np.nan
    )

    accepted_error_rate = (
        n_errors_accepted / n_accepted
        if n_accepted else np.nan
    )

    penalized_accuracy = n_correct_accepted / N

    penalized_error = (
        n_errors_accepted + n_abstained
    ) / N

    # Verify the count identities.
    assert n_correct_accepted + n_errors_accepted == n_accepted
    assert n_accepted + n_abstained == N

    assert (
        n_correct_accepted
        + n_errors_accepted
        + n_abstained
    ) == N

    assert np.isclose(
        penalized_accuracy + penalized_error,
        1.0,
    )

    if n_accepted:
        assert np.isclose(
            selective_accuracy + accepted_error_rate,
            1.0,
        )

        assert np.isclose(
            penalized_accuracy,
            coverage * selective_accuracy,
        )

    return {
        "method": method,
        "mode": mode,
        "N": N,
        "n_correct_forced": n_correct_forced,
        "forced_accuracy": n_correct_forced / N,
        "forced_error": (N - n_correct_forced) / N,
        "n_accepted": n_accepted,
        "n_correct_accepted": n_correct_accepted,
        "n_errors_accepted": n_errors_accepted,
        "n_abstained": n_abstained,
        "coverage": coverage,
        "abstention_rate": n_abstained / N,
        "selective_accuracy": selective_accuracy,
        "accepted_misclassification_rate": accepted_error_rate,
        "abstention_penalized_accuracy": penalized_accuracy,
        "abstention_penalized_error": penalized_error,
    }


# ============================================================
# 5. Recalculate forced metrics for all nine methods
# ============================================================

prediction_groups = {}
result_rows = []

for method, group in predictions.groupby("method", sort=True):
    group = group.set_index("record_id").sort_index()

    assert len(group) == N
    assert set(group.index) == test_ids

    expected_labels = true_label_by_id.reindex(group.index)

    assert expected_labels.notna().all()

    assert np.array_equal(
        group["true_label"].to_numpy(dtype=str),
        expected_labels.to_numpy(dtype=str),
    ), f"{method}: labels differ from the frozen test manifest."

    # Recalculate correctness from predictions and manifest labels.
    # Do not rely on the saved "correct" column.
    correct = (
        group["prediction"].to_numpy(dtype=str)
        == expected_labels.to_numpy(dtype=str)
    )

    prediction_groups[method] = (group, correct)

    result_rows.append(
        count_metrics(
            method=method,
            mode="forced_before_abstention",
            correct=correct,
            accepted=np.ones(N, dtype=bool),
        )
    )


# ============================================================
# 6. Verify native CRUS gates and recalculate selective metrics
# ============================================================

gate_audit = pd.read_csv(
    GATE_AUDIT_PATH,
    dtype={
        "record_id": "string",
        "method": "string",
        "accepted": "string",
    },
)

required_gate_columns = {
    "record_id",
    "method",
    "accepted",
    "confidence",
    "CRUS",
    "tau_c",
    "tau_u",
}

assert required_gate_columns.issubset(gate_audit.columns)

assert gate_audit[
    list(required_gate_columns)
].notna().all().all()

assert not gate_audit.duplicated(
    ["method", "record_id"]
).any()

DOCUMENT_METHODS = {
    "SingleStageFull",
    "SingleStageTruncated",
    "Cascade",
    "LengthNormLegacy",
}

assert set(gate_audit["method"]) == DOCUMENT_METHODS


def parse_boolean_column(series):
    normalized = series.str.strip().str.lower()

    mapping = {
        "true": True,
        "false": False,
        "1": True,
        "0": False,
    }

    assert normalized.notna().all()

    assert normalized.isin(mapping).all(), (
        "Unrecognized values in saved acceptance decisions."
    )

    return normalized.map(mapping).to_numpy(dtype=bool)


for method, gate_group in gate_audit.groupby("method", sort=True):
    prediction_group, correct = prediction_groups[method]

    assert len(gate_group) == N
    assert set(gate_group["record_id"]) == test_ids

    gate_group = (
        gate_group
        .set_index("record_id")
        .reindex(prediction_group.index)
        .copy()
    )

    assert gate_group[
        list(required_gate_columns - {"record_id"})
    ].notna().all().all()

    policy = protocol["frozen_policies"][f"{method}:CRUS"]

    for field in ["confidence", "CRUS", "tau_c", "tau_u"]:
        gate_group[field] = pd.to_numeric(
            gate_group[field],
            errors="raise",
        )

        assert np.isfinite(
            gate_group[field].to_numpy()
        ).all()

    assert np.allclose(
        gate_group["tau_c"],
        policy["tau_c"],
        rtol=0,
        atol=1e-12,
    )

    assert np.allclose(
        gate_group["tau_u"],
        policy["tau_u"],
        rtol=0,
        atol=1e-12,
    )

    saved_accepted = parse_boolean_column(
        gate_group["accepted"]
    )

    expected_accepted = (
        (
            gate_group["confidence"].to_numpy()
            >= policy["tau_c"]
        )
        & (
            gate_group["CRUS"].to_numpy()
            <= policy["tau_u"]
        )
    )

    assert np.array_equal(
        saved_accepted,
        expected_accepted,
    ), f"{method}: saved decisions differ from the frozen gate."

    result_rows.append(
        count_metrics(
            method=method,
            mode="native_CRUS_gate",
            correct=correct,
            accepted=saved_accepted,
        )
    )

COMMENT3_RESULTS = pd.DataFrame(result_rows)


# ============================================================
# 7. Compare against the saved main-results summary
# ============================================================

existing_summary = pd.read_csv(SUMMARY_PATH)

comparison_fields = [
    "N",
    "n_accepted",
    "n_correct_accepted",
    "n_errors_accepted",
    "n_abstained",
    "coverage",
    "forced_accuracy",
    "selective_accuracy",
    "abstention_penalized_accuracy",
    "abstention_penalized_error",
]

assert {
    "method", "mode", *comparison_fields
}.issubset(existing_summary.columns)

assert not existing_summary.duplicated(
    ["method", "mode"]
).any()

recalculated_keys = set(
    COMMENT3_RESULTS[["method", "mode"]]
    .itertuples(index=False, name=None)
)

saved_keys = set(
    existing_summary[["method", "mode"]]
    .itertuples(index=False, name=None)
)

assert recalculated_keys == saved_keys, (
    "The saved summary and recalculated audit contain "
    "different method/mode rows."
)

comparison = COMMENT3_RESULTS.merge(
    existing_summary[["method", "mode"] + comparison_fields],
    on=["method", "mode"],
    suffixes=("_recalculated", "_saved"),
    validate="one_to_one",
)

assert len(comparison) == len(COMMENT3_RESULTS)

for field in comparison_fields:
    assert np.allclose(
        comparison[f"{field}_recalculated"],
        comparison[f"{field}_saved"],
        rtol=0,
        atol=1e-12,
        equal_nan=True,
    ), (
        "Recalculated and saved summaries disagree for "
        f"{field}. Stop and inspect; do not overwrite."
    )


# ============================================================
# 8. Save a separate audit without overwriting original results
# ============================================================

audit_path = OUTPUT_DIR / "comment3_verified_count_metrics.csv"
audit_text = COMMENT3_RESULTS.to_csv(index=False)

if audit_path.exists():
    assert audit_path.read_text(encoding="utf-8") == audit_text, (
        "An existing Comment 3 audit differs. "
        "Stop rather than overwrite it."
    )
    audit_status = "Existing matching audit verified."
else:
    audit_path.write_text(audit_text, encoding="utf-8")
    audit_status = "New Comment 3 audit saved."


# ============================================================
# 9. Print the verified results
# ============================================================

display_columns = [
    "method",
    "mode",
    "N",
    "n_correct_forced",
    "forced_accuracy",
    "n_accepted",
    "n_correct_accepted",
    "n_errors_accepted",
    "n_abstained",
    "coverage",
    "selective_accuracy",
    "accepted_misclassification_rate",
    "abstention_penalized_accuracy",
    "abstention_penalized_error",
]

print("\nCOMMENT 3: VERIFIED COUNT-BASED METRICS\n")

print(
    COMMENT3_RESULTS[display_columns].to_string(
        index=False,
        float_format=lambda value: f"{value:.6f}",
    )
)

cascade_native = COMMENT3_RESULTS.loc[
    (COMMENT3_RESULTS["method"] == "Cascade")
    & (COMMENT3_RESULTS["mode"] == "native_CRUS_gate")
].iloc[0]

print("\nPRIMARY CASCADE–CRUS RESULTS")

print(
    f"Forced correct: "
    f"{int(cascade_native['n_correct_forced'])}/{N}"
)

print(
    f"Forced accuracy: "
    f"{100 * cascade_native['forced_accuracy']:.2f}%"
)

print(
    f"Accepted: {int(cascade_native['n_accepted'])}/{N}"
)

print(
    "Correct accepted:",
    int(cascade_native["n_correct_accepted"]),
)

print(
    "Accepted misclassifications:",
    int(cascade_native["n_errors_accepted"]),
)

print(
    "Abstentions:",
    int(cascade_native["n_abstained"]),
)

for label, field in [
    ("Coverage", "coverage"),
    ("Selective accuracy", "selective_accuracy"),
    (
        "Accepted-case misclassification rate",
        "accepted_misclassification_rate",
    ),
    (
        "Abstention-penalized accuracy",
        "abstention_penalized_accuracy",
    ),
    (
        "Abstention-penalized error",
        "abstention_penalized_error",
    ),
]:
    print(f"{label}: {100 * cascade_native[field]:.2f}%")


# ============================================================
# 10. Confirm that every original source file is unchanged
# ============================================================

for path, original_hash in original_file_hashes.items():
    assert sha256_file(path) == original_hash, (
        f"Source file changed during verification: {path.name}"
    )

print("\nAll count identities passed.")
print("All frozen acceptance decisions verified.")
print("All recalculated metrics match the saved main summary.")
print(audit_status)
print("Audit filename:", audit_path.name)
print("No models trained and no new predictions generated.")
print("All original source files unchanged.")

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

# ---------- CONFIGURATION: EDIT TO MATCH YOUR SAVED DATA ----------

INPUT_CSV = Path("saved_case_level_predictions.csv")
OUTPUT_DIR = Path("comment4_audit")

CASE_COL = "case_id"
METHOD_COL = "method"
TRUE_COL = "y_true"
PRED_COL = "y_pred"

EXPECTED_N = 257
RETAINED_COUNTS = [117, 171]

# Keys must exactly match METHOD_COL values.
# score_col: existing saved score, not a newly constructed score.
# higher_is_better: True for acceptance confidence, False for uncertainty.
# description: actual definition used in the frozen protocol.
#
# Example shape only:
# "actual_method_name": {
#     "score_col": "actual_saved_score_column",
#     "higher_is_better": True,
#     "description": "Actual frozen score definition",
# }

RANKING = {
    # Fill this dictionary before running.
}

# This script implements ties by ascending case ID as a string.
# Confirm consistency with the protocol before setting True.
CONFIRM_CASE_ID_TIE_RULE = False

# ---------- VALIDATION ----------

if not RANKING:
    raise ValueError("Fill RANKING with actual method names and saved scores.")

if not CONFIRM_CASE_ID_TIE_RULE:
    raise ValueError(
        "Verify the tie rule. This script breaks score ties by ascending "
        "case ID as a string. Set CONFIRM_CASE_ID_TIE_RULE=True only "
        "after confirming or explicitly documenting this rule."
    )

df = pd.read_csv(INPUT_CSV, dtype={CASE_COL: "string"})

required = {CASE_COL, METHOD_COL, TRUE_COL, PRED_COL}
required.update(cfg["score_col"] for cfg in RANKING.values())
missing = required - set(df.columns)
if missing:
    raise ValueError(f"Missing columns: {sorted(missing)}")

if df[[CASE_COL, METHOD_COL, TRUE_COL, PRED_COL]].isna().any().any():
    raise ValueError("Missing case IDs, method names, true labels, or predictions.")

if df.duplicated([METHOD_COL, CASE_COL]).any():
    raise ValueError("Duplicate method/case rows found.")

if df.groupby(CASE_COL)[TRUE_COL].nunique().gt(1).any():
    raise ValueError("True labels disagree across methods for the same case.")

if any(k < 1 or k > EXPECTED_N for k in RETAINED_COUNTS):
    raise ValueError("Retained counts must lie between 1 and EXPECTED_N.")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

summaries = []
ranked_outputs = []
curves = []
metadata = []
reference_ids = None

# ---------- RANKING AND COUNT-BASED METRICS ----------

for method, cfg in RANKING.items():
    part = df.loc[df[METHOD_COL].eq(method)].copy()

    if len(part) != EXPECTED_N:
        raise ValueError(
            f"{method}: expected {EXPECTED_N} records, found {len(part)}."
        )

    ids = set(part[CASE_COL])
    if reference_ids is None:
        reference_ids = ids
    elif ids != reference_ids:
        raise ValueError(f"{method}: test case IDs differ from other methods.")

    score_col = cfg["score_col"]
    part["_score"] = pd.to_numeric(part[score_col], errors="raise")
    if not np.isfinite(part["_score"].to_numpy()).all():
        raise ValueError(f"{method}: ranking scores contain NaN or infinity.")

    part["_correct"] = part[PRED_COL].eq(part[TRUE_COL]).astype(int)

    part = part.sort_values(
        ["_score", CASE_COL],
        ascending=[not cfg["higher_is_better"], True],
        kind="mergesort",
    ).reset_index(drop=True)

    part["acceptance_rank"] = np.arange(1, EXPECTED_N + 1)
    cumulative_correct = part["_correct"].cumsum().to_numpy()
    k_all = np.arange(1, EXPECTED_N + 1)

    curves.append(pd.DataFrame({
        "method": method,
        "n_retained": k_all,
        "n_correct_retained": cumulative_correct,
        "n_errors_retained": k_all - cumulative_correct,
        "coverage": k_all / EXPECTED_N,
        "selective_accuracy": cumulative_correct / k_all,
        "selective_misclassification_rate":
            (k_all - cumulative_correct) / k_all,
    }))

    for k in RETAINED_COUNTS:
        selected = part.iloc[:k]
        correct = int(selected["_correct"].sum())
        errors = k - correct
        boundary_score = part.loc[k - 1, "_score"]
        tied = part["_score"].eq(boundary_score)
        tie_total = int(tied.sum())
        tie_selected = int(selected["_score"].eq(boundary_score).sum())

        summaries.append({
            "method": method,
            "N": EXPECTED_N,
            "n_retained": k,
            "n_correct_retained": correct,
            "n_errors_retained": errors,
            "n_not_retained": EXPECTED_N - k,
            "coverage": k / EXPECTED_N,
            "selective_accuracy": correct / k,
            "selective_misclassification_rate": errors / k,
            "accuracy_percent": 100 * correct / k,
            "correct_count_fraction": f"{correct}/{k}",
            "boundary_score": boundary_score,
            "boundary_tie_total": tie_total,
            "boundary_tie_retained": tie_selected,
            "boundary_tie_split": tie_selected < tie_total,
        })

        part[f"retained_at_{k}"] = part["acceptance_rank"].le(k)

    ranked_outputs.append(
        part.rename(columns={
            "_score": "audit_ranking_score",
            "_correct": "audit_correct",
        })
    )

    metadata.append({
        "method": method,
        "score_column": score_col,
        "ranking_description": cfg["description"],
        "score_direction": (
            "higher first" if cfg["higher_is_better"] else "lower first"
        ),
        "tie_rule": "ascending case ID, compared as strings",
    })

summary = pd.DataFrame(summaries).sort_values(["n_retained", "method"])
curve_df = pd.concat(curves, ignore_index=True)
ranked_df = pd.concat(ranked_outputs, ignore_index=True)
metadata_df = pd.DataFrame(metadata)

assert (
    summary["n_correct_retained"] + summary["n_errors_retained"]
    == summary["n_retained"]
).all()

assert (
    summary["n_retained"] + summary["n_not_retained"]
    == summary["N"]
).all()

summary.to_csv(
    OUTPUT_DIR / "comment4_matched_coverage_counts.csv", index=False
)
ranked_df.to_csv(
    OUTPUT_DIR / "comment4_ranked_case_audit.csv", index=False
)
curve_df.to_csv(
    OUTPUT_DIR / "comment4_risk_coverage_points.csv", index=False
)
metadata_df.to_csv(
    OUTPUT_DIR / "comment4_ranking_and_tie_rules.csv", index=False
)

# ---------- REGENERATED LATEX TABLES ----------

def latex_escape(value):
    replacements = {
        "\\": r"\textbackslash{}",
        "&": r"\&",
        "%": r"\%",
        "$": r"\$",
        "#": r"\#",
        "_": r"\_",
        "{": r"\{",
        "}": r"\}",
        "~": r"\textasciitilde{}",
        "^": r"\textasciicircum{}",
    }
    return "".join(replacements.get(c, c) for c in str(value))

for k in RETAINED_COUNTS:
    subset = summary.loc[summary["n_retained"].eq(k)]
    coverage_percent = 100 * k / EXPECTED_N
    lines = [
        r"\begin{table}[ht]",
        r"\centering",
        (
            r"\caption{Descriptive matched-coverage comparison retaining "
            f"{k} of {EXPECTED_N} test records "
            f"({coverage_percent:.2f}\\% coverage) for each method. "
            r"Each method retains records according to its declared "
            r"ranking score and tie rule; retained subsets may differ.}"
        ),
        rf"\label{{tab:matched_coverage_{k}}}",
        r"\begin{tabular}{lrrr}",
        r"\toprule",
        r"Method & Correct/retained & Errors & Accuracy (\%) \\",
        r"\midrule",
    ]
    for _, row in subset.iterrows():
        lines.append(
            f"{latex_escape(row['method'])} & "
            f"{int(row['n_correct_retained'])}/{k} & "
            f"{int(row['n_errors_retained'])} & "
            f"{row['accuracy_percent']:.2f} " + r"\\"
        )
    lines += [
        r"\bottomrule",
        r"\end{tabular}",
        r"\end{table}",
    ]
    (OUTPUT_DIR / f"comment4_table_{k}.tex").write_text(
        "\n".join(lines), encoding="utf-8"
    )

print(summary[
    ["method", "n_retained", "correct_count_fraction",
     "n_errors_retained", "accuracy_percent",
     "boundary_tie_split"]
].to_string(index=False))

print("\nRanking and tie rules:")
print(metadata_df.to_string(index=False))
print(f"\nAudit outputs saved in: {OUTPUT_DIR.resolve()}")

In [ ]:
from pathlib import Path
import pandas as pd

print("CSV files in the current directory:")
for path in sorted(Path(".").glob("*.csv")):
    print(path.name)

INPUT_CSV = Path("YOUR_ACTUAL_CASE_LEVEL_PREDICTIONS.csv")

preview = pd.read_csv(INPUT_CSV, nrows=5)

print("\nColumn names:")
print(preview.columns.tolist())

print("\nFirst five rows:")
print(preview.to_string(index=False))

if "method" in preview.columns:
    methods = pd.read_csv(INPUT_CSV, usecols=["method"])
    print("\nMethod names:")
    print(methods["method"].dropna().unique().tolist())

In [ ]:
from pathlib import Path
import os

print("Current working directory:", Path.cwd())

roots = [Path(_crus_local_path('runtime', ''))]

drive_root = Path(_crus_local_path('mount', ''))
if not (drive_root / "MyDrive").exists():
    pass  # Colab service not required locally.
    _crus_prepare_saved_workspace()

# Search local runtime files separately from mounted Drive.
found = set()
extensions = {".csv", ".parquet", ".json", ".jsonl", ".pkl", ".npz"}

for root in roots:
    if not root.exists():
        continue

    for directory, subdirs, filenames in os.walk(root):
        subdirs[:] = [
            name for name in subdirs
            if name not in {
                "drive", ".git", "__pycache__", ".cache",
                "sample_data", "node_modules"
            }
        ]

        for name in filenames:
            path = Path(directory) / name
            if path.suffix.lower() in extensions:
                found.add(str(path))

for root in [
    drive_root / "MyDrive",
    drive_root / "Shareddrives",
]:
    if not root.exists():
        continue

    for directory, subdirs, filenames in os.walk(root):
        subdirs[:] = [
            name for name in subdirs
            if name not in {".git", "__pycache__", ".cache", "node_modules"}
        ]

        for name in filenames:
            path = Path(directory) / name
            if path.suffix.lower() in extensions:
                found.add(str(path))

CANDIDATE_FILES = sorted(found)

print(f"\nFound {len(CANDIDATE_FILES)} candidate data files:")
for index, path in enumerate(CANDIDATE_FILES):
    print(f"[{index}] {path}")

if not CANDIDATE_FILES:
    print(
        "\nNo candidate data files found in the searched locations."
    )


In [ ]:
from pathlib import Path
import os

print("Current working directory:", Path.cwd())

if not Path(_crus_local_path('drive', '')).exists():
    pass  # Colab service not required locally.
    _crus_prepare_saved_workspace()

extensions = {".csv", ".parquet", ".json", ".jsonl", ".pkl", ".npz"}
found = set()

roots = [
    Path(_crus_local_path('runtime', '')),
    Path(_crus_local_path('drive', '')),
    Path(_crus_local_path('mount', 'Shareddrives')),
]

for root in roots:
    if not root.exists():
        continue

    for directory, subdirs, filenames in os.walk(root):
        excluded = {".git", "__pycache__", ".cache", "node_modules"}

        if root == Path(_crus_local_path('runtime', '')):
            excluded.update({"drive", "sample_data"})

        subdirs[:] = [name for name in subdirs if name not in excluded]

        for name in filenames:
            path = Path(directory) / name
            if path.suffix.lower() in extensions:
                found.add(str(path))

CANDIDATE_FILES = sorted(found)

print()
print("Number of candidate files:", len(CANDIDATE_FILES))

for index, path in enumerate(CANDIDATE_FILES):
    print(f"[{index}] {path}")

if not CANDIDATE_FILES:
    print("No candidate files found. Check the paths in your Comment 3 script.")

In [ ]:
from pathlib import Path
import json
import pandas as pd

BASE = Path(
    _crus_local_path('project', 'corrected_experiment_outputs')
)

csv_files = [
    "test_all_method_predictions.csv",
    "test_matched_count_acceptance_audit.csv",
    "test_matched_count_results.csv",
    "test_ranking_risk_coverage_curves.csv",
    "test_ranking_risk_coverage_summary.csv",
]

for filename in csv_files:
    path = BASE / filename
    print("\n" + "=" * 75)
    print(filename)

    if not path.exists():
        print("FILE NOT FOUND")
        continue

    table = pd.read_csv(path)
    print("Shape:", table.shape)
    print("Columns:", table.columns.tolist())

    if filename in {
        "test_matched_count_results.csv",
        "test_ranking_risk_coverage_summary.csv",
    }:
        print(table.to_string(index=False))
    else:
        print(table.head(3).to_string(index=False))

    for column in ["method", "ranking_score", "score_name", "tie_rule"]:
        if column in table.columns:
            values = table[column].dropna().unique()
            print(f"{column} unique values:", values[:30].tolist())

json_files = [
    "final_pretest_protocol.json",
    "validation_selected_policies.json",
]

for filename in json_files:
    path = BASE / filename
    print("\n" + "=" * 75)
    print(filename)

    if not path.exists():
        print("FILE NOT FOUND")
        continue

    with path.open("r", encoding="utf-8") as handle:
        content = json.load(handle)

    print(json.dumps(content, indent=2, ensure_ascii=False))

In [ ]:
from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd

BASE = Path(
    _crus_local_path('project', 'corrected_experiment_outputs')
)
OUT = BASE / "comment4_audit"
OUT.mkdir(exist_ok=True)

names = [
    "final_pretest_protocol.json",
    "test_all_method_predictions.csv",
    "test_matched_count_acceptance_audit.csv",
    "test_matched_count_results.csv",
    "test_ranking_risk_coverage_curves.csv",
    "test_ranking_risk_coverage_summary.csv",
    "comment3_verified_count_metrics.csv",
]

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

original_hashes = {name: sha256(BASE / name) for name in names}

with (BASE / names[0]).open(encoding="utf-8") as handle:
    protocol = json.load(handle)

assert protocol["ranking_ties"] == "score descending; record ID ascending"
assert (
    protocol["reporting_rules"]["matched_coverage"]
    == "match native Cascade accepted count"
)

N = int(protocol["split_sizes"]["test"])

def read_csv(name):
    return pd.read_csv(BASE / name, dtype={"record_id": "string"})

def parse_bool(series):
    mapping = {"true": True, "false": False, "1": True, "0": False}
    normalized = series.astype(str).str.strip().str.lower()
    if not normalized.isin(mapping).all():
        raise ValueError(f"Invalid boolean values in {series.name}")
    return normalized.map(mapping).astype(bool)

pred = read_csv("test_all_method_predictions.csv")
audit = read_csv("test_matched_count_acceptance_audit.csv")
saved = read_csv("test_matched_count_results.csv")
saved_curves = read_csv("test_ranking_risk_coverage_curves.csv")
saved_summary = read_csv("test_ranking_risk_coverage_summary.csv")
comment3 = read_csv("comment3_verified_count_metrics.csv")

required = [
    "record_id", "method", "true_label", "prediction",
    "correct", "ranking_score",
]
assert not pred[required].isna().any().any()
assert not pred.duplicated(["method", "record_id"]).any()
assert not audit.duplicated(
    ["method", "target_n_accepted", "record_id"]
).any()
assert not saved.duplicated(["method", "target_n_accepted"]).any()
assert not saved_curves.duplicated(["method", "n_accepted"]).any()

pred["correct_saved"] = parse_bool(pred["correct"])
pred["correct_recomputed"] = pred["true_label"].eq(pred["prediction"])
assert pred["correct_saved"].eq(pred["correct_recomputed"]).all()

pred["ranking_score"] = pd.to_numeric(pred["ranking_score"], errors="raise")
assert np.isfinite(pred["ranking_score"]).all()
assert pred.groupby("record_id")["true_label"].nunique().eq(1).all()

methods = sorted(pred["method"].unique())
assert set(audit["method"]) == set(methods)
assert set(saved["method"]) == set(methods)
assert set(saved_curves["method"]) == set(methods)
assert set(saved_summary["method"]) == set(methods)

native = comment3.loc[
    comment3["method"].eq("Cascade")
    & comment3["mode"].eq("native_CRUS_gate")
]
assert len(native) == 1
native = native.iloc[0]
assert int(native["N"]) == N

PRIMARY_K = int(native["n_accepted"])
saved_targets = sorted(saved["target_n_accepted"].astype(int).unique())
targets = sorted(set(saved_targets + [PRIMARY_K, 171]))
assert all(1 <= k <= N for k in targets)

rule = "ranking_score_descending; record_id_ascending_for_ties"
assert saved["selection_rule"].eq(rule).all()
assert saved_curves["selection_rule"].eq(rule).all()

ranked_parts = []
curve_parts = []
rows = []
reference_ids = None

for method in methods:
    part = pred.loc[pred["method"].eq(method)].copy()
    assert len(part) == N

    ids = set(part["record_id"])
    if reference_ids is None:
        reference_ids = ids
    else:
        assert ids == reference_ids

    part = part.sort_values(
        ["ranking_score", "record_id"],
        ascending=[False, True],
        kind="mergesort",
    ).reset_index(drop=True)

    part["acceptance_rank"] = np.arange(1, N + 1)
    correct_prefix = part["correct_recomputed"].astype(int).cumsum()
    counts = np.arange(1, N + 1)

    curve = pd.DataFrame({
        "method": method,
        "n_accepted": counts,
        "n_correct_accepted": correct_prefix.to_numpy(),
        "n_errors_accepted": counts - correct_prefix.to_numpy(),
        "coverage": counts / N,
        "selective_accuracy": correct_prefix.to_numpy() / counts,
        "risk": (counts - correct_prefix.to_numpy()) / counts,
        "abstention_penalized_accuracy":
            correct_prefix.to_numpy() / N,
        "selection_rule": rule,
    })
    curve_parts.append(curve)

    for k in targets:
        selected = part.iloc[:k]
        correct = int(selected["correct_recomputed"].sum())
        boundary = float(part.loc[k - 1, "ranking_score"])
        total_tied = int(part["ranking_score"].eq(boundary).sum())
        retained_tied = int(selected["ranking_score"].eq(boundary).sum())

        rows.append({
            "method": method,
            "N": N,
            "target_n_accepted": k,
            "n_accepted": k,
            "n_correct_accepted": correct,
            "n_errors_accepted": k - correct,
            "n_abstained": N - k,
            "coverage": k / N,
            "selective_accuracy": correct / k,
            "accepted_misclassification_rate": (k - correct) / k,
            "forced_accuracy": part["correct_recomputed"].mean(),
            "abstention_penalized_accuracy": correct / N,
            "abstention_penalized_error": (N - correct) / N,
            "boundary_score": boundary,
            "boundary_tie_total": total_tied,
            "boundary_tie_retained": retained_tied,
            "boundary_tie_split": retained_tied < total_tied,
            "selection_rule": rule,
        })
        part[f"retained_top_{k}"] = part["acceptance_rank"].le(k)

        if k in saved_targets:
            old = audit.loc[
                audit["method"].eq(method)
                & audit["target_n_accepted"].eq(k)
            ].copy()

            assert len(old) == N
            assert set(old["record_id"]) == ids

            old = old.set_index("record_id").loc[part["record_id"]]
            np.testing.assert_array_equal(
                parse_bool(old["accepted"]).to_numpy(),
                part["acceptance_rank"].le(k).to_numpy(),
            )
            np.testing.assert_array_equal(
                parse_bool(old["correct"]).to_numpy(),
                part["correct_recomputed"].to_numpy(),
            )
            np.testing.assert_allclose(
                old["ranking_score"].to_numpy(),
                part["ranking_score"].to_numpy(),
                rtol=1e-10, atol=1e-12,
            )

    ranked_parts.append(part)

result = pd.DataFrame(rows)
curves = pd.concat(curve_parts, ignore_index=True)
ranked = pd.concat(ranked_parts, ignore_index=True)

count_columns = [
    "N", "n_accepted", "n_correct_accepted",
    "n_errors_accepted", "n_abstained",
]
rate_columns = [
    "coverage", "selective_accuracy",
    "accepted_misclassification_rate", "forced_accuracy",
    "abstention_penalized_accuracy", "abstention_penalized_error",
]

for _, old in saved.iterrows():
    new = result.loc[
        result["method"].eq(old["method"])
        & result["target_n_accepted"].eq(int(old["target_n_accepted"]))
    ]
    assert len(new) == 1
    new = new.iloc[0]
    for col in count_columns:
        assert int(new[col]) == int(old[col]), (old["method"], col)
    for col in rate_columns:
        np.testing.assert_allclose(
            float(new[col]), float(old[col]),
            rtol=1e-9, atol=1e-12,
        )

curve_columns = [
    "n_correct_accepted", "n_errors_accepted", "coverage",
    "selective_accuracy", "risk", "abstention_penalized_accuracy",
]
old_indexed = saved_curves.set_index(["method", "n_accepted"]).sort_index()
new_indexed = curves.set_index(["method", "n_accepted"]).sort_index()
assert old_indexed.index.equals(new_indexed.index)

for col in curve_columns:
    np.testing.assert_allclose(
        old_indexed[col], new_indexed[col],
        rtol=1e-9, atol=1e-12,
    )

mean_risk = curves.groupby("method")["risk"].mean()
for _, old in saved_summary.iterrows():
    np.testing.assert_allclose(
        mean_risk.loc[old["method"]], old["mean_prefix_risk"],
        rtol=1e-9, atol=1e-12,
    )
    assert old["protocol_sha256"] == original_hashes[
        "final_pretest_protocol.json"
    ]

result.to_csv(OUT / "comment4_verified_matched_counts.csv", index=False)
ranked.to_csv(OUT / "comment4_verified_ranked_predictions.csv", index=False)
curves.to_csv(OUT / "comment4_verified_risk_coverage_curves.csv", index=False)

def escape_latex(value):
    return str(value).replace("_", r"\_").replace("&", r"\&")

for k in [PRIMARY_K, 171]:
    table = result.loc[result["target_n_accepted"].eq(k)]
    label = (
        "tab:coverage_matched" if k == PRIMARY_K
        else "tab:additional_top171_audit"
    )
    lines = [
        r"\begin{table}[ht]",
        r"\centering",
        r"\small",
        (
            r"\caption{Descriptive ranking-based matched-coverage "
            f"comparison retaining {k} of {N} records "
            f"({100 * k / N:.2f}\\% coverage). "
            r"Records are ranked by saved ranking score, descending, "
            r"with ascending record ID for ties. Retained subsets "
            r"may differ across methods.}"
        ),
        rf"\label{{{label}}}",
        r"\begin{tabular}{lrrr}",
        r"\toprule",
        r"Method & Correct/retained & Errors & Accuracy (\%) \\",
        r"\midrule",
    ]
    for _, row in table.iterrows():
        lines.append(
            f"{escape_latex(row['method'])} & "
            f"{int(row['n_correct_accepted'])}/{k} & "
            f"{int(row['n_errors_accepted'])} & "
            f"{100 * row['selective_accuracy']:.2f} " + r"\\"
        )
    lines += [r"\bottomrule", r"\end{tabular}", r"\end{table}"]
    (OUT / f"comment4_table_top_{k}.tex").write_text(
        "\n".join(lines), encoding="utf-8"
    )

assert all(
    sha256(BASE / name) == original_hashes[name]
    for name in names
)

print("COMMENT 4 AUDIT PASSED")
print("Test records:", N)
print("Primary matched retained count:", PRIMARY_K)
print("All saved matched acceptance decisions verified.")
print("All saved count-based matched metrics verified.")
print("All saved ranking risk-coverage points verified.")
print("All original input files unchanged.")

for k in [PRIMARY_K, 171]:
    table = result.loc[
        result["target_n_accepted"].eq(k),
        ["method", "n_correct_accepted", "n_errors_accepted",
         "selective_accuracy", "boundary_tie_split"],
    ].copy()
    table["accuracy_percent"] = 100 * table["selective_accuracy"]
    print(f"\nTOP {k}: coverage = {100 * k / N:.2f}%")
    print(table.drop(columns="selective_accuracy").to_string(index=False))

print(
    "\nNative Cascade CRUS gate:",
    f"{int(native['n_correct_accepted'])}/{PRIMARY_K}",
    f"= {100 * float(native['selective_accuracy']):.2f}%",
)
print("\nAudit outputs:", OUT)

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

BASE = Path(
    _crus_local_path('project', 'corrected_experiment_outputs/comment4_audit')
)

curves = pd.read_csv(
    BASE / "comment4_verified_risk_coverage_curves.csv"
)

fig, ax = plt.subplots(figsize=(8, 5.5))

for method, part in curves.groupby("method", sort=True):
    part = part.sort_values("n_accepted")
    ax.plot(
        part["coverage"],
        part["risk"],
        label=method.replace("_", " "),
        linewidth=1.7,
    )

ax.set_xlabel("Coverage")
ax.set_ylabel("Selective misclassification rate")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.grid(alpha=0.25)
ax.legend(fontsize=8, loc="best", ncol=2)
fig.tight_layout()

pdf_path = BASE / "comment4_verified_risk_coverage.pdf"
png_path = BASE / "comment4_verified_risk_coverage.png"

fig.savefig(pdf_path, bbox_inches="tight")
fig.savefig(png_path, dpi=300, bbox_inches="tight")
plt.show()

print("PDF saved:", pdf_path)
print("PNG saved:", png_path)

# **COMMENT 5**

In [ ]:
from pathlib import Path
import hashlib
import json
import pandas as pd

ROOT = Path(_crus_local_path('project', ''))
DATA = ROOT / "data_audit"
RESULTS = ROOT / "corrected_experiment_outputs"

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def inspect_csv(path, full=False):
    print("\n" + "=" * 78)
    print(path.name)

    if not path.exists():
        print("FILE NOT FOUND")
        return

    table = pd.read_csv(path)
    print("Shape:", table.shape)
    print("Columns:", table.columns.tolist())

    if full and len(table) <= 150:
        print(table.to_string(index=False))
    else:
        print(table.head(3).to_string(index=False))

    for column in ["split", "method"]:
        if column in table.columns:
            print(
                f"Counts by {column}:",
                table[column].value_counts(dropna=False).to_dict()
            )

def inspect_json(path):
    print("\n" + "=" * 78)
    print(path.name)

    if not path.exists():
        print("FILE NOT FOUND")
        return

    with path.open(encoding="utf-8") as handle:
        content = json.load(handle)

    print(json.dumps(content, indent=2, ensure_ascii=False))

protocol_path = RESULTS / "final_pretest_protocol.json"
with protocol_path.open(encoding="utf-8") as handle:
    protocol = json.load(handle)

print("COMMENT 5: SOURCE AND SCHEMA INSPECTION")
print("Protocol version:", protocol.get("protocol_version"))
print("Frozen split sizes:", protocol.get("split_sizes"))
print("Frozen classes:", protocol.get("classes"))
print("Expected main manifest hash:", protocol.get("main_manifest_sha256"))

manifest_names = [
    "constrained_split_manifest.csv",
    "corrected_split_manifest.csv",
]

matching_manifests = []

for name in manifest_names:
    path = DATA / name
    if not path.exists():
        continue

    digest = sha256(path)
    matches = digest == protocol.get("main_manifest_sha256")

    print("\nManifest:", name)
    print("SHA256:", digest)
    print("Matches frozen main manifest:", matches)

    if matches:
        matching_manifests.append(str(path))

print("\nMatching main manifests:", matching_manifests)

csv_specs = [
    (DATA / "constrained_split_manifest.csv", False),
    (DATA / "corrected_split_manifest.csv", False),
    (DATA / "constrained_dialogue_pool_with_splits.csv", False),
    (DATA / "corrected_dialogue_pool_with_splits.csv", False),
    (DATA / "final_cleaned_class_counts.csv", True),
    (DATA / "constrained_class_distribution_per_split.csv", True),
    (DATA / "corrected_class_distribution_per_split.csv", True),
    (DATA / "rare_class_exclusion_log.csv", False),
    (DATA / "exact_duplicate_removal_log.csv", False),
    (DATA / "exact_dialogue_match_groups.csv", False),
    (DATA / "exact_dialogue_match_records.csv", False),
    (DATA / "near_duplicate_candidates_for_review.csv", False),
    (DATA / "near_duplicate_review_decisions.csv", False),
    (DATA / "retained_conflicting_label_groups.csv", False),
    (RESULTS / "test_all_method_predictions.csv", False),
    (RESULTS / "validation_retrieval_prediction_audit.csv", False),
    (RESULTS / "validation_TFIDF_LR_predictions.csv", False),
    (RESULTS / "validation_TFIDF_SVM_predictions.csv", False),
]

for path, full in csv_specs:
    inspect_csv(path, full=full)

json_paths = [
    DATA / "final_cleaning_grouping_policy.json",
    DATA / "near_duplicate_screen_settings.json",
    DATA / "constrained_split_metadata.json",
    DATA / "corrected_split_metadata.json",
]

for path in json_paths:
    inspect_json(path)

print("\nInspection complete. No source files were modified.")

In [ ]:
from pathlib import Path
pass  # Colab service not required locally.

if not Path(_crus_local_path('drive', '')).exists():
    _crus_prepare_saved_workspace()

ROOT = Path(_crus_local_path('project', ''))
DATA = ROOT / "data_audit"
RESULTS = ROOT / "corrected_experiment_outputs"
protocol_path = RESULTS / "final_pretest_protocol.json"

print("Project folder exists:", ROOT.exists())
print("Data audit folder exists:", DATA.exists())
print("Results folder exists:", RESULTS.exists())
print("Frozen protocol exists:", protocol_path.exists())

if not protocol_path.exists():
    raise FileNotFoundError(
        "The frozen protocol is still unavailable. "
        "Check that you mounted the same Google account "
        "used for the Comment 3 and Comment 4 audits."
    )

print("Setup passed. Rerun the Comment 5 inspection cell.")

In [ ]:
from pathlib import Path
import hashlib
import json
import pandas as pd

In [ ]:
from pathlib import Path
import hashlib
import json
import pandas as pd

ROOT = Path(_crus_local_path('project', ''))
DATA = ROOT / "data_audit"
RESULTS = ROOT / "corrected_experiment_outputs"

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def inspect_csv(path, full=False):
    print("\n" + "=" * 78)
    print(path.name)

    if not path.exists():
        print("FILE NOT FOUND")
        return

    table = pd.read_csv(path)
    print("Shape:", table.shape)
    print("Columns:", table.columns.tolist())

    if full and len(table) <= 150:
        print(table.to_string(index=False))
    else:
        print(table.head(3).to_string(index=False))

    for column in ["split", "method"]:
        if column in table.columns:
            print(
                f"Counts by {column}:",
                table[column].value_counts(dropna=False).to_dict()
            )

def inspect_json(path):
    print("\n" + "=" * 78)
    print(path.name)

    if not path.exists():
        print("FILE NOT FOUND")
        return

    with path.open(encoding="utf-8") as handle:
        content = json.load(handle)

    print(json.dumps(content, indent=2, ensure_ascii=False))

protocol_path = RESULTS / "final_pretest_protocol.json"
with protocol_path.open(encoding="utf-8") as handle:
    protocol = json.load(handle)

print("COMMENT 5: SOURCE AND SCHEMA INSPECTION")
print("Protocol version:", protocol.get("protocol_version"))
print("Frozen split sizes:", protocol.get("split_sizes"))
print("Frozen classes:", protocol.get("classes"))
print("Expected main manifest hash:", protocol.get("main_manifest_sha256"))

manifest_names = [
    "constrained_split_manifest.csv",
    "corrected_split_manifest.csv",
]

matching_manifests = []

for name in manifest_names:
    path = DATA / name
    if not path.exists():
        continue

    digest = sha256(path)
    matches = digest == protocol.get("main_manifest_sha256")

    print("\nManifest:", name)
    print("SHA256:", digest)
    print("Matches frozen main manifest:", matches)

    if matches:
        matching_manifests.append(str(path))

print("\nMatching main manifests:", matching_manifests)

csv_specs = [
    (DATA / "constrained_split_manifest.csv", False),
    (DATA / "corrected_split_manifest.csv", False),
    (DATA / "constrained_dialogue_pool_with_splits.csv", False),
    (DATA / "corrected_dialogue_pool_with_splits.csv", False),
    (DATA / "final_cleaned_class_counts.csv", True),
    (DATA / "constrained_class_distribution_per_split.csv", True),
    (DATA / "corrected_class_distribution_per_split.csv", True),
    (DATA / "rare_class_exclusion_log.csv", False),
    (DATA / "exact_duplicate_removal_log.csv", False),
    (DATA / "exact_dialogue_match_groups.csv", False),
    (DATA / "exact_dialogue_match_records.csv", False),
    (DATA / "near_duplicate_candidates_for_review.csv", False),
    (DATA / "near_duplicate_review_decisions.csv", False),
    (DATA / "retained_conflicting_label_groups.csv", False),
    (RESULTS / "test_all_method_predictions.csv", False),
    (RESULTS / "validation_retrieval_prediction_audit.csv", False),
    (RESULTS / "validation_TFIDF_LR_predictions.csv", False),
    (RESULTS / "validation_TFIDF_SVM_predictions.csv", False),
]

for path, full in csv_specs:
    inspect_csv(path, full=full)

json_paths = [
    DATA / "final_cleaning_grouping_policy.json",
    DATA / "near_duplicate_screen_settings.json",
    DATA / "constrained_split_metadata.json",
    DATA / "corrected_split_metadata.json",
]

for path in json_paths:
    inspect_json(path)

print("\nInspection complete. No source files were modified.")

In [ ]:
from IPython.utils.capture import capture_output

with capture_output() as captured:
    print("COMMENT 5: SOURCE AND SCHEMA INSPECTION")
    print("Protocol version:", protocol.get("protocol_version"))
    print("Frozen split sizes:", protocol.get("split_sizes"))
    print("Frozen classes:", protocol.get("classes"))
    print("Expected main manifest hash:", protocol.get("main_manifest_sha256"))

    for name in manifest_names:
        path = DATA / name
        if path.exists():
            print("\nManifest:", name)
            print("Matches frozen main manifest:",
                  sha256(path) == protocol.get("main_manifest_sha256"))

    for path, full in csv_specs:
        inspect_csv(path, full=full)

    for path in json_paths:
        inspect_json(path)

output_text = captured.stdout
output_path = Path(_crus_local_path('runtime', 'comment5_inspection_output.txt'))
output_path.write_text(output_text, encoding="utf-8")

print(output_text)
print("\nInspection output saved to:", output_path)

In [ ]:
pass  # Colab service not required locally.
_crus_show_file(_crus_local_path('runtime', 'comment5_inspection_output.txt'))

In [ ]:
print("COMMENT 5 INSPECTION STARTED", flush=True)

from pathlib import Path
import hashlib
import json
import pandas as pd

ROOT = Path(_crus_local_path('project', ''))
DATA = ROOT / "data_audit"
RESULTS = ROOT / "corrected_experiment_outputs"

protocol_path = RESULTS / "final_pretest_protocol.json"

if not protocol_path.exists():
    raise FileNotFoundError(
        "Frozen protocol unavailable. Run the Google Drive setup cell first."
    )

with protocol_path.open(encoding="utf-8") as handle:
    protocol = json.load(handle)

print("\nFrozen split sizes:", protocol.get("split_sizes"), flush=True)
print("Frozen classes:", protocol.get("classes"), flush=True)

for name in [
    "constrained_split_manifest.csv",
    "corrected_split_manifest.csv",
]:
    path = DATA / name
    print("\nMANIFEST:", name, flush=True)

    if not path.exists():
        print("FILE NOT FOUND", flush=True)
        continue

    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    print(
        "Matches frozen manifest:",
        digest == protocol.get("main_manifest_sha256"),
        flush=True,
    )

    table = pd.read_csv(path)
    print("Shape:", table.shape, flush=True)
    print("Columns:", table.columns.tolist(), flush=True)
    print(table.head(3).to_string(index=False), flush=True)

for name in [
    "final_cleaned_class_counts.csv",
    "constrained_class_distribution_per_split.csv",
    "corrected_class_distribution_per_split.csv",
    "rare_class_exclusion_log.csv",
]:
    path = DATA / name
    print("\nFILE:", name, flush=True)

    if not path.exists():
        print("FILE NOT FOUND", flush=True)
        continue

    table = pd.read_csv(path)
    print("Shape:", table.shape, flush=True)
    print("Columns:", table.columns.tolist(), flush=True)
    print(table.head(25).to_string(index=False), flush=True)

print("\nCOMMENT 5 INSPECTION FINISHED", flush=True)

In [ ]:
from pathlib import Path
import hashlib
import json
import pandas as pd

ROOT = Path(_crus_local_path('project', ''))
DATA = ROOT / "data_audit"
RESULTS = ROOT / "corrected_experiment_outputs"
OUT = RESULTS / "comment5_audit"
OUT.mkdir(exist_ok=True)

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

sources = [
    DATA / "constrained_split_manifest.csv",
    DATA / "constrained_class_distribution_per_split.csv",
    DATA / "final_cleaned_class_counts.csv",
    DATA / "rare_class_exclusion_log.csv",
    RESULTS / "final_pretest_protocol.json",
    RESULTS / "test_all_method_predictions.csv",
]

before = {str(path): sha256(path) for path in sources}

with (RESULTS / "final_pretest_protocol.json").open(
    encoding="utf-8"
) as handle:
    protocol = json.load(handle)

manifest_path = DATA / "constrained_split_manifest.csv"
assert sha256(manifest_path) == protocol["main_manifest_sha256"]

manifest = pd.read_csv(
    manifest_path, dtype={"record_id": "string"}
)
pred = pd.read_csv(
    RESULTS / "test_all_method_predictions.csv",
    dtype={"record_id": "string"},
)
excluded = pd.read_csv(
    DATA / "rare_class_exclusion_log.csv",
    dtype={"record_id": "string"},
)

required = [
    "record_id", "label", "dialogue_hash", "group_id", "split"
]
assert not manifest[required].isna().any().any()
assert not manifest["record_id"].duplicated().any()
assert set(manifest["split"]) == {"train", "validation", "test"}
assert set(manifest["label"]) == set(protocol["classes"])

actual_sizes = manifest["split"].value_counts().to_dict()
assert actual_sizes == protocol["split_sizes"]

counts = pd.crosstab(
    manifest["label"], manifest["split"]
).reindex(
    index=protocol["classes"],
    columns=["train", "validation", "test"],
    fill_value=0,
)
counts["total"] = counts.sum(axis=1)
counts.index.name = "label"

assert counts["total"].ge(5).all()
assert int(counts["total"].sum()) == len(manifest)

saved_counts = pd.read_csv(
    DATA / "constrained_class_distribution_per_split.csv"
).set_index("label")

pd.testing.assert_frame_equal(
    counts.sort_index(),
    saved_counts[counts.columns].sort_index(),
    check_dtype=False,
    check_names=False,
)

saved_totals = pd.read_csv(
    DATA / "final_cleaned_class_counts.csv"
).set_index("label")["count"]

pd.testing.assert_series_equal(
    counts["total"].sort_index(),
    saved_totals.sort_index(),
    check_dtype=False,
    check_names=False,
)

assert not excluded["record_id"].duplicated().any()
assert set(excluded["record_id"]).isdisjoint(manifest["record_id"])
assert set(excluded["label"]).isdisjoint(manifest["label"])

excluded_counts = excluded.groupby("label").size()
assert excluded_counts.lt(5).all()

# Check overlap using the saved identifiers and grouping.
overlap_rows = []
for key in ["record_id", "dialogue_hash", "group_id"]:
    split_counts = manifest.groupby(key)["split"].nunique()
    crossing = split_counts[split_counts.gt(1)]

    overlap_rows.append({
        "identifier": key,
        "n_distinct_values": int(len(split_counts)),
        "n_values_crossing_splits": int(len(crossing)),
    })

    if len(crossing):
        manifest.loc[manifest[key].isin(crossing.index)].to_csv(
            OUT / f"comment5_cross_split_{key}.csv",
            index=False,
        )

overlap = pd.DataFrame(overlap_rows)

# Verify all nine methods against the frozen test manifest.
expected_methods = {
    "TFIDF_LR", "TFIDF_SVM", "BM25", "SBERT", "SK4",
    "SingleStageFull", "SingleStageTruncated",
    "Cascade", "LengthNormLegacy",
}
assert set(pred["method"]) == expected_methods
assert not pred[
    ["record_id", "method", "true_label", "prediction"]
].isna().any().any()
assert not pred.duplicated(["method", "record_id"]).any()

test_labels = manifest.loc[
    manifest["split"].eq("test")
].set_index("record_id")["label"]

method_rows = []
for method, part in pred.groupby("method", sort=True):
    assert set(part["record_id"]) == set(test_labels.index)

    expected = part["record_id"].map(test_labels)
    assert part["true_label"].eq(expected).all()

    method_rows.append({
        "method": method,
        "n_test_records": len(part),
        "identical_test_record_set": True,
        "labels_match_frozen_manifest": True,
        "n_correct_forced": int(
            part["prediction"].eq(part["true_label"]).sum()
        ),
    })

method_audit = pd.DataFrame(method_rows)

counts.reset_index().to_csv(
    OUT / "comment5_verified_class_distribution.csv", index=False
)
overlap.to_csv(
    OUT / "comment5_cross_split_overlap_summary.csv", index=False
)
method_audit.to_csv(
    OUT / "comment5_test_method_consistency.csv", index=False
)
excluded_counts.rename("n_excluded").to_csv(
    OUT / "comment5_excluded_class_counts.csv"
)

# Generate Table 2 directly from the final manifest.
def escape_latex(text):
    return str(text).replace("_", r"\_").replace("&", r"\&")

lines = [
    r"\begin{table}[ht]",
    r"\centering",
    r"\small",
    (
        r"\caption{Class distribution in the final frozen cohort. "
        r"The minimum-five criterion applies to the overall class "
        r"count after exact deduplication and before splitting, "
        r"not separately to each split.}"
    ),
    r"% Preserve the existing Table 2 label below.",
    r"\begin{tabular}{lrrrr}",
    r"\toprule",
    r"Class & Train & Validation & Test & Total \\",
    r"\midrule",
]

for label, row in counts.iterrows():
    lines.append(
        f"{escape_latex(label)} & "
        f"{int(row['train'])} & {int(row['validation'])} & "
        f"{int(row['test'])} & {int(row['total'])} " + r"\\"
    )

totals = counts.sum()
lines += [
    r"\midrule",
    (
        f"Total & {int(totals['train'])} & "
        f"{int(totals['validation'])} & "
        f"{int(totals['test'])} & {int(totals['total'])} "
        + r"\\"
    ),
    r"\bottomrule",
    r"\end{tabular}",
    r"\end{table}",
]
(OUT / "comment5_table2.tex").write_text(
    "\n".join(lines), encoding="utf-8"
)

assert all(
    sha256(Path(path)) == digest for path, digest in before.items()
)

print("COMMENT 5: CLASS COUNTS AND TEST METHOD AUDIT PASSED")
print("Frozen manifest hash verified.")
print("\nRegenerated class distribution:")
print(counts.to_string())
print("\nExcluded class counts:")
print(excluded_counts.to_string())
print("\nCross-split overlap checks:")
print(overlap.to_string(index=False))
print("\nTest method consistency:")
print(method_audit.to_string(index=False))
print("\nOriginal audited input files unchanged.")

# Inspect evidence before writing the duplicate-screening explanation.
print("\nDUPLICATE / RELATED-RECORD EVIDENCE")

for name in [
    "final_cleaning_grouping_policy.json",
    "near_duplicate_screen_settings.json",
    "constrained_split_metadata.json",
]:
    path = DATA / name
    print("\nFILE:", name)
    if path.exists():
        with path.open(encoding="utf-8") as handle:
            print(json.dumps(json.load(handle), indent=2))
    else:
        print("FILE NOT FOUND")

for name in [
    "exact_duplicate_removal_log.csv",
    "exact_dialogue_match_groups.csv",
    "near_duplicate_candidates_for_review.csv",
    "near_duplicate_review_decisions.csv",
    "retained_conflicting_label_groups.csv",
]:
    path = DATA / name
    print("\nFILE:", name)
    if path.exists():
        table = pd.read_csv(path)
        print("Shape:", table.shape)
        print("Columns:", table.columns.tolist())
        print(table.head(5).to_string(index=False))
    else:
        print("FILE NOT FOUND")

if overlap["n_values_crossing_splits"].gt(0).any():
    print("\nWARNING: Cross-split overlaps found; investigate saved reports.")
else:
    print("\nNo saved record IDs, dialogue hashes, or groups cross splits.")

print("\nOutputs saved:", OUT)
print(
    "\nScope: method consistency verified for test predictions. "
    "Training/validation usage and reviewed related-record handling "
    "still require the corresponding saved evidence."
)

In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path(_crus_local_path('project', ''))
DATA = ROOT / "data_audit"
OUT = ROOT / "corrected_experiment_outputs/comment5_audit"

manifest = pd.read_csv(
    DATA / "constrained_split_manifest.csv",
    dtype={"record_id": "string"},
).set_index("record_id")

reviews = pd.read_csv(
    DATA / "near_duplicate_review_decisions.csv",
    dtype={"record_id_a": "string", "record_id_b": "string"},
)

conflicts = pd.read_csv(
    DATA / "retained_conflicting_label_groups.csv",
    dtype={"record_id": "string"},
)

rows = []

for _, pair in reviews.iterrows():
    assert pair["decision"] == "retain_both_group_together"

    a = pair["record_id_a"]
    b = pair["record_id_b"]
    assert a in manifest.index and b in manifest.index

    same_group = manifest.loc[a, "group_id"] == manifest.loc[b, "group_id"]
    same_split = manifest.loc[a, "split"] == manifest.loc[b, "split"]

    assert same_group, f"Reviewed pair has different groups: {a}, {b}"
    assert same_split, f"Reviewed pair crosses splits: {a}, {b}"

    rows.append({
        "record_id_a": a,
        "record_id_b": b,
        "same_group": same_group,
        "same_split": same_split,
        "split": manifest.loc[a, "split"],
    })

assert conflicts["record_id"].isin(manifest.index).all()

for _, group in conflicts.groupby("dialogue_hash"):
    assigned = manifest.loc[group["record_id"].tolist()]
    assert assigned["group_id"].nunique() == 1
    assert assigned["split"].nunique() == 1

    for _, record in group.iterrows():
        actual = manifest.loc[record["record_id"]]
        assert actual["label"] == record["label"]
        assert actual["dialogue_hash"] == record["dialogue_hash"]
        assert actual["group_id"] == record["group_id"]

pair_audit = pd.DataFrame(rows)
pair_audit.to_csv(
    OUT / "comment5_verified_reviewed_pair_assignments.csv",
    index=False,
)

print("Reviewed near-duplicate pairs: grouping and split checks passed.")
print(pair_audit.to_string(index=False))
print("Retained conflicting-label records: manifest checks passed.")

print("\nValidation files still requiring schema inspection:")
RESULTS = ROOT / "corrected_experiment_outputs"

for name in [
    "validation_retrieval_prediction_audit.csv",
    "validation_TFIDF_LR_predictions.csv",
    "validation_TFIDF_SVM_predictions.csv",
]:
    path = RESULTS / name
    print("\nFILE:", name)

    if path.exists():
        table = pd.read_csv(path)
        print("Shape:", table.shape)
        print("Columns:", table.columns.tolist())
        print(table.head(2).to_string(index=False))
    else:
        print("FILE NOT FOUND")

In [ ]:
from pathlib import Path
import hashlib
import json
import pandas as pd

ROOT = Path(_crus_local_path('project', ''))
DATA = ROOT / "data_audit"
RESULTS = ROOT / "corrected_experiment_outputs"
OUT = RESULTS / "comment5_audit"
OUT.mkdir(exist_ok=True)

paths = [
    DATA / "constrained_split_manifest.csv",
    RESULTS / "final_pretest_protocol.json",
    RESULTS / "validation_retrieval_prediction_audit.csv",
    RESULTS / "validation_TFIDF_LR_predictions.csv",
    RESULTS / "validation_TFIDF_SVM_predictions.csv",
]

def digest(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

before = {str(path): digest(path) for path in paths}

with paths[1].open(encoding="utf-8") as handle:
    protocol = json.load(handle)

assert digest(paths[0]) == protocol["main_manifest_sha256"]

manifest = pd.read_csv(paths[0], dtype={"record_id": "string"})
validation_labels = manifest.loc[
    manifest["split"].eq("validation")
].set_index("record_id")["label"]

training_ids = set(
    manifest.loc[manifest["split"].eq("train"), "record_id"]
)

retrieval = pd.read_csv(paths[2], dtype={"record_id": "string"})

expected_retrieval_methods = {
    "BM25", "SBERT", "SK4", "SingleStageFull",
    "SingleStageTruncated", "Cascade", "LengthNormLegacy",
}
assert set(retrieval["method"]) == expected_retrieval_methods

lr = pd.read_csv(paths[3], dtype={"record_id": "string"})
lr["method"] = "TFIDF_LR"

svm = pd.read_csv(paths[4], dtype={"record_id": "string"})
svm["method"] = "TFIDF_SVM"

combined = pd.concat([retrieval, lr, svm], ignore_index=True)

assert not combined[
    ["record_id", "method", "true_label", "prediction"]
].isna().any().any()
assert not combined.duplicated(["method", "record_id"]).any()

rows = []

for method, part in combined.groupby("method", sort=True):
    assert set(part["record_id"]) == set(validation_labels.index)
    expected_labels = part["record_id"].map(validation_labels)
    assert part["true_label"].eq(expected_labels).all()

    rows.append({
        "method": method,
        "n_validation_records": len(part),
        "identical_validation_record_set": True,
        "labels_match_frozen_manifest": True,
    })

reference_rows = []

for method, part in retrieval.groupby("method", sort=True):
    n_references = 0
    unique_references = set()

    for value in part["retrieved_training_ids"]:
        if pd.isna(value):
            raise ValueError(f"{method}: missing retrieved_training_ids")

        ids = json.loads(value)
        if not isinstance(ids, list):
            raise ValueError(f"{method}: retrieved IDs are not a list")

        invalid = set(ids) - training_ids
        assert not invalid, (
            f"{method}: retrieved IDs outside training partition: {invalid}"
        )

        n_references += len(ids)
        unique_references.update(ids)

    reference_rows.append({
        "method": method,
        "n_recorded_retrieval_references": n_references,
        "n_unique_training_records_referenced": len(unique_references),
        "all_recorded_references_in_training_split": True,
    })

validation_audit = pd.DataFrame(rows)
reference_audit = pd.DataFrame(reference_rows)

validation_audit.to_csv(
    OUT / "comment5_validation_method_consistency.csv", index=False
)
reference_audit.to_csv(
    OUT / "comment5_validation_retrieval_reference_check.csv", index=False
)

assert all(
    digest(Path(path)) == saved_hash
    for path, saved_hash in before.items()
)

print("COMMENT 5 VALIDATION CONSISTENCY AUDIT PASSED")
print(validation_audit.to_string(index=False))

print("\nRecorded retrieval references:")
print(reference_audit.to_string(index=False))

print("\nAll audited input files unchanged.")
print(
    "\nScope: validation records and labels are verified. "
    "Recorded retrieval references belong to the training split. "
    "This does not independently verify complete retrieval-index "
    "membership or supervised-model fitting inputs."
)

**comment 6**

In [ ]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path(_crus_local_path('project', ''))
RESULTS = ROOT / "corrected_experiment_outputs"

protocol_path = RESULTS / "final_pretest_protocol.json"

if not protocol_path.exists():
    pass  # Colab service not required locally.
    _crus_prepare_saved_workspace()

with protocol_path.open(encoding="utf-8") as handle:
    protocol = json.load(handle)

print("COMMENT 6: SAVED POLICY AND RANKING INSPECTION")
print("\nProtocol creation time:", protocol.get("created_utc"))
print("\nUncertainty configuration:")
print(json.dumps(
    protocol.get("uncertainty_configuration"),
    indent=2,
))

print("\nFrozen Cascade policies:")
for name, policy in protocol.get("frozen_policies", {}).items():
    if name.startswith("Cascade:"):
        print(name)
        print(json.dumps(policy, indent=2))

files = [
    "test_cascade_frozen_uncertainty_policies.csv",
    "test_cascade_uncertainty_policy_audit.csv",
    "test_cascade_matched_uncertainty_audit.csv",
    "test_cascade_matched_uncertainty_rankings.csv",
    "test_frozen_CRUS_acceptance_audit.csv",
    "validation_retrieval_prediction_audit.csv",
]

for name in files:
    path = RESULTS / name
    print("\n" + "=" * 78)
    print(name)

    if not path.exists():
        print("FILE NOT FOUND")
        continue

    table = pd.read_csv(path)
    print("Shape:", table.shape)
    print("Columns:", table.columns.tolist())

    if len(table) <= 40:
        print(table.to_string(index=False))
    else:
        print(table.head(4).to_string(index=False))

    for column in [
        "signal", "policy", "uncertainty_signal",
        "target_n_accepted", "selection_rule",
    ]:
        if column in table.columns:
            print(
                f"{column} unique values:",
                table[column].dropna().unique().tolist(),
            )

print("\nInspection complete. No source files were modified.")

In [ ]:
from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd

BASE = Path(
    _crus_local_path('project', 'corrected_experiment_outputs')
)
OUT = BASE / "comment6_audit"
OUT.mkdir(exist_ok=True)

names = [
    "final_pretest_protocol.json",
    "test_all_method_predictions.csv",
    "test_cascade_frozen_uncertainty_policies.csv",
    "test_cascade_uncertainty_policy_audit.csv",
    "test_cascade_matched_uncertainty_audit.csv",
    "test_cascade_matched_uncertainty_rankings.csv",
    "validation_retrieval_prediction_audit.csv",
]

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

before = {name: sha256(BASE / name) for name in names}

with (BASE / names[0]).open(encoding="utf-8") as handle:
    protocol = json.load(handle)

def read(name):
    return pd.read_csv(BASE / name, dtype={"record_id": "string"})

def boolean(series):
    normalized = series.astype(str).str.strip().str.lower()
    mapping = {"true": True, "false": False, "1": True, "0": False}
    assert normalized.isin(mapping).all(), series.name
    return normalized.map(mapping).astype(bool)

pred = read("test_all_method_predictions.csv")
pred = pred.loc[pred["method"].eq("Cascade")].copy()
assert not pred["record_id"].duplicated().any()
assert not pred[
    ["record_id", "true_label", "prediction", "confidence"]
].isna().any().any()

N = int(protocol["split_sizes"]["test"])
assert len(pred) == N
pred["recomputed_correct"] = pred["true_label"].eq(pred["prediction"])
assert boolean(pred["correct"]).eq(pred["recomputed_correct"]).all()
reference = pred.set_index("record_id")

native_audit = read("test_cascade_uncertainty_policy_audit.csv")
matched_audit = read("test_cascade_matched_uncertainty_audit.csv")
native_saved = read("test_cascade_frozen_uncertainty_policies.csv")
matched_saved = read("test_cascade_matched_uncertainty_rankings.csv")

signals = {
    "CRUS", "H_pred", "H_retrieval", "V_RRF",
    "OneMinusMaxProb", "OneMinusMargin",
}

for table in [native_audit, matched_audit]:
    assert set(table["uncertainty_signal"]) == signals
    assert not table.duplicated(
        ["record_id", "uncertainty_signal"]
    ).any()

for table in [native_saved, matched_saved]:
    assert set(table["uncertainty_signal"]) == signals
    assert not table["uncertainty_signal"].duplicated().any()

assert protocol["ranking_ties"] == "score descending; record ID ascending"

native_rows = []
matched_rows = []
ranked_parts = []

def metrics(correct, accepted):
    accepted = np.asarray(accepted, dtype=bool)
    correct = np.asarray(correct, dtype=bool)
    a = int(accepted.sum())
    c = int((accepted & correct).sum())
    e = a - c
    return {
        "N": N,
        "n_accepted": a,
        "n_correct_accepted": c,
        "n_errors_accepted": e,
        "n_abstained": N - a,
        "coverage": a / N,
        "selective_accuracy": c / a if a else np.nan,
        "accepted_misclassification_rate": e / a if a else np.nan,
        "abstention_penalized_accuracy": c / N,
        "abstention_penalized_error": (N - c) / N,
        "forced_accuracy": correct.mean(),
    }

def verify_summary(recomputed, saved):
    for key, value in recomputed.items():
        np.testing.assert_allclose(
            value, saved[key], rtol=1e-9, atol=1e-12
        )

for signal in sorted(signals):
    part = native_audit.loc[
        native_audit["uncertainty_signal"].eq(signal)
    ].copy()

    assert len(part) == N
    assert set(part["record_id"]) == set(reference.index)
    assert not part[
        ["uncertainty", "confidence", "tau_c", "tau_u"]
    ].isna().any().any()
    assert np.isfinite(part["uncertainty"]).all()

    correct = part["record_id"].map(reference["recomputed_correct"])
    assert boolean(part["correct"]).eq(correct).all()

    np.testing.assert_allclose(
        part["confidence"],
        part["record_id"].map(reference["confidence"]),
        rtol=1e-9, atol=1e-12,
    )

    policy = protocol["frozen_policies"][f"Cascade:{signal}"]
    np.testing.assert_allclose(
        part["tau_c"], policy["tau_c"], rtol=1e-9, atol=1e-12
    )
    np.testing.assert_allclose(
        part["tau_u"], policy["tau_u"], rtol=1e-9, atol=1e-12
    )

    accepted = (
        part["confidence"].ge(policy["tau_c"])
        & part["uncertainty"].le(policy["tau_u"])
    )
    assert boolean(part["accepted"]).eq(accepted).all()

    row = metrics(correct, accepted)
    old = native_saved.loc[
        native_saved["uncertainty_signal"].eq(signal)
    ].iloc[0]
    verify_summary(row, old)

    np.testing.assert_allclose(
        [old["tau_c"], old["tau_u"]],
        [policy["tau_c"], policy["tau_u"]],
        rtol=1e-9, atol=1e-12,
    )

    native_rows.append({
        "uncertainty_signal": signal,
        "tau_c": policy["tau_c"],
        "tau_u": policy["tau_u"],
        **row,
    })

    matched = matched_audit.loc[
        matched_audit["uncertainty_signal"].eq(signal)
    ].copy()
    assert len(matched) == N
    assert set(matched["record_id"]) == set(reference.index)
    assert matched["target_n_accepted"].nunique() == 1
    k = int(matched["target_n_accepted"].iloc[0])

    uncertainty_map = part.set_index("record_id")["uncertainty"]
    np.testing.assert_allclose(
        matched["uncertainty"],
        matched["record_id"].map(uncertainty_map),
        rtol=1e-9, atol=1e-12,
    )

    matched = matched.sort_values(
        ["uncertainty", "record_id"],
        ascending=[True, True],
        kind="mergesort",
    ).reset_index(drop=True)

    matched["rank"] = np.arange(1, N + 1)
    reconstructed = matched["rank"].le(k)
    assert boolean(matched["accepted"]).eq(reconstructed).all()

    matched_correct = matched["record_id"].map(
        reference["recomputed_correct"]
    )
    assert boolean(matched["correct"]).eq(matched_correct).all()

    row = metrics(matched_correct, reconstructed)
    old = matched_saved.loc[
        matched_saved["uncertainty_signal"].eq(signal)
    ].iloc[0]
    verify_summary(row, old)

    boundary = matched.loc[k - 1, "uncertainty"]
    tied_total = int(matched["uncertainty"].eq(boundary).sum())
    tied_retained = int(
        matched.iloc[:k]["uncertainty"].eq(boundary).sum()
    )

    matched_rows.append({
        "uncertainty_signal": signal,
        "boundary_tie_split": tied_retained < tied_total,
        **row,
    })
    ranked_parts.append(matched)

native_verified = pd.DataFrame(native_rows)
matched_verified = pd.DataFrame(matched_rows)

crus = matched_verified.loc[
    matched_verified["uncertainty_signal"].eq("CRUS")
].iloc[0]
entropy = matched_verified.loc[
    matched_verified["uncertainty_signal"].eq("H_pred")
].iloc[0]

assert crus["n_accepted"] == entropy["n_accepted"]

# Check the proposed component mapping against saved validation values.
validation = read("validation_retrieval_prediction_audit.csv")
validation = validation.loc[
    validation["method"].eq("Cascade")
].copy()

components = ["H_pred", "H_retrieval", "V_RRF"]
assert not validation[components + ["CRUS"]].isna().any().any()
weights = np.asarray(
    protocol["uncertainty_configuration"]["weights"], dtype=float
)

formula_value = validation[components].to_numpy() @ weights
np.testing.assert_allclose(
    formula_value, validation["CRUS"],
    rtol=1e-8, atol=1e-10,
)

# Cross-check the same component mapping on saved test signal values.
wide = native_audit.pivot(
    index="record_id",
    columns="uncertainty_signal",
    values="uncertainty",
)
np.testing.assert_allclose(
    wide[components].to_numpy() @ weights,
    wide["CRUS"],
    rtol=1e-8, atol=1e-10,
)

native_verified.to_csv(
    OUT / "comment6_verified_native_policies.csv", index=False
)
matched_verified.to_csv(
    OUT / "comment6_verified_matched_uncertainty.csv", index=False
)
pd.concat(ranked_parts, ignore_index=True).to_csv(
    OUT / "comment6_verified_signal_rankings.csv", index=False
)

assert all(
    sha256(BASE / name) == value for name, value in before.items()
)

print("COMMENT 6 AUDIT PASSED")
print("\nFrozen native policies:")
print(native_verified.to_string(index=False))
print("\nMatched-coverage uncertainty rankings:")
print(matched_verified.to_string(index=False))
print("\nCRUS component mapping verified on validation and test values:")
print(dict(zip(components, weights)))

print(
    "\nCRUS minus predictive entropy, correct retained:",
    int(crus["n_correct_accepted"] - entropy["n_correct_accepted"]),
)
print(
    "Selective accuracy difference, percentage points:",
    100 * (crus["selective_accuracy"] - entropy["selective_accuracy"]),
)

crus_native_ids = set(native_audit.loc[
    native_audit["uncertainty_signal"].eq("CRUS")
    & boolean(native_audit["accepted"]),
    "record_id",
])
crus_ranked_ids = set(matched_audit.loc[
    matched_audit["uncertainty_signal"].eq("CRUS")
    & boolean(matched_audit["accepted"]),
    "record_id",
])
print(
    "Native CRUS and matched lowest-CRUS selections identical:",
    crus_native_ids == crus_ranked_ids,
)

print("\nOriginal source files unchanged.")
print("Audit outputs:", OUT)
print(
    "\nWeight provenance limitation: the protocol records fixed "
    "heuristic weights and a creation time. This numerical audit "
    "does not independently establish the full chronology of "
    "weight selection and test access."
)

In [ ]:
from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

BASE = Path(
    _crus_local_path('project', 'corrected_experiment_outputs')
)

if not (BASE / "final_pretest_protocol.json").exists():
    pass  # Colab service not required locally.
    _crus_prepare_saved_workspace()

OUT = BASE / "comment6_audit"
OUT.mkdir(exist_ok=True)

protocol_path = BASE / "final_pretest_protocol.json"
audit_path = BASE / "test_cascade_uncertainty_policy_audit.csv"

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

original_hashes = {
    path: sha256(path)
    for path in [protocol_path, audit_path]
}

with protocol_path.open(encoding="utf-8") as handle:
    protocol = json.load(handle)

policy = protocol["frozen_policies"]["Cascade:CRUS"]
tau_c = float(policy["tau_c"])
tau_u = float(policy["tau_u"])
N = int(protocol["split_sizes"]["test"])

table = pd.read_csv(audit_path, dtype={"record_id": "string"})
crus = table.loc[
    table["uncertainty_signal"].eq("CRUS")
].copy()

assert len(crus) == N == 257
assert not crus["record_id"].duplicated().any()

for column in ["uncertainty", "confidence", "tau_c", "tau_u"]:
    crus[column] = pd.to_numeric(crus[column], errors="raise")
    assert np.isfinite(crus[column]).all()

np.testing.assert_allclose(crus["tau_c"], tau_c)
np.testing.assert_allclose(crus["tau_u"], tau_u)

mapping = {"true": True, "false": False, "1": True, "0": False}
accepted_text = crus["accepted"].astype(str).str.strip().str.lower()
assert accepted_text.isin(mapping).all()
saved_accepted = accepted_text.map(mapping).astype(bool)

accepted = (
    crus["confidence"].ge(tau_c)
    & crus["uncertainty"].le(tau_u)
)

assert accepted.eq(saved_accepted).all()
assert int(accepted.sum()) == 117
assert int((~accepted).sum()) == 140
assert tau_c == 0.0

accepted_scores = crus.loc[accepted, "uncertainty"].to_numpy()
abstained_scores = crus.loc[~accepted, "uncertainty"].to_numpy()

# Shared bin edges, including the threshold as an exact edge.
scores = crus["uncertainty"].to_numpy()
low = min(0.0, float(scores.min()))
high = max(float(scores.max()), tau_u)
high += max((high - low) * 0.02, 1e-6)

bins = np.unique(
    np.append(np.linspace(low, high, 31), tau_u)
)

fig, ax = plt.subplots(figsize=(9, 4.8))

ax.hist(
    accepted_scores,
    bins=bins,
    color="#56A6E8",
    alpha=0.85,
    label="Accepted (n = 117)",
    edgecolor="white",
    linewidth=0.4,
)
ax.hist(
    abstained_scores,
    bins=bins,
    color="#EF746C",
    alpha=0.85,
    label="Abstained (n = 140)",
    edgecolor="white",
    linewidth=0.4,
)
ax.axvline(
    tau_u,
    color="black",
    linestyle="--",
    linewidth=1.8,
    label=rf"$\tau_u = {tau_u:.6f}$",
)

ax.set_title("Cascade–CRUS: frozen test acceptance policy", fontsize=13)
ax.set_xlabel("CRUS score", fontsize=11)
ax.set_ylabel("Number of records", fontsize=11)
ax.set_xlim(low, high)
ax.legend(fontsize=9, frameon=True)
ax.grid(axis="y", alpha=0.2)
ax.set_axisbelow(True)
fig.tight_layout()

pdf_path = OUT / "comment6_verified_crus_distribution.pdf"
png_path = OUT / "comment6_verified_crus_distribution.png"

fig.savefig(pdf_path, bbox_inches="tight")
fig.savefig(png_path, dpi=300, bbox_inches="tight")
plt.show()
plt.close(fig)

assert pdf_path.exists() and pdf_path.stat().st_size > 0
assert png_path.exists() and png_path.stat().st_size > 0

assert all(
    sha256(path) == original_hash
    for path, original_hash in original_hashes.items()
)

print("Corrected Figure 5 generated.")
print("Accepted:", len(accepted_scores))
print("Abstained:", len(abstained_scores))
print("Full-precision CRUS threshold:", tau_u)
print("PDF:", pdf_path)
print("PNG:", png_path)
print("Original source files unchanged.")

# **Comment 7**

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score

# Set these from your FINAL corrected experiment.
# Do not use an older notebook's arrays unless they are verified final outputs.
LABELS = None
METHODS = None

# Required example structure:
#
# LABELS = list(final_label_order)
#
# METHODS = {
#     "Cascade–CRUS": {
#         "y_true": final_true_labels,
#         "y_pred": final_cascade_predictions,
#         "accepted": final_saved_acceptance_mask,
#         "probabilities": final_cascade_probability_matrix,
#         "probability_labels": list(final_probability_column_order),
#     },
#     "BM25-only": {
#         "y_true": final_true_labels,
#         "y_pred": final_bm25_predictions,
#         "accepted": np.ones(len(final_true_labels), dtype=bool),
#         "probabilities": None,
#         "probability_labels": None,
#     },
# }
#
# Add every evaluated method using the same structure.
# Non-abstaining methods have an all-True acceptance mask.

OUT = Path("comment7_audit")
OUT.mkdir(parents=True, exist_ok=True)

BOOTSTRAP_N = 2000
BOOTSTRAP_SEED = 42
ECE_BINS = 10
NLL_EPS = 1e-15

if LABELS is None or METHODS is None:
    raise ValueError(
        "Populate LABELS and METHODS from the final corrected "
        "experiment before running this script."
    )

labels = np.asarray(LABELS, dtype=str)
assert len(labels) > 1
assert len(set(labels)) == len(labels)
label_index = {label: i for i, label in enumerate(labels)}


def parse_acceptance(values, n):
    values = np.asarray(values)
    assert values.shape == (n,)
    if values.dtype == bool:
        return values.copy()

    text = pd.Series(values).astype(str).str.strip().str.lower()
    mapping = {"true": True, "false": False, "1": True, "0": False}
    if not text.isin(mapping).all():
        raise ValueError("Acceptance mask must contain Boolean or 0/1 values.")
    return text.map(mapping).to_numpy(dtype=bool)


def align_probabilities(matrix, column_labels, n):
    if matrix is None:
        return None

    p = np.asarray(matrix, dtype=float)
    cols = list(map(str, column_labels))

    assert len(cols) == len(set(cols))
    assert set(cols) == set(labels), (
        "Probability columns must cover exactly the frozen label set."
    )
    assert p.shape == (n, len(cols))
    assert np.isfinite(p).all()
    assert ((p >= 0) & (p <= 1)).all()
    np.testing.assert_allclose(
        p.sum(axis=1), 1.0, rtol=1e-6, atol=1e-8
    )

    return p[:, [cols.index(label) for label in labels]]


def fixed_macro_f1(y_true, y_pred):
    return float(
        f1_score(
            y_true,
            y_pred,
            labels=labels.tolist(),
            average="macro",
            zero_division=0,
        )
    )


def bootstrap_macro_f1(y_true, y_pred):
    n = len(y_true)
    if n == 0:
        return np.nan, np.nan

    rng = np.random.default_rng(BOOTSTRAP_SEED)
    scores = np.empty(BOOTSTRAP_N)

    for b in range(BOOTSTRAP_N):
        idx = rng.integers(0, n, size=n)
        scores[b] = fixed_macro_f1(y_true[idx], y_pred[idx])

    return tuple(np.quantile(scores, [0.025, 0.975]))


def calibration_metrics(y_true, y_pred, probabilities):
    confidence = probabilities.max(axis=1)
    probability_prediction = labels[probabilities.argmax(axis=1)]
    assert np.array_equal(probability_prediction, y_pred), (
        "Saved predictions differ from probability argmax. "
        "Resolve probability ordering or prediction provenance."
    )

    correct = (y_pred == y_true).astype(float)
    bins = np.minimum(
        (confidence * ECE_BINS).astype(int), ECE_BINS - 1
    )

    rows = []
    ece = 0.0

    for b in range(ECE_BINS):
        mask = bins == b
        count = int(mask.sum())
        accuracy = float(correct[mask].mean()) if count else np.nan
        mean_conf = (
            float(confidence[mask].mean()) if count else np.nan
        )
        gap = abs(accuracy - mean_conf) if count else np.nan

        if count:
            ece += count / len(y_true) * gap

        rows.append({
            "bin": b,
            "lower": b / ECE_BINS,
            "upper": (b + 1) / ECE_BINS,
            "upper_inclusive": b == ECE_BINS - 1,
            "count": count,
            "accuracy": accuracy,
            "mean_confidence": mean_conf,
            "absolute_gap": gap,
        })

    true_idx = np.array([label_index[x] for x in y_true])
    one_hot = np.eye(len(labels))[true_idx]

    brier = float(
        np.mean(np.sum((probabilities - one_hot) ** 2, axis=1))
    )
    nll = float(
        -np.mean(
            np.log(
                np.clip(
                    probabilities[np.arange(len(y_true)), true_idx],
                    NLL_EPS,
                    1.0,
                )
            )
        )
    )

    return ece, brier, nll, rows


metric_rows = []
bin_rows = []
class_rows = []
reference_y = None

for method, data in METHODS.items():
    y_true = np.asarray(data["y_true"], dtype=str)
    y_pred = np.asarray(data["y_pred"], dtype=str)

    assert y_true.ndim == y_pred.ndim == 1
    assert len(y_true) == len(y_pred) > 0
    assert set(y_true).issubset(label_index)
    assert set(y_pred).issubset(label_index)

    if reference_y is None:
        reference_y = y_true.copy()
    else:
        assert np.array_equal(reference_y, y_true), (
            "Methods must use the same test records in the same order."
        )

    accepted = parse_acceptance(data["accepted"], len(y_true))
    p = align_probabilities(
        data.get("probabilities"),
        data.get("probability_labels"),
        len(y_true),
    )

    for subset, mask in [
        ("pre_gating_all_test", np.ones(len(y_true), dtype=bool)),
        ("accepted_subset", accepted),
    ]:
        yt, yp = y_true[mask], y_pred[mask]
        n = len(yt)

        row = {
            "method": method,
            "subset": subset,
            "n": n,
            "test_n": len(y_true),
            "accepted_n": int(accepted.sum()),
            "coverage": float(accepted.mean()),
            "label_count": len(labels),
            "accuracy": float((yt == yp).mean()) if n else np.nan,
            "macro_f1": fixed_macro_f1(yt, yp) if n else np.nan,
            "ece": np.nan,
            "brier": np.nan,
            "nll": np.nan,
        }

        row["macro_f1_ci_low"], row["macro_f1_ci_high"] = (
            bootstrap_macro_f1(yt, yp)
        )

        if not n:
            row["calibration_status"] = "Unavailable: empty subset"
        elif p is None:
            row["calibration_status"] = (
                "Unavailable: no genuine class-probability output supplied"
            )
        else:
            ece, brier, nll, reliability = calibration_metrics(
                yt, yp, p[mask]
            )
            row.update(ece=ece, brier=brier, nll=nll)
            row["calibration_status"] = (
                "Computed from supplied class probabilities"
            )

            for entry in reliability:
                bin_rows.append({
                    "method": method,
                    "subset": subset,
                    **entry,
                })

        metric_rows.append(row)

        for label in labels:
            class_rows.append({
                "method": method,
                "subset": subset,
                "label": label,
                "true_support": int((yt == label).sum()),
                "predicted_count": int((yp == label).sum()),
                "absent_from_truth_and_predictions": bool(
                    not (yt == label).any()
                    and not (yp == label).any()
                ),
            })

metrics = pd.DataFrame(metric_rows)
metrics.to_csv(
    OUT / "comment7_metrics.csv",
    index=False,
    na_rep="NA",
)
pd.DataFrame(bin_rows).to_csv(
    OUT / "comment7_reliability_bins.csv",
    index=False,
    na_rep="NA",
)
pd.DataFrame(class_rows).to_csv(
    OUT / "comment7_label_support.csv",
    index=False,
)

metadata = {
    "labels": labels.tolist(),
    "macro_f1": {
        "label_set": "Fixed frozen experimental label set",
        "zero_division": 0,
        "absent_from_truth_and_predictions": (
            "Retained in the macro average with F1=0"
        ),
    },
    "bootstrap": {
        "replicates": BOOTSTRAP_N,
        "seed": BOOTSTRAP_SEED,
        "type": "IID record bootstrap with replacement",
        "interval": "Percentile 95%",
        "accepted_subset": (
            "Resample accepted records only; conditional on frozen selection"
        ),
        "label_set": "Unchanged in every replicate",
        "empty_subset": "Unavailable; no bootstrap performed",
    },
    "calibration": {
        "ece": "Top-label ECE using maximum class probability",
        "bins": ECE_BINS,
        "binning": "Equal width; confidence=1 included in final bin",
        "brier": "Mean sum of squared errors across all classes",
        "nll_log_stabiliser": NLL_EPS,
        "calibration_fitted_by_this_script": False,
        "missing_probabilities": "NA, not zero",
    },
}

with (OUT / "comment7_metric_definitions.json").open(
    "w", encoding="utf-8"
) as handle:
    json.dump(metadata, handle, indent=2)

print(metrics.to_string(index=False))
print("\nAudit outputs saved to:", OUT.resolve())

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

variable_rows = []

for name, value in list(globals().items()):
    if name.startswith("_"):
        continue

    if any(term in name.lower() for term in [
        "label", "class", "pred", "prob", "accept",
        "covered", "ytest", "y_test", "protocol"
    ]):
        if isinstance(value, (np.ndarray, pd.DataFrame, pd.Series)):
            variable_rows.append({
                "variable": name,
                "type": type(value).__name__,
                "shape": str(value.shape),
            })
        elif isinstance(value, (list, tuple, dict)):
            variable_rows.append({
                "variable": name,
                "type": type(value).__name__,
                "shape": f"length={len(value)}",
            })

print("Potential evaluation variables:")
print(
    pd.DataFrame(variable_rows).to_string(index=False)
    if variable_rows else "No matching variables found."
)

base = Path(
    _crus_local_path('project', 'corrected_experiment_outputs')
)

print("\nCorrected output directory exists:", base.exists())

if base.exists():
    print("\nAvailable data files:")
    for path in sorted(base.rglob("*")):
        if path.is_file() and path.suffix.lower() in {
            ".csv", ".json", ".npz", ".npy", ".parquet"
        }:
            print(path.relative_to(base))
else:
    print(
        "Mount Google Drive or set 'base' to the actual "
        "final corrected output directory."
    )

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

BASE = Path(
    _crus_local_path('project', 'corrected_experiment_outputs')
)

assert BASE.exists(), "Corrected output directory not found."

print("=== PROTOCOL STRUCTURE ===")

with (BASE / "final_pretest_protocol.json").open(
    encoding="utf-8"
) as handle:
    protocol = json.load(handle)

for key, value in protocol.items():
    if isinstance(value, dict):
        print(f"{key}: dict; keys={list(value.keys())}")
    elif isinstance(value, list):
        preview = value if len(value) <= 30 else value[:5]
        print(f"{key}: list; length={len(value)}; values={preview}")
    else:
        print(f"{key}: {value}")

print("\n=== SAVED TEST ARRAY STRUCTURES ===")

for path in sorted(BASE.glob("test_*_arrays.npz")):
    print(f"\nFILE: {path.name}")

    try:
        with np.load(path, allow_pickle=False) as arrays:
            for key in arrays.files:
                try:
                    array = arrays[key]
                    print(
                        f"  {key}: shape={array.shape}, "
                        f"dtype={array.dtype}"
                    )

                    if array.ndim == 1:
                        print("    first values:", array[:5].tolist())
                    elif array.ndim == 2:
                        print(
                            "    first row, first values:",
                            array[0, :min(5, array.shape[1])].tolist()
                            if len(array) else []
                        )

                except ValueError as error:
                    print(f"  {key}: cannot read safely: {error}")

    except Exception as error:
        print(f"  Inspection failed: {error}")

print("\n=== CSV STRUCTURES ===")

csv_names = [
    "test_all_method_predictions.csv",
    "test_frozen_CRUS_acceptance_audit.csv",
    "test_native_gate_operating_points.csv",
    "verified_main_test_results.csv",
    "supplementary_frozen_policy_intervals.csv",
]

for name in csv_names:
    path = BASE / name
    if not path.exists():
        print(f"\nMISSING: {name}")
        continue

    table = pd.read_csv(path)
    print(f"\nFILE: {name}")
    print("Shape:", table.shape)
    print("Columns:", table.columns.tolist())

    # Display schema without printing dialogue or document text.
    display_columns = [
        column for column in table.columns
        if not any(term in column.lower() for term in [
            "text", "dialogue", "query", "document"
        ])
    ]
    print(table[display_columns].head(3).to_string(index=False))

print("\n=== EXISTING BOOTSTRAP SPECIFICATION ===")

spec_path = BASE / "supplementary_group_bootstrap_spec.json"

if spec_path.exists():
    with spec_path.open(encoding="utf-8") as handle:
        bootstrap_spec = json.load(handle)
    print(json.dumps(bootstrap_spec, indent=2))
else:
    print("Bootstrap specification file not found.")

In [ ]:
from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score

BASE = Path(
    _crus_local_path('project', 'corrected_experiment_outputs')
)
OUT = BASE / "comment7_audit"
OUT.mkdir(exist_ok=True)

protocol_path = BASE / "final_pretest_protocol.json"

with protocol_path.open(encoding="utf-8") as handle:
    protocol = json.load(handle)

protocol_hash = hashlib.sha256(protocol_path.read_bytes()).hexdigest()
labels = list(map(str, protocol["classes"]))
label_index = {label: i for i, label in enumerate(labels)}

assert len(labels) == len(set(labels)) == 17

predictions = pd.read_csv(
    BASE / "test_all_method_predictions.csv",
    dtype={"record_id": str, "true_label": str, "prediction": str},
)
gates = pd.read_csv(
    BASE / "test_frozen_CRUS_acceptance_audit.csv",
    dtype={"record_id": str},
)
existing = pd.read_csv(BASE / "verified_main_test_results.csv")


def boolean_values(series):
    text = series.astype(str).str.strip().str.lower()
    mapping = {"true": True, "false": False, "1": True, "0": False}
    assert text.isin(mapping).all(), "Unexpected Boolean representation."
    return text.map(mapping).to_numpy(dtype=bool)


def evaluate(table, probabilities, mask):
    yt = table["true_label"].to_numpy()[mask]
    yp = table["prediction"].to_numpy()[mask]
    n = len(yt)

    result = {
        "n_accepted": n,
        "macro_f1": np.nan,
        "ece": np.nan,
        "brier": np.nan,
    }
    reliability = []

    if not n:
        return result, reliability

    result["macro_f1"] = float(
        f1_score(
            yt, yp,
            labels=labels,
            average="macro",
            zero_division=0,
        )
    )

    if probabilities is None:
        return result, reliability

    p = probabilities[mask]
    predicted_from_p = np.asarray(labels)[p.argmax(axis=1)]
    assert np.array_equal(predicted_from_p, yp), (
        "Probability argmax differs from saved predictions."
    )

    confidence = p.max(axis=1)
    correct = (yp == yt).astype(float)
    bin_id = np.minimum((confidence * 10).astype(int), 9)

    ece = 0.0
    for b in range(10):
        selected = bin_id == b
        count = int(selected.sum())
        acc = float(correct[selected].mean()) if count else np.nan
        conf = float(confidence[selected].mean()) if count else np.nan

        if count:
            ece += (count / n) * abs(acc - conf)

        reliability.append({
            "bin": b,
            "lower": b / 10,
            "upper": (b + 1) / 10,
            "upper_inclusive": b == 9,
            "count": count,
            "accuracy": acc,
            "mean_confidence": conf,
        })

    true_index = np.array([label_index[label] for label in yt])
    one_hot = np.eye(len(labels))[true_index]

    result["ece"] = float(ece)
    result["brier"] = float(
        np.mean(np.sum((p - one_hot) ** 2, axis=1))
    )
    return result, reliability


rows = []
bin_rows = []
support_rows = []
reference_ids = None
reference_truth = None

for method in predictions["method"].unique():
    table = predictions.loc[
        predictions["method"].eq(method)
    ].copy().reset_index(drop=True)

    assert not table["record_id"].duplicated().any()
    assert len(table) == int(protocol["split_sizes"]["test"])
    assert set(table["true_label"]).issubset(label_index)
    assert set(table["prediction"]).issubset(label_index)

    if reference_ids is None:
        reference_ids = table["record_id"].tolist()
        reference_truth = table.set_index("record_id")["true_label"]
    else:
        assert set(table["record_id"]) == set(reference_ids)
        truth = table.set_index("record_id")["true_label"]
        assert truth.reindex(reference_ids).equals(
            reference_truth.reindex(reference_ids)
        )

    path = BASE / f"test_{method}_arrays.npz"
    assert path.exists(), f"Missing arrays: {path.name}"

    with np.load(path, allow_pickle=False) as arrays:
        saved_hash = str(arrays["protocol_sha256"].item())
        assert saved_hash == protocol_hash, (
            f"Protocol hash mismatch for {method}"
        )

        column_labels = arrays["classes"].astype(str).tolist()
        array_predictions = arrays["predictions"].astype(str)

        # Numeric probability rows follow the saved NPZ prediction order.
        # Confirm that this agrees with the per-method CSV row order.
        assert np.array_equal(
            array_predictions, table["prediction"].to_numpy()
        ), f"NPZ/CSV prediction order mismatch: {method}"

        p = None
        if "probabilities" in arrays.files:
            raw_p = arrays["probabilities"].copy()
            assert set(column_labels) == set(labels)
            assert len(column_labels) == len(set(column_labels))
            p = raw_p[:, [column_labels.index(x) for x in labels]]

            assert p.shape == (len(table), len(labels))
            assert np.isfinite(p).all()
            assert ((p >= 0) & (p <= 1)).all()
            np.testing.assert_allclose(
                p.sum(axis=1), 1.0, rtol=1e-6, atol=1e-8
            )

        if method in {"BM25", "SBERT"}:
            assert p is None, (
                "Unexpected probability output for retrieval-only baseline."
            )

    modes = [
        ("forced_before_abstention", np.ones(len(table), dtype=bool))
    ]

    method_gates = gates.loc[gates["method"].eq(method)].copy()
    if len(method_gates):
        assert not method_gates["record_id"].duplicated().any()
        assert set(method_gates["record_id"]) == set(table["record_id"])

        aligned = method_gates.set_index("record_id").loc[
            table["record_id"]
        ]
        accepted = boolean_values(aligned["accepted"])

        gate_correct = boolean_values(aligned["correct"])
        assert np.array_equal(
            gate_correct,
            table["prediction"].eq(table["true_label"]).to_numpy(),
        )

        if method == "Cascade":
            assert int(accepted.sum()) == 117

        modes.append(("native_CRUS_gate", accepted))

    for mode, mask in modes:
        metrics, reliability = evaluate(table, p, mask)

        rows.append({
            "method": method,
            "mode": mode,
            **metrics,
            "calibration_status": (
                "Available: supplied class probabilities"
                if p is not None
                else "NA: no class-probability output"
            ),
        })

        for entry in reliability:
            bin_rows.append({"method": method, "mode": mode, **entry})

        selected = table.loc[mask]
        for label in labels:
            support_rows.append({
                "method": method,
                "mode": mode,
                "label": label,
                "true_support": int(
                    selected["true_label"].eq(label).sum()
                ),
                "predicted_count": int(
                    selected["prediction"].eq(label).sum()
                ),
            })

audit = pd.DataFrame(rows)

comparison = audit.merge(
    existing[
        ["method", "mode", "n_accepted", "macro_f1", "ece", "brier"]
    ],
    on=["method", "mode"],
    how="outer",
    suffixes=("_audit", "_saved"),
    indicator=True,
    validate="one_to_one",
)

for metric in ["n_accepted", "macro_f1", "ece", "brier"]:
    a = comparison[f"{metric}_audit"]
    b = comparison[f"{metric}_saved"]
    comparison[f"{metric}_matches"] = (
        np.isclose(a, b, rtol=1e-7, atol=1e-8, equal_nan=True)
        & comparison["_merge"].eq("both")
    )

audit.to_csv(
    OUT / "comment7_recomputed_metrics.csv",
    index=False, na_rep="NA",
)
comparison.to_csv(
    OUT / "comment7_comparison_with_saved_results.csv",
    index=False, na_rep="NA",
)
pd.DataFrame(bin_rows).to_csv(
    OUT / "comment7_reliability_bins.csv",
    index=False, na_rep="NA",
)
pd.DataFrame(support_rows).to_csv(
    OUT / "comment7_label_support.csv",
    index=False,
)

print("=== METRIC COMPARISON ===")
print(comparison.to_string(index=False))

print("\n=== FROZEN REPORTING RULES ===")
print(json.dumps(protocol["reporting_rules"], indent=2))

print("\n=== FROZEN UNCERTAINTY CONFIGURATION ===")
print(json.dumps(protocol["uncertainty_configuration"], indent=2))

print("\n=== FROZEN RETRIEVAL CONFIGURATION ===")
print(json.dumps(protocol["retrieval_configuration"], indent=2))

print("\nAudit files saved to:", OUT)
print("No bootstrap, model fitting, or threshold selection performed.")

In [ ]:
from pathlib import Path
import json
import pandas as pd

BASE = Path(
    _crus_local_path('project', 'corrected_experiment_outputs')
)
OUT = BASE / "comment7_audit"
OUT.mkdir(exist_ok=True)

results = pd.read_csv(BASE / "verified_main_test_results.csv")
with (BASE / "final_pretest_protocol.json").open(
    encoding="utf-8"
) as handle:
    protocol = json.load(handle)

required = [
    "method", "mode", "N", "n_accepted",
    "forced_accuracy", "selective_accuracy",
    "macro_f1", "weighted_f1", "kappa", "ece"
]
assert set(required).issubset(results.columns)
assert not results.duplicated(["method", "mode"]).any()
assert results["N"].eq(protocol["split_sizes"]["test"]).all()

names = {
    "TFIDF_LR": "TF-IDF + Logistic Regression",
    "TFIDF_SVM": "TF-IDF + Linear SVM",
    "BM25": "BM25-only",
    "SBERT": "SBERT-only",
    "SK4": "SK4 Hybrid",
    "SingleStageFull": "Single-stage full",
    "SingleStageTruncated": "Single-stage truncated",
    "Cascade": "RiskAware-Cascade",
    "LengthNormLegacy": "RiskAware-LengthNorm (legacy)",
}

order = [
    "TFIDF_LR", "TFIDF_SVM", "BM25", "SBERT", "SK4",
    "SingleStageFull", "SingleStageTruncated",
    "LengthNormLegacy", "Cascade"
]

assert set(results["method"]).issubset(names)
assert results["mode"].isin([
    "forced_before_abstention", "native_CRUS_gate"
]).all()

policy = protocol["frozen_policies"]["Cascade:CRUS"]
tau_c = float(policy["tau_c"])
tau_u = float(policy["tau_u"])
n_test = int(protocol["split_sizes"]["test"])

def fmt(value):
    return "NA" if pd.isna(value) else f"{value:.4f}"

lines = [
    r"\begin{table*}[ht]",
    r"\centering",
    (
        r"\caption{Classification and calibration performance on the "
        f"corrected MTS-Dialog test set ($n={n_test}$). "
        r"Forced rows evaluate predictions before abstention; accepted "
        r"rows evaluate records retained by each method's frozen "
        r"validation-selected CRUS policy. Accuracy denotes forced "
        r"accuracy for forced rows and selective accuracy for accepted "
        r"rows. Accepted-subset results must be interpreted together "
        r"with coverage and abstention-penalized results. "
        f"The Cascade policy uses $\\tau_c={tau_c:g}$ and "
        f"$\\tau_u={tau_u:.6f}$ (rounded for display). "
        r"Macro-F1 uses the fixed 17-label set. NA denotes unavailable "
        r"ECE for retrieval-only methods.}",
    ),
    r"\label{tab:results}",
    r"\resizebox{\textwidth}{!}{%",
    r"\begin{tabular}{llrccccc}",
    r"\toprule",
    (
        r"Method & Evaluation & $n$ & Accuracy & Macro-F1 & "
        r"Weighted-F1 & Cohen's $\kappa$ & ECE \\"
    ),
    r"\midrule",
]

for method in order:
    subset = results.loc[results["method"].eq(method)]

    for mode in ["forced_before_abstention", "native_CRUS_gate"]:
        selected = subset.loc[subset["mode"].eq(mode)]
        if selected.empty:
            continue

        row = selected.iloc[0]
        forced = mode == "forced_before_abstention"
        evaluation = "Forced" if forced else "Accepted"
        n = int(row["N"] if forced else row["n_accepted"])
        accuracy = (
            row["forced_accuracy"] if forced
            else row["selective_accuracy"]
        )

        values = [
            names[method], evaluation, str(n),
            fmt(accuracy), fmt(row["macro_f1"]),
            fmt(row["weighted_f1"]), fmt(row["kappa"]),
            fmt(row["ece"]),
        ]
        lines.append(" & ".join(values) + r" \\")

lines.extend([
    r"\bottomrule",
    r"\end{tabular}}",
    r"\end{table*}",
])

path = OUT / "comment7_verified_results_table.tex"
path.write_text("\n".join(lines), encoding="utf-8")

assert path.exists() and path.stat().st_size > 0
print(path.read_text(encoding="utf-8"))

In [ ]:
lines = [
    "".join(item) if isinstance(item, tuple) else item
    for item in lines
]

assert all(isinstance(item, str) for item in lines)

path = OUT / "comment7_verified_results_table.tex"
path.write_text("\n".join(lines), encoding="utf-8")

assert path.exists() and path.stat().st_size > 0
print(path.read_text(encoding="utf-8"))

In [ ]:
from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd

BASE = Path(
    _crus_local_path('project', 'corrected_experiment_outputs')
)
OUT = BASE / "comment8_audit"
OUT.mkdir(parents=True, exist_ok=True)

source_paths = [
    BASE / "final_pretest_protocol.json",
    BASE / "test_all_method_predictions.csv",
    BASE / "test_frozen_CRUS_acceptance_audit.csv",
    BASE / "verified_main_test_results.csv",
]

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

for path in source_paths:
    assert path.exists(), f"Missing source: {path.name}"

original_hashes = {path: sha256(path) for path in source_paths}

with source_paths[0].open(encoding="utf-8") as handle:
    protocol = json.load(handle)

predictions = pd.read_csv(
    source_paths[1],
    dtype={"record_id": str, "true_label": str, "prediction": str},
)
gates = pd.read_csv(
    source_paths[2],
    dtype={"record_id": str},
)
saved_results = pd.read_csv(source_paths[3])

N = int(protocol["split_sizes"]["test"])

def as_bool(series):
    text = series.astype(str).str.strip().str.lower()
    mapping = {"true": True, "false": False, "1": True, "0": False}
    assert text.isin(mapping).all(), "Invalid Boolean values."
    return text.map(mapping).to_numpy(dtype=bool)

def method_predictions(method):
    table = predictions.loc[
        predictions["method"].eq(method)
    ].copy()

    assert len(table) == N
    assert not table["record_id"].duplicated().any()

    correct = table["prediction"].eq(table["true_label"]).to_numpy()
    assert np.array_equal(correct, as_bool(table["correct"]))

    return table.sort_values("record_id").reset_index(drop=True)

cascade = method_predictions("Cascade")
sk4 = method_predictions("SK4")
lengthnorm = method_predictions("LengthNormLegacy")

for other in [sk4, lengthnorm]:
    assert cascade["record_id"].equals(other["record_id"])
    assert cascade["true_label"].equals(other["true_label"])

rows = []
record_audits = []

for method in ["Cascade", "LengthNormLegacy"]:
    table = method_predictions(method)
    gate = gates.loc[gates["method"].eq(method)].copy()

    assert len(gate) == N
    assert not gate["record_id"].duplicated().any()
    assert set(gate["record_id"]) == set(table["record_id"])

    gate = gate.set_index("record_id").loc[
        table["record_id"]
    ].reset_index()

    correct = table["prediction"].eq(table["true_label"]).to_numpy()
    accepted = as_bool(gate["accepted"])

    assert np.array_equal(correct, as_bool(gate["correct"]))
    np.testing.assert_allclose(
        table["confidence"].to_numpy(dtype=float),
        gate["confidence"].to_numpy(dtype=float),
        rtol=1e-7,
        atol=1e-10,
    )

    policy = protocol["frozen_policies"][f"{method}:CRUS"]
    tau_c = float(policy["tau_c"])
    tau_u = float(policy["tau_u"])

    for column, value in [("tau_c", tau_c), ("tau_u", tau_u)]:
        np.testing.assert_allclose(
            gate[column].to_numpy(dtype=float),
            value,
            rtol=1e-7,
            atol=1e-10,
        )

    confidence = gate["confidence"].to_numpy(dtype=float)
    crus = gate["CRUS"].to_numpy(dtype=float)
    assert np.isfinite(confidence).all()
    assert np.isfinite(crus).all()

    reconstructed_acceptance = (
        (confidence >= tau_c) & (crus <= tau_u)
    )
    assert np.array_equal(accepted, reconstructed_acceptance), (
        f"Saved acceptance differs from frozen rule: {method}"
    )

    accepted_n = int(accepted.sum())
    correct_all = int(correct.sum())
    correct_accepted = int((correct & accepted).sum())
    errors_accepted = int((~correct & accepted).sum())
    abstained_n = N - accepted_n

    rows.append({
        "method": method,
        "N": N,
        "correct_before_abstention": correct_all,
        "errors_before_abstention": N - correct_all,
        "forced_accuracy": correct_all / N,
        "forced_misclassification_rate": 1 - correct_all / N,
        "n_accepted": accepted_n,
        "n_correct_accepted": correct_accepted,
        "n_errors_accepted": errors_accepted,
        "n_abstained": abstained_n,
        "coverage": accepted_n / N,
        "selective_accuracy": (
            correct_accepted / accepted_n if accepted_n else np.nan
        ),
        "accepted_misclassification_rate": (
            errors_accepted / accepted_n if accepted_n else np.nan
        ),
        "abstention_penalized_accuracy": correct_accepted / N,
        "abstention_penalized_error": (
            errors_accepted + abstained_n
        ) / N,
        "tau_c": tau_c,
        "tau_u": tau_u,
    })

    record_audits.append(pd.DataFrame({
        "record_id": table["record_id"],
        "method": method,
        "true_label": table["true_label"],
        "unchanged_prediction": table["prediction"],
        "correct_before_gating": correct,
        "accepted": accepted,
        "correct_and_accepted": correct & accepted,
        "CRUS": crus,
        "confidence": confidence,
    }))

operating_points = pd.DataFrame(rows)

cascade_row = operating_points.loc[
    operating_points["method"].eq("Cascade")
].iloc[0]

assert int(cascade_row["correct_before_abstention"]) == 181
assert int(cascade_row["n_accepted"]) == 117
assert int(cascade_row["n_correct_accepted"]) == 109
assert int(cascade_row["n_errors_accepted"]) == 8
assert int(cascade_row["n_abstained"]) == 140

comparison_fields = [
    "N", "n_accepted", "n_correct_accepted",
    "n_errors_accepted", "n_abstained", "coverage",
    "forced_accuracy", "selective_accuracy",
    "accepted_misclassification_rate",
    "abstention_penalized_accuracy",
    "abstention_penalized_error",
]

for _, row in operating_points.iterrows():
    saved = saved_results.loc[
        saved_results["method"].eq(row["method"])
        & saved_results["mode"].eq("native_CRUS_gate")
    ]
    assert len(saved) == 1
    saved = saved.iloc[0]

    for field in comparison_fields:
        assert np.isclose(
            float(row[field]),
            float(saved[field]),
            rtol=1e-7,
            atol=1e-10,
            equal_nan=True,
        ), f"Saved-result mismatch: {row['method']} / {field}"

sk4_correct = int(
    sk4["prediction"].eq(sk4["true_label"]).sum()
)
forced_comparison = pd.DataFrame([
    {
        "method": "SK4",
        "N": N,
        "correct": sk4_correct,
        "accuracy": sk4_correct / N,
        "misclassification_rate": 1 - sk4_correct / N,
    },
    {
        "method": "Cascade",
        "N": N,
        "correct": int(cascade_row["correct_before_abstention"]),
        "accuracy": float(cascade_row["forced_accuracy"]),
        "misclassification_rate": float(
            cascade_row["forced_misclassification_rate"]
        ),
    },
])

operating_points.to_csv(
    OUT / "comment8_verified_operating_points.csv",
    index=False,
    na_rep="NA",
)
pd.concat(record_audits, ignore_index=True).to_csv(
    OUT / "comment8_same_prediction_gate_audit.csv",
    index=False,
)
forced_comparison.to_csv(
    OUT / "comment8_full_coverage_comparison.csv",
    index=False,
)

latex = [
    r"\begin{table}[ht]",
    r"\centering",
    r"\caption{Cascade performance before and after the frozen CRUS "
    r"gate, using the same underlying predictions. Error rates "
    r"measure misclassification, not clinical harm.}",
    r"\label{tab:cascade_gating}",
    r"\begin{tabular}{lrr}",
    r"\toprule",
    r"Quantity & Count & Value \\",
    r"\midrule",
    f"Before-gating accuracy & 181/257 & "
    f"{cascade_row['forced_accuracy']:.4f}" + r" \\",
    f"Before-gating misclassification & 76/257 & "
    f"{cascade_row['forced_misclassification_rate']:.4f}" + r" \\",
    f"Coverage & 117/257 & "
    f"{cascade_row['coverage']:.4f}" + r" \\",
    f"Selective accuracy & 109/117 & "
    f"{cascade_row['selective_accuracy']:.4f}" + r" \\",
    f"Accepted-case misclassification & 8/117 & "
    f"{cascade_row['accepted_misclassification_rate']:.4f}" + r" \\",
    f"Abstention-penalized accuracy & 109/257 & "
    f"{cascade_row['abstention_penalized_accuracy']:.4f}" + r" \\",
    f"Abstention-penalized error & 148/257 & "
    f"{cascade_row['abstention_penalized_error']:.4f}" + r" \\",
    r"\bottomrule",
    r"\end{tabular}",
    r"\end{table}",
]

tex_path = OUT / "comment8_cascade_gating_table.tex"
tex_path.write_text("\n".join(latex), encoding="utf-8")

with (OUT / "comment8_lengthnorm_metadata.json").open(
    "w", encoding="utf-8"
) as handle:
    json.dump({
        "retrieval_configuration": protocol["retrieval_configuration"],
        "limitations": protocol["limitations"],
        "validation_findings": protocol["validation_findings"],
        "implementation_verification_needed": [
            "Exact document-length measure and reference length",
            "Exact length-penalty formula",
            "Penalty clipping and its numerical threshold",
            "Whether correction is applied before softmax",
            "Whether Stage A filtering is absent",
        ],
    }, handle, indent=2)

assert all(
    sha256(path) == original_hash
    for path, original_hash in original_hashes.items()
)

print("=== VERIFIED OPERATING POINTS ===")
print(operating_points.to_string(index=False))
print("\n=== FULL-COVERAGE COMPARISON ===")
print(forced_comparison.to_string(index=False))
print("\n=== CASCADE TABLE ===")
print(tex_path.read_text(encoding="utf-8"))
print("\nSource files unchanged.")

In [ ]:
import inspect
import json
from pathlib import Path

BASE = Path(
    _crus_local_path('project', 'corrected_experiment_outputs')
)

with (BASE / "final_pretest_protocol.json").open(
    encoding="utf-8"
) as handle:
    protocol = json.load(handle)

print("=== FROZEN LENGTH-NORMALISATION METADATA ===")
print(json.dumps({
    "retrieval_configuration": protocol["retrieval_configuration"],
    "policy": protocol["frozen_policies"].get(
        "LengthNormLegacy:CRUS"
    ),
    "limitations": protocol.get("limitations", []),
    "validation_findings": protocol.get("validation_findings", {}),
}, indent=2))

print("\n=== LIVE FUNCTION SOURCES ===")

found_live_source = False

for name, obj in list(globals().items()):
    if name.startswith("_") or not inspect.isfunction(obj):
        continue
    if getattr(obj, "__module__", None) != "__main__":
        continue

    try:
        source = inspect.getsource(obj)
    except (OSError, TypeError):
        continue

    searchable = (name + "\n" + source).lower()
    if any(term in searchable for term in [
        "lengthnorm", "length_norm", "lengthpenalty", "length_penalty"
    ]):
        found_live_source = True
        print(f"\n--- FUNCTION: {name} ---")
        print(source)

print("\n=== NOTEBOOK CELL HISTORY ===")

ip = get_ipython()
history = ip.history_manager.input_hist_raw

# Exclude this inspection cell.
inspection_markers = [
    "FROZEN LENGTH-NORMALISATION METADATA",
    "LIVE FUNCTION SOURCES",
]

matches = []

for cell_number, source in enumerate(history[:-1]):
    if not source or any(x in source for x in inspection_markers):
        continue

    lower = source.lower()
    has_lengthnorm = any(term in lower for term in [
        "lengthnorm", "length_norm", "lengthpenalty", "length_penalty"
    ])
    has_implementation = any(term in lower for term in [
        "def ", "np.log", "np.clip", "torch.clamp", "penalty"
    ])

    if has_lengthnorm and has_implementation:
        matches.append((cell_number, source))

for cell_number, source in matches:
    print(f"\n--- EXECUTED CELL {cell_number} ---")
    print(source)

if not found_live_source and not matches:
    print(
        "\nNo LengthNorm implementation was found in this session. "
        "Open the notebook that generated the final corrected outputs "
        "and run this inspection cell there. Do not rerun inference."
    )

In [ ]:
from pathlib import Path

ROOT = Path(_crus_local_path('project', ''))
assert ROOT.exists(), "Revision folder not found."

sources = sorted(
    path for path in ROOT.rglob("*")
    if path.is_file() and path.suffix.lower() in {".ipynb", ".py"}
)

for path in sources:
    print(path)

if not sources:
    print(
        "No notebook or Python source found in this folder. "
        "Locate or save the notebook that generated the corrected outputs."
    )

In [ ]:
from pathlib import Path

ROOT = Path(_crus_local_path('drive', ''))

source_candidates = sorted(
    p for p in ROOT.rglob("*")
    if p.is_file() and p.suffix.lower() in {".ipynb", ".py"}
)

if source_candidates:
    for number, path in enumerate(source_candidates):
        print(f"[{number}] {path}")
else:
    print(
        "No notebook or Python files found in MyDrive. "
        "Locate the generating notebook in Colab's recent notebooks "
        "or in a shared Drive location."
    )

In [ ]:
import json
from pathlib import Path

assert source_candidates, "Run the previous file-listing cell first."

markers = [
    "revision2_dialogue_only_v1",
    "corrected_experiment_outputs",
    "final_pretest_protocol",
    "LengthNormLegacy",
]

for path in source_candidates:
    try:
        if path.suffix.lower() == ".ipynb":
            notebook = json.loads(path.read_text(encoding="utf-8"))
            cells = [
                (i, "".join(cell.get("source", [])))
                for i, cell in enumerate(notebook.get("cells", []))
                if cell.get("cell_type") == "code"
            ]
        else:
            cells = [(0, path.read_text(encoding="utf-8"))]
    except Exception as error:
        print(f"Could not read {path.name}: {error}")
        continue

    combined = "\n".join(source for _, source in cells)
    hits = [marker for marker in markers if marker in combined]

    if not hits:
        continue

    print(f"\n=== FILE: {path} ===")
    print("Markers found:", hits)

    found_implementation = False

    for cell_number, source in cells:
        lower = source.lower()

        length_terms = any(term in lower for term in [
            "lengthnorm", "length_norm",
            "lengthpenalty", "length_penalty"
        ])

        computation_terms = any(term in lower for term in [
            "np.log(", "np.clip(", "torch.clamp(",
            "math.log(", "length_penalty =",
            "lengthpenalty =", "penalty ="
        ])

        if length_terms and computation_terms:
            found_implementation = True
            print(f"\n--- IMPLEMENTATION CANDIDATE: CELL {cell_number} ---")
            print(source)

    if not found_implementation:
        print(
            "No direct penalty computation found. "
            "This may be an audit/reporting notebook rather than "
            "the generating pipeline."
        )

# **REVIWER 1 comment in first revision remaining**

In [ ]:
from pathlib import Path
import hashlib
import json
import pandas as pd

DATA_CSV = Path("PATH_TO_SOURCE_DATA.csv")
MANIFEST_CSV = Path("PATH_TO_EXISTING_SPLIT_MANIFEST.csv")
OUT = Path("output/reviewer1_dialogue_only")
OUT.mkdir(parents=True, exist_ok=True)

ID_COL = "record_id"
DIALOGUE_COL = "dialogue"
HEADER_COL = "section_header"
SPLIT_COL = "split"

data = pd.read_csv(DATA_CSV, dtype={ID_COL: str})
manifest = pd.read_csv(MANIFEST_CSV, dtype={ID_COL: str})

for col in [ID_COL, DIALOGUE_COL, HEADER_COL]:
    if col not in data.columns:
        raise ValueError(f"Missing source column: {col}")

for col in [ID_COL, SPLIT_COL]:
    if col not in manifest.columns:
        raise ValueError(f"Missing manifest column: {col}")

if data[ID_COL].duplicated().any():
    raise ValueError("Source IDs are not unique.")

if manifest[ID_COL].duplicated().any():
    raise ValueError("Manifest IDs are not unique.")

missing_ids = set(manifest[ID_COL]) - set(data[ID_COL])
if missing_ids:
    raise ValueError(f"Manifest IDs missing from source: {len(missing_ids)}")

df = manifest[[ID_COL, SPLIT_COL]].merge(
    data[[ID_COL, DIALOGUE_COL, HEADER_COL]],
    on=ID_COL,
    how="left",
    validate="one_to_one",
)

if df[[DIALOGUE_COL, HEADER_COL, SPLIT_COL]].isna().any().any():
    raise ValueError("Missing dialogue, target header, or split.")

df[SPLIT_COL] = df[SPLIT_COL].astype(str).str.strip().str.lower()

if not set(df[SPLIT_COL]).issubset({"train", "validation", "test"}):
    raise ValueError("Normalize split names to train, validation, and test.")

df["query_text"] = df[DIALOGUE_COL].astype(str)

if df["query_text"].str.strip().eq("").any():
    raise ValueError("Empty dialogue detected.")

# Replace this identity mapping if the established task uses
# a predefined header-to-routing-class mapping.
df["target"] = df[HEADER_COL].astype(str)

df["query_sha256"] = df["query_text"].map(
    lambda text: hashlib.sha256(text.encode("utf-8")).hexdigest()
)

for split, part in df.groupby(SPLIT_COL):
    part[[ID_COL, "query_text"]].to_csv(
        OUT / f"{split}_inputs.csv", index=False
    )
    part[[ID_COL, "target"]].to_csv(
        OUT / f"{split}_targets.csv", index=False
    )

df[[ID_COL, SPLIT_COL, "query_sha256"]].to_csv(
    OUT / "query_audit_manifest.csv", index=False
)

protocol = {
    "input_fields": [DIALOGUE_COL],
    "target_source": HEADER_COL,
    "query_construction": "Original dialogue field only; no concatenation",
    "split_manifest": str(MANIFEST_CSV),
    "excludes_gold_header": True,
    "excludes_gold_summary": True,
    "excludes_case_specific_gold_label": True,
    "note": (
        "This records the intended input contract. "
        "The actual pipeline and upstream dialogue field "
        "still require inspection."
    ),
}

(OUT / "input_protocol.json").write_text(
    json.dumps(protocol, indent=2), encoding="utf-8"
)

print(df.groupby(SPLIT_COL).size())
print("Dialogue-only inputs and separate targets created.")

In [ ]:
from pathlib import Path
import pandas as pd
pass  # Colab service not required locally.

DRIVE_ROOT = Path(_crus_local_path('drive', ''))

if not DRIVE_ROOT.exists():
    _crus_prepare_saved_workspace()

PROJECT = DRIVE_ROOT / "RiskAware_Revision2"

if not PROJECT.exists():
    raise FileNotFoundError(
        f"Project folder not found: {PROJECT}\n"
        "Check the project folder name in the Colab Files panel."
    )

csv_files = sorted(PROJECT.rglob("*.csv"))

print(f"PROJECT: {PROJECT}")
print(f"CSV files found: {len(csv_files)}")

for index, path in enumerate(csv_files, start=1):
    print(f"\n[{index}] {path}")
    try:
        sample = pd.read_csv(path, nrows=3)
        print("Columns:", sample.columns.tolist())
    except Exception as exc:
        print("Could not read:", type(exc).__name__, str(exc)[:180])

print("\nOTHER POSSIBLE DATA / SPLIT FILES:")
for path in sorted(PROJECT.rglob("*")):
    if (
        path.is_file()
        and path.suffix.lower() in {".json", ".jsonl", ".parquet", ".pkl"}
        and any(
            term in path.name.lower()
            for term in ["data", "source", "split", "manifest", "assignment"]
        )
    ):
        print(path)

In [ ]:
from pathlib import Path
import hashlib
import json
import pandas as pd

ROOT = Path(_crus_local_path('project', ''))
AUDIT = ROOT / "data_audit"
RESULTS = ROOT / "corrected_experiment_outputs"
OUT = ROOT / "reviewer1_comment1_dialogue_only"
OUT.mkdir(parents=True, exist_ok=True)

ID = "record_id"

def read_csv(path):
    if not path.is_file():
        raise FileNotFoundError(path)
    return pd.read_csv(path, dtype={ID: str})

def normalize_splits(series):
    aliases = {
        "val": "validation",
        "valid": "validation",
        "dev": "validation",
        "training": "train",
        "testing": "test",
    }
    return series.astype(str).str.strip().str.lower().replace(aliases)

def label_map(frame, label_col):
    if frame[[ID, label_col]].isna().any().any():
        raise ValueError(f"Missing IDs or labels in {label_col}")

    pairs = frame[[ID, label_col]].drop_duplicates()

    if pairs[ID].duplicated().any():
        raise ValueError("Conflicting labels for the same record ID.")

    return pairs.set_index(ID)[label_col].astype(str).to_dict()

saved_test = read_csv(RESULTS / "test_all_method_predictions.csv")
saved_validation = read_csv(
    RESULTS / "validation_retrieval_prediction_audit.csv"
)

expected = {
    "test": label_map(saved_test, "true_label"),
    "validation": label_map(saved_validation, "true_label"),
}

candidates = {}
checks = []

for name in ["corrected", "constrained"]:
    manifest = read_csv(AUDIT / f"{name}_split_manifest.csv")

    required = {ID, "label", "group_id", "split"}
    if not required.issubset(manifest.columns):
        raise ValueError(f"{name}: missing required manifest columns.")

    if manifest[ID].duplicated().any():
        raise ValueError(f"{name}: duplicate manifest record IDs.")

    manifest["split"] = normalize_splits(manifest["split"])

    matches = True
    for split, expected_labels in expected.items():
        observed = label_map(
            manifest.loc[manifest["split"].eq(split)], "label"
        )
        ids_match = set(observed) == set(expected_labels)
        labels_match = observed == expected_labels

        checks.append({
            "protocol": name,
            "split": split,
            "manifest_records": len(observed),
            "saved_records": len(expected_labels),
            "ids_match": ids_match,
            "labels_match": labels_match,
        })
        matches = matches and ids_match and labels_match

    if matches:
        candidates[name] = manifest

check_table = pd.DataFrame(checks)
check_table.to_csv(OUT / "split_protocol_check.csv", index=False)
print(check_table.to_string(index=False))

if not candidates:
    raise RuntimeError(
        "Neither manifest matches the saved test/validation outputs. "
        "Stop and inspect the generating notebook."
    )

if len(candidates) == 2:
    def signature(frame):
        return (
            frame[[ID, "split", "label", "group_id"]]
            .astype(str)
            .sort_values(ID)
            .reset_index(drop=True)
        )

    if not signature(candidates["corrected"]).equals(
        signature(candidates["constrained"])
    ):
        raise RuntimeError(
            "Both protocols match test/validation outputs but differ "
            "in their full assignments. Inspect the generating notebook "
            "to identify the training protocol."
        )

    selected = "constrained"
    print("\nBoth manifests have identical full assignments.")
else:
    selected = next(iter(candidates))

manifest = candidates[selected]
pool_path = AUDIT / f"{selected}_dialogue_pool_with_splits.csv"
pool = read_csv(pool_path)

required = {ID, "dialogue"}
if not required.issubset(pool.columns):
    raise ValueError("Selected pool lacks record_id or dialogue.")

if pool[ID].duplicated().any():
    raise ValueError("Selected pool contains duplicate record IDs.")

missing = set(manifest[ID]) - set(pool[ID])
if missing:
    raise ValueError(f"{len(missing)} manifest IDs missing from pool.")

source_cols = [ID, "dialogue"]
if "text" in pool.columns:
    source_cols.append("text")

df = manifest[[ID, "split", "label", "group_id"]].merge(
    pool[source_cols],
    on=ID,
    how="left",
    validate="one_to_one",
)

if not set(df["split"]).issubset({"train", "validation", "test"}):
    raise ValueError("Unexpected split names.")

if df["dialogue"].isna().any():
    raise ValueError("Missing original dialogue.")

df["query_text"] = df["dialogue"].astype(str)

if df["query_text"].str.strip().eq("").any():
    raise ValueError("Empty dialogue.")

cross_split_groups = df.groupby("group_id")["split"].nunique()
if cross_split_groups.gt(1).any():
    raise ValueError("Some group IDs cross split boundaries.")

if "text" in df.columns:
    identical = (
        df["text"].notna()
        & df["text"].fillna("").astype(str).eq(df["query_text"])
    )
    comparison = df[[ID, "split"]].copy()
    comparison["text_equals_original_dialogue"] = identical
    comparison.to_csv(
        OUT / "existing_text_comparison.csv", index=False
    )
    print(
        "\nExisting text equals original dialogue:",
        f"{int(identical.sum())}/{len(df)}",
    )
    print(
        "Differences require inspection; they do not by themselves "
        "prove leakage."
    )

df["query_sha256"] = df["query_text"].map(
    lambda text: hashlib.sha256(text.encode("utf-8")).hexdigest()
)

for split, part in df.groupby("split"):
    part[[ID, "query_text"]].to_csv(
        OUT / f"{split}_inputs.csv", index=False
    )
    part[[ID, "label"]].rename(
        columns={"label": "target"}
    ).to_csv(
        OUT / f"{split}_targets.csv", index=False
    )

df[[ID, "split", "group_id", "query_sha256"]].to_csv(
    OUT / "query_audit_manifest.csv", index=False
)

protocol = {
    "selected_split_protocol": selected,
    "source_pool": str(pool_path),
    "manifest": str(AUDIT / f"{selected}_split_manifest.csv"),
    "query_source_field": "dialogue",
    "target_source_field": "existing manifest label",
    "input_columns": [ID, "query_text"],
    "excluded_annotation_fields": [
        "section_header", "section_text", "label"
    ],
    "counts": {
        split: int(count)
        for split, count in df.groupby("split").size().items()
    },
    "status": "Inputs prepared; model rerun not yet performed",
}

(OUT / "input_protocol.json").write_text(
    json.dumps(protocol, indent=2), encoding="utf-8"
)

print("\nSelected protocol:", selected)
print("\nSplit counts:")
print(df.groupby("split").size().to_string())
print("\nSaved dialogue-only inputs and separate targets to:", OUT)

In [ ]:
from pathlib import Path
import json
import re
import pandas as pd

ROOT = Path(_crus_local_path('project', ''))
POOL = ROOT / "data_audit/constrained_dialogue_pool_with_splits.csv"
NOTEBOOK = Path(
    _crus_local_path('notebooks', 'revised_risk_aware_hybrid_pipeline_v2.ipynb')
)

df = pd.read_csv(POOL, dtype={"record_id": str})

required = {
    "record_id", "dialogue", "text",
    "section_header", "section_text"
}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"Missing columns: {sorted(missing)}")

def normalize(value):
    if pd.isna(value):
        return ""
    return re.sub(r"\s+", " ", str(value)).strip().casefold()

dialogue = df["dialogue"].map(normalize)
text = df["text"].map(normalize)
header = df["section_header"].map(normalize)
summary = df["section_text"].map(normalize)

checks = pd.DataFrame({"record_id": df["record_id"]})
checks["text_equals_dialogue_after_normalization"] = text.eq(dialogue)
checks["dialogue_contained_in_text"] = [
    bool(d) and d in t for d, t in zip(dialogue, text)
]
checks["gold_header_contained_in_text"] = [
    bool(h) and h in t for h, t in zip(header, text)
]
checks["gold_summary_contained_in_text"] = [
    bool(s) and s in t for s, t in zip(summary, text)
]

print("=== TEXT COMPARISON COUNTS ===")
for col in checks.columns[1:]:
    print(f"{col}: {int(checks[col].sum())}/{len(checks)}")

print("\n=== FIRST THREE DIFFERING RECORDS ===")
different = df.loc[
    ~checks["text_equals_dialogue_after_normalization"]
]

for _, row in different.head(3).iterrows():
    print("\nRecord:", row["record_id"])
    for col in ["section_header", "dialogue", "section_text", "text"]:
        value = "" if pd.isna(row[col]) else str(row[col])
        print(f"{col}: {value[:600]!r}")

if not NOTEBOOK.is_file():
    raise FileNotFoundError(
        f"Generating notebook not found: {NOTEBOOK}\n"
        "Use the actual notebook path before continuing."
    )

notebook = json.loads(NOTEBOOK.read_text(encoding="utf-8"))

pattern = re.compile(
    r"section_header|section_text|"
    r"\[['\"]text['\"]\]|\[['\"]dialogue['\"]\]|"
    r"query_text|build_query|"
    r"fit_transform|\.transform\(|\.encode\(|"
    r"constrained_dialogue_pool|constrained_split_manifest"
)

print("\n=== RELEVANT NOTEBOOK CODE ===")

for cell_number, cell in enumerate(notebook.get("cells", []), start=1):
    if cell.get("cell_type") != "code":
        continue

    source = cell.get("source", [])
    source = "".join(source) if isinstance(source, list) else source
    lines = source.splitlines()

    matched = [
        index for index, line in enumerate(lines)
        if pattern.search(line)
    ]

    if not matched:
        continue

    visible = set()
    for index in matched:
        visible.update(
            range(max(0, index - 3), min(len(lines), index + 4))
        )

    print(f"\n--- Notebook cell {cell_number} ---")
    previous = None
    for index in sorted(visible):
        if previous is not None and index > previous + 1:
            print("    ...")
        print(f"{index + 1:4d}: {lines[index]}")
        previous = index

In [ ]:
from pathlib import Path
import json
import re

NOTEBOOK = Path(
    _crus_local_path('notebooks', 'revised_risk_aware_hybrid_pipeline_v2.ipynb')
)

if not NOTEBOOK.is_file():
    raise FileNotFoundError(NOTEBOOK)

notebook = json.loads(NOTEBOOK.read_text(encoding="utf-8"))

# Cell numbers refer to the current notebook's 1-based positions.
full_cells = {56, 57, 86}

pipeline_pattern = re.compile(
    r"\.fit\s*\(|fit_transform\s*\(|"
    r"predict_proba\s*\(|\.predict\s*\(|"
    r"TfidfVectorizer|CalibratedClassifierCV|"
    r"score_retrieval_methods|"
    r"test_all_method_predictions|"
    r"verified_main_test_results|"
    r"validation_retrieval_prediction_audit|"
    r"train_embs\s*=|"
    r"joblib\.load|pickle\.load|np\.load",
    flags=re.IGNORECASE,
)

for cell_number, cell in enumerate(notebook.get("cells", []), start=1):
    if cell.get("cell_type") != "code":
        continue

    source = cell.get("source", [])
    source = "".join(source) if isinstance(source, list) else source

    # Inspect the corrected experiment block, not later audit cells.
    in_corrected_block = 58 <= cell_number <= 90

    if (
        cell_number in full_cells
        or (in_corrected_block and pipeline_pattern.search(source))
    ):
        print(f"\n{'=' * 70}")
        print(
            f"CELL {cell_number} | "
            f"execution_count={cell.get('execution_count')}"
        )
        print("=" * 70)
        print(source)

In [ ]:
from pathlib import Path
import json

NOTEBOOK = Path(
    _crus_local_path('notebooks', 'revised_risk_aware_hybrid_pipeline_v2.ipynb')
)

if not NOTEBOOK.is_file():
    raise FileNotFoundError(NOTEBOOK)

notebook = json.loads(NOTEBOOK.read_text(encoding="utf-8"))

# Use the current notebook's 1-based cell numbers.
selected_cells = (
    {58, 65, 66, 67, 69, 71, 72, 73, 74}
    | set(range(83, 86))
    | set(range(87, 101))
)

for number, cell in enumerate(notebook.get("cells", []), start=1):
    if number not in selected_cells:
        continue
    if cell.get("cell_type") != "code":
        continue

    source = cell.get("source", [])
    source = "".join(source) if isinstance(source, list) else source

    print(f"\n{'=' * 70}")
    print(f"CELL {number}")
    print("=" * 70)
    print(source)

In [ ]:
from pathlib import Path
import json
import re

NOTEBOOK = Path(
    _crus_local_path('notebooks', 'revised_risk_aware_hybrid_pipeline_v2.ipynb')
)

if not NOTEBOOK.is_file():
    raise FileNotFoundError(NOTEBOOK)

nb = json.loads(NOTEBOOK.read_text(encoding="utf-8"))

function_pattern = re.compile(
    r"def\s+(?:prepare_candidates|score_documents|score_sk4)\s*\("
)

test_pattern = re.compile(
    r"test_all_method_predictions|"
    r"verified_main_test_results|"
    r"test_retrieval_checkpoint|"
    r"predict_proba\s*\(\s*X_test|"
    r"\.predict\s*\(\s*X_test|"
    r"for\s+.*\bin\s+(?:tqdm\s*\(\s*)?X_test|"
    r"score_retrieval_methods\s*\(\s*(?:query|q)\s*\)"
)

always_show = {58, 65, 66, 67, 69, 71, 72, 73, 74, 83, 84, 85, 87, 88}
shown = []

for number, cell in enumerate(nb.get("cells", []), start=1):
    if cell.get("cell_type") != "code":
        continue

    source = cell.get("source", [])
    source = "".join(source) if isinstance(source, list) else source

    # Exclude original pipeline and later diagnostic/audit cells.
    if not 58 <= number <= 109:
        continue

    if (
        number in always_show
        or function_pattern.search(source)
        or test_pattern.search(source)
    ):
        shown.append(number)
        print(f"\n{'=' * 70}")
        print(f"CELL {number}")
        print("=" * 70)
        print(source)

print("\nExtracted cells:", shown)

In [ ]:
from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd

ROOT = Path(_crus_local_path('project', ''))
DATA = ROOT / "data_audit"
RESULTS = ROOT / "corrected_experiment_outputs"

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

protocol_path = RESULTS / "final_pretest_protocol.json"
manifest_path = DATA / "constrained_split_manifest.csv"
pool_path = DATA / "constrained_dialogue_pool_with_splits.csv"
predictions_path = RESULTS / "test_all_method_predictions.csv"

for path in [
    protocol_path, manifest_path, pool_path, predictions_path
]:
    if not path.is_file():
        raise FileNotFoundError(path)

protocol = json.loads(protocol_path.read_text())
protocol_hash = sha256(protocol_path)

assert protocol["main_manifest_sha256"] == sha256(manifest_path)

print("=== FROZEN PROTOCOL ===")
for key in [
    "protocol_version",
    "model_input",
    "header_usage",
    "reference_section_text_usage",
    "split_sizes",
]:
    print(f"{key}: {protocol.get(key)}")
print("protocol_sha256:", protocol_hash)

manifest = pd.read_csv(manifest_path, dtype={"record_id": str})
pool = pd.read_csv(pool_path, dtype={"record_id": str})

assert manifest["record_id"].is_unique
assert pool["record_id"].is_unique

assignment_columns = ["record_id", "label", "group_id", "split"]
pd.testing.assert_frame_equal(
    pool[assignment_columns].reset_index(drop=True),
    manifest[assignment_columns].reset_index(drop=True),
    check_dtype=False,
)

counts = {
    split: int(count)
    for split, count in manifest.groupby("split").size().items()
}
assert counts == protocol["split_sizes"]
assert counts == {"train": 773, "validation": 258, "test": 257}

model_files = {
    "lr": "trained_tfidf_lr.joblib",
    "svm": "trained_group_calibrated_svm.joblib",
}

print("\n=== MODEL HASHES ===")
for name, filename in model_files.items():
    path = RESULTS / filename
    assert path.is_file(), f"Missing model: {path}"
    assert sha256(path) == protocol["model_file_hashes"][name]
    print(name, "matches frozen protocol")

calibration_path = RESULTS / "constrained_calibration_manifest.csv"
assert calibration_path.is_file()
assert sha256(calibration_path) == protocol["calibration_manifest_sha256"]

test_manifest = manifest.loc[
    manifest["split"].eq("test"), ["record_id", "label"]
]
expected_labels = test_manifest.set_index("record_id")["label"].astype(str)

csv = pd.read_csv(predictions_path, dtype={"record_id": str})
assert not csv.duplicated(["method", "record_id"]).any()

expected_methods = {
    "TFIDF_LR", "TFIDF_SVM", "BM25", "SBERT", "SK4",
    "SingleStageFull", "SingleStageTruncated",
    "Cascade", "LengthNormLegacy",
}
assert set(csv["method"]) == expected_methods

print("\n=== TEST ARRAYS / PREDICTION CSV ===")
rows = []

for method in sorted(expected_methods):
    path = RESULTS / f"test_{method}_arrays.npz"
    assert path.is_file(), f"Missing test arrays: {path}"

    with np.load(path, allow_pickle=False) as arrays:
        ids = arrays["record_ids"].astype(str)
        labels = arrays["true_labels"].astype(str)
        predictions = arrays["predictions"].astype(str)
        classes = arrays["classes"].astype(str).tolist()
        saved_hash = str(arrays["protocol_sha256"].item())

        assert saved_hash == protocol_hash
        assert classes == protocol["classes"]
        assert len(ids) == 257
        assert len(set(ids)) == 257
        assert set(ids) == set(expected_labels.index)

        np.testing.assert_array_equal(
            labels, expected_labels.loc[ids].to_numpy()
        )

        saved_csv = (
            csv.loc[csv["method"].eq(method)]
            .set_index("record_id")
            .loc[ids]
        )
        assert len(saved_csv) == 257

        np.testing.assert_array_equal(
            predictions,
            saved_csv["prediction"].astype(str).to_numpy(),
        )
        np.testing.assert_array_equal(
            labels,
            saved_csv["true_label"].astype(str).to_numpy(),
        )

        for field in ["confidence", "ranking_score"]:
            np.testing.assert_allclose(
                arrays[field].astype(float),
                saved_csv[field].to_numpy(dtype=float),
                rtol=1e-10,
                atol=1e-12,
                equal_nan=True,
            )

        if "probabilities" in arrays.files:
            probabilities = arrays["probabilities"]
            assert probabilities.shape == (257, len(classes))
            assert np.isfinite(probabilities).all()
            assert (probabilities >= 0).all()
            assert np.allclose(probabilities.sum(axis=1), 1.0)

        rows.append({
            "method": method,
            "test_records": len(ids),
            "correct": int((predictions == labels).sum()),
            "protocol_hash_matches": True,
            "prediction_csv_matches": True,
        })

print(pd.DataFrame(rows).to_string(index=False))

print("\n=== CHECKPOINT FILE PRESENCE ===")
for filename in [
    "frozen_test_retrieval_checkpoint.joblib",
    "frozen_test_supervised_checkpoint.joblib",
]:
    path = RESULTS / filename
    print(filename, "present" if path.is_file() else "missing")

print(
    "\nPASS: saved test arrays, prediction CSV, split, and model "
    "files are internally consistent with the frozen protocol."
)
print(
    "This is an artifact-consistency check, not independent proof "
    "of historical execution or a new experiment rerun."
)

In [ ]:
from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd

ROOT = Path(_crus_local_path('project', ''))
DATA = ROOT / "data_audit"
RESULTS = ROOT / "corrected_experiment_outputs"

# Enable only for artifacts you generated and trust.
TRUST_OWN_ARTIFACTS = True

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def require_file(path):
    if not path.is_file():
        raise FileNotFoundError(f"Required file missing: {path}")

protocol_path = RESULTS / "final_pretest_protocol.json"
manifest_path = DATA / "constrained_split_manifest.csv"
pool_path = DATA / "constrained_dialogue_pool_with_splits.csv"
predictions_path = RESULTS / "test_all_method_predictions.csv"

for path in [
    protocol_path,
    manifest_path,
    pool_path,
    predictions_path,
]:
    require_file(path)

protocol = json.loads(protocol_path.read_text())
protocol_hash = sha256(protocol_path)

assert (
    protocol["main_manifest_sha256"] == sha256(manifest_path)
), "Split manifest hash does not match the frozen protocol."

print("=== FROZEN PROTOCOL ===")
for key in [
    "protocol_version",
    "model_input",
    "header_usage",
    "reference_section_text_usage",
    "split_sizes",
]:
    print(f"{key}: {protocol.get(key)}")

print("protocol_sha256:", protocol_hash)

manifest = pd.read_csv(
    manifest_path,
    dtype={"record_id": str},
)
pool = pd.read_csv(
    pool_path,
    dtype={"record_id": str},
)

assert manifest["record_id"].is_unique, "Duplicate manifest IDs."
assert pool["record_id"].is_unique, "Duplicate pool IDs."

assignment_columns = [
    "record_id", "label", "group_id", "split"
]

pd.testing.assert_frame_equal(
    pool[assignment_columns].reset_index(drop=True),
    manifest[assignment_columns].reset_index(drop=True),
    check_dtype=False,
)

assert manifest[assignment_columns].notna().all().all()
assert pool[["dialogue", "text"]].notna().all().all()

def normalize(text):
    return " ".join(str(text).split()).casefold()

dialogue_matches = (
    pool["dialogue"].map(normalize)
    == pool["text"].map(normalize)
)

assert dialogue_matches.all(), (
    "Some model texts differ from dialogue beyond "
    "whitespace/case normalization."
)

cross_split_groups = manifest.groupby("group_id")["split"].nunique()
assert not cross_split_groups.gt(1).any(), (
    "Some group IDs cross split boundaries."
)

counts = {
    split: int(count)
    for split, count in manifest.groupby("split").size().items()
}

assert counts == protocol["split_sizes"]
assert counts == {
    "train": 773,
    "validation": 258,
    "test": 257,
}

print("\n=== INPUT AND SPLIT CHECKS ===")
print("Pool assignments match manifest.")
print(
    "Model text matches normalized dialogue:",
    f"{int(dialogue_matches.sum())}/{len(pool)}",
)
print("No group IDs cross split boundaries.")
print("Split counts:", counts)

model_files = {
    "lr": "trained_tfidf_lr.joblib",
    "svm": "trained_group_calibrated_svm.joblib",
}

print("\n=== MODEL HASHES ===")
for name, filename in model_files.items():
    path = RESULTS / filename
    require_file(path)

    assert (
        sha256(path) == protocol["model_file_hashes"][name]
    ), f"{name}: model hash mismatch."

    print(name, "matches frozen protocol")

calibration_path = (
    RESULTS / "constrained_calibration_manifest.csv"
)
require_file(calibration_path)

assert (
    sha256(calibration_path)
    == protocol["calibration_manifest_sha256"]
), "Calibration manifest hash mismatch."

print("Calibration manifest matches frozen protocol.")

test_manifest = manifest.loc[
    manifest["split"].eq("test"),
    ["record_id", "label"],
]

expected_labels = (
    test_manifest.set_index("record_id")["label"].astype(str)
)

csv = pd.read_csv(
    predictions_path,
    dtype={"record_id": str},
)

required_csv_columns = {
    "record_id",
    "method",
    "true_label",
    "prediction",
    "correct",
    "confidence",
    "ranking_score",
}
assert required_csv_columns.issubset(csv.columns)
assert csv[[
    "record_id", "method", "true_label", "prediction", "correct"
]].notna().all().all()

assert not csv.duplicated(["method", "record_id"]).any(), (
    "Duplicate method/record pairs in prediction CSV."
)

expected_methods = {
    "TFIDF_LR",
    "TFIDF_SVM",
    "BM25",
    "SBERT",
    "SK4",
    "SingleStageFull",
    "SingleStageTruncated",
    "Cascade",
    "LengthNormLegacy",
}

assert set(csv["method"]) == expected_methods, (
    "Prediction CSV method set differs from expected methods."
)

def parse_boolean(series):
    values = series.astype(str).str.strip().str.lower()
    mapping = {
        "true": True,
        "false": False,
        "1": True,
        "0": False,
        "1.0": True,
        "0.0": False,
    }

    unexpected = set(values) - set(mapping)
    if unexpected:
        raise ValueError(
            f"Unexpected correctness values: {unexpected}"
        )

    return values.map(mapping).to_numpy(dtype=bool)

print("\n=== TEST ARRAYS / PREDICTION CSV ===")
rows = []

for method in sorted(expected_methods):
    path = RESULTS / f"test_{method}_arrays.npz"
    require_file(path)

    with np.load(
        path,
        allow_pickle=TRUST_OWN_ARTIFACTS,
    ) as arrays:
        required_keys = {
            "record_ids",
            "true_labels",
            "predictions",
            "classes",
            "protocol_sha256",
            "confidence",
            "ranking_score",
        }
        assert required_keys.issubset(arrays.files), (
            f"{method}: missing array keys."
        )

        ids = arrays["record_ids"].astype(str)
        labels = arrays["true_labels"].astype(str)
        predictions = arrays["predictions"].astype(str)
        classes = arrays["classes"].astype(str).tolist()
        saved_hash = str(arrays["protocol_sha256"].item())

        assert saved_hash == protocol_hash, (
            f"{method}: protocol hash mismatch."
        )
        assert classes == protocol["classes"], (
            f"{method}: class order mismatch."
        )

        assert ids.shape == (257,)
        assert labels.shape == (257,)
        assert predictions.shape == (257,)
        assert len(set(ids)) == 257
        assert set(ids) == set(expected_labels.index)
        assert set(predictions).issubset(set(classes))

        np.testing.assert_array_equal(
            labels,
            expected_labels.loc[ids].to_numpy(),
            err_msg=f"{method}: labels differ from manifest.",
        )

        method_csv = csv.loc[csv["method"].eq(method)]
        assert len(method_csv) == 257
        assert set(method_csv["record_id"]) == set(ids)

        aligned_csv = method_csv.set_index("record_id").loc[ids]

        np.testing.assert_array_equal(
            predictions,
            aligned_csv["prediction"].astype(str).to_numpy(),
            err_msg=f"{method}: predictions differ from CSV.",
        )
        np.testing.assert_array_equal(
            labels,
            aligned_csv["true_label"].astype(str).to_numpy(),
            err_msg=f"{method}: labels differ from CSV.",
        )

        correct = predictions == labels
        np.testing.assert_array_equal(
            correct,
            parse_boolean(aligned_csv["correct"]),
            err_msg=f"{method}: saved correctness is inconsistent.",
        )

        for field in ["confidence", "ranking_score"]:
            saved_values = arrays[field].astype(float)
            assert saved_values.shape == (257,)

            np.testing.assert_allclose(
                saved_values,
                aligned_csv[field].to_numpy(dtype=float),
                rtol=1e-10,
                atol=1e-12,
                equal_nan=True,
                err_msg=f"{method}: {field} differs from CSV.",
            )

        has_probabilities = "probabilities" in arrays.files

        if has_probabilities:
            probabilities = arrays["probabilities"].astype(float)

            assert probabilities.shape == (257, len(classes))
            assert np.isfinite(probabilities).all()
            assert (probabilities >= 0).all()
            assert np.allclose(probabilities.sum(axis=1), 1.0)

            probability_predictions = np.asarray(classes)[
                probabilities.argmax(axis=1)
            ]
            np.testing.assert_array_equal(
                predictions,
                probability_predictions,
                err_msg=f"{method}: probability argmax mismatch.",
            )

            np.testing.assert_allclose(
                arrays["confidence"].astype(float),
                probabilities.max(axis=1),
                rtol=1e-10,
                atol=1e-12,
                err_msg=f"{method}: confidence/probability mismatch.",
            )

        rows.append({
            "method": method,
            "test_records": len(ids),
            "correct": int(correct.sum()),
            "forced_accuracy": float(correct.mean()),
            "probabilities_available": has_probabilities,
            "protocol_hash_matches": True,
            "prediction_csv_matches": True,
        })

print(pd.DataFrame(rows).to_string(index=False))

print("\n=== CHECKPOINT FILE PRESENCE ===")
for filename in [
    "frozen_test_retrieval_checkpoint.joblib",
    "frozen_test_supervised_checkpoint.joblib",
]:
    path = RESULTS / filename
    print(
        filename,
        "present" if path.is_file() else "missing",
    )

print(
    "\nPASS: dialogue-only pool, split assignments, "
    "model hashes, calibration manifest, test arrays, "
    "and prediction CSV are internally consistent "
    "with the frozen protocol."
)
print(
    "Checkpoint contents were not loaded or inspected. "
    "This check does not rerun experiments or independently "
    "prove historical execution provenance."
)

In [ ]:
from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd

ROOT = Path(_crus_local_path('project', ''))
MAIN = ROOT / "data_audit/constrained_split_manifest.csv"
RESULTS = ROOT / "corrected_experiment_outputs"
CAL = RESULTS / "constrained_calibration_manifest.csv"
PROTOCOL = RESULTS / "final_pretest_protocol.json"

for path in [MAIN, CAL, PROTOCOL]:
    if not path.is_file():
        raise FileNotFoundError(path)

protocol = json.loads(PROTOCOL.read_text())

assert (
    hashlib.sha256(CAL.read_bytes()).hexdigest()
    == protocol["calibration_manifest_sha256"]
), "Calibration manifest hash differs from frozen protocol."

main = pd.read_csv(
    MAIN, dtype={"record_id": str, "group_id": str}
)
cal = pd.read_csv(
    CAL, dtype={"record_id": str, "group_id": str}
)

required = {"record_id", "group_id", "calibration_fold"}
assert required.issubset(cal.columns)
assert main["record_id"].is_unique
assert cal["record_id"].is_unique
assert cal[list(required)].notna().all().all()

training = main.loc[main["split"].eq("train")].copy()
training_ids = set(training["record_id"])
external_ids = set(
    main.loc[
        main["split"].isin(["validation", "test"]),
        "record_id",
    ]
)

assert set(cal["record_id"]) == training_ids, (
    "Calibration manifest does not contain exactly the training IDs."
)
assert not set(cal["record_id"]) & external_ids, (
    "Validation/test IDs appear in the calibration manifest."
)

joined = cal.merge(
    training[["record_id", "group_id", "label"]],
    on="record_id",
    suffixes=("_cal", "_main"),
    validate="one_to_one",
)

assert (
    joined["group_id_cal"] == joined["group_id_main"]
).all(), "Group IDs differ from the main manifest."

if "label_cal" in joined.columns:
    assert (
        joined["label_cal"].astype(str)
        == joined["label_main"].astype(str)
    ).all(), "Labels differ from the main manifest."

assert not cal.groupby("group_id")["calibration_fold"].nunique().gt(1).any(), (
    "A group is assigned to multiple calibration folds."
)

rows = []

for fold in sorted(cal["calibration_fold"].unique()):
    held_out = cal.loc[cal["calibration_fold"].eq(fold)]
    fitting = cal.loc[~cal["calibration_fold"].eq(fold)]

    assert len(held_out) > 0 and len(fitting) > 0
    assert not set(held_out["record_id"]) & set(fitting["record_id"])
    assert not set(held_out["group_id"]) & set(fitting["group_id"])

    rows.append({
        "fold": fold,
        "classifier_fit_records": len(fitting),
        "calibration_records": len(held_out),
        "record_overlap": 0,
        "group_overlap": 0,
    })

print("=== SAVED CALIBRATION MANIFEST ===")
print(pd.DataFrame(rows).to_string(index=False))
print("\nAll calibration records belong to the frozen training split.")
print("No validation/test records are present.")
print("Groups remain separated within each calibration fold.")

if "calibration_splits" in globals() and "train_ids" in globals():
    assert isinstance(calibration_splits, (list, tuple)), (
        "Runtime calibration_splits is not a list/tuple; "
        "inspect its construction rather than consuming a generator."
    )

    runtime_ids = np.asarray(train_ids, dtype=str)
    assert len(set(runtime_ids)) == len(runtime_ids)
    assert set(runtime_ids) == training_ids

    group_lookup = training.set_index("record_id")["group_id"]
    fold_lookup = cal.set_index("record_id")["calibration_fold"]
    all_positions = set(range(len(runtime_ids)))
    seen_calibration_positions = []

    for number, (fit_idx, cal_idx) in enumerate(
        calibration_splits, start=1
    ):
        fit_idx = np.asarray(fit_idx)
        cal_idx = np.asarray(cal_idx)

        for indices in [fit_idx, cal_idx]:
            assert indices.ndim == 1
            assert np.issubdtype(indices.dtype, np.integer)
            assert len(indices) > 0
            assert len(set(indices.tolist())) == len(indices)
            assert ((indices >= 0) & (indices < len(runtime_ids))).all()

        assert not set(fit_idx) & set(cal_idx)
        assert set(fit_idx) | set(cal_idx) == all_positions

        fit_ids = runtime_ids[fit_idx]
        held_ids = runtime_ids[cal_idx]

        assert not (
            set(group_lookup.loc[fit_ids])
            & set(group_lookup.loc[held_ids])
        )

        held_folds = fold_lookup.loc[held_ids].unique()
        assert len(held_folds) == 1

        expected_held_ids = set(
            cal.loc[
                cal["calibration_fold"].eq(held_folds[0]),
                "record_id",
            ]
        )
        assert set(held_ids) == expected_held_ids

        seen_calibration_positions.extend(cal_idx.tolist())
        print(f"Runtime fold {number}: matches saved grouped fold.")

    assert sorted(seen_calibration_positions) == list(
        range(len(runtime_ids))
    ), "Runtime calibration folds do not partition training records once."

    print("\nPASS: runtime folds also match the saved manifest.")
else:
    print(
        "\nRuntime fold variables are unavailable. "
        "Saved assignments passed, but actual runtime indices "
        "were not inspected."
    )

print("\nFrozen baseline settings:")
print(protocol.get("baseline_configuration", {}))

In [ ]:
from pathlib import Path
pass  # Colab service not required locally.

_crus_prepare_saved_workspace()

ROOT = Path(_crus_local_path('project', ''))

required_files = [
    ROOT / "data_audit/constrained_split_manifest.csv",
    ROOT / "corrected_experiment_outputs/constrained_calibration_manifest.csv",
    ROOT / "corrected_experiment_outputs/final_pretest_protocol.json",
]

print("\n=== REQUIRED FILES ===")
for path in required_files:
    print(f"{'FOUND' if path.is_file() else 'MISSING'}: {path}")

if not all(path.is_file() for path in required_files):
    mydrive = Path(_crus_local_path('drive', ''))

    print("\nMyDrive accessible:", mydrive.exists())
    print("Project folder accessible:", ROOT.exists())

    if mydrive.exists():
        print("\nTop-level Drive folders:")
        for path in sorted(mydrive.iterdir()):
            if path.is_dir():
                print(path.name)

    raise FileNotFoundError(
        "Some required files remain unavailable. "
        "Check that you mounted the same Google account "
        "used for the earlier successful verification."
    )

print("\nAll required files are accessible.")
print("Now rerun the calibration verification script.")

In [ ]:
from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd

In [ ]:
from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd

ROOT = Path(_crus_local_path('project', ''))
MAIN = ROOT / "data_audit/constrained_split_manifest.csv"
RESULTS = ROOT / "corrected_experiment_outputs"
CAL = RESULTS / "constrained_calibration_manifest.csv"
PROTOCOL = RESULTS / "final_pretest_protocol.json"

for path in [MAIN, CAL, PROTOCOL]:
    if not path.is_file():
        raise FileNotFoundError(path)

protocol = json.loads(PROTOCOL.read_text())

assert (
    hashlib.sha256(CAL.read_bytes()).hexdigest()
    == protocol["calibration_manifest_sha256"]
), "Calibration manifest hash differs from frozen protocol."

main = pd.read_csv(
    MAIN, dtype={"record_id": str, "group_id": str}
)
cal = pd.read_csv(
    CAL, dtype={"record_id": str, "group_id": str}
)

required = {"record_id", "group_id", "calibration_fold"}
assert required.issubset(cal.columns)
assert main["record_id"].is_unique
assert cal["record_id"].is_unique
assert cal[list(required)].notna().all().all()

training = main.loc[main["split"].eq("train")].copy()
training_ids = set(training["record_id"])
external_ids = set(
    main.loc[
        main["split"].isin(["validation", "test"]),
        "record_id",
    ]
)

assert set(cal["record_id"]) == training_ids, (
    "Calibration manifest does not contain exactly the training IDs."
)
assert not set(cal["record_id"]) & external_ids, (
    "Validation/test IDs appear in the calibration manifest."
)

joined = cal.merge(
    training[["record_id", "group_id", "label"]],
    on="record_id",
    suffixes=("_cal", "_main"),
    validate="one_to_one",
)

assert (
    joined["group_id_cal"] == joined["group_id_main"]
).all(), "Group IDs differ from the main manifest."

if "label_cal" in joined.columns:
    assert (
        joined["label_cal"].astype(str)
        == joined["label_main"].astype(str)
    ).all(), "Labels differ from the main manifest."

assert not cal.groupby("group_id")["calibration_fold"].nunique().gt(1).any(), (
    "A group is assigned to multiple calibration folds."
)

rows = []

for fold in sorted(cal["calibration_fold"].unique()):
    held_out = cal.loc[cal["calibration_fold"].eq(fold)]
    fitting = cal.loc[~cal["calibration_fold"].eq(fold)]

    assert len(held_out) > 0 and len(fitting) > 0
    assert not set(held_out["record_id"]) & set(fitting["record_id"])
    assert not set(held_out["group_id"]) & set(fitting["group_id"])

    rows.append({
        "fold": fold,
        "classifier_fit_records": len(fitting),
        "calibration_records": len(held_out),
        "record_overlap": 0,
        "group_overlap": 0,
    })

print("=== SAVED CALIBRATION MANIFEST ===")
print(pd.DataFrame(rows).to_string(index=False))
print("\nAll calibration records belong to the frozen training split.")
print("No validation/test records are present.")
print("Groups remain separated within each calibration fold.")

if "calibration_splits" in globals() and "train_ids" in globals():
    assert isinstance(calibration_splits, (list, tuple)), (
        "Runtime calibration_splits is not a list/tuple; "
        "inspect its construction rather than consuming a generator."
    )

    runtime_ids = np.asarray(train_ids, dtype=str)
    assert len(set(runtime_ids)) == len(runtime_ids)
    assert set(runtime_ids) == training_ids

    group_lookup = training.set_index("record_id")["group_id"]
    fold_lookup = cal.set_index("record_id")["calibration_fold"]
    all_positions = set(range(len(runtime_ids)))
    seen_calibration_positions = []

    for number, (fit_idx, cal_idx) in enumerate(
        calibration_splits, start=1
    ):
        fit_idx = np.asarray(fit_idx)
        cal_idx = np.asarray(cal_idx)

        for indices in [fit_idx, cal_idx]:
            assert indices.ndim == 1
            assert np.issubdtype(indices.dtype, np.integer)
            assert len(indices) > 0
            assert len(set(indices.tolist())) == len(indices)
            assert ((indices >= 0) & (indices < len(runtime_ids))).all()

        assert not set(fit_idx) & set(cal_idx)
        assert set(fit_idx) | set(cal_idx) == all_positions

        fit_ids = runtime_ids[fit_idx]
        held_ids = runtime_ids[cal_idx]

        assert not (
            set(group_lookup.loc[fit_ids])
            & set(group_lookup.loc[held_ids])
        )

        held_folds = fold_lookup.loc[held_ids].unique()
        assert len(held_folds) == 1

        expected_held_ids = set(
            cal.loc[
                cal["calibration_fold"].eq(held_folds[0]),
                "record_id",
            ]
        )
        assert set(held_ids) == expected_held_ids

        seen_calibration_positions.extend(cal_idx.tolist())
        print(f"Runtime fold {number}: matches saved grouped fold.")

    assert sorted(seen_calibration_positions) == list(
        range(len(runtime_ids))
    ), "Runtime calibration folds do not partition training records once."

    print("\nPASS: runtime folds also match the saved manifest.")
else:
    print(
        "\nRuntime fold variables are unavailable. "
        "Saved assignments passed, but actual runtime indices "
        "were not inspected."
    )

print("\nFrozen baseline settings:")
print(protocol.get("baseline_configuration", {}))

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path(_crus_local_path('project', ''))
RESULTS = ROOT / "corrected_experiment_outputs"

PRED_PATH = RESULTS / "test_all_method_predictions.csv"
GATE_PATH = RESULTS / "test_frozen_CRUS_acceptance_audit.csv"

if not ROOT.exists():
    pass  # Colab service not required locally.
    _crus_prepare_saved_workspace()

for path in [PRED_PATH, GATE_PATH]:
    if not path.is_file():
        raise FileNotFoundError(path)

pred = pd.read_csv(PRED_PATH, dtype={"record_id": str})
gate = pd.read_csv(GATE_PATH, dtype={"record_id": str})

def parse_bool(series):
    mapping = {
        "true": True,
        "false": False,
        "1": True,
        "0": False,
        "1.0": True,
        "0.0": False,
    }
    values = series.astype(str).str.strip().str.lower()
    unexpected = set(values) - set(mapping)

    if unexpected:
        raise ValueError(f"Unexpected Boolean values: {unexpected}")

    return values.map(mapping).astype(bool)

keys = ["method", "record_id"]

assert not pred.duplicated(keys).any(), (
    "Duplicate method/record pairs in predictions."
)
assert not gate.duplicated(keys).any(), (
    "Duplicate method/record pairs in acceptance audit."
)

assert pred[
    keys + ["true_label", "prediction"]
].notna().all().all()

gate["accepted_bool"] = parse_bool(gate["accepted"])

pred["correct_recomputed"] = (
    pred["prediction"].astype(str)
    == pred["true_label"].astype(str)
)

if "correct" in pred.columns:
    assert np.array_equal(
        parse_bool(pred["correct"]).to_numpy(),
        pred["correct_recomputed"].to_numpy(),
    ), "Saved prediction correctness is inconsistent."

rows = []

for method, gate_part in gate.groupby("method", sort=True):
    pred_part = pred.loc[pred["method"].eq(method)].copy()

    assert len(pred_part) > 0, f"No predictions for {method}."
    assert set(gate_part["record_id"]) == set(pred_part["record_id"]), (
        f"{method}: gate and prediction record sets differ."
    )

    joined = gate_part.merge(
        pred_part[keys + ["correct_recomputed"]],
        on=keys,
        how="left",
        validate="one_to_one",
    )

    assert joined["correct_recomputed"].notna().all()

    if "correct" in joined.columns:
        assert np.array_equal(
            parse_bool(joined["correct"]).to_numpy(),
            joined["correct_recomputed"].to_numpy(),
        ), f"{method}: acceptance audit correctness is inconsistent."

    accepted = joined["accepted_bool"].to_numpy(dtype=bool)
    correct = joined["correct_recomputed"].to_numpy(dtype=bool)

    N = len(joined)
    n_correct_forced = int(correct.sum())
    n_accepted = int(accepted.sum())
    n_abstained = N - n_accepted
    n_correct_accepted = int((accepted & correct).sum())
    n_errors_accepted = int((accepted & ~correct).sum())

    assert n_accepted + n_abstained == N
    assert n_correct_accepted + n_errors_accepted == n_accepted

    selective_accuracy = (
        n_correct_accepted / n_accepted
        if n_accepted else np.nan
    )
    accepted_error = (
        n_errors_accepted / n_accepted
        if n_accepted else np.nan
    )

    penalized_accuracy = n_correct_accepted / N
    penalized_error = (n_abstained + n_errors_accepted) / N

    assert np.isclose(penalized_accuracy + penalized_error, 1.0)

    if n_accepted:
        assert np.isclose(
            selective_accuracy + accepted_error, 1.0
        )
        assert np.isclose(
            penalized_accuracy,
            (n_accepted / N) * selective_accuracy,
        )

    rows.append({
        "method": method,
        "N": N,
        "correct_before_abstention": n_correct_forced,
        "n_accepted": n_accepted,
        "n_correct_accepted": n_correct_accepted,
        "n_errors_accepted": n_errors_accepted,
        "n_abstained": n_abstained,
        "pre_abstention_accuracy_pct": 100 * n_correct_forced / N,
        "coverage_pct": 100 * n_accepted / N,
        "selective_accuracy_pct": 100 * selective_accuracy,
        "accepted_misclassification_pct": 100 * accepted_error,
        "abstention_penalized_accuracy_pct": 100 * penalized_accuracy,
        "abstention_penalized_error_pct": 100 * penalized_error,
    })

summary = pd.DataFrame(rows)

print("=== RECOMPUTED FROZEN-GATE METRICS ===")
print(
    summary.to_string(
        index=False,
        float_format=lambda value: f"{value:.4f}",
    )
)

print("\n=== REVIEWER'S ORIGINAL EXAMPLE ===")
original_N = 259
original_accepted = 171
original_abstained = 88
original_accepted_errors = 1
original_correct_accepted = (
    original_accepted - original_accepted_errors
)

assert original_accepted + original_abstained == original_N

print(
    "Abstention-penalized accuracy: "
    f"{original_correct_accepted}/{original_N} = "
    f"{100 * original_correct_accepted / original_N:.2f}%"
)
print(
    "Abstention-penalized error: "
    f"{original_abstained + original_accepted_errors}/{original_N} = "
    f"{100 * (original_abstained + original_accepted_errors) / original_N:.2f}%"
)

print(
    "\nPASS: counts and denominator identities are consistent "
    "for all methods in the frozen CRUS acceptance audit."
)
print("No models were retrained and no existing files were modified.")

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path(_crus_local_path('project', ''))
RESULTS = ROOT / "corrected_experiment_outputs"

if not ROOT.exists():
    pass  # Colab service not required locally.
    _crus_prepare_saved_workspace()

PRED_PATH = RESULTS / "test_all_method_predictions.csv"
AUDIT_PATH = RESULTS / "test_matched_count_acceptance_audit.csv"
SUMMARY_PATH = RESULTS / "test_matched_count_results.csv"

OUT = RESULTS / "R1_3_consistency_audit"
OUT.mkdir(parents=True, exist_ok=True)

for path in [PRED_PATH, AUDIT_PATH, SUMMARY_PATH]:
    if not path.is_file():
        raise FileNotFoundError(path)

pred = pd.read_csv(PRED_PATH, dtype={"record_id": str})
audit = pd.read_csv(AUDIT_PATH, dtype={"record_id": str})
saved = pd.read_csv(SUMMARY_PATH)

def parse_bool(series):
    mapping = {
        "true": True, "false": False,
        "1": True, "0": False,
        "1.0": True, "0.0": False,
    }
    values = series.astype(str).str.strip().str.lower()
    unexpected = set(values) - set(mapping)
    if unexpected:
        raise ValueError(f"Unexpected Boolean values: {unexpected}")
    return values.map(mapping).astype(bool)

pred_keys = ["method", "record_id"]
audit_keys = ["method", "target_n_accepted", "record_id"]

assert not pred.duplicated(pred_keys).any()
assert not audit.duplicated(audit_keys).any()

assert pred[
    pred_keys + ["true_label", "prediction"]
].notna().all().all()

assert audit[
    audit_keys + ["accepted"]
].notna().all().all()

audit["accepted_recomputed_bool"] = parse_bool(audit["accepted"])

pred["correct_recomputed"] = (
    pred["prediction"].astype(str)
    == pred["true_label"].astype(str)
)

joined = audit.merge(
    pred[
        pred_keys
        + ["correct_recomputed"]
    ],
    on=pred_keys,
    how="left",
    validate="many_to_one",
    indicator=True,
)

assert joined["_merge"].eq("both").all(), (
    "Some acceptance-audit records lack matching predictions."
)

if "correct" in joined.columns:
    assert np.array_equal(
        parse_bool(joined["correct"]).to_numpy(),
        joined["correct_recomputed"].to_numpy(),
    ), "Saved correctness differs from predictions."

rows = []

for (method, target), part in joined.groupby(
    ["method", "target_n_accepted"], sort=True
):
    method_pred = pred.loc[pred["method"].eq(method)]

    assert set(part["record_id"]) == set(method_pred["record_id"]), (
        f"{method}, target={target}: incomplete test record set."
    )

    target = int(target)
    accepted = part["accepted_recomputed_bool"].to_numpy(dtype=bool)
    correct = part["correct_recomputed"].to_numpy(dtype=bool)

    N = len(part)
    A = int(accepted.sum())
    C = int((accepted & correct).sum())
    E = int((accepted & ~correct).sum())
    B = N - A

    assert N == 257, f"{method}: unexpected test-set size."
    assert A == target, f"{method}: accepted count differs from target."
    assert C + E == A
    assert A + B == N
    assert A > 0

    rows.append({
        "method": method,
        "N": N,
        "target_n_accepted": target,
        "n_accepted": A,
        "n_correct_accepted": C,
        "n_errors_accepted": E,
        "n_abstained": B,
        "coverage": A / N,
        "selective_accuracy": C / A,
        "accepted_misclassification_rate": E / A,
        "forced_accuracy": float(correct.mean()),
        "abstention_penalized_accuracy": C / N,
        "abstention_penalized_error": (B + E) / N,
    })

verified = pd.DataFrame(rows)

comparison_keys = ["method", "target_n_accepted"]
assert not saved.duplicated(comparison_keys).any(), (
    "Saved summary has multiple rows per method/target. "
    "Inspect mode/selection-rule fields before comparing."
)

comparison = verified.merge(
    saved,
    on=comparison_keys,
    suffixes=("_verified", "_saved"),
    how="outer",
    indicator=True,
    validate="one_to_one",
)

assert comparison["_merge"].eq("both").all(), (
    "Verified and saved summaries contain different method/target rows."
)

metrics = [
    "N",
    "n_accepted",
    "n_correct_accepted",
    "n_errors_accepted",
    "n_abstained",
    "coverage",
    "selective_accuracy",
    "accepted_misclassification_rate",
    "forced_accuracy",
    "abstention_penalized_accuracy",
    "abstention_penalized_error",
]

mismatches = []

for metric in metrics:
    left = f"{metric}_verified"
    right = f"{metric}_saved"

    if left not in comparison or right not in comparison:
        continue

    matches = np.isclose(
        comparison[left].to_numpy(dtype=float),
        comparison[right].to_numpy(dtype=float),
        rtol=1e-8,
        atol=1e-10,
        equal_nan=True,
    )
    comparison[f"{metric}_matches"] = matches

    if not matches.all():
        mismatches.append(metric)

comparison.to_csv(
    OUT / "matched_count_summary_comparison.csv",
    index=False,
)

verified["coverage_percent"] = 100 * verified["coverage"]
verified["selective_accuracy_percent"] = (
    100 * verified["selective_accuracy"]
)
verified["selective_accuracy_display"] = (
    verified["selective_accuracy_percent"]
    .map(lambda value: f"{value:.2f}%")
)

verified.to_csv(
    OUT / "verified_matched_count_reporting.csv",
    index=False,
)

print("=== VERIFIED MATCHED-COUNT RESULTS ===")
print(
    verified[[
        "method",
        "N",
        "target_n_accepted",
        "n_correct_accepted",
        "n_errors_accepted",
        "coverage_percent",
        "selective_accuracy_percent",
    ]].to_string(
        index=False,
        float_format=lambda value: f"{value:.4f}",
    )
)

print("\nSaved-summary discrepancies:", mismatches or "None")
print("\nReporting file:", OUT / "verified_matched_count_reporting.csv")

if mismatches:
    raise AssertionError(
        "Saved summary differs from recomputed counts. "
        "Resolve these discrepancies before updating the manuscript."
    )

print(
    "\nPASS: matched-count summary agrees with "
    "case-level predictions and acceptance decisions."
)

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

ROOT = Path(_crus_local_path('project', ''))

if not ROOT.exists():
    pass  # Colab service not required locally.
    _crus_prepare_saved_workspace()

RESULTS = ROOT / "corrected_experiment_outputs"
OUT = RESULTS / "R1_3_consistency_audit" / "figures"
OUT.mkdir(parents=True, exist_ok=True)

pred_path = RESULTS / "test_all_method_predictions.csv"
gate_path = RESULTS / "test_frozen_CRUS_acceptance_audit.csv"
protocol_path = RESULTS / "final_pretest_protocol.json"

for path in [pred_path, gate_path, protocol_path]:
    if not path.is_file():
        raise FileNotFoundError(path)

pred = pd.read_csv(pred_path, dtype={"record_id": str})
gate = pd.read_csv(gate_path, dtype={"record_id": str})
protocol = json.loads(protocol_path.read_text())

classes = [str(label) for label in protocol["classes"]]
assert len(classes) == 17
assert len(set(classes)) == 17

pred = pred.loc[pred["method"].eq("Cascade")].copy()
gate = gate.loc[gate["method"].eq("Cascade")].copy()

assert pred["record_id"].is_unique
assert gate["record_id"].is_unique
assert len(pred) == len(gate) == 257
assert set(pred["record_id"]) == set(gate["record_id"])

mapping = {
    "true": True, "false": False,
    "1": True, "0": False,
    "1.0": True, "0.0": False,
}
values = gate["accepted"].astype(str).str.strip().str.lower()
assert values.isin(mapping).all(), "Invalid acceptance values."
gate["accepted_bool"] = values.map(mapping).astype(bool)

joined = pred.merge(
    gate[["record_id", "accepted_bool"]],
    on="record_id",
    validate="one_to_one",
)

assert joined[["true_label", "prediction"]].notna().all().all()
assert set(joined["true_label"].astype(str)).issubset(classes)
assert set(joined["prediction"].astype(str)).issubset(classes)

accepted = joined.loc[joined["accepted_bool"]].copy()

y_true = accepted["true_label"].astype(str)
y_pred = accepted["prediction"].astype(str)

N = len(joined)
A = len(accepted)
C = int((y_true == y_pred).sum())
E = A - C

assert (N, A, C, E) == (257, 117, 109, 8), (
    f"Unexpected counts: N={N}, accepted={A}, "
    f"correct={C}, errors={E}"
)

cm = confusion_matrix(y_true, y_pred, labels=classes)
assert int(cm.sum()) == A
assert int(np.trace(cm)) == C
assert int(cm.sum() - np.trace(cm)) == E

pd.DataFrame(
    cm,
    index=pd.Index(classes, name="True"),
    columns=pd.Index(classes, name="Predicted"),
).to_csv(OUT / "confusion_matrix_cascade_native_crus_counts.csv")

fig, ax = plt.subplots(figsize=(13, 11))
display = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=classes,
)
display.plot(
    ax=ax,
    cmap="Blues",
    values_format="d",
    colorbar=True,
    xticks_rotation=90,
    im_kw={"interpolation": "nearest"},
)

for text in display.text_.ravel():
    text.set_fontsize(8)

ax.set_title(
    "Cascade — native frozen CRUS gate\n"
    f"Accepted n={A}; selective accuracy={100*C/A:.2f}%; "
    f"coverage={100*A/N:.2f}%"
)
ax.set_xlabel("Predicted section class")
ax.set_ylabel("True section class")

fig.tight_layout()

png_path = OUT / "confusion_matrix_cascade_native_crus_corrected.png"
pdf_path = OUT / "confusion_matrix_cascade_native_crus_corrected.pdf"

fig.savefig(png_path, dpi=300, bbox_inches="tight")
fig.savefig(pdf_path, bbox_inches="tight")
plt.show()
plt.close(fig)

print(f"Accepted: {A}; correct: {C}; errors: {E}")
print(f"Selective accuracy: {100*C/A:.4f}%")
print(f"Coverage: {100*A/N:.4f}%")
print("PNG:", png_path)
print("PDF:", pdf_path)

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path(_crus_local_path('project', ''))

if not ROOT.exists():
    pass  # Colab service not required locally.
    _crus_prepare_saved_workspace()

RESULTS = ROOT / "corrected_experiment_outputs"
OUT = RESULTS / "R1_3_consistency_audit" / "figures"
OUT.mkdir(parents=True, exist_ok=True)

AUDIT = RESULTS / "validation_retrieval_prediction_audit.csv"
PROTOCOL = RESULTS / "final_pretest_protocol.json"

for path in [AUDIT, PROTOCOL]:
    if not path.is_file():
        raise FileNotFoundError(path)

protocol = json.loads(PROTOCOL.read_text())
policy = protocol["frozen_policies"]["Cascade:CRUS"]

tau_c_selected = float(policy["tau_c"])
tau_u_selected = float(policy["tau_u"])

df = pd.read_csv(AUDIT, dtype={"record_id": str})
df = df.loc[df["method"].eq("Cascade")].copy()

required = {
    "record_id", "true_label", "prediction", "confidence", "CRUS"
}
assert required.issubset(df.columns)
assert df["record_id"].is_unique
assert len(df) == protocol["split_sizes"]["validation"] == 258
assert df[list(required)].notna().all().all()

confidence = df["confidence"].to_numpy(dtype=float)
crus = df["CRUS"].to_numpy(dtype=float)
correct = (
    df["true_label"].astype(str)
    == df["prediction"].astype(str)
).to_numpy()

assert np.isfinite(confidence).all()
assert np.isfinite(crus).all()
assert ((confidence >= 0) & (confidence <= 1)).all()
assert ((crus >= 0) & (crus <= 1)).all()

thresholds = np.unique(
    np.concatenate([
        np.linspace(0.0, 1.0, 21),
        [tau_c_selected],
    ])
)

selected_mask = (
    (confidence >= tau_c_selected)
    & (crus <= tau_u_selected)
)

rows = []

for threshold in thresholds:
    accepted = (
        (confidence >= threshold)
        & (crus <= tau_u_selected)
    )

    A = int(accepted.sum())
    C = int((accepted & correct).sum())
    E = int((accepted & ~correct).sum())

    rows.append({
        "tau_c": float(threshold),
        "tau_u": tau_u_selected,
        "N": len(df),
        "n_accepted": A,
        "n_correct_accepted": C,
        "n_errors_accepted": E,
        "coverage": A / len(df),
        "selective_accuracy": C / A if A else np.nan,
        "accepted_misclassification_rate": E / A if A else np.nan,
        "same_subset_as_selected": bool(
            np.array_equal(accepted, selected_mask)
        ),
    })

sweep = pd.DataFrame(rows)
sweep.to_csv(
    OUT / "threshold_sweep_validation_corrected.csv",
    index=False,
)

fig, axes = plt.subplots(
    2, 1, figsize=(9, 7), sharex=True,
    constrained_layout=True,
)

axes[0].plot(
    sweep["tau_c"],
    100 * sweep["coverage"],
    marker="o",
    color="tab:blue",
    label="Coverage",
)
axes[0].set_ylabel("Coverage (%)")

axes[1].plot(
    sweep["tau_c"],
    100 * sweep["selective_accuracy"],
    marker="s",
    color="tab:green",
    label="Selective accuracy",
)
axes[1].plot(
    sweep["tau_c"],
    100 * sweep["accepted_misclassification_rate"],
    marker="^",
    color="tab:red",
    label="Accepted-case misclassification",
)
axes[1].set_ylabel("Accepted-case metrics (%)")
axes[1].set_xlabel(r"Confidence threshold $\tau_c$")

for ax in axes:
    ax.axvline(
        tau_c_selected,
        color="gray",
        linestyle="--",
        label=rf"Frozen $\tau_c={tau_c_selected:g}$",
    )
    ax.set_ylim(-2, 102)
    ax.set_xlim(-0.02, 1.02)
    ax.grid(alpha=0.25)
    ax.legend(fontsize=9)

fig.suptitle(
    "Cascade — corrected validation threshold sensitivity\n"
    rf"$n=258$; fixed $\tau_u={tau_u_selected:.6f}$"
)

png = OUT / "threshold_sweep_validation_corrected.png"
pdf = OUT / "threshold_sweep_validation_corrected.pdf"

fig.savefig(png, dpi=300, bbox_inches="tight")
fig.savefig(pdf, bbox_inches="tight")
plt.show()
plt.close(fig)

selected = sweep.loc[
    sweep["tau_c"].eq(tau_c_selected)
].iloc[0]

print("=== FROZEN VALIDATION OPERATING POINT ===")
print(f"tau_c: {tau_c_selected:.17g}")
print(f"tau_u: {tau_u_selected:.17g}")
print(f"Accepted: {int(selected['n_accepted'])}/{len(df)}")
print(f"Correct accepted: {int(selected['n_correct_accepted'])}")
print(f"Accepted errors: {int(selected['n_errors_accepted'])}")
print(f"Coverage: {100*selected['coverage']:.4f}%")
print(
    "Selective accuracy: "
    f"{100*selected['selective_accuracy']:.4f}%"
)
print(
    "Accepted-case misclassification: "
    f"{100*selected['accepted_misclassification_rate']:.4f}%"
)

print("\n=== SUBSET CHECK AT EACH SWEPT THRESHOLD ===")
print(
    sweep[[
        "tau_c", "n_accepted", "same_subset_as_selected"
    ]].to_string(index=False)
)

print("\nPNG:", png)
print("PDF:", pdf)
print("Frozen policy unchanged; no model predictions regenerated.")

In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path(_crus_local_path('project', ''))

if not ROOT.exists():
    pass  # Colab service not required locally.
    _crus_prepare_saved_workspace()

POOL = ROOT / "data_audit/constrained_dialogue_pool_with_splits.csv"
RESULTS = ROOT / "corrected_experiment_outputs"

if not POOL.is_file():
    raise FileNotFoundError(POOL)

df = pd.read_csv(
    POOL,
    dtype={"record_id": str, "group_id": str},
)

required = {"record_id", "group_id", "label", "split", "text"}
assert required.issubset(df.columns)
assert df["record_id"].is_unique
assert df[list(required)].notna().all().all()

support = (
    df.groupby("label")
    .agg(
        records=("record_id", "size"),
        distinct_groups=("group_id", "nunique"),
    )
    .sort_values(["distinct_groups", "records"])
)

group_support_by_split = (
    df.groupby(["label", "split"])["group_id"]
    .nunique()
    .unstack(fill_value=0)
    .reindex(columns=["train", "validation", "test"], fill_value=0)
)

print("=== CLASS SUPPORT ===")
print(support.to_string())

print("\n=== DISTINCT GROUPS PER CLASS IN CURRENT SPLIT ===")
print(group_support_by_split.to_string())

print("\n=== GROUPS CONTAINING MULTIPLE LABELS ===")
multi_label_groups = df.groupby("group_id")["label"].nunique().gt(1)
print("Count:", int(multi_label_groups.sum()))

print("\n=== EXISTING UNCERTAINTY OUTPUTS ===")
for filename in [
    "supplementary_frozen_policy_intervals.csv",
    "supplementary_frozen_policy_paired_differences.csv",
]:
    path = RESULTS / filename
    print(f"\n{filename}: {'FOUND' if path.is_file() else 'MISSING'}")
    if path.is_file():
        table = pd.read_csv(path)
        print("Columns:", table.columns.tolist())
        print(table.to_string(index=False))

In [ ]:
from pathlib import Path
import json

NOTEBOOK = Path(
    _crus_local_path('notebooks', 'revised_risk_aware_hybrid_pipeline_v2.ipynb')
)

if not NOTEBOOK.is_file():
    raise FileNotFoundError(NOTEBOOK)

nb = json.loads(NOTEBOOK.read_text(encoding="utf-8"))

# Current 1-based notebook positions:
# constrained splitting and calibration-fold construction.
selected_cells = {55, 59, 60, 61}

for number, cell in enumerate(nb.get("cells", []), start=1):
    if number not in selected_cells:
        continue
    if cell.get("cell_type") != "code":
        continue

    source = cell.get("source", [])
    source = "".join(source) if isinstance(source, list) else source

    print(f"\n{'=' * 70}")
    print(f"CELL {number}")
    print("=" * 70)
    print(source)

In [ ]:
from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd
import scipy

from scipy.optimize import milp, Bounds, LinearConstraint
from scipy.sparse import lil_matrix

ROOT = Path(_crus_local_path('project', ''))

if not ROOT.exists():
    pass  # Colab service not required locally.
    _crus_prepare_saved_workspace()

SOURCE = ROOT / "data_audit/constrained_dialogue_pool_with_splits.csv"
OUT = ROOT / "supplementary_repeated_splits"

if not SOURCE.is_file():
    raise FileNotFoundError(SOURCE)

OUT.mkdir(parents=True, exist_ok=True)

SEEDS = [11, 22, 33, 44, 55]
PARTITIONS = ["train", "validation", "test"]
FRACTIONS = np.array([0.60, 0.20, 0.20])
CALIBRATION_FOLDS = 3

data = pd.read_csv(
    SOURCE,
    dtype={"record_id": str, "group_id": str},
).reset_index(drop=True)

required = {
    "record_id", "group_id", "label",
    "dialogue_hash", "text", "dialogue",
}

assert required.issubset(data.columns), (
    f"Missing columns: {sorted(required - set(data.columns))}"
)
assert data["record_id"].is_unique
assert data[list(required)].notna().all().all()
assert len(data) == 1288

def normalize(value):
    return " ".join(str(value).split()).casefold()

assert data["text"].map(normalize).eq(
    data["dialogue"].map(normalize)
).all(), "Model text differs from normalized dialogue."

assert data.groupby("dialogue_hash")["group_id"].nunique().max() == 1, (
    "An exact-dialogue hash belongs to multiple groups."
)

CLASSES = sorted(data["label"].unique())
assert len(CLASSES) == 17

def allocate(frame, sizes, lower_class, upper_class, seed):
    sizes = np.asarray(sizes, dtype=int)
    lower_class = np.asarray(lower_class, dtype=int)
    upper_class = np.asarray(upper_class, dtype=int)

    counts = (
        pd.crosstab(frame["group_id"], frame["label"])
        .reindex(columns=CLASSES, fill_value=0)
        .sort_index()
    )

    groups = counts.index.to_numpy()
    values = counts.to_numpy()
    group_sizes = values.sum(axis=1)

    G, C = values.shape
    P = len(sizes)

    assert sizes.sum() == len(frame)
    assert lower_class.shape == upper_class.shape == (C, P)

    if np.any(lower_class > upper_class):
        raise ValueError("Incompatible class-support constraints.")

    A = lil_matrix((G + P + C * P, G * P), dtype=float)
    lo = np.zeros(A.shape[0])
    hi = np.zeros(A.shape[0])
    row = 0

    for g in range(G):
        A[row, g * P:(g + 1) * P] = 1
        lo[row] = hi[row] = 1
        row += 1

    for p in range(P):
        for g in range(G):
            A[row, g * P + p] = group_sizes[g]
        lo[row] = hi[row] = sizes[p]
        row += 1

    for c in range(C):
        for p in range(P):
            for g in np.flatnonzero(values[:, c]):
                A[row, g * P + p] = values[g, c]
            lo[row] = lower_class[c, p]
            hi[row] = upper_class[c, p]
            row += 1

    result = milp(
        c=np.random.default_rng(seed).random(G * P),
        integrality=np.ones(G * P, dtype=int),
        bounds=Bounds(np.zeros(G * P), np.ones(G * P)),
        constraints=LinearConstraint(A.tocsr(), lo, hi),
        options={"time_limit": 120},
    )

    if result.x is None:
        raise RuntimeError(
            f"No allocation returned. Status={result.status}; "
            f"{result.message}"
        )

    assignment = result.x.reshape(G, P)
    assert np.allclose(
        assignment, np.round(assignment), atol=1e-6
    ), "Non-integer allocation returned."

    assignment = np.round(assignment).astype(int)

    assert np.isin(assignment, [0, 1]).all()
    assert (assignment.sum(axis=1) == 1).all()

    group_mapping = dict(zip(groups, assignment.argmax(axis=1)))
    membership = (
        frame["group_id"].map(group_mapping).to_numpy(dtype=int)
    )

    actual_sizes = np.bincount(membership, minlength=P)
    actual_counts = np.column_stack([
        frame.loc[membership == p, "label"]
        .value_counts()
        .reindex(CLASSES, fill_value=0)
        .to_numpy()
        for p in range(P)
    ])

    assert np.array_equal(actual_sizes, sizes)
    assert (actual_counts >= lower_class).all()
    assert (actual_counts <= upper_class).all()

    return membership, {
        "solver_status": int(result.status),
        "solver_message": str(result.message),
        "verified_feasible": True,
    }

def write_preserving(path, content):
    if path.exists():
        if path.read_bytes() != content:
            raise RuntimeError(
                f"Different file already exists; not overwritten: {path}"
            )
    else:
        path.write_bytes(content)

raw_sizes = FRACTIONS * len(data)
split_sizes = np.floor(raw_sizes).astype(int)

remainder_order = np.argsort(
    -(raw_sizes - split_sizes),
    kind="stable",
)

remaining = int(len(data) - split_sizes.sum())
for p in remainder_order[:remaining]:
    split_sizes[p] += 1

assert split_sizes.tolist() == [773, 258, 257]

totals = (
    data["label"].value_counts()
    .reindex(CLASSES).to_numpy()
)

lower = np.floor(totals[:, None] * FRACTIONS).astype(int)
upper = np.ceil(totals[:, None] * FRACTIONS).astype(int)
lower = np.maximum(lower, np.array([3, 1, 1]))

status_rows = []

for seed in SEEDS:
    destination = OUT / f"seed_{seed}"
    destination.mkdir(parents=True, exist_ok=True)

    print(f"\n=== SEED {seed} ===")

    try:
        membership, split_info = allocate(
            data, split_sizes, lower, upper, seed
        )

        repeat = data.copy()
        repeat["split"] = np.asarray(PARTITIONS)[membership]

        assert repeat.groupby("group_id")["split"].nunique().max() == 1
        assert repeat.groupby("dialogue_hash")["split"].nunique().max() == 1

        training = repeat.loc[
            repeat["split"].eq("train")
        ].copy().reset_index(drop=True)

        K = CALIBRATION_FOLDS
        fold_sizes = np.full(K, len(training) // K)
        fold_sizes[:len(training) % K] += 1

        train_totals = (
            training["label"].value_counts()
            .reindex(CLASSES, fill_value=0).to_numpy()
        )

        cal_lower = np.repeat(
            np.maximum(
                np.floor(train_totals / K).astype(int), 1
            )[:, None],
            K,
            axis=1,
        )

        cal_upper = np.repeat(
            np.ceil(train_totals / K).astype(int)[:, None],
            K,
            axis=1,
        )

        folds, cal_info = allocate(
            training,
            fold_sizes,
            cal_lower,
            cal_upper,
            seed,
        )

        cal_manifest = training[
            ["record_id", "group_id", "label"]
        ].copy()
        cal_manifest["calibration_fold"] = folds + 1

        assert (
            cal_manifest.groupby("group_id")["calibration_fold"]
            .nunique().max() == 1
        )

        calibration_rows = []

        for fold in range(K):
            fit_mask = folds != fold
            cal_mask = folds == fold

            fit_counts = (
                training.loc[fit_mask, "label"]
                .value_counts()
                .reindex(CLASSES, fill_value=0)
            )
            cal_counts = (
                training.loc[cal_mask, "label"]
                .value_counts()
                .reindex(CLASSES, fill_value=0)
            )

            assert (fit_counts >= 2).all()
            assert (cal_counts >= 1).all()
            assert int(cal_mask.sum()) == int(fold_sizes[fold])

            fit_groups = set(training.loc[fit_mask, "group_id"])
            cal_groups = set(training.loc[cal_mask, "group_id"])
            assert not fit_groups & cal_groups

            for label in CLASSES:
                calibration_rows.append({
                    "fold": fold + 1,
                    "label": label,
                    "classifier_fit_count": int(fit_counts[label]),
                    "calibration_count": int(cal_counts[label]),
                })

        distribution = (
            pd.crosstab(repeat["label"], repeat["split"])
            .reindex(
                index=CLASSES,
                columns=PARTITIONS,
                fill_value=0,
            )
        )

        manifest = repeat[[
            "record_id", "group_id", "label",
            "dialogue_hash", "split",
        ]]

        manifest_bytes = manifest.to_csv(index=False).encode("utf-8")
        cal_bytes = cal_manifest.to_csv(index=False).encode("utf-8")

        metadata = {
            "seed": seed,
            "scipy_version": scipy.__version__,
            "method": "constrained group allocation",
            "split_sizes": dict(
                zip(PARTITIONS, split_sizes.tolist())
            ),
            "calibration_fold_sizes": fold_sizes.tolist(),
            "split_manifest_sha256": hashlib.sha256(
                manifest_bytes
            ).hexdigest(),
            "calibration_manifest_sha256": hashlib.sha256(
                cal_bytes
            ).hexdigest(),
            "split_solver": split_info,
            "calibration_solver": cal_info,
            "models_trained": False,
        }

        files_to_save = {
            "split_manifest.csv": manifest_bytes,
            "calibration_manifest.csv": cal_bytes,
            "class_distribution.csv": (
                distribution.to_csv().encode("utf-8")
            ),
            "calibration_class_counts.csv": (
                pd.DataFrame(calibration_rows)
                .to_csv(index=False).encode("utf-8")
            ),
            "allocation_metadata.json": (
                json.dumps(
                    metadata, indent=2, sort_keys=True
                ).encode("utf-8")
            ),
        }

        for filename, content in files_to_save.items():
            path = destination / filename
            if path.exists() and path.read_bytes() != content:
                raise RuntimeError(
                    f"Different file already exists; "
                    f"not overwritten: {path}"
                )

        for filename, content in files_to_save.items():
            write_preserving(destination / filename, content)

        status_rows.append({
            "seed": seed,
            "allocation_passed": True,
            "train": len(training),
            "validation": int(
                repeat["split"].eq("validation").sum()
            ),
            "test": int(
                repeat["split"].eq("test").sum()
            ),
            "message": "Split and calibration constraints verified",
        })

        print("PASS: split and calibration allocations verified.")
        print(distribution.to_string())

    except (RuntimeError, ValueError, AssertionError) as exc:
        status_rows.append({
            "seed": seed,
            "allocation_passed": False,
            "train": None,
            "validation": None,
            "test": None,
            "message": f"{type(exc).__name__}: {exc}",
        })

        print("FAILED:", str(exc))
        print("No replacement seed was selected.")

status = pd.DataFrame(status_rows)

print("\n=== ALLOCATION STATUS ===")
print(status.to_string(index=False))

status_content = status.to_csv(index=False).encode("utf-8")
status_path = OUT / "allocation_status.csv"

if status_path.exists() and status_path.read_bytes() != status_content:
    print(
        "\nExisting allocation_status.csv differs and was preserved. "
        "Use the printed status above to inspect this run."
    )
else:
    write_preserving(status_path, status_content)
    print("\nStatus file:", status_path)

print(
    "\nNo models trained, policies selected, "
    "or test predictions generated."
)

In [ ]:
from pathlib import Path
import json

NOTEBOOK = Path(
    _crus_local_path('notebooks', 'revised_risk_aware_hybrid_pipeline_v2.ipynb')
)

if not NOTEBOOK.is_file():
    raise FileNotFoundError(NOTEBOOK)

notebook = json.loads(
    NOTEBOOK.read_text(encoding="utf-8")
)

selected_cells = {78, 79, 80, 81, 82, 94}

for number, cell in enumerate(
    notebook.get("cells", []), start=1
):
    if number not in selected_cells:
        continue
    if cell.get("cell_type") != "code":
        continue

    source = cell.get("source", [])
    source = (
        "".join(source)
        if isinstance(source, list)
        else source
    )

    print(f"\n{'=' * 70}")
    print(f"CELL {number}")
    print("=" * 70)
    print(source)

In [ ]:
%pip install -q sentence-transformers rank-bm25

In [ ]:
from pathlib import Path
import hashlib
import json
import random
import numpy as np
import pandas as pd

ROOT = Path(_crus_local_path('project', ''))

if not ROOT.exists():
    pass  # Colab service not required locally.
    _crus_prepare_saved_workspace()

NOTEBOOK = Path(
    _crus_local_path('notebooks', 'revised_risk_aware_hybrid_pipeline_v2.ipynb')
)

SPLIT_SEED = 11
MODEL_SEED = 42

BASE = ROOT / "supplementary_repeated_splits" / f"seed_{SPLIT_SEED}"
OUT = BASE / "experiment_outputs"

if OUT.exists() and any(OUT.iterdir()):
    raise RuntimeError(
        f"Output directory is not empty: {OUT}\n"
        "Do not overwrite a completed or partial run. "
        "Inspect it before choosing a separate rerun directory."
    )

required_paths = {
    "notebook": NOTEBOOK,
    "pool": ROOT / "data_audit/constrained_dialogue_pool_with_splits.csv",
    "split": BASE / "split_manifest.csv",
    "calibration": BASE / "calibration_manifest.csv",
    "metadata": BASE / "allocation_metadata.json",
}

for name, path in required_paths.items():
    if not path.is_file():
        raise FileNotFoundError(f"{name}: {path}")

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

metadata = json.loads(required_paths["metadata"].read_text())

assert metadata["split_manifest_sha256"] == sha256(required_paths["split"])
assert metadata["calibration_manifest_sha256"] == sha256(
    required_paths["calibration"]
)

pool = pd.read_csv(
    required_paths["pool"],
    dtype={"record_id": str, "group_id": str},
)
manifest = pd.read_csv(
    required_paths["split"],
    dtype={"record_id": str, "group_id": str},
)
cal = pd.read_csv(
    required_paths["calibration"],
    dtype={"record_id": str, "group_id": str},
)

assert pool["record_id"].is_unique
assert manifest["record_id"].is_unique
assert cal["record_id"].is_unique

joined = manifest.merge(
    pool[["record_id", "text"]],
    on="record_id",
    how="left",
    validate="one_to_one",
)

assert joined["text"].notna().all()
assert joined.groupby("group_id")["split"].nunique().max() == 1

train = joined.loc[joined["split"].eq("train")].reset_index(drop=True)
val = joined.loc[joined["split"].eq("validation")].reset_index(drop=True)
test = joined.loc[joined["split"].eq("test")].reset_index(drop=True)

assert (len(train), len(val), len(test)) == (773, 258, 257)
assert set(cal["record_id"]) == set(train["record_id"])

cal = cal.set_index("record_id").loc[train["record_id"]]
assert np.array_equal(
    cal["group_id"].to_numpy(), train["group_id"].to_numpy()
)
assert np.array_equal(
    cal["label"].to_numpy(), train["label"].to_numpy()
)

classes = sorted(joined["label"].unique())
calibration_splits = []

for fold in sorted(cal["calibration_fold"].unique()):
    held = np.flatnonzero(cal["calibration_fold"].to_numpy() == fold)
    fit = np.flatnonzero(cal["calibration_fold"].to_numpy() != fold)

    assert not set(train.loc[fit, "group_id"]) & set(
        train.loc[held, "group_id"]
    )
    assert set(train.loc[fit, "label"]) == set(classes)
    assert set(train.loc[held, "label"]) == set(classes)

    calibration_splits.append((fit, held))

assert len(calibration_splits) == 3

nb = json.loads(NOTEBOOK.read_text(encoding="utf-8"))

def source_of(number):
    cell = nb["cells"][number - 1]
    assert cell["cell_type"] == "code", f"Cell {number} changed."
    source = cell.get("source", [])
    return "".join(source) if isinstance(source, list) else source

expected_markers = {
    62: "lr_model.fit",
    63: "svm_model.fit",
    67: "BM25Okapi",
    68: "train_embs = embedder.encode",
    69: "cross_enc = CrossEncoder",
    70: "def ce_raw_logits",
    71: "def descending_order",
    72: "def uncertainty_components",
    73: "def prepare_candidates",
    74: "def score_documents",
    75: "def score_retrieval_methods",
    79: "def choose_validation_policy",
    94: "def summarize_test",
}

sources = {}
for number, marker in expected_markers.items():
    source = source_of(number)
    assert marker in source, (
        f"Notebook cell {number} no longer matches the expected code."
    )
    sources[number] = source

OUT.mkdir(parents=True, exist_ok=True)

random.seed(MODEL_SEED)
np.random.seed(MODEL_SEED)

ns = {
    "__name__": "__supplementary_repeat__",
    "np": np,
    "pd": pd,
    "json": json,
    "Path": Path,
    "MODEL_OUTDIR": OUT,
    "CLASSES": classes,
    "N_CLS": len(classes),
    "CIDX": {label: i for i, label in enumerate(classes)},
    "calibration_splits": calibration_splits,
    "train_ids": train["record_id"].to_numpy(dtype=str),
    "ids_val": val["record_id"].to_numpy(dtype=str),
    "ids_test": test["record_id"].to_numpy(dtype=str),
    "X_train": train["text"].to_numpy(),
    "X_val": val["text"].to_numpy(),
    "X_test": test["text"].to_numpy(),
    "y_train_np": train["label"].to_numpy(),
    "y_val_np": val["label"].to_numpy(),
    "y_test_np": test["label"].to_numpy(),
}

for number in expected_markers:
    print(f"\nExecuting verified pipeline cell {number}")
    exec(compile(sources[number], f"notebook_cell_{number}", "exec"), ns)

import joblib

joblib.dump(ns["lr_model"], OUT / "trained_tfidf_lr.joblib")
joblib.dump(ns["svm_model"], OUT / "trained_group_calibrated_svm.joblib")

def supervised_result(model, texts):
    raw = model.predict_proba(texts)
    probabilities = np.zeros((len(texts), len(classes)))

    for column, label in enumerate(model.classes_):
        probabilities[:, ns["CIDX"][label]] = raw[:, column]

    assert np.isfinite(probabilities).all()
    assert np.allclose(probabilities.sum(axis=1), 1.0)

    return {
        "pred": np.asarray(classes)[probabilities.argmax(axis=1)],
        "probs": probabilities,
        "confidence": probabilities.max(axis=1),
        "ranking_score": probabilities.max(axis=1),
        "components": None,
    }

def score_all(texts, description):
    from tqdm.auto import tqdm

    results = {
        "TFIDF_LR": supervised_result(ns["lr_model"], texts),
        "TFIDF_SVM": supervised_result(ns["svm_model"], texts),
    }

    collected = {method: [] for method in ns["RETRIEVAL_METHODS"]}

    for query in tqdm(texts, desc=description):
        output = ns["score_retrieval_methods"](query)
        for method in collected:
            collected[method].append(output[method])

    for method, rows in collected.items():
        results[method] = {
            "pred": np.asarray([row["pred"] for row in rows]),
            "probs": (
                None if rows[0]["probs"] is None
                else np.stack([row["probs"] for row in rows])
            ),
            "confidence": np.asarray([row["confidence"] for row in rows]),
            "ranking_score": np.asarray([
                row["ranking_score"] for row in rows
            ]),
            "components": (
                None if rows[0]["components"] is None
                else np.stack([row["components"] for row in rows])
            ),
        }

    return results

validation_results = score_all(
    ns["X_val"], f"Seed {SPLIT_SEED}: validation"
)

policies = {}

for method in ns["DOCUMENT_METHODS"]:
    result = validation_results[method]
    signals = ns["uncertainty_signals"](result)
    signal_names = list(signals) if method == "Cascade" else ["CRUS"]

    for signal_name in signal_names:
        policies[f"{method}:{signal_name}"] = (
            ns["choose_validation_policy"](
                result["confidence"],
                signals[signal_name],
                result["pred"] == ns["y_val_np"],
            )
        )

protocol = {
    "split_seed": SPLIT_SEED,
    "model_seed": MODEL_SEED,
    "input": "normalized dialogue only",
    "split_manifest_sha256": sha256(required_paths["split"]),
    "calibration_manifest_sha256": sha256(required_paths["calibration"]),
    "notebook_sha256": sha256(NOTEBOOK),
    "executed_cell_source_hashes": {
        str(number): hashlib.sha256(source.encode()).hexdigest()
        for number, source in sources.items()
    },
    "classes": classes,
    "split_sizes": {"train": 773, "validation": 258, "test": 257},
    "baseline_configuration": ns["BASELINE_CFG"],
    "retrieval_configuration": ns["RETRIEVAL_CFG"],
    "uncertainty_configuration": ns["UNCERTAINTY_CFG"],
    "policy_selection_configuration": ns["POLICY_CFG"],
    "frozen_policies": policies,
}

protocol_path = OUT / "pretest_protocol.json"
protocol_path.write_text(json.dumps(protocol, indent=2, sort_keys=True))
protocol_hash = sha256(protocol_path)

print("\nValidation policies saved before test scoring.")
print("Protocol hash:", protocol_hash)

test_results = score_all(
    ns["X_test"], f"Seed {SPLIT_SEED}: test"
)

assert sha256(protocol_path) == protocol_hash

def save_predictions(results, split, frame):
    audit = []

    for method, result in results.items():
        arrays = {
            "record_ids": frame["record_id"].to_numpy(dtype=str),
            "true_labels": frame["label"].to_numpy(dtype=str),
            "predictions": np.asarray(result["pred"], dtype=str),
            "classes": np.asarray(classes, dtype=str),
            "confidence": result["confidence"],
            "ranking_score": result["ranking_score"],
            "protocol_sha256": np.asarray(protocol_hash, dtype=str),
        }

        for key in ["probs", "components"]:
            if result[key] is not None:
                name = "probabilities" if key == "probs" else key
                arrays[name] = result[key]

        np.savez_compressed(OUT / f"{split}_{method}_arrays.npz", **arrays)

        for i, record_id in enumerate(frame["record_id"]):
            audit.append({
                "record_id": record_id,
                "group_id": frame.iloc[i]["group_id"],
                "method": method,
                "true_label": frame.iloc[i]["label"],
                "prediction": result["pred"][i],
                "correct": bool(
                    result["pred"][i] == frame.iloc[i]["label"]
                ),
                "confidence": result["confidence"][i],
                "ranking_score": result["ranking_score"][i],
            })

    pd.DataFrame(audit).to_csv(
        OUT / f"{split}_predictions.csv", index=False
    )

save_predictions(validation_results, "validation", val)
save_predictions(test_results, "test", test)

summary_rows = []
acceptance_rows = []
native_masks = {}

for method, result in test_results.items():
    summary_rows.append(ns["summarize_test"](
        method,
        result,
        np.ones(len(test), dtype=bool),
        "forced_before_abstention",
    ))

for method in ns["DOCUMENT_METHODS"]:
    result = test_results[method]
    policy = policies[f"{method}:CRUS"]
    uncertainty = ns["uncertainty_signals"](result)["CRUS"]

    accepted = (
        (result["confidence"] >= policy["tau_c"])
        & (uncertainty <= policy["tau_u"])
    )
    native_masks[method] = accepted

    summary_rows.append(ns["summarize_test"](
        method, result, accepted, "native_CRUS_gate"
    ))

    for i, record_id in enumerate(ns["ids_test"]):
        acceptance_rows.append({
            "method": method,
            "record_id": record_id,
            "accepted": bool(accepted[i]),
            "CRUS": float(uncertainty[i]),
            "tau_c": policy["tau_c"],
            "tau_u": policy["tau_u"],
        })

matched_k = int(native_masks["Cascade"].sum())

if matched_k > 0:
    for method, result in test_results.items():
        order = ns["descending_order"](
            result["ranking_score"], ns["ids_test"]
        )
        accepted = np.zeros(len(test), dtype=bool)
        accepted[order[:matched_k]] = True

        row = ns["summarize_test"](
            method, result, accepted, "ranked_matched_count"
        )
        row["target_n_accepted"] = matched_k
        summary_rows.append(row)

summary = pd.DataFrame(summary_rows)
summary.insert(0, "split_seed", SPLIT_SEED)

summary.to_csv(OUT / "test_metric_summary.csv", index=False)
pd.DataFrame(acceptance_rows).to_csv(
    OUT / "native_CRUS_acceptance.csv", index=False
)

(OUT / "RUN_COMPLETE.json").write_text(json.dumps({
    "split_seed": SPLIT_SEED,
    "protocol_sha256": protocol_hash,
    "methods": list(test_results),
    "test_records": len(test),
    "matched_retained_count": matched_k,
}, indent=2))

print("\n=== SUPPLEMENTARY REPEAT COMPLETE ===")
print(summary[[
    "split_seed", "method", "mode",
    "n_accepted", "coverage", "selective_accuracy", "macro_f1",
]].to_string(index=False))
print("\nOutputs:", OUT)

In [ ]:
import re
from pathlib import Path

history = get_ipython().history_manager.input_hist_raw

runner_markers = [
    "def score_all(",
    "def supervised_result(",
    "pretest_protocol.json",
    "SUPPLEMENTARY REPEAT COMPLETE",
]

candidates = [
    source
    for source in history
    if all(marker in source for marker in runner_markers)
    and re.search(
        r"(?m)^SPLIT_SEED\s*=\s*11\s*$",
        source,
    )
]

if not candidates:
    raise RuntimeError(
        "The seed-11 runner was not found in this session's history. "
        "Paste the original complete runner into a cell, change "
        "SPLIT_SEED = 11 to SPLIT_SEED = 22, and run that cell instead."
    )

seed22_source, replacements = re.subn(
    r"(?m)^SPLIT_SEED\s*=\s*11\s*$",
    "SPLIT_SEED = 22",
    candidates[-1],
)

assert replacements == 1
assert re.search(
    r"(?m)^MODEL_SEED\s*=\s*42\s*$",
    seed22_source,
)

seed22_base = Path(
    _crus_local_path('project', 'supplementary_repeated_splits/seed_22')
)

for filename in [
    "split_manifest.csv",
    "calibration_manifest.csv",
    "allocation_metadata.json",
]:
    path = seed22_base / filename
    if not path.is_file():
        raise FileNotFoundError(path)

seed22_output = seed22_base / "experiment_outputs"

if seed22_output.exists() and any(seed22_output.iterdir()):
    raise RuntimeError(
        f"Seed 22 already has output files: {seed22_output}\n"
        "Nothing was overwritten. Inspect those files before proceeding."
    )

compile(seed22_source, "<supplementary_seed_22>", "exec")

print("Seed-22 runner prepared.")
print("Only change: SPLIT_SEED = 11 → SPLIT_SEED = 22")
print("MODEL_SEED remains 42.")
print("Seed-11 outputs will not be modified.")

In [ ]:
if "seed22_source" not in globals():
    raise RuntimeError("Run the preparation cell first.")

exec(
    compile(
        seed22_source,
        "<supplementary_seed_22>",
        "exec",
    ),
    globals(),
)

In [ ]:
import re
from pathlib import Path

if "seed22_source" not in globals():
    raise RuntimeError(
        "seed22_source is not available. "
        "Run the previous seed-22 preparation cell first, "
        "but do not rerun seed 22."
    )

seed33_source, replacements = re.subn(
    r"(?m)^SPLIT_SEED\s*=\s*22\s*$",
    "SPLIT_SEED = 33",
    seed22_source,
)

assert replacements == 1, (
    "Expected exactly one SPLIT_SEED = 22 assignment."
)
assert re.search(
    r"(?m)^MODEL_SEED\s*=\s*42\s*$",
    seed33_source,
), "MODEL_SEED must remain 42."

seed33_base = Path(
    _crus_local_path('project', 'supplementary_repeated_splits/seed_33')
)

for filename in [
    "split_manifest.csv",
    "calibration_manifest.csv",
    "allocation_metadata.json",
]:
    path = seed33_base / filename
    if not path.is_file():
        raise FileNotFoundError(path)

seed33_output = seed33_base / "experiment_outputs"

if seed33_output.exists() and any(seed33_output.iterdir()):
    raise RuntimeError(
        f"Seed 33 already has output files: {seed33_output}\n"
        "Nothing was overwritten. Inspect those files before proceeding."
    )

seed33_code = compile(
    seed33_source,
    "<supplementary_seed_33>",
    "exec",
)

print("Starting seed 33.")
print("MODEL_SEED remains 42.")
print("Seeds 11 and 22 will not be modified.")

exec(seed33_code, globals())

In [ ]:
import re
from pathlib import Path

if "seed33_source" not in globals():
    raise RuntimeError(
        "seed33_source is not available. "
        "Recover the previous runner before proceeding; "
        "do not rerun completed seeds."
    )

seed44_source, replacements = re.subn(
    r"(?m)^SPLIT_SEED\s*=\s*33\s*$",
    "SPLIT_SEED = 44",
    seed33_source,
)

assert replacements == 1, (
    "Expected exactly one SPLIT_SEED = 33 assignment."
)

assert re.search(
    r"(?m)^MODEL_SEED\s*=\s*42\s*$",
    seed44_source,
), "MODEL_SEED must remain 42."

seed44_base = Path(
    _crus_local_path('project', 'supplementary_repeated_splits/seed_44')
)

for filename in [
    "split_manifest.csv",
    "calibration_manifest.csv",
    "allocation_metadata.json",
]:
    path = seed44_base / filename
    if not path.is_file():
        raise FileNotFoundError(path)

seed44_output = seed44_base / "experiment_outputs"

if seed44_output.exists() and any(seed44_output.iterdir()):
    raise RuntimeError(
        f"Seed 44 already has output files: {seed44_output}\n"
        "Nothing was overwritten. Inspect those files before proceeding."
    )

seed44_code = compile(
    seed44_source,
    "<supplementary_seed_44>",
    "exec",
)

print("Starting seed 44.")
print("MODEL_SEED remains 42.")
print("Completed outputs for seeds 11, 22, and 33 will be preserved.")

exec(seed44_code, globals())

In [ ]:
import re
from pathlib import Path

if "seed44_source" not in globals():
    raise RuntimeError(
        "seed44_source is not available. "
        "Recover the previous runner before proceeding; "
        "do not rerun completed seeds."
    )

seed55_source, replacements = re.subn(
    r"(?m)^SPLIT_SEED\s*=\s*44\s*$",
    "SPLIT_SEED = 55",
    seed44_source,
)

assert replacements == 1, (
    "Expected exactly one SPLIT_SEED = 44 assignment."
)

assert re.search(
    r"(?m)^MODEL_SEED\s*=\s*42\s*$",
    seed55_source,
), "MODEL_SEED must remain 42."

seed55_base = Path(
    _crus_local_path('project', 'supplementary_repeated_splits/seed_55')
)

for filename in [
    "split_manifest.csv",
    "calibration_manifest.csv",
    "allocation_metadata.json",
]:
    path = seed55_base / filename
    if not path.is_file():
        raise FileNotFoundError(path)

seed55_output = seed55_base / "experiment_outputs"

if seed55_output.exists() and any(seed55_output.iterdir()):
    raise RuntimeError(
        f"Seed 55 already has output files: {seed55_output}\n"
        "Nothing was overwritten. Inspect those files before proceeding."
    )

seed55_code = compile(
    seed55_source,
    "<supplementary_seed_55>",
    "exec",
)

print("Starting seed 55.")
print("MODEL_SEED remains 42.")
print("Completed outputs for seeds 11, 22, 33, and 44 will be preserved.")

exec(seed55_code, globals())

In [ ]:
from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd

ROOT = Path(
    _crus_local_path('project', 'supplementary_repeated_splits')
)

SEEDS = [11, 22, 33, 44, 55]
SUMMARY_OUT = ROOT / "aggregate_summary"

expected_methods = {
    "TFIDF_LR",
    "TFIDF_SVM",
    "BM25",
    "SBERT",
    "SK4",
    "SingleStageFull",
    "SingleStageTruncated",
    "Cascade",
    "LengthNormLegacy",
}

expected_native_methods = {
    "SingleStageFull",
    "SingleStageTruncated",
    "Cascade",
    "LengthNormLegacy",
}

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def save_preserving(path, content):
    if path.exists():
        if path.read_bytes() != content:
            raise RuntimeError(
                f"Different summary already exists; not overwritten: {path}"
            )
    else:
        path.write_bytes(content)

tables = []
audit_rows = []

for seed in SEEDS:
    base = ROOT / f"seed_{seed}"
    out = base / "experiment_outputs"

    paths = {
        "completion": out / "RUN_COMPLETE.json",
        "protocol": out / "pretest_protocol.json",
        "metrics": out / "test_metric_summary.csv",
        "split": base / "split_manifest.csv",
        "calibration": base / "calibration_manifest.csv",
    }

    for name, path in paths.items():
        if not path.is_file():
            raise FileNotFoundError(f"Seed {seed}, {name}: {path}")

    completion = json.loads(paths["completion"].read_text())
    protocol = json.loads(paths["protocol"].read_text())
    metrics = pd.read_csv(paths["metrics"])

    assert completion["split_seed"] == seed
    assert protocol["split_seed"] == seed
    assert protocol["model_seed"] == 42
    assert completion["test_records"] == 257

    assert completion["protocol_sha256"] == sha256(paths["protocol"])
    assert protocol["split_manifest_sha256"] == sha256(paths["split"])
    assert protocol["calibration_manifest_sha256"] == sha256(
        paths["calibration"]
    )

    assert metrics["split_seed"].eq(seed).all()
    assert metrics["N"].eq(257).all()
    assert not metrics.duplicated(["method", "mode"]).any()

    assert set(metrics["mode"]) == {
        "forced_before_abstention",
        "native_CRUS_gate",
        "ranked_matched_count",
    }

    for mode in [
        "forced_before_abstention",
        "ranked_matched_count",
    ]:
        actual = set(metrics.loc[metrics["mode"].eq(mode), "method"])
        assert actual == expected_methods, (seed, mode, actual)

    native = metrics.loc[metrics["mode"].eq("native_CRUS_gate")]
    assert set(native["method"]) == expected_native_methods

    assert (
        metrics["n_correct_accepted"] + metrics["n_errors_accepted"]
        == metrics["n_accepted"]
    ).all()

    assert (
        metrics["n_accepted"] + metrics["n_abstained"]
        == metrics["N"]
    ).all()

    assert np.allclose(
        metrics["coverage"],
        metrics["n_accepted"] / metrics["N"],
    )

    assert np.allclose(
        metrics["selective_accuracy"],
        metrics["n_correct_accepted"] / metrics["n_accepted"],
        equal_nan=True,
    )

    forced = metrics.loc[
        metrics["mode"].eq("forced_before_abstention")
    ]
    assert forced["n_accepted"].eq(257).all()

    cascade_native = native.loc[native["method"].eq("Cascade")].iloc[0]
    matched = metrics.loc[metrics["mode"].eq("ranked_matched_count")]

    assert matched["n_accepted"].eq(
        int(cascade_native["n_accepted"])
    ).all()

    assert completion["matched_retained_count"] == int(
        cascade_native["n_accepted"]
    )

    tables.append(metrics)

    audit_rows.append({
        "split_seed": seed,
        "completed": True,
        "protocol_hash_verified": True,
        "manifest_hashes_verified": True,
        "metric_rows": len(metrics),
        "test_records": 257,
        "cascade_native_n_accepted": int(
            cascade_native["n_accepted"]
        ),
        "cascade_native_coverage": float(
            cascade_native["coverage"]
        ),
    })

combined = pd.concat(tables, ignore_index=True)

assert combined.groupby(["method", "mode"]).size().eq(5).all()

metric_columns = [
    "coverage",
    "selective_accuracy",
    "macro_f1",
    "weighted_f1",
    "kappa",
    "abstention_penalized_accuracy",
    "ece",
    "brier",
]

long = combined.melt(
    id_vars=["split_seed", "method", "mode"],
    value_vars=metric_columns,
    var_name="metric",
    value_name="value",
)

summary = (
    long.groupby(["mode", "method", "metric"])["value"]
    .agg(
        n_nonmissing="count",
        mean="mean",
        sample_sd="std",
        minimum="min",
        maximum="max",
    )
    .reset_index()
)

paired_rows = []

for seed in SEEDS:
    current = combined.loc[combined["split_seed"].eq(seed)]

    comparisons = [
        (
            "full_coverage",
            "forced_before_abstention",
            "forced_before_abstention",
        ),
        (
            "ranked_matched_count",
            "ranked_matched_count",
            "ranked_matched_count",
        ),
        (
            "native_Cascade_vs_ranked_SVM",
            "native_CRUS_gate",
            "ranked_matched_count",
        ),
    ]

    for comparison, cascade_mode, svm_mode in comparisons:
        cascade = current.loc[
            current["method"].eq("Cascade")
            & current["mode"].eq(cascade_mode)
        ].iloc[0]

        svm = current.loc[
            current["method"].eq("TFIDF_SVM")
            & current["mode"].eq(svm_mode)
        ].iloc[0]

        assert cascade["n_accepted"] == svm["n_accepted"]

        for metric in ["selective_accuracy", "macro_f1"]:
            paired_rows.append({
                "split_seed": seed,
                "comparison": comparison,
                "metric": metric,
                "cascade": cascade[metric],
                "tfidf_svm": svm[metric],
                "difference_cascade_minus_svm": (
                    cascade[metric] - svm[metric]
                ),
                "coverage": cascade["coverage"],
            })

paired = pd.DataFrame(paired_rows)

paired_summary = (
    paired.groupby(["comparison", "metric"])[
        "difference_cascade_minus_svm"
    ]
    .agg(
        n_repeats="count",
        mean_difference="mean",
        sample_sd="std",
        minimum="min",
        maximum="max",
    )
    .reset_index()
)

SUMMARY_OUT.mkdir(parents=True, exist_ok=True)

outputs = {
    "completion_audit.csv": pd.DataFrame(audit_rows),
    "all_repeat_metrics.csv": combined,
    "descriptive_summary.csv": summary,
    "cascade_vs_svm_per_repeat.csv": paired,
    "cascade_vs_svm_difference_summary.csv": paired_summary,
}

for filename, table in outputs.items():
    save_preserving(
        SUMMARY_OUT / filename,
        table.to_csv(index=False).encode("utf-8"),
    )

def display_mode(mode):
    rows = []

    for method in sorted(
        combined.loc[combined["mode"].eq(mode), "method"].unique()
    ):
        values = combined.loc[
            combined["mode"].eq(mode)
            & combined["method"].eq(method)
        ]

        rows.append({
            "method": method,
            "coverage_%_mean": 100 * values["coverage"].mean(),
            "coverage_%_SD": 100 * values["coverage"].std(ddof=1),
            "accuracy_%_mean": (
                100 * values["selective_accuracy"].mean()
            ),
            "accuracy_%_SD": (
                100 * values["selective_accuracy"].std(ddof=1)
            ),
            "macro_f1_mean": values["macro_f1"].mean(),
            "macro_f1_SD": values["macro_f1"].std(ddof=1),
        })

    print(f"\n=== {mode.upper()} ===")
    print(
        pd.DataFrame(rows).to_string(
            index=False,
            float_format=lambda x: f"{x:.4f}",
        )
    )

print("\n=== COMPLETION AUDIT ===")
print(pd.DataFrame(audit_rows).to_string(index=False))

for mode in [
    "forced_before_abstention",
    "native_CRUS_gate",
    "ranked_matched_count",
]:
    display_mode(mode)

print("\n=== CASCADE MINUS TFIDF_SVM: DESCRIPTIVE DIFFERENCES ===")
print(
    paired_summary.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}",
    )
)

print("\nSummary files:", SUMMARY_OUT)
print("No models refitted and no policies changed.")
print("SD describes variability across the five split repeats.")
print("No confidence intervals or significance tests computed.")

In [ ]:
from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd

ROOT = Path(
    _crus_local_path('project', 'supplementary_repeated_splits')
)

SEEDS = [11, 22, 33, 44, 55]
B = 5000
BOOTSTRAP_SEED = 20261015

OUT = ROOT / "r1_4_conditional_bootstrap"

if OUT.exists() and any(OUT.iterdir()):
    raise RuntimeError(
        f"Output directory is not empty: {OUT}\n"
        "Existing statistical results were preserved."
    )

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

records = []
class_reference = None
group_labels = {}
input_hashes = {}

for seed in SEEDS:
    base = ROOT / f"seed_{seed}"
    folder = base / "experiment_outputs"

    protocol_path = folder / "pretest_protocol.json"
    complete_path = folder / "RUN_COMPLETE.json"
    metrics_path = folder / "test_metric_summary.csv"
    manifest_path = base / "split_manifest.csv"

    for path in [
        protocol_path, complete_path, metrics_path, manifest_path
    ]:
        if not path.is_file():
            raise FileNotFoundError(path)
        input_hashes[str(path.relative_to(ROOT))] = sha256(path)

    protocol = json.loads(protocol_path.read_text())
    complete = json.loads(complete_path.read_text())

    assert complete["protocol_sha256"] == sha256(protocol_path)
    assert protocol["split_manifest_sha256"] == sha256(manifest_path)

    classes = protocol["classes"]
    if class_reference is None:
        class_reference = classes
    assert classes == class_reference

    cidx = {label: i for i, label in enumerate(classes)}
    C = len(classes)

    manifest = pd.read_csv(
        manifest_path,
        dtype={"record_id": str, "group_id": str},
    )
    test_manifest = manifest.loc[
        manifest["split"].eq("test")
    ].set_index("record_id")

    metrics = pd.read_csv(metrics_path)
    assert metrics["split_seed"].eq(seed).all()
    assert not metrics.duplicated(["method", "mode"]).any()

    for _, row in metrics.iterrows():
        method = row["method"]
        mode = row["mode"]

        array_path = folder / f"test_{method}_arrays.npz"
        input_hashes[str(array_path.relative_to(ROOT))] = sha256(array_path)

        with np.load(array_path, allow_pickle=False) as saved:
            ids = saved["record_ids"].astype(str)
            true = saved["true_labels"].astype(str)
            pred = saved["predictions"].astype(str)
            ranking = saved["ranking_score"].copy()
            confidence = saved["confidence"].copy()
            components = (
                saved["components"].copy()
                if "components" in saved.files else None
            )

            assert saved["classes"].tolist() == classes
            assert saved["protocol_sha256"].item() == sha256(protocol_path)

        assert len(set(ids)) == len(ids) == 257
        aligned = test_manifest.loc[ids]
        assert np.array_equal(aligned["label"].to_numpy(), true)

        groups = aligned["group_id"].to_numpy(dtype=str)

        for rid, gid, label in zip(ids, groups, true):
            identity = (gid, label)
            if rid in group_labels:
                assert group_labels[rid] == identity
            group_labels[rid] = identity

        if mode == "forced_before_abstention":
            accepted = np.ones(len(ids), dtype=bool)

        elif mode == "native_CRUS_gate":
            policy = protocol["frozen_policies"][f"{method}:CRUS"]
            weights = np.asarray(
                protocol["uncertainty_configuration"]["weights"],
                dtype=float,
            )
            uncertainty = components @ weights
            accepted = (
                (confidence >= policy["tau_c"])
                & (uncertainty <= policy["tau_u"])
            )

        elif mode == "ranked_matched_count":
            k = int(row["n_accepted"])

            # Descending score; ascending record ID breaks ties.
            order = np.lexsort((ids, -ranking))
            accepted = np.zeros(len(ids), dtype=bool)
            accepted[order[:k]] = True

        else:
            raise ValueError(mode)

        assert accepted.sum() == int(row["n_accepted"])

        records.append({
            "seed": seed,
            "method": method,
            "mode": mode,
            "groups": groups,
            "true": np.asarray([cidx[x] for x in true]),
            "pred": np.asarray([cidx[x] for x in pred]),
            "accepted": accepted,
            "expected_accuracy": float(row["selective_accuracy"]),
            "expected_macro_f1": float(row["macro_f1"]),
        })

assert len(records) == 110

all_groups = sorted({
    group
    for record in records
    for group in record["groups"]
})
gidx = {group: i for i, group in enumerate(all_groups)}
G = len(all_groups)

for record in records:
    record["group_indices"] = np.asarray([
        gidx[group] for group in record["groups"]
    ])

def evaluate(record, multiplicities):
    weights = multiplicities[record["group_indices"]]
    total = weights.sum()

    accepted_weights = weights * record["accepted"]
    retained = accepted_weights.sum()

    if total == 0 or retained == 0:
        return np.array([np.nan, np.nan, np.nan])

    cm = np.bincount(
        C * record["true"] + record["pred"],
        weights=accepted_weights,
        minlength=C * C,
    ).reshape(C, C)

    tp = np.diag(cm)
    denominator = cm.sum(axis=0) + cm.sum(axis=1)

    per_class_f1 = np.divide(
        2 * tp,
        denominator,
        out=np.zeros(C, dtype=float),
        where=denominator > 0,
    )

    return np.array([
        retained / total,
        tp.sum() / retained,
        per_class_f1.mean(),
    ])

metric_names = ["coverage", "selective_accuracy", "macro_f1"]

point = np.stack([
    evaluate(record, np.ones(G, dtype=int))
    for record in records
])

for i, record in enumerate(records):
    assert np.isclose(
        point[i, 1], record["expected_accuracy"], atol=1e-8
    )
    assert np.isclose(
        point[i, 2], record["expected_macro_f1"], atol=1e-8
    ), (
        "Saved summary and reconstructed predictions disagree. "
        "Check ranking tie-breaking before continuing."
    )

rng = np.random.default_rng(BOOTSTRAP_SEED)
bootstrap = np.empty((B, len(records), len(metric_names)))

for b in range(B):
    sampled_groups = rng.integers(0, G, size=G)
    multiplicities = np.bincount(sampled_groups, minlength=G)

    for i, record in enumerate(records):
        bootstrap[b, i] = evaluate(record, multiplicities)

    if (b + 1) % 500 == 0:
        print(f"Bootstrap replicates: {b + 1}/{B}")

def interval(values):
    valid = values[np.isfinite(values)]
    if len(valid) < 0.99 * B:
        raise RuntimeError(
            "Too many undefined bootstrap replicates. "
            "Inspect accepted-subset support."
        )
    lo, hi = np.quantile(valid, [0.025, 0.975])
    return float(lo), float(hi), len(valid)

per_repeat_rows = []

for i, record in enumerate(records):
    for j, metric in enumerate(metric_names):
        lo, hi, n_valid = interval(bootstrap[:, i, j])
        per_repeat_rows.append({
            "split_seed": record["seed"],
            "method": record["method"],
            "mode": record["mode"],
            "metric": metric,
            "estimate": point[i, j],
            "ci95_lower": lo,
            "ci95_upper": hi,
            "valid_bootstrap_replicates": n_valid,
        })

keys = sorted({
    (record["method"], record["mode"])
    for record in records
})

mean_rows = []

for method, mode in keys:
    indices = [
        i for i, record in enumerate(records)
        if record["method"] == method and record["mode"] == mode
    ]
    assert len(indices) == 5

    repeat_estimates = point[indices]
    boot_means = bootstrap[:, indices, :].mean(axis=1)

    for j, metric in enumerate(metric_names):
        lo, hi, n_valid = interval(boot_means[:, j])
        mean_rows.append({
            "method": method,
            "mode": mode,
            "metric": metric,
            "n_repeats": 5,
            "mean": repeat_estimates[:, j].mean(),
            "sample_sd_across_repeats": repeat_estimates[:, j].std(ddof=1),
            "conditional_ci95_lower": lo,
            "conditional_ci95_upper": hi,
            "valid_bootstrap_replicates": n_valid,
        })

lookup = {
    (record["seed"], record["method"], record["mode"]): i
    for i, record in enumerate(records)
}

comparisons = []

for mode in ["forced_before_abstention", "ranked_matched_count"]:
    for comparator in [
        "TFIDF_SVM", "SingleStageFull", "SingleStageTruncated"
    ]:
        comparisons.append((
            f"Cascade_vs_{comparator}",
            "Cascade", mode,
            comparator, mode,
        ))

comparisons.append((
    "Native_Cascade_vs_ranked_TFIDF_SVM",
    "Cascade", "native_CRUS_gate",
    "TFIDF_SVM", "ranked_matched_count",
))

paired_rows = []

for name, method_a, mode_a, method_b, mode_b in comparisons:
    a = [lookup[(seed, method_a, mode_a)] for seed in SEEDS]
    b = [lookup[(seed, method_b, mode_b)] for seed in SEEDS]

    observed = (point[a] - point[b]).mean(axis=0)
    differences = (
        bootstrap[:, a, :] - bootstrap[:, b, :]
    ).mean(axis=1)

    for j, metric in enumerate(metric_names):
        if metric == "coverage":
            continue

        lo, hi, n_valid = interval(differences[:, j])
        paired_rows.append({
            "comparison": name,
            "metric": metric,
            "difference_direction": "Cascade minus comparator",
            "mean_difference": observed[j],
            "conditional_ci95_lower": lo,
            "conditional_ci95_upper": hi,
            "valid_bootstrap_replicates": n_valid,
            "analysis": "exploratory; no multiplicity adjustment",
        })

means = pd.DataFrame(mean_rows)
paired = pd.DataFrame(paired_rows)
per_repeat = pd.DataFrame(per_repeat_rows)

OUT.mkdir(parents=True, exist_ok=False)

means.to_csv(OUT / "mean_metrics_conditional_ci95.csv", index=False)
paired.to_csv(OUT / "paired_differences_conditional_ci95.csv", index=False)
per_repeat.to_csv(OUT / "per_repeat_conditional_ci95.csv", index=False)

analysis_metadata = {
    "bootstrap_replicates": B,
    "bootstrap_seed": BOOTSTRAP_SEED,
    "resampling_unit": "group_id",
    "unique_observed_test_groups": G,
    "resampling": (
        "Multinomial cluster bootstrap over union of observed test groups; "
        "same group multiplicity used across repeats and methods."
    ),
    "interval": "95% percentile",
    "conditioning": (
        "Fitted models, observed split assignments, validation-selected "
        "policies, and original ranked selections held fixed."
    ),
    "limitations": [
        "Does not estimate full training-and-refitting uncertainty.",
        "Does not establish external generalization.",
        "Paired comparisons are exploratory and not multiplicity-adjusted.",
        "No significance p-values computed.",
    ],
    "input_hashes": input_hashes,
}

(OUT / "analysis_metadata.json").write_text(
    json.dumps(analysis_metadata, indent=2, sort_keys=True)
)

print("\n=== FIVE-REPEAT MEANS WITH CONDITIONAL 95% CIs ===")
print(means.to_string(index=False, float_format=lambda x: f"{x:.6f}"))

print("\n=== PAIRED BOOTSTRAP DIFFERENCES ===")
print(paired.to_string(index=False, float_format=lambda x: f"{x:.6f}"))

print("\nSaved statistical summaries:", OUT)
print("No model refitting or policy changes performed.")

In [ ]:
from pathlib import Path
pass  # Colab service not required locally.
import pandas as pd

_crus_prepare_saved_workspace()

MY_DRIVE = Path(_crus_local_path('drive', ''))
ROOT = (
    MY_DRIVE
    / "RiskAware_Revision2"
    / "supplementary_repeated_splits"
)

print("MyDrive accessible:", MY_DRIVE.is_dir())
print("Expected experiment root exists:", ROOT.is_dir())
print("Expected root:", ROOT)

checks = []

for seed in [11, 22, 33, 44, 55]:
    base = ROOT / f"seed_{seed}"
    out = base / "experiment_outputs"

    checks.append({
        "seed": seed,
        "split_manifest": (base / "split_manifest.csv").is_file(),
        "calibration_manifest": (
            base / "calibration_manifest.csv"
        ).is_file(),
        "protocol": (out / "pretest_protocol.json").is_file(),
        "completion_marker": (out / "RUN_COMPLETE.json").is_file(),
        "metrics": (out / "test_metric_summary.csv").is_file(),
        "test_array_files": (
            len(list(out.glob("test_*_arrays.npz")))
            if out.is_dir() else 0
        ),
    })

check_table = pd.DataFrame(checks)

print("\n=== SAVED OUTPUT CHECK ===")
print(check_table.to_string(index=False))

seed11_out = ROOT / "seed_11" / "experiment_outputs"

if seed11_out.is_dir():
    print("\n=== SEED 11 OUTPUT DIRECTORY CONTENTS ===")
    for path in sorted(seed11_out.iterdir()):
        print(path.name)

required_checks = [
    "split_manifest",
    "calibration_manifest",
    "protocol",
    "completion_marker",
    "metrics",
]

ready = (
    check_table[required_checks].all().all()
    and check_table["test_array_files"].eq(9).all()
)

if ready:
    print(
        "\nREADY: Required files are accessible for all five seeds.\n"
        "Rerun the statistical-analysis cell."
    )
else:
    print(
        "\nNOT READY: Some required outputs are not accessible.\n"
        "Do not rerun training or manufacture missing protocol files.\n"
        "Paste this diagnostic output so the file location can be checked."
    )

In [ ]:
from pathlib import Path
pass  # Colab service not required locally.
import hashlib
import json
import numpy as np
import pandas as pd

if not Path(_crus_local_path('drive', '')).is_dir():
    _crus_prepare_saved_workspace()

ROOT = Path(
    _crus_local_path('project', 'supplementary_repeated_splits')
)

SEEDS = [11, 22, 33, 44, 55]
B = 5000
BOOTSTRAP_SEED = 20261015
OUT = ROOT / "r1_4_conditional_bootstrap"

if OUT.exists() and any(OUT.iterdir()):
    raise RuntimeError(
        f"Existing statistical outputs were preserved: {OUT}\n"
        "Inspect them before starting another analysis."
    )

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

records = []
input_hashes = {}
identities = {}
class_reference = None

for seed in SEEDS:
    base = ROOT / f"seed_{seed}"
    folder = base / "experiment_outputs"

    protocol_path = folder / "pretest_protocol.json"
    completion_path = folder / "RUN_COMPLETE.json"
    metrics_path = folder / "test_metric_summary.csv"
    manifest_path = base / "split_manifest.csv"
    calibration_path = base / "calibration_manifest.csv"

    for path in [
        protocol_path, completion_path, metrics_path,
        manifest_path, calibration_path,
    ]:
        if not path.is_file():
            raise FileNotFoundError(path)
        input_hashes[str(path.relative_to(ROOT))] = sha256(path)

    protocol = json.loads(protocol_path.read_text())
    completion = json.loads(completion_path.read_text())
    protocol_hash = sha256(protocol_path)

    assert protocol["split_seed"] == seed
    assert completion["split_seed"] == seed
    assert completion["protocol_sha256"] == protocol_hash
    assert protocol["split_manifest_sha256"] == sha256(manifest_path)
    assert protocol["calibration_manifest_sha256"] == sha256(
        calibration_path
    )

    classes = protocol["classes"]
    if class_reference is None:
        class_reference = classes
    assert classes == class_reference

    C = len(classes)
    cidx = {label: i for i, label in enumerate(classes)}

    manifest = pd.read_csv(
        manifest_path,
        dtype={"record_id": str, "group_id": str},
    )
    test_manifest = manifest.loc[
        manifest["split"].eq("test")
    ].set_index("record_id")

    metrics = pd.read_csv(metrics_path)
    assert metrics["split_seed"].eq(seed).all()
    assert len(metrics) == 22
    assert not metrics.duplicated(["method", "mode"]).any()

    cached = {}

    for method in metrics["method"].unique():
        path = folder / f"test_{method}_arrays.npz"
        if not path.is_file():
            raise FileNotFoundError(path)
        input_hashes[str(path.relative_to(ROOT))] = sha256(path)

        with np.load(path, allow_pickle=False) as saved:
            cached[method] = {
                key: saved[key].copy() for key in saved.files
            }

    for _, row in metrics.iterrows():
        method, mode = row["method"], row["mode"]
        saved = cached[method]

        ids = saved["record_ids"].astype(str)
        true = saved["true_labels"].astype(str)
        pred = saved["predictions"].astype(str)

        assert len(ids) == len(set(ids)) == 257
        assert set(ids) == set(test_manifest.index)
        assert saved["classes"].tolist() == classes
        assert saved["protocol_sha256"].item() == protocol_hash

        aligned = test_manifest.loc[ids]
        assert np.array_equal(aligned["label"].to_numpy(), true)
        groups = aligned["group_id"].to_numpy(dtype=str)

        for rid, group, label in zip(ids, groups, true):
            if rid in identities:
                assert identities[rid] == (group, label)
            identities[rid] = (group, label)

        if mode == "forced_before_abstention":
            accepted = np.ones(len(ids), dtype=bool)

        elif mode == "native_CRUS_gate":
            policy = protocol["frozen_policies"][f"{method}:CRUS"]
            weights = np.asarray(
                protocol["uncertainty_configuration"]["weights"],
                dtype=float,
            )
            uncertainty = saved["components"] @ weights
            accepted = (
                (saved["confidence"] >= policy["tau_c"])
                & (uncertainty <= policy["tau_u"])
            )

        elif mode == "ranked_matched_count":
            order = np.lexsort((ids, -saved["ranking_score"]))
            accepted = np.zeros(len(ids), dtype=bool)
            accepted[order[:int(row["n_accepted"])]] = True

        else:
            raise ValueError(f"Unexpected mode: {mode}")

        assert accepted.sum() == int(row["n_accepted"])

        records.append({
            "seed": seed,
            "method": method,
            "mode": mode,
            "groups": groups,
            "true": np.array([cidx[label] for label in true]),
            "pred": np.array([cidx[label] for label in pred]),
            "accepted": accepted,
            "expected_accuracy": float(row["selective_accuracy"]),
            "expected_macro_f1": float(row["macro_f1"]),
        })

    print(f"Seed {seed}: input files and protocol verified.")

assert len(records) == 110

all_groups = sorted({
    group for record in records for group in record["groups"]
})
group_index = {group: i for i, group in enumerate(all_groups)}
G = len(all_groups)

for record in records:
    record["group_indices"] = np.array([
        group_index[group] for group in record["groups"]
    ])
    record["confusion_indices"] = (
        C * record["true"] + record["pred"]
    )

METRICS = ["coverage", "selective_accuracy", "macro_f1"]

def evaluate(record, multiplicities):
    weights = multiplicities[record["group_indices"]]
    total = weights.sum()
    accepted_weights = weights * record["accepted"]
    retained = accepted_weights.sum()

    if total == 0 or retained == 0:
        return np.full(3, np.nan)

    cm = np.bincount(
        record["confusion_indices"],
        weights=accepted_weights,
        minlength=C * C,
    ).reshape(C, C)

    tp = np.diag(cm)
    denominator = cm.sum(axis=0) + cm.sum(axis=1)

    class_f1 = np.divide(
        2 * tp,
        denominator,
        out=np.zeros(C, dtype=float),
        where=denominator > 0,
    )

    return np.array([
        retained / total,
        tp.sum() / retained,
        class_f1.mean(),
    ])

point = np.stack([
    evaluate(record, np.ones(G, dtype=int))
    for record in records
])

for i, record in enumerate(records):
    assert np.isclose(
        point[i, 1], record["expected_accuracy"], atol=1e-8
    ), f"Accuracy mismatch: {record['seed']}, {record['method']}"

    assert np.isclose(
        point[i, 2], record["expected_macro_f1"], atol=1e-8
    ), (
        f"Macro-F1 mismatch: {record['seed']}, "
        f"{record['method']}, {record['mode']}. "
        "Inspect ranking tie-breaking; do not bypass this check."
    )

print("\nAll reconstructed metrics match the saved summaries.")
print(f"Observed unique test groups: {G}")
print(f"Starting {B} synchronized group-bootstrap replicates.")

rng = np.random.default_rng(BOOTSTRAP_SEED)
bootstrap = np.empty((B, len(records), len(METRICS)))

for b in range(B):
    sampled = rng.integers(0, G, size=G)
    multiplicities = np.bincount(sampled, minlength=G)

    for i, record in enumerate(records):
        bootstrap[b, i] = evaluate(record, multiplicities)

    if (b + 1) % 500 == 0:
        print(f"Completed {b + 1}/{B}")

def interval(values):
    valid = values[np.isfinite(values)]
    if len(valid) < 0.99 * B:
        raise RuntimeError(
            "More than 1% of bootstrap replicates are undefined. "
            "Inspect group and accepted-subset support."
        )
    lower, upper = np.quantile(valid, [0.025, 0.975])
    return float(lower), float(upper), len(valid)

per_repeat_rows = []

for i, record in enumerate(records):
    for j, metric in enumerate(METRICS):
        lower, upper, valid = interval(bootstrap[:, i, j])
        per_repeat_rows.append({
            "split_seed": record["seed"],
            "method": record["method"],
            "mode": record["mode"],
            "metric": metric,
            "estimate": point[i, j],
            "conditional_ci95_lower": lower,
            "conditional_ci95_upper": upper,
            "valid_bootstrap_replicates": valid,
        })

mean_rows = []
keys = sorted({
    (record["method"], record["mode"]) for record in records
})

for method, mode in keys:
    indices = [
        i for i, record in enumerate(records)
        if record["method"] == method and record["mode"] == mode
    ]
    assert len(indices) == 5

    repeat_values = point[indices]
    bootstrap_means = bootstrap[:, indices, :].mean(axis=1)

    for j, metric in enumerate(METRICS):
        lower, upper, valid = interval(bootstrap_means[:, j])
        mean_rows.append({
            "method": method,
            "mode": mode,
            "metric": metric,
            "n_repeats": 5,
            "mean": repeat_values[:, j].mean(),
            "sample_sd_across_repeats": repeat_values[:, j].std(ddof=1),
            "conditional_ci95_lower": lower,
            "conditional_ci95_upper": upper,
            "valid_bootstrap_replicates": valid,
        })

lookup = {
    (record["seed"], record["method"], record["mode"]): i
    for i, record in enumerate(records)
}

comparisons = []

for mode in ["forced_before_abstention", "ranked_matched_count"]:
    for comparator in [
        "TFIDF_SVM", "SingleStageFull", "SingleStageTruncated"
    ]:
        comparisons.append((
            f"Cascade_vs_{comparator}",
            "Cascade", mode, comparator, mode,
        ))

comparisons.append((
    "Native_Cascade_vs_ranked_TFIDF_SVM",
    "Cascade", "native_CRUS_gate",
    "TFIDF_SVM", "ranked_matched_count",
))

paired_rows = []

for name, method_a, mode_a, method_b, mode_b in comparisons:
    a = [lookup[(seed, method_a, mode_a)] for seed in SEEDS]
    b = [lookup[(seed, method_b, mode_b)] for seed in SEEDS]

    observed = (point[a] - point[b]).mean(axis=0)
    boot_differences = (
        bootstrap[:, a, :] - bootstrap[:, b, :]
    ).mean(axis=1)

    for j, metric in enumerate(METRICS):
        if metric == "coverage":
            continue

        lower, upper, valid = interval(boot_differences[:, j])
        paired_rows.append({
            "comparison": name,
            "cascade_mode": mode_a,
            "comparator_mode": mode_b,
            "metric": metric,
            "difference_direction": "Cascade minus comparator",
            "mean_difference": observed[j],
            "conditional_ci95_lower": lower,
            "conditional_ci95_upper": upper,
            "valid_bootstrap_replicates": valid,
            "analysis": "exploratory; no multiplicity adjustment",
        })

means = pd.DataFrame(mean_rows)
paired = pd.DataFrame(paired_rows)
per_repeat = pd.DataFrame(per_repeat_rows)

metadata = {
    "bootstrap_replicates": B,
    "bootstrap_seed": BOOTSTRAP_SEED,
    "resampling_unit": "group_id",
    "unique_observed_test_groups": G,
    "interval_method": "95% percentile",
    "resampling_design": (
        "Resample the union of observed test groups with replacement. "
        "Use the same group multiplicities across methods and repeats."
    ),
    "conditioning": (
        "Models, split assignments, validation-selected policies, "
        "and original ranked selections are held fixed."
    ),
    "limitations": [
        "Does not include full model-refitting uncertainty.",
        "Does not establish external generalization.",
        "Paired comparisons are exploratory, without multiplicity adjustment.",
        "No significance p-values are computed.",
    ],
    "input_hashes": input_hashes,
}

OUT.mkdir(parents=True, exist_ok=True)

means.to_csv(
    OUT / "mean_metrics_conditional_ci95.csv", index=False
)
paired.to_csv(
    OUT / "paired_differences_conditional_ci95.csv", index=False
)
per_repeat.to_csv(
    OUT / "per_repeat_conditional_ci95.csv", index=False
)
(OUT / "analysis_metadata.json").write_text(
    json.dumps(metadata, indent=2, sort_keys=True)
)

print("\n=== FIVE-REPEAT MEANS WITH CONDITIONAL 95% CIs ===")
print(means.to_string(
    index=False, float_format=lambda value: f"{value:.6f}"
))

print("\n=== PAIRED BOOTSTRAP DIFFERENCES ===")
print(paired.to_string(
    index=False, float_format=lambda value: f"{value:.6f}"
))

print("\nSaved statistical summaries:", OUT)
print("No models retrained and no policies changed.")
print(
    "Intervals are conditional on the completed experiments; "
    "they do not include full model-refitting uncertainty."
)

In [ ]:
import numpy as np
import pandas as pd

required_variables = ["bootstrap", "point", "records"]
missing = [name for name in required_variables if name not in globals()]

if missing:
    raise RuntimeError(
        f"Missing variables: {missing}. "
        "The diagnostic needs the same session as the bootstrap run."
    )

diagnostic_rows = []

for i, record in enumerate(records):
    values = bootstrap[:, i, 2]
    valid = values[np.isfinite(values)]

    accepted_true = record["true"][record["accepted"]]
    supported_classes = len(np.unique(accepted_true))

    diagnostic_rows.append({
        "seed": record["seed"],
        "method": record["method"],
        "mode": record["mode"],
        "accepted_true_classes": supported_classes,
        "observed_macro_f1": point[i, 2],
        "bootstrap_mean_macro_f1": valid.mean(),
        "bootstrap_mean_minus_observed": valid.mean() - point[i, 2],
        "bootstrap_median_macro_f1": np.median(valid),
        "fraction_bootstrap_below_observed": (
            valid < point[i, 2]
        ).mean(),
    })

diagnostics = pd.DataFrame(diagnostic_rows)

summary = (
    diagnostics.groupby(["method", "mode"])
    .agg(
        mean_observed_macro_f1=("observed_macro_f1", "mean"),
        mean_bootstrap_macro_f1=("bootstrap_mean_macro_f1", "mean"),
        mean_bootstrap_displacement=(
            "bootstrap_mean_minus_observed", "mean"
        ),
        minimum_accepted_true_classes=(
            "accepted_true_classes", "min"
        ),
    )
    .reset_index()
)

print(summary.to_string(
    index=False,
    float_format=lambda value: f"{value:.6f}",
))

In [ ]:
%pip install -q transformers huggingface_hub scikit-learn pandas numpy

In [ ]:
from pathlib import Path
pass  # Colab service not required locally.
from huggingface_hub import model_info
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import (
    accuracy_score, f1_score, cohen_kappa_score
)
import hashlib
import json
import platform
import random
import numpy as np
import pandas as pd
import torch
import transformers
import sklearn

if not Path(_crus_local_path('drive', '')).is_dir():
    _crus_prepare_saved_workspace()

ROOT = Path(_crus_local_path('project', ''))

SPLIT_SEED = 11
MODEL_SEED = 42
MODEL_ID = "ncbi/MedCPT-Query-Encoder"
MAX_LENGTH = 256
BATCH_SIZE = 16

BASE = ROOT / "supplementary_repeated_splits" / f"seed_{SPLIT_SEED}"
EXISTING = BASE / "experiment_outputs"
OUT = BASE / "domain_baselines" / "medcpt_query_v1"

paths = {
    "pool": ROOT / "data_audit/constrained_dialogue_pool_with_splits.csv",
    "split": BASE / "split_manifest.csv",
    "protocol": EXISTING / "pretest_protocol.json",
    "completion": EXISTING / "RUN_COMPLETE.json",
    "metrics": EXISTING / "test_metric_summary.csv",
}

for name, path in paths.items():
    if not path.is_file():
        raise FileNotFoundError(f"{name}: {path}")

if OUT.exists() and any(OUT.iterdir()):
    raise RuntimeError(
        f"Output directory is not empty: {OUT}\n"
        "Existing results were preserved."
    )

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

source_protocol = json.loads(paths["protocol"].read_text())
completion = json.loads(paths["completion"].read_text())

assert source_protocol["split_seed"] == SPLIT_SEED
assert completion["protocol_sha256"] == sha256(paths["protocol"])
assert source_protocol["split_manifest_sha256"] == sha256(paths["split"])

classes = source_protocol["classes"]

pool = pd.read_csv(
    paths["pool"],
    dtype={"record_id": str, "group_id": str},
)
manifest = pd.read_csv(
    paths["split"],
    dtype={"record_id": str, "group_id": str},
)

assert pool["record_id"].is_unique
assert manifest["record_id"].is_unique
assert manifest["dialogue_hash"].notna().all()

joined = manifest.merge(
    pool[["record_id", "label", "group_id", "dialogue_hash", "text", "dialogue"]],
    on="record_id",
    how="left",
    suffixes=("", "_pool"),
    validate="one_to_one",
)

for column in ["label", "group_id", "dialogue_hash"]:
    assert joined[column].eq(joined[f"{column}_pool"]).all()

assert joined[["text", "dialogue"]].notna().all().all()

def normalize(text):
    return " ".join(str(text).split()).casefold()

assert joined["text"].map(normalize).eq(
    joined["dialogue"].map(normalize)
).all()

assert joined.groupby("group_id")["split"].nunique().max() == 1
assert joined.groupby("dialogue_hash")["split"].nunique().max() == 1

frames = {
    name: joined.loc[joined["split"].eq(name)]
    .sort_values("record_id").reset_index(drop=True)
    for name in ["train", "validation", "test"]
}

assert [len(frames[name]) for name in frames] == [773, 258, 257]
assert sorted(joined["label"].unique()) == classes

random.seed(MODEL_SEED)
np.random.seed(MODEL_SEED)
torch.manual_seed(MODEL_SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(MODEL_SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Resolve and record the exact downloaded model revision.
MODEL_REVISION = model_info(MODEL_ID).sha

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID, revision=MODEL_REVISION
)
model = AutoModel.from_pretrained(
    MODEL_ID, revision=MODEL_REVISION
).to(DEVICE)
model.eval()

assert MAX_LENGTH <= model.config.max_position_embeddings

OUT.mkdir(parents=True, exist_ok=True)

configuration = {
    "method": "MedCPT_Query",
    "split_seed": SPLIT_SEED,
    "model_seed": MODEL_SEED,
    "model_id": MODEL_ID,
    "model_revision": MODEL_REVISION,
    "representation": "final hidden state of CLS token",
    "encoder_roles": "Query Encoder for both query and training dialogue",
    "similarity": "unnormalized inner product",
    "max_length_tokens": MAX_LENGTH,
    "batch_size": BATCH_SIZE,
    "input": "normalized dialogue only",
    "index": "training dialogues only",
    "prediction": "label of highest-scoring training dialogue",
    "tie_breaking": "ascending record_id",
    "probability_model": None,
    "classes": classes,
    "device": DEVICE,
    "python_version": platform.python_version(),
    "torch_version": torch.__version__,
    "transformers_version": transformers.__version__,
    "sklearn_version": sklearn.__version__,
    "input_hashes": {
        name: sha256(path) for name, path in paths.items()
    },
}

config_path = OUT / "baseline_configuration.json"
config_path.write_text(json.dumps(configuration, indent=2, sort_keys=True))
config_hash = sha256(config_path)

def encode(texts):
    chunks = []

    with torch.inference_mode():
        for start in range(0, len(texts), BATCH_SIZE):
            batch = list(texts[start:start + BATCH_SIZE])
            inputs = tokenizer(
                batch,
                padding=True,
                truncation=True,
                max_length=MAX_LENGTH,
                return_tensors="pt",
            ).to(DEVICE)

            embeddings = model(**inputs).last_hidden_state[:, 0, :]
            chunks.append(embeddings.cpu().numpy())

    result = np.concatenate(chunks).astype(np.float32)
    assert np.isfinite(result).all()
    return result

diagnostic_rows = []

for split, frame in frames.items():
    lengths = np.array([
        len(tokenizer(
            text,
            truncation=False,
            add_special_tokens=True,
            verbose=False,
        )["input_ids"])
        for text in frame["text"]
    ])

    diagnostic_rows.append({
        "split": split,
        "n_records": len(frame),
        "max_length_tokens": MAX_LENGTH,
        "n_truncated": int((lengths > MAX_LENGTH).sum()),
        "fraction_truncated": float((lengths > MAX_LENGTH).mean()),
        "median_original_tokens": float(np.median(lengths)),
        "maximum_original_tokens": int(lengths.max()),
    })

print("Device:", DEVICE)
print("Model revision:", MODEL_REVISION)
print("\nEncoding training dialogues...")

train = frames["train"]
train_embeddings = encode(train["text"].to_numpy())
train_labels = train["label"].to_numpy()
train_ids = train["record_id"].to_numpy(dtype=str)

np.savez_compressed(
    OUT / "training_embeddings.npz",
    record_ids=train_ids,
    embeddings=train_embeddings,
)

results = {}
audit_rows = []

for split in ["validation", "test"]:
    frame = frames[split]
    print(f"Encoding and retrieving: {split}")

    query_embeddings = encode(frame["text"].to_numpy())
    scores = query_embeddings @ train_embeddings.T

    # Training records are already sorted by record_id.
    top_indices = np.argmax(scores, axis=1)
    predictions = train_labels[top_indices]
    top_scores = scores[np.arange(len(frame)), top_indices]

    ids = frame["record_id"].to_numpy(dtype=str)
    true = frame["label"].to_numpy(dtype=str)

    results[split] = {
        "ids": ids,
        "true": true,
        "pred": predictions,
        "ranking_score": top_scores,
    }

    np.savez_compressed(
        OUT / f"{split}_MedCPT_Query_arrays.npz",
        record_ids=ids,
        true_labels=true,
        predictions=predictions.astype(str),
        classes=np.asarray(classes, dtype=str),
        ranking_score=top_scores,
        retrieved_record_ids=train_ids[top_indices],
        configuration_sha256=np.asarray(config_hash),
    )

    for i, record_id in enumerate(ids):
        audit_rows.append({
            "split": split,
            "record_id": record_id,
            "group_id": frame.iloc[i]["group_id"],
            "true_label": true[i],
            "prediction": predictions[i],
            "correct": bool(predictions[i] == true[i]),
            "top_inner_product": float(top_scores[i]),
            "retrieved_training_record_id": train_ids[top_indices[i]],
        })

assert sha256(config_path) == config_hash

existing_metrics = pd.read_csv(paths["metrics"])
cascade = existing_metrics.loc[
    existing_metrics["method"].eq("Cascade")
    & existing_metrics["mode"].eq("native_CRUS_gate")
]

assert len(cascade) == 1
MATCHED_K = int(cascade.iloc[0]["n_accepted"])
assert MATCHED_K == completion["matched_retained_count"]

def summarize(result, accepted, mode, split):
    correct = result["pred"] == result["true"]
    n = len(correct)
    retained = int(accepted.sum())
    n_correct = int(correct[accepted].sum())

    return {
        "split_seed": SPLIT_SEED,
        "method": "MedCPT_Query",
        "split": split,
        "mode": mode,
        "N": n,
        "n_accepted": retained,
        "n_correct_accepted": n_correct,
        "n_errors_accepted": retained - n_correct,
        "n_abstained": n - retained,
        "coverage": retained / n,
        "forced_accuracy": float(correct.mean()),
        "selective_accuracy": n_correct / retained if retained else np.nan,
        "macro_f1": f1_score(
            result["true"][accepted],
            result["pred"][accepted],
            labels=classes,
            average="macro",
            zero_division=0,
        ) if retained else np.nan,
        "weighted_f1": f1_score(
            result["true"][accepted],
            result["pred"][accepted],
            labels=classes,
            average="weighted",
            zero_division=0,
        ) if retained else np.nan,
        "kappa": cohen_kappa_score(
            result["true"][accepted],
            result["pred"][accepted],
            labels=classes,
        ) if retained else np.nan,
        "ece": np.nan,
        "brier": np.nan,
        "nll": np.nan,
    }

rows = []

for split, result in results.items():
    rows.append(summarize(
        result,
        np.ones(len(result["ids"]), dtype=bool),
        "forced_before_abstention",
        split,
    ))

test_result = results["test"]
matched_mask = np.zeros(len(test_result["ids"]), dtype=bool)

order = np.lexsort((
    test_result["ids"],
    -test_result["ranking_score"],
))
matched_mask[order[:MATCHED_K]] = True

rows.append(summarize(
    test_result, matched_mask, "ranked_matched_count", "test"
))

summary = pd.DataFrame(rows)

summary.to_csv(OUT / "metric_summary.csv", index=False)
pd.DataFrame(audit_rows).to_csv(
    OUT / "prediction_audit.csv", index=False
)
pd.DataFrame(diagnostic_rows).to_csv(
    OUT / "truncation_diagnostics.csv", index=False
)
pd.DataFrame({
    "record_id": test_result["ids"],
    "retained": matched_mask,
}).to_csv(OUT / "test_matched_selection.csv", index=False)

(OUT / "RUN_COMPLETE.json").write_text(json.dumps({
    "split_seed": SPLIT_SEED,
    "method": "MedCPT_Query",
    "configuration_sha256": config_hash,
    "matched_retained_count": MATCHED_K,
    "metric_summary_sha256": sha256(OUT / "metric_summary.csv"),
}, indent=2))

print("\n=== MEDCPT BASELINE COMPLETE ===")
print(summary[[
    "split_seed", "split", "mode",
    "n_accepted", "coverage",
    "selective_accuracy", "macro_f1",
]].to_string(index=False))

print("\n=== TRUNCATION DIAGNOSTICS ===")
print(pd.DataFrame(diagnostic_rows).to_string(index=False))
print("\nOutputs:", OUT)

In [ ]:
from pathlib import Path
import json
import re

seed11_config_path = Path(
    _crus_local_path('project', 'supplementary_repeated_splits/seed_11/domain_baselines/medcpt_query_v1/baseline_configuration.json')
)

if not seed11_config_path.is_file():
    raise FileNotFoundError(seed11_config_path)

seed11_config = json.loads(seed11_config_path.read_text())
pinned_revision = seed11_config["model_revision"]

history = get_ipython().history_manager.input_hist_raw

markers = [
    'MODEL_ID = "ncbi/MedCPT-Query-Encoder"',
    "def encode(texts):",
    "MEDCPT BASELINE COMPLETE",
    "test_matched_selection.csv",
]

candidates = [
    source
    for source in history
    if all(marker in source for marker in markers)
    and re.search(r"(?m)^SPLIT_SEED\s*=\s*11\s*$", source)
]

if not candidates:
    raise RuntimeError(
        "The original MedCPT seed-11 runner was not found "
        "in this session's history. Do not rerun seed 11."
    )

medcpt_seed22_source, replacements = re.subn(
    r"(?m)^SPLIT_SEED\s*=\s*11\s*$",
    "SPLIT_SEED = 22",
    candidates[-1],
)

assert replacements == 1

medcpt_seed22_source, revision_replacements = re.subn(
    r"(?m)^MODEL_REVISION\s*=\s*model_info\(MODEL_ID\)\.sha\s*$",
    f"MODEL_REVISION = {pinned_revision!r}",
    medcpt_seed22_source,
)

assert revision_replacements == 1, (
    "Could not safely pin the original model revision."
)

assert re.search(
    r"(?m)^MAX_LENGTH\s*=\s*256\s*$",
    medcpt_seed22_source,
)
assert re.search(
    r"(?m)^MODEL_SEED\s*=\s*42\s*$",
    medcpt_seed22_source,
)

seed22_output = Path(
    _crus_local_path('project', 'supplementary_repeated_splits/seed_22/domain_baselines/medcpt_query_v1')
)

if seed22_output.exists() and any(seed22_output.iterdir()):
    raise RuntimeError(
        f"Seed-22 MedCPT outputs already exist: {seed22_output}\n"
        "Nothing was overwritten."
    )

print("Starting MedCPT seed 22.")
print("Pinned model revision:", pinned_revision)
print("Input limit remains 256 tokens.")
print("Seed-11 outputs will be preserved.")

exec(
    compile(
        medcpt_seed22_source,
        "<medcpt_seed_22>",
        "exec",
    ),
    globals(),
)

In [ ]:
from pathlib import Path
import json
import re

if "medcpt_seed22_source" not in globals():
    raise RuntimeError(
        "medcpt_seed22_source is not available. "
        "Use the same Colab session as the completed seed-22 run."
    )

remaining_seeds = [33, 44, 55]

baseline_root = Path(
    _crus_local_path('project', 'supplementary_repeated_splits')
)

seed11_config_path = (
    baseline_root
    / "seed_11"
    / "domain_baselines"
    / "medcpt_query_v1"
    / "baseline_configuration.json"
)

original_config = json.loads(seed11_config_path.read_text())
expected_revision = original_config["model_revision"]

prepared_runners = {}

for next_seed in remaining_seeds:
    source, replacements = re.subn(
        r"(?m)^SPLIT_SEED\s*=\s*22\s*$",
        f"SPLIT_SEED = {next_seed}",
        medcpt_seed22_source,
    )

    assert replacements == 1
    assert re.search(
        r"(?m)^MAX_LENGTH\s*=\s*256\s*$", source
    )
    assert re.search(
        r"(?m)^MODEL_SEED\s*=\s*42\s*$", source
    )
    assert re.search(
        rf"(?m)^MODEL_REVISION\s*=\s*"
        rf"['\"]{re.escape(expected_revision)}['\"]\s*$",
        source,
    ), "The model revision must remain unchanged."

    base = baseline_root / f"seed_{next_seed}"

    for required in [
        base / "split_manifest.csv",
        base / "experiment_outputs" / "pretest_protocol.json",
        base / "experiment_outputs" / "RUN_COMPLETE.json",
        base / "experiment_outputs" / "test_metric_summary.csv",
    ]:
        if not required.is_file():
            raise FileNotFoundError(required)

    output = base / "domain_baselines" / "medcpt_query_v1"

    if output.exists() and any(output.iterdir()):
        raise RuntimeError(
            f"Seed {next_seed} already has MedCPT outputs: {output}\n"
            "Nothing was overwritten. No new run has started."
        )

    prepared_runners[next_seed] = compile(
        source,
        f"<medcpt_seed_{next_seed}>",
        "exec",
    )

print("Preflight checks passed for seeds 33, 44, and 55.")
print("Pinned revision:", expected_revision)

for next_seed in remaining_seeds:
    print("\n" + "=" * 70)
    print(f"STARTING MEDCPT SEED {next_seed}")
    print("=" * 70)

    exec(prepared_runners[next_seed], globals())

print("\nAll remaining supplementary MedCPT runs completed.")

In [ ]:
from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score

PROJECT = Path(_crus_local_path('project', ''))
ROOT = PROJECT / "supplementary_repeated_splits"
SEEDS = [11, 22, 33, 44, 55]

OUT = ROOT / "medcpt_aggregate_summary"

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def save_preserving(path, content):
    if path.exists():
        if path.read_bytes() != content:
            raise RuntimeError(
                f"Different existing summary was preserved: {path}"
            )
    else:
        path.write_bytes(content)

tables = []
truncation_tables = []
audit_rows = []
reference_configuration = None

configuration_fields = [
    "model_id",
    "model_revision",
    "representation",
    "encoder_roles",
    "similarity",
    "max_length_tokens",
    "model_seed",
    "classes",
]

for seed in SEEDS:
    base = ROOT / f"seed_{seed}"
    folder = base / "domain_baselines" / "medcpt_query_v1"

    paths = {
        "configuration": folder / "baseline_configuration.json",
        "completion": folder / "RUN_COMPLETE.json",
        "metrics": folder / "metric_summary.csv",
        "truncation": folder / "truncation_diagnostics.csv",
        "arrays": folder / "test_MedCPT_Query_arrays.npz",
        "selection": folder / "test_matched_selection.csv",
        "original_metrics": (
            base / "experiment_outputs" / "test_metric_summary.csv"
        ),
        "split_manifest": base / "split_manifest.csv",
    }

    for name, path in paths.items():
        if not path.is_file():
            raise FileNotFoundError(f"Seed {seed}, {name}: {path}")

    configuration = json.loads(
        paths["configuration"].read_text()
    )
    completion = json.loads(paths["completion"].read_text())

    assert configuration["split_seed"] == seed
    assert completion["split_seed"] == seed
    assert completion["configuration_sha256"] == sha256(
        paths["configuration"]
    )
    assert completion["metric_summary_sha256"] == sha256(
        paths["metrics"]
    )

    current_configuration = {
        key: configuration[key] for key in configuration_fields
    }

    if reference_configuration is None:
        reference_configuration = current_configuration
    assert current_configuration == reference_configuration

    for name in ["pool", "split", "protocol", "completion", "metrics"]:
        original_path = {
            "pool": (
                PROJECT
                / "data_audit"
                / "constrained_dialogue_pool_with_splits.csv"
            ),
            "split": paths["split_manifest"],
            "protocol": (
                base / "experiment_outputs" / "pretest_protocol.json"
            ),
            "completion": (
                base / "experiment_outputs" / "RUN_COMPLETE.json"
            ),
            "metrics": paths["original_metrics"],
        }[name]

        assert configuration["input_hashes"][name] == sha256(
            original_path
        )

    metrics = pd.read_csv(paths["metrics"])
    test_metrics = metrics.loc[
        metrics["split"].eq("test")
    ].copy()

    assert len(test_metrics) == 2
    assert test_metrics["split_seed"].eq(seed).all()
    assert set(test_metrics["mode"]) == {
        "forced_before_abstention",
        "ranked_matched_count",
    }

    manifest = pd.read_csv(
        paths["split_manifest"],
        dtype={"record_id": str, "group_id": str},
    )
    test_manifest = manifest.loc[
        manifest["split"].eq("test")
    ].set_index("record_id")

    with np.load(paths["arrays"], allow_pickle=False) as saved:
        ids = saved["record_ids"].astype(str)
        true = saved["true_labels"].astype(str)
        pred = saved["predictions"].astype(str)
        ranking = saved["ranking_score"].copy()

        assert saved["configuration_sha256"].item() == sha256(
            paths["configuration"]
        )
        assert saved["classes"].tolist() == configuration["classes"]

    assert len(ids) == len(set(ids)) == 257
    assert set(ids) == set(test_manifest.index)
    assert np.array_equal(
        test_manifest.loc[ids, "label"].to_numpy(), true
    )

    selection = pd.read_csv(
        paths["selection"], dtype={"record_id": str}
    ).set_index("record_id")

    assert set(selection.index) == set(ids)
    retained = selection.loc[ids, "retained"]

    assert retained.isin([True, False]).all()
    retained = retained.to_numpy(dtype=bool)

    original = pd.read_csv(paths["original_metrics"])
    cascade = original.loc[
        original["method"].eq("Cascade")
        & original["mode"].eq("native_CRUS_gate")
    ]

    assert len(cascade) == 1
    matched_k = int(cascade.iloc[0]["n_accepted"])
    assert int(retained.sum()) == matched_k
    assert completion["matched_retained_count"] == matched_k

    expected_mask = np.zeros(len(ids), dtype=bool)
    order = np.lexsort((ids, -ranking))
    expected_mask[order[:matched_k]] = True
    assert np.array_equal(retained, expected_mask)

    for _, row in test_metrics.iterrows():
        accepted = (
            np.ones(len(ids), dtype=bool)
            if row["mode"] == "forced_before_abstention"
            else retained
        )

        correct = pred[accepted] == true[accepted]
        macro_f1 = f1_score(
            true[accepted],
            pred[accepted],
            labels=configuration["classes"],
            average="macro",
            zero_division=0,
        )

        assert int(accepted.sum()) == int(row["n_accepted"])
        assert np.isclose(
            correct.mean(), row["selective_accuracy"]
        )
        assert np.isclose(macro_f1, row["macro_f1"])

    tables.append(test_metrics)

    truncation = pd.read_csv(paths["truncation"])
    truncation.insert(0, "split_seed", seed)
    truncation_tables.append(truncation)

    audit_rows.append({
        "split_seed": seed,
        "completed_and_verified": True,
        "test_records": len(ids),
        "matched_retained_count": matched_k,
        "model_revision": configuration["model_revision"],
    })

combined = pd.concat(tables, ignore_index=True)
truncation_combined = pd.concat(
    truncation_tables, ignore_index=True
)

metric_columns = [
    "coverage", "selective_accuracy",
    "macro_f1", "weighted_f1", "kappa",
]

summary = (
    combined.melt(
        id_vars=["split_seed", "method", "mode"],
        value_vars=metric_columns,
        var_name="metric",
        value_name="value",
    )
    .groupby(["method", "mode", "metric"])["value"]
    .agg(
        n_repeats="count",
        mean="mean",
        sample_sd="std",
        minimum="min",
        maximum="max",
    )
    .reset_index()
)

OUT.mkdir(parents=True, exist_ok=True)

for filename, table in {
    "completion_audit.csv": pd.DataFrame(audit_rows),
    "all_test_metrics.csv": combined,
    "descriptive_summary.csv": summary,
    "all_truncation_diagnostics.csv": truncation_combined,
}.items():
    save_preserving(
        OUT / filename,
        table.to_csv(index=False).encode("utf-8"),
    )

print("\n=== MEDCPT COMPLETION AUDIT ===")
print(pd.DataFrame(audit_rows).to_string(index=False))

print("\n=== MEDCPT FIVE-REPEAT SUMMARY ===")
print(summary.to_string(
    index=False,
    float_format=lambda value: f"{value:.6f}",
))

print("\n=== TEST TRUNCATION BY REPEAT ===")
print(
    truncation_combined.loc[
        truncation_combined["split"].eq("test")
    ].to_string(index=False)
)

print("\n=== POSSIBLE PRIMARY-SPLIT MANIFEST FILES ===")

candidates = sorted(
    path
    for path in PROJECT.rglob("*manifest*.csv")
    if ROOT not in path.parents
)

if candidates:
    for path in candidates:
        print(path.relative_to(PROJECT))
else:
    print("No manifest-named CSV files found outside supplementary outputs.")

print("\nSummary outputs:", OUT)
print("No confidence intervals computed in this aggregation cell.")
print("No primary-split manifest selected automatically.")

In [ ]:
from pathlib import Path
import hashlib
import json
import pandas as pd

PROJECT = Path(_crus_local_path('project', ''))
PRIMARY_OUT = PROJECT / "corrected_experiment_outputs"

candidates = [
    PROJECT / "data_audit/constrained_split_manifest.csv",
    PROJECT / "data_audit/corrected_split_manifest.csv",
]

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

manifest_frames = {}

print("=== PRIMARY-SPLIT MANIFEST CANDIDATES ===")

for path in candidates:
    if not path.is_file():
        print("\nMISSING:", path)
        continue

    frame = pd.read_csv(
        path,
        dtype={"record_id": str, "group_id": str},
    )
    manifest_frames[path.name] = frame

    print("\nFILE:", path.relative_to(PROJECT))
    print("SHA256:", sha256(path))
    print("Rows:", len(frame))
    print("Columns:", frame.columns.tolist())

    if "split" in frame.columns:
        print("Split counts:")
        print(frame["split"].value_counts(dropna=False).to_string())

    if "record_id" in frame.columns:
        print("Unique record IDs:", frame["record_id"].is_unique)

    if {"group_id", "split"}.issubset(frame.columns):
        print(
            "Maximum partitions per group:",
            frame.groupby("group_id")["split"].nunique().max(),
        )

    print("First three rows:")
    print(frame.head(3).to_string(index=False))

print("\n=== CANDIDATE SPLIT-ASSIGNMENT COMPARISON ===")

names = list(manifest_frames)

if len(names) == 2:
    left = manifest_frames[names[0]]
    right = manifest_frames[names[1]]

    if all(
        {"record_id", "split"}.issubset(frame.columns)
        for frame in [left, right]
    ):
        comparison = left[["record_id", "split"]].merge(
            right[["record_id", "split"]],
            on="record_id",
            how="outer",
            suffixes=("_left", "_right"),
            indicator=True,
            validate="one_to_one",
        )

        shared = comparison["_merge"].eq("both")

        print("Left:", names[0])
        print("Right:", names[1])
        print(
            "Record-ID membership:",
            comparison["_merge"].value_counts().to_dict(),
        )
        print(
            "Different assignments among shared records:",
            int(
                (
                    comparison.loc[shared, "split_left"]
                    != comparison.loc[shared, "split_right"]
                ).sum()
            ),
        )

print("\n=== PRIMARY OUTPUT DIRECTORY CONTENTS ===")

if PRIMARY_OUT.is_dir():
    for path in sorted(PRIMARY_OUT.iterdir()):
        if path.is_file():
            print(path.name)
else:
    print("Directory not found:", PRIMARY_OUT)

print("\n=== PRIMARY TEST CSV SCHEMAS AND MANIFEST MATCHES ===")

prediction_candidates = (
    sorted({
        *PRIMARY_OUT.glob("*test*.csv"),
        *PRIMARY_OUT.glob("*prediction*.csv"),
    })
    if PRIMARY_OUT.is_dir()
    else []
)

for path in prediction_candidates:
    try:
        header = pd.read_csv(path, nrows=0)
        print("\nFILE:", path.name)
        print("Columns:", header.columns.tolist())

        if "record_id" not in header.columns:
            continue

        frame = pd.read_csv(path, dtype={"record_id": str})

        # Only treat rows explicitly labelled test as test rows
        # when a split column is present.
        if "split" in frame.columns:
            frame = frame.loc[
                frame["split"].astype(str).str.casefold().eq("test")
            ]

        ids = set(frame["record_id"].dropna())
        print("Unique record IDs in inspected rows:", len(ids))

        for name, manifest in manifest_frames.items():
            if not {"record_id", "split"}.issubset(manifest.columns):
                continue

            test_ids = set(
                manifest.loc[
                    manifest["split"].astype(str)
                    .str.casefold().eq("test"),
                    "record_id",
                ]
            )

            print(
                f"Exact test-ID match with {name}:",
                bool(ids) and ids == test_ids,
            )

    except Exception as exc:
        print(type(exc).__name__, str(exc))

print("\n=== PRIMARY JSON METADATA PREVIEW ===")

metadata_candidates = (
    sorted({
        *PRIMARY_OUT.glob("*protocol*.json"),
        *PRIMARY_OUT.glob("*config*.json"),
        *PRIMARY_OUT.glob("*metadata*.json"),
        *PRIMARY_OUT.glob("*COMPLETE*.json"),
    })
    if PRIMARY_OUT.is_dir()
    else []
)

for path in metadata_candidates[:12]:
    try:
        payload = json.loads(path.read_text())
        print("\nFILE:", path.name)

        if not isinstance(payload, dict):
            print("JSON is not a dictionary.")
            continue

        print("Top-level keys:", list(payload))

        for key, value in payload.items():
            if any(
                word in key.casefold()
                for word in ["manifest", "split", "seed", "hash"]
            ):
                print(
                    key,
                    "=",
                    json.dumps(value, ensure_ascii=False)[:1600],
                )

    except Exception as exc:
        print(type(exc).__name__, str(exc))

print("\nRead-only inspection complete. No files were modified.")

In [ ]:
from pathlib import Path
import hashlib
import json
import random
import numpy as np
import pandas as pd
import torch
import transformers

from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import f1_score, cohen_kappa_score

ROOT = Path(_crus_local_path('project', ''))
PRIMARY = ROOT / "corrected_experiment_outputs"
OUT = PRIMARY / "domain_baselines" / "medcpt_query_v1"

paths = {
    "pool": ROOT / "data_audit/constrained_dialogue_pool_with_splits.csv",
    "manifest": ROOT / "data_audit/constrained_split_manifest.csv",
    "protocol": PRIMARY / "final_pretest_protocol.json",
    "predictions": PRIMARY / "test_all_method_predictions.csv",
    "native": PRIMARY / "test_native_gate_operating_points.csv",
    "reference_config": (
        ROOT / "supplementary_repeated_splits/seed_11/"
        "domain_baselines/medcpt_query_v1/baseline_configuration.json"
    ),
}

for name, path in paths.items():
    if not path.is_file():
        raise FileNotFoundError(f"{name}: {path}")

if OUT.exists() and any(OUT.iterdir()):
    raise RuntimeError(
        f"Existing MedCPT primary outputs were preserved: {OUT}"
    )

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

protocol = json.loads(paths["protocol"].read_text())
reference = json.loads(paths["reference_config"].read_text())

EXPECTED_MANIFEST_HASH = (
    "ca62ec75e905a4dbf4508ad172fe33e50"
    "f7b7779e4a90ad6f58c59c91a7c4d91"
)

assert sha256(paths["manifest"]) == EXPECTED_MANIFEST_HASH
assert protocol["main_manifest_sha256"] == EXPECTED_MANIFEST_HASH
assert protocol["classes"] == reference["classes"]

CLASSES = protocol["classes"]
MODEL_ID = reference["model_id"]
REVISION = reference["model_revision"]
MAX_LENGTH = reference["max_length_tokens"]
BATCH_SIZE = reference["batch_size"]
MODEL_SEED = reference["model_seed"]

assert MODEL_ID == "ncbi/MedCPT-Query-Encoder"
assert MAX_LENGTH == 256

pool = pd.read_csv(
    paths["pool"], dtype={"record_id": str, "group_id": str}
)
manifest = pd.read_csv(
    paths["manifest"], dtype={"record_id": str, "group_id": str}
)

assert pool["record_id"].is_unique
assert manifest["record_id"].is_unique

joined = manifest.merge(
    pool[[
        "record_id", "label", "group_id",
        "dialogue_hash", "text", "dialogue",
    ]],
    on="record_id",
    how="left",
    suffixes=("", "_pool"),
    validate="one_to_one",
)

for column in ["label", "group_id", "dialogue_hash"]:
    assert joined[column].eq(joined[f"{column}_pool"]).all()

assert joined[["text", "dialogue"]].notna().all().all()

def normalize(value):
    return " ".join(str(value).split()).casefold()

assert joined["text"].map(normalize).eq(
    joined["dialogue"].map(normalize)
).all()

assert joined.groupby("group_id")["split"].nunique().max() == 1
assert joined.groupby("dialogue_hash")["split"].nunique().max() == 1

frames = {
    split: joined.loc[joined["split"].eq(split)]
    .sort_values("record_id").reset_index(drop=True)
    for split in ["train", "validation", "test"]
}

assert [len(frames[x]) for x in frames] == [773, 258, 257]
assert sorted(joined["label"].unique()) == CLASSES

existing = pd.read_csv(
    paths["predictions"], dtype={"record_id": str}
)
test_labels = frames["test"].set_index("record_id")["label"]

assert existing["method"].nunique() == 9

for method, group in existing.groupby("method"):
    assert group["record_id"].is_unique
    assert set(group["record_id"]) == set(test_labels.index)
    assert np.array_equal(
        test_labels.loc[group["record_id"]].to_numpy(),
        group["true_label"].to_numpy(),
    )

native = pd.read_csv(paths["native"])
cascade_native = native.loc[native["method"].eq("Cascade")]

assert len(cascade_native) == 1
MATCHED_K = int(cascade_native.iloc[0]["n_accepted"])
assert MATCHED_K == 117

random.seed(MODEL_SEED)
np.random.seed(MODEL_SEED)
torch.manual_seed(MODEL_SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(MODEL_SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID, revision=REVISION
)
model = AutoModel.from_pretrained(
    MODEL_ID, revision=REVISION
).to(DEVICE)
model.eval()

OUT.mkdir(parents=True, exist_ok=True)

configuration = {
    "method": "MedCPT_Query",
    "split": "primary_constrained",
    "model_id": MODEL_ID,
    "model_revision": REVISION,
    "model_seed": MODEL_SEED,
    "max_length_tokens": MAX_LENGTH,
    "batch_size": BATCH_SIZE,
    "representation": reference["representation"],
    "encoder_roles": reference["encoder_roles"],
    "similarity": reference["similarity"],
    "input": "normalized dialogue only",
    "index": "training dialogues only",
    "prediction": "label of highest-scoring training dialogue",
    "tie_breaking": "ascending record_id",
    "probability_model": None,
    "classes": CLASSES,
    "device": DEVICE,
    "torch_version": torch.__version__,
    "transformers_version": transformers.__version__,
    "input_hashes": {
        name: sha256(path) for name, path in paths.items()
    },
}

config_path = OUT / "baseline_configuration.json"
config_path.write_text(
    json.dumps(configuration, indent=2, sort_keys=True)
)
config_hash = sha256(config_path)

def encode(texts):
    chunks = []

    with torch.inference_mode():
        for start in range(0, len(texts), BATCH_SIZE):
            inputs = tokenizer(
                list(texts[start:start + BATCH_SIZE]),
                padding=True,
                truncation=True,
                max_length=MAX_LENGTH,
                return_tensors="pt",
            ).to(DEVICE)

            values = model(**inputs).last_hidden_state[:, 0, :]
            chunks.append(values.cpu().numpy())

    embeddings = np.concatenate(chunks).astype(np.float32)
    assert np.isfinite(embeddings).all()
    return embeddings

diagnostics = []

for split, frame in frames.items():
    lengths = np.array([
        len(tokenizer(
            text,
            truncation=False,
            add_special_tokens=True,
            verbose=False,
        )["input_ids"])
        for text in frame["text"]
    ])

    diagnostics.append({
        "split": split,
        "n_records": len(frame),
        "max_length_tokens": MAX_LENGTH,
        "n_truncated": int((lengths > MAX_LENGTH).sum()),
        "fraction_truncated": float((lengths > MAX_LENGTH).mean()),
        "median_original_tokens": float(np.median(lengths)),
        "maximum_original_tokens": int(lengths.max()),
    })

train = frames["train"]
print("Encoding primary training dialogues...")
train_embeddings = encode(train["text"].to_numpy())
train_ids = train["record_id"].to_numpy(dtype=str)
train_labels = train["label"].to_numpy(dtype=str)

np.savez_compressed(
    OUT / "training_embeddings.npz",
    record_ids=train_ids,
    embeddings=train_embeddings,
)

results = {}
audit = []

for split in ["validation", "test"]:
    frame = frames[split]
    print(f"Encoding and retrieving: {split}")

    embeddings = encode(frame["text"].to_numpy())
    scores = embeddings @ train_embeddings.T
    top = np.argmax(scores, axis=1)

    ids = frame["record_id"].to_numpy(dtype=str)
    true = frame["label"].to_numpy(dtype=str)
    pred = train_labels[top]
    ranking = scores[np.arange(len(frame)), top]

    results[split] = {
        "ids": ids, "true": true,
        "pred": pred, "ranking_score": ranking,
    }

    np.savez_compressed(
        OUT / f"{split}_MedCPT_Query_arrays.npz",
        record_ids=ids,
        true_labels=true,
        predictions=pred,
        classes=np.asarray(CLASSES, dtype=str),
        ranking_score=ranking,
        retrieved_record_ids=train_ids[top],
        configuration_sha256=np.asarray(config_hash),
    )

    for i, record_id in enumerate(ids):
        audit.append({
            "split": split,
            "record_id": record_id,
            "group_id": frame.iloc[i]["group_id"],
            "true_label": true[i],
            "prediction": pred[i],
            "correct": bool(pred[i] == true[i]),
            "top_inner_product": float(ranking[i]),
            "retrieved_training_record_id": train_ids[top[i]],
        })

def summarize(result, accepted, split, mode):
    n = len(result["ids"])
    retained = int(accepted.sum())
    correct = result["pred"] == result["true"]
    n_correct = int(correct[accepted].sum())

    yt = result["true"][accepted]
    yp = result["pred"][accepted]

    return {
        "method": "MedCPT_Query",
        "split": split,
        "mode": mode,
        "N": n,
        "n_accepted": retained,
        "n_correct_accepted": n_correct,
        "n_errors_accepted": retained - n_correct,
        "n_abstained": n - retained,
        "coverage": retained / n,
        "forced_accuracy": float(correct.mean()),
        "selective_accuracy": n_correct / retained,
        "macro_f1": f1_score(
            yt, yp, labels=CLASSES,
            average="macro", zero_division=0,
        ),
        "weighted_f1": f1_score(
            yt, yp, labels=CLASSES,
            average="weighted", zero_division=0,
        ),
        "kappa": cohen_kappa_score(yt, yp, labels=CLASSES),
        "ece": np.nan,
        "brier": np.nan,
        "nll": np.nan,
    }

rows = []

for split, result in results.items():
    rows.append(summarize(
        result,
        np.ones(len(result["ids"]), dtype=bool),
        split,
        "forced_before_abstention",
    ))

test = results["test"]
order = np.lexsort((test["ids"], -test["ranking_score"]))
accepted = np.zeros(len(test["ids"]), dtype=bool)
accepted[order[:MATCHED_K]] = True

rows.append(summarize(
    test, accepted, "test", "ranked_matched_count"
))

assert sha256(config_path) == config_hash

summary = pd.DataFrame(rows)
summary.to_csv(OUT / "metric_summary.csv", index=False)
pd.DataFrame(audit).to_csv(
    OUT / "prediction_audit.csv", index=False
)
pd.DataFrame(diagnostics).to_csv(
    OUT / "truncation_diagnostics.csv", index=False
)
pd.DataFrame({
    "record_id": test["ids"],
    "retained": accepted,
}).to_csv(OUT / "test_matched_selection.csv", index=False)

(OUT / "RUN_COMPLETE.json").write_text(json.dumps({
    "split": "primary_constrained",
    "method": "MedCPT_Query",
    "configuration_sha256": config_hash,
    "matched_retained_count": MATCHED_K,
    "metric_summary_sha256": sha256(OUT / "metric_summary.csv"),
}, indent=2))

print("\n=== PRIMARY MEDCPT COMPLETE ===")
print(summary.to_string(
    index=False,
    float_format=lambda value: f"{value:.6f}",
))

print("\n=== PRIMARY TRUNCATION DIAGNOSTICS ===")
print(pd.DataFrame(diagnostics).to_string(index=False))
print("\nOutputs:", OUT)

In [ ]:
from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd

ROOT = Path(
    _crus_local_path('project', 'supplementary_repeated_splits')
)

SEEDS = [11, 22, 33, 44, 55]
B = 5000
BOOTSTRAP_SEED = 20261015

OUT = ROOT / "medcpt_conditional_bootstrap"

if OUT.exists() and any(OUT.iterdir()):
    raise RuntimeError(
        f"Existing statistical outputs were preserved: {OUT}"
    )

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

records = []
input_hashes = {}

for seed in SEEDS:
    base = ROOT / f"seed_{seed}"
    med = base / "domain_baselines" / "medcpt_query_v1"
    original = base / "experiment_outputs"

    manifest_path = base / "split_manifest.csv"
    manifest = pd.read_csv(
        manifest_path,
        dtype={"record_id": str, "group_id": str},
    )
    test_manifest = manifest.loc[
        manifest["split"].eq("test")
    ].set_index("record_id")

    config_path = med / "baseline_configuration.json"
    metrics_path = med / "metric_summary.csv"
    complete_path = med / "RUN_COMPLETE.json"

    configuration = json.loads(config_path.read_text())
    completion = json.loads(complete_path.read_text())

    assert completion["configuration_sha256"] == sha256(config_path)
    assert completion["metric_summary_sha256"] == sha256(metrics_path)
    assert configuration["input_hashes"]["split"] == sha256(manifest_path)

    med_metrics = pd.read_csv(metrics_path)
    original_metrics = pd.read_csv(
        original / "test_metric_summary.csv"
    )

    cascade_native = original_metrics.loc[
        original_metrics["method"].eq("Cascade")
        & original_metrics["mode"].eq("native_CRUS_gate")
    ]
    assert len(cascade_native) == 1
    k = int(cascade_native.iloc[0]["n_accepted"])

    for method in ["MedCPT_Query", "SBERT", "Cascade"]:
        path = (
            med / "test_MedCPT_Query_arrays.npz"
            if method == "MedCPT_Query"
            else original / f"test_{method}_arrays.npz"
        )
        input_hashes[str(path.relative_to(ROOT))] = sha256(path)

        with np.load(path, allow_pickle=False) as saved:
            ids = saved["record_ids"].astype(str)
            true = saved["true_labels"].astype(str)
            pred = saved["predictions"].astype(str)
            score = saved["ranking_score"].copy()

        assert len(ids) == len(set(ids)) == 257
        assert set(ids) == set(test_manifest.index)
        assert np.array_equal(
            test_manifest.loc[ids, "label"].to_numpy(), true
        )

        groups = test_manifest.loc[
            ids, "group_id"
        ].to_numpy(dtype=str)

        ranked = np.zeros(len(ids), dtype=bool)
        order = np.lexsort((ids, -score))
        ranked[order[:k]] = True

        source_metrics = (
            med_metrics.loc[med_metrics["split"].eq("test")]
            if method == "MedCPT_Query"
            else original_metrics.loc[
                original_metrics["method"].eq(method)
            ]
        )

        for mode, accepted in [
            (
                "forced_before_abstention",
                np.ones(len(ids), dtype=bool),
            ),
            ("ranked_matched_count", ranked),
        ]:
            reference = source_metrics.loc[
                source_metrics["mode"].eq(mode)
            ]
            assert len(reference) == 1
            reference = reference.iloc[0]

            correct = pred == true

            assert int(accepted.sum()) == int(reference["n_accepted"])
            assert np.isclose(
                correct[accepted].mean(),
                reference["selective_accuracy"],
            )

            records.append({
                "seed": seed,
                "method": method,
                "mode": mode,
                "groups": groups,
                "accepted": accepted,
                "correct": correct,
            })

all_groups = sorted({
    group
    for record in records
    for group in record["groups"]
})
group_index = {group: i for i, group in enumerate(all_groups)}
G = len(all_groups)

# Confirm compatibility with the earlier bootstrap specification.
previous_metadata_path = (
    ROOT / "r1_4_conditional_bootstrap" / "analysis_metadata.json"
)

if not previous_metadata_path.is_file():
    raise FileNotFoundError(previous_metadata_path)

previous_metadata = json.loads(previous_metadata_path.read_text())

assert previous_metadata["bootstrap_seed"] == BOOTSTRAP_SEED
assert previous_metadata["bootstrap_replicates"] == B
assert previous_metadata["unique_observed_test_groups"] == G

for record in records:
    record["group_indices"] = np.asarray([
        group_index[group] for group in record["groups"]
    ])

def evaluate(record, multiplicities):
    weights = multiplicities[record["group_indices"]]
    total = weights.sum()
    accepted_weights = weights * record["accepted"]
    retained = accepted_weights.sum()

    if total == 0 or retained == 0:
        return np.array([np.nan, np.nan])

    return np.array([
        retained / total,
        np.dot(accepted_weights, record["correct"]) / retained,
    ])

point = np.stack([
    evaluate(record, np.ones(G, dtype=int))
    for record in records
])

rng = np.random.default_rng(BOOTSTRAP_SEED)
bootstrap = np.empty((B, len(records), 2))

for b in range(B):
    sampled = rng.integers(0, G, size=G)
    multiplicities = np.bincount(sampled, minlength=G)

    for i, record in enumerate(records):
        bootstrap[b, i] = evaluate(record, multiplicities)

    if (b + 1) % 1000 == 0:
        print(f"Completed {b + 1}/{B}")

def interval(values):
    valid = values[np.isfinite(values)]
    if len(valid) < 0.99 * B:
        raise RuntimeError("Too many undefined bootstrap replicates.")

    lower, upper = np.quantile(valid, [0.025, 0.975])
    return float(lower), float(upper), len(valid)

lookup = {
    (record["seed"], record["method"], record["mode"]): i
    for i, record in enumerate(records)
}

mean_rows = []
paired_rows = []

for mode in ["forced_before_abstention", "ranked_matched_count"]:
    med_indices = [
        lookup[(seed, "MedCPT_Query", mode)] for seed in SEEDS
    ]

    observed = point[med_indices]
    bootstrap_means = bootstrap[:, med_indices, :].mean(axis=1)

    for j, metric in enumerate(["coverage", "selective_accuracy"]):
        lower, upper, valid = interval(bootstrap_means[:, j])

        mean_rows.append({
            "method": "MedCPT_Query",
            "mode": mode,
            "metric": metric,
            "mean": observed[:, j].mean(),
            "sample_sd": observed[:, j].std(ddof=1),
            "conditional_ci95_lower": lower,
            "conditional_ci95_upper": upper,
            "valid_bootstrap_replicates": valid,
        })

    for comparator in ["SBERT", "Cascade"]:
        comparator_indices = [
            lookup[(seed, comparator, mode)] for seed in SEEDS
        ]

        observed_difference = (
            point[med_indices, 1]
            - point[comparator_indices, 1]
        ).mean()

        bootstrap_difference = (
            bootstrap[:, med_indices, 1]
            - bootstrap[:, comparator_indices, 1]
        ).mean(axis=1)

        lower, upper, valid = interval(bootstrap_difference)

        paired_rows.append({
            "comparison": f"MedCPT_Query minus {comparator}",
            "mode": mode,
            "metric": "selective_accuracy",
            "mean_difference": observed_difference,
            "conditional_ci95_lower": lower,
            "conditional_ci95_upper": upper,
            "valid_bootstrap_replicates": valid,
            "analysis": "exploratory; no multiplicity adjustment",
        })

means = pd.DataFrame(mean_rows)
paired = pd.DataFrame(paired_rows)

OUT.mkdir(parents=True, exist_ok=True)

means.to_csv(
    OUT / "medcpt_mean_conditional_ci95.csv", index=False
)
paired.to_csv(
    OUT / "medcpt_paired_accuracy_differences.csv", index=False
)

(OUT / "analysis_metadata.json").write_text(json.dumps({
    "bootstrap_replicates": B,
    "bootstrap_seed": BOOTSTRAP_SEED,
    "unique_observed_test_groups": G,
    "resampling_unit": "group_id",
    "interval_method": "95% percentile",
    "conditioning": (
        "Models, splits, policies, and original selections fixed."
    ),
    "difference_direction": "MedCPT_Query minus comparator",
    "input_hashes": input_hashes,
}, indent=2, sort_keys=True))

print("\n=== MEDCPT CONDITIONAL INTERVALS ===")
print(means.to_string(
    index=False, float_format=lambda value: f"{value:.6f}"
))

print("\n=== MEDCPT PAIRED ACCURACY DIFFERENCES ===")
print(paired.to_string(
    index=False, float_format=lambda value: f"{value:.6f}"
))

print("\nOutputs:", OUT)
print("No models retrained and no existing results modified.")

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path(
    _crus_local_path('project', 'supplementary_repeated_splits')
)

SEEDS = [11, 22, 33, 44, 55]
MODES = [
    "forced_before_abstention",
    "ranked_matched_count",
]

rows = []

for seed in SEEDS:
    path = (
        ROOT
        / f"seed_{seed}"
        / "domain_baselines"
        / "medcpt_query_v1"
        / "metric_summary.csv"
    )

    if not path.is_file():
        raise FileNotFoundError(path)

    df = pd.read_csv(path)

    required = {"method", "split", "mode", "macro_f1"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"{path}: missing columns {missing}")

    for mode in MODES:
        selected = df.loc[
            df["method"].eq("MedCPT_Query")
            & df["split"].eq("test")
            & df["mode"].eq(mode)
        ]

        if len(selected) != 1:
            raise ValueError(
                f"Seed {seed}, mode {mode}: "
                f"expected one row, found {len(selected)}"
            )

        value = float(selected.iloc[0]["macro_f1"])

        if not np.isfinite(value) or not 0 <= value <= 1:
            raise ValueError(
                f"Invalid macro-F1 for seed {seed}: {value}"
            )

        rows.append({
            "seed": seed,
            "mode": mode,
            "macro_f1": value,
        })

per_repeat = pd.DataFrame(rows)

summary = (
    per_repeat.groupby("mode", sort=False)["macro_f1"]
    .agg(
        n_repeats="count",
        mean="mean",
        sample_sd=lambda values: values.std(ddof=1),
    )
    .reset_index()
)

print("=== TEST MACRO-F1 FOR EACH REPEAT ===")
print(
    per_repeat.pivot(
        index="seed", columns="mode", values="macro_f1"
    ).to_string(float_format=lambda value: f"{value:.8f}")
)

print("\n=== VERIFIED FIVE-REPEAT SUMMARY ===")
print(
    summary.to_string(
        index=False,
        float_format=lambda value: f"{value:.8f}",
    )
)

draft_values = {
    "forced_before_abstention": ("0.3231", "0.0215"),
    "ranked_matched_count": ("0.2553", "0.0224"),
}

print("\n=== TABLE 5 VALUES AND DRAFT CHECK ===")

for row in summary.itertuples(index=False):
    actual = (f"{row.mean:.4f}", f"{row.sample_sd:.4f}")
    expected = draft_values[row.mode]

    panel = (
        "Panel A"
        if row.mode == "forced_before_abstention"
        else "Panel C"
    )

    print(
        f"{panel}: {actual[0]} ± {actual[1]} | "
        f"Matches earlier draft: {actual == expected}"
    )
    print(
        "LaTeX macro-F1 cell: "
        + rf"\({row.mean:.4f} \pm {row.sample_sd:.4f}\)"
    )

In [ ]:
from pathlib import Path

MYDRIVE = Path(_crus_local_path('drive', ''))

if not MYDRIVE.is_dir():
    pass  # Colab service not required locally.
    _crus_prepare_saved_workspace()

ROOT = MYDRIVE / "RiskAware_Revision2"

print("MyDrive accessible:", MYDRIVE.is_dir())
print("Project folder exists:", ROOT.is_dir())

if not ROOT.is_dir():
    print("\nFolders visible in MyDrive:")
    if MYDRIVE.is_dir():
        for folder in sorted(MYDRIVE.iterdir()):
            if folder.is_dir():
                print(folder.name)

    raise FileNotFoundError(
        "Project folder not found. Check the mounted Google "
        "account and the project folder name."
    )

print("\n=== MEDCPT METRIC FILES FOUND ===")

metric_paths = sorted(
    path
    for path in ROOT.rglob("metric_summary.csv")
    if "medcpt" in str(path).lower()
)

for path in metric_paths:
    print(path)

print("\nTotal MedCPT metric files:", len(metric_paths))

In [ ]:
import numpy as np
import pandas as pd

SEEDS = [11, 22, 33, 44, 55]
MODES = [
    "forced_before_abstention",
    "ranked_matched_count",
]

rows = []

for seed in SEEDS:
    candidates = [
        path for path in metric_paths
        if f"seed_{seed}" in path.parts
    ]

    if len(candidates) != 1:
        raise RuntimeError(
            f"Seed {seed}: expected exactly one MedCPT "
            f"metric file, found {len(candidates)}.\n"
            + "\n".join(str(path) for path in candidates)
        )

    path = candidates[0]
    df = pd.read_csv(path)

    required = {"method", "split", "mode", "macro_f1"}
    missing = required - set(df.columns)

    if missing:
        raise ValueError(f"{path}: missing columns {missing}")

    for mode in MODES:
        selected = df.loc[
            df["method"].eq("MedCPT_Query")
            & df["split"].eq("test")
            & df["mode"].eq(mode)
        ]

        if len(selected) != 1:
            raise ValueError(
                f"Seed {seed}, {mode}: "
                f"expected one test row, found {len(selected)}"
            )

        value = float(selected.iloc[0]["macro_f1"])

        if not np.isfinite(value) or not 0 <= value <= 1:
            raise ValueError(f"Invalid macro-F1: {value}")

        rows.append({
            "seed": seed,
            "mode": mode,
            "macro_f1": value,
        })

results = pd.DataFrame(rows)

print("=== PER-REPEAT TEST MACRO-F1 ===")
print(
    results.pivot(
        index="seed",
        columns="mode",
        values="macro_f1",
    ).to_string(float_format=lambda x: f"{x:.8f}")
)

print("\n=== VERIFIED TABLE 5 CELLS ===")

for mode, panel in [
    ("forced_before_abstention", "Panel A"),
    ("ranked_matched_count", "Panel C"),
]:
    values = results.loc[
        results["mode"].eq(mode), "macro_f1"
    ]

    assert len(values) == 5

    mean = values.mean()
    sd = values.std(ddof=1)

    print(f"{panel}: mean={mean:.8f}, sample SD={sd:.8f}")
    print(rf"LaTeX: \({mean:.4f} \pm {sd:.4f}\)")

In [ ]:
from pathlib import Path
import numpy as np

ROOT = Path(_crus_local_path('project', ''))

if not ROOT.is_dir():
    raise FileNotFoundError(
        "Project folder unavailable. Mount Google Drive first."
    )

targets = [
    ROOT / "corrected_experiment_outputs"
         / "test_Cascade_arrays.npz",

    ROOT / "supplementary_repeated_splits"
         / "seed_11"
         / "experiment_outputs"
         / "test_Cascade_arrays.npz",
]

for path in targets:
    print("\nFILE:", path)

    if not path.is_file():
        print("Not found at expected location.")
        continue

    with np.load(path, allow_pickle=False) as saved:
        for key in saved.files:
            try:
                array = saved[key]
                print(
                    f"  {key}: "
                    f"shape={array.shape}, dtype={array.dtype}"
                )
            except ValueError as exc:
                print(f"  {key}: could not inspect safely: {exc}")

print("\n=== CASCADE ARRAY FILES IN PRIMARY OUTPUTS ===")

primary = ROOT / "corrected_experiment_outputs"

if primary.is_dir():
    for path in sorted(primary.rglob("*Cascade*arrays.npz")):
        print(path)

print("\n=== VALIDATION CASCADE ARRAY FILES IN SEED 11 ===")

seed11 = ROOT / "supplementary_repeated_splits" / "seed_11"

if seed11.is_dir():
    for path in sorted(seed11.rglob("*Cascade*arrays.npz")):
        if "val" in path.name.lower():
            print(path)

In [ ]:
from pathlib import Path
import json
import re

ROOT = Path(_crus_local_path('project', ''))

patterns = re.compile(
    r"\bCRUS\b|\bcrus\b|tau_u|tau_c|"
    r"ranking_score\s*=|[\"']ranking_score[\"']\s*:"
)

matches = 0
MAX_MATCHES = 40

for path in sorted(ROOT.rglob("*")):
    if path.suffix not in {".py", ".ipynb"}:
        continue
    if ".ipynb_checkpoints" in path.parts:
        continue

    try:
        if path.suffix == ".py":
            sections = [
                ("script", path.read_text(encoding="utf-8"))
            ]
        else:
            notebook = json.loads(
                path.read_text(encoding="utf-8")
            )
            sections = [
                (
                    f"cell {i}",
                    "".join(cell.get("source", [])),
                )
                for i, cell in enumerate(
                    notebook.get("cells", [])
                )
                if cell.get("cell_type") == "code"
            ]
    except (OSError, UnicodeError, json.JSONDecodeError):
        continue

    for section_name, text in sections:
        lines = text.splitlines()
        hits = [
            i for i, line in enumerate(lines)
            if patterns.search(line)
        ]

        if not hits:
            continue

        print(f"\n=== {path} | {section_name} ===")

        shown = set()

        for hit in hits:
            for i in range(
                max(0, hit - 5),
                min(len(lines), hit + 9),
            ):
                if i not in shown:
                    print(f"{i + 1:4d}: {lines[i]}")
                    shown.add(i)

            matches += 1

            if matches >= MAX_MATCHES:
                break

        if matches >= MAX_MATCHES:
            break

    if matches >= MAX_MATCHES:
        print("\nOutput capped at 40 matching regions.")
        break

print("\nMatching regions printed:", matches)

In [ ]:
from pathlib import Path
import json
import numpy as np

ROOT = Path(_crus_local_path('project', ''))

if not ROOT.is_dir():
    raise FileNotFoundError(
        "Project folder unavailable. Mount Google Drive first."
    )

BASES = {
    "primary": ROOT / "corrected_experiment_outputs",
    "seed_11": (
        ROOT / "supplementary_repeated_splits"
        / "seed_11" / "experiment_outputs"
    ),
}

KEYWORDS = (
    "crus", "tau_", "threshold", "ranking",
    "uncertainty", "component", "confidence",
    "entropy", "margin", "policy", "gate",
)

def compare_scores(name, score, candidates):
    score = np.asarray(score, dtype=float).reshape(-1)

    if not np.isfinite(score).all():
        print(f"  {name}: contains non-finite values")
        return

    print(
        f"\n  {name}: "
        f"min={score.min():.8f}, "
        f"median={np.median(score):.8f}, "
        f"max={score.max():.8f}"
    )

    matches = []

    for candidate_name, values in candidates.items():
        values = np.asarray(values, dtype=float).reshape(-1)

        if values.shape != score.shape:
            continue
        if not np.isfinite(values).all():
            continue

        if np.allclose(score, values, rtol=1e-7, atol=1e-9):
            matches.append(candidate_name)

    if matches:
        print("  Numerical matches:", ", ".join(matches))
    else:
        print("  No exact match among tested candidate formulas.")

def inspect_arrays(path):
    print(f"\nFILE: {path}")

    if not path.is_file():
        print("  FILE NOT FOUND")
        return

    # Object-valued fields are skipped; pickle is never enabled.
    arrays = {}

    with np.load(path, allow_pickle=False) as saved:
        for key in saved.files:
            try:
                arrays[key] = saved[key]
                print(
                    f"  {key}: shape={arrays[key].shape}, "
                    f"dtype={arrays[key].dtype}"
                )
            except ValueError:
                print(
                    f"  {key}: object-valued field skipped "
                    "(allow_pickle=False)"
                )

    if "probabilities" not in arrays:
        print("  No probabilities found.")
        return

    p = np.asarray(arrays["probabilities"], dtype=float)

    if p.ndim != 2 or p.shape[1] < 2:
        print("  INVALID probability array shape:", p.shape)
        return

    finite = bool(np.isfinite(p).all())
    nonnegative = bool(finite and np.all(p >= -1e-12))
    bounded = bool(finite and np.all(p <= 1 + 1e-12))
    sums = p.sum(axis=1)
    normalized = bool(
        finite
        and np.allclose(sums, 1.0, rtol=1e-6, atol=1e-8)
    )

    print("\n  PROBABILITY CHECKS")
    print("  All finite:", finite)
    print("  Nonnegative:", nonnegative)
    print("  At most one:", bounded)
    print("  Rows sum to one:", normalized)

    if finite:
        print(
            "  Row-sum range:",
            f"{sums.min():.10f} to {sums.max():.10f}"
        )

    if not all([finite, nonnegative, bounded, normalized]):
        print("  STOP: investigate probability validity.")
        return

    maximum = p.max(axis=1)
    ordered = np.sort(p, axis=1)
    margin = ordered[:, -1] - ordered[:, -2]

    safe = np.clip(p, 1e-300, 1.0)
    entropy = -(p * np.log(safe)).sum(axis=1)
    normalized_entropy = entropy / np.log(p.shape[1])

    candidates = {
        "maximum_probability": maximum,
        "1 - maximum_probability": 1 - maximum,
        "probability_margin": margin,
        "1 - probability_margin": 1 - margin,
        "entropy_nats": entropy,
        "normalized_entropy": normalized_entropy,
        "1 - normalized_entropy": 1 - normalized_entropy,
    }

    classes = arrays.get("classes")
    predictions = arrays.get("predictions")

    if classes is not None and predictions is not None:
        if (
            len(classes) == p.shape[1]
            and len(predictions) == p.shape[0]
        ):
            argmax_predictions = classes[p.argmax(axis=1)]
            agreement = (
                argmax_predictions.astype(str)
                == predictions.astype(str)
            )

            print(
                "\n  Saved prediction / probability-argmax "
                f"agreement: {agreement.sum()}/{len(agreement)}"
            )
        else:
            print("\n  Class/prediction dimensions do not match.")

    components = arrays.get("components")

    if components is not None:
        components = np.asarray(components, dtype=float)

        if (
            components.ndim == 2
            and components.shape[0] == p.shape[0]
        ):
            print("\n  COMPONENT CHECKS")

            for j in range(components.shape[1]):
                component = components[:, j]

                compare_scores(
                    f"component_{j}",
                    component,
                    candidates,
                )

                candidates[f"component_{j}"] = component
                candidates[f"-component_{j}"] = -component
                candidates[f"1 - component_{j}"] = 1 - component

            print("\n  First five component rows:")
            print(components[:5])

    for key in ["confidence", "ranking_score"]:
        if key in arrays:
            compare_scores(key, arrays[key], candidates)

    print("\n  NOTE: numerical matches do not establish")
    print("  the documented CRUS definition or score direction.")

def inspect_json(path, max_hits=25):
    try:
        data = json.loads(path.read_text(encoding="utf-8"))
    except (OSError, UnicodeError, json.JSONDecodeError):
        return 0

    hits = []

    def walk(value, prefix=""):
        if len(hits) >= max_hits:
            return

        if isinstance(value, dict):
            for key, child in value.items():
                location = f"{prefix}.{key}" if prefix else str(key)

                if any(word in str(key).lower() for word in KEYWORDS):
                    text = json.dumps(child, ensure_ascii=False)
                    hits.append((location, text[:1400]))
                else:
                    walk(child, location)

                if len(hits) >= max_hits:
                    break

        elif isinstance(value, list):
            for index, child in enumerate(value):
                walk(child, f"{prefix}[{index}]")

                if len(hits) >= max_hits:
                    break

    walk(data)

    if hits:
        print(f"\nCONFIG FILE: {path}")
        for location, value in hits:
            print(f"  {location}: {value}")

    return len(hits)

print("=== ARRAY CROSS-CHECK ===")

for name, base in BASES.items():
    print(f"\n========== {name} ==========")

    for split in ["validation", "test"]:
        inspect_arrays(base / f"{split}_Cascade_arrays.npz")

print("\n=== SAVED CONFIGURATION / POLICY SEARCH ===")

total_hits = 0

for name, base in BASES.items():
    if not base.is_dir():
        continue

    for path in sorted(base.rglob("*.json")):
        # Exclude unrelated added domain-baseline configurations.
        if "domain_baselines" in path.parts:
            continue

        total_hits += inspect_json(path)

print("\nConfiguration matches:", total_hits)
print("Read-only diagnostic complete. No files changed.")



In [ ]:
from pathlib import Path
import json
import numpy as np

ROOT = Path(_crus_local_path('project', ''))

# Enable only for your own trusted, locally generated NPZ files.
TRUST_LOCAL_PICKLE = False

RUNS = [
    (
        "primary",
        ROOT / "corrected_experiment_outputs",
        "final_pretest_protocol.json",
    ),
] + [
    (
        f"seed_{seed}",
        ROOT / "supplementary_repeated_splits"
             / f"seed_{seed}" / "experiment_outputs",
        "pretest_protocol.json",
    )
    for seed in [11, 22, 33, 44, 55]
]

def load_fields(path):
    result = {}

    with np.load(path, allow_pickle=False) as saved:
        for key in [
            "components",
            "component_names",
            "confidence",
            "predictions",
            "true_labels",
        ]:
            if key not in saved.files:
                continue

            try:
                result[key] = saved[key]
            except ValueError:
                if not TRUST_LOCAL_PICKLE:
                    print(
                        f"  Skipped object-valued {key}; "
                        "trusted-file permission is disabled."
                    )
                    continue

                with np.load(path, allow_pickle=True) as trusted:
                    result[key] = trusted[key]

    return result

checks = []

for name, base, protocol_name in RUNS:
    print(f"\n========== {name} ==========")

    protocol_path = base / protocol_name

    if not protocol_path.is_file():
        raise FileNotFoundError(protocol_path)

    protocol = json.loads(protocol_path.read_text())

    config = protocol["uncertainty_configuration"]
    weights = np.asarray(config["weights"], dtype=float)
    policy = protocol["frozen_policies"]["Cascade:CRUS"]

    print("Saved uncertainty configuration:", config)
    print("Saved CRUS policy:", policy)

    validation_pass = None

    for split in ["validation", "test"]:
        path = base / f"{split}_Cascade_arrays.npz"

        if not path.is_file():
            raise FileNotFoundError(path)

        fields = load_fields(path)

        components = np.asarray(
            fields["components"], dtype=float
        )
        confidence = np.asarray(
            fields["confidence"], dtype=float
        ).reshape(-1)

        if components.shape != (len(confidence), len(weights)):
            raise ValueError(
                f"{name}/{split}: component dimensions mismatch."
            )

        if not (
            np.isfinite(components).all()
            and np.isfinite(confidence).all()
            and np.isfinite(weights).all()
        ):
            raise ValueError(
                f"{name}/{split}: non-finite inputs."
            )

        if "component_names" in fields:
            print(
                f"\n{split} component names:",
                fields["component_names"].tolist(),
            )

        uncertainty = components @ weights

        accepted = (
            (confidence >= float(policy["tau_c"]))
            & (uncertainty <= float(policy["tau_u"]))
        )

        n_accepted = int(accepted.sum())

        print(
            f"\n{split}: candidate accepted={n_accepted}/"
            f"{len(accepted)}, "
            f"coverage={accepted.mean():.8f}"
        )

        n_correct = None

        if "true_labels" in fields:
            true = fields["true_labels"].astype(str)
            pred = fields["predictions"].astype(str)

            if len(true) != len(accepted) or len(pred) != len(true):
                raise ValueError(
                    f"{name}/{split}: label length mismatch."
                )

            n_correct = int(((true == pred) & accepted).sum())

            accuracy = (
                n_correct / n_accepted
                if n_accepted else np.nan
            )

            print(
                f"{split}: correct accepted={n_correct}, "
                f"selective accuracy={accuracy:.8f}"
            )

        if split == "validation":
            accepted_match = (
                n_accepted == int(policy["n_accepted"])
            )

            correct_match = (
                n_correct == int(policy["n_correct_accepted"])
                if n_correct is not None
                else None
            )

            print(
                "Validation accepted count matches:",
                accepted_match,
            )
            print(
                "Validation correct count matches:",
                correct_match
                if correct_match is not None
                else "NOT CHECKED",
            )

            validation_pass = (
                accepted_match and correct_match
                if correct_match is not None
                else None
            )

    checks.append((name, validation_pass))

print("\n========== VALIDATION RECONSTRUCTION SUMMARY ==========")

for name, passed in checks:
    status = (
        "PASS"
        if passed is True
        else "FAIL"
        if passed is False
        else "INCOMPLETE: reference labels not loaded"
    )
    print(f"{name}: {status}")

print("\nNo files changed.")
print(
    "Matching aggregate counts supports the candidate reconstruction, "
    "but does not alone prove identical per-record acceptance."
)


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

ROOT = Path(_crus_local_path('project', ''))
TRUST_LOCAL_PICKLE = True

RUNS = [
    (
        "primary",
        ROOT / "corrected_experiment_outputs",
        "final_pretest_protocol.json",
    ),
] + [
    (
        f"seed_{seed}",
        ROOT / "supplementary_repeated_splits"
             / f"seed_{seed}" / "experiment_outputs",
        "pretest_protocol.json",
    )
    for seed in [11, 22, 33, 44, 55]
]

def load_field(path, key):
    with np.load(path, allow_pickle=False) as saved:
        try:
            return saved[key]
        except ValueError:
            if not TRUST_LOCAL_PICKLE:
                raise RuntimeError(
                    f"{path}: {key} requires trusted-file permission."
                )

    with np.load(path, allow_pickle=True) as saved:
        return saved[key]

audit_rows = []

for name, base, protocol_name in RUNS:
    protocol = json.loads((base / protocol_name).read_text())
    policy = protocol["frozen_policies"]["Cascade:CRUS"]
    weights = np.asarray(
        protocol["uncertainty_configuration"]["weights"],
        dtype=float,
    )

    for split in ["validation", "test"]:
        path = base / f"{split}_Cascade_arrays.npz"

        components = load_field(path, "components").astype(float)
        confidence = load_field(path, "confidence").astype(float)
        predictions = load_field(path, "predictions").astype(str)
        true_labels = load_field(path, "true_labels").astype(str)

        uncertainty = components @ weights
        accepted = (
            (confidence >= float(policy["tau_c"]))
            & (uncertainty <= float(policy["tau_u"]))
        )

        correct = predictions == true_labels

        row = {
            "run": name,
            "split": split,
            "N": len(true_labels),
            "n_accepted": int(accepted.sum()),
            "n_correct_accepted": int((accepted & correct).sum()),
            "coverage": accepted.mean(),
            "selective_accuracy": (
                correct[accepted].mean()
                if accepted.any() else np.nan
            ),
            "reference_check": "not found",
        }

        if split == "validation":
            assert row["N"] == int(policy["n_validation"])
            assert row["n_accepted"] == int(policy["n_accepted"])
            assert (
                row["n_correct_accepted"]
                == int(policy["n_correct_accepted"])
            )
            row["reference_check"] = "PASS: frozen validation policy"

        else:
            # Search only likely metric-summary files in this run.
            candidates = sorted(set(
                list(base.glob("*metric*summary*.csv"))
                + list(base.glob("metric_summary.csv"))
            ))

            matches = []

            for metric_path in candidates:
                frame = pd.read_csv(metric_path)

                if not {"method", "mode"}.issubset(frame.columns):
                    continue

                selected = frame.loc[
                    frame["method"].eq("Cascade")
                    & frame["mode"].eq("native_CRUS_gate")
                ]

                if "split" in frame.columns:
                    selected = selected.loc[
                        selected["split"].eq("test")
                    ]

                for _, saved_row in selected.iterrows():
                    matches.append((metric_path, saved_row))

            if not matches:
                print(
                    f"\n{name}: no native test metric reference found."
                    "\nAvailable CSV files:"
                )
                for csv_path in sorted(base.glob("*.csv")):
                    print(" ", csv_path.name)

            for metric_path, saved_row in matches:
                for column in [
                    "N", "n_accepted", "n_correct_accepted"
                ]:
                    if column in saved_row.index:
                        assert int(saved_row[column]) == row[column], (
                            f"{name}: mismatch in {column} "
                            f"against {metric_path}"
                        )

                for column in ["coverage", "selective_accuracy"]:
                    if column in saved_row.index:
                        assert np.isclose(
                            float(saved_row[column]), row[column]
                        ), (
                            f"{name}: mismatch in {column} "
                            f"against {metric_path}"
                        )

            if matches:
                row["reference_check"] = (
                    "PASS: saved native test metrics"
                )

        audit_rows.append(row)

audit = pd.DataFrame(audit_rows)

print("\n=== CRUS RECONSTRUCTION AUDIT ===")
print(audit.to_string(
    index=False,
    float_format=lambda value: f"{value:.8f}",
))

print("\nNo files changed.")
print(
    "These checks verify aggregate agreement. "
    "Per-record equivalence requires an independently saved "
    "native acceptance mask or the original implementation."
)

In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path(_crus_local_path('project', ''))
PRIMARY = ROOT / "corrected_experiment_outputs"

FILES = [
    "verified_main_test_results.csv",
    "test_native_gate_operating_points.csv",
    "test_frozen_CRUS_acceptance_audit.csv",
    "test_cascade_frozen_uncertainty_policies.csv",
    "test_cascade_matched_uncertainty_audit.csv",
    "test_cascade_matched_uncertainty_rankings.csv",
    "test_cascade_uncertainty_policy_audit.csv",
    "test_ranking_risk_coverage_summary.csv",
    "test_ranking_risk_coverage_curves.csv",
]

for filename in FILES:
    path = PRIMARY / filename
    print(f"\n{'=' * 75}\n{filename}")

    if not path.is_file():
        print("NOT FOUND")
        continue

    df = pd.read_csv(path)

    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())

    # Print distinct method/score names when available.
    for column in df.columns:
        name = column.lower()

        if any(token in name for token in [
            "method", "score_name", "uncertainty_name",
            "policy", "mode", "ranking_name",
        ]):
            if df[column].nunique(dropna=False) <= 30:
                print(
                    f"{column} values:",
                    df[column].drop_duplicates().tolist()
                )

    if len(df) <= 30:
        print(df.to_string(index=False))
    else:
        print(df.head(8).to_string(index=False))
        print(f"... {len(df) - 8} additional rows")

print("\n=== RELATED SUPPLEMENTARY OUTPUT FILENAMES ===")

for seed in [11, 22, 33, 44, 55]:
    base = (
        ROOT / "supplementary_repeated_splits"
        / f"seed_{seed}" / "experiment_outputs"
    )

    print(f"\nseed_{seed}:")

    matches = [
        path.name
        for path in sorted(base.glob("*.csv"))
        if any(token in path.name.lower() for token in [
            "uncertainty", "risk_coverage", "acceptance",
            "operating", "calibrat", "conformal",
        ])
    ]

    for filename in matches:
        print(" ", filename)

    if not matches:
        print("  No matching CSV filenames found.")

print("\nRead-only inspection complete. No files changed.")

In [ ]:
from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, average_precision_score

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import numpy as np
import pandas as pd

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_recall_curve,
    auc,
    brier_score_loss,
)

ROOT = Path(_crus_local_path('project', ''))
TRUST_LOCAL_PICKLE = True

STAMP = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
OUT = ROOT / f"r1_7_uncertainty_comparison_{STAMP}"

RUNS = [
    (
        "primary",
        ROOT / "corrected_experiment_outputs",
        "final_pretest_protocol.json",
    ),
] + [
    (
        f"seed_{seed}",
        ROOT / "supplementary_repeated_splits"
             / f"seed_{seed}" / "experiment_outputs",
        "pretest_protocol.json",
    )
    for seed in [11, 22, 33, 44, 55]
]

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def load_field(path, key):
    with np.load(path, allow_pickle=False) as saved:
        try:
            return saved[key]
        except ValueError:
            if not TRUST_LOCAL_PICKLE:
                raise RuntimeError(
                    f"Trusted-file permission required: {path}, {key}"
                )

    with np.load(path, allow_pickle=True) as saved:
        return saved[key]

def load_run_arrays(path):
    keys = [
        "record_ids", "true_labels", "predictions",
        "classes", "probabilities", "confidence", "components",
    ]
    a = {key: load_field(path, key) for key in keys}

    for key in ["record_ids", "true_labels", "predictions", "classes"]:
        a[key] = a[key].astype(str)

    for key in ["probabilities", "confidence", "components"]:
        a[key] = a[key].astype(float)

    ids = a["record_ids"]
    p = a["probabilities"]
    n = len(ids)

    assert len(set(ids)) == n, f"Duplicate IDs: {path}"
    assert len(a["true_labels"]) == len(a["predictions"]) == n
    assert a["confidence"].shape == (n,)
    assert p.shape == (n, len(a["classes"]))
    assert a["components"].shape == (n, 3)
    assert np.isfinite(p).all()
    assert np.isfinite(a["components"]).all()
    assert np.isfinite(a["confidence"]).all()
    assert np.all(p >= 0) and np.all(p <= 1)
    assert np.allclose(p.sum(axis=1), 1.0)
    assert np.array_equal(
        a["classes"][p.argmax(axis=1)], a["predictions"]
    )
    assert np.allclose(a["confidence"], p.max(axis=1))

    entropy = -(
        p * np.log(np.clip(p, 1e-300, 1.0))
    ).sum(axis=1) / np.log(p.shape[1])

    ordered = np.sort(p, axis=1)
    margin = ordered[:, -1] - ordered[:, -2]

    assert np.allclose(
        a["components"][:, 0], entropy,
        rtol=1e-7, atol=1e-9,
    )

    a["entropy"] = entropy
    a["margin"] = margin
    a["maximum"] = p.max(axis=1)
    a["correct"] = (
        a["predictions"] == a["true_labels"]
    ).astype(int)
    a["features"] = np.column_stack([
        a["maximum"], entropy, margin
    ])
    return a

def to_bool(series):
    mapped = (
        series.astype(str).str.strip().str.lower()
        .map({
            "true": True, "false": False,
            "1": True, "0": False,
            "1.0": True, "0.0": False,
        })
    )
    if mapped.isna().any():
        raise ValueError("Unrecognized audit boolean values.")
    return mapped.to_numpy(dtype=bool)

def binary_ece(y, confidence, bins=10):
    index = np.minimum(
        (np.asarray(confidence) * bins).astype(int), bins - 1
    )
    total = len(y)
    value = 0.0

    for b in range(bins):
        mask = index == b
        if mask.any():
            value += (
                mask.sum() / total
                * abs(y[mask].mean() - confidence[mask].mean())
            )
    return value

metric_rows = []
risk_rows = []
curve_rows = []
record_rows = []
calibration_rows = []
input_hashes = {}
calibrator_parameters = {}

for run, base, protocol_name in RUNS:
    protocol_path = base / protocol_name
    val_path = base / "validation_Cascade_arrays.npz"
    test_path = base / "test_Cascade_arrays.npz"

    for path in [protocol_path, val_path, test_path]:
        if not path.is_file():
            raise FileNotFoundError(path)
        input_hashes[str(path.relative_to(ROOT))] = sha256(path)

    protocol = json.loads(protocol_path.read_text())
    policy = protocol["frozen_policies"]["Cascade:CRUS"]
    weights = np.asarray(
        protocol["uncertainty_configuration"]["weights"],
        dtype=float,
    )
    assert weights.shape == (3,)
    assert np.isfinite(weights).all()

    val = load_run_arrays(val_path)
    test = load_run_arrays(test_path)

    assert not (
        set(val["record_ids"]) & set(test["record_ids"])
    ), f"{run}: overlapping validation/test IDs"
    assert np.array_equal(val["classes"], test["classes"])

    # Verify the candidate native gate against frozen validation counts.
    val_crus = val["components"] @ weights
    val_accept = (
        (val["confidence"] >= float(policy["tau_c"]))
        & (val_crus <= float(policy["tau_u"]))
    )

    assert len(val_accept) == int(policy["n_validation"])
    assert int(val_accept.sum()) == int(policy["n_accepted"])
    assert int(
        val["correct"][val_accept].sum()
    ) == int(policy["n_correct_accepted"])

    ids = test["record_ids"]
    n = len(ids)
    errors = 1 - test["correct"]
    crus = test["components"] @ weights

    native_accept = (
        (test["confidence"] >= float(policy["tau_c"]))
        & (crus <= float(policy["tau_u"]))
    )
    native_k = int(native_accept.sum())
    assert native_k > 0

    if run == "primary":
        audit_path = base / "test_frozen_CRUS_acceptance_audit.csv"
        audit = pd.read_csv(audit_path, dtype={"record_id": str})
        audit = audit.loc[audit["method"].eq("Cascade")].copy()

        assert not audit["record_id"].duplicated().any()
        assert set(audit["record_id"]) == set(ids)

        audit = audit.set_index("record_id").loc[ids]

        assert np.allclose(
            audit["CRUS"].to_numpy(dtype=float),
            crus, rtol=1e-7, atol=1e-9,
        )
        assert np.array_equal(
            to_bool(audit["accepted"]), native_accept
        )
        assert np.array_equal(
            to_bool(audit["correct"]), test["correct"].astype(bool)
        )

        input_hashes[str(audit_path.relative_to(ROOT))] = sha256(
            audit_path
        )
        print("Primary per-record CRUS audit: PASS")

    # Fixed specification: no test-driven feature or hyperparameter tuning.
    if len(np.unique(val["correct"])) != 2:
        raise ValueError(
            f"{run}: validation correctness has only one class."
        )

    calibrator = make_pipeline(
        StandardScaler(),
        LogisticRegression(
            C=1.0,
            solver="lbfgs",
            max_iter=2000,
            random_state=0,
        ),
    )
    calibrator.fit(val["features"], val["correct"])

    calibrated_correctness = calibrator.predict_proba(
        test["features"]
    )[:, 1]

    scaler = calibrator.named_steps["standardscaler"]
    logistic = calibrator.named_steps["logisticregression"]

    calibrator_parameters[run] = {
        "feature_order": [
            "maximum_probability",
            "normalized_predictive_entropy",
            "probability_margin",
        ],
        "scaler_mean": scaler.mean_.tolist(),
        "scaler_scale": scaler.scale_.tolist(),
        "logistic_coefficient": logistic.coef_.tolist(),
        "logistic_intercept": logistic.intercept_.tolist(),
    }

    for name, q in [
        ("MaximumProbability", test["maximum"]),
        ("CalibratedCorrectness", calibrated_correctness),
    ]:
        calibration_rows.append({
            "run": run,
            "confidence_method": name,
            "correctness_Brier": brier_score_loss(
                test["correct"], q
            ),
            "correctness_ECE_10_equal_width": binary_ece(
                test["correct"], q
            ),
        })

    # Higher values always mean greater uncertainty.
    scores = {
        "CRUS": crus,
        "MaximumProbability": 1 - test["maximum"],
        "PredictiveEntropy": test["entropy"],
        "ProbabilityMargin": 1 - test["margin"],
        "CalibratedCorrectness": 1 - calibrated_correctness,
    }

    targets = [
        (
            f"nominal_{fraction:.2f}",
            max(1, min(n, int(np.ceil(fraction * n)))),
        )
        for fraction in [0.25, 0.50, 0.75, 0.90, 1.00]
    ]
    targets.append(("native_CRUS_count", native_k))

    for score_name, uncertainty in scores.items():
        assert np.isfinite(uncertainty).all()

        order = np.lexsort((ids, uncertainty))
        cumulative_errors = np.cumsum(errors[order])
        retained = np.arange(1, n + 1)
        risks = cumulative_errors / retained

        has_both = len(np.unique(errors)) == 2

        if has_both:
            precision, recall, _ = precision_recall_curve(
                errors, uncertainty
            )
            pr_area = auc(recall, precision)
            error_auroc = roc_auc_score(errors, uncertainty)
            error_ap = average_precision_score(errors, uncertainty)
        else:
            pr_area = error_auroc = error_ap = np.nan

        metric_rows.append({
            "run": run,
            "score": score_name,
            "N": n,
            "n_errors": int(errors.sum()),
            "AURC": risks.mean(),
            "error_AUROC": error_auroc,
            "error_AP": error_ap,
            "error_AUPRC_trapezoidal": pr_area,
            "error_prevalence": errors.mean(),
        })

        matched_mask = np.zeros(n, dtype=bool)
        matched_mask[order[:native_k]] = True

        for target, k in targets:
            risk_rows.append({
                "run": run,
                "score": score_name,
                "target": target,
                "n_accepted": k,
                "coverage": k / n,
                "n_errors_accepted": int(cumulative_errors[k - 1]),
                "risk": risks[k - 1],
                "selective_accuracy": 1 - risks[k - 1],
            })

        for k, count, risk in zip(
            retained, cumulative_errors, risks
        ):
            curve_rows.append({
                "run": run,
                "score": score_name,
                "n_accepted": int(k),
                "coverage": k / n,
                "n_errors_accepted": int(count),
                "risk": float(risk),
            })

        for i, record_id in enumerate(ids):
            record_rows.append({
                "run": run,
                "record_id": record_id,
                "score": score_name,
                "uncertainty": float(uncertainty[i]),
                "correct": bool(test["correct"][i]),
                "accepted_at_native_count": bool(matched_mask[i]),
                "native_CRUS_accepted": bool(native_accept[i]),
            })

    native_errors = int(errors[native_accept].sum())

    risk_rows.append({
        "run": run,
        "score": "Native_CRUS_gate",
        "target": "native_CRUS_count",
        "n_accepted": native_k,
        "coverage": native_k / n,
        "n_errors_accepted": native_errors,
        "risk": native_errors / native_k,
        "selective_accuracy": 1 - native_errors / native_k,
    })

    print(f"{run}: validation reconstruction PASS; analysis complete")

metrics = pd.DataFrame(metric_rows)
risk_table = pd.DataFrame(risk_rows)
curves = pd.DataFrame(curve_rows)
records = pd.DataFrame(record_rows)
calibration = pd.DataFrame(calibration_rows)

def repeat_summary(frame, group_columns, value_columns):
    result = (
        frame.loc[frame["run"].ne("primary")]
        .groupby(group_columns)[value_columns]
        .agg(["mean", "std"])
    )
    result.columns = [
        f"{metric}_{stat}" for metric, stat in result.columns
    ]
    return result.reset_index()

metric_summary = repeat_summary(
    metrics,
    ["score"],
    ["AURC", "error_AUROC", "error_AP", "error_AUPRC_trapezoidal"],
)

risk_summary = repeat_summary(
    risk_table,
    ["score", "target"],
    ["coverage", "risk", "selective_accuracy"],
)

calibration_summary = repeat_summary(
    calibration,
    ["confidence_method"],
    ["correctness_Brier", "correctness_ECE_10_equal_width"],
)

# Descriptive paired differences; no significance claim.
paired_rows = []

for run in metrics["run"].unique():
    subset = metrics.loc[metrics["run"].eq(run)].set_index("score")
    for comparator in scores:
        if comparator == "CRUS":
            continue
        for metric in [
            "AURC", "error_AUROC", "error_AP",
            "error_AUPRC_trapezoidal",
        ]:
            paired_rows.append({
                "run": run,
                "comparison": f"CRUS minus {comparator}",
                "metric": metric,
                "difference": (
                    subset.loc["CRUS", metric]
                    - subset.loc[comparator, metric]
                ),
            })

paired = pd.DataFrame(paired_rows)

OUT.mkdir(parents=True, exist_ok=False)

outputs = {
    "score_metrics_by_run.csv": metrics,
    "fixed_coverage_risk_by_run.csv": risk_table,
    "risk_coverage_curves.csv": curves,
    "per_record_score_and_selection_audit.csv": records,
    "five_repeat_score_summary.csv": metric_summary,
    "five_repeat_risk_summary.csv": risk_summary,
    "correctness_calibration_by_run.csv": calibration,
    "five_repeat_calibration_summary.csv": calibration_summary,
    "descriptive_paired_metric_differences.csv": paired,
}

for filename, frame in outputs.items():
    frame.to_csv(OUT / filename, index=False)

metadata = {
    "classifier": "Cascade; saved predictions fixed",
    "CRUS": "components @ weights from each saved protocol",
    "ranking": "uncertainty ascending; record ID ascending for ties",
    "AURC": "mean prefix risk for retained counts k=1,...,N",
    "error_positive_class": "incorrect prediction",
    "PR_metrics": [
        "average precision",
        "trapezoidal area under precision-recall curve",
    ],
    "nominal_coverage_rounding": "ceil(fraction * N)",
    "repeat_sd": "sample standard deviation, ddof=1",
    "calibrated_comparator": {
        "target": "correctness of fixed Cascade prediction",
        "fit_data": "validation only",
        "model": "StandardScaler + LogisticRegression",
        "C": 1.0,
        "test_tuning": False,
        "guaranteed_calibration": False,
        "conformal_method": False,
    },
    "confidence_intervals": "not computed in this script",
    "calibrator_parameters": calibrator_parameters,
    "input_hashes": input_hashes,
}

(OUT / "analysis_metadata.json").write_text(
    json.dumps(metadata, indent=2, sort_keys=True)
)

print("\n=== PRIMARY SCORE METRICS ===")
print(metrics.loc[
    metrics["run"].eq("primary")
].to_string(index=False, float_format=lambda x: f"{x:.6f}"))

print("\n=== PRIMARY RISK AT NATIVE CRUS RETAINED COUNT ===")
print(risk_table.loc[
    risk_table["run"].eq("primary")
    & risk_table["target"].eq("native_CRUS_count")
].to_string(index=False, float_format=lambda x: f"{x:.6f}"))

print("\n=== FIVE-REPEAT SCORE SUMMARY ===")
print(metric_summary.to_string(
    index=False, float_format=lambda x: f"{x:.6f}"
))

print("\n=== FIVE-REPEAT MATCHED-COUNT RISK SUMMARY ===")
print(risk_summary.loc[
    risk_summary["target"].eq("native_CRUS_count")
].to_string(index=False, float_format=lambda x: f"{x:.6f}"))

print("\n=== TEST CORRECTNESS-CALIBRATION DIAGNOSTICS ===")
print(calibration.to_string(
    index=False, float_format=lambda x: f"{x:.6f}"
))

print("\nNew analysis folder:", OUT)
print("Existing experiment outputs were not modified.")

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import numpy as np
import pandas as pd

ROOT = Path(_crus_local_path('project', ''))
TRUST_LOCAL_PICKLE = True

STAMP = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
OUT = ROOT / f"r1_8_validation_weight_search_{STAMP}"

RUNS = [
    (
        "primary",
        ROOT / "corrected_experiment_outputs",
        "final_pretest_protocol.json",
    ),
] + [
    (
        f"seed_{seed}",
        ROOT / "supplementary_repeated_splits"
             / f"seed_{seed}" / "experiment_outputs",
        "pretest_protocol.json",
    )
    for seed in [11, 22, 33, 44, 55]
]

MIN_COVERAGE = 0.5
CONFIDENCE_POINTS = 21
UNCERTAINTY_POINTS = 101

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def load_field(path, key):
    with np.load(path, allow_pickle=False) as saved:
        try:
            return saved[key]
        except ValueError:
            if not TRUST_LOCAL_PICKLE:
                raise RuntimeError(
                    f"Trusted-file permission required: {path}, {key}"
                )

    with np.load(path, allow_pickle=True) as saved:
        return saved[key]

def select_gate(confidence, uncertainty, correct):
    n = len(correct)
    minimum_n = int(np.ceil(MIN_COVERAGE * n))

    confidence_thresholds = np.unique(np.r_[
        0.0,
        np.quantile(
            confidence,
            np.linspace(0, 1, CONFIDENCE_POINTS),
            method="linear",
        ),
    ])

    uncertainty_thresholds = np.unique(
        np.quantile(
            uncertainty,
            np.linspace(0, 1, UNCERTAINTY_POINTS),
            method="linear",
        )
    )

    best = None
    best_key = None

    for tau_c in confidence_thresholds:
        accepted = (
            (confidence >= tau_c)[None, :]
            & (
                uncertainty[None, :]
                <= uncertainty_thresholds[:, None]
            )
        )

        counts = accepted.sum(axis=1)
        correct_counts = (accepted * correct[None, :]).sum(axis=1)

        for j in np.flatnonzero(counts >= minimum_n):
            accuracy = float(correct_counts[j] / counts[j])
            tau_u = float(uncertainty_thresholds[j])

            # Same stated preference order as the frozen protocol.
            key = (
                accuracy,
                int(counts[j]),
                -float(tau_c),
                -tau_u,
            )

            if best_key is None or key > best_key:
                best_key = key
                best = {
                    "tau_c": float(tau_c),
                    "tau_u": tau_u,
                    "n_validation": n,
                    "n_accepted": int(counts[j]),
                    "n_correct_accepted": int(correct_counts[j]),
                    "coverage": float(counts[j] / n),
                    "selective_accuracy": accuracy,
                }

    if best is None:
        raise RuntimeError("No feasible validation gate found.")

    return best

grid = [
    (i / 10, j / 10, (10 - i - j) / 10)
    for i in range(11)
    for j in range(11 - i)
]

assert len(grid) == 66

candidates = [
    (f"grid_{i}_{j}_{10-i-j}", weights)
    for weights in grid
    for i, j in [(
        int(round(weights[0] * 10)),
        int(round(weights[1] * 10)),
    )]
]
candidates.append(("equal_weights", (1/3, 1/3, 1/3)))

all_rows = []
frozen = {}
input_hashes = {}

for run, base, protocol_name in RUNS:
    protocol_path = base / protocol_name
    validation_path = base / "validation_Cascade_arrays.npz"

    protocol = json.loads(protocol_path.read_text())
    original_policy = protocol["frozen_policies"]["Cascade:CRUS"]
    original_weights = np.asarray(
        protocol["uncertainty_configuration"]["weights"],
        dtype=float,
    )

    input_hashes[str(protocol_path.relative_to(ROOT))] = sha256(
        protocol_path
    )
    input_hashes[str(validation_path.relative_to(ROOT))] = sha256(
        validation_path
    )

    components = load_field(
        validation_path, "components"
    ).astype(float)
    confidence = load_field(
        validation_path, "confidence"
    ).astype(float)
    true = load_field(
        validation_path, "true_labels"
    ).astype(str)
    pred = load_field(
        validation_path, "predictions"
    ).astype(str)
    correct = (true == pred).astype(int)

    assert components.shape == (len(true), 3)
    assert confidence.shape == (len(true),)
    assert np.isfinite(components).all()
    assert np.isfinite(confidence).all()

    # Verify the original frozen gate directly.
    original_uncertainty = components @ original_weights
    original_mask = (
        (confidence >= float(original_policy["tau_c"]))
        & (original_uncertainty <= float(original_policy["tau_u"]))
    )

    assert int(original_mask.sum()) == int(
        original_policy["n_accepted"]
    )
    assert int(correct[original_mask].sum()) == int(
        original_policy["n_correct_accepted"]
    )

    # Check whether this quantile search recovers the saved outcome.
    recovered = select_gate(
        confidence, original_uncertainty, correct
    )

    if (
        recovered["n_accepted"] != int(original_policy["n_accepted"])
        or recovered["n_correct_accepted"]
        != int(original_policy["n_correct_accepted"])
    ):
        raise RuntimeError(
            f"{run}: gate-search reproduction failed.\n"
            f"Recovered: {recovered}\n"
            f"Saved: {original_policy}\n"
            "Do not continue until the original threshold "
            "candidate-generation procedure is verified."
        )

    rows = []

    for name, weights in candidates:
        weights = np.asarray(weights, dtype=float)
        uncertainty = components @ weights
        policy = select_gate(confidence, uncertainty, correct)

        row = {
            "run": run,
            "candidate": name,
            "w_H_pred": float(weights[0]),
            "w_H_retrieval": float(weights[1]),
            "w_V_RRF": float(weights[2]),
            "is_original_weights": bool(
                np.allclose(weights, original_weights)
            ),
            **policy,
        }
        rows.append(row)
        all_rows.append(row)

    # Prespecified tie-breaking for weight selection:
    # gate objective, then proximity to original weights,
    # then lexicographic weight order.
    def weight_key(row):
        weights = np.array([
            row["w_H_pred"],
            row["w_H_retrieval"],
            row["w_V_RRF"],
        ])
        return (
            -row["selective_accuracy"],
            -row["coverage"],
            float(np.abs(weights - original_weights).sum()),
            row["w_H_pred"],
            row["w_H_retrieval"],
            row["w_V_RRF"],
        )

    best = min(rows, key=weight_key)
    equal = next(
        row for row in rows
        if row["candidate"] == "equal_weights"
    )

    frozen[run] = {
        "original_frozen": {
            "weights": original_weights.tolist(),
            "policy": original_policy,
        },
        "equal_validation_selected": {
            "weights": [1/3, 1/3, 1/3],
            "policy": {
                key: equal[key] for key in recovered
            },
        },
        "simplex_validation_selected": {
            "weights": [
                best["w_H_pred"],
                best["w_H_retrieval"],
                best["w_V_RRF"],
            ],
            "policy": {
                key: best[key] for key in recovered
            },
        },
    }

    print(f"\n=== {run}: VALIDATION ONLY ===")
    print("Original gate reproduction: PASS")
    print(
        "Selected weights:",
        frozen[run]["simplex_validation_selected"]["weights"],
    )
    print(
        pd.DataFrame([
            row for row in rows
            if row["is_original_weights"]
            or row["candidate"] == "equal_weights"
            or row["candidate"] == best["candidate"]
        ]).to_string(
            index=False,
            float_format=lambda value: f"{value:.6f}",
        )
    )

results = pd.DataFrame(all_rows)

OUT.mkdir(parents=True, exist_ok=False)
results.to_csv(
    OUT / "validation_simplex_sensitivity.csv", index=False
)

(OUT / "frozen_weight_policies.json").write_text(
    json.dumps(frozen, indent=2, sort_keys=True)
)

(OUT / "analysis_metadata.json").write_text(json.dumps({
    "data_used": "validation arrays only; no test arrays loaded",
    "grid_step": 0.1,
    "simplex_grid_candidates": 66,
    "additional_candidate": "equal weighting",
    "minimum_validation_coverage": MIN_COVERAGE,
    "confidence_quantile_points": CONFIDENCE_POINTS,
    "uncertainty_quantile_points": UNCERTAINTY_POINTS,
    "quantile_method": "linear",
    "confidence_candidates_include_zero": True,
    "gate_objective": (
        "maximize accuracy; maximize coverage; "
        "smaller tau_c; smaller tau_u"
    ),
    "weight_ties": (
        "accuracy; coverage; L1 proximity to original weights; "
        "lexicographic weights"
    ),
    "selection_status": (
        "revision-stage validation selection; "
        "not part of the original pretest protocol"
    ),
    "input_hashes": input_hashes,
}, indent=2, sort_keys=True))

print("\nSaved validation-only results:", OUT)
print("No test arrays loaded. Existing files unchanged.")

In [ ]:
from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd

ROOT = Path(_crus_local_path('project', ''))

SEARCH = ROOT / (
    "r1_8_validation_weight_search_20261016T104727378070Z"
)
OUT = SEARCH / "frozen_test_evaluation"

TRUST_LOCAL_PICKLE = True

if OUT.exists() and any(OUT.iterdir()):
    raise RuntimeError(
        f"Existing test evaluation preserved: {OUT}"
    )

frozen_path = SEARCH / "frozen_weight_policies.json"
metadata_path = SEARCH / "analysis_metadata.json"

frozen = json.loads(frozen_path.read_text())
search_metadata = json.loads(metadata_path.read_text())

RUNS = [
    (
        "primary",
        ROOT / "corrected_experiment_outputs",
    ),
] + [
    (
        f"seed_{seed}",
        ROOT / "supplementary_repeated_splits"
             / f"seed_{seed}" / "experiment_outputs",
    )
    for seed in [11, 22, 33, 44, 55]
]

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def load_field(path, key):
    with np.load(path, allow_pickle=False) as saved:
        try:
            return saved[key]
        except ValueError:
            if not TRUST_LOCAL_PICKLE:
                raise RuntimeError(
                    f"Trusted-file permission required: {path}, {key}"
                )

    with np.load(path, allow_pickle=True) as saved:
        return saved[key]

# Confirm the validation inputs have not changed since selection.
for relative_path, expected_hash in (
    search_metadata["input_hashes"].items()
):
    path = ROOT / relative_path
    if sha256(path) != expected_hash:
        raise RuntimeError(
            f"Validation-search input changed: {path}"
        )

rows = []
record_rows = []
test_hashes = {}

for run, base in RUNS:
    path = base / "test_Cascade_arrays.npz"
    test_hashes[str(path.relative_to(ROOT))] = sha256(path)

    ids = load_field(path, "record_ids").astype(str)
    true = load_field(path, "true_labels").astype(str)
    pred = load_field(path, "predictions").astype(str)
    confidence = load_field(path, "confidence").astype(float)
    components = load_field(path, "components").astype(float)

    n = len(ids)
    assert len(set(ids)) == n
    assert len(true) == len(pred) == len(confidence) == n
    assert components.shape == (n, 3)
    assert np.isfinite(components).all()
    assert np.isfinite(confidence).all()

    errors = (pred != true).astype(int)

    original = frozen[run]["original_frozen"]
    original_uncertainty = (
        components @ np.asarray(original["weights"], dtype=float)
    )
    original_policy = original["policy"]

    original_mask = (
        (confidence >= float(original_policy["tau_c"]))
        & (
            original_uncertainty
            <= float(original_policy["tau_u"])
        )
    )

    target_k = int(original_mask.sum())
    if target_k == 0:
        raise RuntimeError(f"{run}: original gate accepts no records.")

    for configuration, specification in frozen[run].items():
        weights = np.asarray(
            specification["weights"], dtype=float
        )
        policy = specification["policy"]

        assert np.all(weights >= 0)
        assert np.isclose(weights.sum(), 1.0)

        uncertainty = components @ weights

        gate_mask = (
            (confidence >= float(policy["tau_c"]))
            & (uncertainty <= float(policy["tau_u"]))
        )

        gate_n = int(gate_mask.sum())
        gate_errors = int(errors[gate_mask].sum())

        order = np.lexsort((ids, uncertainty))
        prefix_errors = np.cumsum(errors[order])
        prefix_risk = prefix_errors / np.arange(1, n + 1)

        matched_mask = np.zeros(n, dtype=bool)
        matched_mask[order[:target_k]] = True

        matched_errors = int(errors[matched_mask].sum())

        rows.append({
            "run": run,
            "configuration": configuration,
            "w_H_pred": weights[0],
            "w_H_retrieval": weights[1],
            "w_V_RRF": weights[2],
            "tau_c": float(policy["tau_c"]),
            "tau_u": float(policy["tau_u"]),
            "validation_coverage": float(policy["coverage"]),
            "validation_selective_accuracy": float(
                policy["selective_accuracy"]
            ),
            "test_N": n,
            "gate_n_accepted": gate_n,
            "gate_n_errors": gate_errors,
            "gate_coverage": gate_n / n,
            "gate_risk": (
                gate_errors / gate_n if gate_n else np.nan
            ),
            "gate_selective_accuracy": (
                1 - gate_errors / gate_n if gate_n else np.nan
            ),
            "ranking_AURC": prefix_risk.mean(),
            "matched_n_accepted": target_k,
            "matched_coverage": target_k / n,
            "matched_n_errors": matched_errors,
            "matched_risk": matched_errors / target_k,
            "matched_selective_accuracy": (
                1 - matched_errors / target_k
            ),
        })

        for i, record_id in enumerate(ids):
            record_rows.append({
                "run": run,
                "record_id": record_id,
                "configuration": configuration,
                "uncertainty": float(uncertainty[i]),
                "correct": bool(errors[i] == 0),
                "gate_accepted": bool(gate_mask[i]),
                "matched_accepted": bool(matched_mask[i]),
            })

    print(f"{run}: frozen test evaluation complete")

results = pd.DataFrame(rows)
records = pd.DataFrame(record_rows)

summary_columns = [
    "gate_coverage",
    "gate_risk",
    "gate_selective_accuracy",
    "ranking_AURC",
    "matched_coverage",
    "matched_risk",
    "matched_selective_accuracy",
]

summary = (
    results.loc[results["run"].ne("primary")]
    .groupby("configuration")[summary_columns]
    .agg(["mean", "std"])
)
summary.columns = [
    f"{metric}_{stat}"
    for metric, stat in summary.columns
]
summary = summary.reset_index()

OUT.mkdir(parents=True, exist_ok=True)

results.to_csv(
    OUT / "frozen_weight_test_results.csv", index=False
)
summary.to_csv(
    OUT / "five_repeat_weight_summary.csv", index=False
)
records.to_csv(
    OUT / "per_record_weight_selection_audit.csv", index=False
)

(OUT / "evaluation_metadata.json").write_text(json.dumps({
    "frozen_weight_policies_sha256": sha256(frozen_path),
    "validation_search_metadata_sha256": sha256(metadata_path),
    "test_input_hashes": test_hashes,
    "selection": "weights and thresholds frozen before this evaluation",
    "matched_count": (
        "original frozen CRUS gate's test retained count "
        "within each split"
    ),
    "ranking": "uncertainty ascending; record ID ascending for ties",
    "AURC": "mean prefix risk over k=1,...,N",
    "repeat_sd": "sample standard deviation, ddof=1",
    "confidence_intervals": "not computed",
    "original_results": "preserved; sensitivity analysis only",
}, indent=2, sort_keys=True))

print("\n=== PRIMARY FROZEN-WEIGHT TEST RESULTS ===")
print(results.loc[
    results["run"].eq("primary")
].to_string(
    index=False,
    float_format=lambda value: f"{value:.6f}",
))

print("\n=== FIVE-REPEAT WEIGHT SUMMARY ===")
print(summary.to_string(
    index=False,
    float_format=lambda value: f"{value:.6f}",
))

print("\nOutputs:", OUT)
print("No weights or thresholds were selected using test outcomes.")
print("Existing experiment outputs were not modified.")

In [ ]:
from pathlib import Path
import pandas as pd
from sklearn.metrics import roc_auc_score, average_precision_score

BASE = Path(
    _crus_local_path('project', 'r1_8_validation_weight_search_20261016T104727378070Z/frozen_test_evaluation')
)

df = pd.read_csv(
    BASE / "per_record_weight_selection_audit.csv",
    dtype={"record_id": str},
)

correct = (
    df["correct"].astype(str).str.strip().str.lower()
    .map({
        "true": True, "false": False,
        "1": True, "0": False,
        "1.0": True, "0.0": False,
    })
)

if correct.isna().any():
    raise ValueError("Unrecognized values in the correct column.")

df["error"] = (~correct.astype(bool)).astype(int)

rows = []

for (run, configuration), group in df.groupby(
    ["run", "configuration"]
):
    if group["record_id"].duplicated().any():
        raise ValueError(f"Duplicate IDs: {run}, {configuration}")

    if group["error"].nunique() != 2:
        raise ValueError(f"Only one outcome class: {run}")

    rows.append({
        "run": run,
        "configuration": configuration,
        "AUROC": roc_auc_score(
            group["error"], group["uncertainty"]
        ),
        "AP": average_precision_score(
            group["error"], group["uncertainty"]
        ),
    })

metrics = pd.DataFrame(rows)

configurations = [
    "equal_validation_selected",
    "simplex_validation_selected",
]

print("=== PANEL A: PRIMARY CELLS ===")

for configuration in configurations:
    selected = metrics.loc[
        metrics["run"].eq("primary")
        & metrics["configuration"].eq(configuration)
    ]
    assert len(selected) == 1
    row = selected.iloc[0]

    print(f"\n{configuration}")
    print(f"AUROC: {row['AUROC']:.4f}")
    print(f"AP:    {row['AP']:.4f}")

print("\n=== PANEL B: FIVE-REPEAT CELLS ===")

for configuration in configurations:
    selected = metrics.loc[
        metrics["run"].ne("primary")
        & metrics["configuration"].eq(configuration)
    ]
    assert len(selected) == 5

    print(f"\n{configuration}")

    for metric in ["AUROC", "AP"]:
        mean = selected[metric].mean()
        sd = selected[metric].std(ddof=1)
        print(
            f"{metric}: "
            + rf"\({mean:.4f} \pm {sd:.4f}\)"
        )

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

ROOT = Path(_crus_local_path('project', ''))

BASES = [
    ROOT / "corrected_experiment_outputs",
    ROOT / "supplementary_repeated_splits"
         / "seed_11" / "experiment_outputs",
]

TOKENS = (
    "retriev", "candidate", "stage", "trace",
    "rank", "rrf", "bm25", "sbert", "cascade",
)

def inspect_file(path):
    print(f"\n--- {path} ---")

    try:
        suffix = path.suffix.lower()

        if suffix == ".csv":
            preview = pd.read_csv(path, nrows=3)
            print("Columns:", preview.columns.tolist())

            # Keep diagnostic output manageable.
            print(
                preview.to_string(index=False)[:5000]
            )

        elif suffix == ".npz":
            with np.load(path, allow_pickle=False) as saved:
                print("Fields:", saved.files)

                for key in saved.files:
                    try:
                        array = saved[key]
                        print(
                            f"  {key}: "
                            f"shape={array.shape}, "
                            f"dtype={array.dtype}"
                        )
                    except ValueError:
                        print(
                            f"  {key}: object-valued; "
                            "not loaded"
                        )

        elif suffix == ".jsonl":
            with path.open(encoding="utf-8") as handle:
                for line in handle:
                    if not line.strip():
                        continue

                    record = json.loads(line)
                    if isinstance(record, dict):
                        print("First-record keys:", list(record))
                    else:
                        print("First-record type:", type(record).__name__)

                    print(
                        json.dumps(
                            record, ensure_ascii=False
                        )[:5000]
                    )
                    break

        elif suffix == ".parquet":
            import pyarrow.parquet as pq
            parquet = pq.ParquetFile(path)
            print("Columns:", parquet.schema.names)
            print("Rows:", parquet.metadata.num_rows)

        elif suffix == ".json":
            data = json.loads(path.read_text(encoding="utf-8"))

            if isinstance(data, dict):
                print("Top-level keys:", list(data)[:60])

                for key, value in data.items():
                    if any(token in key.lower() for token in [
                        "retriev", "candidate", "stage", "top",
                        "rrf", "bm25", "sbert", "cascade",
                    ]):
                        print(
                            f"{key}: "
                            + json.dumps(
                                value, ensure_ascii=False
                            )[:3000]
                        )
            else:
                print("JSON type:", type(data).__name__)

    except Exception as exc:
        print(f"Inspection error: {type(exc).__name__}: {exc}")

for base in BASES:
    print(f"\n{'=' * 80}")
    print("BASE:", base)

    if not base.is_dir():
        print("DIRECTORY NOT FOUND")
        continue

    candidates = sorted(
        path
        for path in base.rglob("*")
        if path.is_file()
        and path.suffix.lower() in {
            ".csv", ".json", ".jsonl", ".npz", ".parquet"
        }
        and "domain_baselines" not in path.parts
        and any(token in path.name.lower() for token in TOKENS)
    )

    print("\n=== LIKELY ARTIFACTS ===")
    for path in candidates:
        print(path.relative_to(base))

    # Prioritize files likely to contain intermediate candidates.
    priority = [
        path for path in candidates
        if any(token in path.name.lower() for token in [
            "trace", "candidate", "retriev", "stage"
        ])
    ]

    print("\n=== TRACE / CANDIDATE SCHEMAS ===")
    for path in priority[:15]:
        inspect_file(path)

    print("\n=== PROTOCOL CONFIGURATION ===")
    for filename in [
        "final_pretest_protocol.json",
        "pretest_protocol.json",
    ]:
        path = base / filename
        if path.is_file():
            inspect_file(path)

print("\nRead-only discovery complete. No files changed.")

In [ ]:
from pathlib import Path
pass  # Colab service not required locally.

MYDRIVE = Path(_crus_local_path('drive', ''))

if not MYDRIVE.is_dir():
    _crus_prepare_saved_workspace()

ROOT = MYDRIVE / "RiskAware_Revision2"

print("MyDrive accessible:", MYDRIVE.is_dir())
print("Project folder exists:", ROOT.is_dir())

if not ROOT.is_dir():
    print("\nFolders visible in MyDrive:")
    if MYDRIVE.is_dir():
        for path in sorted(MYDRIVE.iterdir()):
            if path.is_dir():
                print(" ", path.name)

    raise FileNotFoundError(
        "RiskAware_Revision2 is not visible. "
        "Check that the mounted Google account is the one "
        "used for the experiments."
    )

expected = [
    ROOT / "corrected_experiment_outputs",
    ROOT / "supplementary_repeated_splits"
         / "seed_11" / "experiment_outputs",
]

print("\n=== EXPECTED OUTPUT DIRECTORIES ===")

for path in expected:
    print(f"{path}: exists={path.is_dir()}")

print("\n=== PROJECT TOP-LEVEL FOLDERS ===")

for path in sorted(ROOT.iterdir()):
    if path.is_dir():
        print(" ", path.name)

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

MYDRIVE = Path(_crus_local_path('drive', ''))

if not MYDRIVE.is_dir():
    pass  # Colab service not required locally.
    _crus_prepare_saved_workspace()

ROOT = MYDRIVE / "RiskAware_Revision2"

if not ROOT.is_dir():
    raise FileNotFoundError(
        "Project folder not found. Check the mounted Google account."
    )

BASES = [
    ROOT / "corrected_experiment_outputs",
    ROOT / "supplementary_repeated_splits"
         / "seed_11" / "experiment_outputs",
]

FILE_TOKENS = (
    "retriev", "candidate", "stage", "trace",
    "rank", "rrf", "bm25", "sbert", "cascade",
)

CONFIG_TOKENS = (
    "retriev", "candidate", "stage", "top_k", "topk",
    "rrf", "bm25", "sbert", "cascade", "rerank",
    "cross_encoder", "crossencoder",
)

EXTENSIONS = {".csv", ".json", ".jsonl", ".npz", ".parquet"}

MAX_PREVIEW_FILES = 20
MAX_CONFIG_HITS = 35
MAX_TEXT_LENGTH = 2200

def short_text(value, limit=MAX_TEXT_LENGTH):
    try:
        text = json.dumps(value, ensure_ascii=False)
    except (TypeError, ValueError):
        text = str(value)
    return text[:limit] + (" ..." if len(text) > limit else "")

def inspect_artifact(path):
    print(f"\n--- {path.name} ---")

    try:
        suffix = path.suffix.lower()

        if suffix == ".csv":
            frame = pd.read_csv(path, nrows=3)
            print("Columns:", frame.columns.tolist())
            with pd.option_context("display.max_colwidth", 160):
                print(frame.to_string(index=False)[:4500])

        elif suffix == ".npz":
            with np.load(path, allow_pickle=False) as saved:
                print("Fields:", saved.files)

                for key in saved.files:
                    try:
                        array = saved[key]
                        print(
                            f"  {key}: shape={array.shape}, "
                            f"dtype={array.dtype}"
                        )

                        if (
                            any(token in key.lower() for token in [
                                "candidate", "retriev", "rank",
                                "stage", "neighbor", "neighbour",
                            ])
                            and array.size <= 30
                        ):
                            print("   ", array)

                    except ValueError:
                        print(
                            f"  {key}: object-valued; "
                            "not loaded with pickle disabled"
                        )

        elif suffix == ".jsonl":
            with path.open(encoding="utf-8") as handle:
                for line in handle:
                    if not line.strip():
                        continue

                    record = json.loads(line)
                    print(
                        "First-record keys:",
                        list(record)
                        if isinstance(record, dict)
                        else type(record).__name__,
                    )
                    print(short_text(record))
                    break

        elif suffix == ".json":
            data = json.loads(path.read_text(encoding="utf-8"))

            if isinstance(data, dict):
                print("Top-level keys:", list(data)[:50])
            elif isinstance(data, list):
                print("List length:", len(data))
                if data:
                    print("First item:", short_text(data[0]))
            else:
                print(short_text(data))

        elif suffix == ".parquet":
            try:
                import pyarrow.parquet as pq
                parquet = pq.ParquetFile(path)
                print("Columns:", parquet.schema.names)
                print("Rows:", parquet.metadata.num_rows)
            except ImportError:
                print(
                    "Parquet file found; pyarrow unavailable. "
                    "No package installed by this script."
                )

    except Exception as exc:
        print(f"Inspection error: {type(exc).__name__}: {exc}")

def inspect_configuration(path):
    try:
        data = json.loads(path.read_text(encoding="utf-8"))
    except (OSError, UnicodeError, json.JSONDecodeError):
        return

    hits = []

    def walk(value, prefix=""):
        if len(hits) >= MAX_CONFIG_HITS:
            return

        if isinstance(value, dict):
            for key, child in value.items():
                location = f"{prefix}.{key}" if prefix else str(key)

                if any(
                    token in str(key).lower()
                    for token in CONFIG_TOKENS
                ):
                    hits.append((location, short_text(child)))

                # Also inspect nested settings, even if parent matched.
                if isinstance(child, (dict, list)):
                    walk(child, location)

                if len(hits) >= MAX_CONFIG_HITS:
                    break

        elif isinstance(value, list):
            for index, child in enumerate(value):
                if isinstance(child, (dict, list)):
                    walk(child, f"{prefix}[{index}]")

                if len(hits) >= MAX_CONFIG_HITS:
                    break

    walk(data)

    if hits:
        print(f"\nCONFIG: {path.name}")
        for location, value in hits:
            print(f"  {location}: {value}")

for base in BASES:
    print("\n" + "=" * 80)
    print("BASE:", base)

    if not base.is_dir():
        print("DIRECTORY NOT FOUND")
        continue

    files = sorted(
        path
        for path in base.rglob("*")
        if path.is_file()
        and path.suffix.lower() in EXTENSIONS
        and "domain_baselines" not in path.parts
    )

    likely = [
        path for path in files
        if any(
            token in path.name.lower()
            for token in FILE_TOKENS
        )
    ]

    print("\n=== LIKELY ARTIFACTS ===")
    if not likely:
        print("No matching filenames.")
    for path in likely:
        print(path.relative_to(base))

    # Trace-like files first, followed by prediction/array files.
    priority = sorted(
        likely,
        key=lambda path: (
            not any(
                token in path.name.lower()
                for token in [
                    "trace", "candidate", "retriev", "stage"
                ]
            ),
            str(path),
        ),
    )

    print("\n=== TRACE / CANDIDATE SCHEMAS ===")
    for path in priority[:MAX_PREVIEW_FILES]:
        inspect_artifact(path)

    if len(priority) > MAX_PREVIEW_FILES:
        print(
            f"\nPreview limited to {MAX_PREVIEW_FILES} files; "
            "all matching filenames are listed above."
        )

    print("\n=== PROTOCOL CONFIGURATION ===")
    for filename in [
        "final_pretest_protocol.json",
        "pretest_protocol.json",
    ]:
        path = base / filename
        if path.is_file():
            inspect_configuration(path)

    # Inspect other likely retrieval configurations.
    for path in files:
        if (
            path.suffix.lower() == ".json"
            and path.name not in {
                "final_pretest_protocol.json",
                "pretest_protocol.json",
            }
            and any(
                token in path.name.lower()
                for token in [
                    "config", "retriev", "cascade", "stage"
                ]
            )
        ):
            inspect_configuration(path)

print("\nRead-only discovery complete. No files changed.")

In [ ]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path(_crus_local_path('project', ''))

BASES = [
    ROOT / "corrected_experiment_outputs",
    ROOT / "supplementary_repeated_splits"
         / "seed_11" / "experiment_outputs",
]

TRACE_TOKENS = (
    "retrieved", "shortlist", "candidate",
    "stage_a", "stage_b", "stagea", "stageb",
    "bm25_ids", "sbert_ids", "rrf_ids",
    "document_ids", "doc_ids",
)

def preview_value(value, limit=3500):
    text = str(value)
    return text[:limit] + (" ..." if len(text) > limit else "")

for base in BASES:
    print("\n" + "=" * 80)
    print("BASE:", base)

    if not base.is_dir():
        print("DIRECTORY NOT FOUND")
        continue

    found = 0

    for path in sorted(base.glob("*.csv")):
        try:
            columns = pd.read_csv(path, nrows=0).columns.tolist()

            trace_columns = [
                column for column in columns
                if any(
                    token in column.lower()
                    for token in TRACE_TOKENS
                )
            ]

            if not trace_columns:
                continue

            found += 1
            frame = pd.read_csv(path, dtype={"record_id": str})

            print(f"\nFILE: {path.name}")
            print("Shape:", frame.shape)
            print("Trace columns:", trace_columns)

            if "method" in frame.columns:
                print(
                    "Methods:",
                    frame["method"].drop_duplicates().tolist()
                )
                selected = frame.loc[
                    frame["method"].eq("Cascade")
                ]
            else:
                selected = frame

            print("Selected trace rows:", len(selected))

            if selected.empty:
                continue

            id_columns = [
                column
                for column in [
                    "record_id", "method", "true_label",
                    "prediction", "fallback",
                ]
                if column in selected.columns
            ]

            for _, row in selected.head(3).iterrows():
                print("\nTRACE ROW")
                for column in id_columns + trace_columns:
                    print(
                        f"  {column}: "
                        f"{preview_value(row[column])}"
                    )

            # Summarize JSON-list lengths without assuming rank order.
            for column in trace_columns:
                lengths = []
                parse_failures = 0

                for value in selected[column]:
                    if pd.isna(value):
                        continue

                    try:
                        parsed = json.loads(str(value))
                    except (json.JSONDecodeError, TypeError):
                        parse_failures += 1
                        continue

                    if isinstance(parsed, list):
                        lengths.append(len(parsed))

                if lengths:
                    counts = pd.Series(lengths).value_counts().sort_index()
                    print(
                        f"\n  {column} list-length counts:",
                        counts.to_dict(),
                    )

                if parse_failures:
                    print(
                        f"  {column}: non-JSON or unparsed values:",
                        parse_failures,
                    )

        except Exception as exc:
            print(
                f"\nCould not inspect {path.name}: "
                f"{type(exc).__name__}: {exc}"
            )

    print("\nFiles with trace-like columns:", found)

print("\n=== AVAILABLE SPLIT / DATA MANIFEST FILENAMES ===")

for directory in [
    ROOT / "data_audit",
    ROOT / "reviewer1_comment1_dialogue_only",
    ROOT / "supplementary_repeated_splits" / "seed_11",
]:
    if directory.is_dir():
        print("\nDIRECTORY:", directory)
        for path in sorted(directory.glob("*.csv")):
            print(" ", path.name)

print("\nRead-only inspection complete. No files changed.")

In [ ]:
import inspect
import re

PATTERN = re.compile(
    r"\bbm25\b|\brrf\b|"
    r"retrieved_training_ids|"
    r"cascade_top_k_classes|"
    r"doc_truncate_chars|"
    r"\bshortlist\b|"
    r"def\s+\w*(?:cascade|retriev)\w*",
    re.IGNORECASE,
)

MAX_OBJECTS = 12
MAX_LINES_PER_OBJECT = 160
MAX_HISTORY_CELLS = 8

print("=== LIVE RETRIEVAL FUNCTIONS / CLASSES ===")

found_objects = 0
seen_objects = set()

for name, obj in list(globals().items()):
    if name.startswith("_"):
        continue

    if not (inspect.isfunction(obj) or inspect.isclass(obj)):
        continue

    if getattr(obj, "__module__", None) != "__main__":
        continue

    if id(obj) in seen_objects:
        continue
    seen_objects.add(id(obj))

    try:
        source = inspect.getsource(obj)
    except (OSError, TypeError):
        continue

    if not PATTERN.search(source):
        continue

    print(f"\n--- {name} ---")
    try:
        print("Signature:", inspect.signature(obj))
    except (TypeError, ValueError):
        pass

    lines = source.splitlines()

    for line in lines[:MAX_LINES_PER_OBJECT]:
        print(line)

    if len(lines) > MAX_LINES_PER_OBJECT:
        print(
            f"... source truncated after "
            f"{MAX_LINES_PER_OBJECT} lines"
        )

    found_objects += 1

    if found_objects >= MAX_OBJECTS:
        print("\nObject-output limit reached.")
        break

print("\nSource-bearing objects found:", found_objects)

print("\n=== EARLIER EXECUTED NOTEBOOK CELLS ===")

try:
    ip = get_ipython()
    history = list(
        ip.history_manager.get_range(
            session=0,
            start=1,
            stop=None,
            raw=True,
        )
    )

    # Exclude this currently executing diagnostic cell.
    current_execution = ip.execution_count
    matches = []

    for session, cell_number, source in history:
        if cell_number >= current_execution:
            continue

        # Focus on implementation-bearing cells, not file discovery.
        if (
            PATTERN.search(source)
            and re.search(r"\bdef\s+|\bclass\s+", source)
            and any(token in source for token in [
                "get_scores",
                ".encode(",
                ".predict(",
                "retrieved_training_ids",
                "cascade_top_k_classes",
            ])
        ):
            matches.append((cell_number, source))

    for cell_number, source in matches[-MAX_HISTORY_CELLS:]:
        print(f"\n--- Executed cell {cell_number} ---")

        lines = source.splitlines()
        relevant = [
            i for i, line in enumerate(lines)
            if PATTERN.search(line)
        ]

        shown = set()

        for hit in relevant:
            for i in range(
                max(0, hit - 8),
                min(len(lines), hit + 22),
            ):
                if i not in shown:
                    print(f"{i + 1:4d}: {lines[i]}")
                    shown.add(i)

    print("\nImplementation-bearing history cells:", len(matches))

except Exception as exc:
    print(
        f"Notebook history unavailable: "
        f"{type(exc).__name__}: {exc}"
    )

print("\nRead-only implementation discovery complete.")

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import numpy as np
import pandas as pd

ROOT = Path(_crus_local_path('project', ''))
BASE = ROOT / "corrected_experiment_outputs"

AUDIT_PATH = BASE / "validation_retrieval_prediction_audit.csv"
MANIFEST_PATH = (
    ROOT / "data_audit" / "corrected_split_manifest.csv"
)

STAMP = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
OUT = ROOT / f"r1_9_partial_validation_audit_{STAMP}"

for path in [AUDIT_PATH, MANIFEST_PATH]:
    if not path.is_file():
        raise FileNotFoundError(path)

manifest = pd.read_csv(
    MANIFEST_PATH, dtype={"record_id": str}
)
audit = pd.read_csv(
    AUDIT_PATH, dtype={"record_id": str}
)

print("Manifest columns:", manifest.columns.tolist())

required_manifest = {"record_id", "split"}
if not required_manifest.issubset(manifest.columns):
    raise ValueError("Manifest requires record_id and split.")

# Do not silently infer labels from unrelated fields.
label_candidates = [
    column for column in ["label", "true_label"]
    if column in manifest.columns
]

if len(label_candidates) != 1:
    raise ValueError(
        "Specify the manifest's verified routing-label column. "
        f"Available columns: {manifest.columns.tolist()}"
    )

LABEL_COLUMN = label_candidates[0]

if manifest["record_id"].duplicated().any():
    raise ValueError("Duplicate record IDs in manifest.")

training = manifest.loc[
    manifest["split"].eq("train")
].copy()

validation = manifest.loc[
    manifest["split"].eq("validation")
].copy()

if training.empty or validation.empty:
    raise ValueError("Training or validation manifest subset is empty.")

train_labels = training.set_index("record_id")[LABEL_COLUMN].astype(str)
validation_labels = validation.set_index("record_id")[
    LABEL_COLUMN
].astype(str)

training_counts = train_labels.value_counts()
rows = []

def parse_list(value, field, record_id):
    if pd.isna(value):
        raise ValueError(f"Missing {field}: {record_id}")

    parsed = json.loads(str(value))
    if not isinstance(parsed, list):
        raise ValueError(f"{field} is not a list: {record_id}")

    return [str(item) for item in parsed]

for method, group in audit.groupby("method", sort=False):
    if group["record_id"].duplicated().any():
        raise ValueError(f"Duplicate query IDs for {method}.")

    if set(group["record_id"]) != set(validation_labels.index):
        raise ValueError(
            f"{method}: query IDs do not match validation manifest."
        )

    for _, row in group.iterrows():
        query_id = row["record_id"]
        true_label = str(row["true_label"])

        if true_label != validation_labels.loc[query_id]:
            raise ValueError(f"Reference-label mismatch: {query_id}")

        document_ids = parse_list(
            row["retrieved_training_ids"],
            "retrieved_training_ids",
            query_id,
        )
        shortlist = parse_list(
            row["shortlist"], "shortlist", query_id
        )

        if len(document_ids) != len(set(document_ids)):
            raise ValueError(
                f"{method}/{query_id}: duplicate candidate document IDs."
            )

        outside_training = set(document_ids) - set(train_labels.index)
        if outside_training:
            raise ValueError(
                f"{method}/{query_id}: candidates outside training: "
                f"{sorted(outside_training)}"
            )

        relevant_count = sum(
            train_labels.loc[doc_id] == true_label
            for doc_id in document_ids
        )
        relevant_total = int(training_counts.get(true_label, 0))

        rows.append({
            "method": method,
            "record_id": query_id,
            "true_label": true_label,
            "correct": str(row["prediction"]) == true_label,
            "n_saved_documents": len(document_ids),
            "n_correct_label_documents": relevant_count,
            "document_set_has_correct_label": (
                relevant_count > 0 if document_ids else np.nan
            ),
            "document_set_recall": (
                relevant_count / relevant_total
                if document_ids and relevant_total else np.nan
            ),
            "n_shortlisted_labels": len(shortlist),
            "shortlist_has_correct_label": (
                true_label in shortlist if shortlist else np.nan
            ),
        })

records = pd.DataFrame(rows)
summaries = []

for method, group in records.groupby("method", sort=False):
    document_rows = group.loc[
        group["n_saved_documents"].gt(0)
    ]
    shortlist_rows = group.loc[
        group["n_shortlisted_labels"].gt(0)
    ]

    summaries.append({
        "method": method,
        "n_queries": len(group),
        "n_queries_with_saved_documents": len(document_rows),
        "mean_saved_document_count": (
            document_rows["n_saved_documents"].mean()
        ),
        "saved_document_correct_label_rate": (
            document_rows["document_set_has_correct_label"].mean()
        ),
        "mean_document_set_recall": (
            document_rows["document_set_recall"].mean()
        ),
        "n_queries_with_nonempty_shortlist": len(shortlist_rows),
        "mean_nonempty_shortlist_size": (
            shortlist_rows["n_shortlisted_labels"].mean()
        ),
        "nonempty_shortlist_correct_label_rate": (
            shortlist_rows["shortlist_has_correct_label"].mean()
        ),
        "final_accuracy": group["correct"].mean(),
    })

summary = pd.DataFrame(summaries)

cascade = records.loc[records["method"].eq("Cascade")].copy()
conditional_rows = []

for field in [
    "document_set_has_correct_label",
    "shortlist_has_correct_label",
]:
    available = cascade.loc[cascade[field].notna()]

    for present in [True, False]:
        subset = available.loc[available[field].eq(present)]

        conditional_rows.append({
            "candidate_field": field,
            "correct_label_present": present,
            "n_queries": len(subset),
            "n_correct_predictions": int(subset["correct"].sum()),
            "final_accuracy": (
                subset["correct"].mean()
                if len(subset) else np.nan
            ),
        })

conditional = pd.DataFrame(conditional_rows)

OUT.mkdir(parents=True, exist_ok=False)

records.to_csv(
    OUT / "validation_candidate_presence_by_record.csv",
    index=False,
)
summary.to_csv(
    OUT / "validation_candidate_presence_summary.csv",
    index=False,
)
conditional.to_csv(
    OUT / "validation_cascade_conditional_accuracy.csv",
    index=False,
)

(OUT / "scope_and_limitations.json").write_text(json.dumps({
    "split": "validation only",
    "relevance": "training-document routing label matches query target",
    "candidate_order": "not assumed",
    "stage_identity": "not inferred",
    "document_recall_denominator": (
        "all training documents with the query's routing label"
    ),
    "empty_lists": (
        "treated as unavailable traces, not automatically retrieval failures"
    ),
    "not_computed": [
        "test-stage retrieval metrics",
        "rank-based MRR or nDCG",
        "separate BM25/SBERT/RRF/Stage A/Stage B transitions",
    ],
}, indent=2))

print("\n=== VALIDATION CANDIDATE-PRESENCE SUMMARY ===")
print(summary.to_string(
    index=False, float_format=lambda x: f"{x:.6f}"
))

print("\n=== CASCADE CONDITIONAL ACCURACY ===")
print(conditional.to_string(
    index=False, float_format=lambda x: f"{x:.6f}"
))

print("\nAll saved document references verified as training members.")
print("New diagnostic outputs:", OUT)
print("No existing results modified.")


In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path(_crus_local_path('project', ''))
AUDIT = (
    ROOT / "corrected_experiment_outputs"
    / "validation_retrieval_prediction_audit.csv"
)

MANIFESTS = [
    ROOT / "data_audit" / "corrected_split_manifest.csv",
    ROOT / "data_audit" / "constrained_split_manifest.csv",
]

audit = pd.read_csv(AUDIT, dtype={"record_id": str})

required = {"record_id", "method"}
if not required.issubset(audit.columns):
    raise ValueError(f"Missing audit columns: {required - set(audit.columns)}")

if audit["record_id"].isna().any():
    raise ValueError("Audit contains missing record IDs.")

print("=== AUDIT COVERAGE ===")
print("Audit shape:", audit.shape)

method_ids = {}

for method, group in audit.groupby("method", sort=False):
    method_ids[method] = set(group["record_id"])
    print({
        "method": method,
        "rows": len(group),
        "unique_query_ids": group["record_id"].nunique(),
        "duplicate_query_rows": int(
            group["record_id"].duplicated().sum()
        ),
    })

first_method = next(iter(method_ids))
print(
    "\nAll methods have identical query-ID sets:",
    all(
        ids == method_ids[first_method]
        for ids in method_ids.values()
    ),
)

for path in MANIFESTS:
    if not path.is_file():
        continue

    manifest = pd.read_csv(path, dtype={"record_id": str})

    print("\n" + "=" * 78)
    print("MANIFEST:", path.name)
    print("Columns:", manifest.columns.tolist())

    if not {"record_id", "split"}.issubset(manifest.columns):
        print("Skipped: requires record_id and split.")
        continue

    print("\nRaw split values and row counts:")
    print(
        manifest["split"].map(repr)
        .value_counts(dropna=False)
        .to_string()
    )

    if manifest["record_id"].isna().any():
        print("WARNING: missing manifest record IDs.")

    duplicate_count = int(
        manifest["record_id"].duplicated().sum()
    )
    print("Duplicate manifest record IDs:", duplicate_count)

    split_names = (
        manifest["split"].astype("string").str.strip().str.lower()
    )
    all_manifest_ids = set(manifest["record_id"].dropna())

    for method, ids in method_ids.items():
        matched = manifest.loc[manifest["record_id"].isin(ids)]

        print(f"\n--- {method} ---")
        print("Query IDs:", len(ids))
        print(
            "Query IDs absent from this manifest:",
            len(ids - all_manifest_ids),
        )
        print("Manifest split membership of matched queries:")
        print(
            matched["split"].value_counts(dropna=False).to_dict()
        )

        for split in sorted(split_names.dropna().unique()):
            split_ids = set(
                manifest.loc[
                    split_names.eq(split), "record_id"
                ].dropna()
            )

            missing_queries = split_ids - ids
            extra_queries = ids - split_ids

            print({
                "manifest_split": split,
                "manifest_unique_ids": len(split_ids),
                "overlap": len(ids & split_ids),
                "manifest_ids_not_in_audit": len(missing_queries),
                "audit_ids_not_in_split": len(extra_queries),
                "exact_id_match": ids == split_ids,
            })

            if split in {"validation", "val", "valid", "dev"}:
                if missing_queries:
                    print(
                        "  Missing-from-audit examples:",
                        sorted(missing_queries)[:8],
                    )
                if extra_queries:
                    print(
                        "  Outside-this-split examples:",
                        sorted(extra_queries)[:8],
                    )
                    columns = [
                        column for column in [
                            "record_id", "split", "label"
                        ]
                        if column in manifest.columns
                    ]
                    print(
                        manifest.loc[
                            manifest["record_id"].isin(
                                sorted(extra_queries)[:8]
                            ),
                            columns,
                        ].to_string(index=False)
                    )

        if "label" in manifest.columns and "true_label" in audit.columns:
            if duplicate_count == 0:
                label_check = audit.loc[
                    audit["method"].eq(method),
                    ["record_id", "true_label"],
                ].merge(
                    manifest[["record_id", "label"]],
                    on="record_id",
                    how="inner",
                    validate="many_to_one",
                )
                disagreements = label_check.loc[
                    label_check["true_label"].astype(str)
                    != label_check["label"].astype(str)
                ]
                print(
                    "Target-label disagreements among matched rows:",
                    len(disagreements),
                )
                if len(disagreements):
                    print(disagreements.head(8).to_string(index=False))

print("\nRead-only diagnostic complete. No files changed.")

In [ ]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path(_crus_local_path('project', ''))

audit = pd.read_csv(
    ROOT / "corrected_experiment_outputs"
    / "validation_retrieval_prediction_audit.csv",
    dtype={"record_id": str},
)
manifest = pd.read_csv(
    ROOT / "data_audit" / "constrained_split_manifest.csv",
    dtype={"record_id": str},
)

if manifest["record_id"].duplicated().any():
    raise ValueError("Duplicate record IDs in constrained manifest.")

training_ids = set(
    manifest.loc[manifest["split"].eq("train"), "record_id"]
)
validation_ids = set(
    manifest.loc[manifest["split"].eq("validation"), "record_id"]
)
split_lookup = manifest.set_index("record_id")["split"].to_dict()

problems = []
summary = []

for method, group in audit.groupby("method", sort=False):
    if group["record_id"].duplicated().any():
        raise ValueError(f"{method}: duplicate query IDs.")

    if set(group["record_id"]) != validation_ids:
        raise ValueError(f"{method}: validation-ID mismatch.")

    nonempty_traces = 0
    candidate_references = 0
    empty_or_missing_traces = 0

    for _, row in group.iterrows():
        query_id = row["record_id"]
        raw = row["retrieved_training_ids"]

        if pd.isna(raw):
            empty_or_missing_traces += 1
            continue

        try:
            candidates = json.loads(str(raw))
        except (TypeError, ValueError) as exc:
            raise ValueError(
                f"{method}/{query_id}: invalid candidate JSON."
            ) from exc

        if not isinstance(candidates, list):
            raise ValueError(
                f"{method}/{query_id}: candidates are not a list."
            )

        candidates = [str(value) for value in candidates]

        if not candidates:
            empty_or_missing_traces += 1
            continue

        nonempty_traces += 1
        candidate_references += len(candidates)

        if len(candidates) != len(set(candidates)):
            problems.append({
                "method": method,
                "query_id": query_id,
                "candidate_id": None,
                "issue": "duplicate_candidate_ids",
                "manifest_split": None,
            })

        for candidate_id in candidates:
            if candidate_id not in training_ids:
                problems.append({
                    "method": method,
                    "query_id": query_id,
                    "candidate_id": candidate_id,
                    "issue": (
                        "self_retrieval"
                        if candidate_id == query_id
                        else "outside_constrained_training"
                    ),
                    "manifest_split": split_lookup.get(
                        candidate_id, "absent_from_manifest"
                    ),
                })

    summary.append({
        "method": method,
        "validation_queries": len(group),
        "nonempty_candidate_traces": nonempty_traces,
        "empty_or_missing_candidate_traces": empty_or_missing_traces,
        "candidate_references": candidate_references,
    })

print("=== SAVED CANDIDATE COVERAGE ===")
print(pd.DataFrame(summary).to_string(index=False))

print("\n=== MEMBERSHIP / DUPLICATE ISSUES ===")
if problems:
    issue_table = pd.DataFrame(problems)
    print(issue_table.to_string(index=False))
    print("\nIssue counts:")
    print(issue_table.groupby(["method", "issue"]).size().to_string())
else:
    print("None: every saved candidate belongs to constrained training.")
    print("No self-retrieval or duplicate candidate IDs detected.")

print("\nRead-only check complete. No files changed.")

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import numpy as np
import pandas as pd

ROOT = Path(_crus_local_path('project', ''))
AUDIT_PATH = (
    ROOT / "corrected_experiment_outputs"
    / "validation_retrieval_prediction_audit.csv"
)
MANIFEST_PATH = (
    ROOT / "data_audit" / "constrained_split_manifest.csv"
)

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
OUT = ROOT / f"r1_9_constrained_validation_audit_{stamp}"

audit = pd.read_csv(AUDIT_PATH, dtype={"record_id": str})
manifest = pd.read_csv(MANIFEST_PATH, dtype={"record_id": str})

required_audit = {
    "record_id", "method", "true_label", "prediction",
    "retrieved_training_ids", "shortlist",
}
required_manifest = {"record_id", "split", "label"}

if not required_audit.issubset(audit.columns):
    raise ValueError(
        f"Missing audit columns: {required_audit - set(audit.columns)}"
    )
if not required_manifest.issubset(manifest.columns):
    raise ValueError(
        f"Missing manifest columns: "
        f"{required_manifest - set(manifest.columns)}"
    )

if manifest["record_id"].isna().any():
    raise ValueError("Missing manifest record IDs.")
if manifest["record_id"].duplicated().any():
    raise ValueError("Duplicate manifest record IDs.")
if manifest["label"].isna().any():
    raise ValueError("Missing manifest labels.")
if audit[
    ["record_id", "method", "true_label", "prediction"]
].isna().any().any():
    raise ValueError("Missing audit identifiers, targets, or predictions.")

train = manifest.loc[manifest["split"].eq("train")]
validation = manifest.loc[manifest["split"].eq("validation")]

if train.empty or validation.empty:
    raise ValueError("Training or validation partition is empty.")

train_label = train.set_index("record_id")["label"].astype(str)
val_label = validation.set_index("record_id")["label"].astype(str)
train_ids = set(train_label.index)
val_ids = set(val_label.index)
class_sizes = train_label.value_counts()

def parse_saved_list(value, field, query_id):
    if pd.isna(value):
        return []

    try:
        parsed = json.loads(str(value))
    except (TypeError, ValueError) as exc:
        raise ValueError(
            f"{query_id}: invalid JSON in {field}"
        ) from exc

    if not isinstance(parsed, list):
        raise ValueError(f"{query_id}: {field} is not a list.")

    parsed = [str(item) for item in parsed]
    if len(parsed) != len(set(parsed)):
        raise ValueError(f"{query_id}: duplicates in {field}.")

    return parsed

records = []

for method, group in audit.groupby("method", sort=False):
    if group["record_id"].duplicated().any():
        raise ValueError(f"{method}: duplicate validation queries.")
    if set(group["record_id"]) != val_ids:
        raise ValueError(f"{method}: validation query-ID mismatch.")

    for _, row in group.iterrows():
        query_id = row["record_id"]
        target = str(row["true_label"])

        if target != val_label.loc[query_id]:
            raise ValueError(f"{method}/{query_id}: target mismatch.")

        docs = parse_saved_list(
            row["retrieved_training_ids"],
            "retrieved_training_ids",
            query_id,
        )
        shortlist = parse_saved_list(
            row["shortlist"], "shortlist", query_id
        )

        outside_train = set(docs) - train_ids
        if outside_train:
            raise ValueError(
                f"{method}/{query_id}: nontraining candidates: "
                f"{sorted(outside_train)}"
            )

        n_relevant = sum(
            train_label.loc[doc_id] == target for doc_id in docs
        )
        total_relevant = int(class_sizes.get(target, 0))

        records.append({
            "method": method,
            "record_id": query_id,
            "true_label": target,
            "prediction": str(row["prediction"]),
            "correct": str(row["prediction"]) == target,
            "document_trace_available": bool(docs),
            "saved_document_count": len(docs),
            "same_label_document_count": n_relevant,
            "document_correct_label_present": (
                int(n_relevant > 0) if docs else np.nan
            ),
            "document_set_recall": (
                n_relevant / total_relevant
                if docs and total_relevant else np.nan
            ),
            "shortlist_trace_available": bool(shortlist),
            "saved_shortlist_size": len(shortlist),
            "shortlist_correct_label_present": (
                int(target in shortlist) if shortlist else np.nan
            ),
            "prediction_in_shortlist": (
                int(str(row["prediction"]) in shortlist)
                if shortlist else np.nan
            ),
        })

records = pd.DataFrame(records)

summary_rows = []
conditional_rows = []

for method, group in records.groupby("method", sort=False):
    docs = group.loc[group["document_trace_available"]]
    lists = group.loc[group["shortlist_trace_available"]]

    summary_rows.append({
        "method": method,
        "n_validation_queries": len(group),
        "n_document_traces": len(docs),
        "min_saved_documents": (
            docs["saved_document_count"].min()
            if len(docs) else np.nan
        ),
        "mean_saved_documents": docs["saved_document_count"].mean(),
        "max_saved_documents": (
            docs["saved_document_count"].max()
            if len(docs) else np.nan
        ),
        "document_correct_label_presence_rate": (
            docs["document_correct_label_present"].mean()
        ),
        "mean_document_set_recall": docs["document_set_recall"].mean(),
        "n_shortlist_traces": len(lists),
        "mean_saved_shortlist_size": lists["saved_shortlist_size"].mean(),
        "shortlist_correct_label_presence_rate": (
            lists["shortlist_correct_label_present"].mean()
        ),
        "prediction_in_shortlist_rate": (
            lists["prediction_in_shortlist"].mean()
        ),
        "final_accuracy": group["correct"].mean(),
    })

    for field in [
        "document_correct_label_present",
        "shortlist_correct_label_present",
    ]:
        available = group.loc[group[field].notna()]

        for present in [1, 0]:
            subset = available.loc[available[field].eq(present)]

            conditional_rows.append({
                "method": method,
                "candidate_field": field,
                "correct_label_present": bool(present),
                "n_queries": len(subset),
                "n_correct_predictions": int(subset["correct"].sum()),
                "final_accuracy": (
                    subset["correct"].mean()
                    if len(subset) else np.nan
                ),
            })

summary = pd.DataFrame(summary_rows)
conditional = pd.DataFrame(conditional_rows)

OUT.mkdir(parents=True, exist_ok=False)

records.to_csv(OUT / "per_query_diagnostics.csv", index=False)
summary.to_csv(OUT / "method_summary.csv", index=False)
conditional.to_csv(OUT / "conditional_accuracy.csv", index=False)

metadata = {
    "audit_file": str(AUDIT_PATH),
    "manifest_file": str(MANIFEST_PATH),
    "evaluation_partition": "constrained validation",
    "n_training_records": len(train),
    "n_validation_records": len(validation),
    "query_coverage_verified": True,
    "candidate_training_membership_verified": True,
    "relevance_definition": (
        "Candidate training document has the query's routing label."
    ),
    "document_set_recall_definition": (
        "Number of saved same-label training candidates divided by "
        "all training documents with the query's routing label."
    ),
    "candidate_order_assumed": False,
    "stage_identity_assumed": False,
    "empty_trace_policy": (
        "Unavailable; excluded from trace-metric denominators. "
        "Final accuracy still uses all validation queries."
    ),
    "comparison_caution": (
        "Saved candidate depths differ across methods."
    ),
    "not_estimated": [
        "test-set retrieval metrics",
        "unsaved candidate depths",
        "MRR or nDCG",
        "separate intermediate-stage rankings",
    ],
}

(OUT / "audit_metadata.json").write_text(
    json.dumps(metadata, indent=2),
    encoding="utf-8",
)

print("=== FULL VALIDATION SAVED-TRACE SUMMARY ===")
print(summary.to_string(
    index=False, float_format=lambda value: f"{value:.6f}"
))

print("\n=== CASCADE CONDITIONAL ACCURACY ===")
print(
    conditional.loc[
        conditional["method"].eq("Cascade")
    ].to_string(
        index=False, float_format=lambda value: f"{value:.6f}"
    )
)

print("\nSaved new diagnostic outputs to:", OUT)
print("No existing experiment results modified.")

In [ ]:
from pathlib import Path
import json
import re

PROJECT = Path(_crus_local_path('project', ''))
DRIVE = Path(_crus_local_path('drive', ''))

MAX_RESULTS = 10
MAX_EXCERPTS_PER_FILE = 8
CONTEXT_LINES = 7

# Add other known code folders here if needed.
PRIMARY_ROOTS = [PROJECT, Path(_crus_local_path('runtime', ''))]

PATTERNS = {
    "retrieval_depth": re.compile(
        r"\b(?:retrieval_k|retrieval_top_k|top_k_docs|"
        r"bm25_top_k|sbert_top_k|rerank_top_k|"
        r"n_candidates|top_k|topk)\b",
        re.I,
    ),
    "rrf": re.compile(
        r"\brrf\b|k_rrf|rrf_k|rank_constant|"
        r"reciprocal.?rank|def\s+\w*rrf\w*",
        re.I,
    ),
    "shortlist": re.compile(
        r"cascade_top_k_classes|class_shortlist|"
        r"shortlist|top_k_classes",
        re.I,
    ),
    "truncation": re.compile(
        r"doc_truncate_chars|truncate_chars|"
        r"truncat|:\s*400\s*\]|max_chars",
        re.I,
    ),
    "retrieval_implementation": re.compile(
        r"get_scores|BM25Okapi|SentenceTransformer|"
        r"CrossEncoder|semantic_search|"
        r"retrieved_training_ids",
        re.I,
    ),
    "experiment_identity": re.compile(
        r"corrected_experiment_outputs|"
        r"constrained_split_manifest|"
        r"corrected_split_manifest|"
        r"SingleStageTruncated|LengthNormLegacy",
        re.I,
    ),
}

EXCLUDED_PARTS = {
    ".git", ".ipynb_checkpoints", "__pycache__",
    "site-packages", "node_modules", ".cache",
}

# Avoid selecting the recent diagnostic scripts as pipeline code.
DIAGNOSTIC_MARKERS = [
    "FULL VALIDATION SAVED-TRACE SUMMARY",
    "SAVED CANDIDATE COVERAGE",
    "LIVE RETRIEVAL FUNCTIONS / CLASSES",
    "FULL SAVED VALIDATION AUDIT",
    "candidate_membership",
]

def read_code(path):
    if path.suffix.lower() == ".ipynb":
        notebook = json.loads(path.read_text(encoding="utf-8"))
        units = []
        for cell_number, cell in enumerate(
            notebook.get("cells", []), start=1
        ):
            if cell.get("cell_type") != "code":
                continue

            source = cell.get("source", [])
            text = (
                "".join(source) if isinstance(source, list)
                else str(source)
            )
            units.append((f"cell {cell_number}", text))
        return units

    return [("Python file", path.read_text(encoding="utf-8"))]

def discover(root, seen, include_drive=False):
    if not root.exists():
        return

    # /content is scanned outside mounted Drive in the primary pass.
    for suffix in ("*.ipynb", "*.py"):
        for path in root.rglob(suffix):
            if any(part in EXCLUDED_PARTS for part in path.parts):
                continue
            if not include_drive and root == Path(_crus_local_path('runtime', '')):
                if "drive" in path.relative_to(root).parts:
                    continue

            key = str(path)
            if key not in seen:
                seen.add(key)
                yield path

def inspect_candidate(path):
    try:
        units = read_code(path)
    except Exception as exc:
        return None, f"{path}: {type(exc).__name__}"

    eligible = [
        (name, text) for name, text in units
        if not any(marker in text for marker in DIAGNOSTIC_MARKERS)
    ]

    categories = {
        category
        for _, text in eligible
        for category, pattern in PATTERNS.items()
        if pattern.search(text)
    }

    # Require implementation-related evidence, not just mentions.
    implementation = any(
        PATTERNS["retrieval_implementation"].search(text)
        or re.search(
            r"def\s+\w*(?:retriev|cascade|rrf|rerank)\w*",
            text, re.I
        )
        for _, text in eligible
    )

    if not implementation or len(categories) < 2:
        return None, None

    score = len(categories)
    if "experiment_identity" in categories:
        score += 4
    if "rrf" in categories:
        score += 2

    return {
        "path": path,
        "units": eligible,
        "categories": sorted(categories),
        "score": score,
    }, None

seen = set()
matches = []
errors = []

print("=== SEARCHING PROJECT AND LOCAL COLAB CODE ===")

for root in PRIMARY_ROOTS:
    for path in discover(root, seen):
        result, error = inspect_candidate(path)
        if result:
            matches.append(result)
        if error:
            errors.append(error)

# A strong match includes an experiment identifier and retrieval code.
strong_matches = [
    item for item in matches
    if "experiment_identity" in item["categories"]
]

if not strong_matches and DRIVE.exists():
    print(
        "No project-identified implementation found locally; "
        "searching remaining MyDrive notebooks and Python files..."
    )
    for path in discover(DRIVE, seen, include_drive=True):
        result, error = inspect_candidate(path)
        if result:
            matches.append(result)
        if error:
            errors.append(error)

matches.sort(
    key=lambda item: (-item["score"], str(item["path"]))
)

print("\n=== RANKED PIPELINE CANDIDATES ===")
print("Files inspected:", len(seen))
print("Candidate files found:", len(matches))
print("Unreadable files:", len(errors))

for number, item in enumerate(matches[:MAX_RESULTS], start=1):
    print(f"\n{number}. {item['path']}")
    print("   Matched categories:", ", ".join(item["categories"]))

print("\n=== MATCHING CODE EXCERPTS ===")

for item in matches[:MAX_RESULTS]:
    print("\n" + "=" * 90)
    print("FILE:", item["path"])

    excerpts_shown = 0

    for unit_name, text in item["units"]:
        lines = text.splitlines()

        hits = [
            index for index, line in enumerate(lines)
            if any(pattern.search(line) for pattern in PATTERNS.values())
        ]
        if not hits:
            continue

        # Merge overlapping context windows.
        windows = []
        for hit in hits:
            start = max(0, hit - CONTEXT_LINES)
            end = min(len(lines), hit + CONTEXT_LINES + 1)

            if windows and start <= windows[-1][1]:
                windows[-1] = (windows[-1][0], max(end, windows[-1][1]))
            else:
                windows.append((start, end))

        for start, end in windows:
            if excerpts_shown >= MAX_EXCERPTS_PER_FILE:
                break

            print(f"\n--- {unit_name}, lines {start + 1}-{end} ---")
            for index in range(start, end):
                print(f"{index + 1:4d}: {lines[index]}")

            excerpts_shown += 1

        if excerpts_shown >= MAX_EXCERPTS_PER_FILE:
            print("\n[Excerpt limit reached for this file.]")
            break

if not matches:
    print(
        "\nNo matching saved pipeline source was found. "
        "A Colab notebook may not be stored as a searchable "
        ".ipynb file in the mounted filesystem. Download it "
        "from Colab and upload it to /content, then rerun."
    )

if errors:
    print("\nFirst unreadable-file examples:")
    for error in errors[:5]:
        print(" ", error)

print("\nRead-only search complete. No files changed or code executed.")

In [ ]:
from pathlib import Path
import json
import re

PATH = Path(
    _crus_local_path('notebooks', 'revised_risk_aware_hybrid_pipeline_v2.ipynb')
)

if not PATH.is_file():
    raise FileNotFoundError(PATH)

notebook = json.loads(PATH.read_text(encoding="utf-8"))

groups = {
    "CONFIGURATION": [
        r"\bCFG\s*=",
    ],
    "DATA / SPLITS / INPUT CONSTRUCTION": [
        r"constrained_split_manifest",
        r"corrected_split_manifest",
        r"constrained_dialogue_pool",
        r"corrected_dialogue_pool",
        r"cleaned_dialogue_pool",
        r"\btrain_test_split\s*\(",
        r"\bX_train\s*=",
        r"\bX_val\s*=",
        r"\bX_validation\s*=",
        r"\by_train_np\s*=",
        r"\bCLASSES\s*=",
        r"\bread_csv\s*\(",
    ],
    "CASCADE / FUSION / SCORING": [
        r"def\s+stage_a_label_shortlist",
        r"def\s+predict_risk_aware_cascade",
        r"def\s+rrf_fuse",
        r"def\s+predictive_entropy",
        r"def\s+retrieval_label_entropy",
        r"def\s+retrieval_score_variance",
    ],
    "AUDIT GENERATION / OUTPUT DESTINATION": [
        r"validation_retrieval_prediction_audit",
        r"retrieved_training_ids",
        r"corrected_experiment_outputs",
        r"constrained_experiment_outputs",
    ],
}

compiled = {
    name: [re.compile(pattern, re.I) for pattern in patterns]
    for name, patterns in groups.items()
}

matches = []

for cell_number, cell in enumerate(
    notebook.get("cells", []), start=1
):
    if cell.get("cell_type") != "code":
        continue

    source = cell.get("source", [])
    text = (
        "".join(source)
        if isinstance(source, list)
        else str(source)
    )

    categories = [
        name
        for name, patterns in compiled.items()
        if any(pattern.search(text) for pattern in patterns)
    ]

    if categories:
        matches.append((cell_number, categories, text))

print("NOTEBOOK:", PATH.name)
print("Matching code cells:", len(matches))

print("\n=== MATCHING CELL INDEX ===")
for cell_number, categories, text in matches:
    print(
        f"Cell {cell_number}: "
        f"{', '.join(categories)} "
        f"({len(text.splitlines())} lines)"
    )

print("\n=== COMPLETE MATCHING CELLS ===")
for cell_number, categories, text in matches:
    print("\n" + "=" * 90)
    print(f"CELL {cell_number}")
    print("Categories:", ", ".join(categories))
    print("-" * 90)

    for line_number, line in enumerate(text.splitlines(), start=1):
        print(f"{line_number:4d}: {line}")

print("\n=== AVAILABLE DATA SCHEMAS ===")

import pandas as pd

ROOT = Path(_crus_local_path('project', ''))
data_paths = [
    ROOT / "data_audit" / "constrained_dialogue_pool_with_splits.csv",
    ROOT / "data_audit" / "constrained_split_manifest.csv",
    ROOT / "reviewer1_comment1_dialogue_only" / "train_inputs.csv",
    ROOT / "reviewer1_comment1_dialogue_only" / "validation_inputs.csv",
]

for path in data_paths:
    if not path.is_file():
        continue

    preview = pd.read_csv(path, nrows=0)
    print(f"\n{path}")
    print("Columns:", preview.columns.tolist())

print("\nRead-only extraction complete.")
print("No notebook cells executed. No files modified.")

In [ ]:
from pathlib import Path
import ast
import json
import re

PATH = Path(
    _crus_local_path('notebooks', 'revised_risk_aware_hybrid_pipeline_v2.ipynb')
)

# Start with 1. Change to 2 only if another page is reported.
PAGE = 1
LINES_PER_PAGE = 280

notebook = json.loads(PATH.read_text(encoding="utf-8"))

function_pattern = re.compile(
    r"^(?:bm25_retrieve|dense_retrieve|rrf_fuse|"
    r"stage_a_label_shortlist|predict_risk_aware_cascade)$"
)

input_pattern = re.compile(
    r"^\s*(?:X_train|X_val|X_validation|y_train_np|"
    r"CLASSES|CIDX|N_CLS)\s*=|"
    r"constrained_dialogue_pool_with_splits\.csv|"
    r"constrained_split_manifest\.csv|"
    r"validation_retrieval_prediction_audit\.csv"
)

diagnostic_markers = [
    "def inspect_file(",
    "def inspect_artifact(",
    "def inspect_candidate(",
    "=== AUDIT COVERAGE ===",
    "=== SAVED CANDIDATE COVERAGE ===",
    "=== FULL VALIDATION SAVED-TRACE SUMMARY ===",
    "=== COMPLETE MATCHING CELLS ===",
    "=== MATCHING CODE EXCERPTS ===",
    "=== PRIMARY-SPLIT MANIFEST CANDIDATES ===",
    "function_pattern =",
    "PATTERNS =",
]

blocks = []
index = []

for cell_number, cell in enumerate(notebook["cells"], start=1):
    if cell.get("cell_type") != "code":
        continue

    source = cell.get("source", [])
    text = "".join(source) if isinstance(source, list) else str(source)

    if any(marker in text for marker in diagnostic_markers):
        continue

    lines = text.splitlines()
    selected_lines = set()

    try:
        tree = ast.parse(text)
    except SyntaxError:
        tree = None

    if tree is not None:
        for node in ast.walk(tree):
            if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
                if function_pattern.fullmatch(node.name):
                    index.append(
                        f"Cell {cell_number}: {node.name} "
                        f"(lines {node.lineno}-{node.end_lineno})"
                    )
                    selected_lines.update(
                        range(node.lineno - 1, node.end_lineno)
                    )

    for i, line in enumerate(lines):
        if input_pattern.search(line):
            selected_lines.update(
                range(max(0, i - 5), min(len(lines), i + 11))
            )

    if selected_lines:
        block = [f"\n=== CELL {cell_number} ==="]
        previous = None

        for i in sorted(selected_lines):
            if previous is not None and i > previous + 1:
                block.append("    ...")
            block.append(f"{i + 1:4d}: {lines[i]}")
            previous = i

        blocks.extend(block)

print("=== ACTUAL FUNCTION DEFINITIONS FOUND ===")
for entry in index:
    print(entry)

total_pages = max(
    1, (len(blocks) + LINES_PER_PAGE - 1) // LINES_PER_PAGE
)

if not 1 <= PAGE <= total_pages:
    raise ValueError(f"PAGE must be between 1 and {total_pages}.")

start = (PAGE - 1) * LINES_PER_PAGE
end = start + LINES_PER_PAGE

print(f"\n=== EXTRACT PAGE {PAGE}/{total_pages} ===")
for line in blocks[start:end]:
    print(line)

if PAGE < total_pages:
    print(f"\nNEXT: change PAGE to {PAGE + 1} and rerun.")
else:
    print("\nExtraction complete.")

print("Read-only: no pipeline code executed; no files modified.")

In [ ]:
from pathlib import Path
import json

PATH = Path(
    _crus_local_path('notebooks', 'revised_risk_aware_hybrid_pipeline_v2.ipynb')
)

CELL_NUMBER = 68

notebook = json.loads(PATH.read_text(encoding="utf-8"))
cell = notebook["cells"][CELL_NUMBER - 1]

source = cell.get("source", [])
text = "".join(source) if isinstance(source, list) else str(source)

print(f"=== ORIGINAL NOTEBOOK CELL {CELL_NUMBER} ===")
print("Cell type:", cell.get("cell_type"))
print("Total lines:", len(text.splitlines()))

for number, line in enumerate(text.splitlines(), start=1):
    print(f"{number:4d}: {line}")

print("\nRead-only: source displayed, not executed.")

In [ ]:
from pathlib import Path
import json
import re

PATH = Path(
    _crus_local_path('notebooks', 'revised_risk_aware_hybrid_pipeline_v2.ipynb')
)

CELL_NUMBER = 69

notebook = json.loads(PATH.read_text(encoding="utf-8"))

def cell_source(cell):
    source = cell.get("source", [])
    return "".join(source) if isinstance(source, list) else str(source)

config_cells = []

for number, cell in enumerate(notebook["cells"], start=1):
    if cell.get("cell_type") != "code":
        continue

    text = cell_source(cell)

    if re.search(r"^\s*RETRIEVAL_CFG\s*=", text, re.MULTILINE):
        config_cells.append(number)

print("RETRIEVAL_CFG definition cells:", config_cells)

# Show configuration definitions and the requested next cell once each.
selected = list(dict.fromkeys(config_cells + [CELL_NUMBER]))

for number in selected:
    cell = notebook["cells"][number - 1]
    text = cell_source(cell)
    lines = text.splitlines()

    print("\n" + "=" * 75)
    print(f"ORIGINAL NOTEBOOK CELL {number}")
    print("Cell type:", cell.get("cell_type"))
    print("Total lines:", len(lines))

    # Bound output if a cell is unexpectedly large.
    for line_number, line in enumerate(lines[:180], start=1):
        print(f"{line_number:4d}: {line}")

    if len(lines) > 180:
        print(f"[Remaining {len(lines) - 180} lines not displayed.]")

print("\nRead-only: no extracted code executed; no files modified.")

In [ ]:
from pathlib import Path
import json

PATH = Path(
    _crus_local_path('notebooks', 'revised_risk_aware_hybrid_pipeline_v2.ipynb')
)

PAGE = 1
LINES_PER_PAGE = 180

notebook = json.loads(PATH.read_text(encoding="utf-8"))
output = []

for number in range(70, 78):
    cell = notebook["cells"][number - 1]
    source = cell.get("source", [])
    text = "".join(source) if isinstance(source, list) else str(source)

    output.append(f"\n=== ORIGINAL NOTEBOOK CELL {number} ===")
    output.append(f"Cell type: {cell.get('cell_type')}")

    for line_number, line in enumerate(text.splitlines(), start=1):
        output.append(f"{line_number:4d}: {line}")

pages = max(1, (len(output) + LINES_PER_PAGE - 1) // LINES_PER_PAGE)

if not 1 <= PAGE <= pages:
    raise ValueError(f"PAGE must be between 1 and {pages}.")

start = (PAGE - 1) * LINES_PER_PAGE

print(f"=== PAGE {PAGE}/{pages} ===")
print("\n".join(output[start:start + LINES_PER_PAGE]))

if PAGE < pages:
    print(f"\nNext: set PAGE = {PAGE + 1} and rerun.")
else:
    print("\nCells 70–77 fully displayed.")

print("Read-only: no extracted code executed; no files modified.")

In [ ]:
from pathlib import Path
import json

PATH = Path(
    _crus_local_path('notebooks', 'revised_risk_aware_hybrid_pipeline_v2.ipynb')
)

PAGE = 2
LINES_PER_PAGE = 180

notebook = json.loads(PATH.read_text(encoding="utf-8"))
output = []

for number in range(70, 78):
    cell = notebook["cells"][number - 1]
    source = cell.get("source", [])
    text = "".join(source) if isinstance(source, list) else str(source)

    output.append(f"\n=== ORIGINAL NOTEBOOK CELL {number} ===")
    output.append(f"Cell type: {cell.get('cell_type')}")

    for line_number, line in enumerate(text.splitlines(), start=1):
        output.append(f"{line_number:4d}: {line}")

pages = max(1, (len(output) + LINES_PER_PAGE - 1) // LINES_PER_PAGE)

if not 1 <= PAGE <= pages:
    raise ValueError(f"PAGE must be between 1 and {pages}.")

start = (PAGE - 1) * LINES_PER_PAGE

print(f"=== PAGE {PAGE}/{pages} ===")
print("\n".join(output[start:start + LINES_PER_PAGE]))

if PAGE < pages:
    print(f"\nNext: set PAGE = {PAGE + 1} and rerun.")
else:
    print("\nCells 70–77 fully displayed.")

print("Read-only: no extracted code executed; no files modified.")

In [ ]:
PAGE = 3
LINES_PER_PAGE = 180

pages = max(
    1,
    (len(output) + LINES_PER_PAGE - 1) // LINES_PER_PAGE
)
start = (PAGE - 1) * LINES_PER_PAGE

print(f"=== PAGE {PAGE}/{pages} ===")
print("\n".join(output[start:start + LINES_PER_PAGE]))

if PAGE < pages:
    print(f"\nNext: set PAGE = {PAGE + 1} and rerun.")
else:
    print("\nCells 70–77 fully displayed.")

print("Read-only: no extracted code executed; no files modified.")

In [ ]:
PAGE = 4
LINES_PER_PAGE = 180

pages = max(
    1,
    (len(output) + LINES_PER_PAGE - 1) // LINES_PER_PAGE
)
start = (PAGE - 1) * LINES_PER_PAGE

print(f"=== PAGE {PAGE}/{pages} ===")
print("\n".join(output[start:start + LINES_PER_PAGE]))

print("\nRead-only: no extracted code executed; no files modified.")

In [ ]:
from pathlib import Path
import ast
import copy
import json
from collections import Counter

import numpy as np
import pandas as pd
import torch
from scipy.special import softmax
from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi
from tqdm.auto import tqdm

ROOT = Path(_crus_local_path('project', ''))
BASE = ROOT / "corrected_experiment_outputs"
NOTEBOOK = Path(
    _crus_local_path('notebooks', 'revised_risk_aware_hybrid_pipeline_v2.ipynb')
)

paths = {
    "notebook": NOTEBOOK,
    "pool": ROOT / "data_audit/constrained_dialogue_pool_with_splits.csv",
    "manifest": ROOT / "data_audit/constrained_split_manifest.csv",
    "embeddings": BASE / "training_dense_embeddings.npz",
    "audit": BASE / "validation_retrieval_prediction_audit.csv",
}

for name, path in paths.items():
    if not path.is_file():
        raise FileNotFoundError(f"{name}: {path}")

nb = json.loads(NOTEBOOK.read_text(encoding="utf-8"))

def source_of(number):
    source = nb["cells"][number - 1].get("source", [])
    return "".join(source) if isinstance(source, list) else str(source)

def literal_assignment(number, variable):
    tree = ast.parse(source_of(number))
    for node in tree.body:
        if isinstance(node, ast.Assign):
            if any(
                isinstance(target, ast.Name) and target.id == variable
                for target in node.targets
            ):
                return ast.literal_eval(node.value)
    raise ValueError(f"{variable} not found in cell {number}")

config = literal_assignment(67, "RETRIEVAL_CFG")
uncertainty_config = literal_assignment(72, "UNCERTAINTY_CFG")

print("Verified source configuration:")
print(json.dumps(config, indent=2))

manifest = pd.read_csv(paths["manifest"], dtype={"record_id": str})
pool = pd.read_csv(paths["pool"], dtype={"record_id": str})
audit = pd.read_csv(paths["audit"], dtype={"record_id": str})

if manifest["record_id"].duplicated().any():
    raise ValueError("Duplicate manifest IDs.")
if pool["record_id"].duplicated().any():
    raise ValueError("Duplicate pool IDs.")

pool = pool.set_index("record_id")
manifest = manifest.set_index("record_id")

# Restrict inference inputs to training and validation.
train_manifest = manifest.loc[manifest["split"].eq("train")]
val_manifest = manifest.loc[manifest["split"].eq("validation")]

with np.load(paths["embeddings"], allow_pickle=False) as saved:
    train_ids = saved["record_ids"].astype(str)
    train_embs = saved["embeddings"].copy()

if len(set(train_ids)) != len(train_ids):
    raise ValueError("Duplicate saved embedding IDs.")
if set(train_ids) != set(train_manifest.index):
    raise ValueError("Saved embeddings do not match constrained training.")
if train_embs.shape[0] != len(train_ids):
    raise ValueError("Embedding row-count mismatch.")
if not np.isfinite(train_embs).all():
    raise ValueError("Nonfinite saved embeddings.")
if not np.allclose(np.linalg.norm(train_embs, axis=1), 1, atol=1e-4):
    raise ValueError("Saved training embeddings are not normalized.")

reference = audit.loc[audit["method"].eq("Cascade")].copy()

if reference["record_id"].duplicated().any():
    raise ValueError("Duplicate Cascade validation IDs.")
if set(reference["record_id"]) != set(val_manifest.index):
    raise ValueError("Cascade query IDs do not match validation.")

reference = reference.set_index("record_id")
val_ids = reference.index.to_numpy(dtype=str)

train = pool.loc[train_ids]
validation = pool.loc[val_ids]

for frame, expected in [
    (train, train_manifest),
    (validation, val_manifest),
]:
    if frame["text"].isna().any():
        raise ValueError("Missing dialogue input.")
    if not np.array_equal(
        frame["label"].astype(str).to_numpy(),
        expected.loc[frame.index, "label"].astype(str).to_numpy(),
    ):
        raise ValueError("Pool/manifest label mismatch.")

if not np.array_equal(
    validation["label"].astype(str).to_numpy(),
    reference["true_label"].astype(str).to_numpy(),
):
    raise ValueError("Saved validation targets disagree with manifest.")

classes = sorted(train_manifest["label"].astype(str).unique())
X_train = train["text"].astype(str).to_numpy()
y_train = train["label"].astype(str).to_numpy()

torch.manual_seed(config["seed"])
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(config["seed"])

device = "cuda" if torch.cuda.is_available() else "cpu"

embedder = SentenceTransformer(config["dense_model"], device=device)
cross_enc = CrossEncoder(
    config["ce_model"],
    device=device,
    max_length=config["ce_max_tokens"],
)

# Use an isolated namespace instead of replacing notebook globals.
r10 = {
    "np": np,
    "torch": torch,
    "softmax": softmax,
    "Counter": Counter,
    "RETRIEVAL_CFG": copy.deepcopy(config),
    "UNCERTAINTY_CFG": copy.deepcopy(uncertainty_config),
    "X_train": X_train,
    "y_train_np": y_train,
    "train_ids": train_ids,
    "train_embs": train_embs,
    "CLASSES": classes,
    "N_CLS": len(classes),
    "CIDX": {label: i for i, label in enumerate(classes)},
    "embedder": embedder,
    "cross_enc": cross_enc,
    "bm25": BM25Okapi(
        [text.split() for text in X_train],
        k1=config["bm25_k1"],
        b=config["bm25_b"],
    ),
    "average_training_length": float(
        np.mean([len(text) for text in X_train])
    ),
    "DOCUMENT_METHODS": [
        "SingleStageFull", "SingleStageTruncated",
        "Cascade", "LengthNormLegacy",
    ],
}

expected_functions = {
    "ce_raw_logits", "descending_order",
    "bm25_retrieve", "dense_retrieve", "rrf_fuse",
    "normalized_entropy", "uncertainty_components",
    "composite_crus", "prepare_candidates",
    "score_sk4", "score_documents",
}

loaded = set()

for cell_number in range(70, 75):
    tree = ast.parse(source_of(cell_number))
    definitions = [
        node for node in tree.body
        if isinstance(node, ast.FunctionDef)
        and node.name in expected_functions
    ]
    if definitions:
        module = ast.Module(body=definitions, type_ignores=[])
        exec(
            compile(module, f"notebook_cell_{cell_number}", "exec"),
            r10,
        )
        loaded.update(node.name for node in definitions)

if loaded != expected_functions:
    raise ValueError(
        f"Missing source functions: {expected_functions - loaded}"
    )

checks = []
baseline_results = []

def boolean_value(value):
    text = str(value).strip().lower()
    if text not in {"true", "false", "1", "0"}:
        raise ValueError(f"Unexpected boolean: {value}")
    return text in {"true", "1"}

for query_id, row in tqdm(
    validation.iterrows(),
    total=len(validation),
    desc="Reproducing Cascade validation baseline",
):
    candidates = r10["prepare_candidates"](str(row["text"]))
    result = r10["score_documents"](
        str(row["text"]), candidates, "Cascade"
    )
    old = reference.loc[query_id]

    document_ids = [
        str(train_ids[index]) for index, _ in result["documents"]
    ]

    checks.append({
        "record_id": query_id,
        "prediction_match": str(result["pred"]) == str(old["prediction"]),
        "document_order_match": (
            document_ids == json.loads(old["retrieved_training_ids"])
        ),
        "shortlist_order_match": (
            result["shortlist"] == json.loads(old["shortlist"])
        ),
        "confidence_match": bool(np.isclose(
            result["confidence"], float(old["confidence"]),
            rtol=1e-5, atol=1e-7,
        )),
        "fallback_match": (
            bool(result["fallback"]) == boolean_value(old["fallback"])
        ),
        "components_match": bool(np.allclose(
            result["components"],
            old[["H_pred", "H_retrieval", "V_RRF"]].to_numpy(dtype=float),
            rtol=1e-5, atol=1e-7,
        )),
    })
    baseline_results.append(result)

r10_checks = pd.DataFrame(checks)
check_columns = [c for c in r10_checks if c != "record_id"]

print("\n=== BASELINE REPRODUCTION ===")
for column in check_columns:
    print(f"{column}: {int(r10_checks[column].sum())}/{len(r10_checks)}")

r10_baseline_passed = bool(
    r10_checks[check_columns].to_numpy(dtype=bool).all()
)

if not r10_baseline_passed:
    print("\nFirst mismatches:")
    print(
        r10_checks.loc[
            ~r10_checks[check_columns].all(axis=1)
        ].head(10).to_string(index=False)
    )
    raise RuntimeError(
        "Baseline reproduction failed. Do not run sensitivity variants yet."
    )

r10_validation = validation.copy()
r10_baseline_results = baseline_results

print("\nPASS: baseline predictions, candidates, shortlists,")
print("confidence, fallback, and uncertainty components reproduced.")
print("R1.10 runtime is ready for the sensitivity sweep.")
print("No test predictions loaded. No experiment outputs overwritten.")

In [ ]:
%pip install -q rank-bm25

from rank_bm25 import BM25Okapi
from importlib.metadata import version

print("rank-bm25 installed:", version("rank-bm25"))

In [ ]:
from pathlib import Path
import ast
import copy

In [ ]:
from pathlib import Path
import ast
import copy
import json
from collections import Counter

import numpy as np
import pandas as pd
import torch
from scipy.special import softmax
from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi
from tqdm.auto import tqdm

ROOT = Path(_crus_local_path('project', ''))
BASE = ROOT / "corrected_experiment_outputs"
NOTEBOOK = Path(
    _crus_local_path('notebooks', 'revised_risk_aware_hybrid_pipeline_v2.ipynb')
)

paths = {
    "notebook": NOTEBOOK,
    "pool": ROOT / "data_audit/constrained_dialogue_pool_with_splits.csv",
    "manifest": ROOT / "data_audit/constrained_split_manifest.csv",
    "embeddings": BASE / "training_dense_embeddings.npz",
    "audit": BASE / "validation_retrieval_prediction_audit.csv",
}

for name, path in paths.items():
    if not path.is_file():
        raise FileNotFoundError(f"{name}: {path}")

nb = json.loads(NOTEBOOK.read_text(encoding="utf-8"))

def source_of(number):
    source = nb["cells"][number - 1].get("source", [])
    return "".join(source) if isinstance(source, list) else str(source)

def literal_assignment(number, variable):
    tree = ast.parse(source_of(number))
    for node in tree.body:
        if isinstance(node, ast.Assign):
            if any(
                isinstance(target, ast.Name) and target.id == variable
                for target in node.targets
            ):
                return ast.literal_eval(node.value)
    raise ValueError(f"{variable} not found in cell {number}")

config = literal_assignment(67, "RETRIEVAL_CFG")
uncertainty_config = literal_assignment(72, "UNCERTAINTY_CFG")

print("Verified source configuration:")
print(json.dumps(config, indent=2))

manifest = pd.read_csv(paths["manifest"], dtype={"record_id": str})
pool = pd.read_csv(paths["pool"], dtype={"record_id": str})
audit = pd.read_csv(paths["audit"], dtype={"record_id": str})

if manifest["record_id"].duplicated().any():
    raise ValueError("Duplicate manifest IDs.")
if pool["record_id"].duplicated().any():
    raise ValueError("Duplicate pool IDs.")

pool = pool.set_index("record_id")
manifest = manifest.set_index("record_id")

# Restrict inference inputs to training and validation.
train_manifest = manifest.loc[manifest["split"].eq("train")]
val_manifest = manifest.loc[manifest["split"].eq("validation")]

with np.load(paths["embeddings"], allow_pickle=False) as saved:
    train_ids = saved["record_ids"].astype(str)
    train_embs = saved["embeddings"].copy()

if len(set(train_ids)) != len(train_ids):
    raise ValueError("Duplicate saved embedding IDs.")
if set(train_ids) != set(train_manifest.index):
    raise ValueError("Saved embeddings do not match constrained training.")
if train_embs.shape[0] != len(train_ids):
    raise ValueError("Embedding row-count mismatch.")
if not np.isfinite(train_embs).all():
    raise ValueError("Nonfinite saved embeddings.")
if not np.allclose(np.linalg.norm(train_embs, axis=1), 1, atol=1e-4):
    raise ValueError("Saved training embeddings are not normalized.")

reference = audit.loc[audit["method"].eq("Cascade")].copy()

if reference["record_id"].duplicated().any():
    raise ValueError("Duplicate Cascade validation IDs.")
if set(reference["record_id"]) != set(val_manifest.index):
    raise ValueError("Cascade query IDs do not match validation.")

reference = reference.set_index("record_id")
val_ids = reference.index.to_numpy(dtype=str)

train = pool.loc[train_ids]
validation = pool.loc[val_ids]

for frame, expected in [
    (train, train_manifest),
    (validation, val_manifest),
]:
    if frame["text"].isna().any():
        raise ValueError("Missing dialogue input.")
    if not np.array_equal(
        frame["label"].astype(str).to_numpy(),
        expected.loc[frame.index, "label"].astype(str).to_numpy(),
    ):
        raise ValueError("Pool/manifest label mismatch.")

if not np.array_equal(
    validation["label"].astype(str).to_numpy(),
    reference["true_label"].astype(str).to_numpy(),
):
    raise ValueError("Saved validation targets disagree with manifest.")

classes = sorted(train_manifest["label"].astype(str).unique())
X_train = train["text"].astype(str).to_numpy()
y_train = train["label"].astype(str).to_numpy()

torch.manual_seed(config["seed"])
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(config["seed"])

device = "cuda" if torch.cuda.is_available() else "cpu"

embedder = SentenceTransformer(config["dense_model"], device=device)
cross_enc = CrossEncoder(
    config["ce_model"],
    device=device,
    max_length=config["ce_max_tokens"],
)

# Use an isolated namespace instead of replacing notebook globals.
r10 = {
    "np": np,
    "torch": torch,
    "softmax": softmax,
    "Counter": Counter,
    "RETRIEVAL_CFG": copy.deepcopy(config),
    "UNCERTAINTY_CFG": copy.deepcopy(uncertainty_config),
    "X_train": X_train,
    "y_train_np": y_train,
    "train_ids": train_ids,
    "train_embs": train_embs,
    "CLASSES": classes,
    "N_CLS": len(classes),
    "CIDX": {label: i for i, label in enumerate(classes)},
    "embedder": embedder,
    "cross_enc": cross_enc,
    "bm25": BM25Okapi(
        [text.split() for text in X_train],
        k1=config["bm25_k1"],
        b=config["bm25_b"],
    ),
    "average_training_length": float(
        np.mean([len(text) for text in X_train])
    ),
    "DOCUMENT_METHODS": [
        "SingleStageFull", "SingleStageTruncated",
        "Cascade", "LengthNormLegacy",
    ],
}

expected_functions = {
    "ce_raw_logits", "descending_order",
    "bm25_retrieve", "dense_retrieve", "rrf_fuse",
    "normalized_entropy", "uncertainty_components",
    "composite_crus", "prepare_candidates",
    "score_sk4", "score_documents",
}

loaded = set()

for cell_number in range(70, 75):
    tree = ast.parse(source_of(cell_number))
    definitions = [
        node for node in tree.body
        if isinstance(node, ast.FunctionDef)
        and node.name in expected_functions
    ]
    if definitions:
        module = ast.Module(body=definitions, type_ignores=[])
        exec(
            compile(module, f"notebook_cell_{cell_number}", "exec"),
            r10,
        )
        loaded.update(node.name for node in definitions)

if loaded != expected_functions:
    raise ValueError(
        f"Missing source functions: {expected_functions - loaded}"
    )

checks = []
baseline_results = []

def boolean_value(value):
    text = str(value).strip().lower()
    if text not in {"true", "false", "1", "0"}:
        raise ValueError(f"Unexpected boolean: {value}")
    return text in {"true", "1"}

for query_id, row in tqdm(
    validation.iterrows(),
    total=len(validation),
    desc="Reproducing Cascade validation baseline",
):
    candidates = r10["prepare_candidates"](str(row["text"]))
    result = r10["score_documents"](
        str(row["text"]), candidates, "Cascade"
    )
    old = reference.loc[query_id]

    document_ids = [
        str(train_ids[index]) for index, _ in result["documents"]
    ]

    checks.append({
        "record_id": query_id,
        "prediction_match": str(result["pred"]) == str(old["prediction"]),
        "document_order_match": (
            document_ids == json.loads(old["retrieved_training_ids"])
        ),
        "shortlist_order_match": (
            result["shortlist"] == json.loads(old["shortlist"])
        ),
        "confidence_match": bool(np.isclose(
            result["confidence"], float(old["confidence"]),
            rtol=1e-5, atol=1e-7,
        )),
        "fallback_match": (
            bool(result["fallback"]) == boolean_value(old["fallback"])
        ),
        "components_match": bool(np.allclose(
            result["components"],
            old[["H_pred", "H_retrieval", "V_RRF"]].to_numpy(dtype=float),
            rtol=1e-5, atol=1e-7,
        )),
    })
    baseline_results.append(result)

r10_checks = pd.DataFrame(checks)
check_columns = [c for c in r10_checks if c != "record_id"]

print("\n=== BASELINE REPRODUCTION ===")
for column in check_columns:
    print(f"{column}: {int(r10_checks[column].sum())}/{len(r10_checks)}")

r10_baseline_passed = bool(
    r10_checks[check_columns].to_numpy(dtype=bool).all()
)

if not r10_baseline_passed:
    print("\nFirst mismatches:")
    print(
        r10_checks.loc[
            ~r10_checks[check_columns].all(axis=1)
        ].head(10).to_string(index=False)
    )
    raise RuntimeError(
        "Baseline reproduction failed. Do not run sensitivity variants yet."
    )

r10_validation = validation.copy()
r10_baseline_results = baseline_results

print("\nPASS: baseline predictions, candidates, shortlists,")
print("confidence, fallback, and uncertainty components reproduced.")
print("R1.10 runtime is ready for the sensitivity sweep.")
print("No test predictions loaded. No experiment outputs overwritten.")

In [ ]:
from pathlib import Path
import ast
import copy
import json
from collections import Counter

import numpy as np
import pandas as pd
import torch
from scipy.special import softmax
from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi
from tqdm.auto import tqdm

ROOT = Path(_crus_local_path('project', ''))
BASE = ROOT / "corrected_experiment_outputs"
NOTEBOOK = Path(
    _crus_local_path('notebooks', 'revised_risk_aware_hybrid_pipeline_v2.ipynb')
)

# Enable only for your own original, trusted experiment artifact.
TRUST_LOCAL_EMBEDDING_FILE = True

paths = {
    "notebook": NOTEBOOK,
    "pool": ROOT / "data_audit/constrained_dialogue_pool_with_splits.csv",
    "manifest": ROOT / "data_audit/constrained_split_manifest.csv",
    "embeddings": BASE / "training_dense_embeddings.npz",
    "audit": BASE / "validation_retrieval_prediction_audit.csv",
}

for name, path in paths.items():
    if not path.is_file():
        raise FileNotFoundError(f"{name}: {path}")

nb = json.loads(NOTEBOOK.read_text(encoding="utf-8"))

def source_of(number):
    source = nb["cells"][number - 1].get("source", [])
    return "".join(source) if isinstance(source, list) else str(source)

def literal_assignment(number, variable):
    tree = ast.parse(source_of(number))
    for node in tree.body:
        if isinstance(node, ast.Assign):
            if any(
                isinstance(target, ast.Name) and target.id == variable
                for target in node.targets
            ):
                return ast.literal_eval(node.value)
    raise ValueError(f"{variable} not found in cell {number}")

def boolean_value(value):
    text = str(value).strip().lower()
    mapping = {
        "true": True, "false": False,
        "1": True, "0": False,
        "1.0": True, "0.0": False,
    }
    if text not in mapping:
        raise ValueError(f"Unexpected boolean: {value}")
    return mapping[text]

config = literal_assignment(67, "RETRIEVAL_CFG")
uncertainty_config = literal_assignment(72, "UNCERTAINTY_CFG")

print("Verified source configuration:")
print(json.dumps(config, indent=2))

manifest = pd.read_csv(paths["manifest"], dtype={"record_id": str})
pool = pd.read_csv(paths["pool"], dtype={"record_id": str})
audit = pd.read_csv(paths["audit"], dtype={"record_id": str})

for name, frame in [("manifest", manifest), ("pool", pool)]:
    if frame["record_id"].isna().any():
        raise ValueError(f"Missing {name} record IDs.")
    if frame["record_id"].duplicated().any():
        raise ValueError(f"Duplicate {name} record IDs.")

pool = pool.set_index("record_id")
manifest = manifest.set_index("record_id")

train_manifest = manifest.loc[manifest["split"].eq("train")]
val_manifest = manifest.loc[manifest["split"].eq("validation")]

if len(train_manifest) != 773 or len(val_manifest) != 258:
    raise ValueError("Unexpected constrained training/validation sizes.")

# Load numeric embeddings without enabling pickle.
with np.load(paths["embeddings"], allow_pickle=False) as saved:
    train_embs = saved["embeddings"].copy()

    try:
        train_ids = saved["record_ids"].astype(str)
    except ValueError as exc:
        if "Object arrays cannot be loaded" not in str(exc):
            raise

        if not TRUST_LOCAL_EMBEDDING_FILE:
            raise RuntimeError(
                "Object-valued record IDs require trusted-file permission."
            ) from exc

        with np.load(paths["embeddings"], allow_pickle=True) as trusted:
            raw_ids = trusted["record_ids"]

            if raw_ids.ndim != 1:
                raise ValueError("Expected a one-dimensional record-ID array.")
            if not all(isinstance(value, str) for value in raw_ids):
                raise ValueError("Saved record IDs contain non-string values.")

            train_ids = raw_ids.astype(str)

if train_ids.ndim != 1:
    raise ValueError("Invalid training-ID dimensions.")
if len(set(train_ids)) != len(train_ids):
    raise ValueError("Duplicate saved embedding IDs.")
if set(train_ids) != set(train_manifest.index):
    raise ValueError("Saved embeddings do not match constrained training.")
if train_embs.ndim != 2 or train_embs.shape[0] != len(train_ids):
    raise ValueError("Embedding dimensions do not match training IDs.")
if not np.isfinite(train_embs).all():
    raise ValueError("Nonfinite saved embeddings.")
if not np.allclose(np.linalg.norm(train_embs, axis=1), 1, atol=1e-4):
    raise ValueError("Saved training embeddings are not normalized.")

print("\nLoaded training IDs:", len(train_ids))
print("Training embedding shape:", train_embs.shape)

reference = audit.loc[audit["method"].eq("Cascade")].copy()

if reference["record_id"].duplicated().any():
    raise ValueError("Duplicate Cascade validation IDs.")
if set(reference["record_id"]) != set(val_manifest.index):
    raise ValueError("Cascade query IDs do not match validation.")

reference = reference.set_index("record_id")
val_ids = reference.index.to_numpy(dtype=str)

# Preserve the original embedding order for training documents.
train = pool.loc[train_ids]
validation = pool.loc[val_ids]

for frame, expected in [
    (train, train_manifest),
    (validation, val_manifest),
]:
    if frame["text"].isna().any():
        raise ValueError("Missing dialogue input.")
    if frame["text"].astype(str).str.strip().eq("").any():
        raise ValueError("Empty dialogue input.")
    if not np.array_equal(
        frame["label"].astype(str).to_numpy(),
        expected.loc[frame.index, "label"].astype(str).to_numpy(),
    ):
        raise ValueError("Pool/manifest label mismatch.")

if not np.array_equal(
    validation["label"].astype(str).to_numpy(),
    reference["true_label"].astype(str).to_numpy(),
):
    raise ValueError("Saved validation targets disagree with manifest.")

classes = sorted(train_manifest["label"].astype(str).unique())
X_train = train["text"].astype(str).to_numpy()
y_train = train["label"].astype(str).to_numpy()

torch.manual_seed(config["seed"])
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(config["seed"])

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

embedder = SentenceTransformer(config["dense_model"], device=device)
cross_enc = CrossEncoder(
    config["ce_model"],
    device=device,
    max_length=config["ce_max_tokens"],
)

r10 = {
    "np": np,
    "torch": torch,
    "softmax": softmax,
    "Counter": Counter,
    "RETRIEVAL_CFG": copy.deepcopy(config),
    "UNCERTAINTY_CFG": copy.deepcopy(uncertainty_config),
    "X_train": X_train,
    "y_train_np": y_train,
    "train_ids": train_ids,
    "train_embs": train_embs,
    "CLASSES": classes,
    "N_CLS": len(classes),
    "CIDX": {label: i for i, label in enumerate(classes)},
    "embedder": embedder,
    "cross_enc": cross_enc,
    "bm25": BM25Okapi(
        [text.split() for text in X_train],
        k1=config["bm25_k1"],
        b=config["bm25_b"],
    ),
    "average_training_length": float(
        np.mean([len(text) for text in X_train])
    ),
    "DOCUMENT_METHODS": [
        "SingleStageFull",
        "SingleStageTruncated",
        "Cascade",
        "LengthNormLegacy",
    ],
}

expected_functions = {
    "ce_raw_logits",
    "descending_order",
    "bm25_retrieve",
    "dense_retrieve",
    "rrf_fuse",
    "normalized_entropy",
    "uncertainty_components",
    "composite_crus",
    "prepare_candidates",
    "score_sk4",
    "score_documents",
}

loaded = set()

# Execute only the explicitly selected function definitions.
for cell_number in range(70, 75):
    tree = ast.parse(source_of(cell_number))
    definitions = [
        node for node in tree.body
        if isinstance(node, ast.FunctionDef)
        and node.name in expected_functions
    ]

    if definitions:
        module = ast.Module(body=definitions, type_ignores=[])
        exec(
            compile(module, f"notebook_cell_{cell_number}", "exec"),
            r10,
        )
        loaded.update(node.name for node in definitions)

if loaded != expected_functions:
    raise ValueError(
        f"Missing source functions: {expected_functions - loaded}"
    )

checks = []
baseline_results = []
r10_baseline_passed = False

for query_id, row in tqdm(
    validation.iterrows(),
    total=len(validation),
    desc="Reproducing Cascade validation baseline",
):
    query = str(row["text"])
    candidates = r10["prepare_candidates"](query)
    result = r10["score_documents"](query, candidates, "Cascade")
    old = reference.loc[query_id]

    document_ids = [
        str(train_ids[index])
        for index, _ in result["documents"]
    ]

    checks.append({
        "record_id": query_id,
        "prediction_match": (
            str(result["pred"]) == str(old["prediction"])
        ),
        "document_order_match": (
            document_ids == json.loads(old["retrieved_training_ids"])
        ),
        "shortlist_order_match": (
            result["shortlist"] == json.loads(old["shortlist"])
        ),
        "confidence_match": bool(np.isclose(
            result["confidence"],
            float(old["confidence"]),
            rtol=1e-5,
            atol=1e-7,
        )),
        "fallback_match": (
            bool(result["fallback"]) == boolean_value(old["fallback"])
        ),
        "components_match": bool(np.allclose(
            result["components"],
            old[
                ["H_pred", "H_retrieval", "V_RRF"]
            ].to_numpy(dtype=float),
            rtol=1e-5,
            atol=1e-7,
        )),
    })

    baseline_results.append(result)

r10_checks = pd.DataFrame(checks)
check_columns = [
    column for column in r10_checks.columns
    if column != "record_id"
]

print("\n=== BASELINE REPRODUCTION ===")
for column in check_columns:
    print(
        f"{column}: "
        f"{int(r10_checks[column].sum())}/{len(r10_checks)}"
    )

r10_baseline_passed = bool(
    r10_checks[check_columns].to_numpy(dtype=bool).all()
)

if not r10_baseline_passed:
    print("\nFirst mismatches:")
    print(
        r10_checks.loc[
            ~r10_checks[check_columns].all(axis=1)
        ].head(10).to_string(index=False)
    )
    raise RuntimeError(
        "Baseline reproduction failed. "
        "Do not run sensitivity variants yet."
    )

r10_validation = validation.copy()
r10_baseline_results = baseline_results

print("\nPASS: baseline predictions, candidate ordering, shortlists,")
print("confidence, fallback, and uncertainty components reproduced.")
print("R1.10 runtime is ready for the sensitivity sweep.")
print("No test predictions loaded. No experiment outputs overwritten.")

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import copy
import hashlib
import json
import time
import importlib.metadata

import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score
from tqdm.auto import tqdm

if not globals().get("r10_baseline_passed", False):
    raise RuntimeError("Run and pass baseline verification first.")

ROOT = Path(_crus_local_path('project', ''))
STAMP = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
OUT = ROOT / f"r1_10_validation_sensitivity_{STAMP}"

baseline_cfg = copy.deepcopy(r10["RETRIEVAL_CFG"])
validation_frame = r10_validation.copy()
classes = list(r10["CLASSES"])
train_ids_local = np.asarray(r10["train_ids"], dtype=str)
train_labels_local = np.asarray(r10["y_train_np"], dtype=str)
training_texts = list(r10["X_train"])

if len(validation_frame) != 258:
    raise ValueError("Expected 258 constrained validation queries.")

full_character_limit = max(len(text) for text in training_texts)
n_classes = len(classes)

# Deduplicate configurations, including repeated baseline settings.
configurations = {}
sweep_links = []

def register(parameter, displayed_value, settings):
    signature = json.dumps(settings, sort_keys=True)
    if signature not in configurations:
        configurations[signature] = {
            "run_id": f"config_{len(configurations):02d}",
            "configuration": copy.deepcopy(settings),
        }

    sweep_links.append({
        "parameter": parameter,
        "value": str(displayed_value),
        "run_id": configurations[signature]["run_id"],
    })

register("baseline", "original", baseline_cfg)

grids = {
    "topk_retrieve": [5, 10, 20, 40],
    "rrf_k": [10, 30, 60, 100],
    "cascade_top_k_classes": sorted({
        value for value in [1, 3, 5, 7, n_classes]
        if value <= n_classes
    }),
    "doc_truncate_chars": [100, 200, 400, 800],
}

for parameter, values in grids.items():
    # Always include the original baseline value.
    values = sorted(set(values + [baseline_cfg[parameter]]))

    for value in values:
        settings = copy.deepcopy(baseline_cfg)
        settings[parameter] = value
        register(parameter, value, settings)

full_cfg = copy.deepcopy(baseline_cfg)
full_cfg["doc_truncate_chars"] = full_character_limit
register("doc_truncate_chars", "full_text", full_cfg)

def file_hash(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

metadata = {
    "analysis": "R1.10 one-parameter-at-a-time sensitivity",
    "partition": "constrained validation",
    "n_validation": len(validation_frame),
    "n_training": len(train_ids_local),
    "classes": classes,
    "baseline_reproduction": {
        "prediction_match": 258,
        "document_order_match": 258,
        "shortlist_order_match": 258,
        "confidence_match": 258,
        "fallback_match": 258,
        "components_match": 258,
    },
    "baseline_configuration": baseline_cfg,
    "configurations": list(configurations.values()),
    "sweep_links": sweep_links,
    "retrieval_depth_scope": (
        "Both Stage A candidate-label preparation and document retrieval."
    ),
    "final_document_limit_fixed": baseline_cfg["rrf_top_n"],
    "full_text_definition": (
        "No explicit candidate-document character truncation; "
        "cross-encoder token limit remains fixed."
    ),
    "full_text_character_limit_used": full_character_limit,
    "cross_encoder_max_tokens": baseline_cfg["ce_max_tokens"],
    "prediction_evaluation": "forced predictions before abstention",
    "uncertainty_weights": r10["UNCERTAINTY_CFG"]["weights"],
    "gate_thresholds": "not selected or evaluated in this sweep",
    "test_predictions_loaded": False,
    "selection_status": (
        "Revision-stage validation sensitivity; "
        "does not replace original reported baseline."
    ),
    "package_versions": {
        name: importlib.metadata.version(name)
        for name in [
            "numpy", "scikit-learn", "sentence-transformers",
            "transformers", "rank-bm25", "torch",
        ]
    },
    "input_hashes": {
        name: file_hash(path)
        for name, path in paths.items()
    },
}

OUT.mkdir(parents=True, exist_ok=False)
(OUT / "analysis_metadata.json").write_text(
    json.dumps(metadata, indent=2), encoding="utf-8"
)

summary_rows = []

def evaluate_configuration(run_id, settings, results):
    record_rows = []

    for (query_id, row), result in zip(
        validation_frame.iterrows(), results
    ):
        docs = result["documents"]
        indices = [index for index, _ in docs]
        target = str(row["label"])
        shortlist = list(result["shortlist"])

        original_lengths = [
            len(training_texts[index]) for index in indices
        ]
        effective_lengths = [
            min(length, settings["doc_truncate_chars"])
            for length in original_lengths
        ]

        record_rows.append({
            "run_id": run_id,
            "record_id": str(query_id),
            "true_label": target,
            "prediction": str(result["pred"]),
            "correct": str(result["pred"]) == target,
            "confidence": float(result["confidence"]),
            "n_documents": len(docs),
            "n_shortlist_labels": len(shortlist),
            "document_label_present": any(
                train_labels_local[index] == target
                for index in indices
            ),
            "shortlist_label_present": target in shortlist,
            "fallback": bool(result["fallback"]),
            "n_char_truncated_documents": sum(
                length > settings["doc_truncate_chars"]
                for length in original_lengths
            ),
            "total_original_document_chars": sum(original_lengths),
            "total_scoring_document_chars": sum(effective_lengths),
            "shortlist": json.dumps(shortlist),
            "retrieved_training_ids": json.dumps([
                str(train_ids_local[index]) for index in indices
            ]),
            "rrf_scores": json.dumps([
                float(score) for _, score in docs
            ]),
            "H_pred": float(result["components"][0]),
            "H_retrieval": float(result["components"][1]),
            "V_RRF": float(result["components"][2]),
            "CRUS": r10["composite_crus"](result["components"]),
        })

    frame = pd.DataFrame(record_rows)
    total_docs = int(frame["n_documents"].sum())

    summary = {
        "run_id": run_id,
        "topk_retrieve": settings["topk_retrieve"],
        "rrf_k": settings["rrf_k"],
        "rrf_top_n": settings["rrf_top_n"],
        "cascade_top_k_classes": settings["cascade_top_k_classes"],
        "doc_truncate_chars": settings["doc_truncate_chars"],
        "n_queries": len(frame),
        "accuracy": accuracy_score(
            frame["true_label"], frame["prediction"]
        ),
        "macro_f1": f1_score(
            frame["true_label"], frame["prediction"],
            labels=classes, average="macro", zero_division=0,
        ),
        "document_label_presence": frame["document_label_present"].mean(),
        "shortlist_label_presence": frame["shortlist_label_present"].mean(),
        "mean_documents": frame["n_documents"].mean(),
        "mean_shortlist_size": frame["n_shortlist_labels"].mean(),
        "fallback_rate": frame["fallback"].mean(),
        "character_truncation_fraction": (
            frame["n_char_truncated_documents"].sum() / total_docs
            if total_docs else np.nan
        ),
        "mean_scoring_chars_per_document": (
            frame["total_scoring_document_chars"].sum() / total_docs
            if total_docs else np.nan
        ),
    }

    frame.to_csv(OUT / f"{run_id}_per_query.csv", index=False)

    np.savez_compressed(
        OUT / f"{run_id}_arrays.npz",
        record_ids=frame["record_id"].to_numpy(dtype=str),
        true_labels=frame["true_label"].to_numpy(dtype=str),
        predictions=frame["prediction"].to_numpy(dtype=str),
        classes=np.asarray(classes, dtype=str),
        probabilities=np.stack([result["probs"] for result in results]),
        confidence=frame["confidence"].to_numpy(dtype=float),
        components=np.stack([result["components"] for result in results]),
    )

    return summary

try:
    for specification in configurations.values():
        run_id = specification["run_id"]
        settings = specification["configuration"]
        r10["RETRIEVAL_CFG"] = copy.deepcopy(settings)

        is_baseline = settings == baseline_cfg

        print(f"\n=== {run_id} ===")
        print({
            key: settings[key]
            for key in [
                "topk_retrieve", "rrf_k",
                "cascade_top_k_classes", "doc_truncate_chars",
            ]
        })

        start = time.perf_counter()

        if is_baseline:
            results = r10_baseline_results
        else:
            results = []

            for _, row in tqdm(
                validation_frame.iterrows(),
                total=len(validation_frame),
                desc=run_id,
            ):
                query = str(row["text"])
                candidates = r10["prepare_candidates"](query)
                results.append(
                    r10["score_documents"](query, candidates, "Cascade")
                )

        summary = evaluate_configuration(run_id, settings, results)
        summary["reused_verified_baseline"] = is_baseline
        summary["run_elapsed_seconds"] = time.perf_counter() - start

        summary_rows.append(summary)

        pd.DataFrame(summary_rows).to_csv(
            OUT / "completed_configuration_summary.csv", index=False
        )

        print(
            f"Accuracy={summary['accuracy']:.4f}; "
            f"macro-F1={summary['macro_f1']:.4f}; "
            f"document-label presence="
            f"{summary['document_label_presence']:.4f}"
        )

finally:
    r10["RETRIEVAL_CFG"] = copy.deepcopy(baseline_cfg)

summary = pd.DataFrame(summary_rows)
sensitivity = pd.DataFrame(sweep_links).merge(
    summary, on="run_id", how="left", validate="many_to_one"
)

baseline_row = summary.loc[
    summary["reused_verified_baseline"]
].iloc[0]

sensitivity["accuracy_change_pp"] = (
    sensitivity["accuracy"] - baseline_row["accuracy"]
) * 100

sensitivity["macro_f1_change"] = (
    sensitivity["macro_f1"] - baseline_row["macro_f1"]
)

sensitivity.to_csv(
    OUT / "sensitivity_table.csv", index=False
)

(OUT / "RUN_COMPLETE.json").write_text(
    json.dumps({
        "completed_configurations": len(summary),
        "validation_queries_per_configuration": len(validation_frame),
        "baseline_configuration_restored": (
            r10["RETRIEVAL_CFG"] == baseline_cfg
        ),
        "test_predictions_loaded": False,
    }, indent=2),
    encoding="utf-8",
)

display_columns = [
    "parameter", "value",
    "accuracy", "macro_f1",
    "accuracy_change_pp",
    "document_label_presence",
    "shortlist_label_presence",
    "mean_documents",
    "mean_shortlist_size",
    "character_truncation_fraction",
]

print("\n=== R1.10 VALIDATION SENSITIVITY RESULTS ===")
print(sensitivity[display_columns].to_string(
    index=False,
    float_format=lambda value: f"{value:.6f}",
))

print("\nOutputs:", OUT)
print("Baseline configuration restored.")
print("Original experiment outputs were not modified.")

In [ ]:
print("Baseline verification passed:", r10_baseline_passed)
print("Current configuration:", r10["RETRIEVAL_CFG"])

print("\n=== DIALOGUE STRUCTURE: FIRST THREE VALIDATION INPUTS ===")
for record_id, row in r10_validation.head(3).iterrows():
    text = str(row["text"])
    print("\nRecord:", record_id)
    print("Characters:", len(text))
    print("Newline count:", text.count("\n"))
    print("Text representation:", repr(text[:1600]))

In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path(_crus_local_path('project', ''))

# Mount Drive if the project directory is not available.
if not ROOT.exists():
    pass  # Colab service not required locally.
    _crus_prepare_saved_workspace()

paths = {
    "pool": ROOT / "data_audit/constrained_dialogue_pool_with_splits.csv",
    "manifest": ROOT / "data_audit/constrained_split_manifest.csv",
    "audit": (
        ROOT / "corrected_experiment_outputs"
        / "validation_retrieval_prediction_audit.csv"
    ),
}

for name, path in paths.items():
    if not path.is_file():
        raise FileNotFoundError(f"Missing {name} file: {path}")

frames = {
    name: pd.read_csv(path, dtype={"record_id": str})
    for name, path in paths.items()
}

required_columns = {
    "pool": {"record_id", "text", "label"},
    "manifest": {"record_id", "split", "label"},
    "audit": {"record_id", "method", "true_label"},
}

for name, frame in frames.items():
    missing = required_columns[name] - set(frame.columns)
    if missing:
        raise ValueError(
            f"{name}: missing columns {sorted(missing)}.\n"
            f"Available columns: {list(frame.columns)}"
        )
    if frame["record_id"].isna().any():
        raise ValueError(f"{name}: missing record IDs.")

pool = frames["pool"]
manifest = frames["manifest"]
audit = frames["audit"]

for name, frame in [("pool", pool), ("manifest", manifest)]:
    if frame["record_id"].duplicated().any():
        raise ValueError(f"{name}: duplicate record IDs.")

validation_manifest = manifest.loc[
    manifest["split"].eq("validation")
].set_index("record_id")

cascade = audit.loc[
    audit["method"].eq("Cascade")
].set_index("record_id")

if validation_manifest.empty:
    raise ValueError(
        "No validation rows found. Available split values: "
        f"{manifest['split'].dropna().unique().tolist()}"
    )

if cascade.empty:
    raise ValueError(
        "No Cascade rows found. Available methods: "
        f"{audit['method'].dropna().unique().tolist()}"
    )

if not cascade.index.is_unique:
    raise ValueError("Duplicate Cascade audit query IDs.")

if set(cascade.index) != set(validation_manifest.index):
    raise ValueError(
        "Audit IDs do not match constrained validation.\n"
        f"Validation IDs: {len(validation_manifest)}\n"
        f"Cascade audit IDs: {len(cascade)}\n"
        f"Missing from audit: "
        f"{len(set(validation_manifest.index) - set(cascade.index))}\n"
        f"Extra in audit: "
        f"{len(set(cascade.index) - set(validation_manifest.index))}"
    )

pool_by_id = pool.set_index("record_id")
missing_ids = set(cascade.index) - set(pool_by_id.index)
if missing_ids:
    raise ValueError(f"{len(missing_ids)} audit IDs are missing from the pool.")

validation = pool_by_id.loc[cascade.index].copy()

if validation["text"].isna().any():
    raise ValueError("Missing validation text.")

if validation["text"].astype(str).str.strip().eq("").any():
    raise ValueError("Empty validation text.")

for name, labels in [
    (
        "manifest",
        validation_manifest.loc[validation.index, "label"],
    ),
    (
        "audit",
        cascade.loc[validation.index, "true_label"],
    ),
]:
    if labels.isna().any() or validation["label"].isna().any():
        raise ValueError(f"Missing labels in pool or {name}.")
    matches = validation["label"].astype(str).eq(labels.astype(str))
    if not matches.all():
        raise ValueError(
            f"Pool/{name} target-label mismatch: "
            f"{int((~matches).sum())} rows."
        )

print("=== VALIDATION INPUT CHECK ===")
print("Validation queries:", len(validation))
print("Query IDs and target labels match the saved Cascade audit.")

texts = validation["text"].astype(str)
newline_counts = texts.map(lambda text: text.count("\n"))

print("\n=== TEXT FORMATTING SUMMARY ===")
print("Queries containing newlines:", int(newline_counts.gt(0).sum()))
print(
    "Newline-count distribution:",
    newline_counts.value_counts().sort_index().to_dict(),
)
print(
    "Text-length summary:",
    texts.str.len().describe().round(1).to_dict(),
)

print("\n=== FIRST THREE INPUTS ===")
for record_id, row in validation.head(3).iterrows():
    text = str(row["text"])
    print("\nRecord:", record_id)
    print("Characters:", len(text))
    print("Newlines:", text.count("\n"))
    print("Text representation:", repr(text[:1600]))

print("\nRead-only check complete. No models loaded or files modified.")

In [ ]:
import re
import pandas as pd

if "validation" not in globals():
    raise RuntimeError("Run the validation-input check cell first.")

# Require a speaker marker at the start or after whitespace.
speaker_pattern = re.compile(
    r"(?<!\S)(doctor|patient)\s*:",
    flags=re.IGNORECASE,
)

def parse_dialogue_turns(text):
    text = str(text)
    matches = list(speaker_pattern.finditer(text))

    if not matches:
        return [], text.strip()

    prefix = text[:matches[0].start()].strip()
    turns = []

    for i, match in enumerate(matches):
        end = (
            matches[i + 1].start()
            if i + 1 < len(matches)
            else len(text)
        )
        turns.append({
            "speaker": match.group(1).lower(),
            "text": text[match.start():end].strip(),
            "body": text[match.end():end].strip(),
        })

    return turns, prefix

parsed_turns = {}
structure_rows = []

for record_id, row in validation.iterrows():
    text = str(row["text"])
    turns, prefix = parse_dialogue_turns(text)
    parsed_turns[record_id] = turns

    reconstructed = " ".join(turn["text"] for turn in turns)
    normalized_original = " ".join(text.split())
    normalized_reconstructed = " ".join(reconstructed.split())

    structure_rows.append({
        "record_id": record_id,
        "turn_count": len(turns),
        "has_unparsed_prefix": bool(prefix),
        "empty_turn_count": sum(not turn["body"] for turn in turns),
        "reconstruction_matches": (
            normalized_original == normalized_reconstructed
        ),
    })

turn_structure = pd.DataFrame(structure_rows).set_index("record_id")

print("=== TURN-PARSING CHECK ===")
print("Validation queries:", len(turn_structure))
print(
    "Queries with no recognized turns:",
    int(turn_structure["turn_count"].eq(0).sum()),
)
print(
    "Queries with fewer than two turns:",
    int(turn_structure["turn_count"].lt(2).sum()),
)
print(
    "Queries with text before the first speaker:",
    int(turn_structure["has_unparsed_prefix"].sum()),
)
print(
    "Queries containing empty turns:",
    int(turn_structure["empty_turn_count"].gt(0).sum()),
)
print(
    "Queries reconstructed without text loss:",
    int(turn_structure["reconstruction_matches"].sum()),
)
print(
    "Turn-count summary:",
    turn_structure["turn_count"].describe().round(1).to_dict(),
)

problem_mask = (
    turn_structure["turn_count"].eq(0)
    | turn_structure["has_unparsed_prefix"]
    | turn_structure["empty_turn_count"].gt(0)
    | ~turn_structure["reconstruction_matches"]
)

if problem_mask.any():
    print("\n=== PARSING ISSUES: FIRST FIVE ===")
    print(turn_structure.loc[problem_mask].head(5).to_string())

    for record_id in turn_structure.index[problem_mask][:5]:
        print("\nRecord:", record_id)
        print(repr(str(validation.loc[record_id, "text"])[:1200]))

print("\n=== FIRST PARSED DIALOGUE ===")
first_id = validation.index[0]
print("Record:", first_id)

for number, turn in enumerate(parsed_turns[first_id], start=1):
    print(f"{number:02d}. {turn['text']}")

print("\n=== PREVIOUS RUNTIME OBJECTS ===")
for name in ("r10", "r10_baseline_passed", "r10_validation"):
    if name not in globals():
        print(f"{name}: NOT PRESENT")
        continue

    obj = globals()[name]
    print(f"{name}: present; type={type(obj).__name__}")

    if name == "r10" and isinstance(obj, dict):
        print("r10 keys:", sorted(map(str, obj.keys())))
        if "RETRIEVAL_CFG" in obj:
            print("RETRIEVAL_CFG:", obj["RETRIEVAL_CFG"])

    elif name == "r10_baseline_passed":
        print("Value:", repr(obj))

    elif name == "r10_validation" and hasattr(obj, "shape"):
        print("Shape:", obj.shape)

print("\nNo files modified. No inference performed.")

In [ ]:
import re
from pathlib import Path
import pandas as pd

if "validation" not in globals():
    raise RuntimeError("Run the validation-input check cell first.")

ROOT = Path(_crus_local_path('project', ''))

speaker_pattern = re.compile(
    r"(?<!\S)(guest_clinician|doctor|patient)\s*:",
    flags=re.IGNORECASE,
)

def parse_dialogue_turns(text):
    text = str(text)
    matches = list(speaker_pattern.finditer(text))

    if not matches:
        return [], text.strip()

    prefix = text[:matches[0].start()].strip()
    turns = []

    for i, match in enumerate(matches):
        end = (
            matches[i + 1].start()
            if i + 1 < len(matches)
            else len(text)
        )
        turns.append({
            "speaker": match.group(1).lower(),
            "text": text[match.start():end].strip(),
            "body": text[match.end():end].strip(),
        })

    return turns, prefix

parsed_turns = {}
rows = []

for record_id, row in validation.iterrows():
    text = str(row["text"])
    turns, prefix = parse_dialogue_turns(text)
    parsed_turns[record_id] = turns

    reconstructed = " ".join(turn["text"] for turn in turns)

    rows.append({
        "record_id": record_id,
        "turn_count": len(turns),
        "has_unparsed_prefix": bool(prefix),
        "empty_turn_count": sum(not turn["body"] for turn in turns),
        "reconstruction_matches": (
            " ".join(text.split()) == " ".join(reconstructed.split())
        ),
    })

turn_structure = pd.DataFrame(rows).set_index("record_id")

print("=== CORRECTED TURN-PARSING CHECK ===")
print("Queries:", len(turn_structure))
print(
    "No recognized turns:",
    int(turn_structure["turn_count"].eq(0).sum()),
)
print(
    "Fewer than two turns:",
    int(turn_structure["turn_count"].lt(2).sum()),
)
print(
    "Unparsed prefixes:",
    int(turn_structure["has_unparsed_prefix"].sum()),
)
print(
    "Queries with empty turns:",
    int(turn_structure["empty_turn_count"].gt(0).sum()),
)
print(
    "Lossless reconstructions:",
    int(turn_structure["reconstruction_matches"].sum()),
)
print(
    "Turn-count summary:",
    turn_structure["turn_count"].describe().round(1).to_dict(),
)

problem_mask = (
    turn_structure["turn_count"].eq(0)
    | turn_structure["has_unparsed_prefix"]
    | turn_structure["empty_turn_count"].gt(0)
    | ~turn_structure["reconstruction_matches"]
)

turn_parsing_passed = not problem_mask.any()

if not turn_parsing_passed:
    print("\n=== REMAINING PARSING ISSUES ===")
    print(turn_structure.loc[problem_mask].to_string())

    for record_id in turn_structure.index[problem_mask][:5]:
        print("\nRecord:", record_id)
        print(repr(str(validation.loc[record_id, "text"])[:1600]))

print("\nParsing checks passed:", turn_parsing_passed)

print("\n=== SAVED SCRIPTS AND NOTEBOOKS ===")
code_files = sorted(
    path
    for path in ROOT.rglob("*")
    if path.is_file()
    and path.suffix.lower() in {".py", ".ipynb"}
    and ".ipynb_checkpoints" not in path.parts
)

print("Files found:", len(code_files))
for path in code_files:
    print(path.relative_to(ROOT))

print("\n=== TOP-LEVEL PROJECT CONTENTS ===")
for path in sorted(ROOT.iterdir()):
    print(
        "[DIR] " if path.is_dir() else "[FILE]",
        path.name,
    )

print("\n=== SAVED AUDIT SCHEMA ===")
if "audit" in globals() and isinstance(audit, pd.DataFrame):
    print("Columns:", audit.columns.tolist())
    print("Rows:", len(audit))
else:
    print("Audit DataFrame is not present.")

print("\nRead-only inspection complete. No files modified or code files executed.")

In [ ]:
import re
from pathlib import Path
import pandas as pd

if "validation" not in globals():
    raise RuntimeError("Run the validation-input check cell first.")

ROOT = Path(_crus_local_path('project', ''))

speaker_pattern = re.compile(
    r"(?<!\S)(guest_clinician|guest_family|doctor|patient)\s*:",
    flags=re.IGNORECASE,
)

def parse_dialogue_turns(text):
    text = str(text)
    matches = list(speaker_pattern.finditer(text))

    if not matches:
        return [], text.strip()

    prefix = text[:matches[0].start()].strip()
    turns = []

    for i, match in enumerate(matches):
        end = (
            matches[i + 1].start()
            if i + 1 < len(matches)
            else len(text)
        )
        turns.append({
            "speaker": match.group(1).lower(),
            "text": text[match.start():end].strip(),
            "body": text[match.end():end].strip(),
        })

    return turns, prefix

parsed_turns = {}
rows = []

for record_id, row in validation.iterrows():
    text = str(row["text"])
    turns, prefix = parse_dialogue_turns(text)
    parsed_turns[record_id] = turns

    reconstructed = " ".join(turn["text"] for turn in turns)

    rows.append({
        "record_id": record_id,
        "turn_count": len(turns),
        "has_unparsed_prefix": bool(prefix),
        "empty_turn_count": sum(not turn["body"] for turn in turns),
        "reconstruction_matches": (
            " ".join(text.split()) == " ".join(reconstructed.split())
        ),
    })

turn_structure = pd.DataFrame(rows).set_index("record_id")

problem_mask = (
    turn_structure["turn_count"].eq(0)
    | turn_structure["has_unparsed_prefix"]
    | turn_structure["empty_turn_count"].gt(0)
    | ~turn_structure["reconstruction_matches"]
)
turn_parsing_passed = not problem_mask.any()

print("=== FINAL TURN-PARSING CHECK ===")
print("Queries:", len(turn_structure))
print(
    "Lossless reconstructions:",
    int(turn_structure["reconstruction_matches"].sum()),
)
print(
    "Queries with fewer than two turns:",
    int(turn_structure["turn_count"].lt(2).sum()),
)
print("Parsing checks passed:", turn_parsing_passed)

if problem_mask.any():
    print(turn_structure.loc[problem_mask].to_string())
    for record_id in turn_structure.index[problem_mask][:5]:
        print("\nRecord:", record_id)
        print(repr(str(validation.loc[record_id, "text"])[:1200]))

print("\n=== SAVED EXPERIMENT ARTIFACTS ===")

artifact_dirs = [
    ROOT / "corrected_experiment_outputs",
    *sorted(ROOT.glob("r1_10_*")),
    *sorted(ROOT.glob("r1_9_*")),
]

for folder in artifact_dirs:
    if not folder.is_dir():
        continue

    print(f"\nFolder: {folder.relative_to(ROOT)}")
    files = sorted(path for path in folder.rglob("*") if path.is_file())

    for path in files[:60]:
        print(
            f"  {path.relative_to(folder)} "
            f"({path.stat().st_size:,} bytes)"
        )

    if len(files) > 60:
        print(f"  ... {len(files) - 60} additional files")

print("\n=== NOTEBOOKS IN STANDARD DRIVE FOLDER ===")

notebook_root = Path(_crus_local_path('notebooks', ''))
notebook_candidates = []

if notebook_root.is_dir():
    notebooks = sorted(
        path
        for path in notebook_root.rglob("*.ipynb")
        if ".ipynb_checkpoints" not in path.parts
    )

    keywords = (
        "risk", "crus", "cascade", "revision",
        "retrieval", "uncertainty", "r1",
    )

    preferred = [
        path for path in notebooks
        if any(word in path.stem.lower() for word in keywords)
    ]
    notebook_candidates = preferred if preferred else notebooks

    print("Total notebooks:", len(notebooks))
    print("Filename-matched notebooks:", len(preferred))

    for path in notebook_candidates[:60]:
        print(path.relative_to(Path(_crus_local_path('drive', ''))))

    if len(notebook_candidates) > 60:
        print(f"... {len(notebook_candidates) - 60} additional candidates")
else:
    print("The standard 'Colab Notebooks' folder was not found.")

print("\nNo files modified. No discovered code executed. No inference performed.")

In [ ]:
import json
import re
from pathlib import Path

ROOT = Path(_crus_local_path('project', ''))
NOTEBOOK_ROOT = Path(_crus_local_path('notebooks', ''))
OUTPUT_ROOT = ROOT / "corrected_experiment_outputs"

candidate_names = [
    "RiskAware_v2_Historical_Reporting_Audit.ipynb",
    "SK10_clinical_rag_reviewer_revision_v2.ipynb",
    "Sk10_revised2_clinical_RAG_reviewer_revision_pipeline_python_20th_Aug.ipynb",
    "revised_risk_aware_hybrid_pipeline_v2 (1).ipynb",
    "revised_risk_aware_hybrid_pipeline_v2.ipynb",
]

terms = [
    "RETRIEVAL_CFG",
    "r10_baseline_passed",
    "frozen_test_retrieval_checkpoint",
    "H_pred",
    "H_retrieval",
    "V_RRF",
    "def run_cascade",
    "def retrieve",
]

print("=== NOTEBOOK SOURCE SEARCH ===")

source_matches = []

for name in candidate_names:
    path = NOTEBOOK_ROOT / name

    if not path.is_file():
        print(f"\nNOT FOUND: {name}")
        continue

    try:
        notebook = json.loads(path.read_text(encoding="utf-8"))
    except (OSError, ValueError) as exc:
        print(f"\nCould not read {name}: {exc}")
        continue

    matches = []

    for cell_index, cell in enumerate(notebook.get("cells", [])):
        if cell.get("cell_type") != "code":
            continue

        source = cell.get("source", "")
        if isinstance(source, list):
            source = "".join(source)

        hits = [term for term in terms if term in source]
        if hits:
            matches.append((cell_index, source, hits))
            source_matches.append({
                "path": path,
                "cell_index": cell_index,
                "source": source,
                "hits": hits,
            })

    print(f"\nNOTEBOOK: {name}")
    print("Matching code cells:", len(matches))

    for cell_index, source, hits in matches:
        definitions = re.findall(
            r"^\s*(?:async\s+)?def\s+([A-Za-z_]\w*)\s*\(",
            source,
            flags=re.MULTILINE,
        )
        print(
            f"  Cell {cell_index}: "
            f"{len(source):,} characters; hits={hits}"
        )
        if definitions:
            print("    Functions:", ", ".join(definitions))

    # Preview configuration/runtime cells first.
    preferred = [
        item for item in matches
        if (
            "RETRIEVAL_CFG" in item[2]
            or "r10_baseline_passed" in item[2]
        )
    ]

    for cell_index, source, hits in preferred[:2]:
        print(f"\n--- Cell {cell_index}: first 3,500 characters ---")
        print(source[:3500])

print("\n=== SAVED SOURCE AUDIT: STRUCTURE AND MATCHES ===")

source_audits = sorted(
    OUTPUT_ROOT.glob("comment2_saved_source_audit_*.txt")
)

for path in source_audits:
    text = path.read_text(encoding="utf-8", errors="replace")
    lines = text.splitlines()

    print("\nFile:", path.name)
    print("Characters:", len(text))
    print("--- First 20 lines ---")
    print("\n".join(lines[:20]))

    hit_indices = [
        i for i, line in enumerate(lines)
        if any(term in line for term in terms)
    ]

    print("Matching lines:", len(hit_indices))
    for i in hit_indices[:15]:
        print(f"\n--- Lines {max(1, i - 2)}–{min(len(lines), i + 4)} ---")
        print("\n".join(lines[max(0, i - 3):i + 4]))

print("\n=== SAVED CONFIGURATION FILES ===")

config_paths = [
    OUTPUT_ROOT / "retrieval_initialization_metadata.json",
    OUTPUT_ROOT / "final_pretest_protocol.json",
]

for folder in sorted(ROOT.glob("r1_10_*")):
    config_paths.append(folder / "analysis_metadata.json")

for path in config_paths:
    if not path.is_file():
        continue

    print("\nFile:", path.relative_to(ROOT))
    text = path.read_text(encoding="utf-8", errors="replace")

    try:
        formatted = json.dumps(json.loads(text), indent=2)
    except ValueError:
        formatted = text

    print(formatted[:14000])
    if len(formatted) > 14000:
        print("[Remaining content not printed]")

print("\nSource search complete.")
print("No notebook cells executed. No checkpoints loaded. No files modified.")

In [ ]:
import json
from pathlib import Path

notebook_path = Path(
    _crus_local_path('notebooks', 'revised_risk_aware_hybrid_pipeline_v2.ipynb')
)

if not notebook_path.is_file():
    raise FileNotFoundError(notebook_path)

notebook_source = json.loads(
    notebook_path.read_text(encoding="utf-8")
)

def get_saved_cell_source(index):
    cells = notebook_source.get("cells", [])

    if index >= len(cells):
        raise IndexError(
            f"Requested cell {index}; notebook has {len(cells)} cells."
        )

    cell = cells[index]
    if cell.get("cell_type") != "code":
        raise ValueError(f"Cell {index} is not a code cell.")

    source = cell.get("source", "")
    return "".join(source) if isinstance(source, list) else source

for index in (232, 234):
    source = get_saved_cell_source(index)
    print("\n" + "=" * 80)
    print(f"SAVED CELL {index} — {len(source):,} CHARACTERS")
    print("=" * 80)
    print(source)

print("\nInspection only: no saved cells executed or files modified.")

In [ ]:
import json
from pathlib import Path

notebook_path = Path(
    _crus_local_path('notebooks', 'revised_risk_aware_hybrid_pipeline_v2.ipynb')
)

if not notebook_path.is_file():
    raise FileNotFoundError(notebook_path)

saved_notebook = json.loads(
    notebook_path.read_text(encoding="utf-8")
)

# Zero-based index 232 is the restoration cell inspected above.
cell_index = 232
cell = saved_notebook["cells"][cell_index]

if cell.get("cell_type") != "code":
    raise RuntimeError("The selected restoration cell is not a code cell.")

source = cell.get("source", "")
restore_source = (
    "".join(source) if isinstance(source, list) else source
)

# Stop if the notebook cell has changed to something unexpected.
required_fragments = [
    'def source_of(number):',
    'def literal_assignment(number, variable):',
    'expected_functions = {',
    'Reproducing Cascade validation baseline',
    'r10_baseline_passed = False',
    'r10_validation = validation.copy()',
    'r10_baseline_results = baseline_results',
]

missing = [
    fragment
    for fragment in required_fragments
    if fragment not in restore_source
]

if missing:
    raise RuntimeError(
        "The saved restoration cell differs from the inspected version. "
        f"Missing expected fragments: {missing}"
    )

# Prevent a failed restoration from leaving an old passing status.
r10_baseline_passed = False

print("Restoring the original runtime and checking clean validation...")
print("Only the inspected restoration cell will be executed.")

exec(
    compile(
        restore_source,
        f"{notebook_path.name}:cell_index_{cell_index}",
        "exec",
    ),
    globals(),
)

if not globals().get("r10_baseline_passed", False):
    raise RuntimeError(
        "Baseline verification did not pass. "
        "Do not run perturbations."
    )

print("\n=== READY FOR R1.11 ===")
print("Baseline verification passed:", r10_baseline_passed)
print("Validation queries:", len(r10_validation))
print("Configuration:", r10["RETRIEVAL_CFG"])
print("Uncertainty configuration:", r10["UNCERTAINTY_CFG"])
print("No perturbations have been evaluated yet.")

In [ ]:
%pip install -q "rank-bm25==0.2.2"

from importlib.metadata import version
from rank_bm25 import BM25Okapi

print("Installed rank-bm25:", version("rank-bm25"))

if "restore_source" not in globals():
    raise RuntimeError(
        "restore_source is missing. Rerun the previous restoration cell."
    )

r10_baseline_passed = False

exec(
    compile(
        restore_source,
        "verified_baseline_restoration",
        "exec",
    ),
    globals(),
)

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import copy
import hashlib
import json
import re

import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score
from tqdm.auto import tqdm

# ---------- Preconditions ----------

if not globals().get("r10_baseline_passed", False):
    raise RuntimeError("The clean baseline must reproduce before perturbation.")

required = ["r10", "r10_validation", "r10_baseline_results"]
missing = [name for name in required if name not in globals()]
if missing:
    raise RuntimeError(f"Missing runtime objects: {missing}")

ROOT = Path(_crus_local_path('project', ''))
BASE = ROOT / "corrected_experiment_outputs"

protocol = json.loads(
    (BASE / "final_pretest_protocol.json").read_text(encoding="utf-8")
)

frozen_config = copy.deepcopy(r10["RETRIEVAL_CFG"])
if frozen_config != protocol["retrieval_configuration"]:
    raise RuntimeError("Runtime differs from frozen retrieval configuration.")

weights = np.asarray(
    protocol["uncertainty_configuration"]["weights"], dtype=float
)
if not np.allclose(weights, [0.5, 0.3, 0.2]):
    raise RuntimeError("Unexpected uncertainty weights.")

policy = protocol["frozen_policies"]["Cascade:CRUS"]
tau_c = float(policy["tau_c"])
tau_u = float(policy["tau_u"])
classes = list(protocol["classes"])

queries = r10_validation.copy()
if len(queries) != len(r10_baseline_results):
    raise RuntimeError("Baseline result count does not match validation.")

queries.index = queries.index.astype(str)

saved_audit = pd.read_csv(
    BASE / "validation_retrieval_prediction_audit.csv",
    dtype={"record_id": str},
)
reference = saved_audit.loc[
    saved_audit["method"].eq("Cascade")
].set_index("record_id").loc[queries.index]

clean_components = np.asarray(
    [result["components"] for result in r10_baseline_results],
    dtype=float,
)

if clean_components.shape != (len(queries), 3):
    raise RuntimeError("Unexpected uncertainty-component dimensions.")

clean_crus = clean_components @ weights

if not np.allclose(
    clean_crus,
    reference["CRUS"].to_numpy(dtype=float),
    rtol=1e-5,
    atol=1e-7,
):
    raise RuntimeError("Weighted components do not reproduce saved CRUS.")

def result_row(record_id, target, result):
    components = np.asarray(result["components"], dtype=float)
    if components.shape != (3,) or not np.isfinite(components).all():
        raise RuntimeError(f"Invalid components: {record_id}")

    confidence = float(result["confidence"])
    crus = float(components @ weights)

    return {
        "record_id": str(record_id),
        "true_label": str(target),
        "prediction": str(result["pred"]),
        "correct": str(result["pred"]) == str(target),
        "confidence": confidence,
        "H_pred": float(components[0]),
        "H_retrieval": float(components[1]),
        "V_RRF": float(components[2]),
        "CRUS": crus,
        "accepted": confidence >= tau_c and crus <= tau_u,
        "fallback": bool(result["fallback"]),
    }

clean = pd.DataFrame([
    result_row(record_id, row["label"], result)
    for (record_id, row), result
    in zip(queries.iterrows(), r10_baseline_results)
]).set_index("record_id")

if (
    int(clean["accepted"].sum()) != int(policy["n_accepted"])
    or int((clean["accepted"] & clean["correct"]).sum())
    != int(policy["n_correct_accepted"])
):
    raise RuntimeError("Frozen gate does not reproduce saved clean counts.")

# ---------- Lossless turn parsing ----------

speaker_pattern = re.compile(
    r"(?<!\S)(guest_clinician|guest_family|doctor|patient)\s*:",
    flags=re.IGNORECASE,
)

def split_turns(text):
    matches = list(speaker_pattern.finditer(text))
    if not matches or text[:matches[0].start()].strip():
        raise ValueError("Unrecognized dialogue prefix.")

    turns = [
        text[match.start():
             matches[i + 1].start() if i + 1 < len(matches) else len(text)
        ].strip()
        for i, match in enumerate(matches)
    ]

    if " ".join(text.split()) != " ".join(" ".join(turns).split()):
        raise ValueError("Turn parsing lost text.")
    if len(turns) < 2:
        raise ValueError("Turn removal requires at least two turns.")

    return turns

turns_by_id = {
    record_id: split_turns(str(row["text"]))
    for record_id, row in queries.iterrows()
}

# ---------- Reproducible perturbations ----------

def query_rng(record_id, seed, family):
    key = f"{record_id}|{seed}|{family}".encode("utf-8")
    value = int.from_bytes(hashlib.sha256(key).digest()[:8], "little")
    return np.random.default_rng(value)

word_pattern = re.compile(r"\b[A-Za-z]+\b")
protected = {
    "no", "not", "never", "none", "neither", "nor", "without",
    "denies", "denied", "negative",
}

def spelling_noise(text, rate, rng):
    speaker_spans = [match.span() for match in speaker_pattern.finditer(text)]
    eligible = []

    for match in word_pattern.finditer(text):
        word = match.group()
        if len(word) < 4 or word.lower() in protected:
            continue
        if any(match.start() < end and match.end() > start
               for start, end in speaker_spans):
            continue
        swaps = [
            i for i in range(len(word) - 1)
            if word[i].lower() != word[i + 1].lower()
        ]
        if swaps:
            eligible.append((match, swaps))

    if not eligible:
        return text, {
            "eligible_words": 0,
            "edited_words": 0,
            "realized_word_edit_rate": 0.0,
        }

    n_edit = min(len(eligible), max(1, int(np.floor(rate * len(eligible) + 0.5))))
    selected = rng.choice(len(eligible), size=n_edit, replace=False)
    edits = []

    for index in selected:
        match, swaps = eligible[int(index)]
        position = int(rng.choice(swaps))
        letters = list(match.group())
        letters[position], letters[position + 1] = (
            letters[position + 1], letters[position]
        )
        edits.append((match.start(), match.end(), "".join(letters)))

    altered = text
    for start, end, replacement in sorted(edits, reverse=True):
        altered = altered[:start] + replacement + altered[end:]

    return altered, {
        "eligible_words": len(eligible),
        "edited_words": n_edit,
        "realized_word_edit_rate": n_edit / len(eligible),
    }

def perturb(record_id, text, family, level, seed):
    rng = query_rng(record_id, seed, family)

    if family == "spelling":
        return spelling_noise(text, level, rng)

    turns = turns_by_id[record_id]
    n_turns = len(turns)

    if family == "turn_removal":
        n_remove = min(
            n_turns - 1,
            max(1, int(np.floor(level * n_turns + 0.5))),
        )
        removed = set(
            rng.choice(n_turns, size=n_remove, replace=False).tolist()
        )
        retained = [
            turn for i, turn in enumerate(turns) if i not in removed
        ]
    elif family == "prefix":
        n_keep = max(1, min(n_turns - 1, int(np.floor(level * n_turns))))
        retained = turns[:n_keep]
        n_remove = n_turns - n_keep
    else:
        raise ValueError(f"Unknown perturbation: {family}")

    return " ".join(retained), {
        "original_turns": n_turns,
        "retained_turns": len(retained),
        "removed_turns": n_remove,
        "realized_turn_removal_rate": n_remove / n_turns,
    }

seeds = [42, 43, 44]
conditions = [
    (family, level, seed)
    for family, levels in [
        ("spelling", [0.05, 0.10]),
        ("turn_removal", [0.10, 0.25]),
    ]
    for level in levels
    for seed in seeds
] + [
    ("prefix", 0.75, 42),
    ("prefix", 0.50, 42),
]

# ---------- New output directory ----------

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
outdir = ROOT / f"r1_11_validation_perturbations_{stamp}"
outdir.mkdir(parents=True, exist_ok=False)

clean.reset_index().to_csv(outdir / "clean_baseline.csv", index=False)

metadata = {
    "analysis": "R1.11 controlled query perturbations",
    "partition": "constrained validation",
    "n_queries": len(queries),
    "configuration": frozen_config,
    "uncertainty_configuration": protocol["uncertainty_configuration"],
    "frozen_policy": policy,
    "seeds": seeds,
    "conditions": [
        {"family": family, "level": level, "seed": seed}
        for family, level, seed in conditions
    ],
    "turn_removal_rounding": "nearest integer, minimum 1, retain at least 1",
    "prefix_rounding": "floor retained count, retain at least 1",
    "spelling_rounding": "nearest integer, minimum 1 when eligible",
    "spelling_operation": "one unequal adjacent-letter transposition per selected word",
    "limitations": [
        "Synthetic validation stress test, not external clinical validation",
        "Clinical meaning is not guaranteed to be preserved by spelling errors",
        "Incomplete queries retain the original full-dialogue target",
        "Small turn counts can produce realized rates above nominal removal rates",
        "No paraphrase condition evaluated",
    ],
}

(outdir / "analysis_metadata.json").write_text(
    json.dumps(metadata, indent=2), encoding="utf-8"
)

# ---------- Evaluation ----------

def summarize(frame):
    accepted = frame["accepted"].astype(bool)
    correct = frame["correct"].astype(bool)
    accepted_n = int(accepted.sum())

    return {
        "n": len(frame),
        "accuracy": accuracy_score(frame["true_label"], frame["prediction"]),
        "macro_f1": f1_score(
            frame["true_label"], frame["prediction"],
            labels=classes, average="macro", zero_division=0,
        ),
        "mean_CRUS": frame["CRUS"].mean(),
        "coverage": accepted.mean(),
        "n_accepted": accepted_n,
        "accepted_errors": int((accepted & ~correct).sum()),
        "selective_accuracy": (
            float(correct[accepted].mean()) if accepted_n else np.nan
        ),
    }

summaries = [{"condition": "clean", "seed": -1, **summarize(clean)}]
all_frames = []

for family, level, seed in conditions:
    condition = f"{family}_{int(round(level * 100)):02d}"
    run_name = f"{condition}_seed{seed}"

    if r10["RETRIEVAL_CFG"] != frozen_config:
        raise RuntimeError("Runtime configuration changed.")

    records = []

    for record_id, row in tqdm(
        queries.iterrows(), total=len(queries), desc=run_name
    ):
        original = str(row["text"])
        altered, edit_metadata = perturb(
            record_id, original, family, level, seed
        )

        candidates = r10["prepare_candidates"](altered)
        result = r10["score_documents"](altered, candidates, "Cascade")
        item = result_row(record_id, row["label"], result)
        old = clean.loc[record_id]

        item.update({
            "condition": condition,
            "family": family,
            "level": level,
            "seed": seed,
            "perturbed_text": altered,
            "changed": altered != original,
            "original_characters": len(original),
            "perturbed_characters": len(altered),
            "clean_correct": bool(old["correct"]),
            "clean_CRUS": float(old["CRUS"]),
            "delta_CRUS": item["CRUS"] - float(old["CRUS"]),
            "transition": (
                ("correct" if old["correct"] else "incorrect")
                + "_to_"
                + ("correct" if item["correct"] else "incorrect")
            ),
            **edit_metadata,
        })
        records.append(item)

    frame = pd.DataFrame(records)
    frame.to_csv(outdir / f"{run_name}_per_query.csv", index=False)
    all_frames.append(frame)

    newly_wrong = frame["transition"].eq("correct_to_incorrect")

    summaries.append({
        "condition": condition,
        "seed": seed,
        **summarize(frame),
        "mean_delta_CRUS": frame["delta_CRUS"].mean(),
        "changed_fraction": frame["changed"].mean(),
        "new_errors": int(newly_wrong.sum()),
        "new_error_mean_delta_CRUS": (
            frame.loc[newly_wrong, "delta_CRUS"].mean()
        ),
        "new_error_fraction_CRUS_increased": (
            frame.loc[newly_wrong, "delta_CRUS"].gt(0).mean()
            if newly_wrong.any() else np.nan
        ),
    })

    pd.DataFrame(summaries).to_csv(
        outdir / "per_run_summary.csv", index=False
    )

# ---------- Combined descriptive results ----------

per_query = pd.concat(all_frames, ignore_index=True)
per_query.to_csv(outdir / "all_perturbed_queries.csv", index=False)

summary = pd.DataFrame(summaries)
numeric_columns = [
    "accuracy", "macro_f1", "mean_CRUS", "mean_delta_CRUS",
    "coverage", "selective_accuracy", "accepted_errors",
    "new_errors", "new_error_mean_delta_CRUS",
    "new_error_fraction_CRUS_increased", "changed_fraction",
]

condition_summary = (
    summary.groupby("condition", sort=False)[numeric_columns]
    .mean()
    .reset_index()
)
condition_summary.to_csv(outdir / "condition_summary.csv", index=False)

transition_summary = (
    per_query.groupby(["condition", "transition"], sort=False)
    .agg(
        n_query_realizations=("record_id", "size"),
        mean_delta_CRUS=("delta_CRUS", "mean"),
        fraction_CRUS_increased=("delta_CRUS", lambda x: x.gt(0).mean()),
    )
    .reset_index()
)
transition_summary.to_csv(outdir / "transition_summary.csv", index=False)

realized_rates = (
    per_query.groupby("condition", sort=False)[
        ["realized_word_edit_rate", "realized_turn_removal_rate"]
    ]
    .agg(["mean", "min", "max"])
)
realized_rates.to_csv(outdir / "realized_perturbation_rates.csv")

(outdir / "RUN_COMPLETE.json").write_text(
    json.dumps({
        "completed_runs": len(conditions),
        "perturbed_query_evaluations": len(per_query),
        "configuration_unchanged": r10["RETRIEVAL_CFG"] == frozen_config,
    }, indent=2),
    encoding="utf-8",
)

print("\n=== CONDITION SUMMARY: MEAN ACROSS SEEDS ===")
print(condition_summary.round(6).to_string(index=False))

print("\n=== CLEAN-TO-PERTURBED TRANSITIONS ===")
print(transition_summary.round(6).to_string(index=False))

print("\n=== REALIZED PERTURBATION RATES ===")
print(realized_rates.round(4).to_string())

print("\nSaved new experiment folder:", outdir)
print("Descriptive results only; no significance tests performed.")
print("Repeated seeds are not independent validation samples.")

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

ROOT = Path(_crus_local_path('project', ''))
BASE = ROOT / "corrected_experiment_outputs"

if not BASE.is_dir():
    raise FileNotFoundError(BASE)

print("=== TOP-LEVEL EXPERIMENT FILES ===")
for path in sorted(BASE.iterdir()):
    if path.is_file():
        print(f"{path.name} ({path.stat().st_size:,} bytes)")

print("\n=== TOP-LEVEL NPZ ARRAY SCHEMAS ===")
for path in sorted(BASE.glob("*.npz")):
    print("\nFile:", path.name)

    try:
        with np.load(path, allow_pickle=False) as saved:
            for key in saved.files:
                try:
                    array = saved[key]
                    print(
                        f"  {key}: shape={array.shape}, "
                        f"dtype={array.dtype}"
                    )
                except ValueError as exc:
                    print(f"  {key}: not loaded ({exc})")
    except Exception as exc:
        print(f"  Could not inspect: {type(exc).__name__}: {exc}")

print("\n=== SAVED SUPERVISED CALIBRATION CONFIGURATION ===")
config_path = BASE / "supervised_baseline_configuration.json"

if config_path.is_file():
    config = json.loads(config_path.read_text(encoding="utf-8"))
    print(json.dumps(config, indent=2))
else:
    print("Configuration file not found.")

print("\n=== EXISTING CALIBRATION METRICS ===")
metrics_path = BASE / "comment7_audit/comment7_recomputed_metrics.csv"

if metrics_path.is_file():
    frame = pd.read_csv(metrics_path)
    print("Columns:", frame.columns.tolist())
    print(frame.to_string(index=False))
else:
    print("Existing metric file not found.")

print("\n=== EXISTING RELIABILITY-BIN SCHEMA ===")
bins_path = BASE / "comment7_audit/comment7_reliability_bins.csv"

if bins_path.is_file():
    frame = pd.read_csv(bins_path)
    print("Columns:", frame.columns.tolist())
    print(frame.head(10).to_string(index=False))
else:
    print("Existing reliability-bin file not found.")

print("\n=== SAVED CALIBRATION SOURCE: CELL SUMMARIES ===")
notebook_path = Path(
    _crus_local_path('notebooks', 'revised_risk_aware_hybrid_pipeline_v2.ipynb')
)

if notebook_path.is_file():
    notebook = json.loads(notebook_path.read_text(encoding="utf-8"))

    terms = [
        "CalibratedClassifierCV",
        "calibration_manifest",
        "FrozenEstimator",
        "cv=\"prefit\"",
        "cv='prefit'",
        "log_loss(",
        "brier",
    ]

    matches = []

    for index, cell in enumerate(notebook.get("cells", [])):
        if cell.get("cell_type") != "code":
            continue

        source = cell.get("source", "")
        source = "".join(source) if isinstance(source, list) else source
        hits = [term for term in terms if term.lower() in source.lower()]

        if hits:
            matches.append((index, source, hits))
            print(
                f"Cell {index}: {len(source):,} characters; "
                f"hits={hits}"
            )

    print("\n=== CALIBRATOR CONSTRUCTION/FITTING CONTEXT ===")
    printed = 0

    for index, source, hits in matches:
        if "CalibratedClassifierCV" not in source:
            continue

        lines = source.splitlines()
        hit_lines = [
            i for i, line in enumerate(lines)
            if "CalibratedClassifierCV" in line or ".fit(" in line
        ]

        selected_lines = set()
        for i in hit_lines:
            selected_lines.update(
                range(max(0, i - 5), min(len(lines), i + 9))
            )

        print(f"\n--- Cell {index} ---")
        for i in sorted(selected_lines)[:100]:
            print(f"{i + 1:03d}: {lines[i]}")

        printed += 1
        if printed >= 4:
            break
else:
    print("Source notebook not found.")

print("\nRead-only inspection complete. No calibration fitted.")

In [ ]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path(_crus_local_path('project', ''))
BASE = ROOT / "corrected_experiment_outputs"
NOTEBOOK = Path(
    _crus_local_path('notebooks', 'revised_risk_aware_hybrid_pipeline_v2.ipynb')
)

nb = json.loads(NOTEBOOK.read_text(encoding="utf-8"))

def saved_source(index):
    cell = nb["cells"][index]
    source = cell.get("source", "")
    return "".join(source) if isinstance(source, list) else source

print("=== LATER SVM CALIBRATION SOURCE: CELL 159 ===")
print(saved_source(159))

print("\n=== GROUPED CALIBRATION SETUP: CELL 62 ===")
print(saved_source(62))

print("\n=== EXISTING TEST ACCEPTANCE AUDIT ===")
path = BASE / "test_frozen_CRUS_acceptance_audit.csv"
frame = pd.read_csv(path, dtype={"record_id": str})
print("Shape:", frame.shape)
print("Columns:", frame.columns.tolist())
print(frame.head(8).to_string(index=False))

if "method" in frame.columns:
    print("\nMethods:", frame["method"].dropna().unique().tolist())

print("\n=== SAVED VALIDATION POLICIES ===")
path = BASE / "validation_selected_policies.json"
print(path.read_text(encoding="utf-8"))

print("\n=== CALIBRATION SPLIT METADATA ===")
path = BASE / "constrained_calibration_metadata.json"
print(path.read_text(encoding="utf-8"))

print("\n=== EXISTING CALIBRATION ANALYSIS: CELL 206 EXCERPTS ===")
source = saved_source(206)
lines = source.splitlines()

terms = [
    "def ",
    "accepted",
    "confidence",
    "brier",
    "probabilities",
    "allow_pickle",
    "calibration",
]

selected = set()
for i, line in enumerate(lines):
    if any(term in line.lower() for term in terms):
        selected.update(range(max(0, i - 2), min(len(lines), i + 4)))

for i in sorted(selected)[:180]:
    print(f"{i + 1:03d}: {lines[i]}")

print("\nInspection complete. No fitting or file modification performed.")

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import hashlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize_scalar
from scipy.special import logsumexp

ROOT = Path(_crus_local_path('project', ''))
BASE = ROOT / "corrected_experiment_outputs"

# Enable only for your own trusted experiment files.
TRUST_LOCAL_ARRAYS = True
EPS = 1e-12
N_BINS = 10

methods = [
    "TFIDF_LR", "TFIDF_SVM", "SK4",
    "SingleStageFull", "SingleStageTruncated",
    "Cascade", "LengthNormLegacy",
]

protocol_path = BASE / "final_pretest_protocol.json"
protocol = json.loads(protocol_path.read_text(encoding="utf-8"))
classes = np.asarray(protocol["classes"], dtype=str)
weights = np.asarray(
    protocol["uncertainty_configuration"]["weights"], dtype=float
)
policies = protocol["frozen_policies"]

manifest = pd.read_csv(
    ROOT / "data_audit/constrained_split_manifest.csv",
    dtype={"record_id": str},
).set_index("record_id")

if not manifest.index.is_unique:
    raise ValueError("Duplicate manifest IDs.")

acceptance_audit = pd.read_csv(
    BASE / "test_frozen_CRUS_acceptance_audit.csv",
    dtype={"record_id": str},
)

def file_hash(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

input_hashes = {
    str(protocol_path.relative_to(ROOT)): file_hash(protocol_path)
}

def load_field(path, key):
    with np.load(path, allow_pickle=False) as saved:
        try:
            return saved[key].copy()
        except ValueError as exc:
            if "Object arrays cannot be loaded" not in str(exc):
                raise

    if not TRUST_LOCAL_ARRAYS:
        raise RuntimeError("Trusted-file permission required for object IDs.")

    with np.load(path, allow_pickle=True) as saved:
        value = saved[key].copy()

    if value.ndim != 1 or not all(isinstance(x, str) for x in value):
        raise ValueError(f"Unexpected object values: {path.name}/{key}")
    return value.astype(str)

def load_run(split, method):
    suffix = (
        "probabilities"
        if split == "validation" and method.startswith("TFIDF_")
        else "arrays"
    )
    path = BASE / f"{split}_{method}_{suffix}.npz"
    input_hashes[str(path.relative_to(ROOT))] = file_hash(path)

    with np.load(path, allow_pickle=False) as saved:
        keys = set(saved.files)

    required = [
        "record_ids", "true_labels", "classes",
        "predictions", "probabilities", "confidence",
    ]
    data = {key: load_field(path, key) for key in required}

    for key in ("record_ids", "true_labels", "classes", "predictions"):
        data[key] = data[key].astype(str)

    p = np.asarray(data["probabilities"], dtype=float)
    ids = data["record_ids"]

    if len(set(ids)) != len(ids):
        raise ValueError(f"Duplicate IDs: {path.name}")
    if not np.array_equal(data["classes"], classes):
        raise ValueError(f"Class-order mismatch: {path.name}")
    if p.shape != (len(ids), len(classes)):
        raise ValueError(f"Probability-shape mismatch: {path.name}")
    if (
        not np.isfinite(p).all()
        or (p < 0).any()
        or (p > 1).any()
        or not np.allclose(p.sum(axis=1), 1.0)
    ):
        raise ValueError(f"Invalid probabilities: {path.name}")

    expected = manifest.loc[manifest["split"].eq(split)]
    if set(ids) != set(expected.index):
        raise ValueError(f"Split-ID mismatch: {path.name}")
    if not np.array_equal(
        data["true_labels"], expected.loc[ids, "label"].astype(str).to_numpy()
    ):
        raise ValueError(f"Target mismatch: {path.name}")
    if not np.array_equal(classes[p.argmax(axis=1)], data["predictions"]):
        raise ValueError(f"Prediction/probability mismatch: {path.name}")
    if not np.allclose(p.max(axis=1), data["confidence"]):
        raise ValueError(f"Confidence mismatch: {path.name}")

    data["probabilities"] = p
    data["target_index"] = np.asarray([
        np.flatnonzero(classes == label)[0]
        for label in data["true_labels"]
    ])

    if "components" in keys:
        data["components"] = load_field(path, "components").astype(float)

    return data

def as_bool(series):
    mapped = series.astype(str).str.lower().str.strip().map({
        "true": True, "false": False,
        "1": True, "0": False, "1.0": True, "0.0": False,
    })
    if mapped.isna().any():
        raise ValueError("Unrecognized Boolean values.")
    return mapped.to_numpy(dtype=bool)

def original_gate(data, method, split):
    key = f"{method}:CRUS"
    if key not in policies:
        return None

    policy = policies[key]
    components = data["components"]
    if components.shape != (len(data["record_ids"]), 3):
        raise ValueError("Invalid component dimensions.")

    mask = (
        (data["confidence"] >= float(policy["tau_c"]))
        & (components @ weights <= float(policy["tau_u"]))
    )

    correct = data["predictions"] == data["true_labels"]

    if split == "validation":
        if (
            int(mask.sum()) != int(policy["n_accepted"])
            or int(correct[mask].sum()) != int(policy["n_correct_accepted"])
        ):
            raise ValueError(f"Validation gate mismatch: {method}")
    else:
        saved = acceptance_audit.loc[
            acceptance_audit["method"].eq(method)
        ].set_index("record_id")
        if not saved.index.is_unique or set(saved.index) != set(data["record_ids"]):
            raise ValueError(f"Test acceptance IDs mismatch: {method}")
        saved = saved.loc[data["record_ids"]]
        if not np.array_equal(mask, as_bool(saved["accepted"])):
            raise ValueError(f"Test gate mismatch: {method}")
        if not np.array_equal(correct, as_bool(saved["correct"])):
            raise ValueError(f"Test correctness mismatch: {method}")

    return mask

def log_probabilities(p):
    q = np.maximum(p, EPS)
    q = q / q.sum(axis=1, keepdims=True)
    return np.log(q)

def scale_probabilities(p, temperature):
    logits = log_probabilities(p) / temperature
    return np.exp(logits - logsumexp(logits, axis=1, keepdims=True))

def fit_temperature(p, target_index):
    log_p = log_probabilities(p)

    def objective(log_temperature):
        logits = log_p / np.exp(log_temperature)
        log_q = logits - logsumexp(logits, axis=1, keepdims=True)
        return float(-log_q[np.arange(len(target_index)), target_index].mean())

    lower, upper = np.log(0.05), np.log(20.0)
    fit = minimize_scalar(
        objective, bounds=(lower, upper), method="bounded",
        options={"xatol": 1e-8},
    )
    if not fit.success:
        raise RuntimeError(f"Temperature fitting failed: {fit.message}")

    return float(np.exp(fit.x)), bool(
        abs(fit.x - lower) < 1e-3 or abs(fit.x - upper) < 1e-3
    )

def evaluate(p, targets):
    n = len(targets)
    if n == 0:
        raise ValueError("Cannot evaluate an empty accepted subset.")

    prediction = p.argmax(axis=1)
    confidence = p.max(axis=1)
    correct = prediction == targets
    bin_index = np.minimum((confidence * N_BINS).astype(int), N_BINS - 1)

    bins = []
    ece = 0.0

    for b in range(N_BINS):
        mask = bin_index == b
        count = int(mask.sum())
        accuracy = float(correct[mask].mean()) if count else np.nan
        mean_conf = float(confidence[mask].mean()) if count else np.nan
        if count:
            ece += count / n * abs(accuracy - mean_conf)
        bins.append({
            "bin": b, "lower": b / N_BINS,
            "upper": (b + 1) / N_BINS,
            "upper_inclusive": b == N_BINS - 1,
            "count": count, "accuracy": accuracy,
            "mean_confidence": mean_conf,
        })

    one_hot = np.eye(len(classes))[targets]
    metrics = {
        "n": n,
        "accuracy": float(correct.mean()),
        "ECE": float(ece),
        "Brier": float(np.mean(np.sum((p - one_hot) ** 2, axis=1))),
        "NLL": float(
            -log_probabilities(p)[np.arange(n), targets].mean()
        ),
        "zero_true_label_probabilities": int(
            (p[np.arange(n), targets] == 0).sum()
        ),
    }
    return metrics, bins

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
outdir = ROOT / f"r1_12_calibration_{stamp}"
outdir.mkdir(parents=True, exist_ok=False)

metric_rows, bin_rows, temperature_rows = [], [], []
plot_data = {}

for method in methods:
    val = load_run("validation", method)
    test = load_run("test", method)

    if set(val["record_ids"]) & set(test["record_ids"]):
        raise ValueError("Validation/test ID overlap.")

    original_gate(val, method, "validation")
    accepted = original_gate(test, method, "test")

    temperature, boundary = fit_temperature(
        val["probabilities"], val["target_index"]
    )

    val_original, _ = evaluate(val["probabilities"], val["target_index"])
    val_scaled, _ = evaluate(
        scale_probabilities(val["probabilities"], temperature),
        val["target_index"],
    )

    temperature_rows.append({
        "method": method, "temperature": temperature,
        "near_search_boundary": boundary,
        "validation_NLL_original": val_original["NLL"],
        "validation_NLL_temperature_scaled": val_scaled["NLL"],
    })

    variants = {
        "original": test["probabilities"],
        "validation_temperature": scale_probabilities(
            test["probabilities"], temperature
        ),
    }
    subsets = {"full_test": np.ones(len(test["record_ids"]), dtype=bool)}
    if accepted is not None:
        subsets["accepted_fixed_original_gate"] = accepted

    for variant, p in variants.items():
        if not np.array_equal(
            p.argmax(axis=1), test["probabilities"].argmax(axis=1)
        ):
            raise RuntimeError(f"Temperature scaling changed labels: {method}")

        for subset, mask in subsets.items():
            metrics, bins = evaluate(p[mask], test["target_index"][mask])
            identity = {
                "method": method, "probability_variant": variant,
                "subset": subset,
            }
            metric_rows.append({
                **identity, **metrics,
                "coverage": float(mask.mean()),
                "status": "Available",
            })
            bin_rows.extend([{**identity, **row} for row in bins])
            plot_data[(method, variant, subset)] = pd.DataFrame(bins)

    print(f"{method}: T={temperature:.6f}; gate verified")

for method in ["BM25", "SBERT"]:
    metric_rows.append({
        "method": method, "probability_variant": "not_applicable",
        "subset": "full_test", "n": int(protocol["split_sizes"]["test"]),
        "coverage": 1.0, "ECE": np.nan, "Brier": np.nan, "NLL": np.nan,
        "status": "N/A: no class-probability output",
    })

metrics = pd.DataFrame(metric_rows)
bins = pd.DataFrame(bin_rows)
temperatures = pd.DataFrame(temperature_rows)

metrics.to_csv(outdir / "test_calibration_metrics.csv", index=False)
bins.to_csv(outdir / "test_reliability_bins.csv", index=False)
temperatures.to_csv(outdir / "validation_temperature_parameters.csv", index=False)

for variant in ["original", "validation_temperature"]:
    fig, axes = plt.subplots(3, 3, figsize=(13, 12))
    for ax, method in zip(axes.flat, methods):
        for subset, label, color in [
            ("full_test", "Full test", "tab:blue"),
            ("accepted_fixed_original_gate", "Accepted: fixed gate", "tab:orange"),
        ]:
            data = plot_data.get((method, variant, subset))
            if data is None:
                continue
            occupied = data.loc[data["count"].gt(0)]
            ax.plot(
                occupied["mean_confidence"], occupied["accuracy"],
                marker="o", color=color, label=label,
            )
        ax.plot([0, 1], [0, 1], "--", color="gray", linewidth=1)
        ax.set(
            title=method, xlim=(0, 1), ylim=(0, 1),
            xlabel="Mean top-label probability", ylabel="Accuracy",
        )
        ax.grid(alpha=0.2)
        ax.legend(fontsize=8)

    for ax in axes.flat[len(methods):]:
        ax.axis("off")

    fig.suptitle(
        "Test reliability: " + variant.replace("_", " "), fontsize=14
    )
    fig.tight_layout(rect=[0, 0, 1, 0.96])
    fig.savefig(outdir / f"reliability_{variant}.pdf", bbox_inches="tight")
    fig.savefig(
        outdir / f"reliability_{variant}.png",
        dpi=250, bbox_inches="tight",
    )
    plt.close(fig)

metadata = {
    "analysis": "R1.12 calibration clarification",
    "evaluation_split": "test",
    "posthoc_fitting_split": "validation only",
    "posthoc_method": "single scalar temperature per method",
    "temperature_bounds": [0.05, 20.0],
    "objective": "validation multiclass NLL",
    "probability_floor": EPS,
    "Brier_definition": "mean sum over classes of squared probability errors",
    "ECE_definition": "top-label ECE; 10 equal-width bins; include confidence=1",
    "NLL_definition": "mean negative log true-class probability after flooring and renormalization",
    "acceptance": "original frozen masks retained for both probability variants",
    "original_SVM": "training-only grouped sigmoid calibration; not validation-only",
    "retrieval_only": "BM25/SBERT metrics unavailable, not zero",
    "interpretation": "accepted-subset calibration is conditional on selection",
    "analysis_status": "additional revision-stage analysis, not replacement of original models",
    "input_hashes": input_hashes,
}

(outdir / "analysis_metadata.json").write_text(
    json.dumps(metadata, indent=2), encoding="utf-8"
)
(outdir / "RUN_COMPLETE.json").write_text(
    json.dumps({"completed_methods": len(methods)}, indent=2),
    encoding="utf-8",
)

columns = [
    "method", "probability_variant", "subset",
    "n", "coverage", "ECE", "Brier", "NLL",
    "zero_true_label_probabilities",
]

print("\n=== TEST CALIBRATION RESULTS ===")
print(metrics[columns].round(6).to_string(index=False))

print("\n=== VALIDATION-FITTED TEMPERATURES ===")
print(temperatures.round(6).to_string(index=False))

print("\nSaved new analysis folder:", outdir)
print("Original models, probabilities, and policies were not overwritten.")

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from IPython.display import display, Image

FOLDER = Path(
    _crus_local_path('project', 'r1_12_calibration_20261016T142502223934Z')
)

bins_path = FOLDER / "test_reliability_bins.csv"
metrics_path = FOLDER / "test_calibration_metrics.csv"

for path in [bins_path, metrics_path]:
    if not path.is_file():
        raise FileNotFoundError(path)

bins = pd.read_csv(bins_path)
metrics = pd.read_csv(metrics_path)

bins = bins.loc[
    bins["probability_variant"].eq("original")
].copy()

metrics = metrics.loc[
    metrics["probability_variant"].eq("original")
].copy()

panels = [
    ("TFIDF_LR", "TF-IDF + LR"),
    ("TFIDF_SVM", "TF-IDF + SVM"),
    ("SK4", "SK4"),
    ("SingleStageFull", "Single-stage full"),
    ("SingleStageTruncated", "Single-stage truncated"),
    ("Cascade", "Cascade"),
    ("LengthNormLegacy", "LengthNormLegacy"),
]

styles = {
    "full_test": {
        "label": "Full test",
        "color": "#0072B2",
        "marker": "o",
        "linestyle": "-",
    },
    "accepted_fixed_original_gate": {
        "label": "Accepted subset",
        "color": "#D55E00",
        "marker": "s",
        "linestyle": "--",
    },
}

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

fig = plt.figure(figsize=(12.5, 7.6))
grid = fig.add_gridspec(2, 4)

for index, (method, title) in enumerate(panels):
    ax = fig.add_subplot(grid[index // 4, index % 4])
    ax.plot(
        [0, 1], [0, 1],
        color="#777777", linestyle=":", linewidth=1.3, zorder=1,
    )

    annotations = []

    for subset, style in styles.items():
        data = bins.loc[
            bins["method"].eq(method)
            & bins["subset"].eq(subset)
        ].sort_values("bin")

        if data.empty:
            continue

        occupied = data.loc[data["count"].gt(0)]
        sizes = 22 + 3 * np.sqrt(occupied["count"].to_numpy())

        ax.plot(
            occupied["mean_confidence"],
            occupied["accuracy"],
            color=style["color"],
            linestyle=style["linestyle"],
            linewidth=1.5,
            zorder=2,
        )
        ax.scatter(
            occupied["mean_confidence"],
            occupied["accuracy"],
            s=sizes,
            color=style["color"],
            marker=style["marker"],
            edgecolors="white",
            linewidths=0.5,
            zorder=3,
        )

        row = metrics.loc[
            metrics["method"].eq(method)
            & metrics["subset"].eq(subset)
        ]

        if len(row) != 1:
            raise ValueError(f"Unexpected metric rows: {method}/{subset}")

        row = row.iloc[0]
        annotations.append(
            (
                f"{style['label']}: n={int(row['n'])}, "
                f"ECE={row['ECE']:.3f}",
                style["color"],
            )
        )

    ax.set_title(f"({chr(97 + index)}) {title}", pad=8)
    ax.set_xlim(-0.02, 1.02)
    ax.set_ylim(-0.02, 1.02)
    ax.set_xticks(np.linspace(0, 1, 6))
    ax.set_yticks(np.linspace(0, 1, 6))
    ax.set_aspect("equal", adjustable="box")
    ax.grid(alpha=0.18)
    ax.set_axisbelow(True)
    ax.set_xlabel("Mean top-label probability")

    if index % 4 == 0:
        ax.set_ylabel("Observed accuracy")

    for j, (text, color) in enumerate(annotations):
        ax.text(
            0.04, 0.96 - j * 0.085, text,
            transform=ax.transAxes,
            ha="left", va="top", fontsize=8,
            color=color,
            bbox={
                "facecolor": "white",
                "edgecolor": "none",
                "alpha": 0.85,
                "pad": 1,
            },
        )

legend_ax = fig.add_subplot(grid[1, 3])
legend_ax.axis("off")

handles = [
    Line2D(
        [0], [0],
        color=style["color"],
        marker=style["marker"],
        linestyle=style["linestyle"],
        linewidth=1.5,
        label=style["label"],
    )
    for style in styles.values()
]
handles.append(
    Line2D(
        [0], [0], color="#777777", linestyle=":",
        linewidth=1.3, label="Perfect calibration",
    )
)

legend_ax.legend(
    handles=handles,
    loc="upper left",
    bbox_to_anchor=(0.0, 0.95),
    frameon=False,
    fontsize=10,
)

legend_ax.text(
    0.02, 0.48,
    "Original probabilities\n"
    "10 equal-width bins\n"
    "Empty bins omitted\n"
    "Marker size reflects bin count\n\n"
    "Accepted subsets use the\n"
    "original frozen CRUS gates.\n\n"
    "BM25/SBERT: not applicable\n"
    "(no class-probability output)",
    transform=legend_ax.transAxes,
    ha="left", va="top",
    fontsize=9, linespacing=1.5,
)

fig.subplots_adjust(
    left=0.065, right=0.985,
    bottom=0.09, top=0.96,
    wspace=0.27, hspace=0.36,
)

pdf_path = FOLDER / "r112_reliability_original.pdf"
png_path = FOLDER / "r112_reliability_original.png"

fig.savefig(pdf_path, bbox_inches="tight")
fig.savefig(png_path, dpi=350, bbox_inches="tight")
plt.close(fig)

for path in [pdf_path, png_path]:
    if not path.is_file() or path.stat().st_size == 0:
        raise RuntimeError(f"Figure was not saved correctly: {path}")

display(Image(filename=str(png_path)))
print("PDF saved:", pdf_path)
print("PNG saved:", png_path)

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json

import numpy as np
import pandas as pd

ROOT = Path(_crus_local_path('project', ''))
BASE = ROOT / "corrected_experiment_outputs"

manifest = pd.read_csv(
    ROOT / "data_audit/constrained_split_manifest.csv",
    dtype={"record_id": str},
)
audit = pd.read_csv(
    BASE / "test_frozen_CRUS_acceptance_audit.csv",
    dtype={"record_id": str},
)
protocol = json.loads(
    (BASE / "final_pretest_protocol.json").read_text(encoding="utf-8")
)

def parse_bool(series):
    result = series.astype(str).str.strip().str.lower().map({
        "true": True, "false": False,
        "1": True, "0": False, "1.0": True, "0.0": False,
    })
    if result.isna().any():
        raise ValueError("Unrecognized Boolean values.")
    return result.astype(bool)

if manifest["record_id"].duplicated().any():
    raise ValueError("Duplicate manifest record IDs.")

test_manifest = manifest.loc[
    manifest["split"].eq("test")
].set_index("record_id")

cascade = audit.loc[
    audit["method"].eq("Cascade")
].set_index("record_id").copy()

if not cascade.index.is_unique:
    raise ValueError("Duplicate Cascade audit IDs.")
if set(cascade.index) != set(test_manifest.index):
    raise ValueError("Cascade audit does not match the test split.")

cascade["label"] = test_manifest.loc[cascade.index, "label"].astype(str)
cascade["accepted"] = parse_bool(cascade["accepted"])
cascade["correct"] = parse_bool(cascade["correct"])
cascade["deferred"] = ~cascade["accepted"]
cascade["error"] = ~cascade["correct"]

policy = protocol["frozen_policies"]["Cascade:CRUS"]

if not np.allclose(cascade["tau_c"], float(policy["tau_c"])):
    raise ValueError("Confidence threshold mismatch.")
if not np.allclose(cascade["tau_u"], float(policy["tau_u"])):
    raise ValueError("CRUS threshold mismatch.")

recomputed_acceptance = (
    cascade["confidence"].ge(float(policy["tau_c"]))
    & cascade["CRUS"].le(float(policy["tau_u"]))
)
if not recomputed_acceptance.equals(cascade["accepted"]):
    raise ValueError("Saved acceptance does not match frozen thresholds.")

supports = pd.crosstab(manifest["label"], manifest["split"])
classes = protocol["classes"]
overall_deferral = float(cascade["deferred"].mean())

def rate(numerator, denominator):
    return numerator / denominator if denominator else np.nan

rows = []

for label in classes:
    group = cascade.loc[cascade["label"].eq(label)]
    n = len(group)
    accepted = group.loc[group["accepted"]]
    deferred = group.loc[group["deferred"]]

    n_accepted = len(accepted)
    n_deferred = len(deferred)
    forced_errors = int(group["error"].sum())
    accepted_errors = int(accepted["error"].sum())
    deferred_errors = int(deferred["error"].sum())
    deferred_correct = int(deferred["correct"].sum())

    train_n = (
        int(supports.loc[label, "train"])
        if label in supports.index and "train" in supports.columns else 0
    )
    validation_n = (
        int(supports.loc[label, "validation"])
        if label in supports.index and "validation" in supports.columns else 0
    )

    rows.append({
        "class": label,
        "train_n": train_n,
        "validation_n": validation_n,
        "test_n": n,
        "accepted_n": n_accepted,
        "deferred_n": n_deferred,
        "coverage": rate(n_accepted, n),
        "abstention_rate": rate(n_deferred, n),
        "forced_errors_n": forced_errors,
        "forced_error_rate": rate(forced_errors, n),
        "accepted_errors_n": accepted_errors,
        "accepted_error_rate": rate(accepted_errors, n_accepted),
        "deferred_errors_n": deferred_errors,
        "deferred_error_rate": rate(deferred_errors, n_deferred),
        "deferred_correct_n": deferred_correct,
        "deferred_correct_fraction": rate(deferred_correct, n_deferred),
        "mean_CRUS": group["CRUS"].mean(),
        "deferral_rate_ratio_vs_overall": (
            rate(n_deferred, n) / overall_deferral
            if n and overall_deferral else np.nan
        ),
        "share_of_all_deferrals": rate(
            n_deferred, int(cascade["deferred"].sum())
        ),
    })

per_class = pd.DataFrame(rows).sort_values(
    ["test_n", "train_n", "class"]
).reset_index(drop=True)

overall = {
    "test_n": len(cascade),
    "accepted_n": int(cascade["accepted"].sum()),
    "deferred_n": int(cascade["deferred"].sum()),
    "coverage": float(cascade["accepted"].mean()),
    "abstention_rate": overall_deferral,
    "forced_error_rate": float(cascade["error"].mean()),
    "accepted_error_rate": float(
        cascade.loc[cascade["accepted"], "error"].mean()
    ),
    "deferred_error_rate": float(
        cascade.loc[cascade["deferred"], "error"].mean()
    ),
    "deferred_correct_n": int(
        (cascade["deferred"] & cascade["correct"]).sum()
    ),
}

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
outdir = ROOT / f"r1_13_class_deferral_{stamp}"
outdir.mkdir(parents=True, exist_ok=False)

per_class.to_csv(outdir / "per_class_deferral_audit.csv", index=False)
cascade.reset_index().to_csv(
    outdir / "per_query_deferral_audit.csv", index=False
)
(outdir / "overall_summary.json").write_text(
    json.dumps(overall, indent=2), encoding="utf-8"
)

display_columns = [
    "class", "train_n", "test_n", "accepted_n", "deferred_n",
    "coverage", "abstention_rate", "forced_error_rate",
    "accepted_error_rate", "deferred_error_rate",
    "deferred_correct_n", "mean_CRUS",
]

print("=== OVERALL FROZEN CASCADE POLICY ===")
for key, value in overall.items():
    print(f"{key}: {value}")

print("\n=== PER-CLASS RESULTS: ASCENDING TEST SUPPORT ===")
print(
    per_class[display_columns]
    .round(6)
    .to_string(index=False)
)

print("\n=== DEFERRAL CONCENTRATION ===")
print(
    per_class.loc[per_class["test_n"].gt(0), [
        "class", "test_n", "abstention_rate",
        "deferral_rate_ratio_vs_overall", "share_of_all_deferrals",
    ]]
    .sort_values("abstention_rate", ascending=False)
    .round(6)
    .to_string(index=False)
)

print("\nSaved new audit folder:", outdir)
print("Rates are proportions, not percentages.")
print("Undefined rates are NaN, not zero.")
print("No inference, threshold tuning, or existing-file overwriting.")

In [ ]:
from pathlib import Path
import json
import pandas as pd

folder = Path(
    _crus_local_path('project', 'corrected_experiment_outputs/domain_baselines/medcpt_query_v1')
)

print("=== DOMAIN-BASELINE CONFIGURATION ===")
path = folder / "baseline_configuration.json"
print(json.dumps(
    json.loads(path.read_text(encoding="utf-8")),
    indent=2,
))

print("\n=== DOMAIN-BASELINE RESULTS ===")
path = folder / "metric_summary.csv"
print(pd.read_csv(path).to_string(index=False))

print("\n=== COMPLETION METADATA ===")
path = folder / "RUN_COMPLETE.json"
print(path.read_text(encoding="utf-8"))

In [ ]:
import ast
import json
from pathlib import Path

if not globals().get("r10_baseline_passed", False):
    raise RuntimeError("Restore and verify the baseline runtime first.")

notebook_path = Path(
    _crus_local_path('notebooks', 'revised_risk_aware_hybrid_pipeline_v2.ipynb')
)
notebook = json.loads(notebook_path.read_text(encoding="utf-8"))

wanted = {
    "prepare_candidates",
    "score_documents",
    "uncertainty_components",
    "composite_crus",
}

found = set()

for cell_index in range(69, 74):
    cell = notebook["cells"][cell_index]
    source = cell.get("source", "")
    source = "".join(source) if isinstance(source, list) else source
    tree = ast.parse(source)

    for node in tree.body:
        if isinstance(node, ast.FunctionDef) and node.name in wanted:
            print("\n" + "=" * 75)
            print(f"{node.name} — saved cell index {cell_index}")
            print("=" * 75)
            print(ast.get_source_segment(source, node))
            found.add(node.name)

if found != wanted:
    raise RuntimeError(f"Missing definitions: {wanted - found}")

print("\n=== CURRENT CONFIGURATION ===")
print(json.dumps(r10["RETRIEVAL_CFG"], indent=2))

print("\nInspection only. No runtime functions changed or inference performed.")

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from collections import defaultdict
import copy
import json

import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score
from tqdm.auto import tqdm

if not globals().get("r10_baseline_passed", False):
    raise RuntimeError("Verify the original baseline first.")

ROOT = Path(_crus_local_path('project', ''))
cfg = copy.deepcopy(r10["RETRIEVAL_CFG"])
uncertainty_cfg = copy.deepcopy(r10["UNCERTAINTY_CFG"])
queries = r10_validation.copy()
classes = list(r10["CLASSES"])
training_labels = r10["y_train_np"]
training_ids = r10["train_ids"]
training_texts = r10["X_train"]
class_index = r10["CIDX"]

if len(queries) != len(r10_baseline_results):
    raise RuntimeError("Baseline/query count mismatch.")

def eligible_pool(candidates):
    fused = r10["rrf_fuse"](
        candidates["sparse"],
        candidates["dense"],
        top_n=2 * cfg["rrf_top_n"],
    )
    shortlist = set(candidates["shortlist"])
    selected = [
        item for item in fused
        if training_labels[item[0]] in shortlist
    ]
    fallback = not bool(selected)
    return (selected if selected else fused), fallback

def balanced_selection(pool, budget):
    groups = defaultdict(list)
    label_order = []

    for item in pool:
        label = str(training_labels[item[0]])
        if label not in groups:
            label_order.append(label)
        groups[label].append(item)

    chosen = []
    depth = 0

    while len(chosen) < budget:
        added = False
        for label in label_order:
            if depth < len(groups[label]):
                chosen.append(groups[label][depth])
                added = True
                if len(chosen) == budget:
                    break
        if not added:
            break
        depth += 1

    # Keep original relevance order after choosing the balanced set.
    chosen_ids = {int(index) for index, _ in chosen}
    return [item for item in pool if int(item[0]) in chosen_ids]

def score_selected(query, documents):
    logits = r10["ce_raw_logits"]([
        (query, str(training_texts[index])[:cfg["doc_truncate_chars"]])
        for index, _ in documents
    ])
    document_weights = r10["softmax"](logits)
    probabilities = np.zeros(len(classes), dtype=float)

    for (index, _), weight in zip(documents, document_weights):
        probabilities[class_index[training_labels[index]]] += weight

    probabilities /= probabilities.sum()
    components = r10["uncertainty_components"](
        probabilities, documents
    )

    return {
        "pred": classes[int(probabilities.argmax())],
        "confidence": float(probabilities.max()),
        "components": np.asarray(components, dtype=float),
        "CRUS": r10["composite_crus"](components),
    }

rows = []

for (record_id, row), baseline in tqdm(
    zip(queries.iterrows(), r10_baseline_results),
    total=len(queries),
    desc="R1.15 candidate-selection ablation",
):
    if (
        r10["RETRIEVAL_CFG"] != cfg
        or r10["UNCERTAINTY_CFG"] != uncertainty_cfg
    ):
        raise RuntimeError("Runtime configuration changed.")

    query = str(row["text"])
    candidates = r10["prepare_candidates"](query)
    pool, fallback = eligible_pool(candidates)

    relevance_documents = pool[:cfg["rrf_top_n"]]
    balanced_documents = balanced_selection(
        pool, len(relevance_documents)
    )

    if not relevance_documents:
        raise RuntimeError(f"Empty candidate pool: {record_id}")
    if len(balanced_documents) != len(relevance_documents):
        raise RuntimeError(f"Document budget mismatch: {record_id}")

    expected_ids = [int(i) for i, _ in baseline["documents"]]
    actual_ids = [int(i) for i, _ in relevance_documents]

    if (
        actual_ids != expected_ids
        or candidates["shortlist"] != baseline["shortlist"]
        or bool(fallback) != bool(baseline["fallback"])
    ):
        raise RuntimeError(f"Baseline selection mismatch: {record_id}")

    selection_changed = (
        {int(i) for i, _ in relevance_documents}
        != {int(i) for i, _ in balanced_documents}
    )

    relevance_result = score_selected(query, relevance_documents)

    if (
        relevance_result["pred"] != baseline["pred"]
        or not np.isclose(
            relevance_result["confidence"], baseline["confidence"],
            rtol=1e-5, atol=1e-7,
        )
        or not np.allclose(
            relevance_result["components"], baseline["components"],
            rtol=1e-5, atol=1e-7,
        )
    ):
        raise RuntimeError(f"Baseline score mismatch: {record_id}")

    balanced_result = (
        score_selected(query, balanced_documents)
        if selection_changed else relevance_result
    )

    for policy, documents, result in [
        ("relevance", relevance_documents, relevance_result),
        ("class_balanced", balanced_documents, balanced_result),
    ]:
        components = result["components"]
        rows.append({
            "record_id": str(record_id),
            "policy": policy,
            "true_label": str(row["label"]),
            "prediction": str(result["pred"]),
            "correct": str(result["pred"]) == str(row["label"]),
            "confidence": result["confidence"],
            "H_pred": float(components[0]),
            "H_retrieval": float(components[1]),
            "V_RRF": float(components[2]),
            "CRUS": result["CRUS"],
            "eligible_documents_n": len(pool),
            "selected_documents_n": len(documents),
            "represented_labels_n": len({
                str(training_labels[i]) for i, _ in documents
            }),
            "selection_changed": selection_changed,
            "fallback": fallback,
            "selected_training_ids": json.dumps([
                str(training_ids[i]) for i, _ in documents
            ]),
        })

per_query = pd.DataFrame(rows)

summary_rows = []
for policy, group in per_query.groupby("policy", sort=False):
    summary_rows.append({
        "policy": policy,
        "n": len(group),
        "accuracy": accuracy_score(
            group["true_label"], group["prediction"]
        ),
        "macro_f1": f1_score(
            group["true_label"], group["prediction"],
            labels=classes, average="macro", zero_division=0,
        ),
        "mean_H_pred": group["H_pred"].mean(),
        "mean_H_retrieval": group["H_retrieval"].mean(),
        "mean_V_RRF": group["V_RRF"].mean(),
        "mean_CRUS": group["CRUS"].mean(),
        "mean_selected_documents": group["selected_documents_n"].mean(),
        "mean_represented_labels": group["represented_labels_n"].mean(),
        "changed_queries_n": int(group["selection_changed"].sum()),
    })

summary = pd.DataFrame(summary_rows)

original = per_query.loc[
    per_query["policy"].eq("relevance")
].set_index("record_id")
balanced = per_query.loc[
    per_query["policy"].eq("class_balanced")
].set_index("record_id").loc[original.index]

paired = pd.DataFrame(index=original.index)
paired["selection_changed"] = balanced["selection_changed"]
paired["transition"] = [
    ("correct" if old else "incorrect")
    + "_to_" + ("correct" if new else "incorrect")
    for old, new in zip(original["correct"], balanced["correct"])
]

for signal in ["H_pred", "H_retrieval", "V_RRF", "CRUS"]:
    paired[f"delta_{signal}"] = balanced[signal] - original[signal]

transitions = (
    paired.groupby("transition")
    .agg(
        n=("transition", "size"),
        mean_delta_H_pred=("delta_H_pred", "mean"),
        mean_delta_H_retrieval=("delta_H_retrieval", "mean"),
        mean_delta_CRUS=("delta_CRUS", "mean"),
    )
    .reset_index()
)

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
outdir = ROOT / f"r1_15_candidate_selection_{stamp}"
outdir.mkdir(parents=True, exist_ok=False)

per_query.to_csv(outdir / "per_query_results.csv", index=False)
summary.to_csv(outdir / "policy_summary.csv", index=False)
paired.reset_index().to_csv(outdir / "paired_differences.csv", index=False)
transitions.to_csv(outdir / "transition_summary.csv", index=False)

metadata = {
    "analysis": "R1.15 MMRAG-inspired candidate-selection ablation",
    "partition": "constrained validation",
    "configuration": cfg,
    "uncertainty_configuration": uncertainty_cfg,
    "baseline_reproduced": True,
    "balanced_rule": (
        "Round-robin over labels ordered by their first occurrence "
        "in the relevance-ranked eligible pool; relevance order "
        "within labels; restore original order after selection."
    ),
    "fixed": [
        "training index", "initial fused pool", "candidate shortlist",
        "per-query document count", "cross-encoder",
        "truncation", "uncertainty definitions and weights",
    ],
    "test_predictions_used": False,
    "true_query_labels_used_for_selection": False,
    "rejection_thresholds_evaluated": False,
    "limitations": [
        "Not a reproduction of MMRAG retrieval modes",
        "Class balancing restricted to labels in the eligible pool",
        "Descriptive validation ablation; no significance tests",
    ],
}

(outdir / "analysis_metadata.json").write_text(
    json.dumps(metadata, indent=2), encoding="utf-8"
)
(outdir / "RUN_COMPLETE.json").write_text(
    json.dumps({"completed_queries": len(queries)}, indent=2),
    encoding="utf-8",
)

print("\n=== POLICY COMPARISON ===")
print(summary.round(6).to_string(index=False))

print("\n=== PAIRED CORRECTNESS TRANSITIONS ===")
print(transitions.round(6).to_string(index=False))

print("\n=== MEAN SIGNAL CHANGES: BALANCED MINUS RELEVANCE ===")
delta_columns = [
    "delta_H_pred", "delta_H_retrieval", "delta_V_RRF", "delta_CRUS"
]
print(paired[delta_columns].mean().round(6).to_string())

print("\n=== QUERIES WITH CHANGED SELECTION ONLY ===")
changed = paired.loc[paired["selection_changed"]]
print("Changed selections:", len(changed), "/", len(paired))
if len(changed):
    print(changed[delta_columns].mean().round(6).to_string())

print("\nSaved new audit folder:", outdir)
print("Original runtime functions and experiment outputs remain unchanged.")

In [ ]:
import inspect
import sys
import importlib.metadata as metadata

print("Python:", sys.version.split()[0])

print("\nInstalled package versions:")
for package in [
    "numpy",
    "pandas",
    "scikit-learn",
    "torch",
    "transformers",
    "sentence-transformers",
    "rank-bm25",
    "faiss-cpu",
]:
    try:
        print(f"{package}: {metadata.version(package)}")
    except metadata.PackageNotFoundError:
        print(f"{package}: not detected under this package name")

keywords = (
    "cascade",
    "predict",
    "score",
    "retriev",
    "crus",
    "load",
    "infer",
)

print("\nRelevant functions currently available:")
function_count = 0

for name, obj in sorted(list(globals().items())):
    if (
        not name.startswith("_")
        and inspect.isfunction(obj)
        and getattr(obj, "__module__", "") == "__main__"
        and any(word in name.lower() for word in keywords)
    ):
        try:
            signature = str(inspect.signature(obj))
        except (TypeError, ValueError):
            signature = "(signature unavailable)"

        print(f"{name}{signature}")
        function_count += 1

print("\nMatching function count:", function_count)

print("\nRecognized configuration dictionaries:")
configuration_keys = [
    "n_train",
    "n_val",
    "n_test",
    "tau_c",
    "tau_u",
    "crus_lambda1",
    "crus_lambda2",
    "crus_lambda3",
    "topk_retrieve",
    "rrf_k",
    "rrf_top_n",
    "cascade_top_k_classes",
    "doc_truncate_chars",
    "dense_model",
    "ce_model",
]

for name in ("CFG", "cfg", "CONFIG", "config"):
    obj = globals().get(name)

    if isinstance(obj, dict):
        print(f"\n{name}:")
        for key in configuration_keys:
            if key in obj:
                print(f"  {key}: {obj[key]}")

In [ ]:
from pathlib import Path
import importlib.util
import inspect
import os

print("Current working directory:")
print(Path.cwd())

try:
    in_colab = importlib.util.find_spec("google.colab") is not None
except (ModuleNotFoundError, ValueError):
    in_colab = False

print("\nGoogle Colab module available:", in_colab)

print("\nImport availability:")
for module_name in ["rank_bm25", "faiss", "torch"]:
    try:
        available = importlib.util.find_spec(module_name) is not None
    except (ModuleNotFoundError, ValueError):
        available = False
    print(f"{module_name}: {available}")

print("\nRelevant files in the current folder:")
wanted_suffixes = {
    ".ipynb", ".py", ".json", ".csv", ".jsonl",
    ".pkl", ".pickle", ".joblib", ".faiss",
    ".npz", ".pt", ".pth", ".safetensors", ".zip",
}

matches = [
    path for path in Path.cwd().iterdir()
    if path.is_file() and path.suffix.lower() in wanted_suffixes
]

for path in sorted(matches)[:80]:
    print(f"{path.name} | {path.stat().st_size:,} bytes")

if not matches:
    print("No matching files found in the current folder.")

print("\nLikely project-related names in notebook memory:")
keywords = (
    "cascade", "crus", "retriev", "config", "model",
    "index", "encoder", "pipeline", "artifact"
)

names = []

for name, obj in list(globals().items()):
    if name.startswith("_"):
        continue

    if any(word in name.lower() for word in keywords):
        names.append((name, type(obj).__name__))

for name, object_type in sorted(names)[:80]:
    print(f"{name}: {object_type}")

print("\nInspection complete. No model execution was performed.")

In [ ]:
pass  # Colab service replaced by local workflow.

_crus_prepare_saved_workspace()

In [ ]:
from pathlib import Path

DRIVE_ROOT = Path(_crus_local_path('drive', ''))

if not DRIVE_ROOT.is_dir():
    raise FileNotFoundError(
        "MyDrive was not found. Check that Drive mounted successfully."
    )

print("Top-level folders in MyDrive:\n")

folders = sorted(
    path for path in DRIVE_ROOT.iterdir()
    if path.is_dir()
)

for number, folder in enumerate(folders, start=1):
    print(f"{number:3d}. {folder.name}")

In [ ]:
from pathlib import Path

PROJECT_DIR = Path(
    _crus_local_path('project', '')
)

if not PROJECT_DIR.is_dir():
    raise FileNotFoundError(
        f"Project folder not found: {PROJECT_DIR}"
    )

print("Project:", PROJECT_DIR)
print("\nImmediate contents:\n")

entries = sorted(
    PROJECT_DIR.iterdir(),
    key=lambda p: (not p.is_dir(), p.name.lower())
)

for path in entries[:150]:
    if path.is_dir():
        print(f"[FOLDER] {path.name}")
    else:
        print(
            f"[FILE]   {path.name}"
            f" | {path.stat().st_size:,} bytes"
        )

if len(entries) > 150:
    print(f"\nShowing 150 of {len(entries)} entries.")

In [ ]:
from pathlib import Path

PROJECT_DIR = Path(
    _crus_local_path('project', '')
)

folders_to_inspect = [
    "corrected_experiment_outputs",
    "data_audit",
    "source_data",
]

for folder_name in folders_to_inspect:
    folder = PROJECT_DIR / folder_name

    print("\n" + "=" * 75)
    print(folder_name)
    print("=" * 75)

    if not folder.is_dir():
        print("Folder not found.")
        continue

    entries = sorted(
        folder.iterdir(),
        key=lambda p: (not p.is_dir(), p.name.lower())
    )

    for path in entries[:100]:
        if path.is_dir():
            print(f"[FOLDER] {path.name}")
        else:
            print(
                f"[FILE]   {path.name}"
                f" | {path.stat().st_size:,} bytes"
            )

    if len(entries) > 100:
        print(f"\nShowing 100 of {len(entries)} entries.")

In [ ]:
from pathlib import Path
import json
import re
import pandas as pd
import numpy as np

PROJECT_DIR = Path(
    _crus_local_path('project', '')
)

OUTPUT_DIR = PROJECT_DIR / "corrected_experiment_outputs"
AUDIT_DIR = PROJECT_DIR / "data_audit"


# ------------------------------------------------------------
# 1. Read protocol and initialization metadata
# ------------------------------------------------------------
json_files = [
    OUTPUT_DIR / "final_pretest_protocol.json",
    OUTPUT_DIR / "retrieval_initialization_metadata.json",
    OUTPUT_DIR / "validation_selected_policies.json",
    AUDIT_DIR / "constrained_split_metadata.json",
]

for path in json_files:
    print("\n" + "=" * 80)
    print("JSON:", path.name)
    print("=" * 80)

    if not path.is_file():
        print("NOT FOUND")
        continue

    try:
        data = json.loads(path.read_text(encoding="utf-8"))
        print(json.dumps(data, indent=2, ensure_ascii=False))
    except Exception as exc:
        print(type(exc).__name__, str(exc))


# ------------------------------------------------------------
# 2. Inspect CSV column names only — no dialogue rows printed
# ------------------------------------------------------------
csv_files = [
    AUDIT_DIR / "constrained_dialogue_pool_with_splits.csv",
    AUDIT_DIR / "constrained_split_manifest.csv",
    OUTPUT_DIR / "validation_retrieval_prediction_audit.csv",
    OUTPUT_DIR / "validation_policy_selection_summary.csv",
]

print("\n" + "=" * 80)
print("CSV SCHEMAS — COLUMN NAMES ONLY")
print("=" * 80)

for path in csv_files:
    print("\nFILE:", path.name)

    if not path.is_file():
        print("NOT FOUND")
        continue

    try:
        columns = pd.read_csv(path, nrows=0).columns.tolist()
        print(columns)
    except Exception as exc:
        print(type(exc).__name__, str(exc))


# ------------------------------------------------------------
# 3. Inspect NumPy array names, shapes and dtypes
#    Object arrays are skipped; pickle loading stays disabled.
# ------------------------------------------------------------
npz_files = [
    OUTPUT_DIR / "validation_Cascade_arrays.npz",
    OUTPUT_DIR / "test_Cascade_arrays.npz",
    OUTPUT_DIR / "training_dense_embeddings.npz",
]

print("\n" + "=" * 80)
print("NUMPY ARRAY SCHEMAS — NO ARRAY VALUES")
print("=" * 80)

for path in npz_files:
    print("\nFILE:", path.name)

    if not path.is_file():
        print("NOT FOUND")
        continue

    try:
        with np.load(path, allow_pickle=False) as archive:
            for key in archive.files:
                try:
                    array = archive[key]
                    print(
                        f"  {key}: "
                        f"shape={array.shape}, dtype={array.dtype}"
                    )
                except ValueError:
                    print(
                        f"  {key}: skipped; cannot inspect "
                        "without allowing object deserialization"
                    )
    except Exception as exc:
        print(type(exc).__name__, str(exc))


# ------------------------------------------------------------
# 4. Inspect saved source audit as TEXT — never execute it
# ------------------------------------------------------------
source_path = (
    OUTPUT_DIR / "comment2_saved_source_audit_86654d959790.txt"
)

print("\n" + "=" * 80)
print("SAVED SOURCE AUDIT — DEFINITIONS AND IMPORTS ONLY")
print("=" * 80)

if not source_path.is_file():
    print("NOT FOUND:", source_path.name)
else:
    source_text = source_path.read_text(
        encoding="utf-8",
        errors="replace",
    )

    source_lines = source_text.splitlines()
    print("Total lines:", len(source_lines))

    print("\nClass/function definition lines:")
    found = False

    for line_number, line in enumerate(source_lines, start=1):
        if re.match(
            r"^\s*(?:async\s+def|def|class)\s+",
            line,
        ):
            print(f"{line_number}: {line[:240]}")
            found = True

    if not found:
        print("No plain Python definition lines detected.")

    print("\nImport lines:")
    import_count = 0

    for line_number, line in enumerate(source_lines, start=1):
        if re.match(r"^\s*(?:import\s+|from\s+\S+\s+import\s+)", line):
            print(f"{line_number}: {line[:240]}")
            import_count += 1

            if import_count >= 40:
                print("Showing the first 40 import lines.")
                break

print("\nREAD-ONLY INSPECTION COMPLETE.")

In [ ]:
from pathlib import Path

PROJECT_DIR = Path(
    _crus_local_path('project', '')
)

OUTPUT_DIR = PROJECT_DIR / "corrected_experiment_outputs"

SOURCE_AUDIT = (
    OUTPUT_DIR / "comment2_saved_source_audit_86654d959790.txt"
)

if not SOURCE_AUDIT.is_file():
    raise FileNotFoundError(SOURCE_AUDIT)

source_lines = SOURCE_AUDIT.read_text(
    encoding="utf-8",
    errors="replace",
).splitlines()


def show_source_lines(start, end):
    """Display numbered source lines without executing them."""
    if start < 1 or end < start:
        raise ValueError("Invalid line range.")

    end = min(end, len(source_lines))

    print("\n" + "=" * 90)
    print(f"SOURCE AUDIT: LINES {start}–{end}")
    print("=" * 90)

    for line_number in range(start, end + 1):
        print(
            f"{line_number:04d}: "
            f"{source_lines[line_number - 1]}"
        )


# Configuration, model initialization, and training-index setup.
show_source_lines(1900, 2088)

# Corrected retrieval, uncertainty, and prediction functions.
show_source_lines(2089, 2430)

print("\nSource inspection complete. No saved code was executed.")

In [ ]:
from pathlib import Path
import json
import re
import os

MYDRIVE = Path(_crus_local_path('drive', ''))

SEARCH_ROOTS = [
    MYDRIVE / "RiskAware_Revision2",
    MYDRIVE / "Colab Notebooks",
    MYDRIVE / "Colab_Notebooks",
]

HELPER_PATTERNS = {
    "prepare_candidates": re.compile(
        r"\bdef\s+prepare_candidates\s*\("
    ),
    "ce_raw_logits": re.compile(
        r"\bdef\s+ce_raw_logits\s*\("
    ),
    "softmax": re.compile(
        r"\bdef\s+softmax\s*\("
        r"|from\s+scipy\.special\s+import[^\n]*\bsoftmax\b"
        r"|from\s+scipy\s+import[^\n]*\bsoftmax\b"
    ),
}

MAX_FILE_BYTES = 20_000_000
matches = []
read_errors = []
seen_paths = set()

for root in SEARCH_ROOTS:
    if not root.is_dir():
        continue

    for current_dir, directories, filenames in os.walk(root):
        directories[:] = [
            name for name in directories
            if name not in {
                ".git",
                "__pycache__",
                ".ipynb_checkpoints",
                "node_modules",
            }
        ]

        for filename in filenames:
            path = Path(current_dir) / filename

            if path.suffix.lower() not in {".py", ".ipynb"}:
                continue

            resolved = str(path.resolve())

            if resolved in seen_paths:
                continue

            seen_paths.add(resolved)

            try:
                if path.stat().st_size > MAX_FILE_BYTES:
                    continue

                text = path.read_text(
                    encoding="utf-8",
                    errors="replace",
                )

                if path.suffix.lower() == ".ipynb":
                    notebook = json.loads(text)
                    code_units = []

                    for cell_number, cell in enumerate(
                        notebook.get("cells", []),
                        start=1,
                    ):
                        if cell.get("cell_type") != "code":
                            continue

                        source = cell.get("source", "")
                        source = (
                            "".join(source)
                            if isinstance(source, list)
                            else source
                        )

                        code_units.append(
                            (f"code cell {cell_number}", source)
                        )
                else:
                    code_units = [("Python file", text)]

                for location, source in code_units:
                    found_helpers = [
                        name
                        for name, pattern in HELPER_PATTERNS.items()
                        if pattern.search(source)
                    ]

                    if found_helpers:
                        matches.append({
                            "file": str(path),
                            "location": location,
                            "helpers": found_helpers,
                            "modified": path.stat().st_mtime,
                            "source": source,
                        })

            except Exception as exc:
                read_errors.append(
                    f"{path.name}: {type(exc).__name__}"
                )

matches.sort(
    key=lambda item: item["modified"],
    reverse=True,
)

print("Files examined:", len(seen_paths))
print("Matching code blocks:", len(matches))

print("\nMatching locations, newest files first:\n")

for number, item in enumerate(matches[:30], start=1):
    print(f"{number}. {item['file']}")
    print(f"   Location: {item['location']}")
    print(f"   Helpers: {', '.join(item['helpers'])}")

if read_errors:
    print("\nRead errors, first ten:")
    for message in read_errors[:10]:
        print(message)

# Preserve the matches in notebook memory for inspection.
HELPER_SEARCH_RESULTS = matches

print(
    "\nSearch complete. No discovered code was executed."
)

In [ ]:
from pathlib import Path
import json

NOTEBOOK_PATH = Path(
    _crus_local_path('notebooks', 'CRUS_R1_5_External_Transfer.ipynb')
)

if not NOTEBOOK_PATH.is_file():
    raise FileNotFoundError(NOTEBOOK_PATH)

notebook = json.loads(
    NOTEBOOK_PATH.read_text(encoding="utf-8")
)

cells = notebook.get("cells", [])

# These numbers match the search output:
# positions among ALL notebook cells, counting from 1.
for cell_number in [71, 74]:
    if cell_number > len(cells):
        raise IndexError(
            f"Cell {cell_number} is outside this saved notebook."
        )

    cell = cells[cell_number - 1]
    source = cell.get("source", "")

    if isinstance(source, list):
        source = "".join(source)

    print("\n" + "=" * 90)
    print(
        f"NOTEBOOK CELL {cell_number} "
        f"| type: {cell.get('cell_type')}"
    )
    print("=" * 90)

    for line_number, line in enumerate(
        source.splitlines(),
        start=1,
    ):
        print(f"{line_number:03d}: {line}")

print("\nInspection complete. No notebook cells were executed.")

In [ ]:
import subprocess
import sys
import importlib.metadata as metadata

required_version = "0.2.2"

try:
    installed_version = metadata.version("rank-bm25")
except metadata.PackageNotFoundError:
    installed_version = None

if installed_version is None:
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--no-deps",
            f"rank-bm25=={required_version}",
        ],
        check=True,
    )
elif installed_version != required_version:
    raise RuntimeError(
        f"Found rank-bm25 {installed_version}; "
        f"the source run used {required_version}. "
        "Stop before changing the environment."
    )

from rank_bm25 import BM25Okapi

print(
    "BM25 dependency ready:",
    metadata.version("rank-bm25"),
)

In [ ]:
from pathlib import Path
import json
import hashlib
import numpy as np
import pandas as pd
import torch

PROJECT_DIR = Path(
    _crus_local_path('project', '')
)

OUTPUT_DIR = PROJECT_DIR / "corrected_experiment_outputs"
AUDIT_DIR = PROJECT_DIR / "data_audit"

protocol_path = OUTPUT_DIR / "final_pretest_protocol.json"
pool_path = AUDIT_DIR / "constrained_dialogue_pool_with_splits.csv"
manifest_path = AUDIT_DIR / "constrained_split_manifest.csv"
embedding_path = OUTPUT_DIR / "training_dense_embeddings.npz"
validation_path = OUTPUT_DIR / "validation_Cascade_arrays.npz"

required_paths = [
    protocol_path,
    pool_path,
    manifest_path,
    embedding_path,
    validation_path,
]

for path in required_paths:
    if not path.is_file():
        raise FileNotFoundError(path)

protocol = json.loads(
    protocol_path.read_text(encoding="utf-8")
)

expected_sizes = {
    "train": 773,
    "validation": 258,
    "test": 257,
}

if protocol["split_sizes"] != expected_sizes:
    raise RuntimeError(
        "The protocol is not the corrected 773/258/257 run."
    )

source_classes = list(protocol["classes"])
source_policy = protocol["frozen_policies"]["Cascade:CRUS"]

if len(source_classes) != 17 or len(set(source_classes)) != 17:
    raise RuntimeError("Invalid source class order.")

if source_policy["tau_c"] != 0.0:
    raise RuntimeError("Unexpected source confidence threshold.")

if abs(
    source_policy["tau_u"] - 0.128165036474669
) > 1e-15:
    raise RuntimeError("Unexpected source CRUS threshold.")

source_weights = np.asarray(
    protocol["uncertainty_configuration"]["weights"],
    dtype=float,
)

if not np.allclose(
    source_weights,
    [0.5, 0.3, 0.2],
    rtol=0,
    atol=1e-12,
):
    raise RuntimeError("Unexpected source CRUS weights.")


# Read tabular records without displaying dialogue text.
pool = pd.read_csv(
    pool_path,
    dtype=str,
    keep_default_na=False,
)

manifest = pd.read_csv(
    manifest_path,
    dtype=str,
    keep_default_na=False,
)

required_pool_columns = {
    "record_id",
    "label",
    "normalized_dialogue",
    "group_id",
    "split",
}

required_manifest_columns = {
    "record_id",
    "label",
    "group_id",
    "split",
}

if not required_pool_columns.issubset(pool.columns):
    raise RuntimeError("Required pool columns are missing.")

if not required_manifest_columns.issubset(manifest.columns):
    raise RuntimeError("Required manifest columns are missing.")

if pool["record_id"].duplicated().any():
    raise RuntimeError("Duplicate record IDs in source pool.")

if manifest["record_id"].duplicated().any():
    raise RuntimeError("Duplicate record IDs in source manifest.")

if set(pool["record_id"]) != set(manifest["record_id"]):
    raise RuntimeError("Pool and manifest contain different records.")

actual_sizes = pool["split"].value_counts().to_dict()

if actual_sizes != expected_sizes:
    raise RuntimeError(
        f"Unexpected source split sizes: {actual_sizes}"
    )

if pool.groupby("group_id")["split"].nunique().max() != 1:
    raise RuntimeError("Source groups cross partitions.")

pool_index = pool.set_index("record_id")
manifest_index = manifest.set_index("record_id")

for column in ["label", "group_id", "split"]:
    left = pool_index.loc[manifest_index.index, column]
    right = manifest_index[column]

    if not np.array_equal(
        left.to_numpy(),
        right.to_numpy(),
    ):
        raise RuntimeError(
            f"Pool and manifest disagree on {column}."
        )

if set(pool["label"]) != set(source_classes):
    raise RuntimeError(
        "Source pool labels differ from protocol classes."
    )

normalized_again = (
    pool["normalized_dialogue"]
    .map(lambda text: " ".join(text.lower().split()))
)

if not np.array_equal(
    normalized_again.to_numpy(),
    pool["normalized_dialogue"].to_numpy(),
):
    raise RuntimeError(
        "Saved normalized dialogue does not match "
        "the documented normalization."
    )

if pool["normalized_dialogue"].eq("").any():
    raise RuntimeError("Empty normalized source dialogue.")


# These known source NPZ files contain pickled identifier arrays.
confirmation = input(
    "Only continue if these are your own trusted source "
    "experiment files. Type TRUST to load their identifier arrays: "
).strip()

if confirmation != "TRUST":
    raise RuntimeError(
        "Stopped without loading pickled identifier arrays."
    )

with np.load(
    embedding_path,
    allow_pickle=True,
) as archive:
    saved_train_ids = np.asarray(
        archive["record_ids"],
        dtype=str,
    )
    saved_train_embeddings = np.array(
        archive["embeddings"],
        copy=True,
    )

with np.load(
    validation_path,
    allow_pickle=True,
) as archive:
    saved_val_ids = np.asarray(
        archive["record_ids"],
        dtype=str,
    )
    saved_val_labels = np.asarray(
        archive["true_labels"],
        dtype=str,
    )
    saved_val_classes = np.asarray(
        archive["classes"],
        dtype=str,
    )
    saved_val_predictions = np.asarray(
        archive["predictions"],
        dtype=str,
    )
    saved_val_probabilities = np.array(
        archive["probabilities"],
        copy=True,
    )
    saved_val_components = np.array(
        archive["components"],
        copy=True,
    )
    saved_component_names = np.asarray(
        archive["component_names"],
        dtype=str,
    )

if len(set(saved_train_ids)) != 773:
    raise RuntimeError("Training embedding IDs are not unique.")

expected_train_ids = set(
    pool.loc[pool["split"].eq("train"), "record_id"]
)

if set(saved_train_ids) != expected_train_ids:
    raise RuntimeError(
        "Embedding IDs do not match source training records."
    )

if saved_train_embeddings.shape != (773, 384):
    raise RuntimeError(
        "Unexpected saved training embedding shape."
    )

if not np.isfinite(saved_train_embeddings).all():
    raise RuntimeError("Nonfinite saved embeddings.")

if not np.allclose(
    np.linalg.norm(saved_train_embeddings, axis=1),
    1.0,
    atol=1e-4,
):
    raise RuntimeError(
        "Saved training embeddings are not unit-normalized."
    )

expected_val_ids = set(
    pool.loc[pool["split"].eq("validation"), "record_id"]
)

if (
    len(saved_val_ids) != 258
    or len(set(saved_val_ids)) != 258
    or set(saved_val_ids) != expected_val_ids
):
    raise RuntimeError(
        "Saved validation IDs do not match the manifest."
    )

if list(saved_val_classes) != source_classes:
    raise RuntimeError(
        "Saved probability-column order differs from protocol."
    )

if saved_val_probabilities.shape != (258, 17):
    raise RuntimeError("Unexpected validation probability shape.")

if saved_val_components.shape != (258, 3):
    raise RuntimeError("Unexpected validation component shape.")

if not np.isfinite(saved_val_probabilities).all():
    raise RuntimeError("Nonfinite validation probabilities.")

if (
    (saved_val_probabilities < 0).any()
    or not np.allclose(
        saved_val_probabilities.sum(axis=1),
        1.0,
        atol=1e-8,
    )
):
    raise RuntimeError("Invalid saved validation probabilities.")

if (
    not np.isfinite(saved_val_components).all()
    or (saved_val_components < 0).any()
    or (saved_val_components > 1).any()
):
    raise RuntimeError("Invalid saved uncertainty components.")

expected_component_order = [
    "H_pred",
    "H_retrieval",
    "V_RRF",
]

if list(saved_component_names) != expected_component_order:
    raise RuntimeError(
        "Unexpected component order. Report it before proceeding: "
        f"{list(saved_component_names)}"
    )


# Align source TEXT and LABELS to the saved embedding/validation order.
aligned_train = pool_index.loc[saved_train_ids]
aligned_validation = pool_index.loc[saved_val_ids]

if not aligned_train["split"].eq("train").all():
    raise RuntimeError("Non-training record in retrieval index.")

if not aligned_validation["split"].eq("validation").all():
    raise RuntimeError("Non-validation record in validation fixture.")

if not np.array_equal(
    saved_val_labels,
    aligned_validation["label"].to_numpy(dtype=str),
):
    raise RuntimeError(
        "Saved validation labels disagree with the manifest."
    )

expected_argmax_predictions = np.asarray(
    source_classes,
    dtype=str,
)[saved_val_probabilities.argmax(axis=1)]

if not np.array_equal(
    expected_argmax_predictions,
    saved_val_predictions,
):
    raise RuntimeError(
        "Saved predictions disagree with probability argmax."
    )

saved_val_crus = saved_val_components @ source_weights

saved_val_mask = (
    (
        saved_val_probabilities.max(axis=1)
        >= source_policy["tau_c"]
    )
    & (saved_val_crus <= source_policy["tau_u"])
)

saved_val_correct = (
    saved_val_predictions == saved_val_labels
)

verified_counts = {
    "forced_correct": int(saved_val_correct.sum()),
    "accepted": int(saved_val_mask.sum()),
    "accepted_correct": int(
        saved_val_correct[saved_val_mask].sum()
    ),
}

if verified_counts != {
    "forced_correct": 174,
    "accepted": 132,
    "accepted_correct": 124,
}:
    raise RuntimeError(
        f"Saved validation counts do not match: {verified_counts}"
    )


# Keep restored data separate from any original notebook variables.
SOURCE_STATE = {
    "protocol": protocol,
    "classes": source_classes,
    "class_index": {
        label: index
        for index, label in enumerate(source_classes)
    },
    "retrieval_config": dict(
        protocol["retrieval_configuration"]
    ),
    "uncertainty_config": dict(
        protocol["uncertainty_configuration"]
    ),
    "train_ids": saved_train_ids.copy(),
    "train_text": aligned_train[
        "normalized_dialogue"
    ].to_numpy(dtype=str),
    "train_labels": aligned_train[
        "label"
    ].to_numpy(dtype=str),
    "train_embeddings": saved_train_embeddings,
    "validation_ids": saved_val_ids.copy(),
    "validation_text": aligned_validation[
        "normalized_dialogue"
    ].to_numpy(dtype=str),
    "validation_labels": saved_val_labels.copy(),
    "saved_validation_predictions": saved_val_predictions.copy(),
    "saved_validation_probabilities": saved_val_probabilities,
    "saved_validation_components": saved_val_components,
}

print("\nSOURCE ARTIFACT CHECK PASSED")
print("Training records:", len(SOURCE_STATE["train_ids"]))
print("Validation records:", len(SOURCE_STATE["validation_ids"]))
print("Training embeddings:", saved_train_embeddings.shape)
print("Saved validation counts:", verified_counts)
print("Component order:", list(saved_component_names))
print("CUDA currently available:", torch.cuda.is_available())
print(
    "\nSaved-array consistency verified. "
    "New model scoring has NOT yet been reproduced."
)

In [ ]:
import copy
import importlib.metadata as metadata
import numpy as np
import torch

from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder


# ------------------------------------------------------------
# Preconditions: use the already verified source state.
# ------------------------------------------------------------
if "SOURCE_STATE" not in globals():
    raise RuntimeError(
        "SOURCE_STATE is missing. "
        "Run the previous source-artifact verification cell first."
    )

if "SOURCE_RUNTIME" in globals():
    raise RuntimeError(
        "SOURCE_RUNTIME already exists. "
        "Do not reload the models unnecessarily. "
        "Report its current status before proceeding."
    )

source_protocol = SOURCE_STATE["protocol"]
retrieval_cfg = copy.deepcopy(
    SOURCE_STATE["retrieval_config"]
)

expected_versions = source_protocol["package_versions"]

# Verify relevant package versions; do not upgrade automatically.
for package in [
    "numpy",
    "pandas",
    "scikit-learn",
    "sentence-transformers",
    "transformers",
    "rank-bm25",
]:
    current_version = metadata.version(package)
    expected_version = expected_versions[package]

    if current_version != expected_version:
        raise RuntimeError(
            f"{package}: current={current_version}, "
            f"source={expected_version}. "
            "Stop before changing the environment."
        )

# Compare PyTorch release numbers separately from CPU/CUDA suffixes.
current_torch_release = torch.__version__.split("+")[0]
source_torch_release = expected_versions["torch"].split("+")[0]

if current_torch_release != source_torch_release:
    raise RuntimeError(
        f"PyTorch release mismatch: "
        f"current={torch.__version__}, "
        f"source={expected_versions['torch']}"
    )

device = "cuda" if torch.cuda.is_available() else "cpu"

torch.manual_seed(retrieval_cfg["seed"])

if device == "cuda":
    torch.cuda.manual_seed_all(retrieval_cfg["seed"])

print("Current device:", device)
print("Source PyTorch:", expected_versions["torch"])
print("Current PyTorch:", torch.__version__)

if device != "cuda":
    print(
        "NOTE: the source run used CUDA; this restoration uses CPU. "
        "Numerical reproduction must still be checked."
    )


# ------------------------------------------------------------
# Restore BM25 from the original training texts and order.
# This is source-index reconstruction, not external adaptation.
# ------------------------------------------------------------
train_texts = SOURCE_STATE["train_text"]

train_tokens = [
    text.split()
    for text in train_texts
]

source_bm25 = BM25Okapi(
    train_tokens,
    k1=retrieval_cfg["bm25_k1"],
    b=retrieval_cfg["bm25_b"],
)

print("\nBM25 restored from source training records only.")
print("Indexed training records:", len(train_tokens))


# ------------------------------------------------------------
# Load the same named dense encoder.
# Reuse the SAVED training embeddings.
# ------------------------------------------------------------
print("\nLoading dense encoder:")
print(retrieval_cfg["dense_model"])

source_embedder = SentenceTransformer(
    retrieval_cfg["dense_model"],
    device=device,
    trust_remote_code=False,
)

expected_dense_limit = 256

if int(source_embedder.max_seq_length) != expected_dense_limit:
    raise RuntimeError(
        "Dense encoder sequence limit differs from "
        "the saved initialization metadata: "
        f"loaded={source_embedder.max_seq_length}, "
        f"source={expected_dense_limit}. "
        "Do not change it silently."
    )

source_embedder.eval()

print("Dense encoder loaded.")
print("Dense maximum sequence length:",
      source_embedder.max_seq_length)
print(
    "Saved training embeddings reused:",
    SOURCE_STATE["train_embeddings"].shape,
)


# ------------------------------------------------------------
# Load the same named cross-encoder.
# Raw-logit activation will be specified by ce_raw_logits().
# ------------------------------------------------------------
print("\nLoading cross-encoder:")
print(retrieval_cfg["ce_model"])

source_cross_encoder = CrossEncoder(
    retrieval_cfg["ce_model"],
    device=device,
    max_length=retrieval_cfg["ce_max_tokens"],
    trust_remote_code=False,
)

source_cross_encoder.model.eval()

print("Cross-encoder loaded.")
print(
    "Configured cross-encoder token limit:",
    retrieval_cfg["ce_max_tokens"],
)


# ------------------------------------------------------------
# Record model revisions where available.
# The source protocol did not provide explicit model commits.
# ------------------------------------------------------------
dense_config = getattr(
    getattr(source_embedder[0], "auto_model", None),
    "config",
    None,
)

cross_config = getattr(
    source_cross_encoder.model,
    "config",
    None,
)

loaded_revisions = {
    "dense_model_commit": getattr(
        dense_config, "_commit_hash", None
    ),
    "cross_encoder_commit": getattr(
        cross_config, "_commit_hash", None
    ),
}

SOURCE_RUNTIME = {
    "device": device,
    "bm25": source_bm25,
    "embedder": source_embedder,
    "cross_encoder": source_cross_encoder,
    "retrieval_config": retrieval_cfg,
    "loaded_revisions": loaded_revisions,
    "source_regression_passed": False,
}

print("\nSOURCE MODELS LOADED")
print("Recorded model revisions:", loaded_revisions)
print("Source regression passed: False")
print("Training dialogues were NOT re-encoded.")
print("No validation or external queries were scored.")
print("No original experiment files were changed.")

In [ ]:
from pathlib import Path
from collections import Counter
import ast
import copy
import hashlib
import json
import re
import time

import numpy as np
import pandas as pd
import torch
from scipy.special import softmax


# ------------------------------------------------------------
# Preconditions
# ------------------------------------------------------------
if "SOURCE_STATE" not in globals():
    raise RuntimeError("SOURCE_STATE is missing.")

if "SOURCE_RUNTIME" not in globals():
    raise RuntimeError(
        "SOURCE_RUNTIME is missing. Load the models first."
    )

PROJECT_DIR = Path(
    _crus_local_path('project', '')
)

SOURCE_AUDIT = (
    PROJECT_DIR
    / "corrected_experiment_outputs"
    / "comment2_saved_source_audit_86654d959790.txt"
)

NOTEBOOK_PATH = Path(
    _crus_local_path('notebooks', 'CRUS_R1_5_External_Transfer.ipynb')
)

for path in [SOURCE_AUDIT, NOTEBOOK_PATH]:
    if not path.is_file():
        raise FileNotFoundError(path)


# ------------------------------------------------------------
# Extract the corrected functions from the source section
# that we have just inspected.
#
# Only function definitions are compiled below.
# Original top-level operations are NOT executed.
# ------------------------------------------------------------
audit_lines = SOURCE_AUDIT.read_text(
    encoding="utf-8",
    errors="replace",
).splitlines()

# Original source-audit lines 2079–2344, inclusive.
corrected_section = "\n".join(audit_lines[2078:2344])

# Remove the audit's non-Python saved-cell separators.
corrected_section = re.sub(
    r"(?m)^--- SAVED CELL \d+ ---\s*$",
    "",
    corrected_section,
)

parsed_section = ast.parse(corrected_section)

required_audit_functions = [
    "descending_order",
    "bm25_retrieve",
    "dense_retrieve",
    "rrf_fuse",
    "normalized_entropy",
    "uncertainty_components",
    "composite_crus",
    "score_documents",
]

function_nodes = {}

for node in parsed_section.body:
    if (
        isinstance(node, ast.FunctionDef)
        and node.name in required_audit_functions
    ):
        if node.name in function_nodes:
            raise RuntimeError(
                f"Duplicate corrected definition: {node.name}"
            )

        function_nodes[node.name] = node

missing = (
    set(required_audit_functions)
    - set(function_nodes)
)

if missing:
    raise RuntimeError(
        f"Missing corrected source functions: {sorted(missing)}. "
        "Stop; do not substitute historical functions."
    )


# ------------------------------------------------------------
# Extract the two helpers from the inspected notebook cells.
# Cell numbers are positions among ALL cells, counting from 1.
# ------------------------------------------------------------
notebook = json.loads(
    NOTEBOOK_PATH.read_text(encoding="utf-8")
)

helper_locations = {
    "ce_raw_logits": 71,
    "prepare_candidates": 74,
}

for function_name, cell_number in helper_locations.items():
    cell = notebook["cells"][cell_number - 1]

    if cell.get("cell_type") != "code":
        raise RuntimeError(
            f"Expected code in notebook cell {cell_number}."
        )

    cell_source = cell.get("source", "")

    if isinstance(cell_source, list):
        cell_source = "".join(cell_source)

    parsed_cell = ast.parse(cell_source)

    candidates = [
        node
        for node in parsed_cell.body
        if isinstance(node, ast.FunctionDef)
        and node.name == function_name
    ]

    if len(candidates) != 1:
        raise RuntimeError(
            f"Could not uniquely identify {function_name} "
            f"in cell {cell_number}."
        )

    function_nodes[function_name] = candidates[0]


# ------------------------------------------------------------
# Construct a separate namespace using verified source objects.
# No external records are added to the retrieval corpus.
# ------------------------------------------------------------
classes = list(SOURCE_STATE["classes"])

CASCADE_NS = {
    "__builtins__": __builtins__,
    "np": np,
    "torch": torch,
    "Counter": Counter,
    "softmax": softmax,

    "CLASSES": classes,
    "N_CLS": len(classes),
    "CIDX": {
        label: index
        for index, label in enumerate(classes)
    },

    "RETRIEVAL_CFG": copy.deepcopy(
        SOURCE_STATE["retrieval_config"]
    ),
    "UNCERTAINTY_CFG": copy.deepcopy(
        SOURCE_STATE["uncertainty_config"]
    ),

    "X_train": SOURCE_STATE["train_text"],
    "train_ids": SOURCE_STATE["train_ids"],
    "y_train_np": SOURCE_STATE["train_labels"],
    "train_embs": SOURCE_STATE["train_embeddings"],

    "bm25": SOURCE_RUNTIME["bm25"],
    "embedder": SOURCE_RUNTIME["embedder"],
    "cross_enc": SOURCE_RUNTIME["cross_encoder"],

    "DOCUMENT_METHODS": [
        "SingleStageFull",
        "SingleStageTruncated",
        "Cascade",
        "LengthNormLegacy",
    ],
    "average_training_length": float(
        np.mean([
            len(text)
            for text in SOURCE_STATE["train_text"]
        ])
    ),
}

definition_order = required_audit_functions + [
    "ce_raw_logits",
    "prepare_candidates",
]

# The inspected functions have no decorators. Reject unexpected ones.
for name in definition_order:
    if function_nodes[name].decorator_list:
        raise RuntimeError(
            f"Unexpected decorator on {name}; inspect before execution."
        )

definitions = ast.Module(
    body=[
        copy.deepcopy(function_nodes[name])
        for name in definition_order
    ],
    type_ignores=[],
)

ast.fix_missing_locations(definitions)

source_fingerprint = hashlib.sha256(
    ast.dump(
        definitions,
        include_attributes=False,
    ).encode("utf-8")
).hexdigest()

exec(
    compile(
        definitions,
        filename="<verified_source_cascade_functions>",
        mode="exec",
    ),
    CASCADE_NS,
)

SOURCE_RUNTIME["scorer_source_sha256"] = source_fingerprint

print("Restored source functions:")
for name in definition_order:
    print(" ", name)

print("\nScorer definition fingerprint:")
print(source_fingerprint)


# ------------------------------------------------------------
# Predetermined smoke test: first five records in SAVED order.
# This subset is not selected using prediction performance.
# ------------------------------------------------------------
SMOKE_INDICES = np.arange(5)
NUMERICAL_TOLERANCE = 1e-6

policy = SOURCE_STATE["protocol"]["frozen_policies"][
    "Cascade:CRUS"
]

weights = np.asarray(
    SOURCE_STATE["uncertainty_config"]["weights"],
    dtype=float,
)

smoke_rows = []
smoke_raw = []

started = time.perf_counter()

for position in SMOKE_INDICES:
    record_id = str(
        SOURCE_STATE["validation_ids"][position]
    )

    query = SOURCE_STATE["validation_text"][position]

    with torch.inference_mode():
        candidates = CASCADE_NS["prepare_candidates"](query)

        result = CASCADE_NS["score_documents"](
            query,
            candidates,
            "Cascade",
        )

    current_probabilities = np.asarray(
        result["probs"],
        dtype=float,
    )

    current_components = np.asarray(
        result["components"],
        dtype=float,
    )

    saved_probabilities = (
        SOURCE_STATE["saved_validation_probabilities"][position]
    )

    saved_components = (
        SOURCE_STATE["saved_validation_components"][position]
    )

    current_crus = CASCADE_NS["composite_crus"](
        current_components
    )

    saved_crus = float(saved_components @ weights)

    current_accept = bool(
        result["confidence"] >= policy["tau_c"]
        and current_crus <= policy["tau_u"]
    )

    saved_accept = bool(
        saved_probabilities.max() >= policy["tau_c"]
        and saved_crus <= policy["tau_u"]
    )

    probability_difference = float(
        np.max(
            np.abs(
                current_probabilities - saved_probabilities
            )
        )
    )

    component_difference = float(
        np.max(
            np.abs(
                current_components - saved_components
            )
        )
    )

    smoke_rows.append({
        "position": int(position),
        "record_id": record_id,
        "prediction_matches": (
            result["pred"]
            == SOURCE_STATE[
                "saved_validation_predictions"
            ][position]
        ),
        "acceptance_matches": (
            current_accept == saved_accept
        ),
        "probability_max_abs_difference": (
            probability_difference
        ),
        "component_max_abs_difference": (
            component_difference
        ),
        "crus_abs_difference": abs(
            current_crus - saved_crus
        ),
        "probabilities_within_1e_minus6": (
            probability_difference <= NUMERICAL_TOLERANCE
        ),
        "components_within_1e_minus6": (
            component_difference <= NUMERICAL_TOLERANCE
        ),
        "final_document_count": len(result["documents"]),
        "fallback": bool(result["fallback"]),
    })

    smoke_raw.append({
        "position": int(position),
        "record_id": record_id,
        "result": result,
        "crus": current_crus,
    })

    print(
        f"Checked source-validation record "
        f"{len(smoke_rows)}/5"
    )

SMOKE_RESULTS = pd.DataFrame(smoke_rows)
SMOKE_RAW_RESULTS = smoke_raw

elapsed = time.perf_counter() - started

print("\nFIVE-RECORD SOURCE CHECK")
print(SMOKE_RESULTS.to_string(index=False))

print(f"\nElapsed time: {elapsed:.1f} seconds")

print(
    "\nPrediction matches:",
    int(SMOKE_RESULTS["prediction_matches"].sum()),
    "/ 5",
)

print(
    "Acceptance matches:",
    int(SMOKE_RESULTS["acceptance_matches"].sum()),
    "/ 5",
)

print(
    "Maximum probability difference:",
    SMOKE_RESULTS[
        "probability_max_abs_difference"
    ].max(),
)

print(
    "Maximum component difference:",
    SMOKE_RESULTS[
        "component_max_abs_difference"
    ].max(),
)

# Five records are not sufficient for full source validation.
SOURCE_RUNTIME["source_regression_passed"] = False

print(
    "\nFull source regression remains UNVERIFIED. "
    "No external queries were evaluated."
)

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import ast
import hashlib
import json
import time

import numpy as np
import pandas as pd
import torch


# ------------------------------------------------------------
# Preconditions
# ------------------------------------------------------------
for required_name in [
    "SOURCE_STATE",
    "SOURCE_RUNTIME",
    "CASCADE_NS",
]:
    if required_name not in globals():
        raise RuntimeError(
            f"{required_name} is missing. "
            "Complete the previous restoration cells first."
        )

PROJECT_DIR = Path(
    _crus_local_path('project', '')
)

OUTPUT_DIR = PROJECT_DIR / "corrected_experiment_outputs"

AUDIT_FILE = (
    OUTPUT_DIR / "validation_retrieval_prediction_audit.csv"
)

TOLERANCE = 1e-6

SOURCE_RUNTIME["source_regression_passed"] = False


# ------------------------------------------------------------
# Utilities
# ------------------------------------------------------------
def r15_file_sha256(path):
    digest = hashlib.sha256()

    with open(path, "rb") as handle:
        for chunk in iter(
            lambda: handle.read(1024 * 1024),
            b"",
        ):
            digest.update(chunk)

    return digest.hexdigest()


def r15_model_sha256(model):
    """Fingerprint the model weights currently loaded in memory."""
    digest = hashlib.sha256()

    for name, tensor in sorted(model.state_dict().items()):
        value = tensor.detach().cpu().contiguous()

        digest.update(name.encode("utf-8"))
        digest.update(str(value.dtype).encode("utf-8"))
        digest.update(str(tuple(value.shape)).encode("utf-8"))

        raw_bytes = (
            value.reshape(-1)
            .view(torch.uint8)
            .numpy()
            .tobytes()
        )

        digest.update(raw_bytes)

    return digest.hexdigest()


def r15_parse_list(value):
    """Read a saved list without executing its contents."""
    if isinstance(value, list):
        return value

    text = str(value).strip()

    try:
        result = json.loads(text)
    except json.JSONDecodeError:
        result = ast.literal_eval(text)

    if not isinstance(result, (list, tuple)):
        raise ValueError(
            "Expected a serialized list in the saved audit."
        )

    return list(result)


def r15_parse_bool(value):
    text = str(value).strip().lower()

    if text in {"true", "1"}:
        return True

    if text in {"false", "0"}:
        return False

    raise ValueError(
        f"Unrecognized saved boolean: {value!r}"
    )


# ------------------------------------------------------------
# Verify the saved validation audit.
# ------------------------------------------------------------
audit = pd.read_csv(
    AUDIT_FILE,
    dtype=str,
    keep_default_na=False,
)

audit = audit.loc[
    audit["method"].eq("Cascade")
].copy()

if len(audit) != 258:
    raise RuntimeError(
        f"Expected 258 Cascade audit rows; found {len(audit)}."
    )

if audit["record_id"].duplicated().any():
    raise RuntimeError("Duplicate Cascade audit IDs.")

validation_ids = [
    str(value)
    for value in SOURCE_STATE["validation_ids"]
]

if set(audit["record_id"]) != set(validation_ids):
    raise RuntimeError(
        "Saved audit IDs differ from the validation fixture."
    )

audit = audit.set_index("record_id").loc[validation_ids]

if not np.array_equal(
    audit["true_label"].to_numpy(dtype=str),
    SOURCE_STATE["validation_labels"],
):
    raise RuntimeError("Audit reference-label mismatch.")

if not np.array_equal(
    audit["prediction"].to_numpy(dtype=str),
    SOURCE_STATE["saved_validation_predictions"],
):
    raise RuntimeError("Audit prediction mismatch.")


# ------------------------------------------------------------
# Establish a run identity for safe checkpoint reuse.
# These hashes identify CURRENT restored artifacts.
# They do not prove historical source model-weight identity.
# ------------------------------------------------------------
print("Fingerprinting the loaded models...")

run_identity = {
    "scorer_sha256": SOURCE_RUNTIME[
        "scorer_source_sha256"
    ],
    "protocol_sha256": r15_file_sha256(
        OUTPUT_DIR / "final_pretest_protocol.json"
    ),
    "validation_arrays_sha256": r15_file_sha256(
        OUTPUT_DIR / "validation_Cascade_arrays.npz"
    ),
    "training_embeddings_sha256": r15_file_sha256(
        OUTPUT_DIR / "training_dense_embeddings.npz"
    ),
    "validation_audit_sha256": r15_file_sha256(AUDIT_FILE),
    "source_pool_sha256": r15_file_sha256(
        PROJECT_DIR
        / "data_audit"
        / "constrained_dialogue_pool_with_splits.csv"
    ),
    "dense_weights_sha256": r15_model_sha256(
        SOURCE_RUNTIME["embedder"]
    ),
    "cross_encoder_weights_sha256": r15_model_sha256(
        SOURCE_RUNTIME["cross_encoder"].model
    ),
    "device": SOURCE_RUNTIME["device"],
    "torch_version": torch.__version__,
    "numerical_tolerance": TOLERANCE,
}

if "SOURCE_REGRESSION_RUN" not in globals():
    timestamp = datetime.now(timezone.utc).strftime(
        "%Y%m%dT%H%M%S%fZ"
    )

    SOURCE_REGRESSION_RUN = (
        PROJECT_DIR
        / f"r1_5_source_reproduction_{timestamp}"
    )

    SOURCE_REGRESSION_RUN.mkdir(
        parents=True,
        exist_ok=False,
    )

    (SOURCE_REGRESSION_RUN / "records").mkdir()

    (SOURCE_REGRESSION_RUN / "run_identity.json").write_text(
        json.dumps(run_identity, indent=2),
        encoding="utf-8",
    )

else:
    SOURCE_REGRESSION_RUN = Path(
        SOURCE_REGRESSION_RUN
    )

    saved_identity = json.loads(
        (
            SOURCE_REGRESSION_RUN / "run_identity.json"
        ).read_text(encoding="utf-8")
    )

    if saved_identity != run_identity:
        raise RuntimeError(
            "Checkpoint identity differs from the current "
            "models or source files. Do not reuse this run."
        )

print("Reproduction folder:", SOURCE_REGRESSION_RUN)


# ------------------------------------------------------------
# Score or restore each validation record.
# No gold label is passed into the scorer.
# ------------------------------------------------------------
policy = SOURCE_STATE["protocol"]["frozen_policies"][
    "Cascade:CRUS"
]

weights = np.asarray(
    SOURCE_STATE["uncertainty_config"]["weights"],
    dtype=float,
)

checks = []
regression_records = []

started = time.perf_counter()
newly_scored = 0
reused = 0

for position, record_id in enumerate(validation_ids):
    query = SOURCE_STATE["validation_text"][position]

    query_sha256 = hashlib.sha256(
        query.encode("utf-8")
    ).hexdigest()

    checkpoint = (
        SOURCE_REGRESSION_RUN
        / "records"
        / f"{position:04d}.json"
    )

    if checkpoint.is_file():
        current = json.loads(
            checkpoint.read_text(encoding="utf-8")
        )

        if (
            current["record_id"] != record_id
            or current["query_sha256"] != query_sha256
        ):
            raise RuntimeError(
                f"Checkpoint/input mismatch at position {position}."
            )

        reused += 1

    else:
        with torch.inference_mode():
            candidates = CASCADE_NS[
                "prepare_candidates"
            ](query)

            result = CASCADE_NS["score_documents"](
                query,
                candidates,
                "Cascade",
            )

        components = np.asarray(
            result["components"],
            dtype=float,
        )

        current = {
            "record_id": record_id,
            "query_sha256": query_sha256,
            "pred_label": str(result["pred"]),
            "probabilities": np.asarray(
                result["probs"],
                dtype=float,
            ).tolist(),
            "h_pred": float(components[0]),
            "h_retrieval": float(components[1]),
            "v_rrf": float(components[2]),
            "crus": float(
                CASCADE_NS["composite_crus"](components)
            ),
            "candidate_ids": [
                str(SOURCE_STATE["train_ids"][index])
                for index, _ in result["documents"]
            ],
            "candidate_rrf_scores": [
                float(score)
                for _, score in result["documents"]
            ],
            "shortlist": [
                str(label)
                for label in result["shortlist"]
            ],
            "fallback": bool(result["fallback"]),
        }

        temporary = checkpoint.with_suffix(".json.tmp")

        temporary.write_text(
            json.dumps(
                current,
                ensure_ascii=False,
                allow_nan=False,
            ),
            encoding="utf-8",
        )

        temporary.replace(checkpoint)
        newly_scored += 1

    probabilities = np.asarray(
        current["probabilities"],
        dtype=float,
    )

    components = np.array([
        current["h_pred"],
        current["h_retrieval"],
        current["v_rrf"],
    ])

    expected_probabilities = (
        SOURCE_STATE[
            "saved_validation_probabilities"
        ][position]
    )

    expected_components = (
        SOURCE_STATE[
            "saved_validation_components"
        ][position]
    )

    expected_crus = float(
        expected_components @ weights
    )

    current_accept = bool(
        probabilities.max() >= policy["tau_c"]
        and current["crus"] <= policy["tau_u"]
    )

    expected_accept = bool(
        expected_probabilities.max() >= policy["tau_c"]
        and expected_crus <= policy["tau_u"]
    )

    expected_documents = [
        str(value)
        for value in r15_parse_list(
            audit.loc[record_id, "retrieved_training_ids"]
        )
    ]

    expected_shortlist = [
        str(value)
        for value in r15_parse_list(
            audit.loc[record_id, "shortlist"]
        )
    ]

    probability_difference = float(
        np.max(
            np.abs(
                probabilities - expected_probabilities
            )
        )
    )

    component_difference = float(
        np.max(
            np.abs(
                components - expected_components
            )
        )
    )

    checks.append({
        "record_id": record_id,
        "prediction_matches": (
            current["pred_label"]
            == SOURCE_STATE[
                "saved_validation_predictions"
            ][position]
        ),
        "acceptance_matches": (
            current_accept == expected_accept
        ),
        "document_ids_and_order_match": (
            current["candidate_ids"] == expected_documents
        ),
        "shortlist_and_order_match": (
            current["shortlist"] == expected_shortlist
        ),
        "fallback_matches": (
            current["fallback"]
            == r15_parse_bool(
                audit.loc[record_id, "fallback"]
            )
        ),
        "probability_max_abs_difference": (
            probability_difference
        ),
        "component_max_abs_difference": (
            component_difference
        ),
        "crus_abs_difference": abs(
            current["crus"] - expected_crus
        ),
        "probabilities_within_tolerance": (
            probability_difference <= TOLERANCE
        ),
        "components_within_tolerance": (
            component_difference <= TOLERANCE
        ),
        "accepted": current_accept,
        "correct": (
            current["pred_label"]
            == SOURCE_STATE["validation_labels"][position]
        ),
    })

    regression_records.append(current)

    if (position + 1) % 10 == 0 or position == 257:
        elapsed = time.perf_counter() - started

        print(
            f"Completed {position + 1}/258 "
            f"| newly scored: {newly_scored} "
            f"| reused: {reused} "
            f"| elapsed: {elapsed / 60:.1f} min"
        )


# ------------------------------------------------------------
# Summarize all checks.
# ------------------------------------------------------------
comparison = pd.DataFrame(checks)

required_checks = [
    "prediction_matches",
    "acceptance_matches",
    "document_ids_and_order_match",
    "shortlist_and_order_match",
    "fallback_matches",
    "probabilities_within_tolerance",
    "components_within_tolerance",
]

correct = comparison["correct"].to_numpy(dtype=bool)
accepted = comparison["accepted"].to_numpy(dtype=bool)

actual_counts = {
    "forced_correct": int(correct.sum()),
    "accepted": int(accepted.sum()),
    "accepted_correct": int(correct[accepted].sum()),
}

expected_counts = {
    "forced_correct": 174,
    "accepted": 132,
    "accepted_correct": 124,
}

passed = bool(
    comparison[required_checks].all().all()
    and actual_counts == expected_counts
)

comparison.to_csv(
    SOURCE_REGRESSION_RUN / "comparison.csv",
    index=False,
)

mismatches = comparison.loc[
    ~comparison[required_checks].all(axis=1)
]

mismatches.to_csv(
    SOURCE_REGRESSION_RUN / "mismatches.csv",
    index=False,
)

summary = {
    "passed": passed,
    "records": len(comparison),
    "counts": actual_counts,
    "check_matches": {
        column: int(comparison[column].sum())
        for column in required_checks
    },
    "maximum_probability_difference": float(
        comparison[
            "probability_max_abs_difference"
        ].max()
    ),
    "maximum_component_difference": float(
        comparison[
            "component_max_abs_difference"
        ].max()
    ),
    "maximum_crus_difference": float(
        comparison["crus_abs_difference"].max()
    ),
    "tolerance": TOLERANCE,
    "device": SOURCE_RUNTIME["device"],
    "scope": (
        "Numerical reproduction on the corrected source "
        "validation set; not external validation or proof "
        "of historical model-file identity."
    ),
}

(
    SOURCE_REGRESSION_RUN / "regression_summary.json"
).write_text(
    json.dumps(summary, indent=2),
    encoding="utf-8",
)

# Export the ORIGINAL saved probability/component values,
# combined with document traces only after trace agreement.
if passed:
    fixture_path = (
        SOURCE_REGRESSION_RUN
        / "source_validation_regression.jsonl"
    )

    with fixture_path.open(
        "w",
        encoding="utf-8",
    ) as handle:
        for position, record in enumerate(regression_records):
            original_components = (
                SOURCE_STATE[
                    "saved_validation_components"
                ][position]
            )

            fixture = {
                **record,
                "pred_label": str(
                    SOURCE_STATE[
                        "saved_validation_predictions"
                    ][position]
                ),
                "probabilities": (
                    SOURCE_STATE[
                        "saved_validation_probabilities"
                    ][position].tolist()
                ),
                "h_pred": float(original_components[0]),
                "h_retrieval": float(original_components[1]),
                "v_rrf": float(original_components[2]),
                "crus": float(original_components @ weights),
            }

            handle.write(
                json.dumps(
                    fixture,
                    ensure_ascii=False,
                    allow_nan=False,
                )
                + "\n"
            )

SOURCE_RUNTIME["source_regression_passed"] = passed
SOURCE_RUNTIME["source_regression_directory"] = str(
    SOURCE_REGRESSION_RUN
)

print("\nFULL SOURCE-VALIDATION REPRODUCTION")
print(json.dumps(summary, indent=2))
print("\nResults saved in:", SOURCE_REGRESSION_RUN)

if not passed:
    print(
        "\nSTOP: inspect mismatches before external inference. "
        "Do not retune thresholds or relax tolerance to force a pass."
    )
else:
    print(
        "\nSource-validation reproduction passed. "
        "No external queries have been evaluated."
    )

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import ast
import hashlib
import json
import time

import numpy as np
import pandas as pd
import torch


# ------------------------------------------------------------
# Preconditions
# ------------------------------------------------------------
for required_name in [
    "SOURCE_STATE",
    "SOURCE_RUNTIME",
    "CASCADE_NS",
]:
    if required_name not in globals():
        raise RuntimeError(
            f"{required_name} is missing. "
            "Complete the previous restoration cells first."
        )

PROJECT_DIR = Path(
    _crus_local_path('project', '')
)

OUTPUT_DIR = PROJECT_DIR / "corrected_experiment_outputs"

AUDIT_FILE = (
    OUTPUT_DIR / "validation_retrieval_prediction_audit.csv"
)

TOLERANCE = 1e-6

SOURCE_RUNTIME["source_regression_passed"] = False


# ------------------------------------------------------------
# Utilities
# ------------------------------------------------------------
def r15_file_sha256(path):
    digest = hashlib.sha256()

    with open(path, "rb") as handle:
        for chunk in iter(
            lambda: handle.read(1024 * 1024),
            b"",
        ):
            digest.update(chunk)

    return digest.hexdigest()


def r15_model_sha256(model):
    """Fingerprint the model weights currently loaded in memory."""
    digest = hashlib.sha256()

    for name, tensor in sorted(model.state_dict().items()):
        value = tensor.detach().cpu().contiguous()

        digest.update(name.encode("utf-8"))
        digest.update(str(value.dtype).encode("utf-8"))
        digest.update(str(tuple(value.shape)).encode("utf-8"))

        raw_bytes = (
            value.reshape(-1)
            .view(torch.uint8)
            .numpy()
            .tobytes()
        )

        digest.update(raw_bytes)

    return digest.hexdigest()


def r15_parse_list(value):
    """Read a saved list without executing its contents."""
    if isinstance(value, list):
        return value

    text = str(value).strip()

    try:
        result = json.loads(text)
    except json.JSONDecodeError:
        result = ast.literal_eval(text)

    if not isinstance(result, (list, tuple)):
        raise ValueError(
            "Expected a serialized list in the saved audit."
        )

    return list(result)


def r15_parse_bool(value):
    text = str(value).strip().lower()

    if text in {"true", "1"}:
        return True

    if text in {"false", "0"}:
        return False

    raise ValueError(
        f"Unrecognized saved boolean: {value!r}"
    )


# ------------------------------------------------------------
# Verify the saved validation audit.
# ------------------------------------------------------------
audit = pd.read_csv(
    AUDIT_FILE,
    dtype=str,
    keep_default_na=False,
)

audit = audit.loc[
    audit["method"].eq("Cascade")
].copy()

if len(audit) != 258:
    raise RuntimeError(
        f"Expected 258 Cascade audit rows; found {len(audit)}."
    )

if audit["record_id"].duplicated().any():
    raise RuntimeError("Duplicate Cascade audit IDs.")

validation_ids = [
    str(value)
    for value in SOURCE_STATE["validation_ids"]
]

if set(audit["record_id"]) != set(validation_ids):
    raise RuntimeError(
        "Saved audit IDs differ from the validation fixture."
    )

audit = audit.set_index("record_id").loc[validation_ids]

if not np.array_equal(
    audit["true_label"].to_numpy(dtype=str),
    SOURCE_STATE["validation_labels"],
):
    raise RuntimeError("Audit reference-label mismatch.")

if not np.array_equal(
    audit["prediction"].to_numpy(dtype=str),
    SOURCE_STATE["saved_validation_predictions"],
):
    raise RuntimeError("Audit prediction mismatch.")


# ------------------------------------------------------------
# Establish a run identity for safe checkpoint reuse.
# These hashes identify CURRENT restored artifacts.
# They do not prove historical source model-weight identity.
# ------------------------------------------------------------
print("Fingerprinting the loaded models...")

run_identity = {
    "scorer_sha256": SOURCE_RUNTIME[
        "scorer_source_sha256"
    ],
    "protocol_sha256": r15_file_sha256(
        OUTPUT_DIR / "final_pretest_protocol.json"
    ),
    "validation_arrays_sha256": r15_file_sha256(
        OUTPUT_DIR / "validation_Cascade_arrays.npz"
    ),
    "training_embeddings_sha256": r15_file_sha256(
        OUTPUT_DIR / "training_dense_embeddings.npz"
    ),
    "validation_audit_sha256": r15_file_sha256(AUDIT_FILE),
    "source_pool_sha256": r15_file_sha256(
        PROJECT_DIR
        / "data_audit"
        / "constrained_dialogue_pool_with_splits.csv"
    ),
    "dense_weights_sha256": r15_model_sha256(
        SOURCE_RUNTIME["embedder"]
    ),
    "cross_encoder_weights_sha256": r15_model_sha256(
        SOURCE_RUNTIME["cross_encoder"].model
    ),
    "device": SOURCE_RUNTIME["device"],
    "torch_version": torch.__version__,
    "numerical_tolerance": TOLERANCE,
}

if "SOURCE_REGRESSION_RUN" not in globals():
    timestamp = datetime.now(timezone.utc).strftime(
        "%Y%m%dT%H%M%S%fZ"
    )

    SOURCE_REGRESSION_RUN = (
        PROJECT_DIR
        / f"r1_5_source_reproduction_{timestamp}"
    )

    SOURCE_REGRESSION_RUN.mkdir(
        parents=True,
        exist_ok=False,
    )

    (SOURCE_REGRESSION_RUN / "records").mkdir()

    (SOURCE_REGRESSION_RUN / "run_identity.json").write_text(
        json.dumps(run_identity, indent=2),
        encoding="utf-8",
    )

else:
    SOURCE_REGRESSION_RUN = Path(
        SOURCE_REGRESSION_RUN
    )

    saved_identity = json.loads(
        (
            SOURCE_REGRESSION_RUN / "run_identity.json"
        ).read_text(encoding="utf-8")
    )

    if saved_identity != run_identity:
        raise RuntimeError(
            "Checkpoint identity differs from the current "
            "models or source files. Do not reuse this run."
        )

print("Reproduction folder:", SOURCE_REGRESSION_RUN)


# ------------------------------------------------------------
# Score or restore each validation record.
# No gold label is passed into the scorer.
# ------------------------------------------------------------
policy = SOURCE_STATE["protocol"]["frozen_policies"][
    "Cascade:CRUS"
]

weights = np.asarray(
    SOURCE_STATE["uncertainty_config"]["weights"],
    dtype=float,
)

checks = []
regression_records = []

started = time.perf_counter()
newly_scored = 0
reused = 0

for position, record_id in enumerate(validation_ids):
    query = SOURCE_STATE["validation_text"][position]

    query_sha256 = hashlib.sha256(
        query.encode("utf-8")
    ).hexdigest()

    checkpoint = (
        SOURCE_REGRESSION_RUN
        / "records"
        / f"{position:04d}.json"
    )

    if checkpoint.is_file():
        current = json.loads(
            checkpoint.read_text(encoding="utf-8")
        )

        if (
            current["record_id"] != record_id
            or current["query_sha256"] != query_sha256
        ):
            raise RuntimeError(
                f"Checkpoint/input mismatch at position {position}."
            )

        reused += 1

    else:
        with torch.inference_mode():
            candidates = CASCADE_NS[
                "prepare_candidates"
            ](query)

            result = CASCADE_NS["score_documents"](
                query,
                candidates,
                "Cascade",
            )

        components = np.asarray(
            result["components"],
            dtype=float,
        )

        current = {
            "record_id": record_id,
            "query_sha256": query_sha256,
            "pred_label": str(result["pred"]),
            "probabilities": np.asarray(
                result["probs"],
                dtype=float,
            ).tolist(),
            "h_pred": float(components[0]),
            "h_retrieval": float(components[1]),
            "v_rrf": float(components[2]),
            "crus": float(
                CASCADE_NS["composite_crus"](components)
            ),
            "candidate_ids": [
                str(SOURCE_STATE["train_ids"][index])
                for index, _ in result["documents"]
            ],
            "candidate_rrf_scores": [
                float(score)
                for _, score in result["documents"]
            ],
            "shortlist": [
                str(label)
                for label in result["shortlist"]
            ],
            "fallback": bool(result["fallback"]),
        }

        temporary = checkpoint.with_suffix(".json.tmp")

        temporary.write_text(
            json.dumps(
                current,
                ensure_ascii=False,
                allow_nan=False,
            ),
            encoding="utf-8",
        )

        temporary.replace(checkpoint)
        newly_scored += 1

    probabilities = np.asarray(
        current["probabilities"],
        dtype=float,
    )

    components = np.array([
        current["h_pred"],
        current["h_retrieval"],
        current["v_rrf"],
    ])

    expected_probabilities = (
        SOURCE_STATE[
            "saved_validation_probabilities"
        ][position]
    )

    expected_components = (
        SOURCE_STATE[
            "saved_validation_components"
        ][position]
    )

    expected_crus = float(
        expected_components @ weights
    )

    current_accept = bool(
        probabilities.max() >= policy["tau_c"]
        and current["crus"] <= policy["tau_u"]
    )

    expected_accept = bool(
        expected_probabilities.max() >= policy["tau_c"]
        and expected_crus <= policy["tau_u"]
    )

    expected_documents = [
        str(value)
        for value in r15_parse_list(
            audit.loc[record_id, "retrieved_training_ids"]
        )
    ]

    expected_shortlist = [
        str(value)
        for value in r15_parse_list(
            audit.loc[record_id, "shortlist"]
        )
    ]

    probability_difference = float(
        np.max(
            np.abs(
                probabilities - expected_probabilities
            )
        )
    )

    component_difference = float(
        np.max(
            np.abs(
                components - expected_components
            )
        )
    )

    checks.append({
        "record_id": record_id,
        "prediction_matches": (
            current["pred_label"]
            == SOURCE_STATE[
                "saved_validation_predictions"
            ][position]
        ),
        "acceptance_matches": (
            current_accept == expected_accept
        ),
        "document_ids_and_order_match": (
            current["candidate_ids"] == expected_documents
        ),
        "shortlist_and_order_match": (
            current["shortlist"] == expected_shortlist
        ),
        "fallback_matches": (
            current["fallback"]
            == r15_parse_bool(
                audit.loc[record_id, "fallback"]
            )
        ),
        "probability_max_abs_difference": (
            probability_difference
        ),
        "component_max_abs_difference": (
            component_difference
        ),
        "crus_abs_difference": abs(
            current["crus"] - expected_crus
        ),
        "probabilities_within_tolerance": (
            probability_difference <= TOLERANCE
        ),
        "components_within_tolerance": (
            component_difference <= TOLERANCE
        ),
        "accepted": current_accept,
        "correct": (
            current["pred_label"]
            == SOURCE_STATE["validation_labels"][position]
        ),
    })

    regression_records.append(current)

    if (position + 1) % 10 == 0 or position == 257:
        elapsed = time.perf_counter() - started

        print(
            f"Completed {position + 1}/258 "
            f"| newly scored: {newly_scored} "
            f"| reused: {reused} "
            f"| elapsed: {elapsed / 60:.1f} min"
        )


# ------------------------------------------------------------
# Summarize all checks.
# ------------------------------------------------------------
comparison = pd.DataFrame(checks)

required_checks = [
    "prediction_matches",
    "acceptance_matches",
    "document_ids_and_order_match",
    "shortlist_and_order_match",
    "fallback_matches",
    "probabilities_within_tolerance",
    "components_within_tolerance",
]

correct = comparison["correct"].to_numpy(dtype=bool)
accepted = comparison["accepted"].to_numpy(dtype=bool)

actual_counts = {
    "forced_correct": int(correct.sum()),
    "accepted": int(accepted.sum()),
    "accepted_correct": int(correct[accepted].sum()),
}

expected_counts = {
    "forced_correct": 174,
    "accepted": 132,
    "accepted_correct": 124,
}

passed = bool(
    comparison[required_checks].all().all()
    and actual_counts == expected_counts
)

comparison.to_csv(
    SOURCE_REGRESSION_RUN / "comparison.csv",
    index=False,
)

mismatches = comparison.loc[
    ~comparison[required_checks].all(axis=1)
]

mismatches.to_csv(
    SOURCE_REGRESSION_RUN / "mismatches.csv",
    index=False,
)

summary = {
    "passed": passed,
    "records": len(comparison),
    "counts": actual_counts,
    "check_matches": {
        column: int(comparison[column].sum())
        for column in required_checks
    },
    "maximum_probability_difference": float(
        comparison[
            "probability_max_abs_difference"
        ].max()
    ),
    "maximum_component_difference": float(
        comparison[
            "component_max_abs_difference"
        ].max()
    ),
    "maximum_crus_difference": float(
        comparison["crus_abs_difference"].max()
    ),
    "tolerance": TOLERANCE,
    "device": SOURCE_RUNTIME["device"],
    "scope": (
        "Numerical reproduction on the corrected source "
        "validation set; not external validation or proof "
        "of historical model-file identity."
    ),
}

(
    SOURCE_REGRESSION_RUN / "regression_summary.json"
).write_text(
    json.dumps(summary, indent=2),
    encoding="utf-8",
)

# Export the ORIGINAL saved probability/component values,
# combined with document traces only after trace agreement.
if passed:
    fixture_path = (
        SOURCE_REGRESSION_RUN
        / "source_validation_regression.jsonl"
    )

    with fixture_path.open(
        "w",
        encoding="utf-8",
    ) as handle:
        for position, record in enumerate(regression_records):
            original_components = (
                SOURCE_STATE[
                    "saved_validation_components"
                ][position]
            )

            fixture = {
                **record,
                "pred_label": str(
                    SOURCE_STATE[
                        "saved_validation_predictions"
                    ][position]
                ),
                "probabilities": (
                    SOURCE_STATE[
                        "saved_validation_probabilities"
                    ][position].tolist()
                ),
                "h_pred": float(original_components[0]),
                "h_retrieval": float(original_components[1]),
                "v_rrf": float(original_components[2]),
                "crus": float(original_components @ weights),
            }

            handle.write(
                json.dumps(
                    fixture,
                    ensure_ascii=False,
                    allow_nan=False,
                )
                + "\n"
            )

SOURCE_RUNTIME["source_regression_passed"] = passed
SOURCE_RUNTIME["source_regression_directory"] = str(
    SOURCE_REGRESSION_RUN
)

print("\nFULL SOURCE-VALIDATION REPRODUCTION")
print(json.dumps(summary, indent=2))
print("\nResults saved in:", SOURCE_REGRESSION_RUN)

if not passed:
    print(
        "\nSTOP: inspect mismatches before external inference. "
        "Do not retune thresholds or relax tolerance to force a pass."
    )
else:
    print(
        "\nSource-validation reproduction passed. "
        "No external queries have been evaluated."
    )

In [ ]:
import numpy as np
import torch

In [ ]:
# Historical bodyless duplicate declaration omitted.
# The complete guarded predict_external_cascade implementation
# is preserved in the following cell.


In [ ]:
import numpy as np
import torch


# Require the verified source system.
for required_name in [
    "SOURCE_STATE",
    "SOURCE_RUNTIME",
    "CASCADE_NS",
]:
    if required_name not in globals():
        raise RuntimeError(
            f"{required_name} is missing. "
            "Stop and report this message."
        )

if not SOURCE_RUNTIME.get(
    "source_regression_passed",
    False,
):
    raise RuntimeError(
        "The full source-validation reproduction "
        "must pass before external prediction."
    )


def predict_external_cascade(records):
    """
    Predict using the verified frozen source Cascade.

    Each input must contain exactly:
        record_id: unique identifier
        query: dialogue text only

    Do not include gold labels or reference notes.
    """

    if not isinstance(records, list):
        raise TypeError(
            "records must be a list of dictionaries."
        )

    prepared_records = []
    seen_ids = set()

    # Validate all inputs before scoring.
    for record in records:
        if not isinstance(record, dict):
            raise TypeError(
                "Each record must be a dictionary."
            )

        if set(record) != {"record_id", "query"}:
            raise ValueError(
                "Each record must contain exactly "
                "'record_id' and 'query'. "
                "Do not pass gold labels or reference notes."
            )

        record_id = str(record["record_id"]).strip()

        if not record_id:
            raise ValueError("Empty record_id.")

        if record_id in seen_ids:
            raise ValueError(
                f"Duplicate record_id: {record_id}"
            )

        seen_ids.add(record_id)

        if not isinstance(record["query"], str):
            raise TypeError(
                f"{record_id}: query must be text."
            )

        query = " ".join(
            record["query"].lower().split()
        )

        if not query:
            raise ValueError(
                f"{record_id}: empty normalized dialogue."
            )

        prepared_records.append({
            "record_id": record_id,
            "query": query,
        })

    outputs = []

    for record in prepared_records:
        with torch.inference_mode():
            candidates = CASCADE_NS[
                "prepare_candidates"
            ](record["query"])

            result = CASCADE_NS["score_documents"](
                record["query"],
                candidates,
                "Cascade",
            )

        components = np.asarray(
            result["components"],
            dtype=float,
        )

        probabilities = np.asarray(
            result["probs"],
            dtype=float,
        )

        outputs.append({
            "record_id": record["record_id"],
            "pred_label": str(result["pred"]),
            "probabilities": probabilities.tolist(),
            "h_pred": float(components[0]),
            "h_retrieval": float(components[1]),
            "v_rrf": float(components[2]),
            "crus": float(
                CASCADE_NS["composite_crus"](components)
            ),
            "candidate_ids": [
                str(SOURCE_STATE["train_ids"][index])
                for index, _ in result["documents"]
            ],
            "candidate_rrf_scores": [
                float(score)
                for _, score in result["documents"]
            ],
            "shortlist": [
                str(label)
                for label in result["shortlist"]
            ],
            "fallback": bool(result["fallback"]),
        })

    return outputs


SOURCE_RUNTIME["external_predictor"] = (
    predict_external_cascade
)

print("IN-NOTEBOOK ADAPTER READY")
print("Uses the verified source Cascade.")
print("No training or threshold selection performed.")
print("No external records scored.")

In [ ]:
from pathlib import Path
import io
import zipfile

import pandas as pd


# ------------------------------------------------------------
# OPTION A: Enter existing Colab/Google Drive file paths.
#
# Example:
# INPUT_PATHS = [
#     "/content/drive/MyDrive/your_folder/your_file.zip",
#     "/content/drive/MyDrive/your_folder/your_file.csv",
# ]
#
# OPTION B: Leave empty to select files from your computer.
# ------------------------------------------------------------
INPUT_PATHS = []

EXTERNAL_SCHEMA_INSPECTION = []


def inspect_csv(file_object, file_name):
    try:
        df = pd.read_csv(
            file_object,
            dtype=str,
            keep_default_na=False,
        )

        result = {
            "file": file_name,
            "rows": len(df),
            "columns": df.columns.tolist(),
        }

        EXTERNAL_SCHEMA_INSPECTION.append(result)

        print("\nCSV FILE:", file_name)
        print("Rows:", result["rows"])
        print("Columns:", result["columns"])

    except Exception as exc:
        print("\nCould not inspect:", file_name)
        print(type(exc).__name__, str(exc))


def inspect_zip(file_object, archive_name):
    with zipfile.ZipFile(file_object, "r") as archive:
        members = [
            item
            for item in archive.infolist()
            if not item.is_dir()
            and "__macosx" not in item.filename.lower()
        ]

        print("\nZIP FILE:", archive_name)
        print("Archive contents:")

        for item in members[:100]:
            print(
                f"  {item.filename} "
                f"({item.file_size:,} bytes)"
            )

        if len(members) > 100:
            print(f"  ... {len(members) - 100} more files")

        for item in members:
            if not item.filename.lower().endswith(".csv"):
                continue

            if item.file_size > 50_000_000:
                print("Skipped large CSV:", item.filename)
                continue

            with archive.open(item, "r") as handle:
                inspect_csv(
                    handle,
                    f"{archive_name} :: {item.filename}",
                )


def inspect_file(file_object, file_name):
    suffix = Path(file_name).suffix.lower()

    if suffix == ".zip":
        inspect_zip(file_object, file_name)

    elif suffix == ".csv":
        inspect_csv(file_object, file_name)

    else:
        print(
            f"Not parsed: {file_name}. "
            "Expected a .zip or .csv file."
        )


# ------------------------------------------------------------
# Inspect existing files. Nothing is downloaded from the web.
# ------------------------------------------------------------
if INPUT_PATHS:
    for file_path in INPUT_PATHS:
        path = Path(file_path)

        if not path.is_file():
            print("File not found:", path)
            continue

        with path.open("rb") as handle:
            inspect_file(handle, str(path))

else:
    pass  # Colab service replaced by local workflow.

    print("Select the ZIP or CSV files you already downloaded.")
    uploaded_files = _crus_upload_local_files()

    for file_name, file_bytes in uploaded_files.items():
        inspect_file(
            io.BytesIO(file_bytes),
            file_name,
        )


print("\n" + "=" * 70)
print("INSPECTION COMPLETE")
print("CSV files successfully inspected:",
      len(EXTERNAL_SCHEMA_INSPECTION))
print("No dialogue or note text printed.")
print("No labels generated.")
print("No model inference performed.")

In [ ]:
from pathlib import Path
import zipfile

kit_path = Path("CRUS_External_Transfer_Starter_Kit.zip")

if not kit_path.is_file():
    raise FileNotFoundError(
        "Starter-kit ZIP was not found in the current Colab folder. "
        "Use the filename shown in your upload output."
    )

files_to_read = [
    "README_external_transfer.md",
    "config_template.json",
    "external_attestation_template.json",
    "requirements.txt",
]

with zipfile.ZipFile(kit_path, "r") as archive:
    members = [
        name
        for name in archive.namelist()
        if not name.endswith("/")
    ]

    for requested_name in files_to_read:
        matches = [
            name
            for name in members
            if Path(name).name == requested_name
        ]

        print("\n" + "=" * 80)
        print(requested_name)
        print("=" * 80)

        if len(matches) != 1:
            print(
                "Expected exactly one matching file; "
                f"found {len(matches)}. Skipping."
            )
            continue

        content = archive.read(matches[0]).decode("utf-8-sig")
        print(content)

print("\nDocumentation inspection complete.")
print("No scripts executed, files extracted, or predictions generated.")

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import zipfile

import pandas as pd
import requests
pass  # Colab service replaced by local workflow.


# ------------------------------------------------------------
# 1. Mount Drive and create a NEW dataset-audit folder.
# ------------------------------------------------------------
if not Path(_crus_local_path('drive', '')).is_dir():
    _crus_prepare_saved_workspace()

project_dir = Path(
    _crus_local_path('project', '')
)

timestamp = datetime.now(timezone.utc).strftime(
    "%Y%m%dT%H%M%S%fZ"
)

ACI_AUDIT_DIR = (
    project_dir
    / f"aci_bench_external_data_audit_{timestamp}"
)

ACI_AUDIT_DIR.mkdir(
    parents=True,
    exist_ok=False,
)

article_id = 22494601
expected_file_id = 41498793

metadata_url = (
    f"https://api.figshare.com/v2/articles/{article_id}"
)


# ------------------------------------------------------------
# 2. Resolve the official archive from Figshare metadata.
# ------------------------------------------------------------
print("Reading ACI-Bench download metadata...")

response = requests.get(metadata_url, timeout=60)
response.raise_for_status()
metadata = response.json()

matching_files = [
    item
    for item in metadata.get("files", [])
    if str(item.get("id")) == str(expected_file_id)
]

if len(matching_files) != 1:
    print("Available files:")

    for item in metadata.get("files", []):
        print(item.get("id"), item.get("name"))

    raise RuntimeError(
        "Expected official archive was not found. "
        "Stop here and share this output."
    )

file_info = matching_files[0]
download_url = file_info.get("download_url")

if not download_url:
    raise RuntimeError("No download URL found in metadata.")

print("Dataset:", metadata.get("title"))
print("Archive:", file_info.get("name"))


# ------------------------------------------------------------
# 3. Download and validate the archive.
# ------------------------------------------------------------
archive_path = ACI_AUDIT_DIR / "aci_bench_original.zip"
partial_path = ACI_AUDIT_DIR / "aci_bench_original.zip.part"

sha256 = hashlib.sha256()
md5 = hashlib.md5()
downloaded_bytes = 0
maximum_bytes = 50_000_000

with requests.get(
    download_url,
    stream=True,
    timeout=(30, 180),
) as download:
    download.raise_for_status()

    with partial_path.open("xb") as handle:
        for chunk in download.iter_content(
            chunk_size=1024 * 1024
        ):
            if not chunk:
                continue

            downloaded_bytes += len(chunk)

            if downloaded_bytes > maximum_bytes:
                raise RuntimeError(
                    "Download exceeded the 50 MB audit limit."
                )

            handle.write(chunk)
            sha256.update(chunk)
            md5.update(chunk)

expected_md5 = (
    file_info.get("computed_md5")
    or file_info.get("supplied_md5")
)

if (
    expected_md5
    and md5.hexdigest().lower() != expected_md5.lower()
):
    raise RuntimeError("Archive checksum mismatch.")

if not zipfile.is_zipfile(partial_path):
    raise RuntimeError("Downloaded file is not a valid ZIP.")

partial_path.replace(archive_path)

provenance = {
    "article_id": article_id,
    "file_id": expected_file_id,
    "title": metadata.get("title"),
    "doi": metadata.get("doi"),
    "license": metadata.get("license"),
    "download_url": download_url,
    "downloaded_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "archive_bytes": downloaded_bytes,
    "sha256": sha256.hexdigest(),
    "md5": md5.hexdigest(),
    "status": "candidate dataset; suitability not yet verified",
}

(ACI_AUDIT_DIR / "download_provenance.json").write_text(
    json.dumps(provenance, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

(ACI_AUDIT_DIR / "figshare_metadata.json").write_text(
    json.dumps(metadata, indent=2, ensure_ascii=False),
    encoding="utf-8",
)


# ------------------------------------------------------------
# 4. Inspect CSV files without extracting archive contents.
#    No dialogue or note text is printed.
# ------------------------------------------------------------
ACI_SCHEMA_INSPECTION = []

with zipfile.ZipFile(archive_path, "r") as archive:
    members = [
        item
        for item in archive.infolist()
        if not item.is_dir()
        and "__macosx" not in item.filename.lower()
    ]

    print("\nARCHIVE CONTENTS")
    print("=" * 80)

    for item in members[:100]:
        print(
            f"{item.filename} "
            f"({item.file_size:,} bytes)"
        )

    if len(members) > 100:
        print(f"... {len(members) - 100} additional files")

    print("\nCSV STRUCTURE")
    print("=" * 80)

    for item in members:
        if not item.filename.lower().endswith(".csv"):
            continue

        if item.file_size > maximum_bytes:
            print("Skipped oversized CSV:", item.filename)
            continue

        try:
            with archive.open(item, "r") as handle:
                frame = pd.read_csv(
                    handle,
                    dtype=str,
                    keep_default_na=False,
                )

            entry = {
                "file": item.filename,
                "rows": len(frame),
                "columns": frame.columns.tolist(),
            }

            ACI_SCHEMA_INSPECTION.append(entry)

            print("\nFILE:", entry["file"])
            print("Rows:", entry["rows"])
            print("Columns:", entry["columns"])

        except Exception as exc:
            print("\nCould not inspect:", item.filename)
            print(type(exc).__name__, str(exc))

(ACI_AUDIT_DIR / "csv_schema_inspection.json").write_text(
    json.dumps(
        ACI_SCHEMA_INSPECTION,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

print("\nAUDIT COMPLETE")
print("Dataset audit folder:", ACI_AUDIT_DIR)
print("CSV files inspected:", len(ACI_SCHEMA_INSPECTION))
print("No routing labels generated.")
print("No source artifacts changed.")
print("No model inference performed.")

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import re
import zipfile

import pandas as pd


# ------------------------------------------------------------
# 1. Locate the archive downloaded in the previous cell.
# ------------------------------------------------------------
if "ACI_AUDIT_DIR" not in globals():
    raise RuntimeError(
        "ACI_AUDIT_DIR is not defined. Set it to the audit "
        "folder printed by the previous download cell."
    )

audit_dir = Path(ACI_AUDIT_DIR)
archive_path = audit_dir / "aci_bench_original.zip"

if not archive_path.is_file():
    raise FileNotFoundError(archive_path)

challenge_files = {
    "train": "train.csv",
    "validation": "valid.csv",
    "test1": "clinicalnlp_taskB_test1.csv",
    "test2": "clinicalnlp_taskC_test2.csv",
    "test3": "clef_taskC_test3.csv",
}

required_columns = {
    "dataset",
    "encounter_id",
    "dialogue",
    "note",
}


# ------------------------------------------------------------
# 2. Read only the five challenge dialogue files.
#    Notes are deliberately excluded from the exported inventory.
# ------------------------------------------------------------
parts = []

with zipfile.ZipFile(archive_path, "r") as archive:
    for split_name, filename in challenge_files.items():
        member = (
            "aci-bench-corpus/challenge_data/"
            + filename
        )

        with archive.open(member, "r") as handle:
            frame = pd.read_csv(
                handle,
                dtype=str,
                keep_default_na=False,
            )

        missing = required_columns - set(frame.columns)

        if missing:
            raise RuntimeError(
                f"{member}: missing columns {sorted(missing)}"
            )

        part = frame[
            ["dataset", "encounter_id", "dialogue"]
        ].copy()

        part = part.rename(columns={
            "encounter_id": "original_encounter_id"
        })

        part.insert(0, "archive_member", member)
        part.insert(0, "aci_split", split_name)
        part.insert(0, "row_in_file", range(1, len(part) + 1))

        parts.append(part)

encounters = pd.concat(parts, ignore_index=True)


# ------------------------------------------------------------
# 3. Create inventory IDs and duplicate-review indicators.
#    These IDs identify rows, not proven independent encounters.
# ------------------------------------------------------------
encounters.insert(
    0,
    "inventory_record_id",
    [
        f"aci_inventory_{i:04d}"
        for i in range(1, len(encounters) + 1)
    ],
)

def normalize_dialogue(text):
    return re.sub(r"\s+", " ", text.lower().strip())

normalized = encounters["dialogue"].map(normalize_dialogue)

encounters["normalized_dialogue_sha256"] = normalized.map(
    lambda text: hashlib.sha256(
        text.encode("utf-8")
    ).hexdigest()
)

encounters["dialogue_word_count"] = (
    encounters["dialogue"].str.split().str.len()
)

empty_dialogue = normalized.eq("")

duplicate_source_key = encounters.duplicated(
    ["dataset", "original_encounter_id"],
    keep=False,
)

duplicate_dialogue = (
    normalized.duplicated(keep=False)
    & ~empty_dialogue
)

encounters["duplicate_source_key_review"] = (
    duplicate_source_key
)
encounters["duplicate_dialogue_review"] = (
    duplicate_dialogue
)

summary = {
    "inventory_rows": len(encounters),
    "rows_by_aci_split": (
        encounters.groupby("aci_split").size().to_dict()
    ),
    "blank_original_encounter_ids": int(
        encounters["original_encounter_id"]
        .str.strip()
        .eq("")
        .sum()
    ),
    "blank_dataset_values": int(
        encounters["dataset"].str.strip().eq("").sum()
    ),
    "blank_dialogues": int(empty_dialogue.sum()),
    "rows_with_repeated_dataset_encounter_key": int(
        duplicate_source_key.sum()
    ),
    "rows_with_repeated_normalized_dialogue": int(
        duplicate_dialogue.sum()
    ),
    "unique_dataset_encounter_keys": int(
        encounters[
            ["dataset", "original_encounter_id"]
        ].drop_duplicates().shape[0]
    ),
    "unique_nonblank_normalized_dialogues": int(
        normalized[~empty_dialogue].nunique()
    ),
    "external_source_overlap_checked": False,
    "segmentation_completed": False,
    "gold_labels_assigned": False,
    "ready_for_external_prediction": False,
}


# ------------------------------------------------------------
# 4. Save a NEW inventory folder without overwriting prior work.
# ------------------------------------------------------------
timestamp = datetime.now(timezone.utc).strftime(
    "%Y%m%dT%H%M%S%fZ"
)

inventory_dir = audit_dir / f"encounter_inventory_{timestamp}"
inventory_dir.mkdir(exist_ok=False)

encounters.to_csv(
    inventory_dir / "aci_encounter_inventory_NOT_FOR_INFERENCE.csv",
    index=False,
)

(inventory_dir / "inventory_summary.json").write_text(
    json.dumps(summary, indent=2),
    encoding="utf-8",
)

print("ENCOUNTER INVENTORY SUMMARY")
print("=" * 70)
print(json.dumps(summary, indent=2))

print("\nSaved inventory folder:", inventory_dir)
print("Notes excluded from inventory.")
print("No records automatically removed.")
print("No dialogue segments or gold labels generated.")
print("No model inference performed.")

ACI_ENCOUNTER_INVENTORY = encounters

In [ ]:
from pathlib import Path

PROJECT_DIR = Path(
    _crus_local_path('project', '')
)

if not PROJECT_DIR.is_dir():
    raise FileNotFoundError(PROJECT_DIR)

candidate_files = []

for path in PROJECT_DIR.rglob("*"):
    if not path.is_file():
        continue

    if path.suffix.lower() not in {
        ".json", ".txt", ".csv", ".yaml", ".yml"
    }:
        continue

    name = path.name.lower()

    if any(
        keyword in name
        for keyword in [
            "config",
            "label",
            "class",
            "manifest",
        ]
    ):
        candidate_files.append(path)

candidate_files = sorted(candidate_files)

print("POTENTIAL SOURCE CONFIGURATION / LABEL FILES")
print("=" * 80)

if not candidate_files:
    print("No matching files found in this project folder.")
else:
    for number, path in enumerate(candidate_files, start=1):
        relative_path = path.relative_to(PROJECT_DIR)

        print(
            f"{number:03d}. {relative_path} "
            f"({path.stat().st_size:,} bytes)"
        )

print("\nRead-only file discovery complete.")
print("No configuration selected or changed.")
print("No model files loaded.")
print("No inference performed.")

In [ ]:
from pathlib import Path
import json

import pandas as pd


PROJECT_DIR = Path(
    _crus_local_path('project', '')
)

# Selected from the file list you just provided.
# These are evidence to inspect, not automatically trusted
# as the final Cascade configuration.
JSON_FILES = [
    "source_data/source_manifest.json",
    "corrected_experiment_outputs/"
    "supervised_baseline_configuration.json",
    "corrected_experiment_outputs/domain_baselines/"
    "medcpt_query_v1/baseline_configuration.json",
]

CLASS_TABLES = [
    "corrected_experiment_outputs/comment5_audit/"
    "comment5_verified_class_distribution.csv",
    "data_audit/constrained_class_distribution_per_split.csv",
    "data_audit/corrected_class_distribution_per_split.csv",
    "data_audit/final_cleaned_class_counts.csv",
]

# Inspect headers only for larger files that may contain dialogue.
HEADER_ONLY_FILES = [
    "data_audit/constrained_split_manifest.csv",
    "data_audit/corrected_split_manifest.csv",
    "r1_10_validation_sensitivity_20261016T125818056792Z/"
    "config_00_per_query.csv",
]


print("JSON CONFIGURATION / MANIFEST CONTENTS")
print("=" * 80)

for relative_path in JSON_FILES:
    path = PROJECT_DIR / relative_path

    print("\nFILE:", relative_path)

    if not path.is_file():
        print("Missing; skipped.")
        continue

    try:
        with path.open("r", encoding="utf-8-sig") as handle:
            content = json.load(handle)

        print(
            json.dumps(
                content,
                indent=2,
                ensure_ascii=False,
            )
        )

    except Exception as exc:
        print(type(exc).__name__, str(exc))


print("\n\nCLASS-DISTRIBUTION TABLES")
print("=" * 80)

for relative_path in CLASS_TABLES:
    path = PROJECT_DIR / relative_path

    print("\nFILE:", relative_path)

    if not path.is_file():
        print("Missing; skipped.")
        continue

    try:
        frame = pd.read_csv(
            path,
            dtype=str,
            keep_default_na=False,
        )

        print("Rows:", len(frame))
        print("Columns:", frame.columns.tolist())
        print(frame.to_string(index=False))

    except Exception as exc:
        print(type(exc).__name__, str(exc))


print("\n\nLARGER FILES — COLUMN NAMES ONLY")
print("=" * 80)

for relative_path in HEADER_ONLY_FILES:
    path = PROJECT_DIR / relative_path

    print("\nFILE:", relative_path)

    if not path.is_file():
        print("Missing; skipped.")
        continue

    try:
        columns = pd.read_csv(
            path,
            nrows=0,
        ).columns.tolist()

        print("Columns:", columns)

    except Exception as exc:
        print(type(exc).__name__, str(exc))


print("\nREAD-ONLY INSPECTION COMPLETE")
print("No label order inferred.")
print("No configuration changed.")
print("No model files loaded.")
print("No inference performed.")

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json

import pandas as pd


# ------------------------------------------------------------
# 1. Verify the source-label vocabulary against both audit tables.
# ------------------------------------------------------------
project_dir = Path(
    _crus_local_path('project', '')
)

verified_path = (
    project_dir
    / "corrected_experiment_outputs"
    / "comment5_audit"
    / "comment5_verified_class_distribution.csv"
)

constrained_path = (
    project_dir
    / "data_audit"
    / "constrained_class_distribution_per_split.csv"
)

verified = pd.read_csv(
    verified_path,
    dtype={"label": str},
)

constrained = pd.read_csv(
    constrained_path,
    dtype={"label": str},
)

columns = ["label", "train", "validation", "test", "total"]

for name, frame in [
    ("verified", verified),
    ("constrained", constrained),
]:
    missing = set(columns) - set(frame.columns)

    if missing:
        raise RuntimeError(
            f"{name} table is missing columns: {sorted(missing)}"
        )

    if frame["label"].duplicated().any():
        raise RuntimeError(f"{name} table has duplicate labels.")

    for column in columns[1:]:
        frame[column] = pd.to_numeric(
            frame[column],
            errors="raise",
        )

pd.testing.assert_frame_equal(
    verified[columns].sort_values("label").reset_index(drop=True),
    constrained[columns].sort_values("label").reset_index(drop=True),
    check_dtype=False,
)

labels = verified["label"].tolist()

if len(labels) != 17:
    raise RuntimeError(f"Expected 17 labels; found {len(labels)}.")

split_counts = {
    name: int(verified[name].sum())
    for name in ["train", "validation", "test"]
}

if split_counts != {
    "train": 773,
    "validation": 258,
    "test": 257,
}:
    raise RuntimeError(
        f"Unexpected constrained split counts: {split_counts}"
    )


# ------------------------------------------------------------
# 2. Use the previously created encounter inventory.
# ------------------------------------------------------------
if "ACI_ENCOUNTER_INVENTORY" not in globals():
    raise RuntimeError(
        "ACI_ENCOUNTER_INVENTORY is not in memory. "
        "Rerun the encounter-inventory cell first."
    )

inventory = ACI_ENCOUNTER_INVENTORY.copy()

required = [
    "inventory_record_id",
    "dataset",
    "original_encounter_id",
    "aci_split",
    "dialogue",
]

missing = set(required) - set(inventory.columns)

if missing:
    raise RuntimeError(
        f"Inventory is missing columns: {sorted(missing)}"
    )

if inventory["inventory_record_id"].duplicated().any():
    raise RuntimeError("Inventory row IDs are not unique.")


# ------------------------------------------------------------
# 3. Create a separate, new annotation workspace.
# ------------------------------------------------------------
timestamp = datetime.now(timezone.utc).strftime(
    "%Y%m%dT%H%M%S%fZ"
)

workspace = (
    project_dir
    / f"aci_external_annotation_workspace_{timestamp}"
)

workspace.mkdir(parents=True, exist_ok=False)

queue = inventory[required].copy()

# A deterministic grouping key retaining both source identifiers.
queue["encounter_id"] = [
    json.dumps(
        [dataset, encounter],
        ensure_ascii=False,
        separators=(",", ":"),
    )
    for dataset, encounter in zip(
        queue["dataset"],
        queue["original_encounter_id"],
    )
]

if queue["encounter_id"].duplicated().any():
    raise RuntimeError("Encounter grouping keys are not unique.")

queue["review_status"] = "not_started"
queue["eligibility_decision"] = ""
queue["exclusion_reason"] = ""
queue["review_notes"] = ""

queue.to_csv(
    workspace / "encounter_review_queue.csv",
    index=False,
)


# ------------------------------------------------------------
# 4. Create a genuinely EMPTY segment-annotation table.
#    Full encounters are NOT assigned one routing label.
# ------------------------------------------------------------
annotation_columns = [
    "record_id",
    "encounter_id",
    "inventory_record_id",
    "start_char",
    "end_char",
    "dialogue",
    "annotator_1_label",
    "annotator_2_label",
    "gold",
    "annotation_status",
    "adjudication_notes",
]

pd.DataFrame(
    columns=annotation_columns
).to_csv(
    workspace / "segment_annotations_BLANK.csv",
    index=False,
)

pd.DataFrame({
    "allowed_label": sorted(labels),
}).to_csv(
    workspace / "allowed_label_vocabulary_NOT_probability_order.csv",
    index=False,
)

status = {
    "source_split_counts": split_counts,
    "source_label_count": len(labels),
    "label_vocabulary_verified": True,
    "cascade_probability_order_verified": False,
    "encounters_in_review_queue": len(queue),
    "segmentation_protocol_finalized": False,
    "segments_annotated": 0,
    "external_source_overlap_checked": False,
    "ready_for_external_prediction": False,
}

(workspace / "workspace_status.json").write_text(
    json.dumps(status, indent=2),
    encoding="utf-8",
)

instructions = """
STATUS: Annotation workspace only; not an evaluation-ready dataset.

1. Finalize and document segmentation, inclusion, exclusion, and
   label-mapping rules BEFORE external model scoring.

2. Review the encounter queue. Retain original encounter identifiers.
   Record exclusions and their reasons.

3. Add one row per eligible dialogue segment to the annotation table.
   Do not assign one arbitrary label to an entire consultation.

4. If character offsets are used, use Python string indices:
   start_char inclusive, end_char exclusive, relative to the
   unmodified dialogue in encounter_review_queue.csv.
   The segment dialogue must equal full_dialogue[start_char:end_char].

5. Assign labels independently of model predictions. Document who
   annotated and reviewed them. Leave unresolved gold labels blank.
   Annotator columns are workflow fields, not evidence that review
   has occurred.

6. Use only the exact allowed source-label strings for resolved gold.
   Do not force unsupported or ambiguous material into a class.

7. Reference notes are not included in this workspace and must not
   be appended to inference dialogue.

8. The vocabulary CSV is alphabetical for readability.
   It does NOT declare Cascade's probability-column order.

9. Source-overlap/provenance review and frozen-system regression
   remain mandatory before external inference.

10. Do not pass the encounter queue or blank annotation CSV directly
    to the evaluation runner. A validated external_segments.csv
    must be exported after annotation and review.
""".strip()

(workspace / "ANNOTATION_INSTRUCTIONS.txt").write_text(
    instructions,
    encoding="utf-8",
)

print("ANNOTATION WORKSPACE CREATED")
print("=" * 70)
print("Folder:", workspace)
print("Encounters queued:", len(queue))
print("Allowed source labels:", len(labels))
print("Source split counts:", split_counts)
print("Annotated segments: 0")
print("Cascade probability order: not yet verified")
print("External inference readiness: FALSE")
print("No models loaded and no predictions generated.")

ACI_ANNOTATION_WORKSPACE = workspace

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from urllib.parse import quote
import hashlib
import io
import json
import re

import pandas as pd
import requests


# ------------------------------------------------------------
# 1. Settings and helpers.
# ------------------------------------------------------------
PROJECT_DIR = Path(
    _crus_local_path('project', '')
)

SOURCE_DIR = PROJECT_DIR / "source_data"
REPOSITORY = "abachaa/MTS-Dialog"

TEST_FILENAMES = [
    "MTS-Dialog-TestSet-1-MEDIQA-Chat-2023.csv",
    "MTS-Dialog-TestSet-2-MEDIQA-Sum-2023.csv",
]

SOURCE_FILENAMES = [
    "MTS-Dialog-TrainingSet.csv",
    "MTS-Dialog-ValidationSet.csv",
]

session = requests.Session()


def get_response(url):
    response = session.get(url, timeout=90)
    response.raise_for_status()
    return response


def normalize_dialogue(text):
    return re.sub(r"\s+", " ", str(text).lower().strip())


def find_column(frame, aliases):
    def key(value):
        return re.sub(r"[\s_-]+", "", str(value).lower())

    accepted = {key(alias) for alias in aliases}

    matches = [
        column
        for column in frame.columns
        if key(column) in accepted
    ]

    if len(matches) != 1:
        raise RuntimeError(
            f"Expected exactly one column matching {aliases}; "
            f"found {matches}. Available: {frame.columns.tolist()}"
        )

    return matches[0]


# ------------------------------------------------------------
# 2. Verify local source files against their saved manifest.
#    Use the full original source pool for exact-overlap screening.
# ------------------------------------------------------------
manifest_path = SOURCE_DIR / "source_manifest.json"

with manifest_path.open("r", encoding="utf-8-sig") as handle:
    source_manifest = json.load(handle)

source_dialogues = []

for filename in SOURCE_FILENAMES:
    matches = [
        path
        for path in SOURCE_DIR.rglob(filename)
        if path.is_file()
    ]

    if len(matches) != 1:
        raise RuntimeError(
            f"Expected one local source file named {filename}; "
            f"found {len(matches)}. Stop and share this message."
        )

    path = matches[0]
    file_bytes = path.read_bytes()
    actual_sha256 = hashlib.sha256(file_bytes).hexdigest()

    expected = source_manifest["files"][filename]

    if actual_sha256 != expected["sha256"]:
        raise RuntimeError(
            f"Source checksum mismatch: {filename}"
        )

    frame = pd.read_csv(
        io.BytesIO(file_bytes),
        dtype=str,
        keep_default_na=False,
    )

    if len(frame) != expected["rows"]:
        raise RuntimeError(
            f"Source row-count mismatch: {filename}"
        )

    dialogue_column = find_column(frame, ["dialogue"])

    source_dialogues.extend(
        frame[dialogue_column].map(normalize_dialogue).tolist()
    )

source_dialogue_set = set(source_dialogues)
source_dialogue_set.discard("")


# ------------------------------------------------------------
# 3. Load verified source label membership.
#    This does NOT establish Cascade probability-column order.
# ------------------------------------------------------------
label_path = (
    PROJECT_DIR
    / "corrected_experiment_outputs"
    / "comment5_audit"
    / "comment5_verified_class_distribution.csv"
)

label_table = pd.read_csv(
    label_path,
    dtype=str,
    keep_default_na=False,
)

allowed_labels = set(label_table["label"])

if len(allowed_labels) != 17:
    raise RuntimeError("Expected exactly 17 verified source labels.")


# ------------------------------------------------------------
# 4. Pin the current official repository commit.
# ------------------------------------------------------------
commit_metadata = get_response(
    f"https://api.github.com/repos/{REPOSITORY}/commits/main"
).json()

download_commit = commit_metadata["sha"]

tree = get_response(
    f"https://api.github.com/repos/{REPOSITORY}/git/trees/"
    f"{download_commit}?recursive=1"
).json()

if tree.get("truncated"):
    raise RuntimeError(
        "Repository tree was truncated; stop rather than guess paths."
    )

timestamp = datetime.now(timezone.utc).strftime(
    "%Y%m%dT%H%M%S%fZ"
)

HELDOUT_AUDIT_DIR = (
    PROJECT_DIR
    / f"additional_mts_heldout_audit_{timestamp}"
)

HELDOUT_AUDIT_DIR.mkdir(
    parents=True,
    exist_ok=False,
)

download_records = []
parts = []


# ------------------------------------------------------------
# 5. Download the two CSVs from that exact commit.
# ------------------------------------------------------------
for test_number, filename in enumerate(TEST_FILENAMES, start=1):
    matches = [
        item["path"]
        for item in tree.get("tree", [])
        if item.get("type") == "blob"
        and Path(item["path"]).name == filename
    ]

    if len(matches) != 1:
        raise RuntimeError(
            f"Expected exactly one repository file named {filename}; "
            f"found {matches}."
        )

    repository_path = matches[0]

    download_url = (
        f"https://raw.githubusercontent.com/{REPOSITORY}/"
        f"{download_commit}/{quote(repository_path, safe='/')}"
    )

    payload = get_response(download_url).content
    (HELDOUT_AUDIT_DIR / filename).write_bytes(payload)

    frame = pd.read_csv(
        io.BytesIO(payload),
        dtype=str,
        keep_default_na=False,
    )

    print("\nFILE:", filename)
    print("Rows:", len(frame))
    print("Columns:", frame.columns.tolist())

    if len(frame) != 200:
        raise RuntimeError(
            "Unexpected test-set size. Download preserved; "
            "stop and inspect before proceeding."
        )

    dialogue_column = find_column(frame, ["dialogue"])
    gold_column = find_column(frame, ["section_header"])

    # Preserve original fields in the downloaded raw CSV.
    # Export dialogue and provided labels only, never summary text.
    part = pd.DataFrame({
        "record_id": [
            f"mts_official_test{test_number}_row_{i:04d}"
            for i in range(1, len(frame) + 1)
        ],
        "official_test_set": f"test{test_number}",
        "source_filename": filename,
        "source_csv_row": range(1, len(frame) + 1),
        "dialogue": frame[dialogue_column],
        "gold": frame[gold_column],
    })

    parts.append(part)

    download_records.append({
        "filename": filename,
        "repository_path": repository_path,
        "download_url": download_url,
        "sha256": hashlib.sha256(payload).hexdigest(),
        "rows": len(frame),
    })


# ------------------------------------------------------------
# 6. Flag compatibility and exact overlap.
#    Do not silently relabel or deduplicate.
# ------------------------------------------------------------
candidates = pd.concat(parts, ignore_index=True)

normalized = candidates["dialogue"].map(normalize_dialogue)

candidates["blank_dialogue"] = normalized.eq("")
candidates["supported_source_label"] = (
    candidates["gold"].isin(allowed_labels)
)
candidates["exact_normalized_source_overlap"] = (
    normalized.isin(source_dialogue_set)
)
candidates["repeated_dialogue_in_new_test_pool"] = (
    normalized.duplicated(keep=False)
    & ~normalized.eq("")
)

candidates["normalized_dialogue_sha256"] = normalized.map(
    lambda text: hashlib.sha256(
        text.encode("utf-8")
    ).hexdigest()
)

summary = {
    "evaluation_description": (
        "additional held-out MTS-Dialog evaluation; "
        "not independent cross-dataset validation"
    ),
    "download_repository_commit": download_commit,
    "source_records_checked": len(source_dialogues),
    "downloaded_test_records": len(candidates),
    "supported_label_records": int(
        candidates["supported_source_label"].sum()
    ),
    "unsupported_label_records": int(
        (~candidates["supported_source_label"]).sum()
    ),
    "unsupported_label_counts": (
        candidates.loc[
            ~candidates["supported_source_label"], "gold"
        ].value_counts().to_dict()
    ),
    "blank_dialogues": int(
        candidates["blank_dialogue"].sum()
    ),
    "exact_source_overlap_records": int(
        candidates["exact_normalized_source_overlap"].sum()
    ),
    "repeated_dialogue_records_in_new_pool": int(
        candidates["repeated_dialogue_in_new_test_pool"].sum()
    ),
    "previous_development_use_verified_absent": False,
    "near_duplicate_review_completed": False,
    "cascade_probability_order_verified": False,
    "ready_for_inference": False,
}

candidates.to_csv(
    HELDOUT_AUDIT_DIR / "heldout_candidate_audit.csv",
    index=False,
)

support = pd.crosstab(
    candidates["gold"],
    candidates["official_test_set"],
)

support.to_csv(
    HELDOUT_AUDIT_DIR / "provided_label_support.csv"
)

(HELDOUT_AUDIT_DIR / "download_provenance.json").write_text(
    json.dumps(
        {
            "repository": REPOSITORY,
            "commit": download_commit,
            "files": download_records,
        },
        indent=2,
    ),
    encoding="utf-8",
)

(HELDOUT_AUDIT_DIR / "audit_summary.json").write_text(
    json.dumps(summary, indent=2),
    encoding="utf-8",
)

print("\nHELD-OUT TEST AUDIT")
print("=" * 80)
print(json.dumps(summary, indent=2))

print("\nPROVIDED LABEL SUPPORT")
print("=" * 80)
print(support.to_string())

print("\nAudit folder:", HELDOUT_AUDIT_DIR)
print("No model inference performed.")
print("No source artifacts changed.")
print("No gold labels invented or remapped.")
print("No candidate records automatically removed.")

MTS_HELDOUT_CANDIDATES = candidates

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from difflib import SequenceMatcher
import json
import re

import pandas as pd


# ------------------------------------------------------------
# 1. Require the verified data from the preceding audit cell.
# ------------------------------------------------------------
required_variables = [
    "MTS_HELDOUT_CANDIDATES",
    "HELDOUT_AUDIT_DIR",
    "source_dialogues",
]

missing_variables = [
    name
    for name in required_variables
    if name not in globals()
]

if missing_variables:
    raise RuntimeError(
        "Rerun the previous download/audit cell first. "
        f"Missing variables: {missing_variables}"
    )

audit = MTS_HELDOUT_CANDIDATES.copy()

required_columns = [
    "record_id",
    "official_test_set",
    "dialogue",
    "gold",
    "blank_dialogue",
    "supported_source_label",
    "exact_normalized_source_overlap",
    "repeated_dialogue_in_new_test_pool",
]

missing_columns = set(required_columns) - set(audit.columns)

if missing_columns:
    raise RuntimeError(
        f"Missing audit columns: {sorted(missing_columns)}"
    )

if audit["record_id"].duplicated().any():
    raise RuntimeError("Candidate record IDs are not unique.")


def normalize_text(text):
    return re.sub(r"\s+", " ", str(text).lower().strip())


# ------------------------------------------------------------
# 2. Record exclusion reasons without dropping audit rows.
# ------------------------------------------------------------
def exclusion_reasons(row):
    reasons = []

    if row["blank_dialogue"]:
        reasons.append("blank_dialogue")

    if not row["supported_source_label"]:
        reasons.append("unsupported_source_label")

    if row["exact_normalized_source_overlap"]:
        reasons.append("exact_source_overlap")

    if row["repeated_dialogue_in_new_test_pool"]:
        reasons.append("repeated_dialogue_in_new_test_pool")

    return ";".join(reasons)


audit["automatic_exclusion_reasons"] = audit.apply(
    exclusion_reasons,
    axis=1,
)

audit["passes_automatic_checks"] = (
    audit["automatic_exclusion_reasons"].eq("")
)


# ------------------------------------------------------------
# 3. Build source-text groups.
#    Positions refer to the source_dialogues list created earlier:
#    original training rows followed by original validation rows.
#    These are NOT the runner's canonical source record IDs.
# ------------------------------------------------------------
source_positions_by_text = {}

for position, dialogue in enumerate(source_dialogues, start=1):
    normalized = normalize_text(dialogue)

    if normalized:
        source_positions_by_text.setdefault(
            normalized, []
        ).append(position)

source_items = [
    (text, len(text), positions)
    for text, positions in source_positions_by_text.items()
]

screening_candidates = audit.loc[
    audit["passes_automatic_checks"]
].copy()

near_pairs = []

print(
    "Screening",
    len(screening_candidates),
    "candidate records against",
    len(source_items),
    "unique nonblank source dialogues.",
)

print("This may take several minutes.")


# ------------------------------------------------------------
# 4. Apply the starter kit's near-duplicate screen.
#    quick_ratio is used only as an upper-bound rejection step.
# ------------------------------------------------------------
for number, row in enumerate(
    screening_candidates.itertuples(index=False),
    start=1,
):
    query = normalize_text(row.dialogue)
    query_length = len(query)

    matcher = SequenceMatcher(
        None,
        "",
        query,
        autojunk=False,
    )

    for source_text, source_length, positions in source_items:
        length_ratio = (
            min(query_length, source_length)
            / max(query_length, source_length)
        )

        if length_ratio < 0.90:
            continue

        matcher.set_seq1(source_text)

        if matcher.real_quick_ratio() < 0.95:
            continue

        if matcher.quick_ratio() < 0.95:
            continue

        similarity = matcher.ratio()

        if similarity >= 0.95:
            near_pairs.append({
                "heldout_record_id": row.record_id,
                "official_test_set": row.official_test_set,
                "source_pool_positions": json.dumps(positions),
                "length_ratio": length_ratio,
                "sequence_similarity": similarity,
                "heldout_dialogue": row.dialogue,
                "source_dialogue": source_text,
                "review_decision": "",
                "review_reason": "",
            })

    if number % 25 == 0:
        print(
            f"Screened {number}/{len(screening_candidates)}"
        )

pair_columns = [
    "heldout_record_id",
    "official_test_set",
    "source_pool_positions",
    "length_ratio",
    "sequence_similarity",
    "heldout_dialogue",
    "source_dialogue",
    "review_decision",
    "review_reason",
]

near_review = pd.DataFrame(
    near_pairs,
    columns=pair_columns,
)

flagged_ids = set(near_review["heldout_record_id"])

audit["near_source_overlap_review_required"] = (
    audit["record_id"].isin(flagged_ids)
)

audit["passes_automated_screen_only"] = (
    audit["passes_automatic_checks"]
    & ~audit["near_source_overlap_review_required"]
)


# ------------------------------------------------------------
# 5. Save a new audit folder.
#    No evaluation-ready file or attestation is generated.
# ------------------------------------------------------------
timestamp = datetime.now(timezone.utc).strftime(
    "%Y%m%dT%H%M%S%fZ"
)

output_dir = (
    Path(HELDOUT_AUDIT_DIR)
    / f"eligibility_screen_{timestamp}"
)

output_dir.mkdir(exist_ok=False)

audit.to_csv(
    output_dir / "all_records_with_screening_flags.csv",
    index=False,
)

near_review.to_csv(
    output_dir / "near_source_overlap_REVIEW_REQUIRED.csv",
    index=False,
)

audit.loc[
    audit["repeated_dialogue_in_new_test_pool"],
    [
        "record_id",
        "official_test_set",
        "gold",
        "dialogue",
        "exact_normalized_source_overlap",
    ],
].to_csv(
    output_dir / "repeated_test_dialogue_records.csv",
    index=False,
)

summary = {
    "total_downloaded_records": len(audit),
    "fails_one_or_more_automatic_checks": int(
        (~audit["passes_automatic_checks"]).sum()
    ),
    "passes_automatic_checks": int(
        audit["passes_automatic_checks"].sum()
    ),
    "near_source_overlap_candidate_pairs": len(near_review),
    "records_requiring_near_overlap_review": len(flagged_ids),
    "passes_automated_screen_only": int(
        audit["passes_automated_screen_only"].sum()
    ),
    "prior_development_use_verified_absent": False,
    "provenance_and_semantic_overlap_review_completed": False,
    "cascade_probability_order_verified": False,
    "ready_for_inference": False,
}

(output_dir / "screening_summary.json").write_text(
    json.dumps(summary, indent=2),
    encoding="utf-8",
)

print("\nSCREENING SUMMARY")
print("=" * 80)
print(json.dumps(summary, indent=2))

print("\nSaved audit folder:", output_dir)
print("No model inference performed.")
print("No records removed from the master audit.")
print("No review decisions automatically assigned.")
print("Passing this screen does not establish independence.")

MTS_HELDOUT_SCREENED_AUDIT = audit
MTS_HELDOUT_NEAR_REVIEW = near_review

In [ ]:
from pathlib import Path
import json
import re

import pandas as pd


PROJECT_DIR = Path(
    _crus_local_path('project', '')
)

# Add your ORIGINAL experiment notebook or scripts here
# if they are outside PROJECT_DIR.
EXTRA_FILES = [
    # "/content/drive/MyDrive/your_original_notebook.ipynb",
]

if not PROJECT_DIR.is_dir():
    raise FileNotFoundError(PROJECT_DIR)


# ------------------------------------------------------------
# 1. Exclude the newly created dataset-preparation/audit folders.
# ------------------------------------------------------------
EXCLUDED_FOLDER_PREFIXES = (
    "additional_mts_heldout_audit_",
    "aci_bench_external_data_audit_",
    "aci_external_annotation_workspace_",
    "r1_5_aci_bench_dataset_audit_",
)

EXCLUDED_EXACT_FOLDERS = {
    ".git",
    ".ipynb_checkpoints",
    "__pycache__",
    "venv",
    ".venv",
    "node_modules",
}

TEXT_SUFFIXES = {
    ".py", ".ipynb", ".json", ".jsonl",
    ".yaml", ".yml", ".toml", ".ini", ".cfg",
    ".sh", ".txt", ".log", ".out", ".md",
}

MAX_FILE_BYTES = 20_000_000

PATTERNS = {
    "official_test_filename": re.compile(
        r"MTS[-_]Dialog[-_]TestSet[-_][12]",
        re.IGNORECASE,
    ),
    "challenge_test_filename": re.compile(
        r"MEDIQA[-_]Chat[-_]2023\.csv|"
        r"MEDIQA[-_]Sum[-_]2023\.csv",
        re.IGNORECASE,
    ),
    "testset_reference": re.compile(
        r"\btest[\s_-]*set[\s_-]*[12]\b",
        re.IGNORECASE,
    ),
    "challenge_reference": re.compile(
        r"MEDIQA[\s_-]*(?:Chat|Sum)",
        re.IGNORECASE,
    ),
}

def excluded(path):
    relative_parts = path.relative_to(PROJECT_DIR).parts

    return any(
        part in EXCLUDED_EXACT_FOLDERS
        or part.startswith(EXCLUDED_FOLDER_PREFIXES)
        for part in relative_parts[:-1]
    )


# ------------------------------------------------------------
# 2. Collect project files and explicitly supplied extra files.
# ------------------------------------------------------------
all_project_files = [
    path
    for path in PROJECT_DIR.rglob("*")
    if path.is_file()
    and not excluded(path)
]

files_to_scan = [
    path
    for path in all_project_files
    if path.suffix.lower() in TEXT_SUFFIXES
]

missing_extra_files = []

for filename in EXTRA_FILES:
    path = Path(filename)

    if path.is_file():
        files_to_scan.append(path)
    else:
        missing_extra_files.append(str(path))

files_to_scan = sorted(set(files_to_scan))

hits = []
scan_errors = []
oversized_files = []
scanned_files = []
notebooks_scanned = []


# ------------------------------------------------------------
# 3. Search text, using CODE CELLS ONLY for notebooks.
#    No notebook cells or model artifacts are executed.
# ------------------------------------------------------------
def search_text(text, path, location):
    for line_number, line in enumerate(text.splitlines(), start=1):
        matched_rules = [
            name
            for name, pattern in PATTERNS.items()
            if pattern.search(line)
        ]

        if matched_rules:
            hits.append({
                "file": str(path),
                "location": location,
                "line": line_number,
                "matched_rules": ";".join(matched_rules),
                "matching_line": line.strip()[:500],
            })


for path in files_to_scan:
    if path.stat().st_size > MAX_FILE_BYTES:
        oversized_files.append(str(path))
        continue

    try:
        text = path.read_text(encoding="utf-8-sig")

        if path.suffix.lower() == ".ipynb":
            notebook = json.loads(text)
            notebooks_scanned.append(str(path))

            for cell_number, cell in enumerate(
                notebook.get("cells", []),
                start=1,
            ):
                if cell.get("cell_type") != "code":
                    continue

                source = cell.get("source", [])

                if isinstance(source, list):
                    source = "".join(source)

                search_text(
                    source,
                    path,
                    f"code_cell_{cell_number}",
                )

        else:
            search_text(text, path, "text_file")

        scanned_files.append(str(path))

    except Exception as exc:
        scan_errors.append({
            "file": str(path),
            "error": f"{type(exc).__name__}: {exc}",
        })


# ------------------------------------------------------------
# 4. Inventory relevant filenames, including binary artifacts.
#    Do NOT deserialize pickle/joblib/model files.
# ------------------------------------------------------------
filename_hits = []

for path in all_project_files:
    if any(
        pattern.search(path.name)
        for pattern in PATTERNS.values()
    ):
        filename_hits.append({
            "file": str(path),
            "bytes": path.stat().st_size,
        })


# ------------------------------------------------------------
# 5. Present findings. Nothing is written or modified.
# ------------------------------------------------------------
HISTORY_REFERENCE_HITS = pd.DataFrame(
    hits,
    columns=[
        "file",
        "location",
        "line",
        "matched_rules",
        "matching_line",
    ],
)

HISTORY_FILENAME_HITS = pd.DataFrame(
    filename_hits,
    columns=["file", "bytes"],
)

summary = {
    "project_folder": str(PROJECT_DIR),
    "successfully_scanned_files": len(scanned_files),
    "notebooks_scanned": len(notebooks_scanned),
    "reference_hit_count": len(hits),
    "matching_filename_count": len(filename_hits),
    "oversized_files_skipped": len(oversized_files),
    "read_or_parse_errors": len(scan_errors),
    "missing_extra_files": missing_extra_files,
    "prior_non_use_established": False,
    "models_executed": False,
    "files_modified": False,
}

print("DEVELOPMENT-HISTORY AUDIT")
print("=" * 80)
print(json.dumps(summary, indent=2))

print("\nNOTEBOOKS SCANNED")
print("=" * 80)
for filename in notebooks_scanned:
    print(filename)

if not notebooks_scanned:
    print(
        "WARNING: No saved notebooks were scanned. "
        "If development occurred in a notebook, add its path "
        "to EXTRA_FILES and rerun."
    )

print("\nCODE / CONFIGURATION / LOG REFERENCES — FIRST 100")
print("=" * 80)

if HISTORY_REFERENCE_HITS.empty:
    print("No matching references found in the scanned files.")
else:
    with pd.option_context(
        "display.max_colwidth", 500,
        "display.width", 240,
    ):
        print(
            HISTORY_REFERENCE_HITS.head(100).to_string(index=False)
        )

print("\nMATCHING FILENAMES — FIRST 100")
print("=" * 80)

if HISTORY_FILENAME_HITS.empty:
    print("No matching filenames found.")
else:
    print(
        HISTORY_FILENAME_HITS.head(100).to_string(index=False)
    )

print("\nSKIPPED LARGE FILES")
for filename in oversized_files:
    print(filename)

print("\nREAD / PARSE ERRORS")
for error in scan_errors:
    print(error)

print("\nRead-only audit complete.")
print("Matches require interpretation; they are not proof of use.")
print("Absence of matches is not proof of non-use.")

In [ ]:
from pathlib import Path
import os

DRIVE_ROOT = Path(_crus_local_path('drive', ''))

if not DRIVE_ROOT.is_dir():
    raise RuntimeError(
        "Google Drive is not mounted. Mount it before running this cell."
    )

SKIP_FOLDERS = {
    ".git",
    ".ipynb_checkpoints",
    "__pycache__",
    ".venv",
    "venv",
    "node_modules",
}

notebook_paths = []
walk_errors = []

def record_walk_error(error):
    walk_errors.append(str(error))

for folder, subfolders, filenames in os.walk(
    DRIVE_ROOT,
    onerror=record_walk_error,
):
    subfolders[:] = [
        name
        for name in subfolders
        if name not in SKIP_FOLDERS
    ]

    for filename in filenames:
        if filename.lower().endswith(".ipynb"):
            notebook_paths.append(
                Path(folder) / filename
            )

NOTEBOOK_CANDIDATES = sorted(notebook_paths)

print("SAVED NOTEBOOKS IN MY DRIVE")
print("=" * 80)

for number, path in enumerate(NOTEBOOK_CANDIDATES, start=1):
    print(f"{number:03d}. {path}")

print("\nNotebook count:", len(NOTEBOOK_CANDIDATES))

if walk_errors:
    print("\nFolders that could not be inspected:")
    for error in walk_errors:
        print(error)

print("\nRead-only discovery complete.")
print("No notebooks opened or executed.")
print("No files changed.")

In [ ]:
from pathlib import Path
import json
import re

import pandas as pd


if "NOTEBOOK_CANDIDATES" not in globals():
    raise RuntimeError(
        "Run the notebook-discovery cell first."
    )

PATTERNS = {
    "official_test_reference": re.compile(
        r"MTS[-_]Dialog[-_]TestSet|"
        r"MEDIQA[-_ ](?:Chat|Sum)[-_ ]2023\.csv|"
        r"\btest[\s_-]*set[\s_-]*[12]\b",
        re.IGNORECASE,
    ),

    "project_or_split_reference": re.compile(
        r"RiskAware_Revision2|"
        r"constrained_split_manifest|"
        r"primary_constrained|"
        r"corrected_experiment_outputs",
        re.IGNORECASE,
    ),

    "source_data_loading": re.compile(
        r"MTS[-_]Dialog[-_](?:TrainingSet|ValidationSet)|"
        r"read_csv\s*\(|"
        r"load_dataset\s*\(|"
        r"\bglob\s*\(|"
        r"\brglob\s*\(|"
        r"pd\.concat\s*\(",
        re.IGNORECASE,
    ),

    "frozen_threshold": re.compile(
        r"0\.128165036474669"
    ),

    "corrected_cohort_count": re.compile(
        r"\b(?:773|1288|1,288)\b"
    ),
}


def as_text(value):
    if isinstance(value, list):
        return "".join(str(item) for item in value)
    return str(value or "")


def saved_output_text(output):
    pieces = []

    if "text" in output:
        pieces.append(as_text(output["text"]))

    plain_text = output.get("data", {}).get("text/plain")

    if plain_text is not None:
        pieces.append(as_text(plain_text))

    if output.get("output_type") == "error":
        pieces.append(
            f"{output.get('ename', '')}: "
            f"{output.get('evalue', '')}"
        )

    return "\n".join(pieces)


hits = []
summaries = []
errors = []


def inspect_text(text, notebook_path, cell_number,
                 section, execution_count):
    for line_number, line in enumerate(text.splitlines(), start=1):
        matched = [
            name
            for name, pattern in PATTERNS.items()
            if pattern.search(line)
        ]

        if matched:
            hits.append({
                "notebook": str(notebook_path),
                "cell": cell_number,
                "section": section,
                "execution_count": execution_count,
                "line": line_number,
                "rules": ";".join(matched),
                "text": line.strip()[:600],
            })


for notebook_path in NOTEBOOK_CANDIDATES:
    path = Path(notebook_path)
    start_index = len(hits)

    try:
        notebook = json.loads(
            path.read_text(encoding="utf-8-sig")
        )

        cells = notebook.get("cells", [])

        for cell_number, cell in enumerate(cells, start=1):
            cell_type = cell.get("cell_type", "unknown")
            execution_count = cell.get("execution_count")

            if cell_type in {"code", "markdown"}:
                inspect_text(
                    as_text(cell.get("source", [])),
                    path,
                    cell_number,
                    cell_type,
                    execution_count,
                )

            for output in cell.get("outputs", []):
                inspect_text(
                    saved_output_text(output),
                    path,
                    cell_number,
                    "saved_output",
                    execution_count,
                )

        notebook_hits = hits[start_index:]

        summary = {
            "notebook": str(path),
            "code_cells": sum(
                cell.get("cell_type") == "code"
                for cell in cells
            ),
            "total_matching_lines": len(notebook_hits),
        }

        for rule in PATTERNS:
            summary[rule] = sum(
                rule in hit["rules"].split(";")
                for hit in notebook_hits
            )

        summaries.append(summary)

    except Exception as exc:
        errors.append({
            "notebook": str(path),
            "error": f"{type(exc).__name__}: {exc}",
        })


NOTEBOOK_HISTORY_SUMMARY = pd.DataFrame(summaries)

NOTEBOOK_HISTORY_HITS = pd.DataFrame(
    hits,
    columns=[
        "notebook",
        "cell",
        "section",
        "execution_count",
        "line",
        "rules",
        "text",
    ],
)


print("NOTEBOOK HISTORY SUMMARY")
print("=" * 100)

with pd.option_context(
    "display.max_colwidth", 140,
    "display.width", 260,
):
    print(
        NOTEBOOK_HISTORY_SUMMARY.to_string(index=False)
    )


print("\nOFFICIAL TEST-SET REFERENCES — FIRST 60")
print("=" * 100)

test_hits = NOTEBOOK_HISTORY_HITS.loc[
    NOTEBOOK_HISTORY_HITS["rules"].str.contains(
        "official_test_reference",
        regex=False,
    )
]

if test_hits.empty:
    print("No direct official-test references found.")
else:
    print(test_hits.head(60).to_string(index=False))


print("\nFROZEN-THRESHOLD REFERENCES — FIRST 30")
print("=" * 100)

threshold_hits = NOTEBOOK_HISTORY_HITS.loc[
    NOTEBOOK_HISTORY_HITS["rules"].str.contains(
        "frozen_threshold",
        regex=False,
    )
]

if threshold_hits.empty:
    print("No exact frozen-threshold references found.")
else:
    print(threshold_hits.head(30).to_string(index=False))


print("\nERRORS")
print("=" * 100)

if errors:
    for error in errors:
        print(error)
else:
    print("None.")

print("\nRead-only notebook audit complete.")
print("No notebook code executed.")
print("No model artifacts loaded or changed.")
print("No historical non-use conclusion assigned.")

In [ ]:
from pathlib import Path
import ast
import json
import re


ORIGINAL_NOTEBOOK = Path(
    _crus_local_path('notebooks', 'revised_risk_aware_hybrid_pipeline_v2.ipynb')
)

if not ORIGINAL_NOTEBOOK.is_file():
    raise FileNotFoundError(ORIGINAL_NOTEBOOK)

notebook = json.loads(
    ORIGINAL_NOTEBOOK.read_text(encoding="utf-8-sig")
)

code_cells = []

for cell_number, cell in enumerate(
    notebook.get("cells", []),
    start=1,
):
    if cell.get("cell_type") != "code":
        continue

    source = cell.get("source", [])

    if isinstance(source, list):
        source = "".join(source)

    code_cells.append({
        "cell": cell_number,
        "execution_count": cell.get("execution_count"),
        "source": source,
    })


def print_excerpts(title, pattern, max_excerpts, context=5):
    print("\n" + "=" * 90)
    print(title)
    print("=" * 90)

    printed = 0

    for item in code_cells:
        lines = item["source"].splitlines()
        last_printed_end = -1

        for index, line in enumerate(lines):
            if not pattern.search(line):
                continue

            # Avoid repeatedly printing overlapping contexts.
            if index < last_printed_end:
                continue

            start = max(0, index - context)
            end = min(len(lines), index + context + 1)

            print(
                f"\nCELL {item['cell']} | "
                f"execution_count={item['execution_count']} | "
                f"lines {start + 1}-{end}"
            )

            for line_index in range(start, end):
                print(
                    f"{line_index + 1:04d}: "
                    f"{lines[line_index][:500]}"
                )

            last_printed_end = end
            printed += 1

            if printed >= max_excerpts:
                print(
                    "\nDisplay limit reached. "
                    "Additional matches may exist."
                )
                return

    if printed == 0:
        print("No matching code found.")


# ------------------------------------------------------------
# 1. Source filenames, download paths, and wildcard ingestion.
# ------------------------------------------------------------
source_pattern = re.compile(
    r"MTS[-_]Dialog[-_](?:TrainingSet|ValidationSet)|"
    r"source_manifest|"
    r"\b(?:glob|rglob)\s*\(",
    re.IGNORECASE,
)

print_excerpts(
    "A. SOURCE FILE SELECTION AND POSSIBLE WILDCARD INGESTION",
    source_pattern,
    max_excerpts=10,
    context=5,
)


# ------------------------------------------------------------
# 2. Explicit class-order construction or saved probability order.
#    These are clues, not automatic order verification.
# ------------------------------------------------------------
label_pattern = re.compile(
    r"\bclasses_\b|"
    r"\blabel_encoder\b|"
    r"\blabel_order\b|"
    r"\bclass_order\b|"
    r"\blabel_to_idx\b|"
    r"\blabel2id\b|"
    r"\bid2label\b|"
    r"^\s*(?:CLASSES|LABELS|classes|labels)\s*=",
)

print_excerpts(
    "B. LABEL / PROBABILITY-ORDER CONSTRUCTION",
    label_pattern,
    max_excerpts=12,
    context=4,
)


# ------------------------------------------------------------
# 3. Locate likely prediction, loading, and uncertainty functions.
#    AST parsing only: no function calls or notebook execution.
# ------------------------------------------------------------
function_pattern = re.compile(
    r"cascade|predict|crus|retrieve|retrieval|"
    r"load.*(?:model|artifact|index|frozen)|"
    r"entropy|rrf|regression",
    re.IGNORECASE,
)

function_locations = []
parse_failures = []

for item in code_cells:
    source = item["source"]

    # Preserve line numbering while removing common IPython-only
    # lines for this static function-location inventory.
    if any(
        line.lstrip().startswith("%%")
        for line in source.splitlines()
    ):
        parse_failures.append({
            "cell": item["cell"],
            "reason": "cell magic; not parsed",
        })
        continue

    sanitized = "\n".join(
        ""
        if line.lstrip().startswith(("!", "%", "?"))
        else line
        for line in source.splitlines()
    )

    try:
        tree = ast.parse(sanitized)
    except SyntaxError as exc:
        parse_failures.append({
            "cell": item["cell"],
            "reason": str(exc),
        })
        continue

    for node in ast.walk(tree):
        if isinstance(
            node,
            (ast.FunctionDef, ast.AsyncFunctionDef),
        ) and function_pattern.search(node.name):
            function_locations.append({
                "cell": item["cell"],
                "line": node.lineno,
                "end_line": getattr(node, "end_lineno", None),
                "name": node.name,
                "arguments": ast.unparse(node.args),
            })

print("\n" + "=" * 90)
print("C. LIKELY SOURCE FUNCTIONS — FIRST 80")
print("=" * 90)

for item in function_locations[:80]:
    print(
        f"Cell {item['cell']}, "
        f"lines {item['line']}-{item['end_line']}: "
        f"{item['name']}({item['arguments']})"
    )

print("\nMatching function definitions:", len(function_locations))

print("\nCells not AST-parsed:", len(parse_failures))
for item in parse_failures[:10]:
    print(item)

# Keep discoveries in notebook memory for targeted inspection.
ORIGINAL_SOURCE_CODE_CELLS = {
    item["cell"]: item["source"]
    for item in code_cells
}

ORIGINAL_SOURCE_FUNCTION_LOCATIONS = function_locations

print("\nREAD-ONLY SOURCE INSPECTION COMPLETE")
print("No notebook code executed.")
print("No models loaded or changed.")
print("No historical non-use attestation assigned.")
print("No held-out predictions generated.")

In [ ]:
# ------------------------------------------------------------
# Read selected ORIGINAL notebook cells.
# Nothing is executed, imported from the notebook, or modified.
# ------------------------------------------------------------
if "ORIGINAL_SOURCE_CODE_CELLS" not in globals():
    raise RuntimeError(
        "Run the preceding original-notebook inspection cell first."
    )

CELLS_TO_INSPECT = [
    (46, "SOURCE FILE RESOLUTION"),
    (47, "SOURCE POOL AND MANIFEST CONSTRUCTION"),
    (57, "CORRECTED MODEL INPUTS, CLASSES, AND SPLITS"),
    (72, "UNCERTAINTY COMPONENTS AND CRUS"),
    (73, "SCORING IMPLEMENTATION / SETUP"),
    (74, "SCORING IMPLEMENTATION / SETUP"),
    (75, "RETRIEVAL RESULTS AND METHOD WRAPPERS"),
]

MAX_LINES_PER_CELL = 220

for cell_number, description in CELLS_TO_INSPECT:
    print("\n" + "=" * 100)
    print(f"CELL {cell_number}: {description}")
    print("=" * 100)

    source = ORIGINAL_SOURCE_CODE_CELLS.get(cell_number)

    if source is None:
        print("No code cell found at this notebook position.")
        continue

    lines = source.splitlines()

    for line_number, line in enumerate(
        lines[:MAX_LINES_PER_CELL],
        start=1,
    ):
        print(f"{line_number:04d}: {line}")

    if len(lines) > MAX_LINES_PER_CELL:
        print(
            f"\nDISPLAY TRUNCATED: showing "
            f"{MAX_LINES_PER_CELL} of {len(lines)} lines."
        )

print("\nTARGETED INSPECTION COMPLETE")
print("No notebook code executed.")
print("No model fitting or inference performed.")
print("No source files or frozen artifacts changed.")

In [ ]:
from pathlib import Path
import json
import re

import numpy as np
import pandas as pd


# Snapshot existing globals before this inspection adds more names.
live = dict(globals())

project_dir = Path(
    _crus_local_path('project', '')
)

required_names = [
    "prepare_candidates",
    "score_documents",
    "composite_crus",
    "uncertainty_components",
    "normalized_entropy",
    "bm25_retrieve",
    "dense_retrieve",
    "rrf_fuse",
    "ce_raw_logits",
    "descending_order",
    "softmax",
    "CLASSES",
    "CIDX",
    "N_CLS",
    "X_train",
    "y_train_np",
    "train_data",
    "RETRIEVAL_CFG",
    "UNCERTAINTY_CFG",
]

print("LIVE SOURCE OBJECTS")
print("=" * 90)

for name in required_names:
    if name not in live:
        print(f"{name}: MISSING")
        continue

    value = live[name]
    print(
        f"{name}: "
        f"{type(value).__module__}.{type(value).__name__}"
    )


# ------------------------------------------------------------
# 1. Compare the live class order with the inspected source rule.
# ------------------------------------------------------------
verified_labels = pd.read_csv(
    project_dir
    / "corrected_experiment_outputs"
    / "comment5_audit"
    / "comment5_verified_class_distribution.csv",
    dtype=str,
    keep_default_na=False,
)

expected_order = sorted(verified_labels["label"].tolist())

checks = {}

if "CLASSES" in live:
    actual_order = list(live["CLASSES"])

    print("\nLIVE CLASS ORDER")
    print(json.dumps(actual_order, indent=2))

    checks["live_classes_match_inspected_source_order"] = (
        actual_order == expected_order
    )

    if "CIDX" in live:
        checks["live_cidx_matches_class_order"] = (
            live["CIDX"]
            == {
                label: position
                for position, label in enumerate(actual_order)
            }
        )

    if "N_CLS" in live:
        checks["live_class_count_is_17"] = (
            int(live["N_CLS"]) == 17
        )


# ------------------------------------------------------------
# 2. Compare live training IDs/labels with the constrained split.
#    Set equality checks membership; array checks verify alignment
#    with the live training DataFrame, not a guessed row order.
# ------------------------------------------------------------
manifest = pd.read_csv(
    project_dir
    / "data_audit"
    / "constrained_split_manifest.csv",
    dtype=str,
    keep_default_na=False,
)

if manifest["record_id"].duplicated().any():
    raise RuntimeError("Saved split manifest has duplicate IDs.")

expected_train = manifest.loc[
    manifest["split"].eq("train")
].copy()

checks["saved_constrained_training_count_is_773"] = (
    len(expected_train) == 773
)

train_frame = live.get("train_data")

if isinstance(train_frame, pd.DataFrame):
    print("\nLIVE TRAINING DATA")
    print("Rows:", len(train_frame))
    print("Columns:", train_frame.columns.tolist())

    needed_columns = {"record_id", "label", "text"}

    if needed_columns.issubset(train_frame.columns):
        ids = train_frame["record_id"].astype(str)

        checks["live_training_ids_are_unique"] = (
            not ids.duplicated().any()
        )

        checks["live_training_ids_match_constrained_train"] = (
            len(train_frame) == len(expected_train)
            and set(ids)
            == set(expected_train["record_id"])
        )

        expected_label_by_id = expected_train.set_index(
            "record_id"
        )["label"]

        expected_live_labels = ids.map(expected_label_by_id)

        checks["live_training_labels_match_saved_manifest"] = (
            expected_live_labels.notna().all()
            and np.array_equal(
                expected_live_labels.to_numpy(),
                train_frame["label"].astype(str).to_numpy(),
            )
        )

        if "X_train" in live:
            checks["x_train_matches_live_frame_order"] = (
                np.array_equal(
                    np.asarray(live["X_train"]),
                    train_frame["text"].to_numpy(),
                )
            )

        if "y_train_np" in live:
            checks["y_train_matches_live_frame_order"] = (
                np.array_equal(
                    np.asarray(live["y_train_np"]),
                    train_frame["label"].to_numpy(),
                )
            )
    else:
        print(
            "Cannot perform ID/text alignment checks; "
            "required columns are missing."
        )


# ------------------------------------------------------------
# 3. Display only the two relevant source configurations.
# ------------------------------------------------------------
print("\nLIVE RETRIEVAL / UNCERTAINTY CONFIGURATION")
print("=" * 90)

for name in ["RETRIEVAL_CFG", "UNCERTAINTY_CFG"]:
    print("\n" + name)

    if name not in live:
        print("MISSING")
    else:
        print(
            json.dumps(
                live[name],
                indent=2,
                default=str,
            )
        )


# ------------------------------------------------------------
# 4. Identify possible nested restoration state or model objects.
#    Print types only; do not load or call them.
# ------------------------------------------------------------
print("\nPOSSIBLE RESTORED STATE / MODEL OBJECTS — TYPES ONLY")
print("=" * 90)

object_name_pattern = re.compile(
    r"frozen|restor|adapter|"
    r"^(?:state|bm25|index|dense_index|"
    r"encoder|dense_encoder|ce_model|"
    r"cross_encoder|tokenizer)$",
    re.IGNORECASE,
)

for name, value in sorted(live.items()):
    if name.startswith("_"):
        continue

    if object_name_pattern.search(name):
        print(
            f"{name}: "
            f"{type(value).__module__}.{type(value).__name__}"
        )


print("\nAVAILABLE CONSISTENCY CHECKS")
print("=" * 90)

print(
    json.dumps(
        {name: bool(value) for name, value in checks.items()},
        indent=2,
    )
)

print("\nMissing required names:")
print([
    name
    for name in required_names
    if name not in live
])

print("\nInspection only.")
print("Training-text provenance and index alignment are not yet verified.")
print("Function equivalence and source regression are not yet verified.")
print("No models fitted, prediction functions called, or files changed.")

LIVE_CASCADE_INSPECTION_CHECKS = checks

In [ ]:
from pathlib import Path
import json

import pandas as pd


PROJECT_DIR = Path(
    _crus_local_path('project', '')
)

TRANSFER_NOTEBOOK = Path(
    _crus_local_path('notebooks', 'CRUS_R1_5_External_Transfer.ipynb')
)

if not TRANSFER_NOTEBOOK.is_file():
    raise FileNotFoundError(TRANSFER_NOTEBOOK)


# ------------------------------------------------------------
# 1. Inspect saved cells near the earlier source-restoration checks.
#    Notebook positions include both code and Markdown cells.
# ------------------------------------------------------------
notebook = json.loads(
    TRANSFER_NOTEBOOK.read_text(encoding="utf-8-sig")
)

cells = notebook.get("cells", [])

CELLS_TO_READ = [261, 262, 263, 264, 265]
MAX_LINES_PER_CELL = 180

RESTORATION_CELL_SOURCES = {}

print("SAVED RESTORATION-AREA CODE")
print("=" * 100)

for cell_number in CELLS_TO_READ:
    if cell_number > len(cells):
        print(f"\nCell {cell_number}: not present in saved notebook.")
        continue

    cell = cells[cell_number - 1]

    print("\n" + "=" * 100)
    print(
        f"CELL {cell_number} | "
        f"type={cell.get('cell_type')} | "
        f"execution_count={cell.get('execution_count')}"
    )
    print("=" * 100)

    if cell.get("cell_type") != "code":
        print("Not a code cell; skipped.")
        continue

    source = cell.get("source", [])

    if isinstance(source, list):
        source = "".join(source)

    RESTORATION_CELL_SOURCES[cell_number] = source
    lines = source.splitlines()

    for line_number, line in enumerate(
        lines[:MAX_LINES_PER_CELL],
        start=1,
    ):
        print(f"{line_number:04d}: {line}")

    if len(lines) > MAX_LINES_PER_CELL:
        print(
            f"\nTRUNCATED: displayed {MAX_LINES_PER_CELL} "
            f"of {len(lines)} lines. "
            "Full source retained in RESTORATION_CELL_SOURCES."
        )


# ------------------------------------------------------------
# 2. Inventory artifact filenames only.
#    Do not deserialize or execute any artifact.
# ------------------------------------------------------------
artifact_suffixes = {
    ".npz", ".npy",
    ".pkl", ".pickle", ".joblib",
    ".faiss", ".index",
    ".pt", ".pth", ".bin", ".safetensors",
}

excluded_prefixes = (
    "additional_mts_heldout_audit_",
    "aci_bench_external_data_audit_",
    "aci_external_annotation_workspace_",
    "r1_5_aci_bench_dataset_audit_",
)

artifact_rows = []

for path in PROJECT_DIR.rglob("*"):
    if not path.is_file():
        continue

    relative_path = path.relative_to(PROJECT_DIR)

    if any(
        part.startswith(excluded_prefixes)
        for part in relative_path.parts[:-1]
    ):
        continue

    lower_name = path.name.lower()

    binary_candidate = (
        path.suffix.lower() in artifact_suffixes
    )

    restoration_metadata_candidate = (
        path.suffix.lower() in {".json", ".jsonl", ".py"}
        and any(
            word in lower_name
            for word in [
                "frozen",
                "restore",
                "regression",
                "artifact",
                "checkpoint",
                "adapter",
            ]
        )
    )

    if binary_candidate or restoration_metadata_candidate:
        artifact_rows.append({
            "relative_path": str(relative_path),
            "bytes": path.stat().st_size,
        })

RESTORATION_ARTIFACT_INVENTORY = pd.DataFrame(
    artifact_rows,
    columns=["relative_path", "bytes"],
)

if not RESTORATION_ARTIFACT_INVENTORY.empty:
    RESTORATION_ARTIFACT_INVENTORY = (
        RESTORATION_ARTIFACT_INVENTORY
        .sort_values("relative_path")
        .reset_index(drop=True)
    )

print("\n\nPOTENTIAL SAVED ARTIFACTS — FIRST 100")
print("=" * 100)

if RESTORATION_ARTIFACT_INVENTORY.empty:
    print("No matching artifacts found inside this project folder.")
else:
    print(
        RESTORATION_ARTIFACT_INVENTORY.head(100)
        .to_string(index=False)
    )

print(
    "\nTotal matching artifact files:",
    len(RESTORATION_ARTIFACT_INVENTORY),
)

print("\nREAD-ONLY DISCOVERY COMPLETE")
print("No saved notebook code executed.")
print("No model artifacts deserialized.")
print("No dependencies installed.")
print("No training, indexing, or predictions performed.")
print("No files changed.")

In [ ]:
from pathlib import Path
import json


PROJECT_DIR = Path(
    _crus_local_path('project', '')
)

REPRODUCTION_DIR = (
    PROJECT_DIR
    / "r1_5_source_reproduction_20261017T081620155979Z"
)

summary_path = REPRODUCTION_DIR / "regression_summary.json"
trace_path = (
    REPRODUCTION_DIR / "source_validation_regression.jsonl"
)


# ------------------------------------------------------------
# 1. Read the saved regression summary.
# ------------------------------------------------------------
print("SAVED SOURCE-REGRESSION SUMMARY")
print("=" * 100)

if not summary_path.is_file():
    raise FileNotFoundError(summary_path)

summary = json.loads(
    summary_path.read_text(encoding="utf-8-sig")
)

print(json.dumps(summary, indent=2, ensure_ascii=False))


# ------------------------------------------------------------
# 2. Inspect trace structure and record counts only.
#    Do not print probabilities, dialogue, or individual traces.
# ------------------------------------------------------------
print("\nSAVED REGRESSION TRACE STRUCTURE")
print("=" * 100)

if not trace_path.is_file():
    raise FileNotFoundError(trace_path)

required_fields = {
    "record_id",
    "pred_label",
    "probabilities",
    "h_pred",
    "h_retrieval",
    "v_rrf",
    "crus",
    "candidate_ids",
    "candidate_rrf_scores",
}

record_ids = []
field_sets = set()
missing_field_rows = 0

with trace_path.open("r", encoding="utf-8-sig") as handle:
    for line_number, line in enumerate(handle, start=1):
        if not line.strip():
            continue

        record = json.loads(line)

        if not isinstance(record, dict):
            raise RuntimeError(
                f"Trace line {line_number} is not a JSON object."
            )

        field_sets.add(tuple(sorted(record.keys())))

        if not required_fields.issubset(record):
            missing_field_rows += 1

        record_ids.append(record.get("record_id"))

trace_structure = {
    "trace_records": len(record_ids),
    "unique_nonblank_record_ids": len({
        value
        for value in record_ids
        if isinstance(value, str) and value.strip()
    }),
    "rows_missing_runner_required_fields": missing_field_rows,
    "observed_field_sets": [
        list(fields)
        for fields in sorted(field_sets)
    ],
    "trace_values_recomputed": False,
}

print(json.dumps(trace_structure, indent=2))


# ------------------------------------------------------------
# 3. Display the remainder of the saved restoration cell.
#    Include a small overlap with the previous output.
# ------------------------------------------------------------
print("\nRESTORATION CELL 265 — LINES 160 ONWARD")
print("=" * 100)

if (
    "RESTORATION_CELL_SOURCES" not in globals()
    or 265 not in RESTORATION_CELL_SOURCES
):
    raise RuntimeError(
        "Run the preceding restoration-discovery cell first."
    )

source = RESTORATION_CELL_SOURCES[265]
source_lines = source.splitlines()

for line_number in range(160, len(source_lines) + 1):
    print(
        f"{line_number:04d}: "
        f"{source_lines[line_number - 1]}"
    )

print("\nREAD-ONLY INSPECTION COMPLETE")
print("Saved regression results read, not rerun.")
print("No pickled arrays or model objects loaded.")
print("No restoration code executed.")
print("No files changed.")

SAVED_SOURCE_REGRESSION_SUMMARY = summary

In [ ]:
from pathlib import Path
import json


# ------------------------------------------------------------
# 1. Retrieve the exact restoration code already inspected.
# ------------------------------------------------------------
if (
    "RESTORATION_CELL_SOURCES" not in globals()
    or 265 not in RESTORATION_CELL_SOURCES
):
    raise RuntimeError(
        "Missing inspected cell-265 source. "
        "Rerun the restoration-discovery cell first."
    )

if "SOURCE_RESTORATION_NAMESPACE" in globals():
    raise RuntimeError(
        "A restoration namespace already exists. "
        "Do not overwrite it automatically. "
        "Report the previous restoration output or error."
    )

restoration_source = RESTORATION_CELL_SOURCES[265]

# Isolate the restored source variables from the dataset-audit globals.
SOURCE_RESTORATION_NAMESPACE = {
    "__name__": "reviewed_source_data_restoration"
}

print("Restoring saved source data using inspected cell 265.")
print("No model fitting or query scoring is requested.")
print(
    "At the prompt, type TRUST only if these are "
    "your own trusted source NPZ files."
)

exec(
    compile(
        restoration_source,
        "<reviewed_transfer_notebook_cell_265>",
        "exec",
    ),
    SOURCE_RESTORATION_NAMESPACE,
)

if "SOURCE_STATE" not in SOURCE_RESTORATION_NAMESPACE:
    raise RuntimeError(
        "Restoration finished without creating SOURCE_STATE."
    )

restored_state = SOURCE_RESTORATION_NAMESPACE["SOURCE_STATE"]

print("\nRESTORED SOURCE DATA AVAILABLE")
print("Training records:", len(restored_state["train_ids"]))
print("Validation records:", len(restored_state["validation_ids"]))
print("Class count:", len(restored_state["classes"]))
print("Models have not yet been restored by this step.")


# ------------------------------------------------------------
# 2. Read the following saved cells, WITHOUT executing them.
# ------------------------------------------------------------
notebook_path = Path(
    _crus_local_path('notebooks', 'CRUS_R1_5_External_Transfer.ipynb')
)

notebook = json.loads(
    notebook_path.read_text(encoding="utf-8-sig")
)

NEXT_RESTORATION_CELL_SOURCES = {}

for cell_number in [266, 267]:
    cell = notebook["cells"][cell_number - 1]

    print("\n" + "=" * 100)
    print(
        f"SAVED CELL {cell_number} "
        f"| type={cell.get('cell_type')} "
        "| DISPLAY ONLY"
    )
    print("=" * 100)

    if cell.get("cell_type") != "code":
        print("Not a code cell; skipped.")
        continue

    source = cell.get("source", [])

    if isinstance(source, list):
        source = "".join(source)

    NEXT_RESTORATION_CELL_SOURCES[cell_number] = source

    for line_number, line in enumerate(
        source.splitlines(),
        start=1,
    ):
        print(f"{line_number:04d}: {line}")

print("\nSource-data restoration complete.")
print("Cells 266 and 267 were displayed, not executed.")
print("No held-out predictions generated.")

In [ ]:
import importlib.metadata as metadata
import subprocess
import sys


# ------------------------------------------------------------
# 1. Preconditions: use the previously restored isolated state.
# ------------------------------------------------------------
if "SOURCE_RESTORATION_NAMESPACE" not in globals():
    raise RuntimeError(
        "The source-restoration namespace is missing."
    )

ns = SOURCE_RESTORATION_NAMESPACE

if "SOURCE_STATE" not in ns:
    raise RuntimeError(
        "Verified SOURCE_STATE is missing from the namespace."
    )

if "SOURCE_RUNTIME" in ns:
    raise RuntimeError(
        "SOURCE_RUNTIME already exists. "
        "Do not reload automatically; report its current status."
    )

if "NEXT_RESTORATION_CELL_SOURCES" not in globals():
    raise RuntimeError(
        "The inspected model-restoration code is missing."
    )

for cell_number in [266, 267]:
    if cell_number not in NEXT_RESTORATION_CELL_SOURCES:
        raise RuntimeError(
            f"Inspected source for cell {cell_number} is missing."
        )

expected_versions = ns["SOURCE_STATE"]["protocol"][
    "package_versions"
]


# ------------------------------------------------------------
# 2. Check dependencies before importing the model packages.
#    Only the previously inspected missing-BM25 installation
#    is allowed here. No automatic upgrades.
# ------------------------------------------------------------
packages = [
    "numpy",
    "pandas",
    "scikit-learn",
    "sentence-transformers",
    "transformers",
    "rank-bm25",
    "torch",
    "scipy",
]

problems = []

print("SOURCE ENVIRONMENT PREFLIGHT")
print("=" * 80)

for package in packages:
    expected = expected_versions.get(package)

    try:
        current = metadata.version(package)
    except metadata.PackageNotFoundError:
        current = None

    if (
        package == "rank-bm25"
        and current is None
        and expected == "0.2.2"
    ):
        print("Installing missing rank-bm25==0.2.2 with --no-deps.")

        subprocess.run(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "--no-deps",
                "rank-bm25==0.2.2",
            ],
            check=True,
        )

        current = metadata.version(package)

    print(
        f"{package}: installed={current}, "
        f"source={expected if expected is not None else 'not recorded'}"
    )

    if current is None:
        problems.append(f"{package}: not installed")
        continue

    if expected is None:
        # Do not invent a source-version requirement.
        continue

    if package == "torch":
        matches = (
            current.split("+")[0]
            == str(expected).split("+")[0]
        )
    else:
        matches = current == str(expected)

    if not matches:
        problems.append(
            f"{package}: installed={current}, source={expected}"
        )

if problems:
    raise RuntimeError(
        "Environment check failed. Do not upgrade automatically.\n"
        + "\n".join(problems)
    )


# ------------------------------------------------------------
# 3. Execute inspected cell 266: source model restoration.
# ------------------------------------------------------------
print("\nLOADING SOURCE MODELS")
print("=" * 80)

exec(
    compile(
        NEXT_RESTORATION_CELL_SOURCES[266],
        "<inspected_source_model_restoration_cell_266>",
        "exec",
    ),
    ns,
)

if "SOURCE_RUNTIME" not in ns:
    raise RuntimeError(
        "Model-restoration cell did not create SOURCE_RUNTIME."
    )


# ------------------------------------------------------------
# 4. Execute inspected cell 267: exact source functions plus
#    the predetermined first-five-validation-record smoke test.
# ------------------------------------------------------------
print("\nRUNNING FIVE-RECORD SOURCE SMOKE TEST")
print("=" * 80)

exec(
    compile(
        NEXT_RESTORATION_CELL_SOURCES[267],
        "<inspected_source_smoke_test_cell_267>",
        "exec",
    ),
    ns,
)


# ------------------------------------------------------------
# 5. Require all five checks to pass without relaxing tolerance.
# ------------------------------------------------------------
smoke = ns["SMOKE_RESULTS"]

required_checks = [
    "prediction_matches",
    "acceptance_matches",
    "probabilities_within_1e_minus6",
    "components_within_1e_minus6",
]

if len(smoke) != 5:
    raise RuntimeError(
        f"Expected five smoke-test records; found {len(smoke)}."
    )

failed_checks = [
    column
    for column in required_checks
    if not smoke[column].eq(True).all()
]

# A smoke pass must never substitute for full regression.
ns["SOURCE_RUNTIME"]["source_regression_passed"] = False

if failed_checks:
    raise RuntimeError(
        "Source smoke test failed. Stop without changing "
        "models, thresholds, or tolerance. Failed checks: "
        + ", ".join(failed_checks)
    )

print("\nFIVE-RECORD SMOKE TEST PASSED")
print("Source models and scorer are available in the isolated namespace.")
print("Full 258-record source regression is still required.")
print("No held-out records scored.")

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from numbers import Integral
import hashlib
import json
import time

import numpy as np
import pandas as pd
import torch


# ------------------------------------------------------------
# 1. Preconditions and immutable reference inputs.
# ------------------------------------------------------------
if "SOURCE_RESTORATION_NAMESPACE" not in globals():
    raise RuntimeError("Source-restoration namespace is missing.")

ns = SOURCE_RESTORATION_NAMESPACE

for name in ["SOURCE_STATE", "SOURCE_RUNTIME", "CASCADE_NS"]:
    if name not in ns:
        raise RuntimeError(f"{name} is missing.")

state = ns["SOURCE_STATE"]
runtime = ns["SOURCE_RUNTIME"]
scorer = ns["CASCADE_NS"]

# Never retain a passing flag while a new verification is incomplete.
runtime["source_regression_passed"] = False

PROJECT_DIR = Path(
    _crus_local_path('project', '')
)

REFERENCE_TRACE = (
    PROJECT_DIR
    / "r1_5_source_reproduction_20261017T081620155979Z"
    / "source_validation_regression.jsonl"
)

TOLERANCE = 1e-6

with REFERENCE_TRACE.open("r", encoding="utf-8-sig") as handle:
    reference_records = [
        json.loads(line)
        for line in handle
        if line.strip()
    ]

reference_ids = [
    record["record_id"] for record in reference_records
]

if len(reference_ids) != 258 or len(set(reference_ids)) != 258:
    raise RuntimeError("Reference trace must contain 258 unique IDs.")

reference_by_id = {
    record["record_id"]: record
    for record in reference_records
}

validation_ids = [
    str(value) for value in state["validation_ids"]
]

if (
    len(validation_ids) != 258
    or len(set(validation_ids)) != 258
    or set(validation_ids) != set(reference_by_id)
):
    raise RuntimeError("Validation IDs and reference trace disagree.")

classes = list(state["classes"])
train_ids = [str(value) for value in state["train_ids"]]
train_id_set = set(train_ids)

if len(train_ids) != 773 or len(train_id_set) != 773:
    raise RuntimeError("Invalid source training IDs.")

if list(scorer["CLASSES"]) != classes:
    raise RuntimeError("Scorer class order differs from source state.")

if not np.array_equal(scorer["train_ids"], state["train_ids"]):
    raise RuntimeError("Scorer training-ID order has changed.")

if not np.array_equal(scorer["X_train"], state["train_text"]):
    raise RuntimeError("Scorer training text has changed.")

if not np.array_equal(scorer["y_train_np"], state["train_labels"]):
    raise RuntimeError("Scorer training labels have changed.")

if not np.array_equal(scorer["train_embs"], state["train_embeddings"]):
    raise RuntimeError("Scorer training embeddings have changed.")

if scorer["RETRIEVAL_CFG"] != state["retrieval_config"]:
    raise RuntimeError("Scorer retrieval configuration has changed.")

if scorer["UNCERTAINTY_CFG"] != state["uncertainty_config"]:
    raise RuntimeError("Scorer uncertainty configuration has changed.")

policy = state["protocol"]["frozen_policies"]["Cascade:CRUS"]
weights = np.asarray(
    state["uncertainty_config"]["weights"],
    dtype=float,
)


def max_difference(current, reference):
    current = np.asarray(current, dtype=float)
    reference = np.asarray(reference, dtype=float)

    if (
        current.shape != reference.shape
        or not np.isfinite(current).all()
        or not np.isfinite(reference).all()
    ):
        return float("inf")

    if current.size == 0:
        return 0.0

    return float(np.max(np.abs(current - reference)))


timestamp = datetime.now(timezone.utc).strftime(
    "%Y%m%dT%H%M%S%fZ"
)

run_dir = (
    PROJECT_DIR
    / f"r1_5_source_reverification_{timestamp}"
)

run_dir.mkdir(exist_ok=False)

rows = []
current_traces = []
started = time.perf_counter()

print("FULL SOURCE-VALIDATION REGRESSION")
print("Records:", len(validation_ids))
print("Device:", runtime["device"])
print("Tolerance:", TOLERANCE)
print("No held-out records will be scored.")


# ------------------------------------------------------------
# 2. Score every validation record in SAVED order.
# ------------------------------------------------------------
try:
    for position, record_id in enumerate(validation_ids):
        query = str(state["validation_text"][position])
        gold = str(state["validation_labels"][position])
        reference = reference_by_id[record_id]

        with torch.inference_mode():
            candidates = scorer["prepare_candidates"](query)
            result = scorer["score_documents"](
                query, candidates, "Cascade"
            )

        probabilities = np.asarray(result["probs"], dtype=float)
        components = np.asarray(result["components"], dtype=float)

        if (
            probabilities.shape != (17,)
            or not np.isfinite(probabilities).all()
            or (probabilities < 0).any()
            or not np.isclose(probabilities.sum(), 1.0)
        ):
            raise RuntimeError(
                f"Invalid probabilities for {record_id}"
            )

        if (
            components.shape != (3,)
            or not np.isfinite(components).all()
            or (components < 0).any()
            or (components > 1).any()
        ):
            raise RuntimeError(
                f"Invalid components for {record_id}"
            )

        prediction = str(result["pred"])
        crus = float(scorer["composite_crus"](components))
        confidence = float(result["confidence"])

        document_ids = []
        document_scores = []

        for index, rrf_score in result["documents"]:
            if (
                isinstance(index, bool)
                or not isinstance(index, Integral)
                or not 0 <= int(index) < len(train_ids)
            ):
                raise RuntimeError(
                    f"Invalid retrieval index for {record_id}"
                )

            document_ids.append(train_ids[int(index)])
            document_scores.append(float(rrf_score))

        if (
            not document_ids
            or len(set(document_ids)) != len(document_ids)
            or not set(document_ids).issubset(train_id_set)
        ):
            raise RuntimeError(
                f"Invalid final training-document set for {record_id}"
            )

        shortlist = [str(value) for value in result["shortlist"]]
        fallback = bool(result["fallback"])

        accepted = bool(
            confidence >= policy["tau_c"]
            and crus <= policy["tau_u"]
        )

        saved_probabilities = (
            state["saved_validation_probabilities"][position]
        )
        saved_components = (
            state["saved_validation_components"][position]
        )

        saved_crus = float(
            np.asarray(saved_components, dtype=float) @ weights
        )

        saved_acceptance = bool(
            np.max(saved_probabilities) >= policy["tau_c"]
            and saved_crus <= policy["tau_u"]
        )

        reference_components = [
            reference["h_pred"],
            reference["h_retrieval"],
            reference["v_rrf"],
        ]

        query_hash = hashlib.sha256(
            query.encode("utf-8")
        ).hexdigest()

        differences = {
            "probability_difference_from_original_arrays":
                max_difference(probabilities, saved_probabilities),

            "component_difference_from_original_arrays":
                max_difference(components, saved_components),

            "crus_difference_from_original_arrays":
                abs(crus - saved_crus),

            "probability_difference_from_reference_trace":
                max_difference(
                    probabilities, reference["probabilities"]
                ),

            "component_difference_from_reference_trace":
                max_difference(
                    components, reference_components
                ),

            "crus_difference_from_reference_trace":
                abs(crus - float(reference["crus"])),

            "rrf_score_difference_from_reference_trace":
                max_difference(
                    document_scores,
                    reference["candidate_rrf_scores"],
                ),
        }

        checks = {
            "prediction_matches_original":
                prediction == str(
                    state["saved_validation_predictions"][position]
                ),

            "prediction_matches_reference_trace":
                prediction == reference["pred_label"],

            "acceptance_matches_original":
                accepted == saved_acceptance,

            "document_ids_and_order_match":
                document_ids == reference["candidate_ids"],

            "shortlist_and_order_match":
                shortlist == reference["shortlist"],

            "fallback_matches":
                fallback == reference["fallback"],

            "query_hash_matches":
                query_hash == reference["query_sha256"],

            "prediction_matches_current_argmax":
                prediction == classes[int(probabilities.argmax())],

            "confidence_matches_current_probabilities":
                abs(confidence - float(probabilities.max()))
                <= TOLERANCE,

            "crus_matches_fixed_weight_composite":
                abs(crus - float(components @ weights))
                <= TOLERANCE,
        }

        checks.update({
            f"{name}_within_tolerance": value <= TOLERANCE
            for name, value in differences.items()
        })

        rows.append({
            "record_id": record_id,
            "correct": prediction == gold,
            "accepted": accepted,
            "accepted_correct": accepted and prediction == gold,
            **checks,
            **differences,
        })

        current_traces.append({
            "record_id": record_id,
            "query_sha256": query_hash,
            "pred_label": prediction,
            "probabilities": probabilities.tolist(),
            "h_pred": float(components[0]),
            "h_retrieval": float(components[1]),
            "v_rrf": float(components[2]),
            "crus": crus,
            "candidate_ids": document_ids,
            "candidate_rrf_scores": document_scores,
            "shortlist": shortlist,
            "fallback": fallback,
        })

        if (position + 1) % 25 == 0 or position == 257:
            print(f"Checked {position + 1}/258")

except Exception:
    # Preserve completed checks, without assigning a passing flag.
    pd.DataFrame(rows).to_csv(
        run_dir / "partial_regression_checks.csv",
        index=False,
    )
    print("\nStopped. Completed checks saved in:", run_dir)
    print("Full source regression remains FALSE.")
    raise


# ------------------------------------------------------------
# 3. Require every record-level check and the original counts.
# ------------------------------------------------------------
checks_frame = pd.DataFrame(rows)

check_names = list(checks)
difference_names = list(differences)

counts = {
    "forced_correct": int(checks_frame["correct"].sum()),
    "accepted": int(checks_frame["accepted"].sum()),
    "accepted_correct": int(
        checks_frame["accepted_correct"].sum()
    ),
}

expected_counts = {
    "forced_correct": 174,
    "accepted": 132,
    "accepted_correct": 124,
}

failed_check_counts = {
    name: int((~checks_frame[name]).sum())
    for name in check_names
    if not checks_frame[name].all()
}

passed = bool(
    len(checks_frame) == 258
    and not failed_check_counts
    and counts == expected_counts
)

summary = {
    "passed": passed,
    "records": len(checks_frame),
    "counts": counts,
    "expected_counts": expected_counts,
    "check_matches": {
        name: int(checks_frame[name].sum())
        for name in check_names
    },
    "failed_check_counts": failed_check_counts,
    "maximum_differences": {
        name: float(checks_frame[name].max())
        for name in difference_names
    },
    "tolerance": TOLERANCE,
    "device": runtime["device"],
    "loaded_revisions": runtime["loaded_revisions"],
    "scorer_source_sha256": runtime["scorer_source_sha256"],
    "reference_trace_sha256": hashlib.sha256(
        REFERENCE_TRACE.read_bytes()
    ).hexdigest(),
    "elapsed_seconds": time.perf_counter() - started,
    "scope": (
        "Current-runtime numerical source reproduction; "
        "not held-out evaluation, proof of historical "
        "model-file identity, or proof of test-set non-use."
    ),
}

checks_frame.to_csv(
    run_dir / "source_validation_regression_checks.csv",
    index=False,
)

with (run_dir / "source_validation_regression.jsonl").open(
    "w", encoding="utf-8"
) as handle:
    for record in current_traces:
        handle.write(json.dumps(record) + "\n")

(run_dir / "regression_summary.json").write_text(
    json.dumps(summary, indent=2),
    encoding="utf-8",
)

print("\nFULL SOURCE-REGRESSION SUMMARY")
print("=" * 90)
print(json.dumps(summary, indent=2))
print("\nNew regression folder:", run_dir)

if not passed:
    raise RuntimeError(
        "Full source regression failed. "
        "Do not change tolerance, weights, or thresholds "
        "to force a pass."
    )

# Set the flag only after checks pass and reports are saved.
runtime["source_regression_passed"] = True
runtime["source_regression_directory"] = str(run_dir)

print("\nFULL SOURCE REGRESSION PASSED")
print("No held-out records scored.")

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json

import numpy as np
import pandas as pd
import torch


# ------------------------------------------------------------
# 1. Require the current-runtime source regression to have passed.
# ------------------------------------------------------------
if "SOURCE_RESTORATION_NAMESPACE" not in globals():
    raise RuntimeError("Source-restoration namespace is missing.")

ns = SOURCE_RESTORATION_NAMESPACE

state = ns["SOURCE_STATE"]
runtime = ns["SOURCE_RUNTIME"]
scorer = ns["CASCADE_NS"]

if runtime.get("source_regression_passed") is not True:
    raise RuntimeError(
        "Full current-runtime source regression has not passed."
    )

regression_dir = Path(
    runtime["source_regression_directory"]
)

regression_summary_path = (
    regression_dir / "regression_summary.json"
)

regression_summary = json.loads(
    regression_summary_path.read_text(encoding="utf-8")
)

if (
    regression_summary.get("passed") is not True
    or regression_summary.get("records") != 258
    or regression_summary.get("counts") != {
        "forced_correct": 174,
        "accepted": 132,
        "accepted_correct": 124,
    }
):
    raise RuntimeError("Unexpected source-regression summary.")

if (
    regression_summary["scorer_source_sha256"]
    != runtime["scorer_source_sha256"]
):
    raise RuntimeError("Source scorer fingerprint mismatch.")


# ------------------------------------------------------------
# 2. Verify that the scorer still uses the verified source state.
# ------------------------------------------------------------
for scorer_key, state_key in [
    ("train_ids", "train_ids"),
    ("X_train", "train_text"),
    ("y_train_np", "train_labels"),
    ("train_embs", "train_embeddings"),
]:
    if not np.array_equal(
        scorer[scorer_key], state[state_key]
    ):
        raise RuntimeError(
            f"Scorer source input changed: {scorer_key}"
        )

if scorer["RETRIEVAL_CFG"] != state["retrieval_config"]:
    raise RuntimeError("Retrieval configuration changed.")

if scorer["UNCERTAINTY_CFG"] != state["uncertainty_config"]:
    raise RuntimeError("Uncertainty configuration changed.")

classes = list(state["classes"])

if list(scorer["CLASSES"]) != classes:
    raise RuntimeError("Scorer class order changed.")

policy = state["protocol"]["frozen_policies"]["Cascade:CRUS"]
weights = np.asarray(
    state["uncertainty_config"]["weights"],
    dtype=float,
)

if (
    policy["tau_c"] != 0.0
    or abs(policy["tau_u"] - 0.128165036474669) > 1e-15
    or not np.allclose(
        weights, [0.5, 0.3, 0.2], rtol=0, atol=1e-12
    )
):
    raise RuntimeError("Frozen source policy changed.")


# ------------------------------------------------------------
# 3. Load the previously saved 400-record screening audit.
# ------------------------------------------------------------
screen_path = Path(
    _crus_local_path('project', 'additional_mts_heldout_audit_20261017T085443976861Z/eligibility_screen_20261017T085623681478Z/all_records_with_screening_flags.csv')
)

audit = pd.read_csv(
    screen_path,
    dtype=str,
    keep_default_na=False,
)

def boolean_column(frame, name):
    values = frame[name].str.strip().str.lower()

    if not values.isin(["true", "false"]).all():
        raise RuntimeError(f"Invalid Boolean values in {name}.")

    return values.eq("true")


passes_screen = boolean_column(
    audit, "passes_automated_screen_only"
)

expected_screen = (
    boolean_column(audit, "passes_automatic_checks")
    & ~boolean_column(
        audit, "near_source_overlap_review_required"
    )
)

if not passes_screen.equals(expected_screen):
    raise RuntimeError("Saved screening flags are inconsistent.")

selected = audit.loc[passes_screen].copy()

if len(audit) != 400 or len(selected) != 386:
    raise RuntimeError(
        f"Unexpected cohort sizes: {len(audit)} total, "
        f"{len(selected)} selected."
    )

if selected["record_id"].duplicated().any():
    raise RuntimeError("Selected record IDs are not unique.")

if not selected["gold"].isin(classes).all():
    raise RuntimeError("Unsupported labels remain in the cohort.")

# Preserve the current, pre-performance selection and row order.
selected["normalized_dialogue"] = selected["dialogue"].map(
    lambda text: " ".join(text.lower().split())
)

if selected["normalized_dialogue"].eq("").any():
    raise RuntimeError("Empty inference dialogue.")

if selected["normalized_dialogue"].duplicated().any():
    raise RuntimeError("Repeated inference dialogue remains.")

if selected["normalized_dialogue"].isin(
    set(state["train_text"])
).any():
    raise RuntimeError("Exact normalized training overlap remains.")


# ------------------------------------------------------------
# 4. Explicitly acknowledge the unresolved history limitation.
# ------------------------------------------------------------
print("EVALUATION SCOPE")
print("=" * 80)
print("386 screened additional official MTS-Dialog test records.")
print("Exploratory evaluation; historical development non-use unverified.")
print("Not independent cross-dataset validation.")
print("No target-domain tuning or model changes.")

acknowledgment = input(
    "\nType EXPLORATORY to run with that documented scope: "
).strip()

if acknowledgment != "EXPLORATORY":
    raise RuntimeError("Stopped without running test inference.")


# ------------------------------------------------------------
# 5. Create a NEW run and separate reference labels from inputs.
# ------------------------------------------------------------
project_dir = Path(
    _crus_local_path('project', '')
)

timestamp = datetime.now(timezone.utc).strftime(
    "%Y%m%dT%H%M%S%fZ"
)

run_dir = (
    project_dir / f"r1_5_additional_mts_exploratory_{timestamp}"
)

run_dir.mkdir(exist_ok=False)

audit.to_csv(
    run_dir / "all_400_records_screening_audit.csv",
    index=False,
)

inference_inputs = selected[
    ["record_id", "normalized_dialogue"]
].rename(columns={"normalized_dialogue": "query"})

reference = selected[
    [
        "record_id",
        "official_test_set",
        "source_filename",
        "source_csv_row",
        "gold",
    ]
].copy()

inference_inputs.to_csv(
    run_dir / "inference_inputs.csv", index=False
)

reference.to_csv(
    run_dir / "evaluation_reference.csv", index=False
)

def sha256_file(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()


manifest = {
    "evaluation_type": "exploratory_additional_MTS_Dialog_testing",
    "independent_cross_dataset_validation": False,
    "historical_development_non_use_verified": False,
    "provenance_and_semantic_overlap_review_completed": False,
    "selection": (
        "Supported labels; exclude blank dialogues, exact source "
        "overlaps, all repeated-dialogue group members, and "
        "records flagged by the completed near-source screen."
    ),
    "downloaded_records": 400,
    "selected_records": 386,
    "source_training_records": len(state["train_ids"]),
    "classes_in_probability_order": classes,
    "tau_c": policy["tau_c"],
    "tau_u": policy["tau_u"],
    "crus_weights": weights.tolist(),
    "device": runtime["device"],
    "loaded_revisions": runtime["loaded_revisions"],
    "scorer_source_sha256": runtime["scorer_source_sha256"],
    "source_regression_directory": str(regression_dir),
    "source_regression_summary_sha256": sha256_file(
        regression_summary_path
    ),
    "screening_audit_sha256": sha256_file(screen_path),
    "inference_inputs_sha256": sha256_file(
        run_dir / "inference_inputs.csv"
    ),
    "evaluation_reference_sha256": sha256_file(
        run_dir / "evaluation_reference.csv"
    ),
    "created_utc": datetime.now(timezone.utc).isoformat(),
}

(run_dir / "run_manifest.json").write_text(
    json.dumps(manifest, indent=2),
    encoding="utf-8",
)


# ------------------------------------------------------------
# 6. Score queries only. Gold labels are not passed to the scorer.
#    Stream predictions so completed records survive interruption.
# ------------------------------------------------------------
train_ids = [str(value) for value in state["train_ids"]]
completed = 0

try:
    with (run_dir / "predictions.jsonl").open(
        "x", encoding="utf-8"
    ) as handle:
        for row in inference_inputs.itertuples(index=False):
            with torch.inference_mode():
                candidates = scorer["prepare_candidates"](row.query)

                result = scorer["score_documents"](
                    row.query, candidates, "Cascade"
                )

            probabilities = np.asarray(
                result["probs"], dtype=float
            )
            components = np.asarray(
                result["components"], dtype=float
            )

            if (
                probabilities.shape != (17,)
                or not np.isfinite(probabilities).all()
                or (probabilities < 0).any()
                or not np.isclose(probabilities.sum(), 1.0)
            ):
                raise RuntimeError(
                    f"Invalid probabilities: {row.record_id}"
                )

            if (
                components.shape != (3,)
                or not np.isfinite(components).all()
                or (components < 0).any()
                or (components > 1).any()
            ):
                raise RuntimeError(
                    f"Invalid uncertainty components: {row.record_id}"
                )

            prediction = str(result["pred"])
            confidence = float(result["confidence"])
            crus = float(
                scorer["composite_crus"](components)
            )

            if prediction != classes[int(probabilities.argmax())]:
                raise RuntimeError("Prediction/argmax mismatch.")

            if abs(confidence - probabilities.max()) > 1e-6:
                raise RuntimeError("Confidence mismatch.")

            if abs(crus - float(components @ weights)) > 1e-6:
                raise RuntimeError("CRUS composite mismatch.")

            candidate_ids = []
            candidate_rrf_scores = []

            for index, score in result["documents"]:
                if (
                    isinstance(index, (bool, np.bool_))
                    or not isinstance(index, (int, np.integer))
                    or not 0 <= int(index) < len(train_ids)
                    or not np.isfinite(float(score))
                ):
                    raise RuntimeError("Invalid retrieved document.")

                candidate_ids.append(train_ids[int(index)])
                candidate_rrf_scores.append(float(score))

            if (
                not candidate_ids
                or len(set(candidate_ids)) != len(candidate_ids)
            ):
                raise RuntimeError("Invalid final document set.")

            record = {
                "record_id": row.record_id,
                "query_sha256": hashlib.sha256(
                    row.query.encode("utf-8")
                ).hexdigest(),
                "pred_label": prediction,
                "probabilities": probabilities.tolist(),
                "confidence": confidence,
                "h_pred": float(components[0]),
                "h_retrieval": float(components[1]),
                "v_rrf": float(components[2]),
                "crus": crus,
                "accepted": bool(
                    confidence >= policy["tau_c"]
                    and crus <= policy["tau_u"]
                ),
                "candidate_ids": candidate_ids,
                "candidate_rrf_scores": candidate_rrf_scores,
                "shortlist": [
                    str(value) for value in result["shortlist"]
                ],
                "fallback": bool(result["fallback"]),
            }

            handle.write(json.dumps(record) + "\n")
            completed += 1

            if completed % 25 == 0 or completed == 386:
                handle.flush()
                print(f"Scored {completed}/386")

except BaseException as exc:
    (run_dir / "inference_status.json").write_text(
        json.dumps({
            "completed": False,
            "records_written": completed,
            "error_type": type(exc).__name__,
            "error": str(exc),
        }, indent=2),
        encoding="utf-8",
    )
    print("\nIncomplete run preserved:", run_dir)
    print("Do not evaluate a partial run as the full cohort.")
    raise

status = {
    "completed": True,
    "records_written": completed,
    "predictions_sha256": sha256_file(
        run_dir / "predictions.jsonl"
    ),
    "performance_metrics_computed": False,
}

(run_dir / "inference_status.json").write_text(
    json.dumps(status, indent=2),
    encoding="utf-8",
)

ADDITIONAL_MTS_RUN_DIR = run_dir

print("\nADDITIONAL MTS-DIALOG INFERENCE COMPLETE")
print(json.dumps(status, indent=2))
print("Run folder:", run_dir)
print("Models, thresholds, and source corpus unchanged.")
print("Historical development non-use remains unverified.")

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json

import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
)


# ------------------------------------------------------------
# 1. Locate the completed prediction run.
#    No live model namespace is required.
# ------------------------------------------------------------
RUN_DIR = Path(
    _crus_local_path('project', 'r1_5_additional_mts_exploratory_20261017T101158614697Z')
)

def read_json(path):
    return json.loads(path.read_text(encoding="utf-8"))

def sha256_file(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

manifest = read_json(RUN_DIR / "run_manifest.json")
status = read_json(RUN_DIR / "inference_status.json")

if (
    status.get("completed") is not True
    or status.get("records_written") != 386
):
    raise RuntimeError("Inference run is incomplete.")

integrity_checks = [
    (
        "predictions.jsonl",
        status["predictions_sha256"],
    ),
    (
        "inference_inputs.csv",
        manifest["inference_inputs_sha256"],
    ),
    (
        "evaluation_reference.csv",
        manifest["evaluation_reference_sha256"],
    ),
]

for filename, expected_hash in integrity_checks:
    if sha256_file(RUN_DIR / filename) != expected_hash:
        raise RuntimeError(f"Checksum mismatch: {filename}")

reference = pd.read_csv(
    RUN_DIR / "evaluation_reference.csv",
    dtype=str,
    keep_default_na=False,
)

inputs = pd.read_csv(
    RUN_DIR / "inference_inputs.csv",
    dtype=str,
    keep_default_na=False,
)

with (RUN_DIR / "predictions.jsonl").open(
    "r", encoding="utf-8"
) as handle:
    records = [
        json.loads(line)
        for line in handle
        if line.strip()
    ]

prediction_ids = [row["record_id"] for row in records]

if (
    len(records) != 386
    or len(set(prediction_ids)) != 386
    or reference["record_id"].duplicated().any()
    or inputs["record_id"].duplicated().any()
    or set(prediction_ids) != set(reference["record_id"])
    or set(prediction_ids) != set(inputs["record_id"])
):
    raise RuntimeError("Record IDs or counts are inconsistent.")

# Align everything explicitly to the reference order.
prediction_by_id = {
    row["record_id"]: row for row in records
}
reference = reference.reset_index(drop=True)
records = [
    prediction_by_id[record_id]
    for record_id in reference["record_id"]
]

query_by_id = inputs.set_index("record_id")["query"]

for row in records:
    expected_query_hash = hashlib.sha256(
        query_by_id.loc[row["record_id"]].encode("utf-8")
    ).hexdigest()

    if row["query_sha256"] != expected_query_hash:
        raise RuntimeError(
            f"Query hash mismatch: {row['record_id']}"
        )


# ------------------------------------------------------------
# 2. Reconstruct and verify saved prediction quantities.
# ------------------------------------------------------------
classes = manifest["classes_in_probability_order"]

if len(classes) != 17 or len(set(classes)) != 17:
    raise RuntimeError("Invalid saved source class vocabulary.")

class_index = {
    label: index for index, label in enumerate(classes)
}

gold = reference["gold"].to_numpy(dtype=str)

if not set(gold).issubset(class_index):
    raise RuntimeError("Unsupported gold labels in evaluation.")

probabilities = np.asarray(
    [row["probabilities"] for row in records],
    dtype=float,
)

components = np.asarray(
    [
        [row["h_pred"], row["h_retrieval"], row["v_rrf"]]
        for row in records
    ],
    dtype=float,
)

crus = np.asarray(
    [row["crus"] for row in records], dtype=float
)

if (
    probabilities.shape != (386, 17)
    or not np.isfinite(probabilities).all()
    or (probabilities < 0).any()
    or not np.allclose(probabilities.sum(axis=1), 1.0)
):
    raise RuntimeError("Invalid saved probability matrix.")

if (
    components.shape != (386, 3)
    or not np.isfinite(components).all()
    or (components < 0).any()
    or (components > 1).any()
    or not np.isfinite(crus).all()
):
    raise RuntimeError("Invalid saved uncertainty values.")

weights = np.asarray(manifest["crus_weights"], dtype=float)

if not np.allclose(
    crus, components @ weights, rtol=0, atol=1e-6
):
    raise RuntimeError("Saved CRUS does not match its components.")

predictions = np.asarray(
    [row["pred_label"] for row in records], dtype=str
)

argmax_predictions = np.asarray(classes, dtype=str)[
    probabilities.argmax(axis=1)
]

if not np.array_equal(predictions, argmax_predictions):
    raise RuntimeError("Prediction/argmax mismatch.")

confidence = probabilities.max(axis=1)

if not np.allclose(
    confidence,
    np.asarray([row["confidence"] for row in records]),
    rtol=0,
    atol=1e-6,
):
    raise RuntimeError("Saved confidence mismatch.")

accepted = (
    (confidence >= manifest["tau_c"])
    & (crus <= manifest["tau_u"])
)

if not all(
    isinstance(row["accepted"], bool)
    for row in records
):
    raise RuntimeError("Saved acceptance values are not Boolean.")

if not np.array_equal(
    accepted,
    np.asarray([row["accepted"] for row in records]),
):
    raise RuntimeError("Saved acceptance/gate mismatch.")

correct = predictions == gold
errors = ~correct
ids = reference["record_id"].to_numpy(dtype=str)

# Normalized predictive entropy: zero-probability terms contribute zero.
log_probabilities = np.zeros_like(probabilities)
np.log(
    probabilities,
    out=log_probabilities,
    where=probabilities > 0,
)

entropy = -(
    probabilities * log_probabilities
).sum(axis=1) / np.log(len(classes))

if not np.allclose(
    entropy, components[:, 0], rtol=0, atol=1e-6
):
    raise RuntimeError("Predictive-entropy reconstruction mismatch.")

sorted_probabilities = np.sort(probabilities, axis=1)
margin = (
    sorted_probabilities[:, -1]
    - sorted_probabilities[:, -2]
)


# ------------------------------------------------------------
# 3. Forced and frozen-native-gate metrics.
# ------------------------------------------------------------
def performance(mask):
    mask = np.asarray(mask, dtype=bool)
    n = int(mask.sum())

    if n == 0:
        return {
            "records": 0,
            "forced_accuracy": None,
            "forced_macro_f1_fixed17": None,
            "accepted_records": 0,
            "coverage": None,
            "accepted_correct": 0,
            "accepted_errors": 0,
            "selective_accuracy": None,
            "selective_error": None,
        }

    selected = mask & accepted
    k = int(selected.sum())

    return {
        "records": n,
        "forced_correct": int(correct[mask].sum()),
        "forced_errors": int(errors[mask].sum()),
        "forced_accuracy": float(
            accuracy_score(gold[mask], predictions[mask])
        ),
        "forced_macro_f1_fixed17": float(
            f1_score(
                gold[mask],
                predictions[mask],
                labels=classes,
                average="macro",
                zero_division=0,
            )
        ),
        "accepted_records": k,
        "coverage": k / n,
        "accepted_correct": int(correct[selected].sum()),
        "accepted_errors": int(errors[selected].sum()),
        "selective_accuracy": (
            float(correct[selected].mean()) if k else None
        ),
        "selective_error": (
            float(errors[selected].mean()) if k else None
        ),
        "deferred_records": n - k,
    }

overall = performance(np.ones(len(gold), dtype=bool))

by_test_set = {
    name: performance(
        reference["official_test_set"].eq(name).to_numpy()
    )
    for name in sorted(reference["official_test_set"].unique())
}


# ------------------------------------------------------------
# 4. Calibration: fixed equal-width 15-bin top-label ECE,
#    summed-17 Brier score, and NLL with floor 1e-12.
# ------------------------------------------------------------
gold_indices = np.asarray(
    [class_index[label] for label in gold]
)

one_hot_gold = np.eye(len(classes))[gold_indices]

def calibration_metrics(mask, scope):
    n = int(mask.sum())

    if n == 0:
        return {
            "scope": scope,
            "records": 0,
            "top_label_ece_15_bins": None,
            "brier_sum17": None,
            "nll_floor_1e_minus12": None,
        }

    local_confidence = confidence[mask]
    local_correct = correct[mask]
    bins = np.minimum(
        (local_confidence * 15).astype(int), 14
    )

    ece = 0.0

    for bin_index in range(15):
        members = bins == bin_index

        if members.any():
            ece += (
                members.mean()
                * abs(
                    local_correct[members].mean()
                    - local_confidence[members].mean()
                )
            )

    true_class_probability = probabilities[
        np.arange(len(gold)), gold_indices
    ][mask]

    return {
        "scope": scope,
        "records": n,
        "top_label_ece_15_bins": float(ece),
        "brier_sum17": float(
            (
                (probabilities[mask] - one_hot_gold[mask]) ** 2
            ).sum(axis=1).mean()
        ),
        "nll_floor_1e_minus12": float(
            -np.log(
                np.maximum(true_class_probability, 1e-12)
            ).mean()
        ),
    }

calibration_table = pd.DataFrame([
    calibration_metrics(
        np.ones(len(gold), dtype=bool), "all"
    ),
    calibration_metrics(accepted, "native_accepted"),
])


# ------------------------------------------------------------
# 5. Uncertainty comparisons.
#    Lower score = more certain for every selector.
#    Ties: ascending lexicographic record_id.
#    Matched count: native accepted count, never retuned.
# ------------------------------------------------------------
uncertainty_scores = {
    "CRUS": crus,
    "PredictiveEntropy": entropy,
    "MaxProbability": 1.0 - confidence,
    "ProbabilityMargin": 1.0 - margin,
}

native_count = int(accepted.sum())
has_both_outcomes = np.unique(errors).size == 2

comparison_rows = []
curve_parts = []

for method, scores in uncertainty_scores.items():
    order = np.lexsort((ids, scores))
    prefix_counts = np.arange(1, len(gold) + 1)

    cumulative_errors = np.cumsum(errors[order])
    prefix_risk = cumulative_errors / prefix_counts

    matched_indices = order[:native_count]

    comparison_rows.append({
        "selector": method,
        "aurc_mean_prefix_risk": float(prefix_risk.mean()),
        "error_auroc": (
            float(roc_auc_score(errors.astype(int), scores))
            if has_both_outcomes else None
        ),
        "error_average_precision": (
            float(
                average_precision_score(
                    errors.astype(int), scores
                )
            )
            if has_both_outcomes else None
        ),
        "matched_accepted_count": native_count,
        "matched_errors": (
            int(errors[matched_indices].sum())
            if native_count else None
        ),
        "matched_risk": (
            float(errors[matched_indices].mean())
            if native_count else None
        ),
        "matched_accuracy": (
            float(correct[matched_indices].mean())
            if native_count else None
        ),
    })

    curve_parts.append(pd.DataFrame({
        "selector": method,
        "accepted_count": prefix_counts,
        "coverage": prefix_counts / len(gold),
        "cumulative_errors": cumulative_errors,
        "selective_risk": prefix_risk,
    }))

comparison = pd.DataFrame(comparison_rows)
risk_coverage = pd.concat(curve_parts, ignore_index=True)


# ------------------------------------------------------------
# 6. Per-class support and deferral audit.
# ------------------------------------------------------------
class_rows = []

for label in classes:
    mask = gold == label
    selected = mask & accepted
    deferred = mask & ~accepted

    support = int(mask.sum())
    selected_count = int(selected.sum())
    deferred_count = int(deferred.sum())

    class_rows.append({
        "label": label,
        "support": support,
        "forced_correct": int(correct[mask].sum()),
        "forced_accuracy": (
            float(correct[mask].mean()) if support else None
        ),
        "accepted": selected_count,
        "coverage": (
            selected_count / support if support else None
        ),
        "accepted_correct": int(correct[selected].sum()),
        "accepted_errors": int(errors[selected].sum()),
        "selective_accuracy": (
            float(correct[selected].mean())
            if selected_count else None
        ),
        "deferred": deferred_count,
        "deferred_correct": int(correct[deferred].sum()),
        "deferred_errors": int(errors[deferred].sum()),
    })

class_table = pd.DataFrame(class_rows)


# ------------------------------------------------------------
# 7. Save a new evaluation folder; no model or prediction changes.
# ------------------------------------------------------------
timestamp = datetime.now(timezone.utc).strftime(
    "%Y%m%dT%H%M%S%fZ"
)

evaluation_dir = RUN_DIR / f"evaluation_{timestamp}"
evaluation_dir.mkdir(exist_ok=False)

case_audit = reference.copy()
case_audit["prediction"] = predictions
case_audit["correct"] = correct
case_audit["confidence"] = confidence
case_audit["predictive_entropy"] = entropy
case_audit["probability_margin"] = margin
case_audit["h_retrieval"] = components[:, 1]
case_audit["v_rrf"] = components[:, 2]
case_audit["crus"] = crus
case_audit["native_accepted"] = accepted

metrics = {
    "scope": manifest["evaluation_type"],
    "independent_cross_dataset_validation": False,
    "historical_development_non_use_verified": False,
    "overall": overall,
    "by_official_test_set": by_test_set,
    "class_support": {
        label: int((gold == label).sum())
        for label in classes
    },
    "absent_source_classes": [
        label for label in classes
        if not (gold == label).any()
    ],
    "frozen_policy": {
        "tau_c": manifest["tau_c"],
        "tau_u": manifest["tau_u"],
        "crus_weights": manifest["crus_weights"],
    },
    "conventions": {
        "rates": "fractions from 0 to 1",
        "macro_f1": "fixed 17 source labels; zero_division=0",
        "aurc": "arithmetic mean of risk at all nonempty prefixes",
        "ranking_ties": "ascending lexicographic record_id",
        "matched_count": "native accepted count",
        "ece": "15 equal-width confidence bins",
        "brier": "sum over 17 classes, then mean over records",
        "nll": "natural log with true-class probability floor 1e-12",
        "confidence_intervals": "not calculated in this cell",
    },
    "limitations": [
        "Same-dataset additional testing, not cross-dataset validation.",
        "Historical development non-use remains unverified.",
        "Automated overlap screening does not prove independence.",
        "Unsupported labels and flagged overlaps excluded before scoring.",
        "Original consultation grouping is not established here.",
        "Historical model-file identity is not established.",
    ],
}

(evaluation_dir / "metrics.json").write_text(
    json.dumps(metrics, indent=2, allow_nan=False),
    encoding="utf-8",
)

comparison.to_csv(
    evaluation_dir / "uncertainty_comparison.csv", index=False
)
risk_coverage.to_csv(
    evaluation_dir / "risk_coverage.csv", index=False
)
calibration_table.to_csv(
    evaluation_dir / "calibration.csv", index=False
)
class_table.to_csv(
    evaluation_dir / "class_deferral.csv", index=False
)
case_audit.to_csv(
    evaluation_dir / "case_audit.csv", index=False
)

print("OVERALL PERFORMANCE")
print("=" * 90)
print(json.dumps(overall, indent=2))

print("\nPER OFFICIAL TEST SET")
print("=" * 90)
print(json.dumps(by_test_set, indent=2))

print("\nUNCERTAINTY COMPARISON")
print("=" * 90)
print(comparison.to_string(index=False))

print("\nCALIBRATION")
print("=" * 90)
print(calibration_table.to_string(index=False))

print("\nPER-CLASS DEFERRAL")
print("=" * 90)
print(class_table.to_string(index=False))

print("\nEvaluation folder:", evaluation_dir)
print("No model inference, fitting, or threshold tuning performed.")
print("Exploratory same-dataset scope retained.")

ADDITIONAL_MTS_EVALUATION_DIR = evaluation_dir

In [ ]:
from pathlib import Path
import hashlib
import json

pass  # Colab service replaced by local workflow.


# Mount/reuse Drive; do not force a remount.
_crus_prepare_saved_workspace()

MYDRIVE = Path(_crus_local_path('drive', ''))
PROJECT_DIR = MYDRIVE / "RiskAware_Revision2"

EXPECTED_RUN_NAME = (
    "r1_5_additional_mts_exploratory_20261017T101158614697Z"
)

EXPECTED_PREDICTIONS_SHA256 = (
    "785a90593f2025b6e3f6c3f1a38c5f0a82f0"
    "a8e688bda13d5df81ee2b4f9222a"
)

required_files = [
    "run_manifest.json",
    "inference_status.json",
    "predictions.jsonl",
    "inference_inputs.csv",
    "evaluation_reference.csv",
]

candidate_dirs = {
    PROJECT_DIR / EXPECTED_RUN_NAME,
}

# Include the original in-memory location if still available.
if "ADDITIONAL_MTS_RUN_DIR" in globals():
    candidate_dirs.add(Path(ADDITIONAL_MTS_RUN_DIR))

if PROJECT_DIR.is_dir():
    candidate_dirs.update(
        path
        for path in PROJECT_DIR.glob(
            "r1_5_additional_mts_exploratory_*"
        )
        if path.is_dir()
    )

print("DRIVE / PROJECT CHECK")
print("=" * 80)
print("MyDrive accessible:", MYDRIVE.is_dir())
print("Project folder accessible:", PROJECT_DIR.is_dir())

matching_runs = []

for folder in sorted(candidate_dirs, key=str):
    print("\nCANDIDATE:", folder)

    if not folder.is_dir():
        print("Folder not found.")
        continue

    missing = [
        filename
        for filename in required_files
        if not (folder / filename).is_file()
    ]

    print("Missing required files:", missing)

    predictions_path = folder / "predictions.jsonl"

    if not predictions_path.is_file():
        continue

    actual_hash = hashlib.sha256(
        predictions_path.read_bytes()
    ).hexdigest()

    print(
        "Matches completed prediction checksum:",
        actual_hash == EXPECTED_PREDICTIONS_SHA256,
    )

    if actual_hash != EXPECTED_PREDICTIONS_SHA256:
        continue

    if missing:
        print(
            "Correct prediction file found, but supporting files "
            "are missing. Do not fabricate replacements."
        )
        continue

    status = json.loads(
        (folder / "inference_status.json").read_text(
            encoding="utf-8"
        )
    )

    if (
        status.get("completed") is not True
        or status.get("records_written") != 386
        or status.get("predictions_sha256") != actual_hash
    ):
        print("Completion-status check failed.")
        continue

    matching_runs.append(folder)

if len(matching_runs) != 1:
    if not PROJECT_DIR.is_dir() and MYDRIVE.is_dir():
        print("\nTop-level Drive folders — first 40:")
        for path in sorted(MYDRIVE.iterdir())[:40]:
            if path.is_dir():
                print(path.name)

    raise RuntimeError(
        f"Found {len(matching_runs)} complete matching runs. "
        "Stop here and paste this recovery output. "
        "No results were changed."
    )

RECOVERED_MTS_RUN_DIR = matching_runs[0]

print("\nCOMPLETED RUN RECOVERED")
print("Verified run folder:", RECOVERED_MTS_RUN_DIR)
print("All five required files are present.")
print("Prediction checksum matches your completed 386-record run.")
print("No models loaded, inference rerun, or files modified.")

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json

import numpy as np
import pandas as pd
from sklearn.metrics import (
    f1_score,
    roc_auc_score,
    average_precision_score,
)


# ============================================================
# 1. Locate the completed run.
# ============================================================
if not Path(_crus_local_path('drive', '')).is_dir():
    pass  # Colab service replaced by local workflow.
    _crus_prepare_saved_workspace()

RUN_DIR = Path(
    globals().get(
        "RECOVERED_MTS_RUN_DIR",
        _crus_local_path('project', 'r1_5_additional_mts_exploratory_20261017T101158614697Z'),
    )
)

def read_json(path):
    return json.loads(path.read_text(encoding="utf-8"))

def sha256_file(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

required_files = [
    "run_manifest.json",
    "inference_status.json",
    "predictions.jsonl",
    "inference_inputs.csv",
    "evaluation_reference.csv",
]

for filename in required_files:
    if not (RUN_DIR / filename).is_file():
        raise FileNotFoundError(RUN_DIR / filename)

manifest = read_json(RUN_DIR / "run_manifest.json")
status = read_json(RUN_DIR / "inference_status.json")

if (
    status.get("completed") is not True
    or status.get("records_written") != 386
):
    raise RuntimeError("The prediction run is incomplete.")

expected_prediction_hash = (
    "785a90593f2025b6e3f6c3f1a38c5f0a82f0"
    "a8e688bda13d5df81ee2b4f9222a"
)

if status["predictions_sha256"] != expected_prediction_hash:
    raise RuntimeError("Unexpected prediction-run checksum.")

for filename, expected_hash in [
    ("predictions.jsonl", status["predictions_sha256"]),
    ("inference_inputs.csv", manifest["inference_inputs_sha256"]),
    (
        "evaluation_reference.csv",
        manifest["evaluation_reference_sha256"],
    ),
]:
    if sha256_file(RUN_DIR / filename) != expected_hash:
        raise RuntimeError(f"Checksum mismatch: {filename}")


# ============================================================
# 2. Load and align predictions, queries, and reference labels.
# ============================================================
reference = pd.read_csv(
    RUN_DIR / "evaluation_reference.csv",
    dtype=str,
    keep_default_na=False,
)

inputs = pd.read_csv(
    RUN_DIR / "inference_inputs.csv",
    dtype=str,
    keep_default_na=False,
)

with (RUN_DIR / "predictions.jsonl").open(
    "r", encoding="utf-8"
) as handle:
    records = [
        json.loads(line)
        for line in handle
        if line.strip()
    ]

prediction_ids = [record["record_id"] for record in records]

if (
    len(records) != 386
    or len(reference) != 386
    or len(inputs) != 386
    or len(set(prediction_ids)) != 386
    or reference["record_id"].duplicated().any()
    or inputs["record_id"].duplicated().any()
    or set(prediction_ids) != set(reference["record_id"])
    or set(prediction_ids) != set(inputs["record_id"])
):
    raise RuntimeError("Record IDs or counts are inconsistent.")

prediction_by_id = {
    record["record_id"]: record for record in records
}

reference = reference.reset_index(drop=True)
records = [
    prediction_by_id[record_id]
    for record_id in reference["record_id"]
]

query_by_id = inputs.set_index("record_id")["query"]

for record in records:
    query_hash = hashlib.sha256(
        query_by_id.loc[record["record_id"]].encode("utf-8")
    ).hexdigest()

    if query_hash != record["query_sha256"]:
        raise RuntimeError(
            f"Query checksum mismatch: {record['record_id']}"
        )


# ============================================================
# 3. Verify prediction quantities and the frozen gate.
# ============================================================
classes = list(manifest["classes_in_probability_order"])

if len(classes) != 17 or len(set(classes)) != 17:
    raise RuntimeError("Invalid source class order.")

class_index = {
    label: index for index, label in enumerate(classes)
}

ids = reference["record_id"].to_numpy(dtype=str)
gold = reference["gold"].to_numpy(dtype=str)

if not set(gold).issubset(class_index):
    raise RuntimeError("Unsupported reference labels.")

predictions = np.asarray(
    [record["pred_label"] for record in records],
    dtype=str,
)

probabilities = np.asarray(
    [record["probabilities"] for record in records],
    dtype=float,
)

components = np.asarray([
    [
        record["h_pred"],
        record["h_retrieval"],
        record["v_rrf"],
    ]
    for record in records
], dtype=float)

crus = np.asarray(
    [record["crus"] for record in records],
    dtype=float,
)

if (
    probabilities.shape != (386, 17)
    or not np.isfinite(probabilities).all()
    or (probabilities < 0).any()
    or (probabilities > 1).any()
    or not np.allclose(
        probabilities.sum(axis=1), 1.0, rtol=0, atol=1e-6
    )
):
    raise RuntimeError("Invalid probability matrix.")

if (
    components.shape != (386, 3)
    or not np.isfinite(components).all()
    or (components < 0).any()
    or (components > 1).any()
    or not np.isfinite(crus).all()
):
    raise RuntimeError("Invalid uncertainty values.")

weights = np.asarray(manifest["crus_weights"], dtype=float)
tau_c = float(manifest["tau_c"])
tau_u = float(manifest["tau_u"])

if (
    not np.allclose(
        weights, [0.5, 0.3, 0.2], rtol=0, atol=1e-12
    )
    or tau_c != 0.0
    or abs(tau_u - 0.128165036474669) > 1e-15
):
    raise RuntimeError("Unexpected frozen source policy.")

if not np.allclose(
    crus, components @ weights, rtol=0, atol=1e-6
):
    raise RuntimeError("CRUS reconstruction mismatch.")

argmax_predictions = np.asarray(classes, dtype=str)[
    probabilities.argmax(axis=1)
]

if not np.array_equal(predictions, argmax_predictions):
    raise RuntimeError("Prediction/argmax mismatch.")

confidence = probabilities.max(axis=1)

if not np.allclose(
    confidence,
    np.asarray([record["confidence"] for record in records]),
    rtol=0,
    atol=1e-6,
):
    raise RuntimeError("Confidence mismatch.")

accepted = (confidence >= tau_c) & (crus <= tau_u)

if not all(
    isinstance(record["accepted"], bool)
    for record in records
):
    raise RuntimeError("Saved acceptance values are not Boolean.")

if not np.array_equal(
    accepted,
    np.asarray([record["accepted"] for record in records]),
):
    raise RuntimeError("Frozen-gate acceptance mismatch.")

correct = predictions == gold
errors = ~correct
n = len(gold)

log_p = np.zeros_like(probabilities)
np.log(probabilities, out=log_p, where=probabilities > 0)

entropy = -(
    probabilities * log_p
).sum(axis=1) / np.log(len(classes))

if not np.allclose(
    entropy, components[:, 0], rtol=0, atol=1e-6
):
    raise RuntimeError("Predictive-entropy mismatch.")

sorted_p = np.sort(probabilities, axis=1)
margin = sorted_p[:, -1] - sorted_p[:, -2]


# ============================================================
# 4. Overall and per-test performance.
# ============================================================
def performance(mask):
    mask = np.asarray(mask, dtype=bool)
    count = int(mask.sum())
    selected = mask & accepted
    selected_count = int(selected.sum())

    return {
        "records": count,
        "forced_correct": int(correct[mask].sum()),
        "forced_errors": int(errors[mask].sum()),
        "forced_accuracy": (
            float(correct[mask].mean()) if count else None
        ),
        "forced_macro_f1_fixed17": (
            float(f1_score(
                gold[mask],
                predictions[mask],
                labels=classes,
                average="macro",
                zero_division=0,
            ))
            if count else None
        ),
        "accepted_records": selected_count,
        "coverage": (
            selected_count / count if count else None
        ),
        "accepted_correct": int(correct[selected].sum()),
        "accepted_errors": int(errors[selected].sum()),
        "selective_accuracy": (
            float(correct[selected].mean())
            if selected_count else None
        ),
        "selective_error": (
            float(errors[selected].mean())
            if selected_count else None
        ),
        "deferred_records": count - selected_count,
    }

overall = performance(np.ones(n, dtype=bool))

by_test_set = {
    name: performance(
        reference["official_test_set"].eq(name).to_numpy()
    )
    for name in sorted(
        reference["official_test_set"].unique()
    )
}


# ============================================================
# 5. Uncertainty comparison.
#    Lower score = more certain.
#    Ties use lexicographic record_id.
# ============================================================
uncertainty_scores = {
    "CRUS": crus,
    "PredictiveEntropy": entropy,
    "MaxProbability": 1.0 - confidence,
    "ProbabilityMargin": 1.0 - margin,
}

native_count = int(accepted.sum())
both_outcomes = np.unique(errors).size == 2

comparison_rows = []
curve_parts = []

for selector, scores in uncertainty_scores.items():
    order = np.lexsort((ids, scores))
    prefix_counts = np.arange(1, n + 1)
    cumulative_errors = np.cumsum(errors[order])
    prefix_risk = cumulative_errors / prefix_counts
    matched = order[:native_count]

    comparison_rows.append({
        "selector": selector,
        "aurc_mean_prefix_risk": float(prefix_risk.mean()),
        "error_auroc": (
            float(roc_auc_score(errors.astype(int), scores))
            if both_outcomes else None
        ),
        "error_average_precision": (
            float(average_precision_score(
                errors.astype(int), scores
            ))
            if both_outcomes else None
        ),
        "matched_accepted_count": native_count,
        "matched_errors": (
            int(errors[matched].sum())
            if native_count else None
        ),
        "matched_risk": (
            float(errors[matched].mean())
            if native_count else None
        ),
        "matched_accuracy": (
            float(correct[matched].mean())
            if native_count else None
        ),
    })

    curve_parts.append(pd.DataFrame({
        "selector": selector,
        "accepted_count": prefix_counts,
        "coverage": prefix_counts / n,
        "cumulative_errors": cumulative_errors,
        "selective_risk": prefix_risk,
    }))

comparison = pd.DataFrame(comparison_rows)
risk_coverage = pd.concat(curve_parts, ignore_index=True)


# ============================================================
# 6. Calibration.
# ============================================================
gold_indices = np.asarray([
    class_index[label] for label in gold
])

one_hot = np.eye(len(classes))[gold_indices]
true_class_p = probabilities[np.arange(n), gold_indices]

def calibration(mask, scope):
    count = int(mask.sum())

    if not count:
        return {
            "scope": scope,
            "records": 0,
            "ece_15_bins": None,
            "brier_sum17": None,
            "nll_floor_1e_minus12": None,
        }

    local_confidence = confidence[mask]
    local_correct = correct[mask]

    bins = np.minimum(
        (15 * local_confidence).astype(int), 14
    )

    ece = 0.0

    for bin_index in range(15):
        members = bins == bin_index

        if members.any():
            ece += members.mean() * abs(
                local_correct[members].mean()
                - local_confidence[members].mean()
            )

    return {
        "scope": scope,
        "records": count,
        "ece_15_bins": float(ece),
        "brier_sum17": float(
            ((probabilities[mask] - one_hot[mask]) ** 2)
            .sum(axis=1)
            .mean()
        ),
        "nll_floor_1e_minus12": float(
            -np.log(np.maximum(true_class_p[mask], 1e-12)).mean()
        ),
    }

calibration_table = pd.DataFrame([
    calibration(np.ones(n, dtype=bool), "all"),
    calibration(accepted, "native_accepted"),
])


# ============================================================
# 7. Class-level deferral.
# ============================================================
class_rows = []

for label in classes:
    mask = gold == label
    selected = mask & accepted
    deferred = mask & ~accepted

    support = int(mask.sum())
    selected_count = int(selected.sum())

    class_rows.append({
        "label": label,
        "support": support,
        "forced_correct": int(correct[mask].sum()),
        "forced_accuracy": (
            float(correct[mask].mean()) if support else None
        ),
        "accepted": selected_count,
        "coverage": (
            selected_count / support if support else None
        ),
        "accepted_correct": int(correct[selected].sum()),
        "accepted_errors": int(errors[selected].sum()),
        "selective_accuracy": (
            float(correct[selected].mean())
            if selected_count else None
        ),
        "deferred": int(deferred.sum()),
        "deferred_correct": int(correct[deferred].sum()),
        "deferred_errors": int(errors[deferred].sum()),
    })

class_table = pd.DataFrame(class_rows)


# ============================================================
# 8. Save results in a NEW evaluation folder.
# ============================================================
timestamp = datetime.now(timezone.utc).strftime(
    "%Y%m%dT%H%M%S%fZ"
)

evaluation_dir = RUN_DIR / f"evaluation_{timestamp}"
evaluation_dir.mkdir(exist_ok=False)

case_audit = reference.copy()
case_audit["prediction"] = predictions
case_audit["correct"] = correct
case_audit["confidence"] = confidence
case_audit["predictive_entropy"] = entropy
case_audit["probability_margin"] = margin
case_audit["h_retrieval"] = components[:, 1]
case_audit["v_rrf"] = components[:, 2]
case_audit["crus"] = crus
case_audit["native_accepted"] = accepted

metrics = {
    "scope": "exploratory_additional_MTS_Dialog_testing",
    "independent_cross_dataset_validation": False,
    "historical_development_non_use_verified": False,
    "overall": overall,
    "by_official_test_set": by_test_set,
    "class_support": {
        label: int((gold == label).sum())
        for label in classes
    },
    "absent_source_classes": [
        label for label in classes
        if not (gold == label).any()
    ],
    "frozen_policy": {
        "tau_c": tau_c,
        "tau_u": tau_u,
        "crus_weights": weights.tolist(),
    },
    "conventions": {
        "rates": "fractions from 0 to 1",
        "macro_f1": "fixed 17 labels; zero_division=0",
        "aurc": "mean risk over all nonempty ranked prefixes",
        "ranking_ties": "ascending lexicographic record_id",
        "matched_count": "native accepted count; no tuning",
        "ece": "15 equal-width bins; probability 1 in final bin",
        "brier": "sum over 17 classes, then mean over records",
        "nll": "natural log; probability floor 1e-12",
        "confidence_intervals": "not calculated",
    },
    "limitations": [
        "Additional same-dataset testing, not cross-dataset validation.",
        "Historical development non-use remains unverified.",
        "Automated screening does not establish full independence.",
        "Original consultation grouping has not been established.",
        "Historical model-file identity remains unverified.",
        "Uncertainty comparisons are descriptive.",
    ],
    "input_checksums": {
        filename: sha256_file(RUN_DIR / filename)
        for filename in required_files
    },
}

(evaluation_dir / "metrics.json").write_text(
    json.dumps(metrics, indent=2, allow_nan=False),
    encoding="utf-8",
)

for filename, table in [
    ("uncertainty_comparison.csv", comparison),
    ("risk_coverage.csv", risk_coverage),
    ("calibration.csv", calibration_table),
    ("class_deferral.csv", class_table),
    ("case_audit.csv", case_audit),
]:
    table.to_csv(evaluation_dir / filename, index=False)


# ============================================================
# 9. Print the results needed for interpretation.
# ============================================================
print("OVERALL PERFORMANCE")
print("=" * 90)
print(json.dumps(overall, indent=2))

print("\nPER OFFICIAL TEST SET")
print("=" * 90)
print(json.dumps(by_test_set, indent=2))

print("\nUNCERTAINTY COMPARISON")
print("=" * 90)
print(comparison.to_string(index=False))

print("\nCALIBRATION")
print("=" * 90)
print(calibration_table.to_string(index=False))

print("\nPER-CLASS DEFERRAL")
print("=" * 90)
print(class_table.to_string(index=False))

print("\nEVALUATION COMPLETE")
print("Evaluation folder:", evaluation_dir)
print("All rates are fractions, not percentages.")
print("No models loaded or inference rerun.")
print("No fitting, threshold tuning, or original-file changes.")
print("Exploratory same-dataset scope retained.")

ADDITIONAL_MTS_EVALUATION_DIR = evaluation_dir

In [ ]:
from pathlib import Path
import json
import hashlib
import platform

import numpy as np
import pandas as pd
import sklearn
import matplotlib.pyplot as plt

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_recall_curve,
    auc,
    brier_score_loss,
    log_loss,
)

SEED = 42
PROB_PREFIX = "p__"


def prepare_frame(df, probability_columns=None):
    df = df.copy()

    required = {"record_id", "y_true", "y_pred", "crus"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Missing columns: {sorted(missing)}")

    if df[list(required)].isna().any().any():
        raise ValueError("Missing IDs, labels, predictions, or CRUS values.")

    if df["record_id"].duplicated().any():
        raise ValueError("record_id must be unique within each evaluation set.")

    if probability_columns is None:
        probability_columns = [
            c for c in df.columns if c.startswith(PROB_PREFIX)
        ]

    if len(probability_columns) < 2:
        raise ValueError("Full class probabilities are required.")

    missing_p = set(probability_columns) - set(df.columns)
    if missing_p:
        raise ValueError(f"Missing probability columns: {sorted(missing_p)}")

    p = df[probability_columns].to_numpy(dtype=float)

    if not np.isfinite(p).all():
        raise ValueError("Probabilities contain NaN or infinity.")
    if (p < 0).any() or (p > 1).any():
        raise ValueError("Probabilities must be in [0, 1].")
    if not np.allclose(p.sum(axis=1), 1.0, atol=1e-5):
        raise ValueError(
            "Probability rows do not sum to 1. "
            "Use the saved full probability vectors."
        )

    crus = df["crus"].to_numpy(dtype=float)
    if not np.isfinite(crus).all():
        raise ValueError("CRUS contains NaN or infinity.")
    if (crus < 0).any() or (crus > 1).any():
        raise ValueError("Expected the bounded saved CRUS score in [0, 1].")

    classes = np.array(
        [c[len(PROB_PREFIX):] for c in probability_columns],
        dtype=str,
    )
    if len(set(classes)) != len(classes):
        raise ValueError("Duplicate class names in probability columns.")

    truth = df["y_true"].astype(str).to_numpy()
    pred = df["y_pred"].astype(str).to_numpy()

    if not set(truth).issubset(set(classes)):
        raise ValueError("Ground-truth labels do not match probability columns.")
    if not set(pred).issubset(set(classes)):
        raise ValueError("Predicted labels do not match probability columns.")

    pred_index = pd.Index(classes).get_indexer(pred)
    pred_probability = p[np.arange(len(p)), pred_index]

    if not np.allclose(pred_probability, p.max(axis=1), atol=1e-7):
        raise ValueError(
            "Saved predictions are not maximum-probability predictions. "
            "Resolve this mismatch before comparing the selectors."
        )

    top_two = np.sort(p, axis=1)[:, -2:]
    max_probability = top_two[:, 1]
    margin = top_two[:, 1] - top_two[:, 0]

    log_p = np.zeros_like(p)
    np.log(p, out=log_p, where=p > 0)
    entropy = -(p * log_p).sum(axis=1) / np.log(p.shape[1])

    features = np.column_stack([
        max_probability,
        entropy,
        margin,
    ])

    errors = (pred != truth).astype(int)

    return {
        "df": df,
        "p": p,
        "errors": errors,
        "features": features,
        "max_probability": max_probability,
        "entropy": entropy,
        "margin": margin,
        "crus": crus,
        "probability_columns": probability_columns,
    }


def fit_correctness_estimator(validation):
    # Correctness is the binary target. Test labels are never used here.
    correct = 1 - validation["errors"]
    counts = np.bincount(correct, minlength=2)

    if counts.min() < 2:
        raise ValueError(
            "Validation needs at least two correct and two incorrect "
            "predictions for cross-validated sigmoid calibration."
        )

    folds = min(3, int(counts.min()))

    base = make_pipeline(
        StandardScaler(),
        LogisticRegression(
            C=1.0,
            solver="lbfgs",
            max_iter=2000,
            random_state=SEED,
        ),
    )

    cv = StratifiedKFold(
        n_splits=folds,
        shuffle=True,
        random_state=SEED,
    )

    model = CalibratedClassifierCV(
        estimator=base,
        method="sigmoid",
        cv=cv,
        ensemble=True,
    )
    model.fit(validation["features"], correct)
    return model, folds


def deterministic_order(scores, record_ids):
    scores = np.asarray(scores, dtype=float)
    if not np.isfinite(scores).all():
        raise ValueError("Non-finite selector scores.")

    ids = pd.Series(record_ids).reset_index(drop=True)
    numeric_ids = pd.to_numeric(ids, errors="coerce")

    if numeric_ids.notna().all():
        if numeric_ids.duplicated().any():
            raise ValueError("IDs become duplicated after numeric conversion.")
        tie_key = numeric_ids.to_numpy()
    else:
        tie_key = ids.astype(str).to_numpy()

    return np.lexsort((tie_key, scores))


def frame_hash(df):
    values = pd.util.hash_pandas_object(
        df.reset_index(drop=True), index=False
    ).to_numpy()
    return hashlib.sha256(values.tobytes()).hexdigest()


def evaluate_dataset(
    test_df,
    validation,
    correctness_model,
    output_dir,
    dataset_name,
    native_count,
    expected_n=None,
    expected_errors=None,
    coverages=(0.25, 0.50, 0.75, 0.90, 1.00),
):
    output_dir = Path(output_dir) / dataset_name
    output_dir.mkdir(parents=True, exist_ok=True)

    test = prepare_frame(
        test_df,
        probability_columns=validation["probability_columns"],
    )

    n = len(test_df)
    if expected_n is not None and n != expected_n:
        raise ValueError(
            f"{dataset_name}: expected {expected_n} records, found {n}. "
            "Do not mix manuscript/pipeline versions."
        )

    if not isinstance(native_count, (int, np.integer)):
        raise ValueError("native_count must be an integer.")
    if not 1 <= native_count <= n:
        raise ValueError("native_count must be between 1 and N.")

    val_ids = set(validation["df"]["record_id"].astype(str))
    test_ids = set(test["df"]["record_id"].astype(str))
    overlap = val_ids & test_ids
    if overlap:
        raise ValueError(
            "Validation/test record IDs overlap. Use stable source IDs "
            "and verify that the underlying records are disjoint."
        )

    errors = test["errors"]
    correctness_index = list(correctness_model.classes_).index(1)
    p_correct = correctness_model.predict_proba(
        test["features"]
    )[:, correctness_index]

    selectors = {
        "CRUS": test["crus"],
        "Maximum probability": 1.0 - test["max_probability"],
        "Predictive entropy": test["entropy"],
        "Probability margin": 1.0 - test["margin"],
        "Calibrated correctness": 1.0 - p_correct,
    }

    counts = {native_count}
    requested = {}
    for coverage in coverages:
        if not 0 < coverage <= 1:
            raise ValueError("Requested coverages must be in (0, 1].")
        count = max(1, int(np.floor(coverage * n + 1e-12)))
        counts.add(count)
        requested[str(coverage)] = count

    metrics_rows = []
    fixed_rows = []
    curve_rows = []
    audit = test["df"][
        ["record_id", "y_true", "y_pred", "crus"]
    ].copy()
    audit["error"] = errors

    for selector, uncertainty in selectors.items():
        order = deterministic_order(
            uncertainty, test["df"]["record_id"]
        )
        ordered_errors = errors[order]
        retained_counts = np.arange(1, n + 1)
        cumulative_errors = np.cumsum(ordered_errors)
        risk = cumulative_errors / retained_counts

        if np.unique(errors).size == 2:
            error_auroc = roc_auc_score(errors, uncertainty)
            error_ap = average_precision_score(errors, uncertainty)
            precision, recall, _ = precision_recall_curve(
                errors, uncertainty
            )
            error_pr_auc = auc(recall, precision)
        else:
            error_auroc = np.nan
            error_ap = np.nan
            error_pr_auc = np.nan

        native_errors = int(cumulative_errors[native_count - 1])

        # Optional exact check against the existing primary manuscript table.
        if expected_errors is not None and selector in expected_errors:
            expected = expected_errors[selector]
            if native_errors != expected:
                raise ValueError(
                    f"{selector}: expected {expected} errors at "
                    f"{native_count} retained records, got {native_errors}. "
                    "Check dataset, saved scores, and tie handling. "
                    "Do not change scores to force agreement."
                )

        metrics_rows.append({
            "selector": selector,
            "n": n,
            "full_coverage_errors": int(errors.sum()),
            "error_prevalence": float(errors.mean()),
            "AURC": float(risk.mean()),
            "error_AUROC": float(error_auroc),
            "error_AP": float(error_ap),
            "error_PR_AUC_trapezoidal": float(error_pr_auc),
            "native_retained": native_count,
            "native_coverage_pct": 100.0 * native_count / n,
            "native_errors": native_errors,
            "native_risk_pct": 100.0 * native_errors / native_count,
            "native_accuracy_pct": (
                100.0 * (1.0 - native_errors / native_count)
            ),
        })

        for count in sorted(counts):
            selected = order[:count]
            count_errors = int(errors[selected].sum())

            fixed_rows.append({
                "selector": selector,
                "retained": count,
                "coverage_pct": 100.0 * count / n,
                "errors": count_errors,
                "risk_pct": 100.0 * count_errors / count,
                "accuracy_pct": 100.0 * (1.0 - count_errors / count),
                "is_native_count": count == native_count,
                "boundary_tie_split": bool(
                    count < n and
                    uncertainty[order[count - 1]] ==
                    uncertainty[order[count]]
                ),
            })

        for count, count_errors, count_risk in zip(
            retained_counts, cumulative_errors, risk
        ):
            curve_rows.append({
                "selector": selector,
                "retained": int(count),
                "coverage": float(count / n),
                "errors": int(count_errors),
                "risk": float(count_risk),
            })

        audit[f"uncertainty__{selector}"] = uncertainty
        audit[f"retained_native__{selector}"] = False
        audit.loc[
            audit.index[order[:native_count]],
            f"retained_native__{selector}",
        ] = True

    metrics = pd.DataFrame(metrics_rows)
    fixed = pd.DataFrame(fixed_rows)
    curves = pd.DataFrame(curve_rows)

    calibration = {
        "test_correctness_Brier": float(
            brier_score_loss(1 - errors, p_correct)
        ),
        "test_correctness_log_loss": float(
            log_loss(
                1 - errors,
                np.column_stack([1 - p_correct, p_correct]),
                labels=[0, 1],
            )
        ),
    }

    metrics.to_csv(output_dir / "ranking_metrics.csv", index=False)
    fixed.to_csv(output_dir / "fixed_coverage_risk.csv", index=False)
    curves.to_csv(output_dir / "risk_coverage_points.csv", index=False)
    audit.to_csv(output_dir / "selector_case_audit.csv", index=False)

    table = metrics[[
        "selector",
        "AURC",
        "error_AUROC",
        "error_AP",
        "error_PR_AUC_trapezoidal",
        "native_risk_pct",
    ]].copy()

    table.columns = [
        "Selector",
        "AURC",
        "Error AUROC",
        "Error AP",
        "Error PR-AUC",
        "Matched risk (%)",
    ]

    caption = (
        f"Uncertainty-selector comparison on {n} records with "
        f"fixed Cascade predictions. Matched risk retains "
        f"{native_count} records for every selector "
        f"({100.0 * native_count / n:.2f}% coverage). "
        "Lower AURC and risk are better; higher AUROC, AP, "
        "and PR-AUC are better. Errors are the positive class."
    )

    table.to_latex(
        output_dir / "r17_uncertainty_comparison.tex",
        index=False,
        float_format=lambda value: f"{value:.4f}",
        caption=caption,
        label=f"tab:r17_{dataset_name}",
        escape=True,
    )

    fig, ax = plt.subplots(figsize=(7.5, 5))
    for selector in selectors:
        rows = curves[curves["selector"] == selector]
        ax.plot(
            100 * rows["coverage"],
            100 * rows["risk"],
            label=selector,
        )
    ax.axvline(
        100 * native_count / n,
        color="black",
        linestyle="--",
        linewidth=1,
        label="Primary matched count",
    )
    ax.set(
        xlabel="Coverage (%)",
        ylabel="Selective risk (%)",
        title=f"Fixed-prediction selector comparison: {dataset_name}",
        xlim=(0, 100),
    )
    ax.grid(alpha=0.25)
    ax.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(output_dir / "risk_coverage_selectors.png", dpi=300)
    plt.close(fig)

    metadata = {
        "dataset": dataset_name,
        "n": n,
        "native_count": native_count,
        "requested_coverage_to_count": requested,
        "probability_columns": validation["probability_columns"],
        "validation_hash": frame_hash(validation["df"]),
        "test_hash": frame_hash(test["df"]),
        "AURC_definition": "mean risk over retained counts 1 through N",
        "positive_class_for_detection": "incorrect prediction",
        "tie_rule": (
            "ascending uncertainty, then ascending numeric record ID "
            "when possible; otherwise lexicographic record ID"
        ),
        "correctness_features": [
            "maximum_probability",
            "normalized_predictive_entropy",
            "probability_margin",
        ],
        "test_calibration": calibration,
        "python": platform.python_version(),
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "sklearn": sklearn.__version__,
        "seed": SEED,
        "inference": (
            "descriptive evaluation; no statistical significance claim"
        ),
    }
    (output_dir / "evaluation_metadata.json").write_text(
        json.dumps(metadata, indent=2),
        encoding="utf-8",
    )

    return metrics, fixed, curves, audit


def run_r17(
    validation_df,
    primary_test_df,
    output_dir="R1_7_outputs",
    check_existing_table=True,
    external_test_df=None,
):
    validation = prepare_frame(validation_df)
    model, folds = fit_correctness_estimator(validation)

    expected = None
    if check_existing_table:
        expected = {
            "CRUS": 8,
            "Maximum probability": 11,
            "Predictive entropy": 11,
            "Probability margin": 10,
        }

    primary = evaluate_dataset(
        test_df=primary_test_df,
        validation=validation,
        correctness_model=model,
        output_dir=output_dir,
        dataset_name="primary",
        native_count=117,
        expected_n=257,
        expected_errors=expected,
    )

    output = {
        "primary": primary,
        "correctness_model": model,
        "calibration_folds": folds,
    }

    if external_test_df is not None:
        output["external"] = evaluate_dataset(
            test_df=external_test_df,
            validation=validation,
            correctness_model=model,
            output_dir=output_dir,
            dataset_name="external",
            native_count=162,
            expected_n=386,
            expected_errors=None,
        )

    return output

In [ ]:
from pathlib import Path
from IPython.display import display

required_inputs = ["validation_outputs", "primary_test_outputs"]

missing_inputs = [
    name for name in required_inputs
    if name not in globals()
]

if "run_r17" not in globals():
    raise RuntimeError(
        "Run Section 3 first to define run_r17() and its helper functions."
    )

if missing_inputs:
    print("Evaluation has NOT started.")
    print("Missing input DataFrames:", missing_inputs)
    print(
        "\nThese names were placeholders in my previous answer. "
        "We need to connect your notebook's actual validation/test outputs."
    )

else:
    print("Validation records:", len(validation_outputs))
    print("Primary-test records:", len(primary_test_outputs))
    print("Starting R1.7 evaluation...")

    r17_output_dir = Path("R1_7_outputs")

    results_r17 = run_r17(
        validation_df=validation_outputs,
        primary_test_df=primary_test_outputs,
        output_dir=r17_output_dir,
        check_existing_table=True,
    )

    primary_metrics, fixed_risk, curves, audit = results_r17["primary"]

    print("\nRanking metrics and matched-count results:")
    display(primary_metrics)

    print("\nRisk at the primary matched count:")
    display(
        fixed_risk.loc[fixed_risk["is_native_count"]]
        .reset_index(drop=True)
    )

    print("\nAll evaluated coverage levels:")
    display(fixed_risk)

    print("\nGenerated files:")
    generated_files = sorted(
        path for path in r17_output_dir.rglob("*")
        if path.is_file()
    )

    for path in generated_files:
        print(f"{path} — {path.stat().st_size:,} bytes")

    if not generated_files:
        raise RuntimeError("Evaluation returned but no output files were found.")

    print("\nR1.7 evaluation completed.")

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display

rows = []

for name, obj in list(globals().items()):
    if name.startswith("_"):
        continue

    if isinstance(obj, pd.DataFrame):
        rows.append({
            "variable": name,
            "type": "DataFrame",
            "shape_or_length": str(obj.shape),
            "columns_or_keys": ", ".join(map(str, obj.columns)),
        })

    elif isinstance(obj, pd.Series):
        rows.append({
            "variable": name,
            "type": "Series",
            "shape_or_length": str(obj.shape),
            "columns_or_keys": "",
        })

    elif isinstance(obj, np.ndarray):
        rows.append({
            "variable": name,
            "type": "ndarray",
            "shape_or_length": str(obj.shape),
            "columns_or_keys": "",
        })

    elif isinstance(obj, dict):
        rows.append({
            "variable": name,
            "type": "dict",
            "shape_or_length": str(len(obj)),
            "columns_or_keys": ", ".join(
                map(str, list(obj.keys())[:40])
            ),
        })

    elif isinstance(obj, (list, tuple)) and len(obj):
        rows.append({
            "variable": name,
            "type": type(obj).__name__,
            "shape_or_length": str(len(obj)),
            "columns_or_keys": "",
        })

if rows:
    inventory = pd.DataFrame(rows).sort_values(
        ["type", "variable"]
    ).reset_index(drop=True)

    with pd.option_context(
        "display.max_rows", None,
        "display.max_colwidth", 180,
    ):
        display(inventory)
else:
    print(
        "No candidate data objects found. "
        "The earlier pipeline cells may not have been run "
        "in this notebook session."
    )

In [ ]:
# Select the canonical notebook from this package.
from pathlib import Path
import json

expected_name = "CRUS_R1_5_External_Transfer.ipynb"
NOTEBOOK = PACKAGE_ROOT / "notebooks" / expected_name
if not NOTEBOOK.is_file():
    raise FileNotFoundError(NOTEBOOK)

nb = json.loads(NOTEBOOK.read_text(encoding="utf-8"))
if not isinstance(nb.get("cells"), list):
    raise ValueError("The selected file does not contain notebook cells.")

print("Selected notebook:", NOTEBOOK)
print("Notebook cells:", len(nb["cells"]))
print("Code cells:", sum(
    cell.get("cell_type") == "code" for cell in nb["cells"]
))


In [ ]:
import re

if "nb" not in globals():
    raise RuntimeError("Run the notebook-upload cell first.")

path_pattern = re.compile(
    r"""["']([^"'\n]+\.(?:csv|tsv|parquet|json|jsonl|yaml|yml|pkl|pickle|npz|npy|png|pdf))["']""",
    flags=re.IGNORECASE
)

operation_pattern = re.compile(
    r"read_csv|read_parquet|read_json|read_pickle|"
    r"to_csv|to_parquet|to_json|to_pickle|"
    r"json\.dump|pickle\.dump|joblib\.dump|joblib\.load|"
    r"np\.save|np\.load|savefig",
    flags=re.IGNORECASE
)

directory_pattern = re.compile(
    r"\b\w*(?:output_dir|results_dir|save_dir|base_dir|"
    r"artifact_dir|run_dir|root_dir)\w*\b\s*=",
    flags=re.IGNORECASE
)

sensitive_pattern = re.compile(
    r"api_key|access_token|password|secret|"
    r"hf_[A-Za-z0-9]{15,}|ghp_[A-Za-z0-9]{15,}|"
    r"github_pat_|sk-[A-Za-z0-9_-]{15,}",
    flags=re.IGNORECASE
)

found_paths = {}

print("A. REFERENCED ARTIFACT FILENAMES")
print("-" * 70)

for cell_number, cell in enumerate(nb["cells"], start=1):
    if cell.get("cell_type") != "code":
        continue

    source = cell.get("source", "")
    source = "".join(source) if isinstance(source, list) else source

    for line in source.splitlines():
        if sensitive_pattern.search(line):
            continue

        for match in path_pattern.finditer(line):
            filename = match.group(1)
            found_paths.setdefault(filename, set()).add(cell_number)

for filename, cells in sorted(found_paths.items()):
    cell_list = ", ".join(map(str, sorted(cells)))
    print(f"Cells {cell_list}: {filename}")

print("\nB. FILE OPERATIONS AND DIRECTORY DEFINITIONS")
print("-" * 70)

operation_count = 0

for cell_number, cell in enumerate(nb["cells"], start=1):
    if cell.get("cell_type") != "code":
        continue

    source = cell.get("source", "")
    source = "".join(source) if isinstance(source, list) else source

    for line_number, line in enumerate(source.splitlines(), start=1):
        if not (
            operation_pattern.search(line)
            or directory_pattern.search(line)
        ):
            continue

        if sensitive_pattern.search(line):
            print(
                f"Cell {cell_number}, line {line_number}: "
                "[Potentially sensitive line omitted]"
            )
            continue

        print(
            f"Cell {cell_number}, line {line_number}: "
            f"{line.strip()[:300]}"
        )
        operation_count += 1

print("\nUnique referenced artifact paths:", len(found_paths))
print("Displayed file-operation/directory lines:", operation_count)

In [ ]:
import re

if "nb" not in globals():
    raise RuntimeError("Run the notebook-upload cell first.")

directory_assignment = re.compile(
    r"^\s*(?:"
    r"PROJECT_DIR|BASE|ROOT|ROOT_DIR|OUTDIR|OUT|"
    r"OUTPUT_DIR|RESULTS_DIR|AUDIT_DIR|RUN_DIR|"
    r"DATA_DIR|EXPERIMENT_DIR"
    r")\s*=",
    flags=re.IGNORECASE
)

sensitive = re.compile(
    r"api_key|access_token|password|secret|"
    r"hf_[A-Za-z0-9]{15,}|ghp_[A-Za-z0-9]{15,}|"
    r"github_pat_|sk-[A-Za-z0-9_-]{15,}",
    flags=re.IGNORECASE
)

for cell_number, cell in enumerate(nb["cells"], start=1):
    if cell.get("cell_type") != "code":
        continue

    source = cell.get("source", "")
    source = "".join(source) if isinstance(source, list) else source
    lines = source.splitlines()

    for index, line in enumerate(lines):
        if directory_assignment.search(line):
            # Show a short block because some paths span several lines.
            block = "\n".join(lines[index:index + 4])

            if sensitive.search(block):
                print(f"\nCell {cell_number}: [sensitive block omitted]")
            else:
                print(f"\nCell {cell_number}:")
                print(block)

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path(_crus_local_path('project', ''))

if not PROJECT_ROOT.is_dir():
    pass  # Colab service replaced by local workflow.
    _crus_prepare_saved_workspace()

if not PROJECT_ROOT.is_dir():
    raise FileNotFoundError(
        f"Project folder not accessible: {PROJECT_ROOT}\n"
        "Check that the correct Google Drive account is mounted."
    )

print("Project folder accessible:", PROJECT_ROOT)

print("\nTop-level folders:")
for path in sorted(PROJECT_ROOT.iterdir()):
    if path.is_dir():
        print(path.name)

print("\nKey primary-analysis files:")
checks = [
    "data_audit/constrained_split_manifest.csv",
    "data_audit/constrained_split_metadata.json",
    "corrected_experiment_outputs/final_pretest_protocol.json",
    "corrected_experiment_outputs/validation_selected_policies.json",
    "corrected_experiment_outputs/test_all_method_predictions.csv",
    "corrected_experiment_outputs/test_frozen_CRUS_acceptance_audit.csv",
    "corrected_experiment_outputs/validation_retrieval_prediction_audit.csv",
]

for relative in checks:
    path = PROJECT_ROOT / relative
    status = "FOUND" if path.is_file() else "NOT FOUND"
    print(f"{status}: {relative}")

print("\nReferenced supporting notebook:")
supporting_notebook = Path(
    _crus_local_path('notebooks', 'revised_risk_aware_hybrid_pipeline_v2.ipynb')
)
print(
    "FOUND" if supporting_notebook.is_file() else "NOT FOUND",
    supporting_notebook.name
)

In [ ]:
from pathlib import Path
import csv
import json

ROOT = Path(_crus_local_path('project', ''))

if not ROOT.is_dir():
    raise FileNotFoundError("Project folder is not accessible.")

def inspect_csv(path):
    print(f"\nFILE: {path.relative_to(ROOT)}")

    if not path.is_file():
        print("STATUS: NOT FOUND")
        return

    try:
        with path.open(
            "r", encoding="utf-8-sig", newline=""
        ) as handle:
            reader = csv.reader(handle)
            header = next(reader, [])
            row_count = sum(
                1 for row in reader if any(value.strip() for value in row)
            )

        print("STATUS: FOUND")
        print("ROWS:", row_count)
        print("COLUMNS:", header)

    except Exception as error:
        print("INSPECTION ERROR:", type(error).__name__)

def inspect_json_keys(path):
    print(f"\nFILE: {path.relative_to(ROOT)}")

    if not path.is_file():
        print("STATUS: NOT FOUND")
        return

    try:
        payload = json.loads(path.read_text(encoding="utf-8"))
        print("STATUS: FOUND")

        if isinstance(payload, dict):
            print("TOP-LEVEL KEYS:", list(payload.keys()))
        else:
            print("JSON TYPE:", type(payload).__name__)

    except Exception as error:
        print("INSPECTION ERROR:", type(error).__name__)

print("A. PRIMARY SPLITS AND CASE-LEVEL OUTPUTS")

for relative in [
    "data_audit/constrained_split_manifest.csv",
    "corrected_experiment_outputs/constrained_calibration_manifest.csv",
    "corrected_experiment_outputs/test_all_method_predictions.csv",
    "corrected_experiment_outputs/test_frozen_CRUS_acceptance_audit.csv",
    "corrected_experiment_outputs/validation_retrieval_prediction_audit.csv",
    "corrected_experiment_outputs/test_matched_count_acceptance_audit.csv",
]:
    inspect_csv(ROOT / relative)

print("\n\nB. CONFIGURATION AND POLICY STRUCTURE")

for relative in [
    "data_audit/constrained_split_metadata.json",
    "data_audit/final_cleaning_grouping_policy.json",
    "corrected_experiment_outputs/final_pretest_protocol.json",
    "corrected_experiment_outputs/validation_selected_policies.json",
]:
    inspect_json_keys(ROOT / relative)

print("\n\nC. REPEATED-SPLIT FILE AVAILABILITY")

repeat_root = ROOT / "supplementary_repeated_splits"

seed_folders = (
    sorted(
        path for path in repeat_root.glob("seed_*")
        if path.is_dir()
    )
    if repeat_root.is_dir() else []
)

if not seed_folders:
    print("No seed_* folders found.")

for seed_folder in seed_folders:
    print(f"\nRUN: {seed_folder.name}")

    required = [
        seed_folder / "split_manifest.csv",
        seed_folder / "calibration_manifest.csv",
        seed_folder / "experiment_outputs/pretest_protocol.json",
        seed_folder / "experiment_outputs/test_metric_summary.csv",
        seed_folder / "experiment_outputs/native_CRUS_acceptance.csv",
        seed_folder / "experiment_outputs/RUN_COMPLETE.json",
    ]

    for path in required:
        status = "FOUND" if path.is_file() else "NOT FOUND"
        print(f"{status}: {path.relative_to(ROOT)}")

    arrays = sorted(
        (seed_folder / "experiment_outputs").glob("test_*_arrays.npz")
    )

    print("TEST ARRAY FILES:", [path.name for path in arrays])

print("\n\nD. SUPPORTING NOTEBOOK")

supporting = Path(
    _crus_local_path('notebooks', 'revised_risk_aware_hybrid_pipeline_v2.ipynb')
)

if supporting.is_file():
    supporting_nb = json.loads(
        supporting.read_text(encoding="utf-8")
    )
    print("FOUND:", supporting.name)
    print("CELLS:", len(supporting_nb.get("cells", [])))
else:
    print("NOT FOUND:", supporting.name)

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import csv
import hashlib
import json
import shutil

ROOT = Path(_crus_local_path('project', ''))

UPLOADED_NOTEBOOK = Path(
    "CRUS_R1_5_External_Transfer.ipynb"
).resolve()

SUPPORTING_NOTEBOOK = Path(
    _crus_local_path('notebooks', 'revised_risk_aware_hybrid_pipeline_v2.ipynb')
)

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
PACKAGE = Path(_crus_local_path('runtime', 'editor9_staging')) / f"crus_review_{stamp}"

if not ROOT.is_dir():
    raise FileNotFoundError(ROOT)

for notebook in [UPLOADED_NOTEBOOK, SUPPORTING_NOTEBOOK]:
    if not notebook.is_file():
        raise FileNotFoundError(f"Required notebook not found: {notebook}")

if PACKAGE.exists():
    raise FileExistsError(PACKAGE)

PACKAGE.mkdir(parents=True)
manifest_rows = []
missing = []

def digest(path):
    h = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def copy_artifact(relative, required=True):
    source = ROOT / relative
    destination = PACKAGE / "artifacts" / relative

    if not source.is_file():
        missing.append({
            "path": relative,
            "required": required
        })
        return

    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source, destination)

    manifest_rows.append({
        "package_path": destination.relative_to(PACKAGE).as_posix(),
        "source_relative_path": relative,
        "operation": "unchanged_copy",
        "bytes": destination.stat().st_size,
        "sha256": digest(destination)
    })

def clean_notebook(source, destination_name):
    original = json.loads(source.read_text(encoding="utf-8"))
    cells = []

    for cell in original.get("cells", []):
        kind = cell.get("cell_type")
        if kind not in {"code", "markdown", "raw"}:
            raise ValueError(f"Unexpected cell type in {source.name}")

        cleaned = {
            "cell_type": kind,
            "metadata": {},
            "source": cell.get("source", [])
        }

        if "id" in cell:
            cleaned["id"] = cell["id"]

        if kind == "code":
            cleaned["execution_count"] = None
            cleaned["outputs"] = []

        # Attachments are preserved and require manual review.
        if "attachments" in cell:
            cleaned["attachments"] = cell["attachments"]

        cells.append(cleaned)

    result = {
        "cells": cells,
        "metadata": {
            key: value
            for key, value in original.get("metadata", {}).items()
            if key in {"kernelspec", "language_info"}
        },
        "nbformat": original.get("nbformat", 4),
        "nbformat_minor": original.get("nbformat_minor", 5)
    }

    destination = PACKAGE / "notebooks" / destination_name
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_text(
        json.dumps(result, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )

    manifest_rows.append({
        "package_path": destination.relative_to(PACKAGE).as_posix(),
        "source_relative_path": source.name,
        "operation": "outputs_and_metadata_cleared",
        "bytes": destination.stat().st_size,
        "sha256": digest(destination)
    })

primary_required = [
    "data_audit/constrained_split_manifest.csv",
    "data_audit/constrained_split_metadata.json",
    "data_audit/final_cleaning_grouping_policy.json",
    "corrected_experiment_outputs/constrained_calibration_manifest.csv",
    "corrected_experiment_outputs/final_pretest_protocol.json",
    "corrected_experiment_outputs/validation_selected_policies.json",
    "corrected_experiment_outputs/test_all_method_predictions.csv",
    "corrected_experiment_outputs/test_frozen_CRUS_acceptance_audit.csv",
    "corrected_experiment_outputs/validation_retrieval_prediction_audit.csv",
    "corrected_experiment_outputs/test_matched_count_acceptance_audit.csv",
]

primary_optional = [
    "data_audit/exact_duplicate_removal_log.csv",
    "data_audit/near_duplicate_review_decisions.csv",
    "data_audit/retained_conflicting_label_groups.csv",
    "data_audit/rare_class_exclusion_log.csv",
    "data_audit/final_cleaned_class_counts.csv",
    "data_audit/constrained_class_distribution_per_split.csv",
    "corrected_experiment_outputs/constrained_calibration_metadata.json",
    "corrected_experiment_outputs/retrieval_initialization_metadata.json",
    "corrected_experiment_outputs/supervised_baseline_configuration.json",
    "corrected_experiment_outputs/verified_main_test_results.csv",
    "corrected_experiment_outputs/test_matched_count_results.csv",
    "corrected_experiment_outputs/test_ranking_risk_coverage_curves.csv",
    "corrected_experiment_outputs/test_ranking_risk_coverage_summary.csv",
    "corrected_experiment_outputs/test_native_gate_operating_points.csv",
    "corrected_experiment_outputs/comment4_audit/"
    "comment4_verified_matched_counts.csv",
    "corrected_experiment_outputs/comment4_audit/"
    "comment4_verified_risk_coverage_curves.csv",
    "corrected_experiment_outputs/R1_3_consistency_audit/figures/"
    "confusion_matrix_cascade_native_crus_counts.csv",
    "corrected_experiment_outputs/R1_3_consistency_audit/figures/"
    "threshold_sweep_validation_corrected.csv",
]

for relative in primary_required:
    copy_artifact(relative, required=True)

for relative in primary_optional:
    copy_artifact(relative, required=False)

for seed in [11, 22, 33, 44, 55]:
    base = f"supplementary_repeated_splits/seed_{seed}"

    for suffix in [
        "split_manifest.csv",
        "calibration_manifest.csv",
        "experiment_outputs/pretest_protocol.json",
        "experiment_outputs/test_metric_summary.csv",
        "experiment_outputs/native_CRUS_acceptance.csv",
        "experiment_outputs/RUN_COMPLETE.json",
    ]:
        copy_artifact(f"{base}/{suffix}", required=True)

    array_folder = ROOT / base / "experiment_outputs"

    for array_file in sorted(array_folder.glob("test_*_arrays.npz")):
        copy_artifact(
            array_file.relative_to(ROOT).as_posix(),
            required=True
        )

clean_notebook(
    UPLOADED_NOTEBOOK,
    "CRUS_R1_5_External_Transfer.ipynb"
)

clean_notebook(
    SUPPORTING_NOTEBOOK,
    "revised_risk_aware_hybrid_pipeline_v2.ipynb"
)

manifest_dir = PACKAGE / "manifest"
manifest_dir.mkdir()

with (manifest_dir / "artifact_manifest.csv").open(
    "w", newline="", encoding="utf-8"
) as handle:
    writer = csv.DictWriter(
        handle,
        fieldnames=[
            "package_path", "source_relative_path",
            "operation", "bytes", "sha256"
        ]
    )
    writer.writeheader()
    writer.writerows(manifest_rows)

status = {
    "status": "STAGING_ONLY_NOT_READY_FOR_RELEASE",
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "copied_files": len(manifest_rows),
    "missing_required": [
        item["path"] for item in missing if item["required"]
    ],
    "optional_paths_not_found": [
        item["path"] for item in missing if not item["required"]
    ],
    "experiments_executed": False,
    "privacy_review_completed": False,
    "array_contents_reviewed": False,
    "execution_instructions_verified": False,
    "metric_regeneration_test_completed": False,
    "additional_revision_experiments_packaged": False
}

(manifest_dir / "package_status.json").write_text(
    json.dumps(status, indent=2),
    encoding="utf-8"
)

readme = """# CRUS reproducibility review package

Status: staging package; not yet approved for release.

## Contents
- Output-cleared implementation notebooks.
- Primary split and calibration manifests.
- Existing configuration and policy files.
- Primary predictions and acceptance audits.
- Five repeated-split manifests and output artifacts.
- Artifact manifest and SHA-256 checksums.

Original experiment files were not modified.

## Required before release
1. Review every shared file for clinical text, credentials,
   personal identifiers, and machine-specific paths.
2. Inspect NPZ array keys, dtypes, shapes, and sharing suitability.
3. Document dataset version, source-row indexing, record-ID mapping,
   normalization, and group definitions.
4. Document installation requirements using historical run information.
5. Specify notebook execution order and supporting-code dependencies.
6. Explain path configuration; current notebook copies retain source paths.
7. Provide commands/cells to regenerate metrics, tables, and curves.
8. Verify regenerated values against the final manuscript.
9. Add artifacts for other reported revision experiments.
10. Select appropriate licensing and review data redistribution terms.

## Provenance
Existing configuration, limitation, and historical-exposure notes
are retained in the copied artifacts and must not be removed merely
to simplify presentation.

## Important scope limitation
This staging package is not yet a complete collection of all
additional analyses in the manuscript. Additional MTS evaluation,
domain baselines, uncertainty comparisons, sensitivity analyses,
calibration analyses, perturbations, and candidate-selection
experiments require separate selection and verification.

## Manuscript result mapping
TODO: Map each reported table/figure to input files and
regeneration commands.

## Access and version
TODO: Add repository/review URL, version tag, and commit identifier.
"""

(PACKAGE / "README.md").write_text(readme, encoding="utf-8")

(PACKAGE / ".gitignore").write_text(
    ".env\n.venv/\n__pycache__/\n.ipynb_checkpoints/\n",
    encoding="utf-8"
)

checksum_file = manifest_dir / "SHA256SUMS.txt"
checksum_lines = [
    f"{digest(path)}  {path.relative_to(PACKAGE).as_posix()}"
    for path in sorted(PACKAGE.rglob("*"))
    if path.is_file() and path != checksum_file
]

checksum_file.write_text(
    "\n".join(checksum_lines) + "\n",
    encoding="utf-8"
)

print("Staging package:", PACKAGE)
print("Packaged files:", len(manifest_rows))
print("Missing required files:", len(status["missing_required"]))

for relative in status["missing_required"]:
    print("REQUIRED — NOT FOUND:", relative)

print("\nOptional paths not found:",
      len(status["optional_paths_not_found"]))

for relative in status["optional_paths_not_found"]:
    print("OPTIONAL PATH — NOT FOUND:", relative)

print("\nSource files unchanged. No experiments executed.")
print("Not ready for GitHub release until review is complete.")

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

PACKAGE = Path(
    _crus_local_path('runtime', 'editor9_staging/crus_review_20261019T070629Z')
)
ARTIFACTS = PACKAGE / "artifacts"

if not ARTIFACTS.is_dir():
    raise FileNotFoundError("Staging artifacts folder not found.")

checks = []
array_report = []

def record(name, passed, details):
    checks.append({
        "check": name,
        "passed": bool(passed),
        "details": details
    })
    print(f"{'PASS' if passed else 'REVIEW'}: {name}")
    print(" ", details)

def read_csv(relative):
    return pd.read_csv(
        ARTIFACTS / relative,
        dtype={
            "record_id": "string",
            "group_id": "string"
        }
    )

def check_manifest(frame, name, expected_sizes=None):
    required = {"record_id", "group_id", "label", "split"}
    missing = required - set(frame.columns)

    record(
        f"{name}: required columns",
        not missing,
        {"missing_columns": sorted(missing)}
    )
    if missing:
        return

    record(
        f"{name}: record IDs unique and nonmissing",
        frame["record_id"].notna().all()
        and not frame["record_id"].duplicated().any(),
        {
            "rows": len(frame),
            "duplicate_record_ids": int(
                frame["record_id"].duplicated().sum()
            )
        }
    )

    missing_values = frame[list(required)].isna().sum().to_dict()
    record(
        f"{name}: manifest fields nonmissing",
        not any(missing_values.values()),
        {k: int(v) for k, v in missing_values.items()}
    )

    crossing = (
        frame.groupby("group_id")["split"].nunique() > 1
    ).sum()

    record(
        f"{name}: groups do not cross partitions",
        crossing == 0,
        {"cross_partition_groups": int(crossing)}
    )

    sizes = {
        str(key): int(value)
        for key, value in frame["split"].value_counts().items()
    }

    if expected_sizes is not None:
        record(
            f"{name}: expected split sizes",
            sizes == expected_sizes,
            {"observed": sizes, "expected": expected_sizes}
        )
    else:
        print(f"INFO: {name} split sizes: {sizes}")

# A. Primary manifest
manifest = read_csv("data_audit/constrained_split_manifest.csv")

check_manifest(
    manifest,
    "Primary manifest",
    {"train": 773, "validation": 258, "test": 257}
)

record(
    "Primary manifest: retained labels",
    manifest["label"].nunique() == 17,
    {"label_count": int(manifest["label"].nunique())}
)

test = manifest.loc[manifest["split"] == "test"]
test_ids = set(test["record_id"].dropna())

pred = read_csv(
    "corrected_experiment_outputs/test_all_method_predictions.csv"
)

record(
    "Primary predictions: unique record/method pairs",
    not pred.duplicated(["record_id", "method"]).any(),
    {
        "duplicate_pairs": int(
            pred.duplicated(["record_id", "method"]).sum()
        )
    }
)

for method, part in pred.groupby("method"):
    observed = set(part["record_id"].dropna())
    record(
        f"Primary predictions: test coverage for {method}",
        observed == test_ids and len(part) == len(test_ids),
        {
            "rows": len(part),
            "missing_test_ids": len(test_ids - observed),
            "unexpected_ids": len(observed - test_ids)
        }
    )

joined = pred.merge(
    test[["record_id", "label"]],
    on="record_id",
    how="left",
    validate="many_to_one"
)

agreement = (
    joined["true_label"].astype("string")
    == joined["label"].astype("string")
).fillna(False)

record(
    "Primary predictions: labels agree with manifest",
    agreement.all(),
    {"mismatched_or_unmapped_rows": int((~agreement).sum())}
)

# B. Primary acceptance audits
prediction_keys = pred[["record_id", "method"]].drop_duplicates()

for filename, key_columns in [
    (
        "test_frozen_CRUS_acceptance_audit.csv",
        ["record_id", "method"]
    ),
    (
        "test_matched_count_acceptance_audit.csv",
        ["record_id", "method", "target_n_accepted"]
    ),
]:
    audit = read_csv(f"corrected_experiment_outputs/{filename}")

    record(
        f"{filename}: unique policy keys",
        not audit.duplicated(key_columns).any(),
        {
            "duplicate_keys": int(
                audit.duplicated(key_columns).sum()
            )
        }
    )

    linked = audit.merge(
        prediction_keys.assign(_prediction_present=True),
        on=["record_id", "method"],
        how="left",
        validate="many_to_one"
    )

    missing_links = linked["_prediction_present"].isna().sum()

    record(
        f"{filename}: links to saved predictions",
        missing_links == 0,
        {"unlinked_rows": int(missing_links)}
    )

    accepted = audit["accepted"].astype("string").str.lower()
    valid_boolean = accepted.isin(["true", "false", "1", "0"])

    record(
        f"{filename}: acceptance values interpretable",
        valid_boolean.all(),
        {"invalid_or_missing_rows": int((~valid_boolean).sum())}
    )

# C. Repeated manifests
for seed in [11, 22, 33, 44, 55]:
    relative = (
        f"supplementary_repeated_splits/seed_{seed}/"
        "split_manifest.csv"
    )
    check_manifest(read_csv(relative), f"Repeated split seed {seed}")

# D. Array schemas: never enable pickle
print("\nARRAY SCHEMAS")

for path in sorted(ARTIFACTS.rglob("test_*_arrays.npz")):
    entry = {
        "file": path.relative_to(PACKAGE).as_posix(),
        "arrays": [],
        "archive_error": None
    }

    try:
        with np.load(path, allow_pickle=False) as saved:
            for key in saved.files:
                try:
                    array = saved[key]
                    entry["arrays"].append({
                        "key": key,
                        "shape": list(array.shape),
                        "dtype": str(array.dtype),
                        "status": "inspected_without_pickle"
                    })
                except ValueError:
                    entry["arrays"].append({
                        "key": key,
                        "status": "not_read_without_pickle_review_required"
                    })
    except Exception as error:
        entry["archive_error"] = type(error).__name__

    array_report.append(entry)

    print(entry["file"])
    for item in entry["arrays"]:
        print(" ", item)
    if entry["archive_error"]:
        print(" ARCHIVE ERROR:", entry["archive_error"])

report = {
    "scope": "Internal consistency and array-schema checks only",
    "checks": checks,
    "array_schemas": array_report,
    "privacy_review_completed": False,
    "metric_regeneration_verified": False
}

report_path = PACKAGE / "manifest" / "consistency_validation.json"
report_path.write_text(
    json.dumps(report, indent=2),
    encoding="utf-8"
)

print("\nSUMMARY")
print("Checks:", len(checks))
print("Passed:", sum(item["passed"] for item in checks))
print("Require review:", sum(not item["passed"] for item in checks))
print("Array archives inspected:", len(array_report))
print("Report saved:", report_path)
print("Checksums must be refreshed after all package changes.")

In [ ]:
from pathlib import Path
import csv
import json
import re

PACKAGE = Path(
    _crus_local_path('runtime', 'editor9_staging/crus_review_20261019T070629Z')
)

if not PACKAGE.is_dir():
    raise FileNotFoundError(PACKAGE)

patterns = {
    "possible_token": re.compile(
        r"\b(?:hf_[A-Za-z0-9]{15,}|ghp_[A-Za-z0-9]{15,}|"
        r"github_pat_[A-Za-z0-9_]{15,}|sk-[A-Za-z0-9_-]{15,})"
    ),
    "credential_assignment": re.compile(
        r"""(?i)\b(?:api_key|access_token|password|secret_key)\b"""
        r"""["']?\s*[:=]"""
    ),
    "absolute_local_path": re.compile(
        r"/content/|/home/|/Users/|[A-Za-z]:\\"
    ),
    "possible_email": re.compile(
        r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b"
    ),
}

text_column_pattern = re.compile(
    r"(?i)dialogue|dialog|query_text|patient_text|"
    r"clinical_note|reference_text|response_text|"
    r"utterance|transcript"
)

flags = []

def scan_text(text, relative, location):
    for line_number, line in enumerate(text.splitlines(), 1):
        for flag_name, pattern in patterns.items():
            if pattern.search(line):
                flags.append({
                    "file": relative,
                    "location": f"{location}:line_{line_number}",
                    "flag": flag_name
                })

for path in sorted(PACKAGE.rglob("*")):
    if not path.is_file():
        continue

    relative = path.relative_to(PACKAGE).as_posix()

    # Previous audit reports are not experimental content.
    if relative.startswith("manifest/"):
        continue

    if path.suffix.lower() == ".ipynb":
        notebook = json.loads(path.read_text(encoding="utf-8"))

        for number, cell in enumerate(notebook.get("cells", []), 1):
            source = cell.get("source", "")
            source = "".join(source) if isinstance(source, list) else source

            scan_text(source, relative, f"cell_{number}")

            if cell.get("attachments"):
                flags.append({
                    "file": relative,
                    "location": f"cell_{number}",
                    "flag": "embedded_attachment_requires_manual_review"
                })

    elif path.suffix.lower() in {
        ".csv", ".json", ".jsonl", ".yaml", ".yml",
        ".py", ".md", ".txt", ".toml"
    }:
        try:
            text = path.read_text(encoding="utf-8-sig")
            scan_text(text, relative, "file")

            if path.suffix.lower() == ".csv":
                with path.open(
                    encoding="utf-8-sig", newline=""
                ) as handle:
                    columns = next(csv.reader(handle), [])

                for column_number, column in enumerate(columns, 1):
                    if text_column_pattern.search(column):
                        flags.append({
                            "file": relative,
                            "location": f"column_{column_number}",
                            "flag": "possible_clinical_text_column"
                        })

        except UnicodeError:
            flags.append({
                "file": relative,
                "location": "file",
                "flag": "encoding_requires_manual_review"
            })

report = PACKAGE / "manifest" / "sharing_review_flags.csv"

with report.open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(
        handle, fieldnames=["file", "location", "flag"]
    )
    writer.writeheader()
    writer.writerows(flags)

from collections import Counter

counts = Counter(item["flag"] for item in flags)

print("Flag counts:")
for name, count in sorted(counts.items()):
    print(f"{name}: {count}")

print("\nFiles needing review:")
for name in sorted({item["file"] for item in flags}):
    print(name)

print("\nTotal flags:", len(flags))
print("Report:", report)
print("No experimental files changed.")

In [ ]:
from pathlib import Path
import csv
from collections import Counter

PACKAGE = Path(
    _crus_local_path('runtime', 'editor9_staging/crus_review_20261019T070629Z')
)

report = PACKAGE / "manifest" / "sharing_review_flags.csv"

if not report.is_file():
    raise FileNotFoundError(report)

with report.open(encoding="utf-8", newline="") as handle:
    flags = list(csv.DictReader(handle))

column_flags = [
    row for row in flags
    if row["flag"] == "possible_clinical_text_column"
]

print("A. FLAGGED CSV COLUMN NAMES")

for flag in column_flags:
    path = PACKAGE / flag["file"]
    column_number = int(flag["location"].split("_")[-1])

    with path.open(encoding="utf-8-sig", newline="") as handle:
        header = next(csv.reader(handle), [])

    column_name = (
        header[column_number - 1]
        if 0 < column_number <= len(header)
        else "[column index not found]"
    )

    print(f"\nFile: {flag['file']}")
    print(f"Column: {column_name}")

print("\nB. LOCAL-PATH FLAGS BY FILE")

path_counts = Counter(
    row["file"]
    for row in flags
    if row["flag"] == "absolute_local_path"
)

for filename, count in sorted(path_counts.items()):
    print(f"{count}: {filename}")

print("\nNo source files or staged artifacts changed.")

In [ ]:
from pathlib import Path
import json
import re

PACKAGE = Path(
    _crus_local_path('runtime', 'editor9_staging/crus_review_20261019T070629Z')
)
ARTIFACTS = PACKAGE / "artifacts"
ENVIRONMENT = PACKAGE / "environment"
ENVIRONMENT.mkdir(exist_ok=True)

primary_path = (
    ARTIFACTS / "corrected_experiment_outputs"
    / "final_pretest_protocol.json"
)

primary = json.loads(primary_path.read_text(encoding="utf-8"))
primary_versions = primary.get("package_versions")

if primary_versions is None:
    raise ValueError(
        "No package_versions field found in the primary protocol."
    )

# Preserve the saved information without rewriting the original protocol.
(ENVIRONMENT / "primary_recorded_package_versions.json").write_text(
    json.dumps(primary_versions, indent=2),
    encoding="utf-8"
)

comparison = []

for seed in [11, 22, 33, 44, 55]:
    path = (
        ARTIFACTS / "supplementary_repeated_splits"
        / f"seed_{seed}" / "experiment_outputs"
        / "pretest_protocol.json"
    )

    protocol = json.loads(path.read_text(encoding="utf-8"))
    versions = protocol.get("package_versions")

    comparison.append({
        "seed": seed,
        "package_versions_present": versions is not None,
        "matches_primary": (
            versions == primary_versions
            if versions is not None else None
        ),
        "recorded_package_versions": versions
    })

(ENVIRONMENT / "recorded_environment_comparison.json").write_text(
    json.dumps(comparison, indent=2),
    encoding="utf-8"
)

# Candidate only: package names and installation compatibility
# still need verification.
candidate = []
unresolved = []

if isinstance(primary_versions, dict):
    for name, version in primary_versions.items():
        name = str(name)
        version = str(version)

        valid_name = re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9._-]*", name)
        valid_version = re.fullmatch(
            r"[0-9][A-Za-z0-9.!+_-]*", version
        )

        if valid_name and valid_version:
            candidate.append(f"{name}=={version}")
        else:
            unresolved.append(name)
else:
    unresolved.append("package_versions is not a dictionary")

candidate_path = ENVIRONMENT / "requirements-recorded-candidate.txt"
candidate_path.write_text(
    "\n".join(sorted(candidate)) + ("\n" if candidate else ""),
    encoding="utf-8"
)

notes = """# Recorded environment information

The JSON files in this directory were extracted from saved experiment
protocols, not from the current packaging runtime.

requirements-recorded-candidate.txt is an untested candidate list.
It is not a complete dependency lockfile.

Before using it:
- Verify that entries are Python distribution names, not import aliases.
- Separate Python/runtime versions from pip-installable packages.
- Check missing dependencies and installation-specific packages.
- Document the required Python version and hardware/runtime details.
- Test installation and metric regeneration in a separate environment.
- Do not change frozen experimental artifacts to match a new environment.
"""

(ENVIRONMENT / "README.md").write_text(notes, encoding="utf-8")

print("PRIMARY RECORDED PACKAGE VERSIONS")
print(json.dumps(primary_versions, indent=2))

print("\nREPEATED-RUN ENVIRONMENT COMPARISON")
for item in comparison:
    print(
        f"seed_{item['seed']}: "
        f"versions_present={item['package_versions_present']}, "
        f"matches_primary={item['matches_primary']}"
    )

print("\nCandidate requirement entries:", len(candidate))
print("Entries requiring interpretation:", unresolved)
print("Frozen protocol files unchanged.")
print("Package checksums must be refreshed after final edits.")

In [ ]:
from pathlib import Path
import json
import re

PACKAGE = Path(
    _crus_local_path('runtime', 'editor9_staging/crus_review_20261019T070629Z')
)

# Inspect original saved artifacts without modifying them.
ROOT = Path(_crus_local_path('project', ''))
REPEATS = ROOT / "supplementary_repeated_splits"

key_pattern = re.compile(
    r"package.?versions?|environment|dependencies|"
    r"python.?version|pip.?freeze|cuda.?version|"
    r"torch.?version|software.?versions?",
    flags=re.IGNORECASE
)

findings = []

def walk_json(value, key_path=""):
    if isinstance(value, dict):
        for key, child in value.items():
            child_path = f"{key_path}.{key}" if key_path else str(key)

            if key_pattern.search(str(key)):
                yield child_path, type(child).__name__

            yield from walk_json(child, child_path)

    elif isinstance(value, list):
        for index, child in enumerate(value):
            yield from walk_json(child, f"{key_path}[{index}]")

print("A. ENVIRONMENT-RELATED JSON KEYS")

for seed in [11, 22, 33, 44, 55]:
    folder = REPEATS / f"seed_{seed}"
    print(f"\nseed_{seed}")

    seed_matches = 0

    for path in sorted(folder.rglob("*.json")):
        if not path.is_file() or path.stat().st_size > 5_000_000:
            continue

        try:
            payload = json.loads(path.read_text(encoding="utf-8"))
        except (OSError, UnicodeError, json.JSONDecodeError):
            continue

        for key_path, value_type in walk_json(payload):
            item = {
                "file": path.relative_to(ROOT).as_posix(),
                "key": key_path,
                "value_type": value_type
            }
            findings.append(item)
            print(
                f"{item['file']} | "
                f"{key_path} | {value_type}"
            )
            seed_matches += 1

    if seed_matches == 0:
        print("No matching JSON keys found.")

print("\nB. ENVIRONMENT/DEPENDENCY FILENAMES")

dependency_files = []

for path in sorted(REPEATS.rglob("*")):
    if not path.is_file():
        continue

    name = path.name.lower()

    if (
        name.startswith("requirements")
        or name.startswith("environment")
        or name in {
            "pip_freeze.txt", "pip-freeze.txt",
            "pyproject.toml", "poetry.lock",
            "conda-lock.yml", "uv.lock"
        }
    ):
        relative = path.relative_to(ROOT).as_posix()
        dependency_files.append(relative)
        print(relative)

if not dependency_files:
    print("No matching dependency files found.")

report = {
    "scope": "Environment-record discovery; values not verified",
    "json_key_matches": findings,
    "dependency_files": dependency_files
}

report_path = PACKAGE / "environment" / "environment_record_discovery.json"
report_path.write_text(json.dumps(report, indent=2), encoding="utf-8")

print("\nDiscovery report saved.")
print("No experimental files changed; no packages installed.")

In [ ]:
from pathlib import Path
import json
import shutil
import csv

ROOT = Path(_crus_local_path('project', ''))
PACKAGE = Path(
    _crus_local_path('runtime', 'editor9_staging/crus_review_20261019T070629Z')
)

primary_path = (
    PACKAGE / "artifacts/corrected_experiment_outputs/"
    "final_pretest_protocol.json"
)

primary = json.loads(primary_path.read_text(encoding="utf-8"))
primary_versions = primary.get("package_versions")

if not isinstance(primary_versions, dict):
    raise ValueError("Primary package_versions must be a dictionary.")

rows = []
saved_records = {}

for seed in [11, 22, 33, 44, 55]:
    relative = (
        f"supplementary_repeated_splits/seed_{seed}/"
        "experiment_outputs/retrieval_initialization_metadata.json"
    )

    source = ROOT / relative
    destination = PACKAGE / "artifacts" / relative

    if not source.is_file():
        raise FileNotFoundError(source)

    if destination.exists():
        if destination.read_bytes() != source.read_bytes():
            raise RuntimeError(
                f"Different staged file already exists: {relative}"
            )
    else:
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source, destination)

    metadata = json.loads(source.read_text(encoding="utf-8"))
    versions = metadata.get("package_versions")

    if not isinstance(versions, dict):
        raise ValueError(
            f"seed_{seed}: package_versions is not a dictionary."
        )

    saved_records[f"seed_{seed}"] = versions

    print(f"\nseed_{seed}: recorded package versions")
    print(json.dumps(versions, indent=2))

    for package_name in sorted(set(primary_versions) | set(versions)):
        primary_value = primary_versions.get(package_name)
        repeated_value = versions.get(package_name)

        if package_name not in versions:
            status = "NOT_RECORDED_IN_REPEAT"
        elif package_name not in primary_versions:
            status = "ADDITIONAL_REPEAT_RECORD"
        elif str(primary_value) == str(repeated_value):
            status = "MATCH"
        else:
            status = "DIFFERENT"

        rows.append({
            "seed": seed,
            "package": package_name,
            "primary_recorded_version": primary_value,
            "repeat_recorded_version": repeated_value,
            "status": status
        })

environment_dir = PACKAGE / "environment"
environment_dir.mkdir(exist_ok=True)

with (
    environment_dir / "primary_vs_repeated_recorded_versions.csv"
).open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(
        handle,
        fieldnames=[
            "seed", "package", "primary_recorded_version",
            "repeat_recorded_version", "status"
        ]
    )
    writer.writeheader()
    writer.writerows(rows)

(
    environment_dir / "repeated_recorded_package_versions.json"
).write_text(
    json.dumps(saved_records, indent=2),
    encoding="utf-8"
)

readme = environment_dir / "README.md"
note = """
## Repeated-run environment records

Additional package-version records were located in each seed's
experiment_outputs/retrieval_initialization_metadata.json and copied
unchanged into the package.

primary_vs_repeated_recorded_versions.csv compares these records with
the primary protocol. MATCH means the recorded version strings agree;
it does not establish a complete or tested environment specification.
MedCPT baseline environment records are separate and are not used to
infer the main experiment environment.
"""

existing = readme.read_text(encoding="utf-8") if readme.exists() else ""
if "## Repeated-run environment records" not in existing:
    readme.write_text(existing + "\n" + note, encoding="utf-8")

print("\nCOMPARISON SUMMARY")

for seed in [11, 22, 33, 44, 55]:
    seed_rows = [row for row in rows if row["seed"] == seed]
    counts = {}

    for row in seed_rows:
        counts[row["status"]] = counts.get(row["status"], 0) + 1

    print(f"seed_{seed}: {counts}")

print("\nFive metadata files copied or verified unchanged.")
print("No experiments executed; no packages installed.")
print("Artifact manifest and checksums require a final refresh.")

In [ ]:
from pathlib import Path
import subprocess
import sys

PACKAGE = Path(
    _crus_local_path('runtime', 'editor9_staging/crus_review_20261019T070629Z')
)

scripts_dir = PACKAGE / "scripts"
scripts_dir.mkdir(exist_ok=True)

script_path = scripts_dir / "regenerate_primary_metrics.py"

script = r'''
from pathlib import Path
import argparse
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score

parser = argparse.ArgumentParser()
parser.add_argument(
    "--package-root",
    type=Path,
    default=Path(__file__).resolve().parents[1]
)
args = parser.parse_args()

root = args.package_root.resolve()
artifacts = root / "artifacts"
results = root / "regenerated"
results.mkdir(exist_ok=True)

def read(name):
    return pd.read_csv(
        artifacts / "corrected_experiment_outputs" / name,
        dtype={"record_id": "string"}
    )

def booleans(series, field):
    values = series.astype("string").str.strip().str.lower()
    mapping = {"true": True, "false": False, "1": True, "0": False}
    if not values.isin(mapping).all():
        raise ValueError(f"Invalid or missing Boolean values in {field}")
    return values.map(mapping).astype(bool)

pred = read("test_all_method_predictions.csv")
manifest = pd.read_csv(
    artifacts / "data_audit/constrained_split_manifest.csv",
    dtype={"record_id": "string"}
)

if pred.duplicated(["record_id", "method"]).any():
    raise ValueError("Duplicate prediction keys.")

if pred[["record_id", "method", "true_label", "prediction"]].isna().any().any():
    raise ValueError("Missing prediction identifiers or labels.")

classes = sorted(manifest["label"].dropna().unique())
test_ids = set(
    manifest.loc[manifest["split"] == "test", "record_id"]
)

pred["computed_correct"] = pred["true_label"].eq(pred["prediction"])

if not (
    booleans(pred["correct"], "prediction correct")
    == pred["computed_correct"]
).all():
    raise ValueError("Saved correctness differs from label comparison.")

full_rows = []
for method, part in pred.groupby("method"):
    if set(part["record_id"]) != test_ids or len(part) != len(test_ids):
        raise ValueError(f"Incomplete or unexpected test coverage: {method}")

    full_rows.append({
        "method": method,
        "n_test": len(part),
        "n_correct": int(part["computed_correct"].sum()),
        "accuracy": accuracy_score(part["true_label"], part["prediction"]),
        "macro_f1_all_retained_labels": f1_score(
            part["true_label"],
            part["prediction"],
            labels=classes,
            average="macro",
            zero_division=0
        )
    })

pd.DataFrame(full_rows).to_csv(
    results / "primary_full_coverage_metrics.csv", index=False
)

reference = pred[
    ["record_id", "method", "computed_correct"]
]

def evaluate_audit(filename, policy_columns, output_name):
    audit = read(filename)
    keys = ["record_id", "method"] + policy_columns

    if audit.duplicated(keys).any():
        raise ValueError(f"Duplicate policy keys: {filename}")

    audit["accepted_bool"] = booleans(audit["accepted"], "accepted")

    joined = audit.merge(
        reference,
        on=["record_id", "method"],
        how="left",
        validate="many_to_one",
        indicator=True
    )

    if not joined["_merge"].eq("both").all():
        raise ValueError(f"Unlinked prediction rows: {filename}")

    if "correct" in joined:
        if not (
            booleans(joined["correct"], "audit correct")
            == joined["computed_correct"]
        ).all():
            raise ValueError(f"Correctness mismatch: {filename}")

    group_columns = ["method"] + policy_columns
    rows = []

    for key, part in joined.groupby(group_columns, dropna=False):
        key = key if isinstance(key, tuple) else (key,)
        identifiers = dict(zip(group_columns, key))

        if set(part["record_id"]) != test_ids:
            raise ValueError(f"Incomplete audit coverage: {identifiers}")

        selected = part.loc[part["accepted_bool"]]
        n = len(part)
        accepted = len(selected)
        correct = int(selected["computed_correct"].sum())

        row = {
            **identifiers,
            "n_test": n,
            "n_accepted": accepted,
            "n_correct_accepted": correct,
            "coverage": accepted / n,
            "accepted_accuracy": correct / accepted if accepted else np.nan,
            "selective_risk": (
                1 - correct / accepted if accepted else np.nan
            )
        }

        if "target_n_accepted" in identifiers:
            if accepted != int(identifiers["target_n_accepted"]):
                raise ValueError(
                    f"Accepted count differs from target: {identifiers}"
                )

        rows.append(row)

    pd.DataFrame(rows).to_csv(results / output_name, index=False)

evaluate_audit(
    "test_frozen_CRUS_acceptance_audit.csv",
    [],
    "primary_native_acceptance_metrics.csv"
)

evaluate_audit(
    "test_matched_count_acceptance_audit.csv",
    ["target_n_accepted"],
    "primary_matched_count_metrics.csv"
)

print("Regenerated files:")
for name in [
    "primary_full_coverage_metrics.csv",
    "primary_native_acceptance_metrics.csv",
    "primary_matched_count_metrics.csv"
]:
    print(name)

print("Saved correctness flags verified against labels and predictions.")
print("No model inference executed.")
print("Agreement with manuscript/saved summary values remains to be checked.")
'''

script_path.write_text(script.lstrip(), encoding="utf-8")

completed = subprocess.run(
    [
        sys.executable,
        str(script_path),
        "--package-root",
        str(PACKAGE)
    ],
    capture_output=True,
    text=True
)

print(completed.stdout)

if completed.returncode != 0:
    print(completed.stderr)
    raise RuntimeError(
        "Regeneration failed. Inspect the error; do not alter source "
        "artifacts simply to force the check to pass."
    )

print("\nREGENERATED FULL-COVERAGE METRICS")
import pandas as pd

print(
    pd.read_csv(
        PACKAGE / "regenerated/primary_full_coverage_metrics.csv"
    ).to_string(index=False)
)

print("\nREGENERATED NATIVE ACCEPTANCE METRICS")
print(
    pd.read_csv(
        PACKAGE / "regenerated/primary_native_acceptance_metrics.csv"
    ).to_string(index=False)
)

In [ ]:
from pathlib import Path
import pandas as pd

PACKAGE = Path(
    _crus_local_path('runtime', 'editor9_staging/crus_review_20261019T070629Z')
)

RESULTS = (
    PACKAGE / "artifacts" / "corrected_experiment_outputs"
)

filenames = [
    "verified_main_test_results.csv",
    "test_matched_count_results.csv",
    "comment4_audit/comment4_verified_matched_counts.csv",
]

for filename in filenames:
    path = RESULTS / filename

    print("\n" + "=" * 80)
    print("SAVED SUMMARY:", filename)

    if not path.is_file():
        print("NOT FOUND")
        continue

    table = pd.read_csv(path)

    print("Columns:", table.columns.tolist())
    print("Rows:", len(table))
    print(table.to_string(index=False))

print("\n" + "=" * 80)
print("REGENERATED MATCHED-COUNT METRICS")

regenerated = (
    PACKAGE / "regenerated" / "primary_matched_count_metrics.csv"
)

print(pd.read_csv(regenerated).to_string(index=False))

In [ ]:
from pathlib import Path
import subprocess
import sys

PACKAGE = Path(
    _crus_local_path('runtime', 'editor9_staging/crus_review_20261019T070629Z')
)

script_path = PACKAGE / "scripts" / "verify_matched_rankings.py"
script_path.parent.mkdir(exist_ok=True)

script = r'''
from pathlib import Path
import argparse
import numpy as np
import pandas as pd

parser = argparse.ArgumentParser()
parser.add_argument(
    "--package-root",
    type=Path,
    default=Path(__file__).resolve().parents[1]
)
args = parser.parse_args()

root = args.package_root.resolve()
base = root / "artifacts/corrected_experiment_outputs"
output = root / "regenerated"
output.mkdir(exist_ok=True)

pred = pd.read_csv(
    base / "test_all_method_predictions.csv",
    dtype={"record_id": "string"}
)
saved = pd.read_csv(
    base / "comment4_audit/comment4_verified_matched_counts.csv"
)

keys = ["method", "target_n_accepted"]

if pred.duplicated(["record_id", "method"]).any():
    raise ValueError("Duplicate prediction keys.")

if saved.duplicated(keys).any():
    raise ValueError("Duplicate saved matched-count keys.")

required = ["record_id", "method", "true_label", "prediction", "ranking_score"]

if pred[required].isna().any().any():
    raise ValueError("Missing identifiers, labels, or ranking scores.")

pred["ranking_score"] = pd.to_numeric(
    pred["ranking_score"], errors="raise"
)

if not np.isfinite(pred["ranking_score"]).all():
    raise ValueError("Nonfinite ranking scores.")

if not set(saved["method"]).issubset(set(pred["method"])):
    raise ValueError("Saved table contains an unknown method.")

rows = []

for method, targets in saved.groupby("method"):
    part = pred.loc[pred["method"] == method].copy()

    ranked = part.sort_values(
        ["ranking_score", "record_id"],
        ascending=[False, True],
        kind="mergesort"
    )

    for target_value in sorted(targets["target_n_accepted"].unique()):
        k = int(target_value)

        if k != target_value or not 0 < k <= len(ranked):
            raise ValueError(f"Invalid target: {method}, {target_value}")

        selected = ranked.head(k)
        n = len(ranked)
        correct = int(
            selected["true_label"].eq(selected["prediction"]).sum()
        )

        rows.append({
            "method": method,
            "target_n_accepted": k,
            "N": n,
            "n_accepted": k,
            "n_correct_accepted": correct,
            "n_errors_accepted": k - correct,
            "n_abstained": n - k,
            "coverage": k / n,
            "selective_accuracy": correct / k,
            "accepted_misclassification_rate": (k - correct) / k,
            "forced_accuracy": float(
                ranked["true_label"].eq(ranked["prediction"]).mean()
            ),
            "abstention_penalized_accuracy": correct / n,
            "abstention_penalized_error": 1 - correct / n
        })

regenerated = pd.DataFrame(rows)

regenerated.to_csv(
    output / "matched_rankings_all_targets.csv", index=False
)

comparison = saved.merge(
    regenerated,
    on=keys,
    how="outer",
    suffixes=("_saved", "_regenerated"),
    validate="one_to_one",
    indicator=True
)

if not comparison["_merge"].eq("both").all():
    raise ValueError("Saved and regenerated result keys do not match.")

integer_fields = [
    "N", "n_accepted", "n_correct_accepted",
    "n_errors_accepted", "n_abstained"
]
float_fields = [
    "coverage", "selective_accuracy",
    "accepted_misclassification_rate", "forced_accuracy",
    "abstention_penalized_accuracy", "abstention_penalized_error"
]

checks = []

for field in integer_fields + float_fields:
    a = comparison[f"{field}_saved"]
    b = comparison[f"{field}_regenerated"]

    matches = (
        a.eq(b).to_numpy()
        if field in integer_fields
        else np.isclose(
            a.to_numpy(dtype=float),
            b.to_numpy(dtype=float),
            rtol=0,
            atol=1e-10,
            equal_nan=False
        )
    )

    checks.append({
        "metric": field,
        "rows_compared": len(comparison),
        "mismatched_rows": int((~matches).sum()),
        "max_absolute_difference": float(
            np.max(np.abs(a.to_numpy(dtype=float)
                          - b.to_numpy(dtype=float)))
        )
    })

report = pd.DataFrame(checks)
report.to_csv(
    output / "matched_rankings_agreement_report.csv",
    index=False
)

print("Regenerated rows:", len(regenerated))
print(
    "Targets:",
    sorted(regenerated["target_n_accepted"].unique().tolist())
)
print(report.to_string(index=False))

if report["mismatched_rows"].sum():
    raise RuntimeError(
        "Differences detected. Inspect them without altering source artifacts."
    )

print("PASS: All compared counts and metrics agree within stated tolerance.")
print("Boundary-tie diagnostics and calibration metrics were not compared.")
'''

script_path.write_text(script.lstrip(), encoding="utf-8")

completed = subprocess.run(
    [
        sys.executable,
        str(script_path),
        "--package-root",
        str(PACKAGE)
    ],
    capture_output=True,
    text=True
)

print(completed.stdout)

if completed.returncode != 0:
    print(completed.stderr)
    raise RuntimeError("Matched-ranking verification requires review.")

In [ ]:
from pathlib import Path
import json
import hashlib

PACKAGE = Path(
    _crus_local_path('runtime', 'editor9_staging/crus_review_20261019T070629Z')
)

if not PACKAGE.is_dir():
    raise FileNotFoundError(PACKAGE)

manifest_dir = PACKAGE / "manifest"
manifest_dir.mkdir(exist_ok=True)

agreement_path = (
    PACKAGE / "regenerated"
    / "matched_rankings_agreement_report.csv"
)

if not agreement_path.is_file():
    raise FileNotFoundError(agreement_path)

import pandas as pd

agreement = pd.read_csv(agreement_path)

if len(agreement) != 11:
    raise ValueError("Unexpected agreement-report metric count.")

if not agreement["rows_compared"].eq(45).all():
    raise ValueError("Agreement report does not cover 45 rows.")

if not agreement["mismatched_rows"].eq(0).all():
    raise ValueError("Agreement report contains mismatches.")

guide = """# Reproduction from saved predictions

Run commands from the repository root.

## Required analysis packages

These scripts use Python, NumPy, pandas, and scikit-learn.
The primary saved protocol records:
- numpy: 2.1.3
- pandas: 2.2.3
- scikit-learn: 1.6.1

The current packaging session successfully ran the scripts.
A clean-environment installation test has not yet been completed.

## Commands

python scripts/regenerate_primary_metrics.py --package-root .
python scripts/verify_matched_rankings.py --package-root .

## Outputs

The regenerated/ directory contains:
- primary_full_coverage_metrics.csv
- primary_native_acceptance_metrics.csv
- primary_matched_count_metrics.csv
- matched_rankings_all_targets.csv
- matched_rankings_agreement_report.csv

## Verified scope

The matched-ranking script reconstructs selections from saved
ranking scores, sorting by ranking_score descending and record_id
ascending for ties.

It regenerated 45 method/target rows at accepted-case targets
113, 115, 117, 118, and 171.

Eleven count/metric fields agreed with the saved verified summary.
Integer counts matched exactly. Floating-point fields were checked
with absolute tolerance 1e-10 and relative tolerance zero.

The primary regeneration script also checked saved correctness flags
against labels and predictions and reconstructed full-coverage and
native-acceptance metrics.

## Not yet verified

- Automated agreement of all full-coverage/native metrics with summaries.
- Calibration metrics.
- Saved boundary-tie diagnostic fields.
- All manuscript figures.
- All repeated-run metrics.
- Additional revision experiments.
- End-to-end model inference in a clean environment.

## Policy distinction

Native CRUS gating and ranking-based top-k acceptance are different
selection rules. Equal accepted-case counts do not imply identical
selected records.

## Environment provenance

The primary protocol records eight package versions.
Six recorded versions match in each repeated-run metadata file.
The repeated-run metadata do not record pandas or scipy versions.
These are partial environment records, not a complete lockfile.
"""

(PACKAGE / "REPRODUCING_RESULTS.md").write_text(
    guide, encoding="utf-8"
)

status_path = manifest_dir / "package_status.json"
status = (
    json.loads(status_path.read_text(encoding="utf-8"))
    if status_path.exists() else {}
)

status.update({
    "status": "STAGING_PARTIALLY_VERIFIED",
    "matched_ranking_verification": {
        "rows": 45,
        "metrics": 11,
        "mismatched_values": 0,
        "integer_comparison": "exact",
        "float_absolute_tolerance": 1e-10,
        "float_relative_tolerance": 0,
        "report": "regenerated/matched_rankings_agreement_report.csv"
    },
    "privacy_review_completed": False,
    "execution_instructions_verified": False,
    "metric_regeneration_test_completed": False,
    "metric_regeneration_note": (
        "Specified primary metrics regenerated; matched-ranking "
        "agreement verified. Complete manuscript reproduction pending."
    ),
    "additional_revision_experiments_packaged": False
})

status_path.write_text(
    json.dumps(status, indent=2),
    encoding="utf-8"
)

def sha256(path):
    value = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            value.update(chunk)
    return value.hexdigest()

inventory_path = manifest_dir / "current_file_inventory.json"
checksum_path = manifest_dir / "SHA256SUMS.txt"

# Exclude generated inventory/checksum files from the inventory itself.
inventory = [
    {
        "path": path.relative_to(PACKAGE).as_posix(),
        "bytes": path.stat().st_size,
        "sha256": sha256(path)
    }
    for path in sorted(PACKAGE.rglob("*"))
    if path.is_file()
    and path not in {inventory_path, checksum_path}
]

inventory_path.write_text(
    json.dumps(inventory, indent=2),
    encoding="utf-8"
)

checksum_rows = [
    f"{sha256(path)}  {path.relative_to(PACKAGE).as_posix()}"
    for path in sorted(PACKAGE.rglob("*"))
    if path.is_file() and path != checksum_path
]

checksum_path.write_text(
    "\n".join(checksum_rows) + "\n",
    encoding="utf-8"
)

print("Verification documentation updated.")
print("Current inventory entries:", len(inventory))
print("Checksummed files:", len(checksum_rows))
print("Package status: STAGING_PARTIALLY_VERIFIED")
print("Experimental artifacts unchanged.")

In [ ]:
from pathlib import Path
import csv

ROOT = Path(_crus_local_path('project', ''))
PACKAGE = Path(
    _crus_local_path('runtime', 'editor9_staging/crus_review_20261019T070629Z')
)

folder_names = [
    "r1_5_additional_mts_exploratory_20261017T101158614697Z",
    "r1_7_uncertainty_comparison_20261016T103838081197Z",
    "r1_8_validation_weight_search_20261016T104727378070Z",
    "r1_9_constrained_validation_audit_20261016T120653563941Z",
    "r1_10_validation_sensitivity_20261016T125818056792Z",
    "r1_11_validation_perturbations_20261016T140605757621Z",
    "r1_12_calibration_20261016T142502223934Z",
    "r1_13_class_deferral_20261016T143807313869Z",
    "r1_15_candidate_selection_20261016T150259333685Z",
]

allowed_extensions = {".csv", ".json", ".jsonl", ".npz"}
rows = []

for folder_name in folder_names:
    folder = ROOT / folder_name
    print(f"\nFOLDER: {folder_name}")

    if not folder.is_dir():
        print("NOT FOUND")
        continue

    for path in sorted(folder.rglob("*")):
        if not path.is_file():
            continue
        if path.suffix.lower() not in allowed_extensions:
            continue

        relative = path.relative_to(ROOT).as_posix()

        rows.append({
            "analysis_folder": folder_name,
            "source_relative_path": relative,
            "bytes": path.stat().st_size,
            "release_selection": "NOT_YET_SELECTED"
        })

        print(
            f"{path.relative_to(folder).as_posix()} "
            f"[{path.stat().st_size:,} bytes]"
        )

report = (
    PACKAGE / "manifest"
    / "additional_analysis_candidates.csv"
)

with report.open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(
        handle,
        fieldnames=[
            "analysis_folder",
            "source_relative_path",
            "bytes",
            "release_selection"
        ]
    )
    writer.writeheader()
    writer.writerows(rows)

print("\nCandidate files:", len(rows))
print("Candidate inventory saved:", report)
print("No experimental artifacts copied or modified.")
print("Refresh checksums after final package changes.")

In [ ]:
from pathlib import Path
import csv
import json
import numpy as np

ROOT = Path(_crus_local_path('project', ''))
PACKAGE = Path(
    _crus_local_path('runtime', 'editor9_staging/crus_review_20261019T070629Z')
)

inventory_path = (
    PACKAGE / "manifest/additional_analysis_candidates.csv"
)

with inventory_path.open(encoding="utf-8", newline="") as handle:
    candidates = list(csv.DictReader(handle))

schemas = []

for candidate in candidates:
    relative = candidate["source_relative_path"]
    path = ROOT / relative

    entry = {
        "source_relative_path": relative,
        "extension": path.suffix.lower(),
        "schema": None,
        "error": None
    }

    try:
        if path.suffix.lower() == ".csv":
            with path.open(
                encoding="utf-8-sig", newline=""
            ) as handle:
                entry["schema"] = {
                    "columns": next(csv.reader(handle), [])
                }

        elif path.suffix.lower() == ".json":
            payload = json.loads(path.read_text(encoding="utf-8"))
            entry["schema"] = {
                "type": type(payload).__name__,
                "top_level_keys": (
                    list(payload.keys())
                    if isinstance(payload, dict) else None
                )
            }

        elif path.suffix.lower() == ".jsonl":
            with path.open(encoding="utf-8") as handle:
                first_line = next(
                    (line for line in handle if line.strip()), None
                )

            payload = json.loads(first_line) if first_line else None
            entry["schema"] = {
                "first_record_type": type(payload).__name__,
                "first_record_keys": (
                    list(payload.keys())
                    if isinstance(payload, dict) else None
                ),
                "scope": "First nonempty record only"
            }

        elif path.suffix.lower() == ".npz":
            arrays = []

            with np.load(path, allow_pickle=False) as archive:
                for key in archive.files:
                    try:
                        value = archive[key]
                        arrays.append({
                            "key": key,
                            "shape": list(value.shape),
                            "dtype": str(value.dtype)
                        })
                    except ValueError:
                        arrays.append({
                            "key": key,
                            "status": "requires_review_without_pickle"
                        })

            entry["schema"] = {"arrays": arrays}

    except Exception as error:
        entry["error"] = type(error).__name__

    schemas.append(entry)

report_path = PACKAGE / "manifest/additional_analysis_schemas.json"
report_path.write_text(
    json.dumps(schemas, indent=2),
    encoding="utf-8"
)

# Group identical schemas to avoid printing 103 repetitive entries.
groups = {}

for entry in schemas:
    signature = json.dumps(
        {
            "extension": entry["extension"],
            "schema": entry["schema"],
            "error": entry["error"]
        },
        sort_keys=True
    )
    groups.setdefault(signature, []).append(
        entry["source_relative_path"]
    )

for number, (signature, filenames) in enumerate(groups.items(), 1):
    print(f"\nSCHEMA GROUP {number}")
    print(json.dumps(json.loads(signature), indent=2))
    print("FILES:")
    for filename in filenames:
        print(" -", filename)

print("\nFiles inspected:", len(schemas))
print("Distinct schema groups:", len(groups))
print("Inspection errors:", sum(item["error"] is not None for item in schemas))
print("Schema report saved:", report_path)
print("No candidate artifacts copied or modified.")

In [ ]:
from pathlib import Path
import csv
import hashlib
import json
import shutil

ROOT = Path(_crus_local_path('project', ''))
PACKAGE = Path(
    _crus_local_path('runtime', 'editor9_staging/crus_review_20261019T070629Z')
)

inventory_path = (
    PACKAGE / "manifest/additional_analysis_candidates.csv"
)

with inventory_path.open(encoding="utf-8", newline="") as handle:
    candidates = list(csv.DictReader(handle))

jsonl_allowed_keys = {
    "record_id", "query_sha256", "pred_label", "probabilities",
    "confidence", "h_pred", "h_retrieval", "v_rrf", "crus",
    "accepted", "candidate_ids", "candidate_rrf_scores",
    "shortlist", "fallback"
}

log = []

def sha256(path):
    h = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def copy_unchanged(source, destination):
    destination.parent.mkdir(parents=True, exist_ok=True)

    if destination.exists():
        if sha256(source) != sha256(destination):
            raise RuntimeError(
                f"Different staged file already exists: {destination.name}"
            )
    else:
        shutil.copy2(source, destination)

for candidate in candidates:
    relative = candidate["source_relative_path"]
    source = ROOT / relative
    destination = PACKAGE / "artifacts" / relative

    entry = {
        "source_relative_path": relative,
        "source_sha256": sha256(source),
        "package_path": None,
        "operation": None,
        "removed_columns": [],
        "output_sha256": None
    }

    if source.name == "inference_inputs.csv":
        entry["operation"] = "excluded_pending_text_sharing_review"
        log.append(entry)
        continue

    if source.suffix.lower() == ".csv":
        with source.open(
            encoding="utf-8-sig", newline=""
        ) as handle:
            reader = csv.reader(handle)
            header = next(reader, [])

        removed = [
            column for column in header
            if column in {"dialogue", "perturbed_text"}
        ]

        if removed:
            destination = destination.with_name(
                destination.stem + "__text_excluded.csv"
            )

            if destination.exists():
                raise FileExistsError(
                    f"Derivative already exists: {destination}"
                )

            destination.parent.mkdir(parents=True, exist_ok=True)

            keep_indices = [
                index for index, column in enumerate(header)
                if column not in removed
            ]

            with source.open(
                encoding="utf-8-sig", newline=""
            ) as src, destination.open(
                "w", encoding="utf-8", newline=""
            ) as dst:
                reader = csv.reader(src)
                writer = csv.writer(dst)

                for row in reader:
                    if row and len(row) != len(header):
                        raise ValueError(
                            f"Unexpected CSV row width: {relative}"
                        )
                    writer.writerow(
                        [row[index] for index in keep_indices]
                        if row else []
                    )

            entry["operation"] = "derived_csv_text_columns_excluded"
            entry["removed_columns"] = removed
        else:
            copy_unchanged(source, destination)
            entry["operation"] = "unchanged_copy"

    elif source.suffix.lower() == ".jsonl":
        records = 0

        with source.open(encoding="utf-8") as handle:
            for number, line in enumerate(handle, 1):
                if not line.strip():
                    continue

                record = json.loads(line)

                if not isinstance(record, dict):
                    raise ValueError(
                        f"Non-dictionary JSONL record: {relative}, line {number}"
                    )

                unexpected = set(record) - jsonl_allowed_keys

                if unexpected:
                    raise ValueError(
                        f"Unexpected JSONL fields at line {number}: "
                        f"{sorted(unexpected)}"
                    )

                records += 1

        copy_unchanged(source, destination)
        entry["operation"] = "unchanged_copy_all_record_keys_checked"
        entry["jsonl_records_checked"] = records

    elif source.suffix.lower() in {".json", ".npz"}:
        copy_unchanged(source, destination)
        entry["operation"] = "unchanged_copy"

    else:
        raise ValueError(f"Unreviewed file extension: {relative}")

    entry["package_path"] = destination.relative_to(PACKAGE).as_posix()
    entry["output_sha256"] = sha256(destination)
    log.append(entry)

log_path = PACKAGE / "manifest/additional_artifact_export_log.json"
log_path.write_text(
    json.dumps(log, indent=2),
    encoding="utf-8"
)

note = """# Additional-analysis artifact exports

Files were selected from additional_analysis_candidates.csv.

CSV derivatives ending __text_excluded.csv omit only the columns
listed in manifest/additional_artifact_export_log.json.
Source experimental files remain unchanged.

inference_inputs.csv is not included because its query field contains
input text. Dataset reconstruction/access instructions remain required.

The JSONL prediction file was checked for expected top-level fields
in every nonempty record. This is a schema check, not a privacy audit.

Unchanged JSON metadata may reference checksums of source files that
are excluded or represented by derivatives. Those historical checksum
references remain intact. The export log records derivative checksums.

All included analyses still require manuscript mapping and sharing review.
Copying an artifact does not establish that its metrics were regenerated.
"""

(PACKAGE / "ADDITIONAL_ARTIFACTS.md").write_text(
    note, encoding="utf-8"
)

from collections import Counter

print("Export summary:")
for operation, count in sorted(
    Counter(item["operation"] for item in log).items()
):
    print(f"{operation}: {count}")

print("\nSource files unchanged.")
print("Additional artifacts remain subject to manual sharing review.")
print("Manifest inventory and checksums need a final refresh.")

In [ ]:
from pathlib import Path
import csv
import json

ROOT = Path(_crus_local_path('project', ''))
PACKAGE = Path(
    _crus_local_path('runtime', 'editor9_staging/crus_review_20261019T070629Z')
)

folders = {
    "primary": (
        ROOT / "corrected_experiment_outputs"
        / "domain_baselines/medcpt_query_v1"
    )
}

for seed in [11, 22, 33, 44, 55]:
    folders[f"seed_{seed}"] = (
        ROOT / "supplementary_repeated_splits"
        / f"seed_{seed}/domain_baselines/medcpt_query_v1"
    )

rows = []

for run, folder in folders.items():
    print(f"\nRUN: {run}")

    if not folder.is_dir():
        print("FOLDER NOT FOUND")
        continue

    for path in sorted(folder.rglob("*")):
        if not path.is_file():
            continue

        # Exclude model/embedding binaries from this initial selection.
        if path.suffix.lower() not in {".csv", ".json", ".npz"}:
            continue

        schema = None

        if path.suffix.lower() == ".csv":
            with path.open(
                encoding="utf-8-sig", newline=""
            ) as handle:
                schema = {
                    "columns": next(csv.reader(handle), [])
                }

        elif path.suffix.lower() == ".json":
            payload = json.loads(path.read_text(encoding="utf-8"))
            schema = {
                "type": type(payload).__name__,
                "top_level_keys": (
                    list(payload.keys())
                    if isinstance(payload, dict) else None
                )
            }

        relative = path.relative_to(ROOT).as_posix()

        rows.append({
            "run": run,
            "source_relative_path": relative,
            "bytes": path.stat().st_size,
            "schema": schema
        })

        print(f"FILE: {path.relative_to(folder).as_posix()}")
        if schema is not None:
            print(json.dumps(schema, indent=2))
        else:
            print("NPZ file listed; contents not inspected in this step.")

report = PACKAGE / "manifest/medcpt_candidate_inventory.json"
report.write_text(
    json.dumps(rows, indent=2),
    encoding="utf-8"
)

print("\nCandidate MedCPT files:", len(rows))
print("Inventory saved:", report)
print("No experimental artifacts copied or modified.")

In [ ]:
from pathlib import Path
import json
import hashlib
import shutil
import numpy as np

ROOT = Path(_crus_local_path('project', ''))
PACKAGE = Path(
    _crus_local_path('runtime', 'editor9_staging/crus_review_20261019T070629Z')
)

inventory_path = PACKAGE / "manifest/medcpt_candidate_inventory.json"
candidates = json.loads(inventory_path.read_text(encoding="utf-8"))

selected_names = {
    "RUN_COMPLETE.json",
    "baseline_configuration.json",
    "metric_summary.csv",
    "prediction_audit.csv",
    "test_MedCPT_Query_arrays.npz",
    "validation_MedCPT_Query_arrays.npz",
    "test_matched_selection.csv",
    "truncation_diagnostics.csv",
}

selected = [
    item for item in candidates
    if Path(item["source_relative_path"]).name in selected_names
]

if len(selected) != 48:
    raise ValueError(
        f"Expected 48 selected files; found {len(selected)}. "
        "Review the inventory before proceeding."
    )

def sha256(path):
    h = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

# Inspect all selected arrays before copying any files.
array_schemas = []

for item in selected:
    relative = item["source_relative_path"]
    source = ROOT / relative

    if not source.is_file():
        raise FileNotFoundError(source)

    if source.suffix.lower() != ".npz":
        continue

    fields = []

    with np.load(source, allow_pickle=False) as archive:
        for key in archive.files:
            try:
                value = archive[key]
            except ValueError as error:
                raise RuntimeError(
                    f"Array requires review without pickle: "
                    f"{relative}, field {key}"
                ) from error

            fields.append({
                "key": key,
                "shape": list(value.shape),
                "dtype": str(value.dtype)
            })

    array_schemas.append({
        "source_relative_path": relative,
        "fields": fields
    })

# Copy only the explicitly selected artifacts.
export_log = []

for item in selected:
    relative = item["source_relative_path"]
    source = ROOT / relative
    destination = PACKAGE / "artifacts" / relative

    source_hash = sha256(source)

    if destination.exists():
        if sha256(destination) != source_hash:
            raise RuntimeError(
                f"Different staged file already exists: {relative}"
            )
    else:
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source, destination)

    export_log.append({
        "run": item["run"],
        "source_relative_path": relative,
        "package_path": destination.relative_to(PACKAGE).as_posix(),
        "operation": "unchanged_copy",
        "sha256": source_hash,
        "bytes": destination.stat().st_size
    })

manifest = PACKAGE / "manifest"

(manifest / "medcpt_array_schemas.json").write_text(
    json.dumps(array_schemas, indent=2),
    encoding="utf-8"
)

(manifest / "medcpt_export_log.json").write_text(
    json.dumps(export_log, indent=2),
    encoding="utf-8"
)

notes = """# MedCPT baseline artifacts

The package includes primary and five repeated-run artifacts:
configuration, run-completion records, metric summaries,
prediction audits, test/validation prediction arrays,
matched-selection decisions, and truncation diagnostics.

Selected files were copied unchanged.
Original experiment files were not modified.

Training embeddings are not included in this release selection.
Rebuilding them requires the source data, model configuration,
and documented implementation.

Array fields were inspected with allow_pickle=False.
This establishes readability and schema only, not privacy approval
or numerical agreement with reported results.

MedCPT environment records apply to this baseline and must not be
used to infer the environment of another experiment.

Metric summaries have been packaged but have not yet been
independently regenerated as part of this packaging workflow.
"""

(PACKAGE / "MEDCPT_ARTIFACTS.md").write_text(
    notes, encoding="utf-8"
)

print("Selected MedCPT files copied or verified:", len(export_log))
print("NPZ archives inspected:", len(array_schemas))
print("Training-embedding files excluded:",
      sum(
          Path(item["source_relative_path"]).name
          == "training_embeddings.npz"
          for item in candidates
      ))

print("\nRepresentative array schemas:")
for entry in array_schemas[:2]:
    print(entry["source_relative_path"])
    for field in entry["fields"]:
        print(" ", field)

print("\nSource experimental files unchanged.")
print("Manual sharing review and final checksums remain pending.")

In [ ]:
from pathlib import Path

PACKAGE = Path(
    _crus_local_path('runtime', 'editor9_staging/crus_review_20261019T070629Z')
)

guide = """# Reviewer guide

## 1. Purpose and status

This package provides implementation notebooks, saved configurations,
split identifiers, case-level outputs, and supporting analysis artifacts.

It is not yet a verified clean-environment, end-to-end inference release.
The verification scope and limitations are stated below.

## 2. Start here: regenerate metrics from saved predictions

From the repository root, run:

python scripts/regenerate_primary_metrics.py --package-root .
python scripts/verify_matched_rankings.py --package-root .

These scripts do not run model inference or require raw dialogue inputs.

Outputs are written to regenerated/.

See REPRODUCING_RESULTS.md for metric definitions and verification scope.

## 3. Verified results

- The primary metric script regenerated full-coverage classification
  and native-acceptance metrics and checked saved correctness flags.
- The matched-ranking script reconstructed all 45 method/target rows
  from saved ranking scores.
- Eleven count/metric fields agreed with the saved verified summary.
- Integer counts matched exactly.
- Floating-point comparisons used absolute tolerance 1e-10.

These checks do not establish reproduction of every manuscript result.

## 4. Implementation notebooks

notebooks/CRUS_R1_5_External_Transfer.ipynb
notebooks/revised_risk_aware_hybrid_pipeline_v2.ipynb

These are output-cleared historical implementation notebooks.
They retain original path references and multiple revision analyses.
Do not execute all cells indiscriminately.

A tested section-by-section execution map for fresh model inference
has not yet been supplied.

## 5. Configurations and split identifiers

Primary configuration and policies:
artifacts/corrected_experiment_outputs/final_pretest_protocol.json
artifacts/corrected_experiment_outputs/validation_selected_policies.json

Primary split:
artifacts/data_audit/constrained_split_manifest.csv

Repeated splits:
artifacts/supplementary_repeated_splits/seed_*/split_manifest.csv

Calibration-fold assignments are provided separately.

Use supplied split assignments when reproducing reported analyses;
do not generate new splits as substitutes.

## 6. Data access and reconstruction

Raw input dialogue files are not included in this package selection.

Before claiming end-to-end reproducibility, the authors must document:
- Original dataset access location and version.
- Source CSV filenames and source-row indexing convention.
- Record-ID construction.
- Text normalization, cleaning, and filtering.
- Group construction and duplicate handling.
- Reconstruction of selected additional-MTS records.
- Perturbation generation and seeds.

Do not guess source-row indexing or substitute a different dataset version.

## 7. Text-excluded derivatives

Files ending __text_excluded.csv omit the columns listed in:
manifest/additional_artifact_export_log.json

Their filenames and checksums differ from the original artifacts.
Historical metadata can still refer to original-file checksums.

The query-text file inference_inputs.csv is not distributed here.

## 8. Additional analyses

Additional-analysis artifacts are included as supporting outputs.
Their presence does not mean all metrics were independently regenerated.

See ADDITIONAL_ARTIFACTS.md and the export logs.

The additional-MTS analysis must not automatically be interpreted as
independent cross-dataset validation. Consult its saved scope and
limitation fields.

## 9. MedCPT baseline

See MEDCPT_ARTIFACTS.md.

Primary and repeated-run configurations, predictions, selections,
and summaries are provided. Training embeddings are excluded.

MedCPT metrics have not been independently regenerated in this
packaging workflow.

## 10. Environment records

See environment/.

Recorded versions are partial historical snapshots, not a complete,
tested dependency lockfile.

Six recorded package versions match between the primary run and
each repeated run. pandas and scipy versions are not recorded in
the repeated-run retrieval metadata.

Do not infer the main-run environment from a separate baseline.

## 11. Integrity and provenance

See manifest/ for inventories, transformation logs, and checksums.

Checksums identify files; they do not establish scientific validity,
privacy compliance, or execution success.

## 12. Outstanding release tasks

- Complete manual sharing and redistribution review.
- Complete dataset-access/reconstruction instructions.
- Document fresh-inference execution order and dependencies.
- Map final manuscript tables/figures to artifacts and commands.
- Refresh final inventories/checksums after all edits.
- Arrange and test editor/reviewer access.
"""

(PACKAGE / "REVIEWER_GUIDE.md").write_text(
    guide, encoding="utf-8"
)

print("REVIEWER_GUIDE.md created.")
print("No experimental artifacts changed.")
print("Package remains a partially verified draft.")

In [ ]:
from pathlib import Path
import json
import re

ROOT = Path(_crus_local_path('project', ''))

NOTEBOOK_PATH = Path("CRUS_R1_5_External_Transfer.ipynb").resolve()

if not NOTEBOOK_PATH.is_file():
    raise FileNotFoundError(NOTEBOOK_PATH)

notebook = json.loads(
    NOTEBOOK_PATH.read_text(encoding="utf-8")
)

pattern = re.compile(
    r"source_row|source_csv_row|record_id|"
    r"def normalize|def dialogue_hash|"
    r"hashlib\.sha256|REPOSITORY\s*=|"
    r"download_commit|read_csv",
    flags=re.IGNORECASE
)

print("A. PRIMARY SOURCE-MAPPING CODE")

# One-based notebook cell numbers from the earlier inspection.
for cell_number in [48, 49, 54]:
    cell = notebook["cells"][cell_number - 1]
    source = cell.get("source", "")
    source = "".join(source) if isinstance(source, list) else source
    lines = source.splitlines()

    selected = set()

    for index, line in enumerate(lines):
        if pattern.search(line):
            selected.update(
                range(max(0, index - 2), min(len(lines), index + 4))
            )

    print(f"\nCELL {cell_number}")
    for index in sorted(selected):
        print(f"{index + 1}: {lines[index]}")

print("\nB. SOURCE/ DOWNLOAD PROVENANCE FILES")

provenance_paths = [
    ROOT / "source_data/source_manifest.json",
    ROOT / (
        "additional_mts_heldout_audit_20261017T085443976861Z/"
        "download_provenance.json"
    ),
]

for path in provenance_paths:
    print(f"\nFILE: {path.relative_to(ROOT)}")

    if not path.is_file():
        print("NOT FOUND")
        continue

    data = json.loads(path.read_text(encoding="utf-8"))
    print("TOP-LEVEL KEYS:", list(data) if isinstance(data, dict)
          else type(data).__name__)

    # Print only repository/version fields, not arbitrary values.
    if isinstance(data, dict):
        for key in [
            "repository", "commit", "repository_commit",
            "revision", "version"
        ]:
            if key in data:
                print(f"{key}: {data[key]}")

        files = data.get("files")
        if isinstance(files, dict):
            print("FILE ENTRIES:", list(files.keys()))
        elif isinstance(files, list):
            print("NUMBER OF FILE ENTRIES:", len(files))
            for entry in files:
                if isinstance(entry, dict):
                    print(
                        "FILE:",
                        entry.get("filename", "[not recorded]"),
                        "| metadata fields:",
                        list(entry.keys())
                    )

print("\nInspection only. No source records printed or files changed.")

In [ ]:
from pathlib import Path
import hashlib
import json
import shutil

ROOT = Path(_crus_local_path('project', ''))
PACKAGE = Path(
    _crus_local_path('runtime', 'editor9_staging/crus_review_20261019T070629Z')
)

provenance_files = [
    "source_data/source_manifest.json",
    (
        "additional_mts_heldout_audit_20261017T085443976861Z/"
        "download_provenance.json"
    )
]

def sha256(path):
    h = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

export_log = []

for relative in provenance_files:
    source = ROOT / relative
    destination = PACKAGE / "artifacts" / relative

    if not source.is_file():
        raise FileNotFoundError(source)

    if destination.exists():
        if sha256(destination) != sha256(source):
            raise RuntimeError(
                f"Different staged provenance file exists: {relative}"
            )
    else:
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source, destination)

    export_log.append({
        "source_relative_path": relative,
        "package_path": destination.relative_to(PACKAGE).as_posix(),
        "operation": "unchanged_copy",
        "sha256": sha256(destination)
    })

guide = r"""# Dataset access and record reconstruction

## Source and recorded version

Repository:
https://github.com/abachaa/MTS-Dialog

Recorded commit:
3ff0801933608d6f570468c13125125fb5cabdea

Use the saved provenance files to identify source artifacts and
verify their recorded checksums. Check the source dataset's applicable
access and redistribution terms separately.

Primary provenance:
artifacts/source_data/source_manifest.json

Additional-MTS provenance:
artifacts/additional_mts_heldout_audit_20261017T085443976861Z/download_provenance.json

Raw dialogue inputs are not distributed in this package selection.

## Primary source files

MTS-Dialog-TrainingSet.csv
MTS-Dialog-ValidationSet.csv

Use files corresponding to the recorded provenance and checksums,
not an unverified current repository version.

## Primary record identifiers

The original code reads each source CSV with pandas.read_csv.

source_row is the zero-based parsed data-row position within that file.
It is not the original dataset ID and is not a physical CSV line number.
CSV headers are not counted as data rows.

record_id is constructed as:

    f"{source_filename}:{source_row}"

For example, source_row 7 refers to dataframe.iloc[7].

Use source_file and source_row from:
artifacts/data_audit/constrained_split_manifest.csv

Do not reset source-row identifiers after filtering or deduplication.

## Dialogue normalization and hashing

The recorded normalization is:

    re.sub(r"\s+", " ", str(text).strip().lower())

dialogue_hash is:

    hashlib.sha256(normalized_text.encode("utf-8")).hexdigest()

For each retained primary record, verify the normalized source
dialogue against dialogue_hash in the supplied manifest.

The classifier input is normalized dialogue. Reference section text
is not used as query input.

## Labels

Use the manifest's retained label and the supplied source/preprocessing
code to verify source-label agreement.

Do not guess label-normalization rules or substitute an unrelated
specialty-label mapping.

## Deduplication, grouping, and filtering

The implementation sorts by source order and source-row position
before choosing representatives of exact dialogue/label duplicates.

Exact normalized-dialogue/label duplicates are reduced to one
representative. Records with the same normalized dialogue but
different labels are retained.

Reviewed near-duplicate pairs can share a group despite having
different dialogue hashes. Therefore, group_id must not be regenerated
from dialogue_hash alone.

Consult:
- final_cleaning_grouping_policy.json
- exact_duplicate_removal_log.csv
- near_duplicate_review_decisions.csv
- retained_conflicting_label_groups.csv
- rare_class_exclusion_log.csv

Use the supplied final manifests as the authoritative assignments
for reproducing reported evaluations.

## Primary and repeated partitions

Primary manifest:
artifacts/data_audit/constrained_split_manifest.csv

The primary cohort contains 1,288 records and 17 retained labels:
773 train, 258 validation, and 257 test.

Repeated manifests:
artifacts/supplementary_repeated_splits/seed_*/split_manifest.csv

Preserve record and group identifiers. Do not create fresh splits
as replacements for reported assignments.

## Additional-MTS evaluation

Source files:
MTS-Dialog-TestSet-1-MEDIQA-Chat-2023.csv
MTS-Dialog-TestSet-2-MEDIQA-Sum-2023.csv

For these additional records, source_csv_row is one-based:
source_csv_row 1 refers to dataframe.iloc[0].

This differs from the primary cohort's zero-based source_row.

Use evaluation_reference.csv for selected record/source mappings.
Use all_400_records_screening_audit__text_excluded.csv for screening
flags and exclusions.

Retain the provided record_id values rather than inventing replacements.
Use run_manifest.json and metrics.json for evaluation scope,
class-probability order, frozen policy, and limitations.

This analysis is an exploratory additional-MTS evaluation and must not
automatically be described as independent cross-dataset validation.

## Perturbation outputs

Files ending __text_excluded.csv omit specified dialogue-text fields.
See manifest/additional_artifact_export_log.json.

Recreating perturbed text requires the original input mapping,
transformation implementation, seeds, and recorded settings.

This guide does not claim that perturbation-text reconstruction
has been independently tested.

## Verification status

Source-mapping conventions were inspected in the implementation.
A fresh reconstruction of all input records from independently
obtained source CSVs has not yet been tested in this packaging workflow.

Historical provenance files are retained unchanged. Their presence
does not, by itself, establish licensing approval or independent
verification of every source byte.
"""

(PACKAGE / "DATA_RECONSTRUCTION.md").write_text(
    guide, encoding="utf-8"
)

(PACKAGE / "manifest/source_provenance_export_log.json").write_text(
    json.dumps(export_log, indent=2),
    encoding="utf-8"
)

print("Provenance files copied or verified:", len(export_log))
print("DATA_RECONSTRUCTION.md created.")
print("No raw dialogue copied; no datasets downloaded.")
print("Source experimental files unchanged.")
print("Final inventory/checksum refresh remains pending.")

In [ ]:
from pathlib import Path

PACKAGE = Path(
    _crus_local_path('runtime', 'editor9_staging/crus_review_20261019T070629Z')
)

readme = """# CRUS clinical section-routing reproducibility materials

## Status

Review-package draft.
Manual sharing review and editorial-access testing remain pending.

This package contains implementation notebooks, recorded configurations,
split identifiers and group assignments, cleaning/grouping audits,
case-level outputs, and supporting analysis artifacts.

It is not a certified clean-environment, end-to-end inference release.

## Start here

1. REVIEWER_GUIDE.md
2. REPRODUCING_RESULTS.md
3. DATA_RECONSTRUCTION.md

Additional documentation:
- ADDITIONAL_ARTIFACTS.md
- MEDCPT_ARTIFACTS.md
- environment/README.md

## Regenerate specified metrics from saved predictions

From the repository root:

python scripts/regenerate_primary_metrics.py --package-root .
python scripts/verify_matched_rankings.py --package-root .

These commands do not run model inference.

The scripts use Python, NumPy, pandas, and scikit-learn.
Recorded environment versions are documented under environment/.
A clean-environment installation test has not been completed.

## Verification scope

The primary regeneration script checked saved correctness flags
and reconstructed full-coverage and native-acceptance metrics.

The matched-ranking script independently reconstructed selections
from saved ranking scores for 45 method/target combinations.

Eleven count/metric fields agreed with the saved verified summary:
integer counts exactly, and floating-point fields within absolute
tolerance 1e-10.

Other supplied summaries and figures are not all independently
regenerated by these two scripts.

## Implementation notebooks

notebooks/CRUS_R1_5_External_Transfer.ipynb
notebooks/revised_risk_aware_hybrid_pipeline_v2.ipynb

These are output-cleared historical notebooks containing multiple
experiments, audits, and machine-specific paths.

Do not use Run all indiscriminately.
A tested fresh-inference execution map has not yet been provided.

## Data and identifiers

Raw dialogue inputs are not distributed in this package selection.

DATA_RECONSTRUCTION.md documents the recorded source commit,
source-file mapping, row-index conventions, normalization,
hashing, and authoritative split assignments.

Text-excluded CSV derivatives retain distinct filenames.
Export logs document the removed columns and source/output hashes.

## Additional analyses

The package includes selected artifacts for the additional-MTS,
uncertainty, weight-sensitivity, retrieval-audit, sensitivity,
perturbation, calibration, class-deferral, candidate-selection,
and MedCPT analyses.

Inclusion does not establish independent numerical regeneration.

Consult saved scope and limitation fields. In particular,
additional-MTS evaluation is not automatically independent
cross-dataset validation.

## Environment limitations

The primary protocol records eight package versions.
Six recorded versions match for each repeated run.
Repeated-run retrieval metadata do not record pandas or scipy versions.

The recorded candidate requirements list is not a tested lockfile.
Baseline-specific environment records apply to that baseline.

## Integrity and provenance

manifest/ contains export logs, inventories, validation reports,
and SHA256SUMS.txt.

Historical artifact_manifest.csv covers the initial staging selection.
current_file_inventory.json describes the current package contents.
Later exports are documented in their separate export logs.

Historical checksum references may refer to excluded original
files rather than text-excluded derivatives.

## Before distribution

- Complete manual review of code, Markdown, JSON values,
  attachments, and artifact contents.
- Confirm applicable data-sharing and software-distribution terms.
- Choose and add an appropriate code license.
- Add the actual review URL, release identifier, and commit.
- Test the editor/reviewer access route.
- Refresh inventories and checksums after final changes.

Do not describe this package as fully independently reproduced.
"""

(PACKAGE / "README.md").write_text(readme, encoding="utf-8")

print("README.md updated.")
print("No experimental artifacts changed.")

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import shutil

PACKAGE = Path(
    _crus_local_path('runtime', 'editor9_staging/crus_review_20261019T070629Z')
)

if not PACKAGE.is_dir():
    raise FileNotFoundError(PACKAGE)

def sha256(path):
    h = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

manifest_dir = PACKAGE / "manifest"
manifest_dir.mkdir(exist_ok=True)

inventory_path = manifest_dir / "current_file_inventory.json"
checksum_path = manifest_dir / "SHA256SUMS.txt"

# Inventory excludes itself and the checksum list.
inventory = [
    {
        "path": path.relative_to(PACKAGE).as_posix(),
        "bytes": path.stat().st_size,
        "sha256": sha256(path)
    }
    for path in sorted(PACKAGE.rglob("*"))
    if path.is_file()
    and path not in {inventory_path, checksum_path}
]

inventory_path.write_text(
    json.dumps(inventory, indent=2),
    encoding="utf-8"
)

# Checksums include the inventory but exclude the checksum list itself.
checksum_entries = [
    (
        path.relative_to(PACKAGE).as_posix(),
        sha256(path)
    )
    for path in sorted(PACKAGE.rglob("*"))
    if path.is_file() and path != checksum_path
]

checksum_path.write_text(
    "\n".join(
        f"{digest}  {relative}"
        for relative, digest in checksum_entries
    ) + "\n",
    encoding="utf-8"
)

# Verify the listed files before making the backup.
for relative, expected in checksum_entries:
    if sha256(PACKAGE / relative) != expected:
        raise RuntimeError(f"Checksum verification failed: {relative}")

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
backup_root = Path(
    _crus_local_path('drive', 'Editor9_Reproducibility_Drafts')
)
backup_root.mkdir(parents=True, exist_ok=True)

backup = backup_root / f"crus_review_DRAFT_{stamp}"

if backup.exists():
    raise FileExistsError(backup)

shutil.copytree(PACKAGE, backup)

# Verify that the backup contains identical copies.
for relative, expected in checksum_entries:
    if sha256(backup / relative) != expected:
        raise RuntimeError(f"Backup verification failed: {relative}")

if sha256(backup / "manifest/SHA256SUMS.txt") != sha256(checksum_path):
    raise RuntimeError("Backup checksum-list verification failed.")

archive = Path(
    shutil.make_archive(
        str(backup),
        "zip",
        root_dir=backup.parent,
        base_dir=backup.name
    )
)

if not archive.is_file() or archive.stat().st_size == 0:
    raise RuntimeError("Draft ZIP creation failed.")

archive_hash = sha256(archive)
archive_hash_file = archive.with_suffix(".zip.sha256")
archive_hash_file.write_text(
    f"{archive_hash}  {archive.name}\n",
    encoding="utf-8"
)

print("Inventory entries:", len(inventory))
print("Checksummed files:", len(checksum_entries))
print("Backup verification: PASSED")
print("Draft backup:", backup)
print("Draft ZIP:", archive)
print("ZIP size (MB):", round(archive.stat().st_size / 1_000_000, 2))
print("ZIP SHA-256:", archive_hash)
print("Sharing approval: STILL PENDING")

In [ ]:
from pathlib import Path
pass  # Colab service replaced by local workflow.

archive = Path(
    _crus_local_path('drive', 'Editor9_Reproducibility_Drafts/crus_review_DRAFT_20261019T072942457396Z.zip')
)

if not archive.is_file():
    raise FileNotFoundError(archive)

_crus_show_file(str(archive))

In [ ]:
from pathlib import Path
pass  # Colab service replaced by local workflow.

DRIVE = Path(_crus_local_path('mount', ''))
MYDRIVE = DRIVE / "MyDrive"

if MYDRIVE.is_dir():
    print("Google Drive is already accessible.")
else:
    print("Connecting Google Drive...")
    _crus_prepare_saved_workspace()

if not MYDRIVE.is_dir():
    raise RuntimeError(
        "Google Drive is still unavailable. "
        "Do not run the Table 13 analysis yet."
    )

ROOT = MYDRIVE / "RiskAware_Revision2"

print("\nDrive accessible:", MYDRIVE.is_dir())
print("Expected project folder:", ROOT)
print("Project folder exists:", ROOT.is_dir())

if ROOT.is_dir():
    print("\nFiles/folders inside the project:")
    for item in sorted(ROOT.iterdir()):
        print(item.name)
else:
    print(
        "\nDrive is connected, but RiskAware_Revision2 "
        "is not at the expected location."
    )
    print("Check the folder location and the Google account you connected.")

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import platform

import numpy as np
import pandas as pd
import scipy
from scipy.stats import mannwhitneyu

# ---------------- Configuration ----------------

ROOT = Path(_crus_local_path('project', ''))
METHOD = "Cascade"

EXPECTED_N = 257
EXPECTED_CORRECT = 181
EXPECTED_INCORRECT = 76

BOOTSTRAP_REPLICATES = 10000
BOOTSTRAP_SEED = 20261029

if not ROOT.is_dir():
    pass  # Colab service replaced by local workflow.
    _crus_prepare_saved_workspace()

if not ROOT.is_dir():
    raise FileNotFoundError(
        "RiskAware_Revision2 is not accessible. "
        "Check the mounted Google Drive account."
    )

PRED_PATH = (
    ROOT / "corrected_experiment_outputs"
    / "test_all_method_predictions.csv"
)
CRUS_PATH = (
    ROOT / "corrected_experiment_outputs"
    / "test_frozen_CRUS_acceptance_audit.csv"
)
MANIFEST_PATH = ROOT / "data_audit/constrained_split_manifest.csv"
PROTOCOL_PATH = (
    ROOT / "corrected_experiment_outputs"
    / "final_pretest_protocol.json"
)

for path in [PRED_PATH, CRUS_PATH, MANIFEST_PATH, PROTOCOL_PATH]:
    if not path.is_file():
        raise FileNotFoundError(path)

# ---------------- Helpers ----------------

def sha256(path):
    h = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def require(condition, message):
    if not condition:
        raise ValueError(message)

def parse_boolean(series, name):
    values = series.astype("string").str.strip().str.lower()
    mapping = {
        "true": True, "false": False,
        "1": True, "0": False,
        "1.0": True, "0.0": False
    }
    require(
        values.isin(mapping).all(),
        f"Invalid or missing Boolean values in {name}."
    )
    return values.map(mapping).astype(bool)

source_paths = {
    "predictions": PRED_PATH,
    "crus_audit": CRUS_PATH,
    "manifest": MANIFEST_PATH,
    "protocol": PROTOCOL_PATH
}
input_hashes = {
    name: sha256(path) for name, path in source_paths.items()
}

# ---------------- Read and verify cohort ----------------

pred_all = pd.read_csv(PRED_PATH, dtype={"record_id": "string"})
audit_all = pd.read_csv(CRUS_PATH, dtype={"record_id": "string"})
manifest = pd.read_csv(
    MANIFEST_PATH,
    dtype={"record_id": "string", "group_id": "string"}
)
protocol = json.loads(PROTOCOL_PATH.read_text(encoding="utf-8"))

require(
    protocol.get("main_manifest_sha256") == input_hashes["manifest"],
    "Manifest checksum does not match the frozen primary protocol."
)

require(
    not manifest["record_id"].duplicated().any(),
    "Duplicate record IDs in the full manifest."
)
require(
    manifest[["record_id", "group_id", "split", "label"]]
    .notna().all().all(),
    "Missing required manifest values."
)
require(
    manifest.groupby("group_id")["split"].nunique().le(1).all(),
    "A group crosses manifest partitions."
)

test = manifest.loc[
    manifest["split"].eq("test"),
    ["record_id", "group_id", "label"]
].copy()

pred = pred_all.loc[pred_all["method"].eq(METHOD)].copy()
audit = audit_all.loc[audit_all["method"].eq(METHOD)].copy()

require(len(test) == EXPECTED_N, "Unexpected manifest test size.")

test_ids = set(test["record_id"])

for name, frame in [("predictions", pred), ("CRUS audit", audit)]:
    require(
        frame["record_id"].notna().all(),
        f"Missing record IDs in {name}."
    )
    require(
        not frame["record_id"].duplicated().any(),
        f"Duplicate Cascade record IDs in {name}."
    )
    require(
        set(frame["record_id"]) == test_ids,
        f"Cascade IDs in {name} differ from final test IDs."
    )

require(
    pred[["true_label", "prediction"]].notna().all().all(),
    "Missing prediction labels."
)

pred["computed_correct"] = pred["true_label"].eq(pred["prediction"])

require(
    (
        parse_boolean(pred["correct"], "prediction correct")
        == pred["computed_correct"]
    ).all(),
    "Saved prediction correctness disagrees with labels."
)

audit["audit_correct"] = parse_boolean(
    audit["correct"], "CRUS-audit correct"
)

final = (
    test.merge(
        pred[[
            "record_id", "true_label", "prediction",
            "computed_correct"
        ]],
        on="record_id",
        how="left",
        validate="one_to_one"
    )
    .merge(
        audit[["record_id", "CRUS", "audit_correct"]],
        on="record_id",
        how="left",
        validate="one_to_one"
    )
    .sort_values("record_id")
    .reset_index(drop=True)
)

require(
    final["label"].eq(final["true_label"]).all(),
    "Prediction reference labels differ from manifest labels."
)
require(
    final["audit_correct"].eq(final["computed_correct"]).all(),
    "CRUS-audit correctness disagrees with final predictions."
)

final["CRUS"] = pd.to_numeric(final["CRUS"], errors="raise")
require(
    np.isfinite(final["CRUS"]).all(),
    "CRUS contains nonfinite values."
)
require(
    final["CRUS"].between(0, 1).all(),
    "CRUS falls outside the documented [0, 1] range."
)

n_correct = int(final["computed_correct"].sum())
n_incorrect = int((~final["computed_correct"]).sum())

require(
    len(final) == EXPECTED_N
    and n_correct == EXPECTED_CORRECT
    and n_incorrect == EXPECTED_INCORRECT,
    (
        f"Unexpected final cohort: N={len(final)}, "
        f"correct={n_correct}, incorrect={n_incorrect}. "
        "Stop and investigate; do not force these counts."
    )
)

# ---------------- Descriptive statistics ----------------

correct_scores = final.loc[final["computed_correct"], "CRUS"].to_numpy()
incorrect_scores = final.loc[~final["computed_correct"], "CRUS"].to_numpy()

summary = pd.DataFrame([
    {
        "subset": name,
        "n": len(values),
        "mean_CRUS": float(np.mean(values)),
        "sd_CRUS": float(np.std(values, ddof=1)),
        "median_CRUS": float(np.median(values)),
        "minimum_CRUS": float(np.min(values)),
        "maximum_CRUS": float(np.max(values))
    }
    for name, values in [
        ("Correct predictions", correct_scores),
        ("Incorrect predictions", incorrect_scores)
    ]
])

delta = float(incorrect_scores.mean() - correct_scores.mean())

# ---------------- Group-bootstrap mean difference ----------------

codes, group_names = pd.factorize(final["group_id"], sort=True)
n_groups = len(group_names)

correct_mask = final["computed_correct"].to_numpy(dtype=bool)
scores = final["CRUS"].to_numpy(dtype=float)

correct_counts = np.bincount(
    codes, weights=correct_mask.astype(int), minlength=n_groups
)
incorrect_counts = np.bincount(
    codes, weights=(~correct_mask).astype(int), minlength=n_groups
)
correct_sums = np.bincount(
    codes, weights=scores * correct_mask, minlength=n_groups
)
incorrect_sums = np.bincount(
    codes, weights=scores * (~correct_mask), minlength=n_groups
)

rng = np.random.default_rng(BOOTSTRAP_SEED)
bootstrap_delta = np.full(BOOTSTRAP_REPLICATES, np.nan)

for b in range(BOOTSTRAP_REPLICATES):
    sampled = rng.integers(0, n_groups, size=n_groups)

    nc = correct_counts[sampled].sum()
    ni = incorrect_counts[sampled].sum()

    if nc > 0 and ni > 0:
        bootstrap_delta[b] = (
            incorrect_sums[sampled].sum() / ni
            - correct_sums[sampled].sum() / nc
        )

valid = bootstrap_delta[np.isfinite(bootstrap_delta)]
require(
    len(valid) >= int(0.99 * BOOTSTRAP_REPLICATES),
    "Too many bootstrap replicates have an empty correctness subset."
)

ci_low, ci_high = np.quantile(valid, [0.025, 0.975])

# ---------------- Mann–Whitney and effect size ----------------

test_result = mannwhitneyu(
    incorrect_scores,
    correct_scores,
    alternative="two-sided",
    method="asymptotic",
    use_continuity=True
)

u_incorrect_first = float(test_result.statistic)
p_value = float(test_result.pvalue)

rank_biserial = (
    2 * u_incorrect_first / (n_incorrect * n_correct) - 1
)

group_sizes = final.groupby("group_id").size()
mixed_groups = (
    final.groupby("group_id")["computed_correct"].nunique() > 1
).sum()

# ---------------- Save new outputs only ----------------

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
OUT = ROOT / f"editor_table13_final_cohort_{stamp}"
OUT.mkdir(parents=True, exist_ok=False)

summary.to_csv(OUT / "table13_crus_summary.csv", index=False)

final[[
    "record_id", "group_id", "label", "prediction",
    "computed_correct", "CRUS"
]].to_csv(OUT / "table13_final_case_audit.csv", index=False)

np.savez_compressed(
    OUT / "table13_bootstrap_mean_difference.npz",
    mean_difference=bootstrap_delta
)

metadata = {
    "analysis": "Primary Cascade CRUS by forced-prediction correctness",
    "n_test": len(final),
    "n_correct": n_correct,
    "n_incorrect": n_incorrect,
    "mean_correct": float(correct_scores.mean()),
    "mean_incorrect": float(incorrect_scores.mean()),
    "difference_direction": "incorrect minus correct",
    "mean_difference": delta,
    "mean_difference_95_ci": [float(ci_low), float(ci_high)],
    "bootstrap": {
        "unit": "assigned test group_id",
        "group_sampling": "sample G groups with replacement per replicate",
        "within_group": "retain all records; record-weighted subset means",
        "replicates_attempted": BOOTSTRAP_REPLICATES,
        "replicates_valid": len(valid),
        "replicates_empty_subset": int(
            BOOTSTRAP_REPLICATES - len(valid)
        ),
        "seed": BOOTSTRAP_SEED,
        "interval": "2.5 and 97.5 percentiles; numpy linear quantiles",
        "fixed": "model, split, predictions, correctness, CRUS, group assignments"
    },
    "groups": {
        "n_groups": n_groups,
        "maximum_group_size": int(group_sizes.max()),
        "groups_with_both_correct_and_incorrect": int(mixed_groups)
    },
    "mann_whitney": {
        "first_sample": "incorrect predictions",
        "second_sample": "correct predictions",
        "U": u_incorrect_first,
        "p_two_sided": p_value,
        "method": "asymptotic; tie correction; continuity correction",
        "scope": (
            "Exploratory record-level test; assumes independent samples; "
            "not cluster-adjusted and not multiplicity-adjusted."
        )
    },
    "rank_biserial": {
        "value": float(rank_biserial),
        "formula": "2*U_incorrect_first/(n_incorrect*n_correct)-1",
        "positive_direction": "higher CRUS among incorrect predictions"
    },
    "input_sha256": input_hashes,
    "versions": {
        "python": platform.python_version(),
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "scipy": scipy.__version__
    },
    "limitations": [
        "Post-test descriptive/exploratory analysis.",
        "Conditional group-bootstrap interval excludes model-refitting uncertainty.",
        "Assigned groups may not capture all related-record dependence.",
        "Mann–Whitney p-value is not a group-adjusted inferential result.",
        "Correctness separation does not establish optimal weights or clinical utility."
    ]
}

(OUT / "table13_analysis_metadata.json").write_text(
    json.dumps(metadata, indent=2), encoding="utf-8"
)

table_lines = [
    r"\begin{table}[ht]",
    r"\centering",
    (
        r"\caption{CRUS by forced-prediction correctness for Cascade "
        r"on the final primary test cohort ($N=257$). "
        r"All cases are included, irrespective of acceptance.}"
    ),
    r"\label{tab:crus}",
    r"\begin{tabular}{lrr}",
    r"\hline",
    r"Subset & Mean CRUS & Number of cases \\",
    r"\hline",
    f"Correct predictions & {correct_scores.mean():.6f} & {n_correct}" + r" \\",
    f"Incorrect predictions & {incorrect_scores.mean():.6f} & {n_incorrect}" + r" \\",
    r"\hline",
    r"\end{tabular}",
    r"\end{table}"
]

(OUT / "table13_replacement.tex").write_text(
    "\n".join(table_lines), encoding="utf-8"
)

direction = (
    "higher" if delta > 0 else "lower" if delta < 0 else "equal"
)

paragraph = (
    f"The final primary test cohort comprised 257 Cascade forced "
    f"predictions, including {n_correct} correct and {n_incorrect} incorrect "
    f"predictions. Mean CRUS was {correct_scores.mean():.6f} for correct "
    f"predictions and {incorrect_scores.mean():.6f} for incorrect predictions. "
    f"The mean difference (incorrect minus correct) was {delta:.6f}, "
    f"with a conditional 95% group-bootstrap percentile interval of "
    f"[{ci_low:.6f}, {ci_high:.6f}] based on {len(valid):,} valid replicates. "
    f"An exploratory two-sided Mann–Whitney comparison, with incorrect "
    f"cases as the first sample, gave U={u_incorrect_first:.6f}, "
    f"p={p_value:.6g}, and signed rank-biserial effect "
    f"r={rank_biserial:.6f}. The record-level p-value is not cluster-adjusted. "
    f"Observed mean CRUS was {direction} among incorrect predictions. "
    f"This analysis characterizes association with prediction correctness "
    f"within the fixed primary evaluation; it does not establish optimal "
    f"weights, independent external validity, or clinical harm reduction."
)

(OUT / "crus_signal_quality_replacement.txt").write_text(
    paragraph, encoding="utf-8"
)

response = (
    "Thank you for identifying the historical-result inconsistency. "
    "We regenerated Table 13 and the CRUS Signal Quality analysis from "
    "the final primary Cascade case-level predictions, verifying "
    "257 unique test records: 181 correct and 76 incorrect forced predictions. "
    f"The regenerated means are {correct_scores.mean():.6f} and "
    f"{incorrect_scores.mean():.6f}; the mean difference (incorrect minus "
    f"correct) is {delta:.6f}, with a conditional 95% group-bootstrap "
    f"interval [{ci_low:.6f}, {ci_high:.6f}]. "
    f"The exploratory two-sided Mann–Whitney statistic is "
    f"U={u_incorrect_first:.6f}, p={p_value:.6g}, with signed rank-biserial "
    f"effect r={rank_biserial:.6f}. The resampling unit, test convention, "
    "and conditional/dependence limitations are documented. "
    "[After editing the manuscript, confirm: We replaced the obsolete "
    "259-case table and historical inferential values and revised all "
    "associated interpretations.]"
)

(OUT / "editor_response_draft.txt").write_text(
    response, encoding="utf-8"
)

# Verify that source files have not changed.
require(
    all(sha256(path) == input_hashes[name]
        for name, path in source_paths.items()),
    "An input file changed during analysis."
)

checksum_lines = [
    f"{sha256(path)}  {path.name}"
    for path in sorted(OUT.iterdir())
    if path.is_file()
]

(OUT / "SHA256SUMS.txt").write_text(
    "\n".join(checksum_lines) + "\n", encoding="utf-8"
)

print("FINAL COHORT CHECK: PASSED")
print("N:", len(final))
print("Correct:", n_correct)
print("Incorrect:", n_incorrect)
print("Assigned test groups:", n_groups)
print("Maximum group size:", int(group_sizes.max()))
print("\nREGENERATED TABLE 13")
print(summary.to_string(index=False))

print("\nINFERENTIAL / EFFECT SUMMARY")
print("Mean difference, incorrect minus correct:", delta)
print("95% group-bootstrap CI:", [float(ci_low), float(ci_high)])
print("Valid bootstrap replicates:", len(valid))
print("Mann–Whitney U, incorrect sample first:", u_incorrect_first)
print("Two-sided exploratory p-value:", p_value)
print("Signed rank-biserial effect:", rank_biserial)

print("\nOUTPUT DIRECTORY:", OUT)
print("Original source files unchanged.")
print("No model inference or threshold selection performed.")

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path(_crus_local_path('project', ''))
BASE = ROOT / "corrected_experiment_outputs"

FILES = {
    "protocol": BASE / "final_pretest_protocol.json",
    "predictions": BASE / "test_all_method_predictions.csv",
    "gates": BASE / "test_frozen_CRUS_acceptance_audit.csv",
    "saved_curves": BASE / "test_ranking_risk_coverage_curves.csv",
    "manifest": ROOT / "data_audit/constrained_split_manifest.csv",
}

def require(condition, message):
    if not condition:
        raise ValueError(message)

def sha256(path):
    h = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def as_bool(series):
    values = series.astype("string").str.strip().str.lower()
    mapping = {
        "true": True, "false": False,
        "1": True, "0": False,
        "1.0": True, "0.0": False
    }
    require(
        values.isin(mapping).all(),
        f"Invalid Boolean values in {series.name}."
    )
    return values.map(mapping).to_numpy(dtype=bool)

for path in FILES.values():
    if not path.is_file():
        raise FileNotFoundError(
            f"Missing saved input: {path}\n"
            "Locate the saved file; do not regenerate model predictions."
        )

input_hashes = {
    name: sha256(path) for name, path in FILES.items()
}

protocol = json.loads(
    FILES["protocol"].read_text(encoding="utf-8")
)

require(
    input_hashes["manifest"] == protocol["main_manifest_sha256"],
    "Manifest checksum differs from the frozen protocol."
)
require(
    protocol["ranking_ties"] == "score descending; record ID ascending",
    "Unexpected ranking tie rule. Inspect the frozen protocol."
)

pred = pd.read_csv(
    FILES["predictions"],
    dtype={
        "record_id": "string",
        "method": "string",
        "true_label": "string",
        "prediction": "string",
    }
)
gates = pd.read_csv(
    FILES["gates"],
    dtype={"record_id": "string", "method": "string"}
)
manifest = pd.read_csv(
    FILES["manifest"],
    dtype={
        "record_id": "string",
        "label": "string",
        "split": "string",
    }
)
saved_curves = pd.read_csv(FILES["saved_curves"])

require(
    manifest[["record_id", "label", "split"]].notna().all().all(),
    "Missing manifest fields."
)
require(
    manifest["record_id"].is_unique,
    "Duplicate manifest IDs."
)

test = manifest.loc[
    manifest["split"].str.strip().str.lower().eq("test")
].set_index("record_id")

N = len(test)
require(
    N == 257 and N == int(protocol["split_sizes"]["test"]),
    "Unexpected primary test cohort."
)

required_pred = [
    "record_id", "method", "true_label",
    "prediction", "correct", "ranking_score"
]
require(
    set(required_pred).issubset(pred.columns),
    "Required prediction columns are missing."
)
require(
    pred[required_pred].notna().all().all(),
    "Missing prediction values."
)
require(
    not pred.duplicated(["method", "record_id"]).any(),
    "Duplicate method/record predictions."
)
require(
    not gates.duplicated(["method", "record_id"]).any(),
    "Duplicate native-gate audit records."
)

pred["ranking_score"] = pd.to_numeric(
    pred["ranking_score"], errors="raise"
)
require(
    np.isfinite(pred["ranking_score"]).all(),
    "Nonfinite ranking scores."
)

full_rows = []
curve_parts = []
prediction_groups = {}

# Reconstruct all ranking curves from the saved ranking score.
for method, part in pred.groupby("method", sort=True):
    part = part.copy()

    require(
        len(part) == N and set(part["record_id"]) == set(test.index),
        f"{method}: test IDs or size differ."
    )

    truth = test.loc[part["record_id"], "label"].to_numpy(dtype=str)

    require(
        np.array_equal(
            part["true_label"].to_numpy(dtype=str), truth
        ),
        f"{method}: reference labels disagree with manifest."
    )

    part["recomputed_correct"] = (
        part["prediction"].to_numpy(dtype=str) == truth
    )

    require(
        np.array_equal(
            as_bool(part["correct"]),
            part["recomputed_correct"].to_numpy()
        ),
        f"{method}: saved correctness disagrees with labels."
    )

    prediction_groups[method] = part.set_index("record_id")

    n_correct = int(part["recomputed_correct"].sum())

    full_rows.append({
        "method": method,
        "N": N,
        "correct": n_correct,
        "incorrect": N - n_correct,
        "accuracy": n_correct / N,
        "error": (N - n_correct) / N
    })

    ranked = part.sort_values(
        ["ranking_score", "record_id"],
        ascending=[False, True],
        kind="mergesort"
    )

    k = np.arange(1, N + 1)
    cumulative_correct = (
        ranked["recomputed_correct"].astype(int).cumsum().to_numpy()
    )

    curve_parts.append(pd.DataFrame({
        "method": method,
        "n_accepted": k,
        "n_correct_accepted": cumulative_correct,
        "n_errors_accepted": k - cumulative_correct,
        "coverage": k / N,
        "selective_accuracy": cumulative_correct / k,
        "risk": (k - cumulative_correct) / k,
        "abstention_penalized_accuracy": cumulative_correct / N
    }))

full = pd.DataFrame(full_rows)
curves = pd.concat(curve_parts, ignore_index=True)

require(
    {"Cascade", "SK4"}.issubset(prediction_groups),
    "Cascade or SK4 is missing."
)

# Check saved curves against independent reconstruction.
keys = ["method", "n_accepted"]
require(
    not saved_curves.duplicated(keys).any(),
    "Duplicate points in saved ranking curves."
)

old = saved_curves.set_index(keys).sort_index()
new = curves.set_index(keys).sort_index()

require(
    old.index.equals(new.index),
    "Saved and reconstructed curve keys differ."
)

curve_fields = [
    "n_correct_accepted", "n_errors_accepted",
    "coverage", "selective_accuracy", "risk",
    "abstention_penalized_accuracy"
]

for field in curve_fields:
    np.testing.assert_allclose(
        old[field].to_numpy(dtype=float),
        new[field].to_numpy(dtype=float),
        rtol=0,
        atol=1e-12,
        err_msg=f"Saved ranking curves disagree for {field}."
    )

# Verify every saved native CRUS gate.
gate_rows = []

for method, part in gates.groupby("method", sort=True):
    require(
        method in prediction_groups,
        f"Gate method {method} missing from predictions."
    )
    require(
        len(part) == N and set(part["record_id"]) == set(test.index),
        f"{method}: gate IDs or size differ."
    )

    pg = prediction_groups[method].sort_index()
    part = part.set_index("record_id").loc[pg.index].copy()
    policy = protocol["frozen_policies"][f"{method}:CRUS"]

    tc = float(policy["tau_c"])
    tu = float(policy["tau_u"])

    require(
        np.isfinite([tc, tu]).all(),
        f"{method}: invalid frozen thresholds."
    )

    for field in ["confidence", "CRUS", "tau_c", "tau_u"]:
        part[field] = pd.to_numeric(part[field], errors="raise")
        require(
            np.isfinite(part[field]).all(),
            f"{method}: invalid {field}."
        )

    np.testing.assert_allclose(
        part["tau_c"], tc, rtol=0, atol=1e-12
    )
    np.testing.assert_allclose(
        part["tau_u"], tu, rtol=0, atol=1e-12
    )

    accepted = (
        part["confidence"].ge(tc) & part["CRUS"].le(tu)
    ).to_numpy()

    require(
        np.array_equal(accepted, as_bool(part["accepted"])),
        f"{method}: saved acceptance differs from frozen policy."
    )

    correct = pg["recomputed_correct"].to_numpy(dtype=bool)

    require(
        np.array_equal(correct, as_bool(part["correct"])),
        f"{method}: gate-audit correctness differs."
    )

    a = int(accepted.sum())
    ca = int((accepted & correct).sum())
    ea = int((accepted & ~correct).sum())

    gate_rows.append({
        "method": method,
        "tau_c": tc,
        "tau_u": tu,
        "N": N,
        "accepted": a,
        "correct_accepted": ca,
        "accepted_errors": ea,
        "abstained": N - a,
        "coverage": a / N,
        "selective_accuracy": ca / a if a else np.nan,
        "selective_error": ea / a if a else np.nan,
        "penalized_accuracy": ca / N,
        "penalized_error": (ea + N - a) / N
    })

native = pd.DataFrame(gate_rows)
cascade = native.loc[native["method"].eq("Cascade")]
require(len(cascade) == 1, "Missing or duplicate Cascade gate.")
cascade = cascade.iloc[0]

require(
    np.isclose(cascade["tau_c"], 0, rtol=0, atol=1e-15)
    and np.isclose(
        cascade["tau_u"], 0.128165036474669,
        rtol=0, atol=1e-15
    ),
    "Unexpected primary Cascade thresholds."
)

require(
    (
        int(cascade["accepted"]),
        int(cascade["correct_accepted"]),
        int(cascade["accepted_errors"]),
        int(cascade["abstained"])
    ) == (117, 109, 8, 140),
    "Unexpected Cascade native-gate counts."
)

sk4 = full.loc[full["method"].eq("SK4")].iloc[0]
cascade_full = full.loc[full["method"].eq("Cascade")].iloc[0]

require(
    int(sk4["correct"]) == 82,
    "Unexpected SK4 full-coverage correct count."
)
require(
    int(cascade_full["correct"]) == 181,
    "Unexpected Cascade full-coverage correct count."
)

K = int(cascade["accepted"])
matched = curves.loc[curves["n_accepted"].eq(K)].copy()

cascade_ranked = matched.loc[matched["method"].eq("Cascade")].iloc[0]
require(
    int(cascade_ranked["n_correct_accepted"]) == 106,
    "Unexpected Cascade ranked top-117 result. Inspect the saved ranking."
)

# Save new audit outputs; never overwrite original results.
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
OUT = ROOT / f"editor_comment2_consistency_{stamp}"
OUT.mkdir(exist_ok=False)

full.to_csv(OUT / "verified_full_coverage.csv", index=False)
native.to_csv(OUT / "verified_native_operating_points.csv", index=False)
curves.to_csv(OUT / "verified_ranking_curves.csv", index=False)
matched.to_csv(OUT / "verified_ranked_matched_count.csv", index=False)

fig, ax = plt.subplots(figsize=(9, 6))

for method, part in curves.groupby("method", sort=True):
    ax.plot(
        100 * part["coverage"],
        100 * part["risk"],
        label=method,
        linewidth=1.5
    )

ax.scatter(
    [100 * cascade["coverage"]],
    [100 * cascade["selective_error"]],
    color="black",
    marker="X",
    s=100,
    zorder=10,
    label="Cascade native CRUS gate (separate policy)"
)

ax.set_xlabel("Coverage (%)")
ax.set_ylabel("Selective misclassification rate (%)")
ax.set_xlim(0, 100)
ax.set_ylim(0, 100)
ax.grid(alpha=0.25)
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(OUT / "risk_coverage_verified.pdf")
fig.savefig(OUT / "risk_coverage_verified.png", dpi=300)
plt.close(fig)

metadata = {
    "N": N,
    "ranking": protocol["ranking_ties"],
    "native_gate": "confidence >= tau_c AND CRUS <= tau_u",
    "source_sha256": input_hashes,
    "checks": {
        "manifest_matches_protocol": True,
        "all_prediction_ids_and_labels_verified": True,
        "all_saved_ranking_curve_points_verified": True,
        "all_saved_native_gate_decisions_verified": True
    },
    "scope": (
        "Numerical operating-point and ranking-curve audit only. "
        "Not a complete manuscript, threshold-sweep, or figure-asset audit."
    )
}

(OUT / "audit_metadata.json").write_text(
    json.dumps(metadata, indent=2), encoding="utf-8"
)

require(
    all(
        sha256(path) == input_hashes[name]
        for name, path in FILES.items()
    ),
    "An original input changed during the audit."
)

print("COMMENT 2 NUMERICAL AUDIT: PASSED")
print("\nFROZEN CASCADE THRESHOLDS")
print("tau_c =", repr(float(cascade["tau_c"])))
print("tau_u =", repr(float(cascade["tau_u"])))

print("\nFULL-COVERAGE RESULTS")
print(full.to_string(index=False))

print("\nNATIVE CRUS OPERATING POINTS")
print(native.to_string(index=False))

print("\nRANKED MATCHED-COUNT RESULTS")
print(
    matched[
        ["method", "n_accepted", "n_correct_accepted",
         "n_errors_accepted", "coverage", "risk"]
    ].to_string(index=False)
)

print("\nSK4 full-coverage error (%):", 100 * sk4["error"])
print("All saved ranking-curve points agree with reconstruction.")
print("All saved native gates agree with frozen thresholds.")
print("Original input files unchanged.")
print("No model training, inference, or threshold selection performed.")
print("Output directory:", OUT)

In [ ]:
from pathlib import Path
pass  # Colab service replaced by local workflow.

if not Path(_crus_local_path('drive', '')).is_dir():
    _crus_prepare_saved_workspace()

print(
    "Project available:",
    Path(_crus_local_path('project', '')).is_dir()
)

In [ ]:
print("Runtime connection is working")

In [ ]:
from pathlib import Path
pass  # Colab service replaced by local workflow.

if not Path(_crus_local_path('drive', '')).is_dir():
    _crus_prepare_saved_workspace()

BOOTSTRAP_DIR = Path(
    _crus_local_path('project', 'supplementary_repeated_splits/r1_4_conditional_bootstrap')
)

print("Bootstrap folder available:", BOOTSTRAP_DIR.is_dir())

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

# Google Drive is already mounted.
BASE = Path(
    _crus_local_path('project', 'supplementary_repeated_splits/r1_4_conditional_bootstrap')
)

def require(condition, message):
    if not condition:
        raise ValueError(message)

def read_json(filename):
    path = BASE / filename
    print(f"Reading: {filename}", flush=True)
    if not path.is_file():
        raise FileNotFoundError(path)
    return json.loads(path.read_text(encoding="utf-8"))

def read_csv(filename):
    path = BASE / filename
    print(f"Reading: {filename}", flush=True)
    if not path.is_file():
        raise FileNotFoundError(path)
    return pd.read_csv(path)

require(
    BASE.is_dir(),
    f"Bootstrap folder is unavailable: {BASE}"
)

# --------------------------------------------------
# 1. Read metadata without loading or hashing sources
# --------------------------------------------------

metadata = read_json("analysis_metadata.json")

print("\n=== BOOTSTRAP METHODS METADATA ===", flush=True)

for key, value in metadata.items():
    if key != "input_hashes":
        print(f"{key}: {value}", flush=True)

print(
    "Number of recorded source files:",
    len(metadata.get("input_hashes", {})),
    flush=True
)

B = int(metadata["bootstrap_replicates"])

# --------------------------------------------------
# 2. Read saved mean intervals and paired differences
# --------------------------------------------------

means = read_csv("mean_metrics_conditional_ci95.csv")
paired = read_csv("paired_differences_conditional_ci95.csv")

mean_required = {
    "method", "mode", "metric", "mean",
    "conditional_ci95_lower", "conditional_ci95_upper",
    "valid_bootstrap_replicates"
}
paired_required = {
    "comparison", "metric", "mean_difference",
    "conditional_ci95_lower", "conditional_ci95_upper",
    "valid_bootstrap_replicates"
}

require(
    mean_required.issubset(means.columns),
    f"Missing mean columns: {sorted(mean_required - set(means.columns))}"
)
require(
    paired_required.issubset(paired.columns),
    f"Missing paired columns: {sorted(paired_required - set(paired.columns))}"
)
require(
    not means.duplicated(["method", "mode", "metric"]).any(),
    "Duplicate method/mode/metric rows in saved means."
)

sd_col = next(
    (
        col for col in
        ["sample_sd_across_repeats", "sample_sd"]
        if col in means.columns
    ),
    None
)
require(sd_col is not None, "Missing across-repeat sample-SD column.")

# Basic consistency checks on saved summary files.
for name, frame in [("means", means), ("paired", paired)]:
    require(not frame.empty, f"The {name} summary is empty.")

    lower = pd.to_numeric(
        frame["conditional_ci95_lower"], errors="raise"
    ).to_numpy()
    upper = pd.to_numeric(
        frame["conditional_ci95_upper"], errors="raise"
    ).to_numpy()
    valid = pd.to_numeric(
        frame["valid_bootstrap_replicates"], errors="raise"
    ).to_numpy()

    require(
        np.isfinite(lower).all() and np.isfinite(upper).all(),
        f"Nonfinite interval limits in {name}."
    )
    require(
        (lower <= upper).all(),
        f"Reversed interval limits in {name}."
    )
    require(
        np.isfinite(valid).all()
        and (valid >= 0.99 * B).all()
        and (valid <= B).all()
        and (valid == np.floor(valid)).all(),
        f"Unexpected valid-replicate counts in {name}."
    )

if "n_repeats" in means.columns:
    require(
        means["n_repeats"].eq(5).all(),
        "Some mean-result rows do not summarize five repeats."
    )

# --------------------------------------------------
# 3. Table 7 accuracy: percentages, not proportions
# --------------------------------------------------

accuracy = means.loc[
    means["metric"].eq("selective_accuracy")
].copy()

require(
    not accuracy.empty,
    "No selective_accuracy rows found. Inspect metric names."
)

scale_cols = [
    "mean", sd_col,
    "conditional_ci95_lower", "conditional_ci95_upper"
]

for col in scale_cols:
    accuracy[col] = 100 * pd.to_numeric(
        accuracy[col], errors="raise"
    )

accuracy = accuracy.rename(columns={
    "mean": "mean_percent",
    sd_col: "sample_SD_percentage_points",
    "conditional_ci95_lower": "CI_lower_percent",
    "conditional_ci95_upper": "CI_upper_percent",
})

print("\n=== TABLE 7 ACCURACY: PERCENT SCALE ===", flush=True)
print(
    accuracy[
        [
            "method", "mode",
            "mean_percent", "sample_SD_percentage_points",
            "CI_lower_percent", "CI_upper_percent",
            "valid_bootstrap_replicates"
        ]
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    ),
    flush=True
)

# --------------------------------------------------
# 4. Cascade coverage
# --------------------------------------------------

coverage = means.loc[
    means["method"].eq("Cascade")
    & means["metric"].eq("coverage")
].copy()

require(
    not coverage.empty,
    "No Cascade coverage rows found."
)

for col in scale_cols:
    coverage[col] = 100 * pd.to_numeric(
        coverage[col], errors="raise"
    )

coverage = coverage.rename(columns={
    "mean": "mean_percent",
    sd_col: "sample_SD_percentage_points",
    "conditional_ci95_lower": "CI_lower_percent",
    "conditional_ci95_upper": "CI_upper_percent",
})

print("\n=== CASCADE COVERAGE: PERCENT SCALE ===", flush=True)
print(
    coverage[
        [
            "method", "mode",
            "mean_percent", "sample_SD_percentage_points",
            "CI_lower_percent", "CI_upper_percent",
            "valid_bootstrap_replicates"
        ]
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    ),
    flush=True
)

# --------------------------------------------------
# 5. Paired accuracy differences: percentage points
# --------------------------------------------------

differences = paired.loc[
    paired["metric"].eq("selective_accuracy")
].copy()

require(
    not differences.empty,
    "No paired selective_accuracy rows found."
)

for col in [
    "mean_difference",
    "conditional_ci95_lower", "conditional_ci95_upper"
]:
    differences[col] = 100 * pd.to_numeric(
        differences[col], errors="raise"
    )

differences = differences.rename(columns={
    "mean_difference": "difference_percentage_points",
    "conditional_ci95_lower": "CI_lower_percentage_points",
    "conditional_ci95_upper": "CI_upper_percentage_points",
})

display_cols = [
    col for col in [
        "comparison",
        "cascade_mode",
        "comparator_mode",
        "difference_direction",
        "difference_percentage_points",
        "CI_lower_percentage_points",
        "CI_upper_percentage_points",
        "valid_bootstrap_replicates"
    ]
    if col in differences.columns
]

print(
    "\n=== PAIRED ACCURACY DIFFERENCES: PERCENTAGE POINTS ===",
    flush=True
)
print(
    differences[display_cols].to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    ),
    flush=True
)

# --------------------------------------------------
# 6. State the inspection scope
# --------------------------------------------------

print("\nLIGHTWEIGHT SUMMARY INSPECTION: PASSED", flush=True)
print("No files written or modified.", flush=True)
print("No prediction arrays read.", flush=True)
print("No model inference or bootstrap rerun.", flush=True)
print(
    "This inspection checks saved summary structure and reported metadata. "
    "It does not independently reproduce intervals, verify source hashes, "
    "or establish agreement with the current manuscript.",
    flush=True
)

In [ ]:
from pathlib import Path
import json
import pandas as pd

BASE = Path(
    _crus_local_path('project', 'supplementary_repeated_splits/medcpt_conditional_bootstrap')
)

metadata_path = BASE / "analysis_metadata.json"
results_path = BASE / "medcpt_mean_conditional_ci95.csv"

for path in [metadata_path, results_path]:
    print("Reading:", path.name, flush=True)
    if not path.is_file():
        raise FileNotFoundError(path)

metadata = json.loads(
    metadata_path.read_text(encoding="utf-8")
)

print("\n=== MEDCPT METADATA ===")
for key, value in metadata.items():
    if key != "input_hashes":
        print(f"{key}: {value}")

results = pd.read_csv(results_path)
accuracy = results.loc[
    results["metric"].eq("selective_accuracy")
].copy()

for column in [
    "mean", "sample_sd", "sample_sd_across_repeats",
    "conditional_ci95_lower", "conditional_ci95_upper"
]:
    if column in accuracy.columns:
        accuracy[column] *= 100

print("\n=== MEDCPT ACCURACY: PERCENT SCALE ===")
print(accuracy.to_string(index=False))
print("\nRead-only inspection complete. No files modified.")